
# Proyecto final de Machine Learning

## Actividad 0. Exploración preliminar y evaluación de viabilidad de OULAD

### 0.1. Propósito

Antes de formular definitivamente el problema de investigación, se realizará una exploración estructural del Open University Learning Analytics Dataset (OULAD), con el propósito de evaluar su idoneidad para desarrollar modelos de predicción temprana del riesgo académico.

El proyecto busca investigar si la incorporación de características longitudinales del comportamiento estudiantil mejora la capacidad predictiva de modelos de Machine Learning, en comparación con modelos que utilizan exclusivamente variables estáticas.

### 0.2. Objetivos de la exploración preliminar

1. Identificar los archivos, variables y relaciones del dataset.
2. Determinar el número de estudiantes, matrículas, módulos y períodos académicos.
3. Examinar la distribución de los resultados académicos.
4. Verificar la existencia de registros temporales de participación, evaluaciones y retiro.
5. Estimar la disponibilidad de estudiantes activos en diferentes momentos del curso.
6. Identificar las principales restricciones para construir modelos de alerta temprana.

### 0.3. Consideración metodológica

OULAD contiene información de resultados académicos por módulo y registros longitudinales de interacción con un entorno virtual de aprendizaje.

Sin embargo, no contiene necesariamente una trayectoria institucional completa que permita identificar la deserción definitiva de la universidad.

Por tanto, el estudio se orientará preliminarmente a la predicción temprana de resultados académicos adversos, tales como reprobación o retiro del módulo, como aproximación metodológica a un futuro sistema de alerta temprana de riesgo de deserción universitaria.

### 0.4. Fuente de datos

- Dataset: Open University Learning Analytics Dataset (OULAD).
- Fuente: Kaggle.
- Dirección: https://www.kaggle.com/datasets/anlgrbz/student-demographics-online-education-dataoulad

**Referencia bibliográfica preliminar (IEEE):**

[1] J. Kuzilek, M. Hlosta y Z. Zdrahal, "Open University Learning Analytics Dataset," *Scientific Data*, vol. 4, art. 170171, 2017. doi: 10.1038/sdata.2017.171.


In [ ]:
# ============================================================
# CELDA 0 - DRIVE PERSISTENTE Y COMPUTO LOCAL EN /content
# Ejecutar primero en CADA sesión de Google Colab.
# No sobrescribe resultados científicos congelados.
# ============================================================
from google.colab import drive
from pathlib import Path
from IPython import get_ipython
import os, shutil, hashlib, fnmatch, time

drive.mount('/content/drive')
RUTA_BASE = '/content/drive/MyDrive/Proyecto_ML'
RUTA_DRIVE = Path(RUTA_BASE)
RUTA_LOCAL = Path('/content')
RUTA_ARTEFACTOS = RUTA_DRIVE / 'artefactos'
RUTA_SNAPSHOTS = RUTA_DRIVE / 'snapshots'
RUTA_DATOS = RUTA_DRIVE / 'datos'
RUTA_NOTEBOOK = RUTA_DRIVE / 'notebook'
for carpeta in (RUTA_ARTEFACTOS, RUTA_SNAPSHOTS, RUTA_DATOS, RUTA_NOTEBOOK):
    carpeta.mkdir(parents=True, exist_ok=True)

# IMPORTANTE: se han evaluado los datos de prueba una sola vez.
# Un archivo de test congelado NO debe sustituirse por un nuevo cálculo.
RESULTADOS_PROTEGIDOS_OULAD = {
    'protocolo_oulad_337.json',
    'metricas_test_oulad_339.csv',
    'predicciones_test_oulad_339.csv',
    'bootstrap_test_oulad_339.csv',
    'diferencias_test_oulad_339.csv',
    'retiros_test_oulad_339.csv',
    'subtipos_test_oulad_339.csv',
}
HASH_PROTOCOLO_337_ESPERADO = '283bbddb0c56a615dde6a8a1201e149dfd7020f4672d7d18267e4c93ad880625'
HASH_MODELOS_338_CONOCIDO = 'f5c7b6ba03a7bcd040bf99e338734352fda11300d1f680f615d90a3c0124aa68'
PATRONES_ARCHIVOS_OULAD = (
    '*oulad*', 'transformer_C*', 'figura_*.png',
    'resultados_entrenamiento_*.csv', 'auditoria_rubrica_*.csv',
    'metricas_complementarias_*.csv', 'README_oulad*',
    'indice_notebook_oulad*',
)
_CACHE_RESPALDO_OULAD = {}

def sha256_archivo_oulad(ruta):
    h=hashlib.sha256()
    with open(ruta,'rb') as f:
        for chunk in iter(lambda:f.read(8*1024*1024),b''):
            h.update(chunk)
    return h.hexdigest()

def es_archivo_oulad(ruta):
    ruta=Path(ruta)
    return (ruta.is_file() and not ruta.name.startswith('estado_')
            and any(fnmatch.fnmatch(ruta.name.lower(),p.lower()) for p in PATRONES_ARCHIVOS_OULAD)
            and not ruta.name.endswith(('.tmp','.partial')))

def _modelo_338_completo(ruta):
    # El checkpoint 3.38 contiene hasta ocho pipelines.
    try:
        import joblib
        m=joblib.load(ruta)
        return isinstance(m,dict) and len(m)==8
    except Exception:
        return False

def copiar_verificado_oulad(origen, destino):
    """Respalda después de escritura local, con comprobación de integridad."""
    origen,destino=Path(origen),Path(destino)
    if not origen.is_file(): raise FileNotFoundError(origen)
    destino.parent.mkdir(parents=True, exist_ok=True)
    hash_o=sha256_archivo_oulad(origen)
    if origen.name=='protocolo_oulad_337.json' and hash_o != HASH_PROTOCOLO_337_ESPERADO:
        raise RuntimeError('Protocolo 3.37 distinto al SHA congelado. No se respalda.')
    if destino.exists():
        if sha256_archivo_oulad(destino)==hash_o: return False
        if origen.name in RESULTADOS_PROTEGIDOS_OULAD:
            raise RuntimeError(f'Conflicto con resultado congelado en Drive: {origen.name}')
        if origen.name=='modelos_finales_oulad_338.joblib' and _modelo_338_completo(destino):
            raise RuntimeError('El checkpoint 3.38 ya contiene ocho modelos en Drive. No sobrescribir.')
    temp=destino.with_name('.'+destino.name+'.partial')
    try:
        shutil.copy2(origen,temp)
        if sha256_archivo_oulad(temp)!=hash_o:
            raise IOError('Copia incompleta o alterada: '+origen.name)
        os.replace(str(temp),str(destino))
    finally:
        if temp.exists(): temp.unlink()
    return True

def respaldar_checkpoint_oulad(ruta):
    """Usar DENTRO del bucle de entrenamiento, después de cerrar el checkpoint."""
    ruta=Path(ruta)
    destino=RUTA_ARTEFACTOS/ruta.name
    cambio=copiar_verificado_oulad(ruta,destino)
    if cambio: print(f'[Drive] Checkpoint respaldado: {ruta.name}',flush=True)
    return cambio

def respaldar_oulad(verbose=True):
    total=0; fallos=[]
    for ruta in RUTA_LOCAL.iterdir():
        if not es_archivo_oulad(ruta): continue
        estado=(ruta.stat().st_size,ruta.stat().st_mtime_ns)
        if _CACHE_RESPALDO_OULAD.get(ruta.name)==estado and (RUTA_ARTEFACTOS/ruta.name).exists():
            continue
        try:
            if copiar_verificado_oulad(ruta,RUTA_ARTEFACTOS/ruta.name): total+=1
            _CACHE_RESPALDO_OULAD[ruta.name]=estado
        except Exception as exc:
            fallos.append(f'{ruta.name}: {exc}')
    if verbose or total: print(f'[Drive] Archivos actualizados: {total}')
    for mensaje in fallos: print('[Drive] ERROR DE RESPALDO:',mensaje)
    return total,fallos

def restaurar_oulad():
    count=0; errores=[]
    for ruta in RUTA_ARTEFACTOS.iterdir():
        if not ruta.is_file() or ruta.name.startswith('.'): continue
        local=RUTA_LOCAL/ruta.name
        if local.exists():
            if ruta.name in RESULTADOS_PROTEGIDOS_OULAD or (ruta.name=='modelos_finales_oulad_338.joblib' and _modelo_338_completo(ruta)):
                if sha256_archivo_oulad(local)!=sha256_archivo_oulad(ruta):
                    errores.append(f'Conflicto local/Drive: {ruta.name}')
            continue
        shutil.copy2(ruta,local)
        if sha256_archivo_oulad(local)!=sha256_archivo_oulad(ruta):
            raise IOError('Restauración incompleta: '+ruta.name)
        count+=1
    if errores:
        raise RuntimeError('ARCHIVOS CONFLICTIVOS, revisar antes de continuar: '+'; '.join(errores))
    print(f'[Drive] Restaurados a /content: {count} archivo(s)')

# El evento es idempotente si esta celda se ejecuta dos veces.
ip=get_ipython()
_h=globals().get('_HOOK_OULAD_ANTERIOR')
if _h is not None:
    try: ip.events.unregister('post_run_cell',_h)
    except ValueError: pass

def _post_celda_oulad(resultado):
    # Si la celda lanza excepción, un checkpoint parcial ya cerrado
    # también puede existir; respaldarlo no implica que sea final.
    total, errores=respaldar_oulad(verbose=False)
    if errores: print('[Drive] ADVERTENCIA: corregir errores de respaldo antes de reiniciar.')

# Prioridad: conservar cualquier resultado local que todavía no esté en Drive.
print('=== Primer respaldo de la sesión ===')
_,errores_iniciales=respaldar_oulad(verbose=True)
if errores_iniciales:
    print('ADVERTENCIA: hay conflictos de respaldo. No borrar la sesión actual.')
restaurar_oulad()
ip.events.register('post_run_cell',_post_celda_oulad)
_HOOK_OULAD_ANTERIOR=_post_celda_oulad
print('RUTA_BASE:',RUTA_BASE)
print('Ejecución/entrenamiento: /content')
print('Backups: Drive/artefactos | Drive/snapshots | Drive/datos')


In [ ]:
# ============================================================
# CELDA 0.1 - SNAPSHOTS DE OBJETOS EN MEMORIA
# Se serializa primero en /content (rápido) y se copia a Drive.
# Únicamente cargar archivos propios y confiables de este proyecto.
# ============================================================
from pathlib import Path
from IPython import get_ipython
import joblib, os

SNAPSHOTS_PERMITIDOS_OULAD = {
    'oulad_tablas','cohortes_oulad','experimentos_oulad',
    'cv_splits_oulad','pipelines_oulad',
}

def guardar_estado_oulad(*nombres):
    espacio=get_ipython().user_ns
    for nombre in nombres:
        if nombre not in SNAPSHOTS_PERMITIDOS_OULAD:
            raise ValueError(f'Snapshot no autorizado: {nombre}')
        if nombre not in espacio:
            raise NameError(f'Variable inexistente: {nombre}')
        ruta_local=RUTA_LOCAL / f'estado_{nombre}.joblib'
        temp=RUTA_LOCAL / f'.estado_{nombre}.tmp'
        try:
            joblib.dump(espacio[nombre],temp,compress=3)
            os.replace(str(temp),str(ruta_local))
            respaldado=copiar_verificado_oulad(ruta_local,RUTA_SNAPSHOTS/ruta_local.name)
        finally:
            if temp.exists(): temp.unlink()
        print(f'[Drive] Snapshot {nombre}: {ruta_local.stat().st_size/1024**2:.1f} MB; actualizado={respaldado}')

def recuperar_estado_oulad(*nombres, sobrescribir=False):
    espacio=get_ipython().user_ns
    for nombre in nombres:
        if nombre not in SNAPSHOTS_PERMITIDOS_OULAD:
            raise ValueError(f'Snapshot no autorizado: {nombre}')
        if not sobrescribir and nombre in espacio:
            print(f'[Memoria] {nombre}: ya existe; se conserva')
            continue
        ruta=RUTA_SNAPSHOTS/f'estado_{nombre}.joblib'
        if not ruta.exists():
            print(f'[Drive] Falta snapshot {nombre}. Ejecutar su celda de construcción una vez.')
            continue
        copia_local=RUTA_LOCAL / ruta.name
        if not copia_local.exists() or sha256_archivo_oulad(copia_local)!=sha256_archivo_oulad(ruta):
            shutil.copy2(ruta,copia_local)
        espacio[nombre]=joblib.load(copia_local)
        print(f'[Drive] Recuperado {nombre} ({type(espacio[nombre]).__name__})')

def snapshot_disponible_oulad(nombre):
    return (RUTA_SNAPSHOTS/f'estado_{nombre}.joblib').exists()

def dataset_en_drive_oulad():
    nombres=['courses.csv','assessments.csv','studentAssessment.csv',
      'studentInfo.csv','studentRegistration.csv','vle.csv','studentVle.csv']
    return all((RUTA_DATOS/n).exists() for n in nombres)

def preparar_dataset_oulad_local():
    if not dataset_en_drive_oulad():
        raise FileNotFoundError('No se encontró OULAD completo en Drive')
    carpeta=RUTA_LOCAL/'cache_dataset_oulad'
    carpeta.mkdir(parents=True,exist_ok=True)
    for origen in RUTA_DATOS.glob('*.csv'):
        destino=carpeta/origen.name
        if not destino.exists() or destino.stat().st_size!=origen.stat().st_size:
            shutil.copy2(origen,destino)
            print('[Drive → /content]',origen.name,flush=True)
    return carpeta

def respaldar_dataset_oulad(rutas):
    # Primera vez: ~443 MB; en futuras sesiones no se vuelve a copiar.
    for nombre,ruta in rutas.items():
        origen=Path(ruta)
        destino=RUTA_DATOS/nombre
        if destino.exists() and destino.stat().st_size==origen.stat().st_size:
            continue
        print('[Dataset → Drive]',nombre,flush=True)
        copiar_verificado_oulad(origen,destino)
    print('[Drive] Dataset OULAD almacenado para próximas sesiones.')

print('Snapshots: disponibles. Para recuperar el espacio de entrenamiento:')
print("recuperar_estado_oulad('experimentos_oulad','cv_splits_oulad','pipelines_oulad')")


In [ ]:

# ============================================================
# VERIFICACIÓN DE PERSISTENCIA Y RECUPERACIÓN
# No entrena modelos ni abre el test independiente.
# ============================================================

print("=== 1. SNAPSHOTS EN GOOGLE DRIVE ===")

objetos = [
    "oulad_tablas",
    "cohortes_oulad",
    "experimentos_oulad",
    "cv_splits_oulad",
    "pipelines_oulad"
]

for nombre in objetos:
    disponible = snapshot_disponible_oulad(nombre)

    print(
        nombre,
        "DISPONIBLE" if disponible else "PENDIENTE"
    )

print("\n=== 2. DATASET ORIGINAL EN DRIVE ===")

print(
    "OULAD completo:",
    dataset_en_drive_oulad()
)

print("\n=== 3. RECUPERACIÓN DE MATRICES ===")

recuperar_estado_oulad(
    "experimentos_oulad",
    "cv_splits_oulad",
    "pipelines_oulad"
)

print("\n=== 4. VERIFICACIÓN DE SEMANA 8 ===")

if "experimentos_oulad" in globals():

    datos = experimentos_oulad[8]

    print(
        "X_A_train:",
        datos["X_A_train"].shape
    )

    print(
        "X_B_train:",
        datos["X_B_train"].shape
    )

    print(
        "y_train:",
        datos["y_train"].shape
    )

    print(
        "grupos_train:",
        datos["grupos_train"].shape
    )

    assert datos["X_A_train"].shape == (
        21164, 12
    )

    assert datos["X_B_train"].shape == (
        21164, 48
    )

    print(
        "\nMATRICES ORIGINALES: RECUPERADAS"
    )

else:

    print(
        "\nLas matrices todavía no tienen "
        "un snapshot guardado."
    )

    print(
        "Debemos ejecutar una sola vez "
        "la preparación original hasta 3.4."
    )

print("\n=== VERIFICACIÓN FINALIZADA ===")


### Guía de persistencia y ejecución (Google Drive)

**Ruta permanente:** `/content/drive/MyDrive/Proyecto_ML`. Los datos originales se conservan en `datos/`, los modelos y CSV en `artefactos/` y las matrices de entrenamiento en `snapshots/`. **El procesamiento y entrenamiento se realizan localmente en `/content`.**

**Inicio de cada sesión:** ejecutar celdas **0** y **0.1**. Recuperan automáticamente los archivos locales; para restaurar matrices en memoria usar `recuperar_estado_oulad("experimentos_oulad", "cv_splits_oulad", "pipelines_oulad")`. Cuando no hay snapshots, ejecutar una vez las celdas originales de preparación de datos hasta **3.6** para generarlos.

**Reanudación de entrenamientos largos:** Transformer (3.21, 3.23), Optuna (3.33) y entrenamiento definitivo (3.38) respaldan sus checkpoints por fold o modelo; no desconectar Colab antes de confirmar el aviso `[Drive]` del respaldo. Los artefactos de la **evaluación independiente 3.39 están congelados**: esa celda únicamente recupera los resultados anteriores; nunca ejecuta otra vez `predict` sobre test.

**Ejecutar todo de nuevo:** se pueden ejecutar las actividades de preparación y exploración, pero los experimentos anteriores pueden ser costosos; no es obligatorio recomputar los entrenamientos que tengan checkpoints válidos. Reutilizar los resultados guardados y no seleccionar nuevos modelos con el test. **No subir `datos/` ni `snapshots/` al repositorio GitHub**, por tamaño y posibles consideraciones de datos.

**Advertencia:** los respaldos ocurren al terminar una celda y en los checkpoints añadidos; una caída a mitad de una celda sin checkpoint puede perder el avance de esa celda. Los outputs visibles del notebook no son sustituto de los archivos en Drive.


In [ ]:

# ============================================================
# PROYECTO FINAL DE MACHINE LEARNING
# ACTIVIDAD 0: EXPLORACIÓN PRELIMINAR DE OULAD
# ============================================================

# Asistencia IA: ChatGPT, octubre de 2026.
# Solicitud del usuario:
# "Es necesario que exploremos la base de OULAD
# previamente antes de comenzar, de tal manera
# de verificar la estructura y etiquetas disponibles."
#
# Objetivo: descargar el dataset, identificar su estructura,
# examinar las etiquetas y verificar ventanas temporales.

import sys
import subprocess
from pathlib import Path

import pandas as pd
import numpy as np
from IPython.display import display

# ------------------------------------------------------------
# 1. INSTALACIÓN Y DESCARGA DEL DATASET
# ------------------------------------------------------------

try:
    import kagglehub
except ImportError:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "kagglehub"
    ])
    import kagglehub

DATASET = (
    "anlgrbz/"
    "student-demographics-online-education-dataoulad"
)

print("Descargando o recuperando OULAD desde Kaggle...")

# Reutilizar la copia persistente en Drive cuando exista.
# Nunca leer studentVle.csv directamente desde Drive al entrenar.
if dataset_en_drive_oulad():
    oulad_base = preparar_dataset_oulad_local()
else:
    oulad_base = Path(kagglehub.dataset_download(DATASET))

print("Directorio:", oulad_base)

# ------------------------------------------------------------
# 2. IDENTIFICACIÓN Y CARGA DE ARCHIVOS
# ------------------------------------------------------------

archivos = [
    "courses.csv",
    "assessments.csv",
    "studentAssessment.csv",
    "studentInfo.csv",
    "studentRegistration.csv",
    "studentVle.csv",
    "vle.csv"
]

archivos_encontrados = {
    p.name.lower(): p
    for p in oulad_base.rglob("*.csv")
}

faltantes = [
    nombre for nombre in archivos
    if nombre.lower() not in archivos_encontrados
]

if faltantes:
    raise FileNotFoundError(
        f"No se encontraron estos archivos: {faltantes}"
    )

oulad_rutas = {
    nombre: archivos_encontrados[nombre.lower()]
    for nombre in archivos
}

oulad_tablas = {}
inventario = []

for nombre, ruta in oulad_rutas.items():

    if nombre == "studentVle.csv":
        # Solo lectura de cabecera y 5 filas.
        muestra = pd.read_csv(ruta, nrows=5)

        # Contar filas sin cargar todo el CSV en RAM.
        total_lineas = int(
            subprocess.check_output(
                ["wc", "-l", str(ruta)],
                text=True
            ).strip().split()[0]
        )
        filas = total_lineas - 1
        columnas = len(muestra.columns)

        oulad_vle_muestra = muestra

    else:
        df = pd.read_csv(ruta)
        oulad_tablas[nombre] = df
        filas, columnas = df.shape

    inventario.append({
        "Archivo": nombre,
        "Registros": filas,
        "Variables": columnas,
        "Tamaño MB": round(
            ruta.stat().st_size / (1024 ** 2), 2
        )
    })

print("\n=== INVENTARIO DEL DATASET ===")
display(pd.DataFrame(inventario))

# ------------------------------------------------------------
# 3. VARIABLES DISPONIBLES
# ------------------------------------------------------------

print("\n=== ESTRUCTURA DE VARIABLES ===")

for nombre in archivos:

    if nombre == "studentVle.csv":
        df = oulad_vle_muestra
    else:
        df = oulad_tablas[nombre]

    print(f"\n{nombre}")
    print("Columnas:", list(df.columns))

# ------------------------------------------------------------
# 4. INFORMACIÓN GENERAL DE LOS ESTUDIANTES
# ------------------------------------------------------------

info = oulad_tablas["studentInfo.csv"]
registro = oulad_tablas["studentRegistration.csv"]
cursos = oulad_tablas["courses.csv"]
evaluaciones = oulad_tablas["studentAssessment.csv"]

claves = [
    "code_module",
    "code_presentation",
    "id_student"
]

print("\n=== ESTADÍSTICAS GENERALES ===")

print("Total de matrículas:", len(info))
print("Estudiantes únicos:", info["id_student"].nunique())
print("Módulos:", info["code_module"].nunique())
print(
    "Presentaciones de módulos:",
    info[
        ["code_module", "code_presentation"]
    ].drop_duplicates().shape[0]
)

print(
    "Estudiantes con más de una matrícula:",
    (info["id_student"].value_counts() > 1).sum()
)

print(
    "Claves duplicadas en studentInfo:",
    info.duplicated(subset=claves).sum()
)

print(
    "Claves duplicadas en studentRegistration:",
    registro.duplicated(subset=claves).sum()
)

print("\nDuración de los módulos (días):")
display(cursos["module_presentation_length"].describe())

# ------------------------------------------------------------
# 5. DISTRIBUCIÓN DE RESULTADOS ACADÉMICOS
# ------------------------------------------------------------

print("\n=== DISTRIBUCIÓN DE FINAL_RESULT ===")

distribucion = info["final_result"].value_counts()

tabla_resultados = pd.DataFrame({
    "Resultado": distribucion.index,
    "Estudiantes / matrículas": distribucion.values,
    "Porcentaje": (
        distribucion.values / len(info) * 100
    ).round(2)
})

display(tabla_resultados)

# Etiqueta exploratoria: resultado académico adverso.
riesgo = info["final_result"].isin([
    "Fail", "Withdrawn"
])

print("\n=== ETIQUETA BINARIA EXPLORATORIA ===")
print(
    "Resultados adversos (Fail + Withdrawn):",
    int(riesgo.sum())
)
print(
    "Resultados favorables (Pass + Distinction):",
    int((~riesgo).sum())
)
print(
    "Porcentaje de resultados adversos:",
    round(riesgo.mean() * 100, 2), "%"
)

# ------------------------------------------------------------
# 6. VALIDACIÓN PRELIMINAR DE FECHAS DE RETIRO
# ------------------------------------------------------------

print("\n=== FECHAS DE RETIRO ===")

oulad_integrada = info.merge(
    registro,
    on=claves,
    how="left",
    validate="one_to_one",
    indicator=True
)

print(
    "Matrículas sin coincidencia en registro:",
    int(
        (oulad_integrada["_merge"] != "both").sum()
    )
)

oulad_integrada = oulad_integrada.drop(
    columns="_merge"
)

retirados = (
    oulad_integrada["final_result"] == "Withdrawn"
)

sin_fecha_retiro = (
    retirados &
    oulad_integrada["date_unregistration"].isna()
)

print("Total Withdrawn:", int(retirados.sum()))

print(
    "Withdrawn sin fecha de retiro:",
    int(sin_fecha_retiro.sum())
)

print(
    "Retiros registrados antes del día 0:",
    int(
        (
            retirados &
            (oulad_integrada["date_unregistration"] < 0)
        ).sum()
    )
)

# ------------------------------------------------------------
# 7. VIABILIDAD DE VENTANAS DE PREDICCIÓN
# ------------------------------------------------------------

print("\n=== VENTANAS TEMPORALES EXPLORATORIAS ===")

ventanas = [14, 28, 42, 56]
resultados_ventanas = []

for dia in ventanas:

    # Estudiantes que no se habían retirado al día d.
    # Se excluyen los Withdrawn con fecha desconocida.

    activos = oulad_integrada[
        (
            oulad_integrada["date_unregistration"].isna()
            |
            (oulad_integrada["date_unregistration"] > dia)
        )
        &
        (~sin_fecha_retiro)
    ].copy()

    adverso = activos["final_result"].isin([
        "Fail", "Withdrawn"
    ])

    retiro_futuro = (
        (activos["final_result"] == "Withdrawn")
        &
        (activos["date_unregistration"] > dia)
    )

    resultados_ventanas.append({
        "Semana": dia // 7,
        "Día de corte": dia,
        "Matrículas elegibles": len(activos),
        "Resultados adversos finales": int(adverso.sum()),
        "Retiros posteriores": int(retiro_futuro.sum()),
        "% adversos": round(
            adverso.mean() * 100, 2
        ) if len(activos) else np.nan
    })

tabla_ventanas = pd.DataFrame(
    resultados_ventanas
)

display(tabla_ventanas)

# ------------------------------------------------------------
# 8. REGISTROS LONGITUDINALES
# ------------------------------------------------------------

print("\n=== EJEMPLO DE REGISTROS STUDENTVLE ===")
display(oulad_vle_muestra)

print("\n=== EJEMPLO DE EVALUACIONES ===")
display(evaluaciones.head())

print("\n=== EJEMPLO DE REGISTROS DE MATRÍCULA ===")
display(registro.head())

print("\nEXPLORACIÓN PRELIMINAR FINALIZADA")
print("No se han entrenado modelos ni transformado datos.")

# Respaldo del origen: una sola transferencia; las siguientes sesiones usan cache local.
respaldar_dataset_oulad(oulad_rutas)
guardar_estado_oulad('oulad_tablas')



### 0.5. Resultados de la exploración preliminar

La exploración del dataset OULAD permitió identificar 32.593 matrículas correspondientes a 28.785 estudiantes distintos, distribuidas en 7 módulos y 22 presentaciones académicas.

Se identificaron 10.655.280 registros de actividad en la plataforma virtual y 173.912 registros de evaluaciones, que constituyen fuentes relevantes para la construcción de características longitudinales.

La distribución de resultados académicos fue:

- Pass: 12.361 matrículas (37,93 %).
- Distinction: 3.024 matrículas (9,28 %).
- Fail: 7.052 matrículas (21,64 %).
- Withdrawn: 10.156 matrículas (31,16 %).

Al agrupar los resultados en dos categorías, se obtuvieron 17.208 resultados adversos (52,80 %) y 15.385 favorables (47,20 %). Esta distribución presenta un balance global relativamente equilibrado.

La exploración temporal preliminar identificó retiros posteriores a las semanas 2, 4, 6 y 8, lo que respalda la viabilidad de investigar predicciones anticipadas.

También se identificaron 93 registros de retiro sin fecha conocida y 2.676 retiros anteriores al inicio del módulo. Estos casos requieren un tratamiento específico para evitar inconsistencias temporales.

### 0.6. Conclusión de viabilidad

OULAD reúne condiciones favorables para desarrollar un estudio comparativo entre modelos que utilizan variables estáticas y modelos que incorporan características longitudinales de actividad académica.

La disponibilidad de registros temporales permite investigar si las tendencias de participación, la inactividad y los indicadores académicos tempranos mejoran la predicción de resultados adversos.

Sin embargo, todavía no es posible determinar si estas características aportan una mejora estadísticamente significativa. Esta cuestión deberá resolverse mediante experimentos controlados y comparación estadística de modelos.

El dataset tampoco permite identificar directamente la deserción institucional definitiva, por lo que el alcance predictivo se limitará al retiro y la reprobación de módulos académicos.

Los resultados servirán como estudio piloto para el desarrollo futuro de modelos de alerta temprana de deserción universitaria que integren información académica, longitudinal y espacio-temporal.


In [ ]:

# ============================================================
# ACTIVIDAD 0.7 - VALIDACIÓN DE VENTANAS TEMPORALES
# ============================================================

# Prompt de asistencia IA:
# "Verificar la elegibilidad temporal de cada matrícula
# considerando fecha de inscripción, fecha de retiro,
# registros incompletos y resultados posteriores al corte."

base = oulad_integrada.copy()

base["es_retiro"] = base["final_result"].eq("Withdrawn")

base["retiro_sin_fecha"] = (
    base["es_retiro"]
    & base["date_unregistration"].isna()
)

print("=== AUDITORÍA DE FECHAS ===")

print(
    "Matrículas sin fecha de inscripción:",
    base["date_registration"].isna().sum()
)

print(
    "Retiros sin fecha:",
    base["retiro_sin_fecha"].sum()
)

print(
    "Registros con fecha de retiro, pero resultado "
    "distinto de Withdrawn:",
    (
        ~base["es_retiro"]
        & base["date_unregistration"].notna()
    ).sum()
)

print(
    "Registros cuya fecha de retiro es anterior "
    "a la inscripción:",
    (
        base["date_unregistration"]
        < base["date_registration"]
    ).sum()
)

# ------------------------------------------------------------
# RECONSTRUCCIÓN DE LA POBLACIÓN ELEGIBLE
# ------------------------------------------------------------

filas = []

for semana in [2, 4, 6, 8]:

    dia = semana * 7

    # El estudiante debe haberse inscrito a más tardar
    # en el día de corte.
    inscrito = (
        base["date_registration"].notna()
        & (base["date_registration"] <= dia)
    )

    # El estudiante no puede haberse retirado
    # antes o durante el día de corte.
    no_retirado = (
        base["date_unregistration"].isna()
        | (base["date_unregistration"] > dia)
    )

    # Excluir retirados cuya fecha es desconocida.
    fecha_valida = ~base["retiro_sin_fecha"]

    elegibles = base.loc[
        inscrito & no_retirado & fecha_valida
    ].copy()

    elegibles["target_adverso"] = (
        elegibles["final_result"]
        .isin(["Fail", "Withdrawn"])
        .astype(int)
    )

    retiro_futuro = (
        elegibles["es_retiro"]
        & (elegibles["date_unregistration"] > dia)
    )

    filas.append({
        "Semana": semana,
        "Día": dia,
        "Matrículas elegibles": len(elegibles),
        "Favorables": int(
            (elegibles["target_adverso"] == 0).sum()
        ),
        "Adversos": int(
            (elegibles["target_adverso"] == 1).sum()
        ),
        "Retiros futuros": int(retiro_futuro.sum()),
        "% adversos": round(
            100 * elegibles["target_adverso"].mean(),
            2
        ) if len(elegibles) else np.nan,
        "% retiros futuros": round(
            100 * retiro_futuro.mean(),
            2
        ) if len(elegibles) else np.nan
    })

tabla_ventanas_corregida = pd.DataFrame(filas)

print("\n=== VENTANAS TEMPORALES CORREGIDAS ===")
display(tabla_ventanas_corregida)

print("\n=== DIFERENCIAS CON EL PRIMER CÁLCULO ===")

comparacion = tabla_ventanas[
    ["Semana", "Matrículas elegibles"]
].merge(
    tabla_ventanas_corregida[
        ["Semana", "Matrículas elegibles"]
    ],
    on="Semana",
    suffixes=("_original", "_corregida")
)

comparacion["Diferencia"] = (
    comparacion["Matrículas elegibles_original"]
    - comparacion["Matrículas elegibles_corregida"]
)

display(comparacion)

print("\nValidación temporal preliminar completada.")


### 0.7. Validación de la elegibilidad temporal

La validación temporal permitió corregir la población elegible para cada ventana de predicción, considerando tanto la fecha de inscripción como la fecha de retiro del estudiante.

Se identificaron 45 matrículas sin fecha de inscripción, 93 retiros sin fecha registrada y 9 registros con fecha de retiro, pero con un resultado final diferente de `Withdrawn`. No se encontraron retiros anteriores a la fecha de inscripción.

Las poblaciones elegibles corregidas fueron:

| Semana | Matrículas elegibles | Resultados adversos | Retiros futuros |
|---|---:|---:|---:|
| 2 | 27.962 | 12.602 | 5.580 |
| 4 | 27.422 | 12.044 | 5.012 |
| 6 | 26.912 | 11.533 | 4.500 |
| 8 | 26.413 | 11.032 | 3.998 |

La existencia de resultados adversos y retiros posteriores a cada ventana permite plantear experimentos de clasificación anticipada.

Sin embargo, la cantidad de casos disponibles no demuestra todavía que dichos eventos sean predecibles mediante las variables registradas.

### 0.8. Conclusiones de la exploración preliminar

La exploración confirmó que OULAD ofrece información adecuada para investigar el valor predictivo de las trayectorias longitudinales del comportamiento estudiantil.

Los principales hallazgos fueron:

1. Se identificaron 32.593 matrículas correspondientes a 28.785 estudiantes distintos.
2. Se dispone de 10.655.280 registros de actividad en la plataforma virtual, que pueden utilizarse para construir indicadores longitudinales.
3. La etiqueta exploratoria de resultados académicos adversos presenta una distribución global relativamente equilibrada: 52,80 % de resultados adversos frente a 47,20 % favorables.
4. Existen poblaciones elegibles y eventos posteriores suficientes para investigar predicciones en las semanas 2, 4, 6 y 8.
5. Se identificaron registros temporales incompletos o inconsistentes que deberán tratarse antes del modelado.

**Conclusión principal:** OULAD es viable como dataset para evaluar si la incorporación de características longitudinales mejora la predicción temprana de resultados académicos adversos frente al uso exclusivo de información estática.

Esta conclusión establece la viabilidad del experimento, pero no confirma aún la hipótesis de superioridad predictiva del modelo longitudinal.



# **Título del proyecto: Evaluación del valor predictivo de trayectorias longitudinales del comportamiento estudiantil para la identificación temprana del riesgo académico mediante Machine Learning: estudio comparativo con OULAD**

## 1. Introducción

### 1.1. Contexto del problema

La identificación temprana de estudiantes en riesgo académico representa un desafío para las instituciones de educación superior, debido a que los resultados adversos pueden afectar la continuidad de los estudios y contribuir a trayectorias de rezago o abandono universitario.

Los modelos tradicionales de predicción académica pueden utilizar características relativamente estáticas, como información demográfica, antecedentes educativos y condiciones iniciales del estudiante. Sin embargo, estas variables no describen cómo evoluciona su comportamiento durante el período académico.

Las plataformas virtuales de aprendizaje generan registros temporales de participación, acceso a recursos, actividades y evaluaciones. Esta información permite construir trayectorias longitudinales capaces de representar cambios en el comportamiento académico.

El análisis de estas trayectorias podría proporcionar información adicional para identificar estudiantes en riesgo antes de que se produzcan resultados académicos adversos.

### 1.2. Planteamiento del problema

Aunque las características estáticas pueden utilizarse para estimar el riesgo académico, se desconoce cuánto valor predictivo adicional aportan los patrones longitudinales de participación y desempeño del estudiante.

También resulta necesario determinar en qué momento del período académico es posible obtener predicciones útiles y cómo cambia su calidad a medida que se incorpora nueva información.

En este contexto, el problema consiste en evaluar si las características longitudinales mejoran la capacidad predictiva de modelos de Machine Learning frente al uso exclusivo de variables estáticas, evitando utilizar información posterior al momento de predicción.

### 1.3. Pregunta de investigación

¿En qué medida la incorporación de características longitudinales del comportamiento estudiantil mejora el rendimiento de los modelos de Machine Learning para predecir tempranamente resultados académicos adversos, en comparación con el uso exclusivo de variables estáticas, y con cuánta anticipación pueden identificarse estudiantes en riesgo?

### 1.4. Objetivo general

Evaluar el aporte de las características longitudinales del comportamiento estudiantil a la predicción temprana de resultados académicos adversos, mediante el desarrollo, optimización y comparación estadística de modelos de aprendizaje supervisado utilizando el dataset OULAD.

### 1.5. Objetivos específicos

1. Analizar la estructura, calidad, distribución y disponibilidad temporal de las variables académicas y de actividad virtual contenidas en OULAD.

2. Construir conjuntos de características estáticas y longitudinales que permitan representar el perfil inicial y la evolución del comportamiento académico de los estudiantes.

3. Implementar modelos de aprendizaje supervisado mediante pipelines reproducibles que integren preprocesamiento, selección de características y optimización de hiperparámetros.

4. Comparar el rendimiento de los modelos entrenados con características exclusivamente estáticas frente a aquellos que incorporan información longitudinal, mediante métricas de clasificación y pruebas estadísticas.

5. Evaluar la capacidad predictiva utilizando ventanas temporales correspondientes a las semanas 2, 4, 6 y 8 del período académico.

6. Analizar el potencial y las limitaciones de las características longitudinales como fundamento metodológico para futuros sistemas de alerta temprana de deserción universitaria.

### 1.6. Hipótesis de investigación

**Hipótesis principal (H1):**

La incorporación de características longitudinales del comportamiento estudiantil mejora el rendimiento predictivo de los modelos de Machine Learning frente al uso exclusivo de características estáticas.

**Hipótesis nula (H0):**

La incorporación de características longitudinales no mejora el rendimiento predictivo de los modelos de Machine Learning frente al uso exclusivo de características estáticas.

La hipótesis será evaluada mediante comparaciones pareadas bajo particiones equivalentes de datos, utilizando un nivel de significancia de α = 0,05.

La métrica principal propuesta será F1-score macro, complementada con otras métricas de clasificación que hacens entido al proyecto.

### 1.7. Alcance del estudio

La investigación utilizará el Open University Learning Analytics Dataset (OULAD), integrado por información demográfica, registros de matrícula, evaluaciones y actividad en un entorno virtual de aprendizaje.

Se estudiará una tarea de clasificación supervisada donde los resultados académicos adversos estarán definidos inicialmente como `Fail` o `Withdrawn`, mientras que los resultados favorables corresponderán a `Pass` o `Distinction`.

Se construirán modelos basados exclusivamente en variables estáticas y modelos que incorporen características longitudinales calculadas hasta distintas semanas del curso.

La investigación se limitará a la predicción de resultados académicos por módulo. No se interpretarán las etiquetas observadas como deserción institucional definitiva.

Los resultados permitirán valorar la utilidad de esta metodología como prueba de concepto para una futura investigación sobre predicción de deserción universitaria mediante trayectorias académicas y datos espacio-temporales.

### 1.8. Resultado esperado

Se espera determinar experimentalmente si la información longitudinal aporta una mejora estadísticamente significativa respecto de la información estática.

Asimismo, se buscará identificar la ventana temporal más temprana en la que los modelos alcancen un rendimiento predictivo útil, considerando tanto la capacidad de identificación de estudiantes en riesgo como la anticipación disponible para una posible intervención.

La investigación no presupone que los modelos longitudinales serán superiores. Las conclusiones dependerán de los resultados observados durante la evaluación experimental.



## 2. Análisis Exploratorio de Datos (EDA)

### 2.1. Auditoría estructural, calidad e integridad de los datos

#### Objetivo

Evaluar la calidad y consistencia de las tablas que conforman el dataset OULAD, identificando valores faltantes, registros duplicados, claves de identificación y posibles inconsistencias en las relaciones entre archivos.

#### Justificación metodológica

OULAD es un dataset relacional compuesto por siete tablas que contienen información demográfica, académica y registros de actividad virtual.

Para construir modelos predictivos longitudinales resulta indispensable verificar que estas fuentes puedan integrarse correctamente, conservando la identidad de cada estudiante, el módulo académico y el período de presentación correspondiente.

La auditoría evaluará:

1. Tipos de datos y estructura de las tablas.
2. Valores faltantes y sus proporciones.
3. Duplicación de claves que deberían ser únicas.
4. Integridad referencial entre las tablas.
5. Inconsistencias entre resultados finales y fechas de retiro.
6. Disponibilidad de los registros de actividad longitudinal.

**Consideración:** no todos los valores faltantes representan errores. Por ejemplo, una fecha de retiro vacía puede indicar que el estudiante no abandonó el módulo. La interpretación de cada ausencia dependerá de la definición de la variable.

Durante esta actividad no se eliminarán registros, no se modificarán las variables originales y no se realizará entrenamiento de modelos.

El archivo `studentVle.csv`, que contiene más de diez millones de registros, será auditado de forma incremental en la siguiente actividad temporal, para evitar cargas innecesarias de memoria.

#### Resultado esperado

Identificar los problemas de calidad que requieren tratamiento y confirmar si las claves relacionales permiten construir una base integrada de estudiante-módulo-presentación, apta para posteriores experimentos longitudinales.

**Fundamento bibliográfico:** la exploración y caracterización del dataset se apoyan en la descripción original de OULAD [1]. Para las métricas de clasificación y la implementación, se utiliza scikit-learn [2]; en situaciones con clases desequilibradas conviene complementar ROC-AUC con precisión y recall [12].


In [ ]:

# ============================================================
# PROYECTO FINAL DE MACHINE LEARNING
# ACTIVIDAD 2.1 - AUDITORÍA ESTRUCTURAL Y RELACIONAL DE OULAD
# ============================================================

# Asistencia IA: ChatGPT, octubre de 2026.
# Prompt:
# "Realizar una auditoría de calidad del dataset OULAD,
# incluyendo tipos de datos, valores faltantes, duplicados,
# claves primarias, integridad referencial e inconsistencias
# entre resultados académicos y fechas de retiro."

import pandas as pd
from IPython.display import display

# ------------------------------------------------------------
# 1. VERIFICAR QUE LAS TABLAS ESTÁN DISPONIBLES
# ------------------------------------------------------------

tablas = oulad_tablas

nombres = [
    "courses.csv",
    "assessments.csv",
    "studentAssessment.csv",
    "studentInfo.csv",
    "studentRegistration.csv",
    "vle.csv"
]

faltantes = [n for n in nombres if n not in tablas]

if faltantes:
    raise ValueError(
        f"Faltan tablas de la Actividad 0: {faltantes}"
    )

print("=== 1. ESTRUCTURA GENERAL ===")

resumen = []

for nombre in nombres:
    df = tablas[nombre]

    resumen.append({
        "Tabla": nombre,
        "Filas": len(df),
        "Columnas": len(df.columns),
        "Valores nulos": int(df.isna().sum().sum()),
        "Duplicados exactos": int(df.duplicated().sum())
    })

display(pd.DataFrame(resumen))

# ------------------------------------------------------------
# 2. TIPOS DE DATOS
# ------------------------------------------------------------

print("\n=== 2. TIPOS DE DATOS POR TABLA ===")

for nombre in nombres:
    df = tablas[nombre]

    print(f"\n{nombre}")
    display(
        pd.DataFrame({
            "Variable": df.columns,
            "Tipo": df.dtypes.astype(str).values
        })
    )

# ------------------------------------------------------------
# 3. ANÁLISIS DE VALORES FALTANTES
# ------------------------------------------------------------

print("\n=== 3. VALORES FALTANTES ===")

nulos = []

for nombre in nombres:
    df = tablas[nombre]

    for columna in df.columns:

        cantidad = int(df[columna].isna().sum())

        if cantidad > 0:
            nulos.append({
                "Tabla": nombre,
                "Variable": columna,
                "Nulos": cantidad,
                "Porcentaje": round(
                    cantidad / len(df) * 100, 3
                )
            })

tabla_nulos = pd.DataFrame(nulos)

if len(tabla_nulos):
    display(
        tabla_nulos.sort_values(
            "Porcentaje", ascending=False
        ).reset_index(drop=True)
    )
else:
    print("No se encontraron valores faltantes.")

# ------------------------------------------------------------
# 4. AUDITORÍA DE CLAVES ÚNICAS
# ------------------------------------------------------------

print("\n=== 4. UNICIDAD DE CLAVES ===")

claves_esperadas = {
    "courses.csv": [
        "code_module", "code_presentation"
    ],
    "assessments.csv": [
        "id_assessment"
    ],
    "studentAssessment.csv": [
        "id_assessment", "id_student"
    ],
    "studentInfo.csv": [
        "code_module", "code_presentation",
        "id_student"
    ],
    "studentRegistration.csv": [
        "code_module", "code_presentation",
        "id_student"
    ],
    "vle.csv": [
        "code_module", "code_presentation",
        "id_site"
    ]
}

resultados_claves = []

for nombre, claves in claves_esperadas.items():

    df = tablas[nombre]

    duplicados = int(
        df.duplicated(subset=claves).sum()
    )

    claves_nulas = int(
        df[claves].isna().any(axis=1).sum()
    )

    resultados_claves.append({
        "Tabla": nombre,
        "Clave": " + ".join(claves),
        "Duplicados de clave": duplicados,
        "Claves incompletas": claves_nulas
    })

display(pd.DataFrame(resultados_claves))

# ------------------------------------------------------------
# 5. COMPROBACIÓN DE INTEGRIDAD REFERENCIAL
# ------------------------------------------------------------

print("\n=== 5. INTEGRIDAD ENTRE TABLAS ===")

def comprobar_relacion(
    hijo, padre, claves, descripcion
):
    """
    Comprueba si las claves del archivo hijo
    existen en el archivo padre.
    No modifica los datos originales.
    """

    claves_validas = hijo[claves].notna().all(axis=1)

    filas_validas = hijo.loc[
        claves_validas, claves
    ]

    claves_padre = padre[claves].drop_duplicates()

    comprobacion = filas_validas.merge(
        claves_padre,
        on=claves,
        how="left",
        indicator=True
    )

    huerfanos = int(
        comprobacion["_merge"].eq("left_only").sum()
    )

    return {
        "Relación": descripcion,
        "Filas revisadas": len(hijo),
        "Claves incompletas": int((~claves_validas).sum()),
        "Sin correspondencia": huerfanos
    }


info = tablas["studentInfo.csv"]
registro = tablas["studentRegistration.csv"]
cursos = tablas["courses.csv"]
assessments = tablas["assessments.csv"]
student_assessment = tablas["studentAssessment.csv"]
vle = tablas["vle.csv"]

clave_matricula = [
    "code_module", "code_presentation",
    "id_student"
]

clave_curso = [
    "code_module", "code_presentation"
]

relaciones = []

# Matrículas frente a información académica.
relaciones.append(
    comprobar_relacion(
        registro, info, clave_matricula,
        "Registro -> Información estudiante"
    )
)

# Evaluaciones pertenecientes a módulos existentes.
relaciones.append(
    comprobar_relacion(
        assessments, cursos, clave_curso,
        "Evaluaciones -> Cursos"
    )
)

# Calificaciones referidas a evaluaciones existentes.
relaciones.append(
    comprobar_relacion(
        student_assessment,
        assessments,
        ["id_assessment"],
        "Calificaciones -> Evaluaciones"
    )
)

# Recursos virtuales vinculados a cursos.
relaciones.append(
    comprobar_relacion(
        vle, cursos, clave_curso,
        "Recursos VLE -> Cursos"
    )
)

# Reconstruir módulo y presentación por evaluación.
assessment_con_curso = student_assessment.merge(
    assessments[
        ["id_assessment"] + clave_curso
    ],
    on="id_assessment",
    how="left",
    indicator=True
)

assessment_con_curso = assessment_con_curso[
    assessment_con_curso["_merge"] == "both"
].drop(columns="_merge")

# Verificar que la calificación corresponde a
# una matrícula existente.
relaciones.append(
    comprobar_relacion(
        assessment_con_curso,
        info,
        clave_matricula,
        "Calificaciones -> Matrículas"
    )
)

display(pd.DataFrame(relaciones))

# ------------------------------------------------------------
# 6. CONSISTENCIA DE FECHAS Y RESULTADOS
# ------------------------------------------------------------

print("\n=== 6. CONSISTENCIA TEMPORAL ===")

base_auditoria = info.merge(
    registro,
    on=clave_matricula,
    how="left",
    validate="one_to_one"
)

retiro = base_auditoria["final_result"].eq(
    "Withdrawn"
)

fecha_retiro = (
    base_auditoria["date_unregistration"].notna()
)

contradictorios = base_auditoria[
    (~retiro) & fecha_retiro
]

auditoria_temporal = pd.DataFrame([
    {
        "Control": "Inscripción sin fecha",
        "Casos": int(
            base_auditoria["date_registration"].isna().sum()
        )
    },
    {
        "Control": "Withdrawn sin fecha de retiro",
        "Casos": int(
            (retiro & ~fecha_retiro).sum()
        )
    },
    {
        "Control": "No Withdrawn con fecha de retiro",
        "Casos": len(contradictorios)
    },
    {
        "Control": "Retiro antes de inscripción",
        "Casos": int((
            base_auditoria["date_unregistration"]
            < base_auditoria["date_registration"]
        ).sum())
    }
])

display(auditoria_temporal)

print("\nDetalle de resultados contradictorios:")

if len(contradictorios):
    display(
        contradictorios.groupby(
            "final_result"
        ).size().reset_index(name="Casos")
    )
else:
    print("No se encontraron casos contradictorios.")

# ------------------------------------------------------------
# 7. VERIFICACIÓN DEL ARCHIVO LONGITUDINAL
# ------------------------------------------------------------

print("\n=== 7. ESTRUCTURA DE STUDENTVLE ===")

ruta_vle = oulad_rutas["studentVle.csv"]

muestra_vle = pd.read_csv(
    ruta_vle,
    nrows=5
)

print("Columnas:", muestra_vle.columns.tolist())
print("Tipos de datos:")
display(muestra_vle.dtypes.to_frame("Tipo"))

print("\nEjemplo de registros longitudinales:")
display(muestra_vle)

print(
    "\nLa auditoría completa de studentVle "
    "se realizará mediante lectura por bloques."
)

# ------------------------------------------------------------
# 8. RESUMEN
# ------------------------------------------------------------

print("\n=== AUDITORÍA 2.1 FINALIZADA ===")
print("No se han modificado los archivos originales.")
print("No se han eliminado ni imputado registros.")
print("No se han entrenado modelos.")



### 2.1.1. Resultados e interpretación de la auditoría

La auditoría estructural del dataset OULAD permitió verificar la integridad de las seis tablas principales cargadas en memoria, excluyendo temporalmente la auditoría completa del archivo longitudinal `studentVle.csv`.

**Integridad estructural y relacional**

No se identificaron registros exactamente duplicados ni duplicaciones en las claves únicas esperadas de las tablas analizadas.

Las cinco relaciones verificadas entre estudiantes, matrículas, cursos, evaluaciones, calificaciones y recursos virtuales no presentaron registros sin correspondencia.

Estos resultados respaldan la posibilidad de integrar las fuentes de información utilizando las claves correspondientes a estudiante, módulo y presentación académica.

**Valores faltantes**

Se identificaron valores faltantes con distintas características:

- `date_unregistration`: 22.521 registros (69,098 %).
- `week_from` y `week_to`: 5.243 registros cada una (82,385 %).
- `imd_band`: 1.111 registros (3,409 %).
- `assessments.date`: 11 registros (5,340 %).
- `studentAssessment.score`: 173 registros (0,099 %).
- `date_registration`: 45 registros (0,138 %).

La existencia de valores faltantes no implica necesariamente errores de calidad. Su tratamiento deberá considerar el significado de cada variable y su disponibilidad temporal.

**Consistencia temporal**

Se identificaron 93 retiros sin fecha registrada y 9 matrículas clasificadas como `Fail` que también poseen fecha de desregistro.

No se observaron retiros anteriores a las respectivas fechas de inscripción.

Estas inconsistencias requieren un tratamiento explícito para evitar utilizar como observaciones activas matrículas que ya habían terminado antes de la ventana predictiva.

### 2.1.2. Conclusión

Las tablas auditadas presentan condiciones favorables de integridad estructural y referencial para desarrollar un modelo predictivo supervisado.

No obstante, será necesario controlar los registros temporales incompletos, las discrepancias entre fechas y resultados académicos, y los valores faltantes relevantes durante el preprocesamiento.

Las transformaciones, imputaciones y selección de características se implementarán posteriormente dentro de pipelines, utilizando exclusivamente los datos de entrenamiento para evitar fuga de información.

La auditoría integral de la tabla longitudinal `studentVle.csv` permanece pendiente.



### 2.2. Distribución de clases y heterogeneidad académica

#### Objetivo

Analizar la distribución de los resultados académicos y evaluar el balance de las clases entre los diferentes módulos, presentaciones académicas y ventanas temporales de predicción.

#### Justificación metodológica

En problemas de clasificación supervisada, el balance de las clases puede influir en el entrenamiento, la evaluación y la interpretación de los modelos.

Un dataset globalmente equilibrado puede contener subpoblaciones con distribuciones muy diferentes. Por tanto, es necesario analizar el comportamiento de las etiquetas tanto a nivel general como dentro de los grupos académicos.

Para el experimento principal se propone la siguiente clasificación binaria:

| Clase | Resultados originales | Interpretación |
|---|---|---|
| 0 | Pass, Distinction | Resultado favorable |
| 1 | Fail, Withdrawn | Resultado académico adverso |

La clase positiva representa un resultado adverso del módulo, no una confirmación de deserción universitaria.

#### Análisis propuesto

Se examinarán:

1. Distribución global de las cuatro etiquetas originales.
2. Proporción de resultados favorables y adversos por módulo.
3. Diferencias entre presentaciones académicas.
4. Evolución de la distribución de clases entre las semanas 2, 4, 6 y 8.
5. Exactitud de referencia de un clasificador que siempre predice la clase mayoritaria.

El análisis permitirá determinar si el desbalance de clases debe considerarse durante el entrenamiento y la comparación de algoritmos.

No se modificarán las etiquetas originales ni se aplicarán técnicas de balanceo durante esta etapa.


In [ ]:

# ============================================================
# ACTIVIDAD 2.2 - DISTRIBUCIÓN DE CLASES EN OULAD
# ============================================================

# Asistencia IA: ChatGPT, octubre de 2026.
# Prompt:
# "Analizar el balance de clases del dataset OULAD
# globalmente, por módulo, por presentación académica
# y por ventana temporal de predicción. Comparar la
# proporción de resultados adversos y calcular un
# baseline de clase mayoritaria, sin entrenar modelos."

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# ------------------------------------------------------------
# 1. PREPARACIÓN DE DATOS
# ------------------------------------------------------------

datos = tablas["studentInfo.csv"].copy()
registro = tablas["studentRegistration.csv"].copy()

claves = [
    "code_module", "code_presentation", "id_student"
]

orden_resultados = [
    "Pass", "Distinction", "Fail", "Withdrawn"
]

# Verificar etiquetas conocidas
etiquetas_desconocidas = set(
    datos["final_result"].dropna().unique()
) - set(orden_resultados)

if etiquetas_desconocidas or datos["final_result"].isna().any():
    raise ValueError(
        f"Etiquetas inesperadas: {etiquetas_desconocidas}"
    )

# Etiqueta exploratoria
datos["target_adverso"] = (
    datos["final_result"]
    .isin(["Fail", "Withdrawn"])
    .astype(int)
)

# ------------------------------------------------------------
# 2. DISTRIBUCIÓN GLOBAL
# ------------------------------------------------------------

print("\n=== 2.2.1 DISTRIBUCIÓN GLOBAL ===")

tabla_global = (
    datos["final_result"]
    .value_counts()
    .reindex(orden_resultados, fill_value=0)
    .rename_axis("Resultado")
    .reset_index(name="Matrículas")
)

tabla_global["Porcentaje"] = (
    100 * tabla_global["Matrículas"] / len(datos)
).round(2)

display(tabla_global)

plt.figure(figsize=(9, 4))

plt.bar(
    tabla_global["Resultado"],
    tabla_global["Porcentaje"]
)

plt.title("Distribución global de resultados académicos")
plt.ylabel("Porcentaje de matrículas (%)")
plt.xlabel("Resultado final")
plt.ylim(0, tabla_global["Porcentaje"].max() * 1.2)
plt.grid(axis="y", alpha=0.25)

for i, valor in enumerate(tabla_global["Porcentaje"]):
    plt.text(
        i, valor + 0.5,
        f"{valor:.2f}%",
        ha="center"
    )

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 3. RESULTADOS POR MÓDULO
# ------------------------------------------------------------

print("\n=== 2.2.2 BALANCE POR MÓDULO ===")

tabla_modulos = (
    datos.groupby("code_module")
    .agg(
        Matriculas=("id_student", "size"),
        Estudiantes_unicos=("id_student", "nunique"),
        Adversos=("target_adverso", "sum")
    )
    .reset_index()
)

tabla_modulos["Favorables"] = (
    tabla_modulos["Matriculas"] - tabla_modulos["Adversos"]
)

tabla_modulos["% adversos"] = (
    100 * tabla_modulos["Adversos"]
    / tabla_modulos["Matriculas"]
).round(2)

tabla_modulos = tabla_modulos.sort_values(
    "% adversos", ascending=False
)

display(tabla_modulos)

print("\nPorcentaje de cada resultado original por módulo:")

conteos_modulos = pd.crosstab(
    datos["code_module"],
    datos["final_result"]
).reindex(
    columns=orden_resultados,
    fill_value=0
)

porcentajes_modulos = (
    conteos_modulos
    .div(conteos_modulos.sum(axis=1), axis=0)
    .mul(100)
    .round(2)
)

display(porcentajes_modulos)

plt.figure(figsize=(9, 4))

plt.barh(
    tabla_modulos["code_module"],
    tabla_modulos["% adversos"]
)

plt.axvline(
    100 * datos["target_adverso"].mean(),
    linestyle="--",
    label="Porcentaje global de adversos"
)

plt.xlabel("Resultados adversos (%)")
plt.ylabel("Módulo")
plt.title("Proporción de resultados adversos por módulo")
plt.xlim(0, 100)
plt.legend()
plt.grid(axis="x", alpha=0.25)
plt.gca().invert_yaxis()

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 4. RESULTADOS POR PRESENTACIÓN ACADÉMICA
# ------------------------------------------------------------

print("\n=== 2.2.3 BALANCE POR PRESENTACIÓN ===")

tabla_presentaciones = (
    datos.groupby(
        ["code_module", "code_presentation"]
    )
    .agg(
        Matriculas=("id_student", "size"),
        Adversos=("target_adverso", "sum"),
        Retiros=(
            "final_result",
            lambda x: int((x == "Withdrawn").sum())
        ),
        Reprobados=(
            "final_result",
            lambda x: int((x == "Fail").sum())
        )
    )
    .reset_index()
)

tabla_presentaciones["% adversos"] = (
    100
    * tabla_presentaciones["Adversos"]
    / tabla_presentaciones["Matriculas"]
).round(2)

tabla_presentaciones = tabla_presentaciones.sort_values(
    ["code_module", "code_presentation"]
)

display(tabla_presentaciones)

# ------------------------------------------------------------
# 5. DISTRIBUCIÓN POR VENTANA TEMPORAL
# ------------------------------------------------------------

print("\n=== 2.2.4 BALANCE POR SEMANA ===")

base = datos.merge(
    registro,
    on=claves,
    how="left",
    validate="one_to_one"
)

base["retiro_sin_fecha"] = (
    base["final_result"].eq("Withdrawn")
    & base["date_unregistration"].isna()
)

resultados_semanales = []
resultados_modulo_semana = []

for semana in [2, 4, 6, 8]:

    dia = semana * 7

    # Matriculados antes o durante el corte
    inscrito = (
        base["date_registration"].notna()
        & (base["date_registration"] <= dia)
    )

    # Matrículas sin desregistro hasta la fecha
    activo = (
        base["date_unregistration"].isna()
        | (base["date_unregistration"] > dia)
    )

    elegibles = base.loc[
        inscrito
        & activo
        & (~base["retiro_sin_fecha"])
    ].copy()

    n = len(elegibles)

    adversos = int(elegibles["target_adverso"].sum())
    favorables = n - adversos

    porcentaje = 100 * adversos / n if n else np.nan

    # Baseline teórico:
    # predecir siempre la clase más frecuente
    baseline = (
        100 * max(adversos, favorables) / n
        if n else np.nan
    )

    resultados_semanales.append({
        "Semana": semana,
        "Matrículas": n,
        "Favorables": favorables,
        "Adversos": adversos,
        "% adversos": round(porcentaje, 2),
        "Accuracy clase mayoritaria (%)": round(
            baseline, 2
        )
    })

    por_modulo = (
        elegibles.groupby("code_module")
        .agg(
            Matriculas=("id_student", "size"),
            Adversos=("target_adverso", "sum")
        )
        .reset_index()
    )

    por_modulo["Semana"] = semana

    por_modulo["% adversos"] = (
        100
        * por_modulo["Adversos"]
        / por_modulo["Matriculas"]
    ).round(2)

    resultados_modulo_semana.append(por_modulo)

tabla_balance_semanal = pd.DataFrame(
    resultados_semanales
)

tabla_modulo_semana = pd.concat(
    resultados_modulo_semana,
    ignore_index=True
)

display(tabla_balance_semanal)

print("\nDistribución por módulo y ventana temporal:")
display(tabla_modulo_semana)

# ------------------------------------------------------------
# 6. VISUALIZAR EVOLUCIÓN DEL BALANCE
# ------------------------------------------------------------

plt.figure(figsize=(9, 4))

plt.plot(
    tabla_balance_semanal["Semana"],
    tabla_balance_semanal["% adversos"],
    marker="o",
    label="Resultados adversos"
)

plt.plot(
    tabla_balance_semanal["Semana"],
    100 - tabla_balance_semanal["% adversos"],
    marker="s",
    label="Resultados favorables"
)

plt.xlabel("Semana de predicción")
plt.ylabel("Porcentaje de matrículas (%)")
plt.title("Distribución de clases entre ventanas temporales")
plt.xticks([2, 4, 6, 8])
plt.ylim(0, 100)
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 7. RESUMEN PARA LA INTERPRETACIÓN
# ------------------------------------------------------------

print("\n=== 2.2.5 RESUMEN DESCRIPTIVO ===")

modulo_max = tabla_modulos.iloc[0]
modulo_min = tabla_modulos.iloc[-1]

print(
    "Módulo con mayor porcentaje adverso:",
    modulo_max["code_module"],
    f"({modulo_max['% adversos']:.2f}%)"
)

print(
    "Módulo con menor porcentaje adverso:",
    modulo_min["code_module"],
    f"({modulo_min['% adversos']:.2f}%)"
)

print(
    "Rango de tasas adversas entre presentaciones:",
    f"{tabla_presentaciones['% adversos'].min():.2f}% - "
    f"{tabla_presentaciones['% adversos'].max():.2f}%"
)

print(
    "Balance global adverso:",
    f"{100 * datos['target_adverso'].mean():.2f}%"
)

print("\nACTIVIDAD 2.2 FINALIZADA")
print("No se ha entrenado ningún modelo.")
print("No se ha aplicado balanceo ni transformación.")



### 2.2.6. Resultados e interpretación

**Distribución global de clases**

El dataset contiene 32.593 matrículas, con un 52,80 % de resultados académicos adversos (`Fail` o `Withdrawn`) y un 47,20 % de resultados favorables (`Pass` o `Distinction`).

Esta distribución indica que la clasificación binaria propuesta no presenta un desbalance global severo. No obstante, el balance deberá comprobarse nuevamente dentro de cada conjunto de entrenamiento y ventana temporal.

**Heterogeneidad entre módulos académicos**

Se identificaron diferencias importantes en la proporción de resultados adversos entre los siete módulos.

El módulo CCC presentó la mayor proporción de resultados adversos (62,16 %), mientras que AAA presentó la menor (29,01 %), con una diferencia de 33,15 puntos porcentuales.

Entre las 22 presentaciones académicas, las tasas de resultados adversos variaron entre 27,42 % y 65,75 %.

Estos resultados evidencian heterogeneidad académica y sugieren que la identidad del módulo y su presentación pueden contener información predictiva relevante.

**Diferencias entre reprobación y retiro**

Los eventos que conforman la clase adversa presentan distribuciones distintas según el módulo.

En CCC, el retiro representa el 44,54 % de las matrículas, mientras que la reprobación corresponde al 17,61 %.

Por el contrario, en GGG, la reprobación representa el 28,73 % y el retiro únicamente el 11,52 %.

Esta diferencia justifica conservar las etiquetas originales para realizar análisis secundarios del desempeño predictivo sobre ambos tipos de resultados adversos.

**Evolución temporal del balance**

La proporción de resultados adversos entre las matrículas elegibles disminuyó del 45,07 % en la semana 2 al 41,77 % en la semana 8.

Este cambio debe interpretarse considerando que las poblaciones elegibles son diferentes en cada ventana, debido principalmente a los retiros que ocurren durante el período académico.

Por tanto, la evolución de la distribución no debe interpretarse como una mejora del rendimiento estudiantil.

**Clasificador de referencia**

La exactitud teórica de la clase mayoritaria aumentó del 54,93 % en la semana 2 al 58,23 % en la semana 8.

Estos valores representan referencias descriptivas calculadas sobre las poblaciones observadas y no sustituyen una evaluación predictiva independiente.

### 2.2.7. Conclusiones

El análisis permitió identificar una distribución global relativamente equilibrada, acompañada de diferencias considerables entre módulos y presentaciones académicas.

La heterogeneidad observada hace necesario incorporar controles experimentales que permitan separar el aporte predictivo del contexto académico del aporte adicional de las características longitudinales.

Se conservarán las cuatro etiquetas originales para analizar las diferencias entre reprobación y retiro, aunque el experimento principal utilizará inicialmente una clasificación binaria de resultados favorables y adversos.

La comparación entre modelos estáticos y longitudinales se realizará sobre las mismas observaciones elegibles y las mismas particiones dentro de cada ventana temporal.

**Conclusión principal:** el balance global del dataset es favorable para desarrollar modelos de clasificación, pero la heterogeneidad académica y los cambios en la composición temporal de las cohortes deben controlarse para obtener conclusiones válidas sobre el valor predictivo de las trayectorias longitudinales.



### 2.3. Estadística descriptiva, valores atípicos, asimetría y escalamiento

#### Objetivo

Caracterizar las distribuciones de las variables numéricas y categóricas disponibles antes del modelado, identificando diferencias de escala, asimetría y posibles valores atípicos.

#### Justificación metodológica

Los algoritmos de Machine Learning pueden responder de manera diferente a la escala y distribución de las variables.

Los modelos lineales regularizados, como la regresión logística, suelen beneficiarse del escalamiento de variables numéricas, mientras que los modelos basados en árboles no requieren generalmente el mismo tratamiento.

Asimismo, las variables con distribuciones fuertemente asimétricas pueden requerir transformaciones que reduzcan la influencia de valores extremos.

Sin embargo, no todas las variables deben transformarse. Las características categóricas, las fechas relativas y las variables numéricas discretas requieren tratamientos específicos según su significado.

#### Variables numéricas preliminares

Se analizarán:

- `num_of_prev_attempts`: número de intentos anteriores.
- `studied_credits`: créditos académicos registrados.
- `date_registration`: fecha relativa de inscripción.
- `module_presentation_length`: duración del módulo académico.

La fecha de inscripción es una variable temporal relativa al inicio del módulo. Los valores negativos pueden ser legítimos y no deben clasificarse automáticamente como anomalías.

La duración del módulo será incorporada únicamente para su análisis descriptivo, mediante la unión con la tabla de cursos.

#### Variables categóricas

Se examinarán las distribuciones y categorías de variables como género, región, nivel educativo, grupo de edad, banda de privación socioeconómica y discapacidad.

Estas variables no se tratarán como continuas por el hecho de que algunas de sus categorías representen rangos numéricos.

#### Procedimiento

1. Calcular estadísticos descriptivos y percentiles.
2. Identificar valores mínimos, máximos y proporciones de ceros.
3. Medir la asimetría mediante el coeficiente de skewness.
4. Identificar posibles valores atípicos utilizando el rango intercuartílico (IQR).
5. Comparar visualmente las distribuciones de variables numéricas.
6. Examinar las frecuencias de las variables categóricas.

Los valores identificados como atípicos no se eliminarán automáticamente.

#### Consideración metodológica

Esta actividad será exclusivamente exploratoria. Las decisiones sobre estandarización, imputación o transformaciones no lineales se implementarán posteriormente dentro de pipelines de scikit-learn.

Los parámetros de dichas transformaciones deberán aprenderse exclusivamente de los datos de entrenamiento para evitar fuga de información.


In [ ]:

# ============================================================
# PROYECTO FINAL DE MACHINE LEARNING
# ACTIVIDAD 2.3 - DISTRIBUCIONES Y ESTADÍSTICA DESCRIPTIVA
# ============================================================

# Asistencia IA: ChatGPT, octubre de 2026.
# Prompt:
# "Analizar las variables numéricas y categóricas de
# OULAD mediante estadística descriptiva, asimetría,
# valores atípicos IQR, valores cero, histogramas y
# frecuencias, sin modificar los datos originales."

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

# ------------------------------------------------------------
# 1. CONSTRUCCIÓN DE BASE EXPLORATORIA
# ------------------------------------------------------------

info = oulad_tablas["studentInfo.csv"].copy()
registro = oulad_tablas["studentRegistration.csv"].copy()
cursos = oulad_tablas["courses.csv"].copy()

claves = [
    "code_module",
    "code_presentation",
    "id_student"
]

base_eda = info.merge(
    registro,
    on=claves,
    how="left",
    validate="one_to_one"
)

base_eda = base_eda.merge(
    cursos,
    on=["code_module", "code_presentation"],
    how="left",
    validate="many_to_one"
)

print("=== 2.3.1 BASE EXPLORATORIA ===")

print("Registros:", len(base_eda))
print("Variables:", len(base_eda.columns))

assert len(base_eda) == len(info), (
    "La integración cambió el número de matrículas."
)

# ------------------------------------------------------------
# 2. VARIABLES NUMÉRICAS
# ------------------------------------------------------------

numericas = [
    "num_of_prev_attempts",
    "studied_credits",
    "date_registration",
    "module_presentation_length"
]

print("\n=== 2.3.2 ESTADÍSTICA DESCRIPTIVA ===")

estadisticos = base_eda[numericas].describe(
    percentiles=[0.01, 0.05, 0.25, 0.50,
                 0.75, 0.95, 0.99]
).T.round(3)

display(estadisticos)

# ------------------------------------------------------------
# 3. ASIMETRÍA Y ESCALAMIENTO
# ------------------------------------------------------------

print("\n=== 2.3.3 ASIMETRÍA DE VARIABLES ===")

resultados_asimetria = []

for variable in numericas:

    serie = base_eda[variable].dropna()

    skew = serie.skew()

    if pd.isna(skew):
        categoria = "No determinada"
    elif abs(skew) < 0.5:
        categoria = "Baja"
    elif abs(skew) < 1.0:
        categoria = "Moderada"
    else:
        categoria = "Alta"

    resultados_asimetria.append({
        "Variable": variable,
        "Media": round(serie.mean(), 3),
        "Mediana": round(serie.median(), 3),
        "Desv. estándar": round(serie.std(), 3),
        "Skewness": round(skew, 3),
        "Asimetría absoluta": categoria,
        "% ceros": round(
            100 * (serie == 0).mean(), 2
        ),
        "% nulos": round(
            100 * base_eda[variable].isna().mean(),
            2
        )
    })

tabla_asimetria = pd.DataFrame(
    resultados_asimetria
)

display(tabla_asimetria)

# ------------------------------------------------------------
# 4. DETECCIÓN DE VALORES ATÍPICOS CON IQR
# ------------------------------------------------------------

print("\n=== 2.3.4 VALORES ATÍPICOS IQR ===")

resultados_iqr = []

for variable in numericas:

    serie = base_eda[variable].dropna()

    q1 = serie.quantile(0.25)
    q3 = serie.quantile(0.75)
    iqr = q3 - q1

    limite_inferior = q1 - 1.5 * iqr
    limite_superior = q3 + 1.5 * iqr

    # Si IQR = 0, la regla de Tukey pierde
    # utilidad en variables discretas o concentradas.
    if iqr == 0:
        cantidad = np.nan
        porcentaje = np.nan
        criterio = "IQR=0: revisar distribución"
    else:
        atipicos = (
            (serie < limite_inferior)
            | (serie > limite_superior)
        )

        cantidad = int(atipicos.sum())
        porcentaje = round(
            100 * atipicos.mean(), 2
        )
        criterio = "IQR aplicable"

    resultados_iqr.append({
        "Variable": variable,
        "Q1": round(q1, 3),
        "Q3": round(q3, 3),
        "IQR": round(iqr, 3),
        "Límite inferior": round(
            limite_inferior, 3
        ),
        "Límite superior": round(
            limite_superior, 3
        ),
        "Posibles atípicos": cantidad,
        "% posibles atípicos": porcentaje,
        "Observación": criterio
    })

tabla_iqr = pd.DataFrame(resultados_iqr)

display(tabla_iqr)

# ------------------------------------------------------------
# 5. COMPROBACIONES DE DOMINIO
# ------------------------------------------------------------

print("\n=== 2.3.5 VALIDACIÓN DE RANGOS ===")

dominio = pd.DataFrame([
    {
        "Control": "Intentos previos negativos",
        "Registros": int((
            base_eda["num_of_prev_attempts"] < 0
        ).sum())
    },
    {
        "Control": "Créditos negativos",
        "Registros": int((
            base_eda["studied_credits"] < 0
        ).sum())
    },
    {
        "Control": "Créditos iguales a cero",
        "Registros": int((
            base_eda["studied_credits"] == 0
        ).sum())
    },
    {
        "Control": "Inscripciones posteriores al día 0",
        "Registros": int((
            base_eda["date_registration"] > 0
        ).sum())
    },
    {
        "Control": "Inscripciones sin fecha",
        "Registros": int(
            base_eda["date_registration"].isna().sum()
        )
    }
])

display(dominio)

# ------------------------------------------------------------
# 6. HISTOGRAMAS NUMÉRICOS
# ------------------------------------------------------------

print("\n=== 2.3.6 HISTOGRAMAS ===")

for variable in numericas:

    serie = base_eda[variable].dropna()

    plt.figure(figsize=(8, 3.5))

    plt.hist(
        serie,
        bins=30,
        edgecolor="black",
        alpha=0.8
    )

    plt.axvline(
        serie.median(),
        linestyle="--",
        label=f"Mediana = {serie.median():.2f}"
    )

    plt.title(f"Distribución de {variable}")
    plt.xlabel(variable)
    plt.ylabel("Número de matrículas")
    plt.legend()
    plt.grid(axis="y", alpha=0.2)
    plt.tight_layout()
    plt.show()

# ------------------------------------------------------------
# 7. VARIABLES CATEGÓRICAS
# ------------------------------------------------------------

print("\n=== 2.3.7 VARIABLES CATEGÓRICAS ===")

categoricas = [
    "gender",
    "region",
    "highest_education",
    "imd_band",
    "age_band",
    "disability"
]

resumen_categoricas = []

for variable in categoricas:

    serie = base_eda[variable]

    resumen_categoricas.append({
        "Variable": variable,
        "Categorías observadas": serie.nunique(
            dropna=True
        ),
        "Nulos": int(serie.isna().sum()),
        "% nulos": round(
            100 * serie.isna().mean(), 2
        ),
        "Categoría más frecuente": (
            serie.mode(dropna=True).iloc[0]
            if serie.notna().any()
            else "Sin información"
        )
    })

display(pd.DataFrame(resumen_categoricas))

print("\n=== FRECUENCIAS POR CATEGORÍA ===")

for variable in categoricas:

    print(f"\nVariable: {variable}")

    frecuencias = (
        base_eda[variable]
        .fillna("Desconocido")
        .value_counts()
        .rename_axis("Categoría")
        .reset_index(name="Matrículas")
    )

    frecuencias["Porcentaje"] = (
        100
        * frecuencias["Matrículas"]
        / len(base_eda)
    ).round(2)

    display(frecuencias)

# ------------------------------------------------------------
# 8. RESUMEN FINAL
# ------------------------------------------------------------

print("\n=== ACTIVIDAD 2.3 FINALIZADA ===")
print("Se analizaron las variables sin modificar los datos.")
print("No se aplicaron transformaciones.")
print("No se entrenaron modelos.")



### 2.3.8. Resultados e interpretación

**Estadística descriptiva de las variables numéricas**

El análisis identificó diferencias importantes en las distribuciones y escalas de las variables numéricas disponibles.

La variable `num_of_prev_attempts` presentó una media de 0,163, mediana de 0 y asimetría positiva elevada (skewness = 3,808). El 87,20 % de las matrículas registró cero intentos anteriores, evidenciando una distribución fuertemente concentrada en cero.

La variable `studied_credits` presentó una media de 79,759 créditos, mediana de 60 y asimetría positiva de 1,876. Se observaron valores comprendidos entre 30 y 655 créditos.

La variable `date_registration` presentó una media de -69,411 días, mediana de -57 y asimetría negativa de -1,008. Se identificaron 234 inscripciones posteriores al inicio del módulo y 45 registros sin fecha.

La variable `module_presentation_length` presentó una asimetría relativamente baja (-0,404), con duraciones comprendidas entre 234 y 269 días.

**Identificación de posibles valores atípicos**

Mediante el criterio del rango intercuartílico (IQR) se identificaron:

- 350 observaciones potencialmente atípicas en `studied_credits` (1,07 %).
- 339 observaciones potencialmente atípicas en `date_registration` (1,04 %).
- Ninguna observación atípica en `module_presentation_length`.

Para `num_of_prev_attempts`, el IQR fue igual a cero debido a la elevada concentración de observaciones sin intentos anteriores, por lo que este criterio no resulta adecuado para su evaluación.

Los posibles valores atípicos no se eliminarán automáticamente, debido a que podrían representar situaciones académicas legítimas y contener información predictiva.

**Variables categóricas**

Se identificaron seis características categóricas relevantes: género, región, nivel educativo anterior, banda socioeconómica, grupo de edad y discapacidad declarada.

La variable `imd_band` presentó 1.111 valores faltantes (3,41 %). Las demás variables categóricas analizadas no presentaron valores nulos.

Las variables nominales requerirán una codificación apropiada antes del entrenamiento de los modelos.

### 2.3.9. Conclusiones

El análisis confirmó que las variables numéricas no presentan distribuciones homogéneas ni escalas comparables.

En particular, `num_of_prev_attempts` y `studied_credits` presentan asimetrías positivas elevadas, lo que justifica evaluar estrategias de transformación y escalamiento durante el modelado.

Las diferencias entre medias, medianas y valores extremos evidencian que algunas características requieren un tratamiento específico, sin que ello implique necesariamente eliminar observaciones.

La distribución de `date_registration` confirma la existencia de estudiantes matriculados después del inicio del módulo, reforzando la necesidad de controlar la elegibilidad temporal en cada ventana predictiva.

**Conclusión principal:** las características estáticas de OULAD permiten construir una base de referencia para los modelos supervisados, pero requieren un preprocesamiento diferenciado según la naturaleza de cada variable.

Las transformaciones y parámetros de escalamiento se ajustarán exclusivamente con los datos de entrenamiento y se integrarán en pipelines reproducibles para evitar fuga de información.



### 2.4. Análisis longitudinal de la actividad estudiantil

#### Objetivo

Analizar la evolución temporal de la participación estudiantil en el entorno virtual de aprendizaje, con el propósito de identificar patrones de actividad, períodos de inactividad y diferencias de comportamiento entre módulos académicos.

#### Justificación metodológica

La hipótesis principal del proyecto plantea que las características longitudinales del comportamiento estudiantil podrían mejorar la predicción temprana de resultados académicos adversos.

OULAD contiene registros de interacción asociados a estudiantes, módulos, presentaciones académicas, recursos digitales y fechas relativas al inicio del curso.

Estos registros permiten representar la participación académica como una secuencia de observaciones temporales, en lugar de utilizar exclusivamente características estáticas.

#### Variables longitudinales

Se utilizarán las siguientes variables de `studentVle.csv`:

- `code_module`: módulo académico.
- `code_presentation`: presentación del módulo.
- `id_student`: identificador del estudiante.
- `id_site`: recurso del entorno virtual.
- `date`: día relativo al inicio del módulo.
- `sum_click`: número de interacciones registradas.

#### Procedimiento

1. Examinar la cobertura temporal de los registros.
2. Identificar registros anteriores al inicio del módulo.
3. Agrupar la actividad por estudiante, módulo, presentación y día.
4. Construir agregaciones semanales para las primeras ocho semanas.
5. Calcular frecuencia de participación, volumen de interacciones y días activos.
6. Estimar la proporción de matrículas elegibles con y sin actividad registrada.
7. Comparar descriptivamente la evolución entre módulos académicos.

Para esta exploración, las semanas se definirán como intervalos completos de siete días: semana 1 (días 0–6), semana 2 (7–13), y así sucesivamente hasta la semana 8 (49–55).

Estas ventanas descriptivas son distintas de los instantes de predicción definidos previamente en los días 14, 28, 42 y 56. En la fase de ingeniería de características se utilizarán cortes temporales exactos para garantizar que ninguna variable incorpore información posterior al momento de predicción.

#### Consideraciones metodológicas

La ausencia de actividad en `studentVle.csv` representa ausencia de interacciones registradas en la plataforma, no necesariamente inactividad académica general.

La participación será medida tanto por el volumen de clics como por la frecuencia de días con actividad.

Se analizarán las primeras ocho semanas del módulo y se conservarán las claves compuestas para evitar mezclar registros de estudiantes con múltiples matrículas.

El archivo será procesado por bloques para reducir los requerimientos de memoria de Google Colab.

No se entrenarán modelos ni se realizarán transformaciones predictivas definitivas durante esta actividad.


In [ ]:

# ============================================================
# PROYECTO FINAL DE MACHINE LEARNING
# ACTIVIDAD 2.4 - ANÁLISIS LONGITUDINAL DE STUDENTVLE
# ============================================================

# Asistencia IA: ChatGPT, octubre de 2026.
# Prompt:
# "Analizar los registros longitudinales de studentVle.csv
# mediante lectura por bloques, agregación diaria y semanal,
# cálculo de clics, días activos, participación por matrícula
# elegible y comparación entre módulos, sin entrenar modelos."

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

# ------------------------------------------------------------
# 1. CONFIGURACIÓN
# ------------------------------------------------------------

ruta = oulad_rutas["studentVle.csv"]

claves = [
    "code_module",
    "code_presentation",
    "id_student"
]

columnas = claves + [
    "id_site", "date", "sum_click"
]

tamano_bloque = 400_000

agregados_diarios = []

total_registros = 0
registros_negativos = 0
registros_dia_cero = 0
registros_primeras_8 = 0
clics_no_positivos = 0
filas_incompletas = 0

fecha_min = np.inf
fecha_max = -np.inf

print("=== 2.4.1 LECTURA LONGITUDINAL ===")

# ------------------------------------------------------------
# 2. LECTURA POR BLOQUES
# ------------------------------------------------------------

lector = pd.read_csv(
    ruta,
    usecols=columnas,
    chunksize=tamano_bloque
)

for numero_bloque, bloque in enumerate(lector, start=1):

    total_registros += len(bloque)

    filas_incompletas += int(
        bloque[columnas].isna().any(axis=1).sum()
    )

    registros_negativos += int(
        (bloque["date"] < 0).sum()
    )

    registros_dia_cero += int(
        (bloque["date"] == 0).sum()
    )

    clics_no_positivos += int(
        (bloque["sum_click"] <= 0).sum()
    )

    fecha_min = min(
        fecha_min, bloque["date"].min()
    )

    fecha_max = max(
        fecha_max, bloque["date"].max()
    )

    # Primera etapa:
    # ocho semanas completas: días 0 a 55.
    seleccion = bloque.loc[
        bloque["date"].between(0, 55)
        & bloque[claves].notna().all(axis=1)
    ].copy()

    registros_primeras_8 += len(seleccion)

    if len(seleccion) > 0:

        # Consolidación parcial estudiante-día.
        diario = (
            seleccion.groupby(
                claves + ["date"],
                as_index=False,
                sort=False
            )
            .agg(
                clics=("sum_click", "sum"),
                registros=("sum_click", "size")
            )
        )

        agregados_diarios.append(diario)

    if numero_bloque % 5 == 0:
        print(
            f"Bloques procesados: {numero_bloque} | "
            f"Registros: {total_registros:,}"
        )

# ------------------------------------------------------------
# 3. RESUMEN DE COBERTURA TEMPORAL
# ------------------------------------------------------------

print("\n=== 2.4.2 COBERTURA TEMPORAL ===")

resumen_temporal = pd.DataFrame([
    {
        "Indicador": "Total de registros VLE",
        "Valor": total_registros
    },
    {
        "Indicador": "Fecha mínima (día)",
        "Valor": fecha_min
    },
    {
        "Indicador": "Fecha máxima (día)",
        "Valor": fecha_max
    },
    {
        "Indicador": "Registros anteriores al día 0",
        "Valor": registros_negativos
    },
    {
        "Indicador": "Registros del día 0",
        "Valor": registros_dia_cero
    },
    {
        "Indicador": "Registros en días 0–55",
        "Valor": registros_primeras_8
    },
    {
        "Indicador": "Registros con sum_click <= 0",
        "Valor": clics_no_positivos
    },
    {
        "Indicador": "Filas con algún valor faltante",
        "Valor": filas_incompletas
    }
])

display(resumen_temporal)

# ------------------------------------------------------------
# 4. CONSOLIDACIÓN DIARIA COMPLETA
# ------------------------------------------------------------

print("\n=== 2.4.3 CONSOLIDACIÓN DIARIA ===")

if not agregados_diarios:
    raise ValueError(
        "No se encontraron registros para los días 0–55."
    )

actividad_diaria = pd.concat(
    agregados_diarios,
    ignore_index=True
)

# Consolidar registros del mismo estudiante y día
# que pudieron quedar en bloques diferentes.
actividad_diaria = (
    actividad_diaria.groupby(
        claves + ["date"],
        as_index=False,
        sort=False
    )
    .agg(
        clics=("clics", "sum"),
        registros=("registros", "sum")
    )
)

del agregados_diarios

print(
    "Observaciones únicas estudiante-módulo-día:",
    len(actividad_diaria)
)

# ------------------------------------------------------------
# 5. AGREGACIÓN SEMANAL
# ------------------------------------------------------------

print("\n=== 2.4.4 ACTIVIDAD SEMANAL ===")

actividad_diaria["Semana"] = (
    actividad_diaria["date"] // 7 + 1
).astype(int)

oulad_vle_semanal = (
    actividad_diaria.groupby(
        claves + ["Semana"],
        as_index=False
    )
    .agg(
        clics=("clics", "sum"),
        dias_activos=("date", "nunique"),
        registros=("registros", "sum")
    )
)

print(
    "Observaciones estudiante-módulo-semana:",
    len(oulad_vle_semanal)
)

display(oulad_vle_semanal.head(10))

# ------------------------------------------------------------
# 6. PREPARAR POBLACIÓN ELEGIBLE
# ------------------------------------------------------------

info = oulad_tablas["studentInfo.csv"]
registro = oulad_tablas["studentRegistration.csv"]

base = info.merge(
    registro,
    on=claves,
    how="left",
    validate="one_to_one"
)

base["retiro_sin_fecha"] = (
    base["final_result"].eq("Withdrawn")
    & base["date_unregistration"].isna()
)

resumen_semanal = []
resumen_modulos = []

# ------------------------------------------------------------
# 7. PARTICIPACIÓN ENTRE MATRÍCULAS ELEGIBLES
# ------------------------------------------------------------

for semana in range(1, 9):

    inicio_semana = (semana - 1) * 7

    # Elegibilidad al inicio de la semana.
    inscrito = (
        base["date_registration"].notna()
        & (
            base["date_registration"]
            <= inicio_semana
        )
    )

    no_retirado = (
        base["date_unregistration"].isna()
        | (
            base["date_unregistration"]
            > inicio_semana
        )
    )

    elegibles = base.loc[
        inscrito
        & no_retirado
        & (~base["retiro_sin_fecha"]),
        claves
    ].copy()

    actividad_semana = oulad_vle_semanal.loc[
        oulad_vle_semanal["Semana"] == semana,
        claves + [
            "clics", "dias_activos", "registros"
        ]
    ]

    # LEFT JOIN:
    # conservar matrículas sin actividad VLE.
    cohorte = elegibles.merge(
        actividad_semana,
        on=claves,
        how="left",
        validate="one_to_one"
    )

    for variable in [
        "clics", "dias_activos", "registros"
    ]:
        cohorte[variable] = (
            cohorte[variable].fillna(0)
        )

    cohorte["con_actividad"] = (
        cohorte["registros"] > 0
    )

    n = len(cohorte)
    activos = int(cohorte["con_actividad"].sum())

    resumen_semanal.append({
        "Semana": semana,
        "Matrículas elegibles": n,
        "Con actividad VLE": activos,
        "Sin actividad VLE": n - activos,
        "% con actividad": round(
            100 * activos / n, 2
        ) if n else np.nan,
        "Clics por matrícula": round(
            cohorte["clics"].mean(), 2
        ),
        "Mediana clics": round(
            cohorte["clics"].median(), 2
        ),
        "Días activos promedio": round(
            cohorte["dias_activos"].mean(), 2
        )
    })

    por_modulo = (
        cohorte.groupby("code_module")
        .agg(
            matriculas=("id_student", "size"),
            actividad=("con_actividad", "sum"),
            promedio_clics=("clics", "mean"),
            promedio_dias=("dias_activos", "mean")
        )
        .reset_index()
    )

    por_modulo["Semana"] = semana

    por_modulo["% actividad"] = (
        100 * por_modulo["actividad"]
        / por_modulo["matriculas"]
    ).round(2)

    resumen_modulos.append(por_modulo)

tabla_actividad_semanal = pd.DataFrame(
    resumen_semanal
)

tabla_actividad_modulos = pd.concat(
    resumen_modulos,
    ignore_index=True
)

print("\n=== 2.4.5 PARTICIPACIÓN SEMANAL ===")

display(tabla_actividad_semanal)

print("\n=== 2.4.6 ACTIVIDAD POR MÓDULO ===")

display(tabla_actividad_modulos)

# ------------------------------------------------------------
# 8. GRÁFICO DE PARTICIPACIÓN
# ------------------------------------------------------------

plt.figure(figsize=(9, 4))

plt.plot(
    tabla_actividad_semanal["Semana"],
    tabla_actividad_semanal["% con actividad"],
    marker="o"
)

plt.title(
    "Matrículas elegibles con actividad VLE por semana"
)

plt.xlabel("Semana")
plt.ylabel("Matrículas con actividad (%)")
plt.xticks(range(1, 9))
plt.ylim(0, 100)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 9. GRÁFICO DE INTENSIDAD DE PARTICIPACIÓN
# ------------------------------------------------------------

plt.figure(figsize=(9, 4))

plt.plot(
    tabla_actividad_semanal["Semana"],
    tabla_actividad_semanal["Clics por matrícula"],
    marker="s"
)

plt.title(
    "Promedio semanal de interacciones VLE "
    "por matrícula elegible"
)

plt.xlabel("Semana")
plt.ylabel("Número promedio de clics")
plt.xticks(range(1, 9))
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 10. COMPARACIÓN POR MÓDULO
# ------------------------------------------------------------

tabla_pivote = tabla_actividad_modulos.pivot(
    index="code_module",
    columns="Semana",
    values="% actividad"
)

print("\n=== 2.4.7 MATRIZ DE PARTICIPACIÓN POR MÓDULO ===")
display(tabla_pivote)

plt.figure(figsize=(10, 5))

plt.imshow(
    tabla_pivote.values,
    aspect="auto",
    cmap="viridis",
    vmin=0,
    vmax=100
)

plt.colorbar(label="Matrículas con actividad (%)")

plt.xticks(
    range(len(tabla_pivote.columns)),
    tabla_pivote.columns
)

plt.yticks(
    range(len(tabla_pivote.index)),
    tabla_pivote.index
)

plt.xlabel("Semana")
plt.ylabel("Módulo")
plt.title("Participación semanal VLE por módulo")

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 11. RESUMEN FINAL
# ------------------------------------------------------------

print("\n=== ACTIVIDAD 2.4 FINALIZADA ===")

print(
    "Semanas examinadas:",
    tabla_actividad_semanal["Semana"].nunique()
)

print(
    "Participación mínima:",
    f"{tabla_actividad_semanal['% con actividad'].min():.2f}%"
)

print(
    "Participación máxima:",
    f"{tabla_actividad_semanal['% con actividad'].max():.2f}%"
)

print(
    "Promedio de clics semanal mínimo:",
    f"{tabla_actividad_semanal['Clics por matrícula'].min():.2f}"
)

print(
    "Promedio de clics semanal máximo:",
    f"{tabla_actividad_semanal['Clics por matrícula'].max():.2f}"
)

print("\nNo se han entrenado modelos.")
print("No se han modificado los archivos originales.")



### 2.4.8. Resultados e interpretación

**Cobertura de la información longitudinal**

El procesamiento de `studentVle.csv` permitió analizar 10.655.280 registros de actividad virtual, con fechas relativas comprendidas entre los días -25 y 269.

Se identificaron 688.988 registros anteriores al inicio del módulo y 3.613.663 registros correspondientes a los primeros 56 días académicos (días 0 a 55).

La consolidación de los registros de las primeras ocho semanas produjo 562.124 observaciones únicas por estudiante-módulo-día y 174.930 observaciones estudiante-módulo-semana con actividad registrada.

No se encontraron valores faltantes ni cantidades de clics no positivas.

**Evolución semanal de la participación**

El porcentaje de matrículas elegibles con actividad registrada pasó del 80,10 % en la semana 1 al 74,09 % en la semana 8, lo que representa una reducción de 6,01 puntos porcentuales.

La mayor participación se observó en la semana 3, con un 83,37 %.

El promedio semanal de clics por matrícula elegible disminuyó de 68,58 en la semana 1 a 48,08 en la semana 8.

Asimismo, el promedio de días activos disminuyó de 2,56 a 2,33 días por semana.

Estas diferencias corresponden a estadísticas agregadas de poblaciones elegibles cambiantes y no deben interpretarse directamente como disminuciones individuales de participación.

**Heterogeneidad entre módulos**

Se identificaron diferencias importantes en los niveles y patrones de participación virtual.

El módulo FFF presentó una participación relativamente alta durante las ocho semanas, mientras que GGG mostró niveles de actividad considerablemente menores.

En CCC, la proporción de matrículas con actividad pasó del 91,35 % en la semana 3 al 66,63 % en la semana 8.

Al contrastar estos resultados con las distribuciones académicas de la Actividad 2.2, se observa que los módulos con menor actividad virtual no son necesariamente los que presentan mayor proporción de resultados adversos.

Esta observación sugiere la necesidad de considerar el contexto académico y de evaluar tanto indicadores absolutos como relativos de participación.

### 2.4.9. Conclusiones

El dataset OULAD dispone de registros longitudinales suficientes para construir trayectorias semanales del comportamiento estudiantil.

Las variaciones observadas en frecuencia, intensidad y regularidad de participación justifican investigar su posible aporte como características predictivas.

No obstante, los resultados descriptivos todavía no permiten afirmar que una reducción de actividad anticipa la reprobación o el retiro de un estudiante.

La evaluación de dicha relación requerirá comparar modelos estáticos y longitudinales bajo las mismas condiciones experimentales.

**Conclusión principal:** la actividad virtual presenta variabilidad temporal y heterogeneidad entre módulos, por lo que resulta metodológicamente pertinente evaluar si su evolución aporta información predictiva adicional respecto de las características estáticas.

La superioridad del modelo longitudinal permanece como una hipótesis pendiente de contrastación.



### 2.5. Análisis longitudinal de evaluaciones y entregas académicas

#### Objetivo

Analizar la distribución temporal de las evaluaciones, las calificaciones registradas y los patrones de entrega de actividades académicas, con el propósito de determinar qué información podría utilizarse en modelos de predicción temprana del riesgo estudiantil.

#### Justificación metodológica

Las evaluaciones académicas constituyen una fuente relevante para estudiar la trayectoria del estudiante durante el período académico.

Además del resultado final de una asignatura, el historial de entregas puede proporcionar indicadores relacionados con la regularidad de participación, puntualidad y cumplimiento de actividades.

OULAD contiene información sobre las evaluaciones programadas y los registros individuales de entrega de los estudiantes.

Sin embargo, debe distinguirse entre la fecha de entrega de una evaluación y la fecha en que su calificación estuvo disponible. El dataset no proporciona directamente una fecha de publicación de notas para cada evaluación.

En consecuencia, el uso de calificaciones como variables predictivas requerirá una justificación adicional de disponibilidad temporal.

#### Aspectos que se investigarán

1. Distribución de evaluaciones por tipo.
2. Disponibilidad de fechas límite y ponderaciones.
3. Distribución y calidad de las calificaciones.
4. Evaluaciones reconocidas mediante `is_banked`.
5. Entregas realizadas antes y después de las fechas límite.
6. Disponibilidad de entregas registradas hasta las semanas 2, 4, 6 y 8.
7. Diferencias descriptivas de participación en evaluaciones según el resultado académico final.

#### Consideraciones metodológicas

Una entrega realizada antes de una fecha de predicción constituye un evento registrado que podría utilizarse como característica longitudinal.

En cambio, una calificación registrada en el dataset no debe considerarse automáticamente disponible desde la fecha de entrega.

Tampoco puede asumirse que la ausencia de una entrega implica incumplimiento, sin verificar previamente que la actividad estaba programada y era aplicable al estudiante.

Los análisis realizados en esta actividad serán descriptivos y no implicarán selección definitiva de características ni entrenamiento de modelos.


In [ ]:

# ============================================================
# PROYECTO FINAL DE MACHINE LEARNING
# ACTIVIDAD 2.5 - ANÁLISIS DE EVALUACIONES Y ENTREGAS
# ============================================================

# Asistencia IA: ChatGPT, octubre de 2026.
# Prompt:
# "Analizar las evaluaciones de OULAD, sus tipos,
# fechas, ponderaciones, calificaciones y entregas.
# Examinar disponibilidad temporal en semanas 2, 4,
# 6 y 8, distinguiendo la fecha de entrega de la
# disponibilidad desconocida de las calificaciones.
# No entrenar modelos ni introducir fuga temporal."


# ------------------------------------------------------------
# 1. PREPARACIÓN E INTEGRACIÓN
# ------------------------------------------------------------

assessments = oulad_tablas["assessments.csv"].copy()
student_assessment = (
    oulad_tablas["studentAssessment.csv"].copy()
)
info = oulad_tablas["studentInfo.csv"].copy()
registro = (
    oulad_tablas["studentRegistration.csv"].copy()
)

claves = [
    "code_module",
    "code_presentation",
    "id_student"
]

claves_curso = [
    "code_module",
    "code_presentation"
]

evaluaciones = student_assessment.merge(
    assessments,
    on="id_assessment",
    how="left",
    validate="many_to_one",
    indicator=True
)

sin_evaluacion = int(
    evaluaciones["_merge"].eq("left_only").sum()
)

if sin_evaluacion:
    raise ValueError(
        f"Hay {sin_evaluacion} entregas sin evaluación."
    )

evaluaciones = evaluaciones.drop(columns="_merge")

evaluaciones = evaluaciones.merge(
    info[claves + ["final_result"]],
    on=claves,
    how="left",
    validate="many_to_one"
)

print("=== 2.5.1 ESTRUCTURA GENERAL ===")
print("Evaluaciones definidas:", len(assessments))
print("Registros individuales:", len(evaluaciones))
print(
    "Registros sin resultado final asociado:",
    evaluaciones["final_result"].isna().sum()
)

# ------------------------------------------------------------
# 2. TIPOS DE EVALUACIONES
# ------------------------------------------------------------

print("\n=== 2.5.2 TIPOS DE EVALUACIONES ===")

tipos = (
    assessments.groupby(
        "assessment_type",
        dropna=False
    )
    .agg(
        Evaluaciones=("id_assessment", "size"),
        Sin_fecha=("date", lambda x: x.isna().sum()),
        Peso_promedio=("weight", "mean"),
        Fecha_minima=("date", "min"),
        Fecha_maxima=("date", "max")
    )
    .reset_index()
)

display(tipos)

plt.figure(figsize=(8, 4))

plt.bar(
    tipos["assessment_type"].astype(str),
    tipos["Evaluaciones"]
)

plt.title("Número de evaluaciones por tipo")
plt.xlabel("Tipo de evaluación")
plt.ylabel("Número de evaluaciones")
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 3. DISTRIBUCIÓN DE CALIFICACIONES
# ------------------------------------------------------------

print("\n=== 2.5.3 CALIFICACIONES ===")

display(
    evaluaciones["score"].describe(
        percentiles=[0.05, 0.25, 0.50, 0.75, 0.95]
    ).to_frame("Calificación")
)

auditoria_notas = pd.DataFrame([
    {
        "Control": "Calificaciones faltantes",
        "Registros": int(
            evaluaciones["score"].isna().sum()
        )
    },
    {
        "Control": "Calificaciones menores que 0",
        "Registros": int(
            (evaluaciones["score"] < 0).sum()
        )
    },
    {
        "Control": "Calificaciones mayores que 100",
        "Registros": int(
            (evaluaciones["score"] > 100).sum()
        )
    },
    {
        "Control": "Evaluaciones reconocidas (is_banked=1)",
        "Registros": int(
            evaluaciones["is_banked"].eq(1).sum()
        )
    },
    {
        "Control": "Entregas anteriores al día 0",
        "Registros": int(
            (evaluaciones["date_submitted"] < 0).sum()
        )
    }
])

display(auditoria_notas)

print("\nCalificaciones por tipo de evaluación:")

notas_tipo = (
    evaluaciones.groupby("assessment_type")
    .agg(
        Registros=("id_assessment", "size"),
        Nota_promedio=("score", "mean"),
        Nota_mediana=("score", "median"),
        Notas_faltantes=(
            "score", lambda x: x.isna().sum()
        ),
        Reconocidas=(
            "is_banked", lambda x: x.eq(1).sum()
        )
    )
    .reset_index()
)

display(notas_tipo.round(2))

plt.figure(figsize=(8, 4))

plt.hist(
    evaluaciones["score"].dropna(),
    bins=30,
    edgecolor="black"
)

plt.title("Distribución de calificaciones registradas")
plt.xlabel("Calificación")
plt.ylabel("Número de registros")
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 4. FECHAS LÍMITE Y PUNTUALIDAD
# ------------------------------------------------------------

print("\n=== 2.5.4 PUNTUALIDAD EN LAS ENTREGAS ===")

# Las evaluaciones reconocidas no se consideran
# nuevas entregas ordinarias del módulo.

entregas_ordinarias = evaluaciones.loc[
    evaluaciones["is_banked"].eq(0)
].copy()

entregas_con_plazo = entregas_ordinarias.loc[
    entregas_ordinarias["date"].notna()
].copy()

entregas_con_plazo["diferencia_dias"] = (
    entregas_con_plazo["date_submitted"]
    - entregas_con_plazo["date"]
)

entregas_con_plazo["tardia"] = (
    entregas_con_plazo["diferencia_dias"] > 0
)

puntualidad = pd.DataFrame([
    {
        "Indicador": "Entregas ordinarias",
        "Cantidad": len(entregas_ordinarias)
    },
    {
        "Indicador": "Entregas con plazo conocido",
        "Cantidad": len(entregas_con_plazo)
    },
    {
        "Indicador": "Entregas puntuales",
        "Cantidad": int(
            (~entregas_con_plazo["tardia"]).sum()
        )
    },
    {
        "Indicador": "Entregas posteriores al plazo",
        "Cantidad": int(
            entregas_con_plazo["tardia"].sum()
        )
    }
])

display(puntualidad)

if len(entregas_con_plazo):
    porcentaje_tardias = (
        100 * entregas_con_plazo["tardia"].mean()
    )
    print(
        "Porcentaje de entregas tardías "
        "entre las que tienen plazo conocido:",
        f"{porcentaje_tardias:.2f}%"
    )

# ------------------------------------------------------------
# 5. DISTRIBUCIÓN TEMPORAL DE ENTREGAS
# ------------------------------------------------------------

print("\n=== 2.5.5 DISTRIBUCIÓN TEMPORAL ===")

print("Fecha de entrega:")
display(
    evaluaciones["date_submitted"].describe()
)

print("\nFecha límite de evaluaciones:")
display(
    assessments["date"].describe()
)

# ------------------------------------------------------------
# 6. DISPONIBILIDAD EN LAS VENTANAS PREDICTIVAS
# ------------------------------------------------------------

print("\n=== 2.5.6 DISPONIBILIDAD POR SEMANA ===")

base = info.merge(
    registro,
    on=claves,
    how="left",
    validate="one_to_one"
)

base["retiro_sin_fecha"] = (
    base["final_result"].eq("Withdrawn")
    & base["date_unregistration"].isna()
)

resultados_semanales = []
resultados_por_tipo = []
resultados_por_resultado = []

for semana in [2, 4, 6, 8]:

    dia = semana * 7

    # A) Matrículas elegibles al día de corte
    inscrito = (
        base["date_registration"].notna()
        & (base["date_registration"] <= dia)
    )

    no_retirado = (
        base["date_unregistration"].isna()
        | (base["date_unregistration"] > dia)
    )

    elegibles = base.loc[
        inscrito
        & no_retirado
        & (~base["retiro_sin_fecha"]),
        claves + ["final_result"]
    ].copy()

    # B) Entregas ordinarias realizadas
    # hasta el día de predicción.
    entregas_corte = entregas_ordinarias.loc[
        entregas_ordinarias["date_submitted"] <= dia
    ].copy()

    entregas_corte["tardia_conocida"] = (
        entregas_corte["date"].notna()
        & (
            entregas_corte["date_submitted"]
            > entregas_corte["date"]
        )
    )

    entregas_agg = (
        entregas_corte.groupby(
            claves, as_index=False
        )
        .agg(
            entregas=("id_assessment", "size"),
            tardias=("tardia_conocida", "sum")
        )
    )

    # Conservar estudiantes sin entregas.
    cohorte = elegibles.merge(
        entregas_agg,
        on=claves,
        how="left",
        validate="one_to_one"
    )

    cohorte[["entregas", "tardias"]] = (
        cohorte[["entregas", "tardias"]]
        .fillna(0)
        .astype(int)
    )

    cohorte["con_entrega"] = (
        cohorte["entregas"] > 0
    )

    # C) Evaluaciones con fecha límite conocida
    # que ya llegó al día de corte.
    vencidas = assessments.loc[
        assessments["date"].notna()
        & (assessments["date"] <= dia)
    ]

    plazos_curso = (
        vencidas.groupby(claves_curso)
        .size()
        .rename("evaluaciones_vencidas")
        .reset_index()
    )

    cohorte = cohorte.merge(
        plazos_curso,
        on=claves_curso,
        how="left",
        validate="many_to_one"
    )

    cohorte["evaluaciones_vencidas"] = (
        cohorte["evaluaciones_vencidas"]
        .fillna(0)
        .astype(int)
    )

    n = len(cohorte)

    resultados_semanales.append({
        "Semana": semana,
        "Matrículas elegibles": n,
        "Con entregas": int(
            cohorte["con_entrega"].sum()
        ),
        "Sin entregas": int(
            (~cohorte["con_entrega"]).sum()
        ),
        "% con entregas": round(
            100 * cohorte["con_entrega"].mean(), 2
        ) if n else np.nan,
        "Entregas promedio": round(
            cohorte["entregas"].mean(), 3
        ),
        "Mediana entregas": round(
            cohorte["entregas"].median(), 2
        ),
        "Sin evaluaciones vencidas": int(
            cohorte["evaluaciones_vencidas"].eq(0).sum()
        )
    })

    # D) Participación en entregas según resultado
    # final. SOLO DESCRIPTIVO, no predictivo.
    por_resultado = (
        cohorte.groupby("final_result")
        .agg(
            Matriculas=("id_student", "size"),
            Con_entregas=("con_entrega", "sum"),
            Entregas_promedio=("entregas", "mean")
        )
        .reset_index()
    )

    por_resultado["Semana"] = semana

    por_resultado["% con entregas"] = (
        100
        * por_resultado["Con_entregas"]
        / por_resultado["Matriculas"]
    ).round(2)

    resultados_por_resultado.append(
        por_resultado
    )

    # E) Cantidad de entregas realizadas
    # hasta el corte, por tipo.
    entregas_elegibles = entregas_corte.merge(
        elegibles[claves],
        on=claves,
        how="inner",
        validate="many_to_one"
    )

    por_tipo = (
        entregas_elegibles.groupby(
            "assessment_type"
        )
        .size()
        .rename("Entregas")
        .reset_index()
    )

    por_tipo["Semana"] = semana

    resultados_por_tipo.append(por_tipo)

tabla_entregas_semanal = pd.DataFrame(
    resultados_semanales
)

tabla_entregas_resultado = pd.concat(
    resultados_por_resultado,
    ignore_index=True
)

tabla_entregas_tipo = pd.concat(
    resultados_por_tipo,
    ignore_index=True
)

display(tabla_entregas_semanal)

print("\n=== 2.5.7 ENTREGAS SEGÚN RESULTADO FINAL ===")

display(tabla_entregas_resultado)

print("\n=== 2.5.8 ENTREGAS POR TIPO Y SEMANA ===")

display(tabla_entregas_tipo)

# ------------------------------------------------------------
# 7. VISUALIZACIÓN DE DISPONIBILIDAD TEMPORAL
# ------------------------------------------------------------

plt.figure(figsize=(9, 4))

plt.plot(
    tabla_entregas_semanal["Semana"],
    tabla_entregas_semanal["% con entregas"],
    marker="o"
)

plt.title(
    "Matrículas con al menos una entrega registrada"
)

plt.xlabel("Semana de predicción")
plt.ylabel("Matrículas con entregas (%)")
plt.xticks([2, 4, 6, 8])
plt.ylim(0, 100)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 8. RESUMEN
# ------------------------------------------------------------

print("\n=== ACTIVIDAD 2.5 FINALIZADA ===")

print(
    "Evaluaciones definidas:",
    len(assessments)
)

print(
    "Registros de evaluaciones individuales:",
    len(evaluaciones)
)

print(
    "Semanas examinadas:",
    len(tabla_entregas_semanal)
)

print("\nNo se han entrenado modelos.")
print("No se han modificado los datos originales.")
print(
    "No se ha supuesto que las calificaciones "
    "estuvieran disponibles en la fecha de entrega."
)



### 2.5.9. Resultados e interpretación

**Estructura de las evaluaciones**

El análisis identificó 206 evaluaciones académicas y 173.912 registros individuales asociados a estudiantes.

Se identificaron tres tipos de evaluaciones: CMA (76), TMA (106) y Exam (24).

Las calificaciones registradas presentaron una media de 75,80, una mediana de 80 y una desviación estándar de 18,80. No se identificaron valores inferiores a 0 ni superiores a 100.

Se encontraron 173 calificaciones faltantes y 1.909 evaluaciones reconocidas mediante `is_banked=1`, que deben diferenciarse de las entregas ordinarias del módulo.

**Puntualidad en las entregas**

De 172.003 entregas ordinarias, 169.138 disponían de una fecha límite conocida.

Entre estas, 49.318 fueron registradas después de la fecha límite, equivalentes al 29,16 %.

Este resultado evidencia variabilidad en la puntualidad, aunque requiere considerar la naturaleza de cada evaluación y las posibles particularidades de los plazos registrados.

**Disponibilidad longitudinal de entregas**

La proporción de matrículas elegibles con al menos una entrega registrada evolucionó de la siguiente manera:

| Semana | Matrículas elegibles | Con entregas |
|---|---:|---:|
| 2 | 27.962 | 11,89 % |
| 4 | 27.422 | 72,55 % |
| 6 | 26.912 | 82,86 % |
| 8 | 26.413 | 86,30 % |

En la semana 2, 26.674 matrículas todavía no tenían evaluaciones con fecha límite vencida.

Esto demuestra que la ausencia de entregas debe interpretarse respecto del calendario de evaluaciones y no como un indicador automático de incumplimiento.

**Relación descriptiva con los resultados académicos**

En la semana 8, el 93,21 % de las matrículas que finalizaron con `Pass` tenía al menos una entrega registrada, frente al 73,13 % de aquellas que finalizaron con `Fail`.

La diferencia fue de 20,08 puntos porcentuales.

También se observaron diferencias entre las categorías `Distinction`, `Pass`, `Fail` y `Withdrawn` en otras ventanas temporales.

Estos resultados describen asociaciones observadas, pero no demuestran causalidad ni superioridad predictiva de un modelo longitudinal.

**Limitaciones temporales**

El dataset proporciona fechas de entrega, pero no fechas verificables de publicación de las calificaciones.

Por tanto, las notas no se considerarán automáticamente disponibles desde su fecha de entrega.

Se identificaron también 11 evaluaciones sin fecha límite y entregas registradas hasta el día 608, por lo que se requiere una auditoría adicional antes de construir características predictivas temporales.

### 2.5.10. Conclusiones

La información de evaluaciones y entregas constituye una fuente potencial de características longitudinales complementaria a los registros de interacción virtual.

La disponibilidad de entregas aumenta considerablemente entre las semanas 2 y 8, lo que sugiere que el aporte de esta información podría depender del momento de predicción.

Las diferencias descriptivas observadas entre estudiantes con resultados favorables y adversos justifican evaluar indicadores de cumplimiento y puntualidad.

Sin embargo, la utilidad predictiva de estas características deberá demostrarse mediante modelos evaluados sobre datos independientes, controlando el calendario académico y la disponibilidad temporal.

**Conclusión principal:** OULAD permite estudiar patrones longitudinales de entregas y participación académica, pero las características predictivas deben construirse exclusivamente a partir de información observable antes de cada fecha de predicción, evitando incorporar resultados futuros.



### 2.6. Auditoría de disponibilidad temporal y prevención de fuga de información

#### Objetivo

Evaluar la integridad de los registros longitudinales y determinar qué información puede utilizarse legítimamente en cada ventana de predicción, evitando incorporar datos posteriores al momento en que se genera la alerta académica.

#### Justificación metodológica

En un sistema de predicción temprana, cada observación debe representar la información disponible en el momento en que se efectúa la predicción.

Una característica calculada utilizando información futura puede producir una estimación artificialmente elevada del rendimiento predictivo. Este problema se conoce como fuga de información o *data leakage*.

Se utilizarán cuatro momentos de predicción:

| Semana | Día de predicción | Actividad considerada |
|---|---:|---|
| 2 | 14 | Días 0–13 |
| 4 | 28 | Días 0–27 |
| 6 | 42 | Días 0–41 |
| 8 | 56 | Días 0–55 |

#### Reglas temporales preliminares

1. Se considerarán matrículas cuya fecha de inscripción sea anterior o igual al día de predicción.
2. Se excluirán las matrículas con desregistro anterior o igual al día de predicción.
3. Los registros VLE deberán tener una fecha anterior al corte.
4. Las entregas académicas deberán haberse realizado antes del corte.
5. Los resultados finales (`final_result`) se utilizarán únicamente como etiquetas, nunca como características predictivas.
6. Las fechas futuras de desregistro se utilizarán para construir etiquetas y evaluar la anticipación de eventos, no como entradas del modelo.
7. Las calificaciones no se utilizarán como características tempranas mientras no exista una regla justificada sobre su disponibilidad.

#### Integridad longitudinal

Se verificará que los registros de actividad virtual correspondan a matrículas y recursos existentes en OULAD.

También se identificarán registros anteriores a la fecha oficial de inscripción y posteriores al desregistro, sin eliminarlos automáticamente durante esta auditoría.

#### Resultado esperado

Obtener una evaluación reproducible de la integridad de los eventos VLE y una tabla que muestre la cobertura acumulada de actividad virtual y entregas académicas para las cuatro ventanas predictivas.

Los resultados permitirán establecer reglas de ingeniería de características para los modelos estáticos, longitudinales y, si corresponde, un Transformer temporal.


In [ ]:

# ============================================================
# ACTIVIDAD 2.6 - DISPONIBILIDAD TEMPORAL Y DATA LEAKAGE
# ============================================================

# Asistencia IA: ChatGPT, octubre de 2026.
# Prompt:
# "Auditar la integridad referencial de studentVle.csv
# y verificar la disponibilidad de actividad y entregas
# antes de los días 14, 28, 42 y 56.
# Identificar eventos fuera de matrícula y prevenir
# fuga temporal sin entrenar modelos."

claves = [
    "code_module", "code_presentation", "id_student"
]
claves_curso = ["code_module", "code_presentation"]
claves_recurso = claves_curso + ["id_site"]

info = oulad_tablas["studentInfo.csv"]
registro = oulad_tablas["studentRegistration.csv"]
assessments = oulad_tablas["assessments.csv"]
student_assessment = oulad_tablas["studentAssessment.csv"]
recursos = oulad_tablas["vle.csv"]

# ------------------------------------------------------------
# 1. INTEGRIDAD REFERENCIAL DE STUDENTVLE
# ------------------------------------------------------------

print("=== 2.6.1 INTEGRIDAD DE EVENTOS VLE ===")

matriculas_validas = pd.MultiIndex.from_frame(
    info[claves]
)
recursos_validos = pd.MultiIndex.from_frame(
    recursos[claves_recurso]
)

total_vle = 0
sin_matricula = 0
sin_recurso = 0
nulos_vle = 0

for bloque in pd.read_csv(
    oulad_rutas["studentVle.csv"],
    usecols=claves + ["id_site", "date", "sum_click"],
    chunksize=400_000
):
    total_vle += len(bloque)

    nulos_vle += int(
        bloque.isna().any(axis=1).sum()
    )

    idx_matricula = pd.MultiIndex.from_frame(
        bloque[claves]
    )
    idx_recurso = pd.MultiIndex.from_frame(
        bloque[claves_recurso]
    )

    sin_matricula += int(
        (~idx_matricula.isin(matriculas_validas)).sum()
    )
    sin_recurso += int(
        (~idx_recurso.isin(recursos_validos)).sum()
    )

display(pd.DataFrame([
    {"Control": "Eventos VLE totales", "Cantidad": total_vle},
    {"Control": "Eventos sin matrícula", "Cantidad": sin_matricula},
    {"Control": "Eventos sin recurso", "Cantidad": sin_recurso},
    {"Control": "Eventos con nulos", "Cantidad": nulos_vle}
]))

# ------------------------------------------------------------
# 2. AUDITORÍA DE LOS EVENTOS RESPECTO A MATRÍCULA
# ------------------------------------------------------------

print("\n=== 2.6.2 EVENTOS FUERA DE FECHAS DE MATRÍCULA ===")

fechas = registro[
    claves + ["date_registration", "date_unregistration"]
]

vle_fechas = actividad_diaria.merge(
    fechas,
    on=claves,
    how="left",
    validate="many_to_one",
    indicator=True
)

vle_pre_inscripcion = (
    vle_fechas["date_registration"].notna()
    & (
        vle_fechas["date"]
        < vle_fechas["date_registration"]
    )
)

vle_post_retiro = (
    vle_fechas["date_unregistration"].notna()
    & (
        vle_fechas["date"]
        >= vle_fechas["date_unregistration"]
    )
)

auditoria_eventos = pd.DataFrame([
    {
        "Control": "Días VLE únicos analizados (0–55)",
        "Cantidad": len(vle_fechas)
    },
    {
        "Control": "Días VLE anteriores a inscripción",
        "Cantidad": int(vle_pre_inscripcion.sum())
    },
    {
        "Control": "Días VLE desde el desregistro",
        "Cantidad": int(vle_post_retiro.sum())
    },
    {
        "Control": "Días VLE sin matrícula asociada",
        "Cantidad": int(
            vle_fechas["_merge"].eq("left_only").sum()
        )
    }
])

display(auditoria_eventos)

# ------------------------------------------------------------
# 3. PREPARAR ENTREGAS CON INFORMACIÓN TEMPORAL
# ------------------------------------------------------------

entregas = student_assessment.merge(
    assessments[
        ["id_assessment"] + claves_curso
        + ["date", "assessment_type"]
    ],
    on="id_assessment",
    how="left",
    validate="many_to_one"
)

# Excluir evaluaciones reconocidas previamente
# de los eventos ordinarios de entrega.
entregas = entregas.loc[
    entregas["is_banked"].eq(0)
].copy()

# ------------------------------------------------------------
# 4. CREAR POBLACIÓN BASE
# ------------------------------------------------------------

base = info.merge(
    registro,
    on=claves,
    how="left",
    validate="one_to_one"
)

retiro_sin_fecha = (
    base["final_result"].eq("Withdrawn")
    & base["date_unregistration"].isna()
)

# ------------------------------------------------------------
# 5. DISPONIBILIDAD POR VENTANA
# ------------------------------------------------------------

resultados = []

for semana, corte in [(2,14), (4,28), (6,42), (8,56)]:

    # Elegibilidad al momento de predecir
    elegibles = base.loc[
        base["date_registration"].notna()
        & (base["date_registration"] <= corte)
        & (
            base["date_unregistration"].isna()
            | (base["date_unregistration"] > corte)
        )
        & (~retiro_sin_fecha)
    ].copy()

    # Actividad observada desde el día 0
    # hasta el día anterior a la predicción.
    vle_corte = actividad_diaria.loc[
        (actividad_diaria["date"] >= 0)
        & (actividad_diaria["date"] < corte)
    ]

    vle_agg = vle_corte.groupby(
        claves, as_index=False
    ).agg(
        clics_acumulados=("clics", "sum"),
        dias_activos=("date", "nunique")
    )

    # Entregas ordinarias conocidas antes del corte
    entregas_corte = entregas.loc[
        (entregas["date_submitted"] >= 0)
        & (entregas["date_submitted"] < corte)
    ]

    entregas_agg = entregas_corte.groupby(
        claves, as_index=False
    ).agg(
        entregas_acumuladas=("id_assessment", "size")
    )

    # Calendario de evaluaciones vencidas
    plazos = assessments.loc[
        assessments["date"].notna()
        & (assessments["date"] >= 0)
        & (assessments["date"] < corte)
    ]

    plazos_agg = plazos.groupby(
        claves_curso
    ).size().reset_index(
        name="evaluaciones_vencidas"
    )

    # Integración de disponibilidad
    cohorte = elegibles.merge(
        vle_agg,
        on=claves,
        how="left",
        validate="one_to_one"
    )

    cohorte = cohorte.merge(
        entregas_agg,
        on=claves,
        how="left",
        validate="one_to_one"
    )

    cohorte = cohorte.merge(
        plazos_agg,
        on=claves_curso,
        how="left",
        validate="many_to_one"
    )

    for col in [
        "clics_acumulados",
        "dias_activos",
        "entregas_acumuladas",
        "evaluaciones_vencidas"
    ]:
        cohorte[col] = cohorte[col].fillna(0)

    tiene_vle = cohorte["dias_activos"] > 0
    tiene_entregas = cohorte["entregas_acumuladas"] > 0
    tiene_plazos = cohorte["evaluaciones_vencidas"] > 0

    n = len(cohorte)

    resultados.append({
        "Semana": semana,
        "Día de corte": corte,
        "Matrículas elegibles": n,
        "% con VLE acumulado": round(
            100 * tiene_vle.mean(), 2
        ),
        "% con entregas acumuladas": round(
            100 * tiene_entregas.mean(), 2
        ),
        "% con ambas fuentes": round(
            100 * (tiene_vle & tiene_entregas).mean(), 2
        ),
        "% sin ninguna fuente": round(
            100 * (~tiene_vle & ~tiene_entregas).mean(), 2
        ),
        "% con plazos vencidos": round(
            100 * tiene_plazos.mean(), 2
        ),
        "Mediana clics acumulados": round(
            cohorte["clics_acumulados"].median(), 2
        ),
        "Mediana entregas acumuladas": round(
            cohorte["entregas_acumuladas"].median(), 2
        )
    })

tabla_disponibilidad_temporal = pd.DataFrame(
    resultados
)

print("\n=== 2.6.3 DISPONIBILIDAD TEMPORAL ===")
display(tabla_disponibilidad_temporal)

# ------------------------------------------------------------
# 6. VISUALIZACIÓN DE COBERTURA
# ------------------------------------------------------------

plt.figure(figsize=(9, 4))

for columna in [
    "% con VLE acumulado",
    "% con entregas acumuladas",
    "% con ambas fuentes"
]:
    plt.plot(
        tabla_disponibilidad_temporal["Semana"],
        tabla_disponibilidad_temporal[columna],
        marker="o",
        label=columna
    )

plt.title(
    "Disponibilidad acumulada de fuentes longitudinales"
)
plt.xlabel("Semana de predicción")
plt.ylabel("Matrículas con información (%)")
plt.xticks([2, 4, 6, 8])
plt.ylim(0, 100)
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

print("\n=== ACTIVIDAD 2.6 FINALIZADA ===")
print("No se han entrenado modelos.")
print("No se han generado características predictivas definitivas.")
print("No se han utilizado notas ni resultados finales como entradas.")



### 2.6.4. Resultados e interpretación

**Integridad de los registros longitudinales**

La auditoría de los 10.655.280 registros originales de `studentVle.csv` no identificó eventos sin correspondencia con una matrícula o recurso académico existente. Tampoco se encontraron registros con valores faltantes.

En las primeras ocho semanas se analizaron 562.124 observaciones únicas por estudiante, módulo y día.

No se identificaron observaciones anteriores a la fecha de inscripción dentro del intervalo examinado. Sin embargo, se encontraron 4.802 observaciones estudiante-módulo-día con actividad desde la fecha de desregistro.

Estos eventos deberán controlarse durante la construcción de las trayectorias, evitando incorporar actividad posterior al desregistro como parte del historial de una matrícula activa.

**Disponibilidad temporal de las fuentes longitudinales**

La cobertura acumulada de información evolucionó de la siguiente manera:

| Semana | Con actividad VLE | Con entregas | Con ambas fuentes |
|---|---:|---:|---:|
| 2 | 90,50 % | 10,28 % | 10,06 % |
| 4 | 95,68 % | 71,83 % | 71,51 % |
| 6 | 96,72 % | 82,63 % | 82,45 % |
| 8 | 97,22 % | 85,95 % | 85,87 % |

La actividad VLE presentó una cobertura elevada desde la semana 2, mientras que los registros de entregas académicas mostraron una disponibilidad considerablemente mayor a partir de la semana 4.

La proporción de matrículas sin registros en ninguna de las dos fuentes disminuyó del 9,28 % en la semana 2 al 2,69 % en la semana 8.

**Implicaciones para la predicción temprana**

Estos resultados justifican investigar el aporte de las características longitudinales en diferentes momentos del período académico.

La elevada disponibilidad inicial de actividad VLE permite construir indicadores de participación desde etapas tempranas, mientras que la información de entregas puede aportar señales complementarias conforme avanza el calendario académico.

Sin embargo, la mayor disponibilidad de información no garantiza una mejora del rendimiento predictivo.

La contribución de cada fuente deberá evaluarse mediante comparaciones controladas entre modelos que utilicen las mismas matrículas elegibles y particiones de evaluación.

### 2.6.5. Conclusiones

La auditoría confirmó que OULAD dispone de información longitudinal estructuralmente consistente y temporalmente suficiente para investigar la predicción temprana de resultados académicos adversos.

La cobertura de actividad VLE permite plantear experimentos desde la semana 2, mientras que las entregas académicas ofrecen información complementaria especialmente a partir de la semana 4.

Se identificaron observaciones posteriores al desregistro que requieren controles explícitos para evitar inconsistencias temporales.

Las características predictivas se construirán respetando las fechas de inscripción, desregistro y predicción, sin utilizar resultados futuros como variables de entrada.

**Conclusión principal:** OULAD permite desarrollar experimentos comparativos de predicción temprana basados en variables estáticas y longitudinales. La viabilidad temporal está respaldada por los datos, pero la superioridad predictiva de las trayectorias longitudinales permanece pendiente de contrastación experimental.



### 2.7. Análisis de estacionalidad y patrones cíclicos de actividad virtual

#### Objetivo

Investigar la presencia de patrones temporales repetitivos en la actividad virtual estudiantil durante las primeras ocho semanas académicas.

#### Justificación metodológica

El presente proyecto requiere evaluar la existencia de estacionalidad cuando se trabaja con información temporal.

En OULAD, los registros de actividad contienen fechas relativas al inicio de cada módulo. Esto permite estudiar patrones de participación diaria y posibles ciclos de siete días.

Sin embargo, las fechas relativas no identifican directamente el día de la semana del calendario. Por ello, los grupos calculados mediante `date % 7` se interpretarán como posiciones dentro de un ciclo relativo de siete días, no necesariamente como lunes, martes u otros días específicos.

#### Procedimiento

1. Reconstruir la actividad diaria durante los días 0 a 55.
2. Considerar únicamente matrículas activas en el día observado.
3. Calcular el porcentaje diario de matrículas con actividad VLE.
4. Analizar la intensidad de participación mediante clics por matrícula elegible.
5. Comparar las siete posiciones del ciclo semanal relativo.
6. Examinar descriptivamente la autocorrelación con un desfase de siete días.
7. Distinguir cambios de tendencia de posibles patrones repetitivos.

#### Consideraciones metodológicas

Los resultados permitirán identificar indicios descriptivos de periodicidad, pero no demostrarán necesariamente una estacionalidad estable.

Las variaciones también pueden depender de calendarios de evaluación, diferencias entre módulos y cambios en la población elegible.

No se introducirán transformaciones ni características predictivas definitivas durante esta actividad.


In [ ]:

# ============================================================
# PROYECTO FINAL DE MACHINE LEARNING
# ACTIVIDAD 2.7 - ESTACIONALIDAD Y CICLOS DE ACTIVIDAD
# ============================================================

# Asistencia IA: ChatGPT, octubre de 2026.
# Prompt:
# "Analizar posibles ciclos de siete días en la
# actividad VLE de OULAD, controlando la elegibilidad
# diaria de las matrículas. Examinar tendencias y
# autocorrelación descriptiva sin entrenar modelos."

# ------------------------------------------------------------
# 1. PREPARAR REGISTROS Y FECHAS DE MATRÍCULA
# ------------------------------------------------------------

claves = [
    "code_module",
    "code_presentation",
    "id_student"
]

info = oulad_tablas["studentInfo.csv"]
registro = oulad_tablas["studentRegistration.csv"]

base = registro.merge(
    info[claves + ["final_result"]],
    on=claves,
    how="left",
    validate="one_to_one"
)

base["retiro_sin_fecha"] = (
    base["final_result"].eq("Withdrawn")
    & base["date_unregistration"].isna()
)

# ------------------------------------------------------------
# 2. FILTRAR ACTIVIDAD CON ELEGIBILIDAD DIARIA
# ------------------------------------------------------------

print("=== 2.7.1 CONTROL DE ACTIVIDAD DIARIA ===")

actividad = actividad_diaria.merge(
    base[
        claves +
        [
            "date_registration",
            "date_unregistration",
            "retiro_sin_fecha"
        ]
    ],
    on=claves,
    how="left",
    validate="many_to_one"
)

valida = (
    actividad["date_registration"].notna()
    & (
        actividad["date"]
        >= actividad["date_registration"]
    )
    & (
        actividad["date_unregistration"].isna()
        | (
            actividad["date"]
            < actividad["date_unregistration"]
        )
    )
    & (~actividad["retiro_sin_fecha"])
)

print(
    "Observaciones estudiante-módulo-día examinadas:",
    len(actividad)
)

print(
    "Observaciones excluidas por elegibilidad:",
    int((~valida).sum())
)

actividad_valida = actividad.loc[valida].copy()

# ------------------------------------------------------------
# 3. AGREGACIÓN DIARIA
# ------------------------------------------------------------

actividad_por_dia = (
    actividad_valida.groupby("date")
    .agg(
        matriculas_con_vle=("id_student", "size"),
        clics_totales=("clics", "sum")
    )
    .reset_index()
)

# ------------------------------------------------------------
# 4. DENOMINADOR DE MATRÍCULAS ELEGIBLES
# ------------------------------------------------------------

poblacion = base.loc[
    base["date_registration"].notna()
    & (~base["retiro_sin_fecha"])
].copy()

resumen_poblacion = []

for dia in range(56):

    elegibles = (
        (poblacion["date_registration"] <= dia)
        & (
            poblacion["date_unregistration"].isna()
            | (
                poblacion["date_unregistration"] > dia
            )
        )
    )

    resumen_poblacion.append({
        "date": dia,
        "matriculas_elegibles": int(elegibles.sum())
    })

tabla_poblacion = pd.DataFrame(resumen_poblacion)

# ------------------------------------------------------------
# 5. CONSTRUIR SERIE DIARIA COMPLETA
# ------------------------------------------------------------

serie_diaria = tabla_poblacion.merge(
    actividad_por_dia,
    on="date",
    how="left",
    validate="one_to_one"
)

serie_diaria[
    ["matriculas_con_vle", "clics_totales"]
] = serie_diaria[
    ["matriculas_con_vle", "clics_totales"]
].fillna(0)

serie_diaria["participacion_pct"] = (
    100
    * serie_diaria["matriculas_con_vle"]
    / serie_diaria["matriculas_elegibles"]
)

serie_diaria["clics_por_matricula"] = (
    serie_diaria["clics_totales"]
    / serie_diaria["matriculas_elegibles"]
)

serie_diaria["semana"] = (
    serie_diaria["date"] // 7 + 1
)

serie_diaria["posicion_ciclo"] = (
    serie_diaria["date"] % 7
)

print("\n=== 2.7.2 SERIE DIARIA ===")

display(serie_diaria.head(14))

print("\nEstadística descriptiva:")
display(
    serie_diaria[
        ["participacion_pct", "clics_por_matricula"]
    ].describe().round(3)
)

# ------------------------------------------------------------
# 6. ACTIVIDAD DIARIA Y TENDENCIA
# ------------------------------------------------------------

serie_diaria["media_movil_7"] = (
    serie_diaria["participacion_pct"]
    .rolling(
        window=7,
        center=True,
        min_periods=4
    )
    .mean()
)

plt.figure(figsize=(11, 4))

plt.plot(
    serie_diaria["date"],
    serie_diaria["participacion_pct"],
    label="Participación diaria",
    alpha=0.65
)

plt.plot(
    serie_diaria["date"],
    serie_diaria["media_movil_7"],
    linewidth=2,
    label="Media móvil de 7 días"
)

plt.title("Participación VLE diaria y tendencia suavizada")
plt.xlabel("Día relativo al inicio del módulo")
plt.ylabel("Matrículas con actividad (%)")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 7. PATRONES DEL CICLO DE SIETE DÍAS
# ------------------------------------------------------------

print("\n=== 2.7.3 CICLO RELATIVO DE 7 DÍAS ===")

ciclo = (
    serie_diaria.groupby("posicion_ciclo")
    .agg(
        participacion_media=("participacion_pct", "mean"),
        desviacion=("participacion_pct", "std"),
        clics_medios=("clics_por_matricula", "mean"),
        dias_observados=("date", "size")
    )
    .reset_index()
)

display(ciclo.round(3))

plt.figure(figsize=(9, 4))

plt.bar(
    ciclo["posicion_ciclo"],
    ciclo["participacion_media"]
)

plt.title("Participación media por posición del ciclo")
plt.xlabel("Posición relativa en el ciclo de siete días")
plt.ylabel("Participación diaria promedio (%)")
plt.xticks(range(7))
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 8. AUTOCORRELACIÓN DESCRIPTIVA
# ------------------------------------------------------------

print("\n=== 2.7.4 AUTOCORRELACIÓN ===")

# Serie original
autocorr_7 = (
    serie_diaria["participacion_pct"]
    .autocorr(lag=7)
)

# Serie después de restar la media móvil.
# Este procedimiento es descriptivo y no constituye
# una descomposición estacional formal.

residuo = (
    serie_diaria["participacion_pct"]
    - serie_diaria["media_movil_7"]
)

autocorr_residuo_7 = residuo.autocorr(lag=7)

tabla_autocorrelacion = pd.DataFrame([
    {
        "Indicador": "Autocorrelación original, lag 7",
        "Valor": autocorr_7
    },
    {
        "Indicador": "Autocorrelación tras suavizado, lag 7",
        "Valor": autocorr_residuo_7
    }
])

display(tabla_autocorrelacion.round(4))

# ------------------------------------------------------------
# 9. RESUMEN POR SEMANA
# ------------------------------------------------------------

print("\n=== 2.7.5 RESUMEN SEMANAL ===")

resumen_semanal = (
    serie_diaria.groupby("semana")
    .agg(
        participacion_diaria_media=(
            "participacion_pct", "mean"
        ),
        clics_diarios_por_matricula=(
            "clics_por_matricula", "mean"
        )
    )
    .reset_index()
)

display(resumen_semanal.round(3))

print("\nACTIVIDAD 2.7 FINALIZADA")
print("No se han entrenado modelos.")
print("No se han generado variables predictivas definitivas.")



### 2.7.6. Resultados e interpretación

El análisis temporal examinó 562.124 observaciones únicas estudiante-módulo-día correspondientes a los primeros 56 días académicos.

Tras verificar la elegibilidad temporal, se excluyeron 5.256 observaciones, conservándose 556.868 observaciones para el análisis.

La participación diaria presentó una media del 35,97 % de las matrículas elegibles y una desviación estándar de 5,57 puntos porcentuales.

Se identificaron diferencias sistemáticas entre las siete posiciones relativas del ciclo semanal. La mayor participación media correspondió a la posición 2, con un 42,06 %, mientras que la menor se observó en la posición 0, con un 28,93 %.

La diferencia entre ambas posiciones fue de 13,13 puntos porcentuales.

La autocorrelación de la serie agregada con desfase de siete días fue de 0,8191. Después de restar una media móvil de siete días, la autocorrelación correspondiente fue de 0,9369.

Estos resultados son compatibles con un patrón semanal repetitivo en la actividad virtual.

No obstante, la autocorrelación observada no constituye por sí sola una prueba formal de estacionalidad estable. El análisis se limita a ocho semanas y puede estar influido por calendarios académicos, diferencias entre módulos y cambios en la composición de las matrículas elegibles.

### 2.7.7. Conclusiones

La actividad virtual presenta una estructura temporal compatible con ciclos semanales de participación.

Estos patrones justifican conservar el orden cronológico al construir características longitudinales y evaluar tendencias, variaciones semanales y períodos de inactividad.

También respaldan la pertinencia de investigar una arquitectura Transformer temporal como experimento complementario, sin presuponer que superará a los modelos clásicos.

**Conclusión principal:** la serie agregada de participación de OULAD presenta indicios sólidos de periodicidad semanal, lo que refuerza la pertinencia metodológica de evaluar representaciones longitudinales del comportamiento estudiantil.



### 2.8. Conclusiones generales del análisis exploratorio

El análisis exploratorio de OULAD confirmó la viabilidad de desarrollar un estudio comparativo de predicción temprana de resultados académicos adversos utilizando información estática y longitudinal.

El dataset contiene 32.593 matrículas, 28.785 estudiantes distintos, 173.912 registros de evaluaciones individuales y 10.655.280 registros de interacción virtual.

La distribución binaria de resultados académicos presenta un balance global relativamente equilibrado, aunque se identificaron diferencias importantes entre módulos y presentaciones académicas.

Las variables numéricas estáticas presentan distintos grados de asimetría y dispersión, lo que justifica evaluar procedimientos específicos de escalamiento, codificación y transformación dentro de los pipelines.

La actividad virtual muestra variaciones tanto en frecuencia como en intensidad, además de indicios de periodicidad semanal. Las entregas académicas proporcionan una segunda fuente de información longitudinal cuya disponibilidad aumenta durante las primeras semanas.

La cobertura de actividad VLE acumulada alcanza el 90,50 % de las matrículas elegibles en la semana 2 y el 97,22 % en la semana 8.

Se identificaron registros temporales que requieren controles de elegibilidad, así como limitaciones sobre la fecha de disponibilidad de las calificaciones.

Estos resultados justifican desarrollar experimentos controlados que permitan medir:

1. El rendimiento de modelos basados exclusivamente en características estáticas.
2. El aporte incremental de las características longitudinales.
3. Las diferencias entre algoritmos supervisados.
4. La capacidad predictiva en distintas semanas del período académico.
5. El posible aporte adicional de un Transformer que aprenda directamente las secuencias temporales.

El análisis exploratorio no demuestra aún una relación predictiva causal ni confirma la superioridad de los modelos longitudinales.

**Conclusión:** las características y la cobertura temporal de OULAD son adecuadas para contrastar experimentalmente la hipótesis principal, siempre que se respeten las fechas de disponibilidad, se evite la fuga de información y se utilicen particiones de evaluación independientes.



## 3. Construcción de la base experimental

### 3.1. Definición de cohortes y separación de datos

#### Objetivo

Construir las cohortes de matrículas elegibles para realizar predicciones en las semanas 2, 4, 6 y 8, definiendo una partición reproducible de entrenamiento y prueba que evite compartir estudiantes entre ambos conjuntos.

#### Unidad de análisis

La unidad de observación será una matrícula identificada mediante:

- `code_module`
- `code_presentation`
- `id_student`

Una matrícula podrá contribuir a diferentes ventanas de predicción, pero estas observaciones corresponden a distintos momentos de una misma trayectoria y no son estadísticamente independientes.

#### Variable objetivo

Se define provisionalmente una clasificación binaria:

- Clase 0: resultado favorable (`Pass`, `Distinction`).
- Clase 1: resultado académico adverso (`Fail`, `Withdrawn`).

Adicionalmente, se conservará la etiqueta original para analizar separadamente la reprobación y los retiros.

#### Elegibilidad temporal

Para participar en una ventana, la matrícula deberá estar iniciada y permanecer activa en el día de predicción.

Se excluirán los retiros cuya fecha sea desconocida y las matrículas sin fecha verificable de inscripción.

Las fechas de retiro posteriores al momento de predicción podrán utilizarse para calcular etiquetas y anticipación, pero nunca como entradas del modelo.

#### Separación experimental

Se realizará una separación inicial del 80 % de los estudiantes para desarrollo y entrenamiento, y 20 % para prueba independiente.

La asignación se realizará por identificador de estudiante, no por fila de matrícula, y será reproducible mediante una semilla aleatoria fija.

El conjunto de prueba permanecerá reservado para la evaluación final. Los hiperparámetros, las transformaciones y la selección de características se ajustarán exclusivamente sobre los datos de entrenamiento mediante validación cruzada.

#### Alcance de la evaluación

Esta separación permitirá estimar el rendimiento sobre estudiantes no observados durante el entrenamiento, dentro de los módulos y presentaciones disponibles en OULAD.

No constituye por sí sola una validación de generalización a nuevos años o instituciones. Esa limitación se analizará por separado.

**Fundamentación:** la evaluación mediante datos reservados y validación cruzada se relaciona con los principios de estimación y selección de modelos [2], [10]. La agrupación por estudiante es una adaptación necesaria a las matrículas repetidas de OULAD para impedir solapamiento de identidades entre subconjuntos.


In [ ]:

# ============================================================
# PROYECTO FINAL DE MACHINE LEARNING
# ACTIVIDAD 3.1 - COHORTES Y SPLIT POR ESTUDIANTE
# ============================================================

# Asistencia IA: ChatGPT, octubre de 2026.
# Prompt:
# "Construir cohortes de predicción en los días 14,
# 28, 42 y 56 de OULAD. Mantener solo matrículas
# elegibles en cada corte y generar una partición
# 80/20 fija por id_student, sin compartir estudiantes
# entre entrenamiento y prueba ni utilizar variables
# futuras para entrenar modelos."

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from IPython.display import display

SEMILLA = 42
CORTES = {2: 14, 4: 28, 6: 42, 8: 56}

CLAVES = [
    "code_module",
    "code_presentation",
    "id_student"
]

# ------------------------------------------------------------
# 1. INTEGRAR MATRÍCULAS Y CONTEXTO ACADÉMICO
# ------------------------------------------------------------

info = oulad_tablas["studentInfo.csv"].copy()
registro = oulad_tablas["studentRegistration.csv"].copy()
cursos = oulad_tablas["courses.csv"].copy()

base = info.merge(
    registro,
    on=CLAVES,
    how="left",
    validate="one_to_one"
)

base = base.merge(
    cursos,
    on=["code_module", "code_presentation"],
    how="left",
    validate="many_to_one"
)

assert len(base) == len(info)

# ------------------------------------------------------------
# 2. DEFINIR RESULTADO ACADÉMICO
# ------------------------------------------------------------

mapa_resultado = {
    "Pass": 0,
    "Distinction": 0,
    "Fail": 1,
    "Withdrawn": 1
}

base["target_adverso"] = (
    base["final_result"].map(mapa_resultado)
)

assert base["target_adverso"].notna().all()

base["target_adverso"] = (
    base["target_adverso"].astype(int)
)

base["retiro_sin_fecha"] = (
    base["final_result"].eq("Withdrawn")
    & base["date_unregistration"].isna()
)

# ------------------------------------------------------------
# 3. GENERAR PARTICIÓN FIJA POR ESTUDIANTE
# ------------------------------------------------------------

estudiantes_unicos = np.sort(
    base["id_student"].unique()
)

estudiantes_train, estudiantes_test = (
    train_test_split(
        estudiantes_unicos,
        test_size=0.20,
        random_state=SEMILLA,
        shuffle=True
    )
)

ids_train = set(estudiantes_train)
ids_test = set(estudiantes_test)

assert ids_train.isdisjoint(ids_test)

print("=== 3.1.1 PARTICIÓN POR ESTUDIANTE ===")

print("Estudiantes únicos:", len(estudiantes_unicos))
print("Estudiantes entrenamiento:", len(ids_train))
print("Estudiantes prueba:", len(ids_test))
print("Solapamiento:", len(ids_train & ids_test))

# ------------------------------------------------------------
# 4. CONSTRUIR COHORTES ELEGIBLES POR VENTANA
# ------------------------------------------------------------

cohortes = []

for semana, corte in CORTES.items():

    inscrito = (
        base["date_registration"].notna()
        & (base["date_registration"] <= corte)
    )

    activo = (
        base["date_unregistration"].isna()
        | (base["date_unregistration"] > corte)
    )

    elegibles = base.loc[
        inscrito
        & activo
        & (~base["retiro_sin_fecha"])
    ].copy()

    elegibles["semana_prediccion"] = semana
    elegibles["dia_prediccion"] = corte

    # Resultado adverso original conservado.
    # Solo para etiquetas y análisis posterior.
    elegibles["target_retiro_futuro"] = (
        elegibles["final_result"].eq("Withdrawn")
        & (elegibles["date_unregistration"] > corte)
    ).astype(int)

    elegibles["target_reprobacion"] = (
        elegibles["final_result"].eq("Fail")
    ).astype(int)

    # Anticipación real de retiros posteriores.
    # Nunca será una característica predictiva.
    elegibles["dias_hasta_retiro"] = np.where(
        elegibles["target_retiro_futuro"] == 1,
        elegibles["date_unregistration"] - corte,
        np.nan
    )

    # Asignación fija sin compartir estudiantes.
    elegibles["conjunto"] = np.where(
        elegibles["id_student"].isin(ids_test),
        "Prueba",
        "Entrenamiento"
    )

    cohortes.append(elegibles)

cohortes_oulad = pd.concat(
    cohortes,
    ignore_index=True
)

# ------------------------------------------------------------
# 5. CONTROLES DE SEGURIDAD EXPERIMENTAL
# ------------------------------------------------------------

assert not cohortes_oulad.duplicated(
    subset=["semana_prediccion"] + CLAVES
).any(), "Se identificaron matrículas duplicadas en una ventana."

assert (
    cohortes_oulad.loc[
        cohortes_oulad["conjunto"] == "Entrenamiento",
        "id_student"
    ].isin(ids_test).sum() == 0
)

assert (
    cohortes_oulad.loc[
        cohortes_oulad["conjunto"] == "Prueba",
        "id_student"
    ].isin(ids_train).sum() == 0
)

assert (
    cohortes_oulad["date_registration"]
    <= cohortes_oulad["dia_prediccion"]
).all()

assert (
    cohortes_oulad["date_unregistration"].isna()
    | (
        cohortes_oulad["date_unregistration"]
        > cohortes_oulad["dia_prediccion"]
    )
).all()

# ------------------------------------------------------------
# 6. COMPROBAR POBLACIONES
# ------------------------------------------------------------

print("\n=== 3.1.2 COHORTES POR SEMANA ===")

resumen_cohortes = (
    cohortes_oulad.groupby("semana_prediccion")
    .agg(
        Matriculas=("id_student", "size"),
        Estudiantes_unicos=("id_student", "nunique"),
        Adversos=("target_adverso", "sum"),
        Retiros_futuros=("target_retiro_futuro", "sum")
    )
    .reset_index()
)

resumen_cohortes["% adversos"] = (
    100
    * resumen_cohortes["Adversos"]
    / resumen_cohortes["Matriculas"]
).round(2)

display(resumen_cohortes)

# Las cohortes deben coincidir con la Actividad 2.6.
esperadas = {
    2: 27962,
    4: 27422,
    6: 26912,
    8: 26413
}

for semana, n_esperado in esperadas.items():

    n_observado = int(
        resumen_cohortes.loc[
            resumen_cohortes["semana_prediccion"] == semana,
            "Matriculas"
        ].iloc[0]
    )

    assert n_observado == n_esperado, (
        f"Semana {semana}: se esperaban "
        f"{n_esperado} matrículas y hay {n_observado}."
    )

print("Las cohortes coinciden con la auditoría temporal.")

# ------------------------------------------------------------
# 7. BALANCE POR CONJUNTO Y SEMANA
# ------------------------------------------------------------

print("\n=== 3.1.3 BALANCE ENTRENAMIENTO / PRUEBA ===")

balance = (
    cohortes_oulad.groupby(
        ["semana_prediccion", "conjunto"]
    )
    .agg(
        Matriculas=("id_student", "size"),
        Adversos=("target_adverso", "sum"),
        Retiros_futuros=("target_retiro_futuro", "sum")
    )
    .reset_index()
)

balance["% adversos"] = (
    100
    * balance["Adversos"]
    / balance["Matriculas"]
).round(2)

display(balance)

# ------------------------------------------------------------
# 8. VERIFICAR SOLAPAMIENTO
# ------------------------------------------------------------

print("\n=== 3.1.4 CONTROL DE SOLAPAMIENTO ===")

train = cohortes_oulad.loc[
    cohortes_oulad["conjunto"] == "Entrenamiento"
]

test = cohortes_oulad.loc[
    cohortes_oulad["conjunto"] == "Prueba"
]

solapamiento = (
    set(train["id_student"])
    & set(test["id_student"])
)

print(
    "Estudiantes presentes en ambos conjuntos:",
    len(solapamiento)
)

assert len(solapamiento) == 0

# ------------------------------------------------------------
# 9. DEFINIR VARIABLES ESTÁTICAS CANDIDATAS
# ------------------------------------------------------------

VARIABLES_ESTATICAS = [
    "code_module",
    "code_presentation",
    "gender",
    "region",
    "highest_education",
    "imd_band",
    "age_band",
    "num_of_prev_attempts",
    "studied_credits",
    "disability",
    "date_registration",
    "module_presentation_length"
]

COLUMNAS_PROHIBIDAS_COMO_FEATURES = [
    "final_result",
    "target_adverso",
    "target_retiro_futuro",
    "target_reprobacion",
    "date_unregistration",
    "dias_hasta_retiro"
]

assert not (
    set(VARIABLES_ESTATICAS)
    & set(COLUMNAS_PROHIBIDAS_COMO_FEATURES)
)

print("\n=== 3.1.5 VARIABLES ESTÁTICAS CANDIDATAS ===")

print(VARIABLES_ESTATICAS)

print("\nVariables reservadas para etiquetas y evaluación:")
print(COLUMNAS_PROHIBIDAS_COMO_FEATURES)

# ------------------------------------------------------------
# 10. FINALIZACIÓN
# ------------------------------------------------------------

print("\n=== ACTIVIDAD 3.1 FINALIZADA ===")
print("Se construyeron las cuatro cohortes.")
print("Se creó una partición fija por estudiante.")
print("No existe solapamiento entre entrenamiento y prueba.")
print("No se han entrenado modelos ni seleccionado variables.")


In [ ]:
# SNAPSHOT 3.1 - Cohortes y grupos para reproducibilidad
assert len(cohortes_oulad) > 0
guardar_estado_oulad('cohortes_oulad')



### 3.1.6. Resultados e interpretación

La construcción de las cohortes predictivas permitió verificar la elegibilidad temporal de las matrículas en los días 14, 28, 42 y 56, correspondientes a las semanas 2, 4, 6 y 8 completas.

Se generó una partición reproducible a nivel de estudiante, utilizando el 80 % de los estudiantes para desarrollo y entrenamiento y el 20 % para prueba independiente.

La partición incluyó:

- 23.028 estudiantes en entrenamiento.
- 5.757 estudiantes en prueba.
- 0 estudiantes compartidos entre ambos conjuntos.

Las cohortes elegibles y su distribución fueron:

| Semana | Entrenamiento | Prueba | % adversos entrenamiento | % adversos prueba |
|---|---:|---:|---:|---:|
| 2 | 22.412 | 5.550 | 45,15 % | 44,72 % |
| 4 | 21.984 | 5.438 | 44,02 % | 43,51 % |
| 6 | 21.568 | 5.344 | 42,94 % | 42,51 % |
| 8 | 21.164 | 5.249 | 41,84 % | 41,47 % |

Las diferencias de distribución entre entrenamiento y prueba fueron reducidas en todas las ventanas, sin que se detectara solapamiento de estudiantes.

### 3.1.7. Conclusión

Las cohortes construidas proporcionan una base reproducible para evaluar la predicción temprana de resultados académicos adversos.

La separación por estudiante evita que distintas matrículas de una misma persona se distribuyan simultáneamente entre entrenamiento y prueba.

La utilización de una asignación fija permite comparar los modelos estáticos, longitudinales y Transformer sobre conjuntos equivalentes.

El conjunto de prueba quedará reservado para la evaluación final. El preprocesamiento, la selección de características y la optimización se realizarán dentro del conjunto de entrenamiento mediante validación cruzada.

La separación actual permite evaluar la generalización a estudiantes no observados, aunque no garantiza por sí sola la generalización a períodos académicos o instituciones diferentes.

**Conclusión principal:** se estableció una partición experimental consistente, sin solapamiento de estudiantes y con distribuciones de clases comparables, apta para construir y contrastar los modelos del proyecto.



### 3.2. Construcción del conjunto de características estáticas

#### Objetivo

Construir el conjunto de características estáticas correspondiente a la segunda semana del período académico, que funcionará como referencia para evaluar el aporte incremental de las características longitudinales.

#### Definición del experimento A

El experimento A utilizará variables relacionadas con el perfil académico, demográfico y contextual de cada matrícula.

La predicción se realizará al completar los primeros 14 días académicos, utilizando exclusivamente características consideradas disponibles al inicio de la matrícula o antes de la fecha de predicción.

#### Variables candidatas

**Contexto académico:**
- `code_module`
- `code_presentation`
- `module_presentation_length`

**Antecedentes académicos:**
- `highest_education`
- `num_of_prev_attempts`
- `studied_credits`
- `date_registration`

**Información demográfica:**
- `gender`
- `region`
- `imd_band`
- `age_band`
- `disability`

Estas variables constituyen características candidatas y no características seleccionadas definitivamente.

Se considerará que los créditos registrados y la duración del módulo representan información administrativa disponible al momento de la predicción. Esta suposición deberá documentarse como una limitación del dataset, debido a la ausencia de historiales de actualización de estos atributos.

#### Variable objetivo

La variable `target_adverso` representará:

- 0: resultado académico favorable.
- 1: resultado académico adverso.

#### Prevención de fuga de información

Quedarán excluidas de las características predictivas:

- El identificador personal `id_student`.
- El resultado académico final.
- Las fechas de desregistro.
- Las etiquetas auxiliares de retiro y reprobación.
- Los días restantes hasta el retiro.
- Cualquier evento o resultado posterior a la fecha de predicción.

Las variables de identificación y las etiquetas se conservarán separadamente para controlar las relaciones entre registros y realizar la evaluación.

#### Procedimiento

1. Seleccionar la cohorte elegible de la semana 2.
2. Recuperar la partición fija de entrenamiento y prueba.
3. Construir las matrices `X` de características.
4. Separar las etiquetas `y`.
5. Conservar identificadores de estudiante como grupos de validación.
6. Auditar tipos, categorías y valores faltantes utilizando el conjunto de entrenamiento.
7. Verificar que no existan variables futuras entre las características.

No se realizará imputación, codificación, selección ni escalamiento en esta actividad.

Estas transformaciones se ajustarán posteriormente dentro de pipelines reproducibles, sin utilizar el conjunto de prueba.

**Fundamentación:** la construcción del conjunto estático sigue el carácter demográfico y académico de OULAD [1]. Las variables deben estar disponibles en el instante de predicción; `final_result` se conserva exclusivamente como etiqueta [1].


In [ ]:

# ============================================================
# PROYECTO FINAL DE MACHINE LEARNING
# ACTIVIDAD 3.2 - CARACTERÍSTICAS ESTÁTICAS, SEMANA 2
# ============================================================

# Asistencia IA: ChatGPT, octubre de 2026.
# Prompt:
# "Construir las matrices de características estáticas
# para la cohorte de semana 2 de OULAD, separando
# entrenamiento, prueba, etiquetas y grupos de
# estudiante, auditando tipos y valores faltantes
# sin entrenar modelos ni introducir fuga de información."

SEMANA_BASE = 2

CLAVES = [
    "code_module",
    "code_presentation",
    "id_student"
]

VARIABLES_ESTATICAS = [
    "code_module",
    "code_presentation",
    "gender",
    "region",
    "highest_education",
    "imd_band",
    "age_band",
    "num_of_prev_attempts",
    "studied_credits",
    "disability",
    "date_registration",
    "module_presentation_length"
]

VARIABLES_NUMERICAS = [
    "num_of_prev_attempts",
    "studied_credits",
    "date_registration",
    "module_presentation_length"
]

VARIABLES_CATEGORICAS = [
    c for c in VARIABLES_ESTATICAS
    if c not in VARIABLES_NUMERICAS
]

COLUMNAS_PROHIBIDAS = {
    "id_student",
    "final_result",
    "target_adverso",
    "target_retiro_futuro",
    "target_reprobacion",
    "date_unregistration",
    "dias_hasta_retiro",
    "dia_prediccion",
    "semana_prediccion",
    "conjunto"
}

# ------------------------------------------------------------
# 1. VERIFICAR CARACTERÍSTICAS PERMITIDAS
# ------------------------------------------------------------

assert not (
    set(VARIABLES_ESTATICAS) & COLUMNAS_PROHIBIDAS
), "Una variable prohibida aparece como característica."

# ------------------------------------------------------------
# 2. SELECCIONAR LA COHORTE DE SEMANA 2
# ------------------------------------------------------------

cohorte_s2 = cohortes_oulad.loc[
    cohortes_oulad["semana_prediccion"] == SEMANA_BASE
].copy()

assert len(cohorte_s2) == 27962

assert cohorte_s2["dia_prediccion"].eq(14).all()

faltantes = (
    set(VARIABLES_ESTATICAS)
    - set(cohorte_s2.columns)
)

if faltantes:
    raise ValueError(
        f"No se encontraron variables: {faltantes}"
    )

print("=== 3.2.1 COHORTE DE SEMANA 2 ===")

print("Matrículas elegibles:", len(cohorte_s2))

# ------------------------------------------------------------
# 3. SEPARAR ENTRENAMIENTO Y PRUEBA
# ------------------------------------------------------------

train_s2 = cohorte_s2.loc[
    cohorte_s2["conjunto"] == "Entrenamiento"
].copy().reset_index(drop=True)

test_s2 = cohorte_s2.loc[
    cohorte_s2["conjunto"] == "Prueba"
].copy().reset_index(drop=True)

# ------------------------------------------------------------
# 4. CONSTRUIR X, y Y GRUPOS
# ------------------------------------------------------------

X_estatico_train = train_s2[
    VARIABLES_ESTATICAS
].copy()

X_estatico_test = test_s2[
    VARIABLES_ESTATICAS
].copy()

y_estatico_train = train_s2[
    "target_adverso"
].astype(int).copy()

y_estatico_test = test_s2[
    "target_adverso"
].astype(int).copy()

grupos_train = train_s2[
    "id_student"
].copy()

grupos_test = test_s2[
    "id_student"
].copy()

# Conservar claves para futuras uniones longitudinales.
claves_train_s2 = train_s2[CLAVES].copy()
claves_test_s2 = test_s2[CLAVES].copy()

# ------------------------------------------------------------
# 5. CONTROLES DE INTEGRIDAD
# ------------------------------------------------------------

assert len(X_estatico_train) == len(y_estatico_train)
assert len(X_estatico_test) == len(y_estatico_test)

assert X_estatico_train.columns.tolist() == (
    X_estatico_test.columns.tolist()
)

assert set(grupos_train).isdisjoint(set(grupos_test))

assert y_estatico_train.isin([0, 1]).all()
assert y_estatico_test.isin([0, 1]).all()

assert X_estatico_train.index.equals(
    y_estatico_train.index
)

assert X_estatico_test.index.equals(
    y_estatico_test.index
)

# ------------------------------------------------------------
# 6. DIMENSIONES DE LOS CONJUNTOS
# ------------------------------------------------------------

print("\n=== 3.2.2 DIMENSIONES ===")

dimensiones = pd.DataFrame([
    {
        "Conjunto": "Entrenamiento",
        "Filas": X_estatico_train.shape[0],
        "Variables": X_estatico_train.shape[1],
        "Estudiantes únicos": grupos_train.nunique()
    },
    {
        "Conjunto": "Prueba",
        "Filas": X_estatico_test.shape[0],
        "Variables": X_estatico_test.shape[1],
        "Estudiantes únicos": grupos_test.nunique()
    }
])

display(dimensiones)

# ------------------------------------------------------------
# 7. TIPOS DE VARIABLES
# ------------------------------------------------------------

print("\n=== 3.2.3 TIPOS DE CARACTERÍSTICAS ===")

tipos = pd.DataFrame({
    "Variable": VARIABLES_ESTATICAS,
    "Tipo": [
        str(X_estatico_train[c].dtype)
        for c in VARIABLES_ESTATICAS
    ],
    "Tratamiento previsto": [
        "Numérica" if c in VARIABLES_NUMERICAS
        else "Categórica"
        for c in VARIABLES_ESTATICAS
    ]
})

display(tipos)

# ------------------------------------------------------------
# 8. NULOS EN ENTRENAMIENTO
# ------------------------------------------------------------

print("\n=== 3.2.4 VALORES FALTANTES EN TRAIN ===")

tabla_nulos = pd.DataFrame({
    "Variable": VARIABLES_ESTATICAS,
    "Nulos": [
        int(X_estatico_train[c].isna().sum())
        for c in VARIABLES_ESTATICAS
    ],
    "Porcentaje": [
        round(
            100 * X_estatico_train[c].isna().mean(),
            2
        )
        for c in VARIABLES_ESTATICAS
    ]
})

display(
    tabla_nulos.sort_values(
        "Nulos", ascending=False
    )
)

# ------------------------------------------------------------
# 9. VARIABLES NUMÉRICAS EN ENTRENAMIENTO
# ------------------------------------------------------------

print("\n=== 3.2.5 VARIABLES NUMÉRICAS EN TRAIN ===")

display(
    X_estatico_train[
        VARIABLES_NUMERICAS
    ].describe().T.round(3)
)

# ------------------------------------------------------------
# 10. CATEGORÍAS DISPONIBLES
# ------------------------------------------------------------

print("\n=== 3.2.6 VARIABLES CATEGÓRICAS EN TRAIN ===")

categorias = []

for columna in VARIABLES_CATEGORICAS:

    categorias.append({
        "Variable": columna,
        "Categorías observadas": (
            X_estatico_train[columna].nunique(
                dropna=True
            )
        ),
        "Categoría más frecuente": (
            X_estatico_train[columna]
            .mode(dropna=True).iloc[0]
        ),
        "Nulos": int(
            X_estatico_train[columna].isna().sum()
        )
    })

display(pd.DataFrame(categorias))

# ------------------------------------------------------------
# 11. BALANCE DE ETIQUETAS
# ------------------------------------------------------------

print("\n=== 3.2.7 DISTRIBUCIÓN DE ETIQUETAS ===")

balance = pd.DataFrame([
    {
        "Conjunto": "Entrenamiento",
        "Matrículas": len(y_estatico_train),
        "Favorables": int(
            (y_estatico_train == 0).sum()
        ),
        "Adversos": int(
            (y_estatico_train == 1).sum()
        ),
        "% adversos": round(
            100 * y_estatico_train.mean(), 2
        )
    },
    {
        "Conjunto": "Prueba",
        "Matrículas": len(y_estatico_test),
        "Favorables": int(
            (y_estatico_test == 0).sum()
        ),
        "Adversos": int(
            (y_estatico_test == 1).sum()
        ),
        "% adversos": round(
            100 * y_estatico_test.mean(), 2
        )
    }
])

display(balance)

# ------------------------------------------------------------
# 12. DEPENDENCIA ENTRE MATRÍCULAS
# ------------------------------------------------------------

print("\n=== 3.2.8 MATRÍCULAS POR ESTUDIANTE ===")

conteos = grupos_train.value_counts()

print(
    "Estudiantes con más de una matrícula en train:",
    int((conteos > 1).sum())
)

print(
    "Máximo de matrículas por estudiante:",
    int(conteos.max())
)

# ------------------------------------------------------------
# 13. COMPROBACIÓN FINAL
# ------------------------------------------------------------

print("\n=== ACTIVIDAD 3.2 FINALIZADA ===")

print("Semana:", SEMANA_BASE)
print("Características estáticas:", len(VARIABLES_ESTATICAS))
print("Características numéricas:", len(VARIABLES_NUMERICAS))
print(
    "Características categóricas:",
    len(VARIABLES_CATEGORICAS)
)

print("No hay estudiantes compartidos entre train y test.")
print("No se han ajustado transformaciones.")
print("No se han entrenado modelos.")
print("El conjunto de prueba permanece reservado.")



### 3.2.9. Resultados e interpretación

La construcción del conjunto estático de la semana 2 permitió obtener 27.962 matrículas elegibles, distribuidas en 22.412 matrículas de entrenamiento y 5.550 de prueba.

Se seleccionaron 12 características estáticas candidatas, correspondientes a cuatro variables numéricas y ocho categóricas.

El conjunto de entrenamiento contiene 20.121 estudiantes únicos y el de prueba 5.023, sin solapamiento de identificadores.

**Valores faltantes**

La única característica con valores faltantes fue `imd_band`, con 837 observaciones (3,73 %) en entrenamiento.

La ausencia de información socioeconómica se tratará posteriormente mediante una estrategia de imputación incorporada al pipeline, sin utilizar información del conjunto de prueba para ajustar el preprocesamiento.

**Variables numéricas**

Las variables presentaron diferencias de escala y distribución.

`num_of_prev_attempts` presentó una mediana de cero, mientras que `studied_credits` presentó una mediana de 60 y un máximo de 630 créditos.

Estos resultados justifican evaluar procedimientos de escalamiento y posibles transformaciones no lineales durante la optimización de los modelos.

**Variables categóricas**

Se identificaron ocho características categóricas, incluyendo módulo académico, presentación, región, nivel educativo, banda socioeconómica y características demográficas.

Su codificación se realizará dentro de pipelines reproducibles, utilizando procedimientos apropiados como One-Hot Encoding.

**Balance de etiquetas**

La proporción de resultados adversos fue del 45,15 % en entrenamiento y del 44,72 % en prueba, evidenciando distribuciones similares.

**Observaciones relacionadas**

Se identificaron 2.175 estudiantes con más de una matrícula en entrenamiento, con un máximo de cuatro matrículas por estudiante.

Este hallazgo confirma la necesidad de utilizar procedimientos de validación que respeten la agrupación por identificador de estudiante.

### 3.2.10. Conclusión

Se construyó correctamente el conjunto de características estáticas correspondiente a la semana 2, conservando la separación entre entrenamiento y prueba.

El conjunto contiene 12 características candidatas y constituye la referencia del Experimento A.

Las variables no han sido imputadas, codificadas, escaladas ni seleccionadas definitivamente.

**Conclusión principal:** el conjunto estático está preparado para integrarse posteriormente en pipelines supervisados y servirá como referencia para evaluar objetivamente el aporte incremental de la información longitudinal.



### 3.3. Ingeniería de características longitudinales — Semana 2

#### Objetivo

Construir características longitudinales a partir de la actividad virtual y las entregas académicas registradas durante los primeros catorce días del módulo, con el propósito de elaborar el conjunto de entrada del Experimento B.

#### Diseño de la observación temporal

Se utilizará el día 14 como momento de predicción. Las características longitudinales se calcularán exclusivamente con eventos correspondientes a los días 0 a 13.

Se distinguirán dos períodos:

- Semana 1: días 0 a 6.
- Semana 2: días 7 a 13.

#### Fuente 1: actividad en la plataforma virtual

A partir de los registros de `studentVle.csv`, previamente consolidados a nivel diario, se construirán características de:

- Volumen de clics durante cada semana.
- Número de días activos durante cada semana.
- Actividad total acumulada.
- Diferencias de actividad entre ambas semanas.
- Intensidad de actividad por día de exposición.
- Tiempo transcurrido desde la última interacción registrada.
- Semanas sin actividad VLE.

Estas características buscan representar tanto la intensidad de participación como su evolución temporal.

#### Fuente 2: entregas académicas

Utilizando `studentAssessment.csv` y `assessments.csv`, se calcularán:

- Entregas ordinarias acumuladas hasta el día 13.
- Entregas puntuales con fecha límite conocida.
- Entregas posteriores al plazo.
- Entregas sin fecha límite conocida.
- Evaluaciones del módulo cuyo plazo ya transcurrió.

Las evaluaciones reconocidas mediante `is_banked=1` se excluirán del conteo de nuevas entregas.

La ausencia de entregas no será interpretada automáticamente como incumplimiento.

#### Control de exposición temporal

Para cada matrícula se calculará el número de días observables desde su inscripción o desde el inicio del módulo, según corresponda.

Este indicador permitirá diferenciar a estudiantes matriculados desde el día cero de aquellos que ingresaron posteriormente.

#### Prevención de fuga de información

Las características utilizarán únicamente eventos anteriores al día 14.

El resultado académico final, las fechas futuras de retiro y las calificaciones cuya disponibilidad temprana no pueda verificarse quedarán excluidos de los predictores.

Las fechas de inscripción y desregistro se utilizarán exclusivamente para controlar la elegibilidad temporal.

#### Conjuntos experimentales

Se mantendrá el Experimento A con sus 12 características estáticas.

El Experimento B incorporará las nuevas características longitudinales a las mismas matrículas y particiones.

La comparación posterior utilizará procedimientos de validación equivalentes y transformaciones ajustadas únicamente sobre los datos de entrenamiento.

No se entrenarán modelos durante esta actividad.

**Fundamentación:** la dimensión temporal de los registros VLE y de evaluaciones se basa en la descripción del conjunto OULAD [1]. La ingeniería de características se realiza antes del corte temporal y sin incorporar la calificación final como predictor.


In [ ]:

# ============================================================
# PROYECTO FINAL DE MACHINE LEARNING
# ACTIVIDAD 3.3 - CARACTERÍSTICAS LONGITUDINALES
# SEMANA 2 (DÍAS 0-13)
# ============================================================

# Asistencia IA: ChatGPT, octubre de 2026.
# Prompt:
# "Construir características longitudinales de OULAD
# utilizando actividad VLE y entregas de los días 0-13.
# Comparar las mismas matrículas y particiones del
# experimento estático, evitar fuga temporal y no
# entrenar modelos ni ajustar transformaciones."

import pandas as pd
import numpy as np
from IPython.display import display

CORTE = 14

CLAVES = [
    "code_module",
    "code_presentation",
    "id_student"
]

CLAVES_CURSO = [
    "code_module",
    "code_presentation"
]

# ------------------------------------------------------------
# 1. VERIFICAR LA COHORTE
# ------------------------------------------------------------

assert cohorte_s2["dia_prediccion"].eq(CORTE).all()

assert len(cohorte_s2) == 27962

assert not cohorte_s2.duplicated(
    subset=CLAVES
).any()

print("=== 3.3.1 COHORTE ORIGINAL ===")
print("Matrículas elegibles:", len(cohorte_s2))
print("Día de predicción:", CORTE)
print("Último día observable:", CORTE - 1)

# ------------------------------------------------------------
# 2. SELECCIONAR EVENTOS VLE ANTERIORES AL CORTE
# ------------------------------------------------------------

vle = actividad_diaria.loc[
    actividad_diaria["date"].between(0, CORTE - 1)
].copy()

# Conservar solo las matrículas de nuestra cohorte.
vle = vle.merge(
    cohorte_s2[
        CLAVES +
        ["date_registration", "date_unregistration"]
    ],
    on=CLAVES,
    how="inner",
    validate="many_to_one"
)

# Controlar fechas de inscripción y retiro.
vle = vle.loc[
    (vle["date"] >= vle["date_registration"])
    &
    (
        vle["date_unregistration"].isna()
        |
        (vle["date"] < vle["date_unregistration"])
    )
].copy()

assert vle["date"].lt(CORTE).all()
assert vle["date"].ge(0).all()

# Identificar las dos semanas.
vle["semana_local"] = np.where(
    vle["date"] < 7, 1, 2
)

print("\n=== 3.3.2 ACTIVIDAD VLE DISPONIBLE ===")

print(
    "Observaciones estudiante-módulo-día:",
    len(vle)
)

print(
    "Matrículas con actividad:",
    vle[CLAVES].drop_duplicates().shape[0]
)

print(
    "Clics totales registrados:",
    int(vle["clics"].sum())
)

# ------------------------------------------------------------
# 3. AGREGACIÓN POR MATRÍCULA Y SEMANA
# ------------------------------------------------------------

vle_semanal = (
    vle.groupby(
        CLAVES + ["semana_local"],
        as_index=False
    )
    .agg(
        clics=("clics", "sum"),
        dias_activos=("date", "nunique")
    )
)

# Convertir semanas en columnas independientes.
vle_pivote = vle_semanal.pivot(
    index=CLAVES,
    columns="semana_local",
    values=["clics", "dias_activos"]
).fillna(0)

vle_pivote.columns = [
    f"vle_{metrica}_s{semana}"
    for metrica, semana in vle_pivote.columns
]

vle_pivote = vle_pivote.reset_index()

COLUMNAS_SEMANALES = [
    "vle_clics_s1",
    "vle_clics_s2",
    "vle_dias_activos_s1",
    "vle_dias_activos_s2"
]

for columna in COLUMNAS_SEMANALES:
    if columna not in vle_pivote.columns:
        vle_pivote[columna] = 0

# Último día con interacción registrada.
ultima_actividad = (
    vle.groupby(CLAVES, as_index=False)
    .agg(vle_ultimo_dia=("date", "max"))
)

vle_pivote = vle_pivote.merge(
    ultima_actividad,
    on=CLAVES,
    how="left",
    validate="one_to_one"
)

# ------------------------------------------------------------
# 4. BASE DE CARACTERÍSTICAS LONGITUDINALES
# ------------------------------------------------------------

caracteristicas_s2 = cohorte_s2[
    CLAVES + ["date_registration"]
].copy()

# Días observables dentro del período 0-13.
caracteristicas_s2["vle_dias_exposicion"] = (
    CORTE
    -
    caracteristicas_s2[
        "date_registration"
    ].clip(lower=0)
).clip(lower=0, upper=CORTE)

caracteristicas_s2 = caracteristicas_s2.merge(
    vle_pivote,
    on=CLAVES,
    how="left",
    validate="one_to_one"
)

for columna in COLUMNAS_SEMANALES:
    caracteristicas_s2[columna] = (
        caracteristicas_s2[columna]
        .fillna(0)
        .astype(int)
    )

# ------------------------------------------------------------
# 5. INDICADORES LONGITUDINALES VLE
# ------------------------------------------------------------

f = caracteristicas_s2

f["vle_clics_14d"] = (
    f["vle_clics_s1"] + f["vle_clics_s2"]
)

f["vle_dias_activos_14d"] = (
    f["vle_dias_activos_s1"]
    + f["vle_dias_activos_s2"]
)

f["vle_delta_clics"] = (
    f["vle_clics_s2"] - f["vle_clics_s1"]
)

f["vle_delta_dias_activos"] = (
    f["vle_dias_activos_s2"]
    - f["vle_dias_activos_s1"]
)

# Normalizar respecto de días realmente observables.
# Para exposición cero, se asigna tasa cero.
denominador = f[
    "vle_dias_exposicion"
].replace(0, np.nan)

f["vle_clics_por_dia_exposicion"] = (
    f["vle_clics_14d"] / denominador
).fillna(0)

# Días desde la última actividad anterior al corte.
# Si nunca hubo actividad, usar días de exposición.
f["vle_dias_sin_actividad_reciente"] = np.where(
    f["vle_ultimo_dia"].notna(),
    CORTE - 1 - f["vle_ultimo_dia"],
    f["vle_dias_exposicion"]
).astype(int)

f["vle_s1_sin_actividad"] = (
    f["vle_dias_activos_s1"] == 0
).astype(int)

f["vle_s2_sin_actividad"] = (
    f["vle_dias_activos_s2"] == 0
).astype(int)

# ------------------------------------------------------------
# 6. ENTREGAS DISPONIBLES ANTES DEL CORTE
# ------------------------------------------------------------

assessments = oulad_tablas["assessments.csv"]
student_assessment = (
    oulad_tablas["studentAssessment.csv"]
)

entregas = student_assessment.merge(
    assessments[
        [
            "id_assessment",
            "code_module",
            "code_presentation",
            "date"
        ]
    ],
    on="id_assessment",
    how="left",
    validate="many_to_one"
)

# Solamente entregas nuevas observadas en días 0-13.
entregas = entregas.loc[
    entregas["is_banked"].eq(0)
    & entregas["date_submitted"].ge(0)
    & entregas["date_submitted"].lt(CORTE)
].copy()

entregas = entregas.merge(
    cohorte_s2[
        CLAVES + ["date_registration"]
    ],
    on=CLAVES,
    how="inner",
    validate="many_to_one"
)

# Excluir entregas anteriores a la inscripción.
entregas = entregas.loc[
    entregas["date_submitted"]
    >= entregas["date_registration"]
].copy()

assert entregas["date_submitted"].lt(CORTE).all()

# No se utiliza score.
entregas["puntual"] = (
    entregas["date"].notna()
    & (
        entregas["date_submitted"]
        <= entregas["date"]
    )
)

entregas["tardia"] = (
    entregas["date"].notna()
    & (
        entregas["date_submitted"]
        > entregas["date"]
    )
)

entregas["sin_plazo"] = (
    entregas["date"].isna()
)

entregas_agg = (
    entregas.groupby(CLAVES, as_index=False)
    .agg(
        entregas_14d=("id_assessment", "size"),
        entregas_puntuales_14d=("puntual", "sum"),
        entregas_tardias_14d=("tardia", "sum"),
        entregas_sin_plazo_14d=("sin_plazo", "sum")
    )
)

# ------------------------------------------------------------
# 7. PLAZOS DE EVALUACIONES YA TRANSCURRIDOS
# ------------------------------------------------------------

plazos = assessments.loc[
    assessments["date"].notna()
    & assessments["date"].ge(0)
    & assessments["date"].lt(CORTE)
]

plazos_agg = (
    plazos.groupby(CLAVES_CURSO)
    .size()
    .reset_index(name="evaluaciones_vencidas_14d")
)

# Cantidad de plazos del módulo.
# No implica automáticamente obligación individual.

f = f.merge(
    entregas_agg,
    on=CLAVES,
    how="left",
    validate="one_to_one"
)

f = f.merge(
    plazos_agg,
    on=CLAVES_CURSO,
    how="left",
    validate="many_to_one"
)

COLUMNAS_ENTREGAS = [
    "entregas_14d",
    "entregas_puntuales_14d",
    "entregas_tardias_14d",
    "entregas_sin_plazo_14d",
    "evaluaciones_vencidas_14d"
]

f[COLUMNAS_ENTREGAS] = (
    f[COLUMNAS_ENTREGAS]
    .fillna(0)
    .astype(int)
)

caracteristicas_s2 = f

# ------------------------------------------------------------
# 8. DEFINIR CARACTERÍSTICAS LONGITUDINALES
# ------------------------------------------------------------

VARIABLES_LONGITUDINALES = [
    "vle_clics_s1",
    "vle_clics_s2",
    "vle_dias_activos_s1",
    "vle_dias_activos_s2",
    "vle_clics_14d",
    "vle_dias_activos_14d",
    "vle_delta_clics",
    "vle_delta_dias_activos",
    "vle_dias_exposicion",
    "vle_clics_por_dia_exposicion",
    "vle_dias_sin_actividad_reciente",
    "vle_s1_sin_actividad",
    "vle_s2_sin_actividad",
    "entregas_14d",
    "entregas_puntuales_14d",
    "entregas_tardias_14d",
    "entregas_sin_plazo_14d",
    "evaluaciones_vencidas_14d"
]

assert len(VARIABLES_LONGITUDINALES) == 18

assert not caracteristicas_s2.duplicated(
    subset=CLAVES
).any()

assert len(caracteristicas_s2) == len(cohorte_s2)

print("\n=== 3.3.3 CARACTERÍSTICAS GENERADAS ===")

display(pd.DataFrame({
    "Característica": VARIABLES_LONGITUDINALES
}))

print(
    "Características longitudinales:",
    len(VARIABLES_LONGITUDINALES)
)

# ------------------------------------------------------------
# 9. CONSTRUIR X_B SIN ALTERAR EL ORDEN DE LAS FILAS
# ------------------------------------------------------------

def crear_matriz_longitudinal(cohorte):

    # Verificar la relación usando las claves compuestas.
    datos_long = cohorte[CLAVES].merge(
        caracteristicas_s2[
            CLAVES + VARIABLES_LONGITUDINALES
        ],
        on=CLAVES,
        how="left",
        validate="one_to_one",
        indicator=True
    )

    assert datos_long["_merge"].eq("both").all()

    assert datos_long[CLAVES].reset_index(
        drop=True
    ).equals(
        cohorte[CLAVES].reset_index(drop=True)
    )

    # Mantener las características estáticas
    # originales y agregar las longitudinales.
    X = pd.concat(
        [
            cohorte[
                VARIABLES_ESTATICAS
            ].reset_index(drop=True),

            datos_long[
                VARIABLES_LONGITUDINALES
            ].reset_index(drop=True)
        ],
        axis=1
    )

    return X


X_longitudinal_train = crear_matriz_longitudinal(
    train_s2
)

X_longitudinal_test = crear_matriz_longitudinal(
    test_s2
)

# ------------------------------------------------------------
# 10. VALIDAR COMPARABILIDAD CON EXPERIMENTO A
# ------------------------------------------------------------

assert len(X_longitudinal_train) == len(
    X_estatico_train
)

assert len(X_longitudinal_test) == len(
    X_estatico_test
)

assert X_longitudinal_train[
    VARIABLES_ESTATICAS
].equals(X_estatico_train)

assert X_longitudinal_test[
    VARIABLES_ESTATICAS
].equals(X_estatico_test)

assert "id_student" not in (
    X_longitudinal_train.columns
)

assert all(
    columna not in X_longitudinal_train.columns
    for columna in [
        "final_result",
        "date_unregistration",
        "target_adverso",
        "dias_hasta_retiro",
        "score"
    ]
)

assert np.isfinite(
    X_longitudinal_train[
        VARIABLES_LONGITUDINALES
    ].to_numpy(dtype=float)
).all()

assert np.isfinite(
    X_longitudinal_test[
        VARIABLES_LONGITUDINALES
    ].to_numpy(dtype=float)
).all()

# ------------------------------------------------------------
# 11. DIMENSIONES DE LOS EXPERIMENTOS
# ------------------------------------------------------------

print("\n=== 3.3.4 COMPARACIÓN A VS B ===")

dimensiones = pd.DataFrame([
    {
        "Experimento": "A - Estático",
        "Conjunto": "Entrenamiento",
        "Matrículas": len(X_estatico_train),
        "Características": X_estatico_train.shape[1]
    },
    {
        "Experimento": "B - Longitudinal",
        "Conjunto": "Entrenamiento",
        "Matrículas": len(X_longitudinal_train),
        "Características": X_longitudinal_train.shape[1]
    },
    {
        "Experimento": "A - Estático",
        "Conjunto": "Prueba",
        "Matrículas": len(X_estatico_test),
        "Características": X_estatico_test.shape[1]
    },
    {
        "Experimento": "B - Longitudinal",
        "Conjunto": "Prueba",
        "Matrículas": len(X_longitudinal_test),
        "Características": X_longitudinal_test.shape[1]
    }
])

display(dimensiones)

# ------------------------------------------------------------
# 12. AUDITORÍA DESCRIPTIVA: SOLO ENTRENAMIENTO
# ------------------------------------------------------------

print("\n=== 3.3.5 CARACTERÍSTICAS EN TRAIN ===")

display(
    X_longitudinal_train[
        VARIABLES_LONGITUDINALES
    ].describe().T.round(3)
)

print("\n=== 3.3.6 COBERTURA EN TRAIN ===")

cobertura = pd.DataFrame([
    {
        "Indicador": "Con actividad VLE",
        "Matrículas": int((
            X_longitudinal_train["vle_clics_14d"] > 0
        ).sum())
    },
    {
        "Indicador": "Sin actividad VLE",
        "Matrículas": int((
            X_longitudinal_train["vle_clics_14d"] == 0
        ).sum())
    },
    {
        "Indicador": "Con entregas",
        "Matrículas": int((
            X_longitudinal_train["entregas_14d"] > 0
        ).sum())
    },
    {
        "Indicador": "Con plazos vencidos",
        "Matrículas": int((
            X_longitudinal_train[
                "evaluaciones_vencidas_14d"
            ] > 0
        ).sum())
    },
    {
        "Indicador": "Inscritos desde el inicio",
        "Matrículas": int((
            X_longitudinal_train[
                "vle_dias_exposicion"
            ] == 14
        ).sum())
    },
    {
        "Indicador": "Inscritos después del inicio",
        "Matrículas": int((
            X_longitudinal_train[
                "vle_dias_exposicion"
            ] < 14
        ).sum())
    }
])

cobertura["Porcentaje"] = (
    100
    * cobertura["Matrículas"]
    / len(X_longitudinal_train)
).round(2)

display(cobertura)

# ------------------------------------------------------------
# 13. VERIFICACIÓN FINAL
# ------------------------------------------------------------

print("\n=== 3.3.7 CONTROLES FINALES ===")

print(
    "Máxima fecha VLE usada:",
    int(vle["date"].max())
)

if len(entregas):
    print(
        "Máxima fecha de entrega usada:",
        int(entregas["date_submitted"].max())
    )

print(
    "El resultado final no se utilizó "
    "para generar características."
)

print(
    "No se utilizaron calificaciones "
    "como variables predictivas."
)

print(
    "Mismas matrículas A y B:",
    len(X_estatico_train)
    == len(X_longitudinal_train)
)

print(
    "Total variables A:",
    X_estatico_train.shape[1]
)

print(
    "Total variables B:",
    X_longitudinal_train.shape[1]
)

print("\nACTIVIDAD 3.3 FINALIZADA")
print("No se han entrenado modelos.")
print("No se han ajustado escaladores ni imputadores.")
print("El conjunto de prueba permanece reservado.")



### 3.3.8. Resultados e interpretación

La ingeniería de características longitudinales de la semana 2 permitió construir 18 variables adicionales a partir de los registros VLE y las entregas académicas disponibles durante los días 0 a 13.

El Experimento A conserva 12 características estáticas, mientras que el Experimento B contiene 30 características: 12 estáticas y 18 longitudinales.

Ambos experimentos mantienen 22.412 matrículas de entrenamiento y 5.550 de prueba, con idénticas etiquetas y asignaciones de estudiantes.

**Actividad virtual**

En entrenamiento, el 90,59 % de las matrículas presentó actividad VLE durante las primeras dos semanas. La mediana de clics acumulados fue de 74 y la mediana de días activos fue de 4.

Los clics acumulados presentaron una media de 133,225 y una desviación estándar de 181,588, evidenciando una distribución dispersa con valores extremos.

La diferencia semanal de clics presentó una mediana de cero, aunque se observaron variaciones individuales importantes.

**Entregas académicas**

El 10,33 % de las matrículas de entrenamiento registró al menos una entrega antes del día 14, mientras que solamente el 4,63 % pertenecía a módulos con evaluaciones cuya fecha límite ya había transcurrido.

La característica `entregas_sin_plazo_14d` presentó varianza nula, por lo que no aporta capacidad discriminativa en esta ventana y será considerada durante la selección de características.

**Elegibilidad temporal**

El 99,40 % de las matrículas de entrenamiento estuvo inscrito desde el inicio del módulo. Para las inscripciones posteriores se incorporó una variable de exposición temporal.

Se verificó que la fecha máxima de actividad VLE y de entrega utilizada fue el día 13.

No se incorporaron calificaciones, resultados finales ni fechas futuras de desregistro como características predictivas.

### 3.3.9. Conclusión

Se construyeron conjuntos estáticos y longitudinales comparables, manteniendo las mismas matrículas, etiquetas y particiones de entrenamiento y prueba.

La información longitudinal presenta cobertura suficiente y variabilidad durante las primeras dos semanas para justificar su evaluación predictiva.

Sin embargo, la presencia de señales de comportamiento no demuestra que estas permitan mejorar la clasificación.

**Conclusión principal:** el Experimento B incorpora información temporal adicional al Experimento A bajo condiciones comparables y con controles explícitos de fuga de información. El aporte incremental de estas características deberá determinarse mediante entrenamiento, validación cruzada y pruebas estadísticas.



### 3.4. Generalización de las características longitudinales

#### Objetivo

Extender la ingeniería de características longitudinales a las semanas 4, 6 y 8, utilizando un procedimiento reproducible y temporalmente consistente.

#### Estrategia

Se desarrollará una función parametrizada que reciba la semana de predicción y construya las características a partir de la información disponible antes del día de corte.

Se utilizarán las siguientes ventanas:

| Semana | Día de corte | Último día observable |
|---|---:|---:|
| 2 | 14 | 13 |
| 4 | 28 | 27 |
| 6 | 42 | 41 |
| 8 | 56 | 55 |

#### Características longitudinales

Para cada ventana se generarán:

- Clics registrados durante cada semana.
- Días activos durante cada semana.
- Indicadores de semanas sin actividad.
- Clics y días activos acumulados.
- Variación entre las dos semanas más recientes.
- Días de exposición observables.
- Clics acumulados por día de exposición.
- Días desde la última interacción VLE.
- Entregas acumuladas, puntuales y tardías.
- Evaluaciones con fechas límite anteriores al corte.

Las características utilizarán exclusivamente eventos anteriores al día de predicción.

#### Consistencia experimental

Se mantendrán los identificadores, etiquetas y particiones previamente establecidos.

Las características longitudinales se incorporarán a las 12 variables estáticas del Experimento A.

Se verificará que la función reproduzca las características generadas anteriormente para la semana 2, evitando diferencias introducidas por la generalización del código.

Cada ventana tendrá su propio conjunto de entrenamiento y prueba. El conjunto de prueba permanecerá reservado para la evaluación final.

#### Relación con el Transformer

Las variables semanales individuales permitirán preparar posteriormente secuencias cronológicas para el Experimento C.

El Transformer utilizará el orden temporal explícito, mientras que los modelos clásicos utilizarán las características tabulares diseñadas.

No se entrenarán modelos ni se ajustarán transformaciones durante esta actividad.


In [ ]:

# ============================================================
# PROYECTO FINAL DE MACHINE LEARNING
# ACTIVIDAD 3.4 - GENERALIZACIÓN LONGITUDINAL
# SEMANAS 2, 4, 6 Y 8
# ============================================================

# Asistencia IA: ChatGPT, octubre de 2026.
# Prompt:
# "Generalizar las características longitudinales de
# OULAD a los días 14, 28, 42 y 56. Mantener las
# cohortes y particiones fijas, verificar que semana 2
# coincida con la versión anterior y evitar eventos
# posteriores a cada corte. No entrenar modelos."

CLAVES = [
    "code_module", "code_presentation", "id_student"
]

CLAVES_CURSO = [
    "code_module", "code_presentation"
]

CORTES = {2: 14, 4: 28, 6: 42, 8: 56}

# ------------------------------------------------------------
# 1. PREPARAR FUENTE DE ENTREGAS
# ------------------------------------------------------------

assessments = oulad_tablas["assessments.csv"]
student_assessment = oulad_tablas["studentAssessment.csv"]

entregas_base = student_assessment.merge(
    assessments[
        ["id_assessment"] + CLAVES_CURSO + ["date"]
    ],
    on="id_assessment",
    how="left",
    validate="many_to_one"
)

entregas_base = entregas_base.loc[
    (entregas_base["is_banked"] == 0)
    & (entregas_base["date_submitted"] >= 0)
].copy()

# ------------------------------------------------------------
# 2. FUNCIÓN GENERAL
# ------------------------------------------------------------

def construir_ventana_longitudinal(semana):

    corte = CORTES[semana]

    cohorte = cohortes_oulad.loc[
        cohortes_oulad["semana_prediccion"] == semana
    ].copy().reset_index(drop=True)

    assert cohorte["dia_prediccion"].eq(corte).all()

    assert not cohorte.duplicated(
        subset=CLAVES
    ).any()

    # --------------------------------------------------------
    # A. ACTIVIDAD VLE HASTA EL CORTE
    # --------------------------------------------------------

    vle_corte = actividad_diaria.loc[
        actividad_diaria["date"].between(0, corte - 1)
    ].copy()

    vle_corte = vle_corte.merge(
        cohorte[
            CLAVES + [
                "date_registration",
                "date_unregistration"
            ]
        ],
        on=CLAVES,
        how="inner",
        validate="many_to_one"
    )

    vle_corte = vle_corte.loc[
        (vle_corte["date"] >=
         vle_corte["date_registration"])
        &
        (
            vle_corte["date_unregistration"].isna()
            |
            (vle_corte["date"] <
             vle_corte["date_unregistration"])
        )
    ].copy()

    assert vle_corte["date"].lt(corte).all()

    vle_corte["semana_local"] = (
        vle_corte["date"] // 7 + 1
    ).astype(int)

    semanal = (
        vle_corte.groupby(
            CLAVES + ["semana_local"],
            as_index=False
        )
        .agg(
            clics=("clics", "sum"),
            dias_activos=("date", "nunique")
        )
    )

    wide = semanal.pivot(
        index=CLAVES,
        columns="semana_local",
        values=["clics", "dias_activos"]
    )

    wide.columns = [
        f"vle_{metrica}_s{num_semana}"
        for metrica, num_semana in wide.columns
    ]

    wide = wide.reset_index()

    # Una columna para cada semana observada.
    columnas_clics = [
        f"vle_clics_s{i}"
        for i in range(1, semana + 1)
    ]

    columnas_dias = [
        f"vle_dias_activos_s{i}"
        for i in range(1, semana + 1)
    ]

    columnas_inactividad = [
        f"vle_s{i}_sin_actividad"
        for i in range(1, semana + 1)
    ]

    # Recuperar la fecha de la última actividad.
    ultimo_dia = (
        vle_corte.groupby(
            CLAVES, as_index=False
        )
        .agg(
            vle_ultimo_dia=("date", "max")
        )
    )

    f = cohorte[
        CLAVES + ["date_registration"]
    ].copy()

    f = f.merge(
        wide,
        on=CLAVES,
        how="left",
        validate="one_to_one"
    )

    for columna in columnas_clics + columnas_dias:
        if columna not in f.columns:
            f[columna] = 0

        f[columna] = (
            f[columna].fillna(0).astype(int)
        )

    f = f.merge(
        ultimo_dia,
        on=CLAVES,
        how="left",
        validate="one_to_one"
    )

    # --------------------------------------------------------
    # B. INDICADORES LONGITUDINALES VLE
    # --------------------------------------------------------

    col_clics_total = f"vle_clics_{corte}d"
    col_dias_total = f"vle_dias_activos_{corte}d"

    f[col_clics_total] = f[columnas_clics].sum(axis=1)

    f[col_dias_total] = f[columnas_dias].sum(axis=1)

    f["vle_delta_clics"] = (
        f[columnas_clics[-1]]
        - f[columnas_clics[-2]]
    )

    f["vle_delta_dias_activos"] = (
        f[columnas_dias[-1]]
        - f[columnas_dias[-2]]
    )

    f["vle_dias_exposicion"] = (
        corte
        - f["date_registration"].clip(lower=0)
    ).clip(lower=0, upper=corte)

    exposicion = (
        f["vle_dias_exposicion"]
        .replace(0, np.nan)
    )

    f["vle_clics_por_dia_exposicion"] = (
        f[col_clics_total] / exposicion
    ).fillna(0)

    f["vle_dias_sin_actividad_reciente"] = (
        np.where(
            f["vle_ultimo_dia"].notna(),
            corte - 1 - f["vle_ultimo_dia"],
            f["vle_dias_exposicion"]
        )
    ).astype(int)

    for i in range(1, semana + 1):
        f[f"vle_s{i}_sin_actividad"] = (
            f[f"vle_dias_activos_s{i}"] == 0
        ).astype(int)

    # --------------------------------------------------------
    # C. ENTREGAS ANTERIORES AL CORTE
    # --------------------------------------------------------

    e = entregas_base.loc[
        entregas_base["date_submitted"] < corte
    ].copy()

    e = e.merge(
        cohorte[
            CLAVES + [
                "date_registration",
                "date_unregistration"
            ]
        ],
        on=CLAVES,
        how="inner",
        validate="many_to_one"
    )

    e = e.loc[
        (e["date_submitted"] >=
         e["date_registration"])
        &
        (
            e["date_unregistration"].isna()
            |
            (e["date_submitted"] <
             e["date_unregistration"])
        )
    ].copy()

    assert e["date_submitted"].lt(corte).all()

    e["puntual"] = (
        e["date"].notna()
        & (e["date_submitted"] <= e["date"])
    )

    e["tardia"] = (
        e["date"].notna()
        & (e["date_submitted"] > e["date"])
    )

    e["sin_plazo"] = e["date"].isna()

    # Nombres compatibles con actividad 3.3.
    col_entregas = f"entregas_{corte}d"
    col_puntuales = f"entregas_puntuales_{corte}d"
    col_tardias = f"entregas_tardias_{corte}d"
    col_sin_plazo = f"entregas_sin_plazo_{corte}d"
    col_vencidas = f"evaluaciones_vencidas_{corte}d"

    e_agg = (
        e.groupby(CLAVES, as_index=False)
        .agg(
            **{
                col_entregas: ("id_assessment", "size"),
                col_puntuales: ("puntual", "sum"),
                col_tardias: ("tardia", "sum"),
                col_sin_plazo: ("sin_plazo", "sum")
            }
        )
    )

    # Calendario de fechas límite conocidas.
    plazos = assessments.loc[
        assessments["date"].notna()
        & assessments["date"].ge(0)
        & assessments["date"].lt(corte)
    ]

    plazos_agg = (
        plazos.groupby(CLAVES_CURSO)
        .size()
        .reset_index(name=col_vencidas)
    )

    f = f.merge(
        e_agg,
        on=CLAVES,
        how="left",
        validate="one_to_one"
    )

    f = f.merge(
        plazos_agg,
        on=CLAVES_CURSO,
        how="left",
        validate="many_to_one"
    )

    columnas_entregas = [
        col_entregas,
        col_puntuales,
        col_tardias,
        col_sin_plazo,
        col_vencidas
    ]

    f[columnas_entregas] = (
        f[columnas_entregas]
        .fillna(0)
        .astype(int)
    )

    # --------------------------------------------------------
    # D. DEFINIR EL CONJUNTO DE CARACTERÍSTICAS
    # --------------------------------------------------------

    columnas_totales = [
        col_clics_total,
        col_dias_total,
        "vle_delta_clics",
        "vle_delta_dias_activos",
        "vle_dias_exposicion",
        "vle_clics_por_dia_exposicion",
        "vle_dias_sin_actividad_reciente"
    ]

    longitudinales = (
        columnas_clics
        + columnas_dias
        + columnas_totales
        + columnas_inactividad
        + columnas_entregas
    )

    assert not f.duplicated(subset=CLAVES).any()
    assert len(f) == len(cohorte)

    assert f[CLAVES].equals(cohorte[CLAVES])

    assert np.isfinite(
        f[longitudinales].to_numpy(dtype=float)
    ).all()

    # --------------------------------------------------------
    # E. SEPARAR MATRICES A Y B
    # --------------------------------------------------------

    resultados = {}

    for conjunto, sufijo in [
        ("Entrenamiento", "train"),
        ("Prueba", "test")
    ]:
        mascara = cohorte["conjunto"].eq(conjunto)

        coh = cohorte.loc[
            mascara
        ].reset_index(drop=True)

        car = f.loc[
            mascara, longitudinales
        ].reset_index(drop=True)

        XA = coh[
            VARIABLES_ESTATICAS
        ].copy()

        XB = pd.concat(
            [XA.reset_index(drop=True), car],
            axis=1
        )

        resultados[f"X_A_{sufijo}"] = XA
        resultados[f"X_B_{sufijo}"] = XB

        resultados[f"y_{sufijo}"] = (
            coh["target_adverso"].astype(int)
        )

        resultados[f"grupos_{sufijo}"] = (
            coh["id_student"].copy()
        )

        resultados[f"claves_{sufijo}"] = (
            coh[CLAVES].copy()
        )

    assert set(
        resultados["grupos_train"]
    ).isdisjoint(
        set(resultados["grupos_test"])
    )

    resultados["features_longitudinales"] = longitudinales
    resultados["caracteristicas"] = f

    return resultados


# ------------------------------------------------------------
# 3. CONSTRUIR LAS CUATRO VENTANAS
# ------------------------------------------------------------

experimentos_oulad = {}

for semana in [2, 4, 6, 8]:

    experimentos_oulad[semana] = (
        construir_ventana_longitudinal(semana)
    )

    print(
        f"Semana {semana}: "
        f"{experimentos_oulad[semana]['X_B_train'].shape[1]} "
        "características totales."
    )

# ------------------------------------------------------------
# 4. VERIFICAR REPRODUCCIÓN DE SEMANA 2
# ------------------------------------------------------------

print("\n=== 3.4.1 COMPROBACIÓN DE SEMANA 2 ===")

columnas_originales = VARIABLES_LONGITUDINALES

anterior = (
    caracteristicas_s2
    .set_index(CLAVES)[columnas_originales]
    .sort_index()
)

actual = (
    experimentos_oulad[2]["caracteristicas"]
    .set_index(CLAVES)[columnas_originales]
    .sort_index()
)

assert anterior.index.equals(actual.index)

iguales = np.allclose(
    anterior.to_numpy(dtype=float),
    actual.to_numpy(dtype=float),
    rtol=1e-10,
    atol=1e-10,
    equal_nan=True
)

assert iguales, (
    "La función general no reproduce "
    "las características de la semana 2."
)

print("Características de semana 2 idénticas:", iguales)

# ------------------------------------------------------------
# 5. RESUMEN DE LAS CUATRO VENTANAS
# ------------------------------------------------------------

print("\n=== 3.4.2 DIMENSIONES POR EXPERIMENTO ===")

resumen = []

for semana, datos in experimentos_oulad.items():

    resumen.append({
        "Semana": semana,
        "Matrículas train": len(datos["X_A_train"]),
        "Matrículas test": len(datos["X_A_test"]),
        "Variables A": datos["X_A_train"].shape[1],
        "Variables B": datos["X_B_train"].shape[1],
        "Variables longitudinales": len(
            datos["features_longitudinales"]
        )
    })

tabla_experimentos = pd.DataFrame(resumen)

display(tabla_experimentos)

# ------------------------------------------------------------
# 6. AUDITORÍA DE COBERTURA SOLO EN ENTRENAMIENTO
# ------------------------------------------------------------

print("\n=== 3.4.3 COBERTURA LONGITUDINAL EN TRAIN ===")

cobertura = []

for semana, datos in experimentos_oulad.items():

    X = datos["X_B_train"]
    corte = CORTES[semana]

    tiene_vle = X[f"vle_clics_{corte}d"] > 0
    tiene_entregas = X[f"entregas_{corte}d"] > 0

    cobertura.append({
        "Semana": semana,
        "Matrículas": len(X),
        "% con VLE": round(
            100 * tiene_vle.mean(), 2
        ),
        "% con entregas": round(
            100 * tiene_entregas.mean(), 2
        ),
        "% con ambas": round(
            100 * (tiene_vle & tiene_entregas).mean(), 2
        ),
        "Mediana clics": round(
            X[f"vle_clics_{corte}d"].median(), 2
        ),
        "Mediana entregas": round(
            X[f"entregas_{corte}d"].median(), 2
        )
    })

tabla_cobertura_train = pd.DataFrame(cobertura)

display(tabla_cobertura_train)

# ------------------------------------------------------------
# 7. COMPROBAR VARIABLES CONSTANTES EN TRAIN
# ------------------------------------------------------------

print("\n=== 3.4.4 VARIABLES CONSTANTES EN TRAIN ===")

constantes = []

for semana, datos in experimentos_oulad.items():

    X = datos["X_B_train"]

    for columna in datos["features_longitudinales"]:

        if X[columna].nunique(dropna=False) <= 1:
            constantes.append({
                "Semana": semana,
                "Variable": columna
            })

if constantes:
    display(pd.DataFrame(constantes))
else:
    print("No se encontraron variables constantes.")

# ------------------------------------------------------------
# 8. CONTROLES FINALES
# ------------------------------------------------------------

print("\n=== 3.4.5 CONTROLES FINALES ===")

for semana, datos in experimentos_oulad.items():

    XA = datos["X_A_train"]
    XB = datos["X_B_train"]

    assert len(XA) == len(XB)

    assert XA.equals(
        XB[VARIABLES_ESTATICAS]
    )

    assert not set(
        ["final_result", "date_unregistration",
         "target_adverso", "score", "id_student"]
    ).intersection(XB.columns)

    print(
        f"Semana {semana}: "
        "matrices comparables y controles superados."
    )

print("\nACTIVIDAD 3.4 FINALIZADA")
print("No se han entrenado modelos.")
print("No se ha realizado selección de características.")
print("No se han ajustado transformaciones.")
print("El conjunto de prueba permanece reservado.")


In [ ]:
# SNAPSHOT 3.4 - MATRICES ORIGINALES DE LOS EXPERIMENTOS
assert set(experimentos_oulad)=={2,4,6,8}
assert experimentos_oulad[8]['X_B_train'].shape==(21164,48)
guardar_estado_oulad('experimentos_oulad')
print('[Drive] Las matrices A/B originales ya están persistentes para t-SNE.')


In [ ]:

# ==================================================
# VERIFICACIÓN DEL SNAPSHOT ORIGINAL DE OULAD
# ==================================================

print("=== ESTADO DE GOOGLE DRIVE ===")

print(
    "Dataset:",
    dataset_en_drive_oulad()
)

print(
    "Cohortes:",
    snapshot_disponible_oulad("cohortes_oulad")
)

print(
    "Matrices:",
    snapshot_disponible_oulad("experimentos_oulad")
)

print("\n=== MATRICES DE ENTRENAMIENTO ===")

assert "experimentos_oulad" in globals()

esperadas = {
    2: (22412, 5550, 12, 30),
    4: (21984, 5438, 12, 36),
    6: (21568, 5344, 12, 42),
    8: (21164, 5249, 12, 48)
}

for semana, valores in esperadas.items():

    d = experimentos_oulad[semana]

    train, test, n_a, n_b = valores

    assert d["X_A_train"].shape == (train, n_a)
    assert d["X_B_train"].shape == (train, n_b)

    assert d["X_A_test"].shape == (test, n_a)
    assert d["X_B_test"].shape == (test, n_b)

    print(
        f"Semana {semana}: "
        f"A={d['X_A_train'].shape}, "
        f"B={d['X_B_train'].shape} — OK"
    )

assert snapshot_disponible_oulad(
    "experimentos_oulad"
)

print("\nSNAPSHOT ORIGINAL: GUARDADO EN DRIVE")
print("No se entrenaron modelos.")
print("No se modificaron resultados del test.")



### 3.4.6. Resultados e interpretación

La generalización de las características longitudinales permitió construir conjuntos de datos correspondientes a las semanas 2, 4, 6 y 8, utilizando un procedimiento reproducible y temporalmente consistente.

Las dimensiones obtenidas fueron:

| Semana | Matrículas entrenamiento | Matrículas prueba | Variables estáticas | Variables totales longitudinales |
|---|---:|---:|---:|---:|
| 2 | 22.412 | 5.550 | 12 | 30 |
| 4 | 21.984 | 5.438 | 12 | 36 |
| 6 | 21.568 | 5.344 | 12 | 42 |
| 8 | 21.164 | 5.249 | 12 | 48 |

La función generalizada reprodujo exactamente las características previamente construidas para la semana 2.

Se verificó que los experimentos A y B conservan las mismas matrículas y características estáticas dentro de cada ventana.

**Cobertura longitudinal**

La proporción de matrículas de entrenamiento con actividad VLE acumulada aumentó del 90,59 % en la semana 2 al 97,21 % en la semana 8.

La cobertura de entregas académicas aumentó del 10,33 % al 85,95 % durante el mismo período.

La mediana de clics acumulados aumentó de 74 a 304 y la mediana de entregas registradas pasó de 0 a 2.

Estos incrementos representan una mayor cantidad de información disponible conforme avanza el curso y no constituyen evidencia directa de una mejora predictiva.

**Selección de características**

La característica `entregas_sin_plazo` presentó varianza cero en las cuatro ventanas, por lo que deberá considerarse dentro de un procedimiento de selección tipo filtro.

También se identificaron características relacionadas matemáticamente, como los clics acumulados y los clics registrados individualmente en cada semana.

Estas relaciones justifican utilizar técnicas de selección de características dentro de los pipelines, evitando ajustar procedimientos de selección antes de la validación cruzada.

### 3.4.7. Conclusión

Se construyeron correctamente los conjuntos de características correspondientes a las cuatro ventanas de predicción temprana.

Las matrices de los experimentos A y B son comparables dentro de cada semana y mantienen la separación previamente definida por estudiante.

La mayor disponibilidad de registros longitudinales permite evaluar el comportamiento estudiantil con distintos niveles de información acumulada.

**Conclusión principal:** el proyecto dispone de ocho configuraciones tabulares comparables (dos conjuntos de características por cuatro ventanas temporales), preparadas para evaluar el aporte incremental de la información longitudinal mediante validación cruzada y comparación estadística.

La superioridad predictiva del Experimento B continúa siendo una hipótesis pendiente de contrastación.



### 3.5. Diseño de la validación cruzada repetida y agrupada

#### Objetivo

Establecer una estrategia de validación cruzada reproducible que permita optimizar hiperparámetros y comparar modelos de Machine Learning, evitando que matrículas del mismo estudiante aparezcan simultáneamente en entrenamiento y validación.

#### Justificación metodológica

La rúbrica exige utilizar validación cruzada repetida para la optimización de hiperparámetros y la comparación estadística de técnicas de Machine Learning.

El dataset OULAD presenta una estructura en la que algunos estudiantes participan en múltiples matrículas. Esto genera observaciones relacionadas que no deben distribuirse libremente entre los conjuntos de entrenamiento y validación.

Por esta razón se utilizará `StratifiedGroupKFold`, ejecutado con diferentes semillas aleatorias para construir repeticiones de la validación cruzada.

Esta estrategia combina:

- Estratificación aproximada de la variable objetivo.
- Agrupación por identificador de estudiante.
- Separación entre entrenamiento y validación.
- Reproducibilidad mediante semillas fijas.
- Particiones equivalentes para todos los modelos comparados.

#### Configuración inicial

Se utilizarán cinco folds y dos repeticiones, obteniendo diez evaluaciones por configuración.

La elección responde a la alternativa de menor costo computacional permitida por la rúbrica, considerando que el proyecto evaluará cuatro ventanas temporales, dos conjuntos de características y al menos dos algoritmos supervisados.

#### Métrica principal

Se propone utilizar F1-score macro como métrica principal de selección de modelos.

Esta métrica calcula el promedio no ponderado del F1-score de ambas clases, evitando que el rendimiento sobre la clase mayoritaria domine la evaluación.

Como métricas complementarias se utilizarán accuracy, precision, recall, ROC-AUC y F1-score weighted, además de las métricas de curvas ROC requeridas por la rúbrica.

Se prestará especial atención al recall de la clase adversa, debido a la finalidad del proyecto de identificar estudiantes en riesgo académico.

#### Comparaciones experimentales

La evaluación contemplará:

1. Regresión logística con características estáticas y longitudinales.
2. Random Forest con características estáticas y longitudinales.
3. Comparación de ambos algoritmos bajo el mismo conjunto de características.
4. Comparación del aporte longitudinal manteniendo constante el algoritmo.

Los modelos utilizarán las mismas particiones de validación dentro de cada semana.

#### Prevención de fuga de información

La validación cruzada operará exclusivamente sobre el conjunto de entrenamiento previamente definido.

Las transformaciones, imputaciones, escalamiento, selección de características y optimización de hiperparámetros se ajustarán dentro de cada fold.

El conjunto de prueba independiente permanecerá reservado para la evaluación final.

#### Consideración estadística

Los resultados obtenidos mediante validación cruzada repetida presentan dependencia debido al solapamiento de muestras de entrenamiento.

Por tanto, cualquier prueba de hipótesis aplicada a los resultados por fold deberá interpretarse considerando esta limitación.

Además de los valores p, se reportarán las diferencias de rendimiento, su magnitud y medidas de incertidumbre.

Durante esta actividad se crearán y auditarán las particiones, sin entrenar modelos.

**Referencias metodológicas:** la validación cruzada y la estimación del rendimiento se fundamentan en [2], [10]. Para inferencia estadística sobre medidas de validación repetida, la dependencia entre particiones requiere las cautelas discutidas en [5].


In [ ]:

# ============================================================
# PROYECTO FINAL DE MACHINE LEARNING
# ACTIVIDAD 3.5 - VALIDACIÓN CRUZADA REPETIDA Y AGRUPADA
# ============================================================

# Asistencia IA: ChatGPT, octubre de 2026.
# Prompt:
# "Diseñar validación cruzada estratificada y agrupada
# por estudiante para OULAD, usando cinco folds y dos
# repeticiones. Garantizar separación de grupos,
# verificar proporciones de clases y crear particiones
# idénticas para los experimentos A y B.
# No entrenar modelos ni utilizar el conjunto de prueba."

import numpy as np
import pandas as pd
import sklearn

from sklearn.model_selection import StratifiedGroupKFold
from IPython.display import display

# ------------------------------------------------------------
# 1. CONFIGURACIÓN
# ------------------------------------------------------------

N_FOLDS = 5
N_REPETICIONES = 2
SEMILLA_CV = 42

SEMANAS = [2, 4, 6, 8]

cv_splits_oulad = {}
resultados_cv = []
resumen_repeticiones = []

print("=== 3.5.1 CONFIGURACIÓN ===")

print("Versión scikit-learn:", sklearn.__version__)
print("Folds:", N_FOLDS)
print("Repeticiones:", N_REPETICIONES)
print(
    "Evaluaciones por configuración:",
    N_FOLDS * N_REPETICIONES
)

# ------------------------------------------------------------
# 2. GENERAR FOLDS PARA CADA SEMANA
# ------------------------------------------------------------

for semana in SEMANAS:

    datos = experimentos_oulad[semana]

    # Exclusivamente conjunto de entrenamiento.
    X = datos["X_A_train"]
    y = datos["y_train"].to_numpy()
    grupos = datos["grupos_train"].to_numpy()

    assert len(X) == len(y) == len(grupos)
    assert set(np.unique(y)) == {0, 1}

    prevalencia_global = 100 * np.mean(y)

    splits_semana = []
    firmas_repeticiones = []

    for repeticion in range(N_REPETICIONES):

        splitter = StratifiedGroupKFold(
            n_splits=N_FOLDS,
            shuffle=True,
            random_state=SEMILLA_CV + repeticion
        )

        conteo_validacion = np.zeros(
            len(y), dtype=int
        )

        grupos_validacion_repeticion = []

        for fold, (idx_train, idx_val) in enumerate(
            splitter.split(
                X, y, groups=grupos
            ),
            start=1
        ):

            # --------------------------------------------
            # CONTROL A: índices distintos
            # --------------------------------------------

            assert len(
                np.intersect1d(idx_train, idx_val)
            ) == 0

            # --------------------------------------------
            # CONTROL B: estudiantes distintos
            # --------------------------------------------

            grupos_train_fold = set(
                grupos[idx_train]
            )

            grupos_val_fold = set(
                grupos[idx_val]
            )

            assert grupos_train_fold.isdisjoint(
                grupos_val_fold
            ), "Fuga de estudiantes entre folds."

            # --------------------------------------------
            # CONTROL C: ambas clases presentes
            # --------------------------------------------

            assert set(
                np.unique(y[idx_train])
            ) == {0, 1}

            assert set(
                np.unique(y[idx_val])
            ) == {0, 1}

            # --------------------------------------------
            # CONTROL D: contar validaciones
            # --------------------------------------------

            conteo_validacion[idx_val] += 1

            grupos_validacion_repeticion.append(
                frozenset(grupos_val_fold)
            )

            # Conservar índices para reutilizarlos.
            splits_semana.append(
                (idx_train.copy(), idx_val.copy())
            )

            resultados_cv.append({
                "Semana": semana,
                "Repetición": repeticion + 1,
                "Fold": fold,
                "Matrículas train": len(idx_train),
                "Matrículas validación": len(idx_val),
                "Estudiantes train": len(
                    grupos_train_fold
                ),
                "Estudiantes validación": len(
                    grupos_val_fold
                ),
                "% adversos train": round(
                    100 * np.mean(y[idx_train]), 2
                ),
                "% adversos validación": round(
                    100 * np.mean(y[idx_val]), 2
                ),
                "Diferencia vs cohorte (pp)": round(
                    100 * np.mean(y[idx_val])
                    - prevalencia_global, 2
                ),
                "Grupos compartidos": len(
                    grupos_train_fold
                    & grupos_val_fold
                )
            })

        # Cada matrícula debe utilizarse una vez
        # para validación durante cada repetición.
        assert np.all(
            conteo_validacion == 1
        ), (
            f"Semana {semana}: una matrícula no "
            "participó exactamente una vez "
            "en validación."
        )

        firmas_repeticiones.append(
            frozenset(
                grupos_validacion_repeticion
            )
        )

    assert len(splits_semana) == (
        N_FOLDS * N_REPETICIONES
    )

    cv_splits_oulad[semana] = splits_semana

    # Verificar que ambas repeticiones no generaron
    # exactamente la misma partición de grupos.
    repeticiones_distintas = (
        len(set(firmas_repeticiones))
        == N_REPETICIONES
    )

    resumen_repeticiones.append({
        "Semana": semana,
        "Folds totales": len(splits_semana),
        "Repeticiones distintas": (
            repeticiones_distintas
        ),
        "Matrículas entrenamiento": len(y),
        "% adversos cohorte": round(
            prevalencia_global, 2
        )
    })

# ------------------------------------------------------------
# 3. RESUMEN DE REPRODUCIBILIDAD
# ------------------------------------------------------------

print("\n=== 3.5.2 RESUMEN DE PARTICIONES ===")

tabla_resumen_cv = pd.DataFrame(
    resumen_repeticiones
)

display(tabla_resumen_cv)

# ------------------------------------------------------------
# 4. BALANCE DE LOS FOLDS
# ------------------------------------------------------------

print("\n=== 3.5.3 AUDITORÍA DE FOLDS ===")

tabla_folds_cv = pd.DataFrame(
    resultados_cv
)

display(tabla_folds_cv)

# ------------------------------------------------------------
# 5. DESVIACIONES DE ESTRATIFICACIÓN
# ------------------------------------------------------------

print("\n=== 3.5.4 BALANCE POR SEMANA ===")

balance_cv = (
    tabla_folds_cv.groupby("Semana")
    .agg(
        Porcentaje_minimo=(
            "% adversos validación", "min"
        ),
        Porcentaje_maximo=(
            "% adversos validación", "max"
        ),
        Diferencia_absoluta_maxima=(
            "Diferencia vs cohorte (pp)",
            lambda x: x.abs().max()
        ),
        Matrículas_validacion_min=(
            "Matrículas validación", "min"
        ),
        Matrículas_validacion_max=(
            "Matrículas validación", "max"
        ),
        Grupos_compartidos_max=(
            "Grupos compartidos", "max"
        )
    )
    .reset_index()
)

display(balance_cv)

# ------------------------------------------------------------
# 6. VERIFICAR COMPARABILIDAD ENTRE A Y B
# ------------------------------------------------------------

print("\n=== 3.5.5 COMPARABILIDAD A VS B ===")

for semana in SEMANAS:

    datos = experimentos_oulad[semana]

    XA = datos["X_A_train"]
    XB = datos["X_B_train"]

    y = datos["y_train"]
    grupos = datos["grupos_train"]

    assert len(XA) == len(XB) == len(y) == len(grupos)

    for idx_train, idx_val in cv_splits_oulad[semana]:

        assert idx_train.max() < len(XA)
        assert idx_val.max() < len(XB)

        assert set(
            grupos.iloc[idx_train]
        ).isdisjoint(
            set(grupos.iloc[idx_val])
        )

    print(
        f"Semana {semana}: "
        f"{len(cv_splits_oulad[semana])} folds "
        "válidos para A y B."
    )

# ------------------------------------------------------------
# 7. ESTADO FINAL
# ------------------------------------------------------------

print("\n=== ACTIVIDAD 3.5 FINALIZADA ===")

print(
    "Total de particiones creadas:",
    sum(
        len(s)
        for s in cv_splits_oulad.values()
    )
)

print(
    "Máximo de estudiantes compartidos:",
    tabla_folds_cv["Grupos compartidos"].max()
)

if not tabla_resumen_cv[
    "Repeticiones distintas"
].all():
    print(
        "ATENCIÓN: alguna semana generó "
        "repeticiones idénticas; revisar semillas."
    )
else:
    print(
        "Las repeticiones generaron "
        "particiones diferentes en todas las semanas."
    )

print("No se ha utilizado el conjunto de prueba.")
print("No se han entrenado modelos.")
print("Los folds quedan disponibles en cv_splits_oulad.")


In [ ]:
# SNAPSHOT 3.5 - Folds originales estratificados por estudiante
assert all(len(cv_splits_oulad[s])==10 for s in [2,4,6,8])
guardar_estado_oulad('cv_splits_oulad')



### 3.5.6. Resultados e interpretación

Se implementó una estrategia de validación cruzada estratificada, repetida y agrupada mediante `StratifiedGroupKFold`, utilizando cinco folds y dos repeticiones.

Se generaron diez particiones de evaluación por ventana temporal, para un total de 40 particiones correspondientes a las semanas 2, 4, 6 y 8.

**Independencia entre estudiantes**

La auditoría confirmó que no existen identificadores de estudiantes compartidos entre los subconjuntos de entrenamiento y validación de ninguno de los folds.

Esta condición es especialmente importante porque OULAD contiene estudiantes con múltiples matrículas académicas.

**Balance de las clases**

Las proporciones de resultados adversos en los folds de validación se mantuvieron próximas a las proporciones de las cohortes originales.

La desviación absoluta máxima fue:

- Semana 2: 1,78 puntos porcentuales.
- Semana 4: 0,94 puntos porcentuales.
- Semana 6: 1,20 puntos porcentuales.
- Semana 8: 1,81 puntos porcentuales.

Estas diferencias son compatibles con las restricciones impuestas por la agrupación de estudiantes.

**Reproducibilidad**

Las dos repeticiones generaron particiones diferentes en todas las ventanas y las mismas divisiones podrán reutilizarse para evaluar los experimentos A y B.

El conjunto de prueba independiente no participó en la generación ni en la validación de estas particiones.

### 3.5.7. Consideraciones estadísticas

Los resultados de validación cruzada repetida presentan dependencia, debido al solapamiento entre las muestras de entrenamiento y a la reutilización de observaciones en distintas repeticiones.

Por tanto, las comparaciones entre modelos deberán considerar esta dependencia y complementarse con diferencias de rendimiento, intervalos de incertidumbre y pruebas estadísticas apropiadas.

La selección de hiperparámetros y características se realizará exclusivamente dentro del conjunto de entrenamiento.

### 3.5.8. Conclusión

Se estableció una estrategia reproducible de validación cruzada que combina estratificación aproximada y separación estricta por estudiante.

La estrategia permite comparar los experimentos A y B bajo particiones equivalentes y evita la contaminación derivada de múltiples matrículas de un mismo estudiante.

**Conclusión principal:** las 40 particiones de validación se encuentran preparadas para desarrollar y comparar los modelos supervisados, manteniendo el conjunto de prueba independiente reservado para la evaluación final.



### 3.6. Construcción de pipelines de preprocesamiento, selección y clasificación

#### Objetivo

Diseñar pipelines reproducibles para los experimentos A y B, integrando imputación, codificación, escalamiento, selección de características y los clasificadores supervisados seleccionados.

#### Algoritmos

Se utilizarán dos técnicas principales:

**Regresión logística:** modelo lineal que permitirá establecer una referencia interpretable para la clasificación binaria del riesgo académico.

**Random Forest:** modelo de ensamble que permitirá evaluar relaciones no lineales e interacciones entre variables estáticas y longitudinales.

Ambos algoritmos se evaluarán sobre las mismas cohortes y particiones de validación.

#### Preprocesamiento

Las variables numéricas se procesarán mediante imputación por mediana y escalamiento estándar.

Las variables categóricas se procesarán mediante imputación de una categoría explícita para valores faltantes y codificación One-Hot.

El preprocesamiento se integrará mediante `ColumnTransformer`.

#### Selección de características

Se incorporarán dos procedimientos de filtrado:

1. `VarianceThreshold`, para eliminar características sin variabilidad.
2. `SelectPercentile` con `f_classif`, para seleccionar características mediante una puntuación univariante.

Se utilizará inicialmente un porcentaje de selección del 80 %, que posteriormente podrá optimizarse como hiperparámetro.

#### Prevención de fuga de información

Todos los procedimientos se incluirán dentro de un `Pipeline` de scikit-learn.

Durante la validación cruzada, imputadores, escaladores, codificadores y selectores se ajustarán exclusivamente con los datos de entrenamiento de cada fold.

La selección basada en la variable objetivo también se realizará dentro del fold correspondiente.

El conjunto de prueba independiente permanecerá reservado para la evaluación final.

#### Alcance de la actividad

Se construirán los pipelines para cuatro ventanas temporales, dos experimentos y dos algoritmos.

Para verificar su funcionamiento, se realizará una prueba técnica del preprocesamiento y de la selección de características utilizando únicamente un fold de entrenamiento de la semana 2.

No se entrenarán todavía los clasificadores ni se utilizará el conjunto de prueba.

**Referencias metodológicas:** la selección de características tipo filtro y el diseño de pipelines se relacionan con [9] y con las implementaciones de scikit-learn [2]. Random Forest se fundamenta en [3].


In [ ]:

# ============================================================
# PROYECTO FINAL DE MACHINE LEARNING
# ACTIVIDAD 3.6 - CONSTRUCCIÓN DE PIPELINES
# ============================================================

# Asistencia IA: ChatGPT, octubre de 2026.
# Prompt:
# "Construir pipelines reproducibles de scikit-learn
# para regresión logística y Random Forest, usando
# ColumnTransformer, imputación, OneHotEncoder,
# StandardScaler, VarianceThreshold y SelectPercentile.
# Preparar modelos para experimentos A y B en las
# semanas 2, 4, 6 y 8, comprobando el preprocesamiento
# con un fold de entrenamiento sin entrenar clasificadores
# ni utilizar el conjunto de prueba independiente."

import numpy as np
import pandas as pd
import sklearn

from sklearn.base import clone

from sklearn.compose import ColumnTransformer

from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer

from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)

from sklearn.feature_selection import (
    VarianceThreshold,
    SelectPercentile,
    f_classif
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from IPython.display import display

# ------------------------------------------------------------
# 1. CONFIGURACIÓN
# ------------------------------------------------------------

SEMILLA_MODELOS = 42
PORCENTAJE_SELECCION_INICIAL = 80

print("=== 3.6.1 CONFIGURACIÓN ===")

print("Scikit-learn:", sklearn.__version__)
print("Selección inicial:", PORCENTAJE_SELECCION_INICIAL)
print("Semanas disponibles:", sorted(experimentos_oulad))

# ------------------------------------------------------------
# 2. CONSTRUIR PREPROCESADOR
# ------------------------------------------------------------

def construir_preprocesador(variables_longitudinales):

    columnas_numericas = (
        list(VARIABLES_NUMERICAS)
        + list(variables_longitudinales)
    )

    columnas_categoricas = list(
        VARIABLES_CATEGORICAS
    )

    assert not (
        set(columnas_numericas)
        & set(columnas_categoricas)
    )

    # Variables numéricas
    pipeline_numerico = Pipeline(
        steps=[
            (
                "imputacion",
                SimpleImputer(strategy="median")
            ),
            (
                "escalamiento",
                StandardScaler()
            )
        ]
    )

    # Variables categóricas
    pipeline_categorico = Pipeline(
        steps=[
            (
                "imputacion",
                SimpleImputer(
                    strategy="constant",
                    fill_value="Desconocido"
                )
            ),
            (
                "codificacion",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False,
                    dtype=np.float32
                )
            )
        ]
    )

    preprocesador = ColumnTransformer(
        transformers=[
            (
                "numericas",
                pipeline_numerico,
                columnas_numericas
            ),
            (
                "categoricas",
                pipeline_categorico,
                columnas_categoricas
            )
        ],
        remainder="drop",
        sparse_threshold=0.0,
        verbose_feature_names_out=True
    )

    return preprocesador


# ------------------------------------------------------------
# 3. CONSTRUIR SELECTORES
# ------------------------------------------------------------

def construir_selector():

    return [
        (
            "filtro_varianza",
            VarianceThreshold(threshold=0.0)
        ),
        (
            "seleccion",
            SelectPercentile(
                score_func=f_classif,
                percentile=PORCENTAJE_SELECCION_INICIAL
            )
        )
    ]


# ------------------------------------------------------------
# 4. CONSTRUIR CLASIFICADORES
# ------------------------------------------------------------

def construir_clasificador(algoritmo):

    if algoritmo == "LogisticRegression":

        return LogisticRegression(
            solver="liblinear",
            max_iter=3000,
            random_state=SEMILLA_MODELOS
        )

    elif algoritmo == "RandomForest":

        return RandomForestClassifier(
            n_estimators=150,
            min_samples_leaf=2,
            random_state=SEMILLA_MODELOS,
            n_jobs=-1
        )

    else:
        raise ValueError(
            f"Algoritmo desconocido: {algoritmo}"
        )


# ------------------------------------------------------------
# 5. CREAR PIPELINE COMPLETO
# ------------------------------------------------------------

def construir_pipeline_modelo(
    semana,
    experimento,
    algoritmo
):

    if semana not in experimentos_oulad:
        raise ValueError("Semana no disponible.")

    if experimento not in ["A", "B"]:
        raise ValueError("Experimento debe ser A o B.")

    datos = experimentos_oulad[semana]

    if experimento == "A":

        longitudinales = []

    else:

        longitudinales = datos[
            "features_longitudinales"
        ]

    pasos = [
        (
            "preprocesamiento",
            construir_preprocesador(
                longitudinales
            )
        )
    ]

    pasos.extend(construir_selector())

    pasos.append(
        (
            "modelo",
            construir_clasificador(algoritmo)
        )
    )

    return Pipeline(steps=pasos)


# ------------------------------------------------------------
# 6. PREPARAR LAS 16 CONFIGURACIONES
# ------------------------------------------------------------

print("\n=== 3.6.2 PIPELINES CONSTRUIDOS ===")

pipelines_oulad = {}

resumen_pipelines = []

for semana in [2, 4, 6, 8]:

    for experimento in ["A", "B"]:

        for algoritmo in [
            "LogisticRegression",
            "RandomForest"
        ]:

            clave = (
                semana,
                experimento,
                algoritmo
            )

            pipeline = construir_pipeline_modelo(
                semana,
                experimento,
                algoritmo
            )

            pipelines_oulad[clave] = pipeline

            X = experimentos_oulad[semana][
                f"X_{experimento}_train"
            ]

            resumen_pipelines.append({
                "Semana": semana,
                "Experimento": experimento,
                "Algoritmo": algoritmo,
                "Matrículas": len(X),
                "Variables iniciales": X.shape[1],
                "Pasos": " → ".join(
                    pipeline.named_steps.keys()
                )
            })

tabla_pipelines = pd.DataFrame(
    resumen_pipelines
)

display(tabla_pipelines)

assert len(pipelines_oulad) == 16

# ------------------------------------------------------------
# 7. COMPROBAR COLUMNAS DE ENTRADA
# ------------------------------------------------------------

print("\n=== 3.6.3 CONTROL DE VARIABLES ===")

for semana in [2, 4, 6, 8]:

    datos = experimentos_oulad[semana]

    XA = datos["X_A_train"]
    XB = datos["X_B_train"]

    esperadas_A = set(VARIABLES_ESTATICAS)

    esperadas_B = (
        esperadas_A
        | set(datos["features_longitudinales"])
    )

    assert set(XA.columns) == esperadas_A
    assert set(XB.columns) == esperadas_B

    assert XA.columns.is_unique
    assert XB.columns.is_unique

    assert len(XA) == len(datos["y_train"])
    assert len(XB) == len(datos["y_train"])

    print(
        f"Semana {semana}: "
        "columnas verificadas para A y B."
    )

# ------------------------------------------------------------
# 8. PRUEBA TÉCNICA CON UN SOLO FOLD
# ------------------------------------------------------------

print("\n=== 3.6.4 PRUEBA DE PREPROCESAMIENTO ===")

semana_prueba = 2

idx_fold_train, idx_fold_val = (
    cv_splits_oulad[semana_prueba][0]
)

datos = experimentos_oulad[semana_prueba]

y_train_completo = datos["y_train"]

# Solo se ajusta el preprocesamiento de un fold.
# No se entrena ningún clasificador.

resultados_preprocesamiento = []

caracteristicas_seleccionadas_auditoria = {}

for experimento in ["A", "B"]:

    X_completo = datos[
        f"X_{experimento}_train"
    ]

    X_fold_train = (
        X_completo.iloc[idx_fold_train]
    )

    X_fold_val = (
        X_completo.iloc[idx_fold_val]
    )

    y_fold_train = (
        y_train_completo.iloc[idx_fold_train]
    )

    if experimento == "A":
        longitudinales = []
    else:
        longitudinales = datos[
            "features_longitudinales"
        ]

    # Pipeline técnico: solo preprocesamiento
    # y selección de características.
    pipeline_tecnico = Pipeline(
        steps=[
            (
                "preprocesamiento",
                construir_preprocesador(
                    longitudinales
                )
            ),
            *construir_selector()
        ]
    )

    # Ajustar únicamente sobre el fold de entrenamiento.
    X_train_transformado = (
        pipeline_tecnico.fit_transform(
            X_fold_train,
            y_fold_train
        )
    )

    # Validación: únicamente transformar.
    X_val_transformado = (
        pipeline_tecnico.transform(
            X_fold_val
        )
    )

    # --------------------------------------------------------
    # 9. RECUPERAR NOMBRES DE CARACTERÍSTICAS
    # --------------------------------------------------------

    preprocesador_ajustado = (
        pipeline_tecnico.named_steps[
            "preprocesamiento"
        ]
    )

    nombres_preprocesados = (
        preprocesador_ajustado
        .get_feature_names_out()
    )

    filtro_ajustado = (
        pipeline_tecnico.named_steps[
            "filtro_varianza"
        ]
    )

    selector_ajustado = (
        pipeline_tecnico.named_steps[
            "seleccion"
        ]
    )

    nombres_no_constantes = (
        nombres_preprocesados[
            filtro_ajustado.get_support()
        ]
    )

    nombres_seleccionados = (
        nombres_no_constantes[
            selector_ajustado.get_support()
        ]
    )

    caracteristicas_seleccionadas_auditoria[
        experimento
    ] = list(nombres_seleccionados)

    # --------------------------------------------------------
    # 10. VERIFICACIONES
    # --------------------------------------------------------

    assert X_train_transformado.shape[0] == (
        len(idx_fold_train)
    )

    assert X_val_transformado.shape[0] == (
        len(idx_fold_val)
    )

    assert X_train_transformado.shape[1] == (
        X_val_transformado.shape[1]
    )

    assert X_train_transformado.shape[1] == (
        len(nombres_seleccionados)
    )

    assert np.isfinite(
        X_train_transformado
    ).all()

    assert np.isfinite(
        X_val_transformado
    ).all()

    # Los grupos ya estaban auditados en 3.5.
    grupos = datos["grupos_train"]

    assert set(
        grupos.iloc[idx_fold_train]
    ).isdisjoint(
        set(grupos.iloc[idx_fold_val])
    )

    resultados_preprocesamiento.append({
        "Experimento": experimento,
        "Filas train fold": (
            X_train_transformado.shape[0]
        ),
        "Filas validación": (
            X_val_transformado.shape[0]
        ),
        "Variables originales": X_completo.shape[1],
        "Variables después de One-Hot": (
            len(nombres_preprocesados)
        ),
        "Variables no constantes": (
            len(nombres_no_constantes)
        ),
        "Variables seleccionadas": (
            len(nombres_seleccionados)
        ),
        "Nulos o infinitos finales": 0
    })

    print(
        f"Experimento {experimento}: "
        f"{X_completo.shape[1]} variables originales "
        f"→ {len(nombres_preprocesados)} codificadas "
        f"→ {len(nombres_no_constantes)} no constantes "
        f"→ {len(nombres_seleccionados)} seleccionadas."
    )

tabla_prueba_pipeline = pd.DataFrame(
    resultados_preprocesamiento
)

display(tabla_prueba_pipeline)

# ------------------------------------------------------------
# 11. CARACTERÍSTICAS SELECCIONADAS
# ------------------------------------------------------------

print("\n=== 3.6.5 SELECCIÓN TIPO FILTRO ===")

for experimento, nombres in (
    caracteristicas_seleccionadas_auditoria.items()
):

    print(f"\nExperimento {experimento}:")
    print(
        "Características seleccionadas "
        "en el fold auditado:",
        len(nombres)
    )

    display(
        pd.DataFrame({
            "Característica": nombres
        }).head(20)
    )

# Estos nombres pertenecen únicamente al fold
# auditado. NO representan una selección final.
#
# Durante CV cada fold ajustará nuevamente su
# propio selector usando solo su entrenamiento.

# ------------------------------------------------------------
# 12. VERIFICAR QUE LOS MODELOS NO ESTÉN ENTRENADOS
# ------------------------------------------------------------

print("\n=== 3.6.6 CONTROLES FINALES ===")

for clave, pipeline in pipelines_oulad.items():

    assert "preprocesamiento" in pipeline.named_steps
    assert "filtro_varianza" in pipeline.named_steps
    assert "seleccion" in pipeline.named_steps
    assert "modelo" in pipeline.named_steps

print(
    "Configuraciones disponibles:",
    len(pipelines_oulad)
)

print(
    "Todas incluyen preprocesamiento y "
    "selección de características."
)

print(
    "La prueba técnica utilizó exclusivamente "
    "un fold de entrenamiento."
)

print(
    "No se ajustaron clasificadores."
)

print(
    "No se utilizaron los conjuntos de prueba."
)

print("\n=== ACTIVIDAD 3.6 FINALIZADA ===")


In [ ]:
# SNAPSHOT 3.6 - Pipelines originales de los 16 experimentos
assert len(pipelines_oulad)==16
guardar_estado_oulad('pipelines_oulad')


In [ ]:

print("=== VERIFICACIÓN DE PERSISTENCIA ===")

for nombre in [
    "cohortes_oulad",
    "experimentos_oulad",
    "cv_splits_oulad",
    "pipelines_oulad"
]:
    print(
        nombre,
        "OK" if snapshot_disponible_oulad(nombre)
        else "PENDIENTE"
    )

assert len(cv_splits_oulad) == 4
assert all(
    len(cv_splits_oulad[s]) == 10
    for s in [2, 4, 6, 8]
)
assert len(pipelines_oulad) == 16

print("\nSistema de experimentación respaldado.")



### 3.6.7. Resultados e interpretación

Se construyeron 16 pipelines de Machine Learning correspondientes a cuatro ventanas temporales, dos experimentos y dos algoritmos supervisados.

Cada pipeline integra preprocesamiento mediante `ColumnTransformer`, imputación de valores faltantes, codificación One-Hot, escalamiento, filtrado por varianza, selección univariante y un clasificador.

**Auditoría del preprocesamiento**

Se realizó una prueba técnica utilizando exclusivamente el subconjunto de entrenamiento del primer fold de validación cruzada correspondiente a la semana 2.

En el Experimento A, las 12 características originales generaron 51 columnas después del preprocesamiento. El filtro de varianza conservó las 51 y el procedimiento `SelectPercentile` seleccionó 40.

En el Experimento B, las 30 características originales generaron 69 columnas. El filtro de varianza eliminó una columna constante y el procedimiento de selección conservó 54 características.

Las matrices transformadas no presentaron valores faltantes ni infinitos.

**Selección de características**

Entre las características seleccionadas en el Experimento B se identificaron indicadores de actividad VLE semanal, días activos, actividad acumulada, variaciones de participación y entregas académicas.

Esta selección es preliminar y corresponde únicamente al fold auditado. No representa una selección definitiva ni permite establecer causalidad o importancia multivariante.

La selección se ajustará nuevamente dentro de cada partición de entrenamiento durante la validación cruzada.

**Prevención de fuga de información**

Las transformaciones se ajustaron utilizando únicamente el fold de entrenamiento. Los registros de validación se transformaron con los parámetros aprendidos, sin reajustar los procedimientos.

No se entrenaron clasificadores ni se utilizaron los conjuntos de prueba independientes.

### 3.6.8. Conclusión

Los pipelines construidos permiten integrar de manera reproducible el preprocesamiento y la selección de características en los experimentos A y B.

La auditoría confirmó que las matrices resultantes son numéricas, mantienen las dimensiones correspondientes y no contienen valores faltantes ni infinitos.

**Conclusión principal:** las 16 configuraciones experimentales están preparadas para iniciar el entrenamiento y la optimización de hiperparámetros bajo validación cruzada agrupada por estudiante.

La utilidad predictiva de las características longitudinales todavía debe demostrarse mediante la evaluación de los clasificadores.



### 3.7. Entrenamiento piloto y optimización inicial de hiperparámetros

#### Objetivo

Verificar el entrenamiento y la optimización inicial de Regresión Logística y Random Forest en los experimentos A y B, utilizando la cohorte de la semana 2.

#### Diseño experimental

Se utilizará el primer fold externo de la validación cruzada previamente construida.

La partición contiene:

- 17.859 matrículas para entrenamiento y optimización interna.
- 4.553 matrículas para evaluación piloto.

Ambas particiones respetan la separación por estudiante.

Dentro del subconjunto de entrenamiento se realizará una búsqueda de hiperparámetros mediante tres folds adicionales de `StratifiedGroupKFold`, sin compartir estudiantes entre entrenamiento y validación.

#### Modelos

Se evaluarán cuatro configuraciones:

1. Experimento A con Regresión Logística.
2. Experimento B con Regresión Logística.
3. Experimento A con Random Forest.
4. Experimento B con Random Forest.

La diferencia entre los experimentos A y B será exclusivamente el conjunto de características.

#### Optimización

Se utilizará `GridSearchCV` con F1 macro como métrica principal.

Para Regresión Logística se explorarán valores de regularización y porcentajes de selección de características.

Para Random Forest se explorarán configuraciones iniciales de profundidad máxima y cantidad mínima de observaciones por hoja.

Todas las transformaciones y selecciones de características permanecerán dentro de los pipelines y se ajustarán nuevamente en cada fold interno.

#### Evaluación piloto

Se calcularán:

- Accuracy.
- Precision de la clase adversa.
- Recall de la clase adversa.
- F1 de la clase adversa.
- F1 macro.
- F1 weighted.
- ROC-AUC.
- Matriz de confusión.

También se incluirá una referencia de clase mayoritaria calculada exclusivamente a partir del entrenamiento.

#### Limitación

La evaluación utilizará solamente un fold externo y tendrá carácter diagnóstico.

Sus resultados no serán suficientes para concluir que un algoritmo es superior ni para aceptar o rechazar la hipótesis principal.

La comparación definitiva utilizará las cuatro ventanas, validación repetida y el conjunto de prueba independiente.

El conjunto de prueba no participará en esta actividad.


In [ ]:

# ============================================================
# PROYECTO FINAL DE MACHINE LEARNING
# ACTIVIDAD 3.7 - ENTRENAMIENTO PILOTO Y GRID SEARCH
# SEMANA 2
# ============================================================

# Asistencia IA: ChatGPT, octubre de 2026.
# Prompt:
# "Entrenar Regresión Logística y Random Forest
# para los experimentos A y B de OULAD en semana 2.
# Utilizar el primer fold externo para evaluación,
# una búsqueda interna agrupada de hiperparámetros,
# F1 macro como criterio principal y matrices de
# confusión. No utilizar el conjunto de prueba
# independiente ni extraer conclusiones definitivas."

import numpy as np
import pandas as pd
import time

from sklearn.base import clone
from sklearn.model_selection import (
    StratifiedGroupKFold,
    GridSearchCV
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

from IPython.display import display

# ------------------------------------------------------------
# 1. CONFIGURACIÓN DEL EXPERIMENTO
# ------------------------------------------------------------

SEMANA_PILOTO = 2
FOLD_EXTERNO = 0
SEMILLA_INTERNA = 137

ALGORITMOS = [
    "LogisticRegression",
    "RandomForest"
]

EXPERIMENTOS = ["A", "B"]

print("=== 3.7.1 CONFIGURACIÓN DEL PILOTO ===")

print("Semana:", SEMANA_PILOTO)
print("Fold externo:", FOLD_EXTERNO + 1)
print("Folds internos:", 3)
print("Métrica principal: F1 macro")

# ------------------------------------------------------------
# 2. RECUPERAR LOS DATOS DE ENTRENAMIENTO
# ------------------------------------------------------------

datos = experimentos_oulad[SEMANA_PILOTO]

y_completo = (
    datos["y_train"].reset_index(drop=True)
)

grupos_completos = (
    datos["grupos_train"].reset_index(drop=True)
)

idx_externo_train, idx_externo_val = (
    cv_splits_oulad[SEMANA_PILOTO][FOLD_EXTERNO]
)

y_interno = (
    y_completo.iloc[idx_externo_train]
    .reset_index(drop=True)
)

y_externo = (
    y_completo.iloc[idx_externo_val]
    .reset_index(drop=True)
)

grupos_internos = (
    grupos_completos.iloc[idx_externo_train]
    .reset_index(drop=True)
)

grupos_externos = (
    grupos_completos.iloc[idx_externo_val]
    .reset_index(drop=True)
)

assert set(grupos_internos).isdisjoint(
    set(grupos_externos)
)

assert len(y_interno) == 17859
assert len(y_externo) == 4553

print("\n=== 3.7.2 PARTICIÓN EXTERNA ===")

print("Entrenamiento:", len(y_interno))
print("Validación:", len(y_externo))

print(
    "Estudiantes compartidos:",
    len(
        set(grupos_internos)
        & set(grupos_externos)
    )
)

print(
    "% adversos entrenamiento:",
    round(100 * y_interno.mean(), 2)
)

print(
    "% adversos validación:",
    round(100 * y_externo.mean(), 2)
)

# ------------------------------------------------------------
# 3. CONSTRUIR FOLDS INTERNOS
# ------------------------------------------------------------

splitter_interno = StratifiedGroupKFold(
    n_splits=3,
    shuffle=True,
    random_state=SEMILLA_INTERNA
)

# Utilizamos X_A solo para generar los índices.
# Los mismos índices se aplicarán a X_B.

XA_interno = (
    datos["X_A_train"]
    .iloc[idx_externo_train]
    .reset_index(drop=True)
)

splits_internos = list(
    splitter_interno.split(
        XA_interno,
        y_interno,
        groups=grupos_internos
    )
)

print("\n=== 3.7.3 VALIDACIÓN INTERNA ===")

auditoria_interna = []

for numero, (idx_tr, idx_val) in enumerate(
    splits_internos,
    start=1
):

    g_train = set(
        grupos_internos.iloc[idx_tr]
    )

    g_val = set(
        grupos_internos.iloc[idx_val]
    )

    assert g_train.isdisjoint(g_val)

    assert set(
        y_interno.iloc[idx_tr].unique()
    ) == {0, 1}

    assert set(
        y_interno.iloc[idx_val].unique()
    ) == {0, 1}

    auditoria_interna.append({
        "Fold interno": numero,
        "Matrículas train": len(idx_tr),
        "Matrículas validación": len(idx_val),
        "% adversos validación": round(
            100 * y_interno.iloc[idx_val].mean(),
            2
        ),
        "Estudiantes compartidos": 0
    })

display(pd.DataFrame(auditoria_interna))

# ------------------------------------------------------------
# 4. DEFINIR ESPACIOS DE HIPERPARÁMETROS
# ------------------------------------------------------------

# Búsqueda pequeña para comprobar el procedimiento.
# Se ampliará una vez validada la ejecución.

GRIDS_PILOTO = {

    "LogisticRegression": {
        "modelo__C": [0.1, 1.0],
        "seleccion__percentile": [80, 100]
    },

    "RandomForest": {
        "modelo__n_estimators": [100],
        "modelo__max_depth": [12, None],
        "modelo__min_samples_leaf": [2, 5],
        "seleccion__percentile": [80]
    }

}

print("\n=== 3.7.4 HIPERPARÁMETROS ===")

for algoritmo, grid in GRIDS_PILOTO.items():
    print(f"\n{algoritmo}")
    print(grid)

# ------------------------------------------------------------
# 5. MÉTRICAS DE EVALUACIÓN
# ------------------------------------------------------------

def calcular_metricas(y_real, y_pred, prob_riesgo):

    return {
        "Accuracy": accuracy_score(
            y_real, y_pred
        ),
        "Precision adverso": precision_score(
            y_real, y_pred,
            pos_label=1,
            zero_division=0
        ),
        "Recall adverso": recall_score(
            y_real, y_pred,
            pos_label=1,
            zero_division=0
        ),
        "F1 adverso": f1_score(
            y_real, y_pred,
            pos_label=1,
            zero_division=0
        ),
        "F1 macro": f1_score(
            y_real, y_pred,
            average="macro",
            zero_division=0
        ),
        "F1 weighted": f1_score(
            y_real, y_pred,
            average="weighted",
            zero_division=0
        ),
        "ROC-AUC": roc_auc_score(
            y_real, prob_riesgo
        )
    }


# ------------------------------------------------------------
# 6. ENTRENAMIENTO Y BÚSQUEDA
# ------------------------------------------------------------

print("\n=== 3.7.5 ENTRENAMIENTO DE MODELOS ===")

resultados_piloto = []
busquedas_piloto = []
matrices_confusion_piloto = []

# Conservar modelos ajustados únicamente
# con el subconjunto de entrenamiento externo.
modelos_piloto = {}

inicio_global = time.perf_counter()

for experimento in EXPERIMENTOS:

    X_completo = (
        datos[f"X_{experimento}_train"]
        .reset_index(drop=True)
    )

    X_interno = (
        X_completo.iloc[idx_externo_train]
        .reset_index(drop=True)
    )

    X_externo = (
        X_completo.iloc[idx_externo_val]
        .reset_index(drop=True)
    )

    assert len(X_interno) == len(y_interno)
    assert len(X_externo) == len(y_externo)

    for algoritmo in ALGORITMOS:

        print("\n" + "=" * 55)
        print(
            f"Experimento {experimento} | {algoritmo}"
        )
        print("=" * 55)

        inicio_modelo = time.perf_counter()

        # Recuperar el pipeline SIN ajustar.
        pipeline = clone(
            pipelines_oulad[
                (SEMANA_PILOTO, experimento, algoritmo)
            ]
        )

        # GridSearchCV reajusta completamente
        # el pipeline en cada fold interno.
        busqueda = GridSearchCV(
            estimator=pipeline,
            param_grid=GRIDS_PILOTO[algoritmo],
            scoring="f1_macro",
            cv=splits_internos,
            refit=True,
            n_jobs=1,
            verbose=0,
            return_train_score=True,
            error_score="raise"
        )

        # Solo datos del entrenamiento externo.
        busqueda.fit(
            X_interno,
            y_interno
        )

        # Guardar el modelo optimizado del piloto.
        mejor_modelo = busqueda.best_estimator_

        modelos_piloto[
            (experimento, algoritmo)
        ] = mejor_modelo

        print(
            "Mejor F1 macro CV interno:",
            round(busqueda.best_score_, 4)
        )

        print(
            "Mejores hiperparámetros:",
            busqueda.best_params_
        )

        # --------------------------------------------
        # EVALUACIÓN EN FOLD EXTERNO
        # --------------------------------------------

        # Primera evaluación sobre este fold.
        # No intervenir en la búsqueda después
        # de ver sus etiquetas o resultados.

        y_pred = mejor_modelo.predict(
            X_externo
        )

        clases = list(
            mejor_modelo.classes_
        )

        indice_adverso = clases.index(1)

        prob_adverso = (
            mejor_modelo.predict_proba(
                X_externo
            )[:, indice_adverso]
        )

        metricas = calcular_metricas(
            y_externo,
            y_pred,
            prob_adverso
        )

        duracion = (
            time.perf_counter() - inicio_modelo
        )

        fila_resultado = {
            "Experimento": experimento,
            "Algoritmo": algoritmo,
            "F1 macro CV interno": (
                busqueda.best_score_
            ),
            **metricas,
            "Tiempo segundos": duracion
        }

        resultados_piloto.append(
            fila_resultado
        )

        # Matriz de confusión
        cm = confusion_matrix(
            y_externo,
            y_pred,
            labels=[0, 1]
        )

        matrices_confusion_piloto.append({
            "Experimento": experimento,
            "Algoritmo": algoritmo,
            "VN": int(cm[0, 0]),
            "FP": int(cm[0, 1]),
            "FN": int(cm[1, 0]),
            "VP": int(cm[1, 1])
        })

        # Guardar las configuraciones evaluadas.
        cv_resultados = pd.DataFrame(
            busqueda.cv_results_
        )

        for _, fila in cv_resultados.iterrows():

            busquedas_piloto.append({
                "Experimento": experimento,
                "Algoritmo": algoritmo,
                "Parámetros": str(
                    fila["params"]
                ),
                "F1 macro CV media": (
                    fila["mean_test_score"]
                ),
                "F1 macro CV desviación": (
                    fila["std_test_score"]
                ),
                "F1 macro train media": (
                    fila["mean_train_score"]
                ),
                "Posición": int(
                    fila["rank_test_score"]
                )
            })

        print(
            "F1 macro validación externa:",
            round(metricas["F1 macro"], 4)
        )

        print(
            "Recall adverso:",
            round(metricas["Recall adverso"], 4)
        )

        print(
            "ROC-AUC:",
            round(metricas["ROC-AUC"], 4)
        )

        print(
            "Tiempo:",
            round(duracion, 1),
            "segundos"
        )

duracion_total = (
    time.perf_counter() - inicio_global
)

# ------------------------------------------------------------
# 7. TABLA DE RESULTADOS DEL PILOTO
# ------------------------------------------------------------

tabla_resultados_piloto = pd.DataFrame(
    resultados_piloto
)

print("\n=== 3.7.6 RESULTADOS DE VALIDACIÓN EXTERNA ===")

display(
    tabla_resultados_piloto.round(4)
)

# ------------------------------------------------------------
# 8. MATRICES DE CONFUSIÓN
# ------------------------------------------------------------

tabla_confusion_piloto = pd.DataFrame(
    matrices_confusion_piloto
)

print("\n=== 3.7.7 MATRICES DE CONFUSIÓN ===")

display(tabla_confusion_piloto)

# ------------------------------------------------------------
# 9. DETALLE DE LA BÚSQUEDA
# ------------------------------------------------------------

tabla_busquedas_piloto = pd.DataFrame(
    busquedas_piloto
)

print("\n=== 3.7.8 RESULTADOS GRID SEARCH ===")

display(
    tabla_busquedas_piloto
    .sort_values(
        [
            "Experimento",
            "Algoritmo",
            "Posición"
        ]
    )
    .round(4)
)

# ------------------------------------------------------------
# 10. REFERENCIA DE CLASE MAYORITARIA
# ------------------------------------------------------------

print("\n=== 3.7.9 BASELINE DE CLASE MAYORITARIA ===")

clase_mayoritaria = int(
    y_interno.mean() >= 0.5
)

y_baseline = np.full(
    len(y_externo),
    clase_mayoritaria
)

prob_baseline = np.full(
    len(y_externo),
    float(y_interno.mean())
)

metricas_baseline = calcular_metricas(
    y_externo,
    y_baseline,
    prob_baseline
)

display(
    pd.DataFrame([
        {
            "Modelo": "Clase mayoritaria",
            "Clase predicha": clase_mayoritaria,
            **metricas_baseline
        }
    ]).round(4)
)

# ------------------------------------------------------------
# 11. COMPARACIÓN PRELIMINAR A vs B
# ------------------------------------------------------------

print("\n=== 3.7.10 DIFERENCIAS A VS B ===")

comparacion_piloto = []

for algoritmo in ALGORITMOS:

    fila_A = tabla_resultados_piloto.loc[
        (tabla_resultados_piloto["Experimento"] == "A")
        &
        (tabla_resultados_piloto["Algoritmo"] == algoritmo)
    ].iloc[0]

    fila_B = tabla_resultados_piloto.loc[
        (tabla_resultados_piloto["Experimento"] == "B")
        &
        (tabla_resultados_piloto["Algoritmo"] == algoritmo)
    ].iloc[0]

    comparacion_piloto.append({
        "Algoritmo": algoritmo,
        "F1 macro A": fila_A["F1 macro"],
        "F1 macro B": fila_B["F1 macro"],
        "Diferencia F1 (B-A)": (
            fila_B["F1 macro"] - fila_A["F1 macro"]
        ),
        "Diferencia Recall adverso (B-A)": (
            fila_B["Recall adverso"]
            - fila_A["Recall adverso"]
        ),
        "Diferencia ROC-AUC (B-A)": (
            fila_B["ROC-AUC"]
            - fila_A["ROC-AUC"]
        )
    })

tabla_comparacion_piloto = pd.DataFrame(
    comparacion_piloto
)

display(
    tabla_comparacion_piloto.round(4)
)

# ------------------------------------------------------------
# 12. CONTROLES FINALES
# ------------------------------------------------------------

print("\n=== 3.7.11 CONTROLES FINALES ===")

assert len(modelos_piloto) == 4

assert len(tabla_resultados_piloto) == 4

assert (
    tabla_resultados_piloto[
        [
            "Accuracy",
            "F1 macro",
            "ROC-AUC"
        ]
    ].notna().all().all()
)

assert set(grupos_internos).isdisjoint(
    set(grupos_externos)
)

print(
    "Modelos entrenados:",
    len(modelos_piloto)
)

print(
    "Tiempo total:",
    round(duracion_total, 1),
    "segundos"
)

print(
    "Conjunto de prueba independiente: NO utilizado"
)

print(
    "Comparaciones A vs B: exclusivamente preliminares"
)

print("\n=== ACTIVIDAD 3.7 FINALIZADA ===")



### 3.7.12. Resultados e interpretación

Se realizó un primer experimento piloto de entrenamiento y optimización de hiperparámetros correspondiente a la semana 2.

Se utilizó el primer fold externo de validación, formado por 17.859 matrículas de entrenamiento y 4.553 matrículas de validación, sin estudiantes compartidos entre ambos conjuntos.

La optimización se realizó mediante `GridSearchCV` con tres folds internos agrupados por estudiante y F1 macro como métrica principal.

**Resultados predictivos**

| Experimento | Algoritmo | F1 macro | Recall adverso | ROC-AUC |
|---|---|---:|---:|---:|
| A | Regresión logística | 0,6197 | 0,4944 | 0,6705 |
| B | Regresión logística | 0,6709 | 0,5735 | 0,7358 |
| A | Random Forest | 0,6023 | 0,4871 | 0,6560 |
| B | Random Forest | 0,6585 | 0,5349 | 0,7266 |

Los cuatro modelos superaron la referencia de clase mayoritaria, que obtuvo un F1 macro de 0,3550.

**Comparación entre características estáticas y longitudinales**

La incorporación de características longitudinales mejoró el F1 macro de la regresión logística en 0,0512 y el de Random Forest en aproximadamente 0,0561.

En regresión logística, el recall de la clase adversa aumentó de 0,4944 a 0,5735, mientras que ROC-AUC aumentó de 0,6705 a 0,7358.

En Random Forest, el recall adverso aumentó de 0,4871 a 0,5349 y ROC-AUC pasó de 0,6560 a 0,7266.

Estas diferencias son favorables al Experimento B en la partición examinada.

**Interpretación de errores de clasificación**

En regresión logística, el Experimento B redujo los falsos negativos de 1.035 a 873 y los falsos positivos de 635 a 585.

Esto equivale a identificar correctamente 162 resultados adversos adicionales y reducir en 50 los resultados favorables clasificados erróneamente como adversos.

En Random Forest, el Experimento B redujo los falsos negativos de 1.050 a 952 y los falsos positivos de 704 a 547.

**Análisis preliminar de sobreajuste**

Los modelos de regresión logística presentaron brechas reducidas entre los promedios de F1 macro de entrenamiento y validación interna.

Random Forest presentó diferencias mayores, especialmente en el Experimento B, con un F1 macro medio de entrenamiento de 0,8302 frente a 0,6567 en validación interna.

Esta diferencia sugiere posible sobreajuste y justifica evaluar la regularización del modelo mediante validación cruzada.

### 3.7.13. Limitaciones del experimento piloto

Los resultados corresponden exclusivamente al primer fold externo de la semana 2.

Por tanto, no permiten afirmar superioridad estadística, generalización a otras cohortes ni confirmación definitiva de la hipótesis principal.

Las mejoras observadas describen el comportamiento de los modelos bajo una partición específica y deberán contrastarse mediante validación cruzada repetida.

La exploración del fold piloto también forma parte del desarrollo metodológico. Por ello, la confirmación definitiva deberá reservarse para el conjunto de prueba independiente.

### 3.7.14. Conclusión

Los primeros resultados muestran un incremento del rendimiento predictivo al incorporar características longitudinales, tanto para regresión logística como para Random Forest.

El Experimento B obtuvo mayor F1 macro, recall adverso y ROC-AUC en ambos algoritmos.

La mejora es especialmente relevante para la finalidad de identificar tempranamente estudiantes con resultados académicos adversos.

**Conclusión principal:** el experimento piloto proporciona evidencia preliminar favorable a la hipótesis de que las trayectorias longitudinales aportan información predictiva adicional respecto de las características estáticas. Sin embargo, esta hipótesis todavía requiere validación cruzada repetida, evaluación de estabilidad y contrastación estadística antes de considerarse respaldada de manera concluyente.



### 3.8. Validación cruzada repetida y anidada — Regresión Logística

#### Objetivo

Evaluar la estabilidad del rendimiento de Regresión Logística en los experimentos A y B, utilizando validación cruzada repetida y anidada sobre la cohorte de la semana 2.

#### Diseño experimental

Se utilizarán cinco folds externos y dos repeticiones, generados previamente mediante `StratifiedGroupKFold`.

Para cada fold externo:

1. Se separará el subconjunto de entrenamiento y validación.
2. Se construirán tres folds internos agrupados por estudiante.
3. Se optimizarán los hiperparámetros mediante `GridSearchCV`.
4. Se ajustará el mejor pipeline exclusivamente con el entrenamiento externo.
5. Se evaluará una única vez sobre la validación externa.
6. Se repetirán los mismos procedimientos para los experimentos A y B.

Ambos experimentos utilizarán exactamente las mismas particiones externas e internas.

#### Optimización

Se mantendrá el espacio de búsqueda establecido en la Actividad 3.7:

- Regularización `C`: 0,1 y 1,0.
- Porcentaje de selección de características: 80 % y 100 %.

La métrica de optimización será F1 macro.

#### Métricas de evaluación

Se calcularán accuracy, precisión de la clase adversa, recall de la clase adversa, F1 adverso, F1 macro, F1 weighted y ROC-AUC.

También se conservarán las matrices de confusión y las probabilidades predichas sobre los folds externos.

#### Análisis comparativo

Para cada partición externa se calculará la diferencia de rendimiento entre B y A.

Se examinarán:

- La diferencia promedio de F1 macro.
- La diferencia promedio de recall adverso.
- La diferencia promedio de ROC-AUC.
- La variabilidad de estas diferencias.
- El número de particiones en que B supera a A.
- Las diferencias entre desempeño de entrenamiento y validación interna.

#### Consideraciones estadísticas

Los diez folds externos no son observaciones independientes debido al solapamiento entre los conjuntos de entrenamiento.

Por esta razón, las diferencias se presentarán inicialmente como evidencia descriptiva de estabilidad, sin interpretar una prueba t ordinaria como evidencia confirmatoria.

La evaluación estadística posterior considerará procedimientos apropiados para resultados correlacionados y una comprobación independiente sobre el conjunto de prueba reservado.

#### Alcance

Esta actividad evaluará exclusivamente Regresión Logística en la semana 2.

Random Forest y las demás ventanas temporales se estudiarán posteriormente utilizando el mismo protocolo.

El conjunto de prueba independiente no se utilizará.


In [ ]:

# ============================================================
# PROYECTO FINAL DE MACHINE LEARNING
# ACTIVIDAD 3.8 - VALIDACIÓN CRUZADA ANIDADA
# REGRESIÓN LOGÍSTICA - SEMANA 2
# ============================================================

# Asistencia IA: ChatGPT, octubre de 2026.
# Prompt:
# "Realizar validación cruzada anidada de Regresión
# Logística para experimentos A y B en la semana 2
# de OULAD, con 5 folds externos y 2 repeticiones.
# Optimizar cada fold mediante 3 folds internos
# agrupados por estudiante. Comparar resultados
# pareados de F1 macro, recall y ROC-AUC sin usar
# el conjunto de prueba independiente."

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time

from sklearn.base import clone
from sklearn.model_selection import (
    StratifiedGroupKFold,
    GridSearchCV
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

from IPython.display import display

# ------------------------------------------------------------
# 1. CONFIGURACIÓN
# ------------------------------------------------------------

SEMANA_CV = 2
ALGORITMO_CV = "LogisticRegression"

N_FOLDS_EXTERNOS = 5
N_REPETICIONES = 2
N_FOLDS_INTERNOS = 3

SEMILLA_INTERNA_BASE = 137

GRID_LR = {
    "modelo__C": [0.1, 1.0],
    "seleccion__percentile": [80, 100]
}

datos = experimentos_oulad[SEMANA_CV]

X_A = datos["X_A_train"].reset_index(drop=True)
X_B = datos["X_B_train"].reset_index(drop=True)

y = datos["y_train"].reset_index(drop=True)
grupos = datos["grupos_train"].reset_index(drop=True)

splits_externos = cv_splits_oulad[SEMANA_CV]

assert len(splits_externos) == (
    N_FOLDS_EXTERNOS * N_REPETICIONES
)

assert len(X_A) == len(X_B) == len(y) == len(grupos)

print("=== 3.8.1 CONFIGURACIÓN ===")

print("Semana:", SEMANA_CV)
print("Algoritmo:", ALGORITMO_CV)
print("Matrículas disponibles:", len(y))
print("Folds externos:", len(splits_externos))
print("Folds internos:", N_FOLDS_INTERNOS)
print("Experimentos:", ["A", "B"])
print("Métrica principal: F1 macro")

# ------------------------------------------------------------
# 2. MÉTRICAS
# ------------------------------------------------------------

def metricas_clasificacion(y_real, y_pred, prob_adverso):

    return {
        "Accuracy": accuracy_score(
            y_real, y_pred
        ),
        "Precision_adverso": precision_score(
            y_real, y_pred,
            pos_label=1,
            zero_division=0
        ),
        "Recall_adverso": recall_score(
            y_real, y_pred,
            pos_label=1,
            zero_division=0
        ),
        "F1_adverso": f1_score(
            y_real, y_pred,
            pos_label=1,
            zero_division=0
        ),
        "F1_macro": f1_score(
            y_real, y_pred,
            average="macro",
            zero_division=0
        ),
        "F1_weighted": f1_score(
            y_real, y_pred,
            average="weighted",
            zero_division=0
        ),
        "ROC_AUC": roc_auc_score(
            y_real, prob_adverso
        )
    }

# ------------------------------------------------------------
# 3. CONTENEDORES DE RESULTADOS
# ------------------------------------------------------------

resultados_lr_cv = []
parametros_lr_cv = []
predicciones_lr_cv = []
baselines_lr_cv = []

inicio_global = time.perf_counter()

# ------------------------------------------------------------
# 4. RECORRER FOLDS EXTERNOS
# ------------------------------------------------------------

print("\n=== 3.8.2 VALIDACIÓN CRUZADA ANIDADA ===")

for numero_split, (idx_train, idx_val) in enumerate(
    splits_externos,
    start=1
):

    repeticion = (
        (numero_split - 1) // N_FOLDS_EXTERNOS + 1
    )

    fold = (
        (numero_split - 1) % N_FOLDS_EXTERNOS + 1
    )

    print(
        f"\nRepetición {repeticion} | "
        f"Fold externo {fold}"
    )

    # --------------------------------------------
    # A. ETIQUETAS Y GRUPOS
    # --------------------------------------------

    y_train_fold = (
        y.iloc[idx_train].reset_index(drop=True)
    )

    y_val_fold = (
        y.iloc[idx_val].reset_index(drop=True)
    )

    grupos_train_fold = (
        grupos.iloc[idx_train].reset_index(drop=True)
    )

    grupos_val_fold = (
        grupos.iloc[idx_val].reset_index(drop=True)
    )

    assert set(grupos_train_fold).isdisjoint(
        set(grupos_val_fold)
    )

    assert set(y_train_fold.unique()) == {0, 1}
    assert set(y_val_fold.unique()) == {0, 1}

    # --------------------------------------------
    # B. CREAR FOLDS INTERNOS
    # --------------------------------------------

    splitter_interno = StratifiedGroupKFold(
        n_splits=N_FOLDS_INTERNOS,
        shuffle=True,
        random_state=(
            SEMILLA_INTERNA_BASE
            + numero_split - 1
        )
    )

    # Generar los mismos índices internos para A y B.
    X_A_train_fold = (
        X_A.iloc[idx_train].reset_index(drop=True)
    )

    splits_internos = list(
        splitter_interno.split(
            X_A_train_fold,
            y_train_fold,
            groups=grupos_train_fold
        )
    )

    for idx_interno_train, idx_interno_val in splits_internos:

        assert set(
            grupos_train_fold.iloc[idx_interno_train]
        ).isdisjoint(
            set(
                grupos_train_fold.iloc[idx_interno_val]
            )
        )

        assert set(
            y_train_fold.iloc[idx_interno_train].unique()
        ) == {0, 1}

        assert set(
            y_train_fold.iloc[idx_interno_val].unique()
        ) == {0, 1}

    # --------------------------------------------
    # C. BASELINE DEL FOLD
    # --------------------------------------------

    clase_mayoritaria = int(
        y_train_fold.mean() >= 0.5
    )

    pred_baseline = np.full(
        len(y_val_fold),
        clase_mayoritaria
    )

    prob_baseline = np.full(
        len(y_val_fold),
        float(y_train_fold.mean())
    )

    met_baseline = metricas_clasificacion(
        y_val_fold,
        pred_baseline,
        prob_baseline
    )

    baselines_lr_cv.append({
        "Repeticion": repeticion,
        "Fold": fold,
        **met_baseline
    })

    # --------------------------------------------
    # D. ENTRENAR A Y B
    # --------------------------------------------

    for experimento, X_completo in [
        ("A", X_A),
        ("B", X_B)
    ]:

        inicio_modelo = time.perf_counter()

        X_train_fold = (
            X_completo.iloc[idx_train]
            .reset_index(drop=True)
        )

        X_val_fold = (
            X_completo.iloc[idx_val]
            .reset_index(drop=True)
        )

        pipeline = clone(
            pipelines_oulad[
                (
                    SEMANA_CV,
                    experimento,
                    ALGORITMO_CV
                )
            ]
        )

        busqueda = GridSearchCV(
            estimator=pipeline,
            param_grid=GRID_LR,
            scoring="f1_macro",
            cv=splits_internos,
            refit=True,
            n_jobs=1,
            verbose=0,
            return_train_score=True,
            error_score="raise"
        )

        # Únicamente datos de entrenamiento externo.
        busqueda.fit(
            X_train_fold,
            y_train_fold
        )

        # ----------------------------------------
        # E. PREDICCIÓN EXTERNA
        # ----------------------------------------

        mejor_modelo = busqueda.best_estimator_

        y_pred = mejor_modelo.predict(
            X_val_fold
        )

        indice_adverso = list(
            mejor_modelo.classes_
        ).index(1)

        prob_adverso = (
            mejor_modelo.predict_proba(
                X_val_fold
            )[:, indice_adverso]
        )

        metricas = metricas_clasificacion(
            y_val_fold,
            y_pred,
            prob_adverso
        )

        cm = confusion_matrix(
            y_val_fold,
            y_pred,
            labels=[0, 1]
        )

        duracion = (
            time.perf_counter() - inicio_modelo
        )

        # ----------------------------------------
        # F. RESULTADOS DE ESTE FOLD
        # ----------------------------------------

        resultados_lr_cv.append({
            "Repeticion": repeticion,
            "Fold": fold,
            "Experimento": experimento,
            "N_train": len(idx_train),
            "N_validacion": len(idx_val),
            "F1_CV_interno": busqueda.best_score_,
            **metricas,
            "VN": int(cm[0, 0]),
            "FP": int(cm[0, 1]),
            "FN": int(cm[1, 0]),
            "VP": int(cm[1, 1]),
            "Tiempo_segundos": duracion
        })

        # ----------------------------------------
        # G. HIPERPARÁMETROS POR FOLD
        # ----------------------------------------

        indice_mejor = busqueda.best_index_

        media_train_interno = (
            busqueda.cv_results_[
                "mean_train_score"
            ][indice_mejor]
        )

        media_val_interno = (
            busqueda.cv_results_[
                "mean_test_score"
            ][indice_mejor]
        )

        parametros_lr_cv.append({
            "Repeticion": repeticion,
            "Fold": fold,
            "Experimento": experimento,
            "C": busqueda.best_params_[
                "modelo__C"
            ],
            "Percentile": busqueda.best_params_[
                "seleccion__percentile"
            ],
            "F1_train_interno": media_train_interno,
            "F1_val_interno": media_val_interno,
            "Brecha_train_val": (
                media_train_interno
                - media_val_interno
            )
        })

        # ----------------------------------------
        # H. CONSERVAR PREDICCIONES EXTERNAS
        # ----------------------------------------

        predicciones_lr_cv.append(
            pd.DataFrame({
                "Repeticion": repeticion,
                "Fold": fold,
                "Experimento": experimento,
                "id_student": (
                    grupos_val_fold.to_numpy()
                ),
                "y_real": (
                    y_val_fold.to_numpy()
                ),
                "y_pred": y_pred,
                "prob_adverso": prob_adverso
            })
        )

        print(
            f"  {experimento}: "
            f"F1={metricas['F1_macro']:.4f} | "
            f"Recall={metricas['Recall_adverso']:.4f} | "
            f"AUC={metricas['ROC_AUC']:.4f} | "
            f"{duracion:.1f}s"
        )

# ------------------------------------------------------------
# 5. CONSOLIDAR RESULTADOS
# ------------------------------------------------------------

tabla_lr_cv = pd.DataFrame(resultados_lr_cv)

tabla_parametros_lr_cv = pd.DataFrame(
    parametros_lr_cv
)

tabla_baseline_lr_cv = pd.DataFrame(
    baselines_lr_cv
)

tabla_predicciones_lr_cv = pd.concat(
    predicciones_lr_cv,
    ignore_index=True
)

duracion_global = (
    time.perf_counter() - inicio_global
)

assert len(tabla_lr_cv) == 20

assert len(tabla_parametros_lr_cv) == 20

print("\n=== 3.8.3 RESULTADOS POR FOLD ===")

display(
    tabla_lr_cv[
        [
            "Repeticion",
            "Fold",
            "Experimento",
            "F1_macro",
            "Recall_adverso",
            "ROC_AUC",
            "FN",
            "FP"
        ]
    ].round(4)
)

# ------------------------------------------------------------
# 6. RESUMEN POR EXPERIMENTO
# ------------------------------------------------------------

print("\n=== 3.8.4 ESTADÍSTICAS POR EXPERIMENTO ===")

metricas_resumen = [
    "Accuracy",
    "Precision_adverso",
    "Recall_adverso",
    "F1_adverso",
    "F1_macro",
    "F1_weighted",
    "ROC_AUC"
]

resumen_lr_cv = []

for experimento in ["A", "B"]:

    subconjunto = tabla_lr_cv.loc[
        tabla_lr_cv["Experimento"] == experimento
    ]

    fila = {"Experimento": experimento}

    for metrica in metricas_resumen:

        fila[f"{metrica}_media"] = (
            subconjunto[metrica].mean()
        )

        fila[f"{metrica}_std"] = (
            subconjunto[metrica].std(ddof=1)
        )

    resumen_lr_cv.append(fila)

tabla_resumen_lr_cv = pd.DataFrame(
    resumen_lr_cv
)

display(
    tabla_resumen_lr_cv[
        [
            "Experimento",
            "F1_macro_media",
            "F1_macro_std",
            "Recall_adverso_media",
            "Recall_adverso_std",
            "ROC_AUC_media",
            "ROC_AUC_std",
            "Accuracy_media"
        ]
    ].round(4)
)

# ------------------------------------------------------------
# 7. COMPARACIÓN PAREADA B - A
# ------------------------------------------------------------

print("\n=== 3.8.5 DIFERENCIAS PAREADAS B - A ===")

columnas_pares = [
    "Repeticion",
    "Fold",
    "F1_macro",
    "Recall_adverso",
    "ROC_AUC",
    "Accuracy"
]

pares_A = (
    tabla_lr_cv.loc[
        tabla_lr_cv["Experimento"] == "A",
        columnas_pares
    ]
    .rename(columns={
        "F1_macro": "F1_macro_A",
        "Recall_adverso": "Recall_adverso_A",
        "ROC_AUC": "ROC_AUC_A",
        "Accuracy": "Accuracy_A"
    })
)

pares_B = (
    tabla_lr_cv.loc[
        tabla_lr_cv["Experimento"] == "B",
        columnas_pares
    ]
    .rename(columns={
        "F1_macro": "F1_macro_B",
        "Recall_adverso": "Recall_adverso_B",
        "ROC_AUC": "ROC_AUC_B",
        "Accuracy": "Accuracy_B"
    })
)

tabla_pares_lr = pares_A.merge(
    pares_B,
    on=["Repeticion", "Fold"],
    how="inner",
    validate="one_to_one"
)

for metrica in [
    "F1_macro",
    "Recall_adverso",
    "ROC_AUC",
    "Accuracy"
]:

    tabla_pares_lr[f"Delta_{metrica}"] = (
        tabla_pares_lr[f"{metrica}_B"]
        - tabla_pares_lr[f"{metrica}_A"]
    )

assert len(tabla_pares_lr) == 10

display(
    tabla_pares_lr[
        [
            "Repeticion",
            "Fold",
            "F1_macro_A",
            "F1_macro_B",
            "Delta_F1_macro",
            "Delta_Recall_adverso",
            "Delta_ROC_AUC"
        ]
    ].round(4)
)

# ------------------------------------------------------------
# 8. ESTABILIDAD DE LAS MEJORAS
# ------------------------------------------------------------

print("\n=== 3.8.6 ESTABILIDAD DE LAS DIFERENCIAS ===")

estabilidad = []

for metrica in [
    "F1_macro",
    "Recall_adverso",
    "ROC_AUC",
    "Accuracy"
]:

    deltas = tabla_pares_lr[
        f"Delta_{metrica}"
    ]

    estabilidad.append({
        "Metrica": metrica,
        "Delta_medio_B_menos_A": deltas.mean(),
        "Desviacion_deltas": deltas.std(ddof=1),
        "Delta_minimo": deltas.min(),
        "Delta_maximo": deltas.max(),
        "Folds_B_supera_A": int(
            (deltas > 0).sum()
        ),
        "Folds_empatados": int(
            np.isclose(deltas, 0).sum()
        ),
        "Folds_B_inferior_A": int(
            (deltas < 0).sum()
        )
    })

tabla_estabilidad_lr = pd.DataFrame(
    estabilidad
)

display(tabla_estabilidad_lr.round(4))

# ------------------------------------------------------------
# 9. HIPERPARÁMETROS OPTIMIZADOS
# ------------------------------------------------------------

print("\n=== 3.8.7 HIPERPARÁMETROS POR FOLD ===")

display(
    tabla_parametros_lr_cv.round(4)
)

print("\nFrecuencia de configuraciones seleccionadas:")

frecuencia_parametros = (
    tabla_parametros_lr_cv.groupby(
        ["Experimento", "C", "Percentile"]
    )
    .size()
    .reset_index(name="Numero_de_folds")
)

display(frecuencia_parametros)

# ------------------------------------------------------------
# 10. REFERENCIA DE CLASE MAYORITARIA
# ------------------------------------------------------------

print("\n=== 3.8.8 BASELINE ===")

display(
    pd.DataFrame([{
        "Modelo": "Clase mayoritaria",
        "F1_macro_medio": (
            tabla_baseline_lr_cv[
                "F1_macro"
            ].mean()
        ),
        "Accuracy_media": (
            tabla_baseline_lr_cv[
                "Accuracy"
            ].mean()
        ),
        "Recall_adverso_medio": (
            tabla_baseline_lr_cv[
                "Recall_adverso"
            ].mean()
        ),
        "ROC_AUC_medio": (
            tabla_baseline_lr_cv[
                "ROC_AUC"
            ].mean()
        )
    }]).round(4)
)

# ------------------------------------------------------------
# 11. VISUALIZACIÓN DEL RENDIMIENTO POR FOLD
# ------------------------------------------------------------

print("\n=== 3.8.9 GRÁFICOS ===")

grafico = tabla_pares_lr.copy()

grafico["Numero_fold"] = np.arange(
    1, len(grafico) + 1
)

plt.figure(figsize=(10, 4))

plt.plot(
    grafico["Numero_fold"],
    grafico["F1_macro_A"],
    marker="o",
    label="A - Estático"
)

plt.plot(
    grafico["Numero_fold"],
    grafico["F1_macro_B"],
    marker="s",
    label="B - Longitudinal"
)

plt.title(
    "F1 macro por fold externo - Regresión Logística"
)

plt.xlabel("Fold externo (dos repeticiones)")
plt.ylabel("F1 macro")
plt.xticks(range(1, 11))
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 4))

plt.bar(
    grafico["Numero_fold"],
    grafico["Delta_F1_macro"]
)

plt.axhline(0, linestyle="--")

plt.title(
    "Diferencia de F1 macro entre B y A"
)

plt.xlabel("Fold externo")
plt.ylabel("F1 macro B - A")
plt.xticks(range(1, 11))
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 12. CONTROLES FINALES
# ------------------------------------------------------------

print("\n=== 3.8.10 CONTROLES FINALES ===")

assert (
    tabla_lr_cv.groupby("Experimento")
    .size().eq(10).all()
)

assert (
    tabla_lr_cv[
        ["F1_macro", "Recall_adverso", "ROC_AUC"]
    ].notna().all().all()
)

assert (
    tabla_predicciones_lr_cv.groupby(
        ["Repeticion", "Experimento"]
    ).size().eq(len(y)).all()
)

print("Evaluaciones externas completadas:", len(tabla_lr_cv))

print(
    "Tiempo total:",
    round(duracion_global, 1),
    "segundos"
)

print(
    "Estudiantes compartidos entre "
    "entrenamiento y validación: 0"
)

print(
    "Conjunto de prueba independiente: NO utilizado"
)

print(
    "Diferencias reportadas: descriptivas, "
    "sin prueba de significancia todavía"
)

print("\n=== ACTIVIDAD 3.8 FINALIZADA ===")



### 3.8.11. Resultados e interpretación

Se realizó una validación cruzada anidada y repetida de Regresión Logística en la cohorte de la semana 2, utilizando cinco folds externos y dos repeticiones.

En cada fold externo se optimizaron los hiperparámetros mediante tres folds internos agrupados por estudiante.

Se completaron 20 evaluaciones externas: diez correspondientes al Experimento A y diez al Experimento B.

El conjunto de prueba independiente permaneció reservado.

**Resultados promedio**

| Métrica | Experimento A | Experimento B | Diferencia B - A |
|---|---:|---:|---:|
| F1 macro | 0,6076 | 0,6656 | +0,0580 |
| Recall adverso | 0,4755 | 0,5759 | +0,1005 |
| ROC-AUC | 0,6609 | 0,7315 | +0,0706 |
| Accuracy | 0,6223 | 0,6735 | +0,0513 |

La mejora relativa de F1 macro respecto al modelo estático fue aproximadamente del 9,55 %.

**Estabilidad entre folds**

El Experimento B superó al Experimento A en F1 macro, recall adverso, ROC-AUC y accuracy en las diez particiones externas.

La mejora media de F1 macro fue de 0,0580, con una desviación estándar de 0,0067 y valores comprendidos entre 0,0511 y 0,0675.

Estos resultados evidencian una mejora consistente bajo las diferentes particiones de validación utilizadas.

**Hiperparámetros y sobreajuste**

La búsqueda de hiperparámetros seleccionó distintas combinaciones de regularización y porcentaje de características entre folds.

Las diferencias entre F1 macro de entrenamiento y validación interna fueron reducidas, sin evidencia descriptiva de sobreajuste pronunciado para Regresión Logística.

**Comparación con la referencia**

La referencia de clase mayoritaria obtuvo un F1 macro promedio de 0,3542 y ROC-AUC de 0,5000.

Los dos experimentos de Regresión Logística superaron ampliamente esta referencia en las métricas principales.

### 3.8.12. Limitaciones de la interpretación

Los resultados corresponden exclusivamente a la semana 2 y al algoritmo de Regresión Logística.

Aunque se observaron mejoras positivas en los diez folds, las evaluaciones presentan dependencia estadística por el solapamiento de las muestras de entrenamiento y la repetición de matrículas en validación.

Por tanto, los resultados no permiten asignar automáticamente significancia estadística mediante una prueba ordinaria que suponga independencia entre folds.

También debe recordarse que la etiqueta adversa combina reprobación y retiro de un módulo académico, y no representa necesariamente deserción institucional.

### 3.8.13. Conclusión

La validación cruzada anidada y repetida proporciona evidencia empírica consistente de que las características longitudinales mejoran el rendimiento de Regresión Logística para la predicción de resultados académicos adversos desde la semana 2.

El incremento se observó en F1 macro, recall adverso, ROC-AUC y accuracy, manteniendo constantes las matrículas y las particiones de validación.

**Conclusión principal:** los resultados respaldan provisionalmente la hipótesis de que el comportamiento longitudinal aporta información predictiva adicional a las características estáticas. La conclusión definitiva requiere extender el análisis a Random Forest, las restantes ventanas temporales y una evaluación independiente.



### 3.9. Validación cruzada anidada y repetida — Random Forest

#### Objetivo

Evaluar el rendimiento y la estabilidad de Random Forest en los experimentos A y B durante la semana 2, utilizando el mismo procedimiento de validación cruzada anidada aplicado previamente a Regresión Logística.

#### Diseño experimental

Se utilizarán cinco folds externos y dos repeticiones, generados mediante `StratifiedGroupKFold`.

Cada fold externo será evaluado después de realizar una búsqueda de hiperparámetros con tres folds internos agrupados por estudiante.

Se mantendrán las mismas divisiones externas e internas empleadas en la Actividad 3.8 para permitir comparaciones entre algoritmos bajo condiciones equivalentes.

#### Espacio de optimización

Se conservará el espacio de búsqueda establecido en el experimento piloto:

- Número de árboles: 100.
- Profundidad máxima: 12 o sin límite explícito.
- Número mínimo de muestras por hoja: 2 o 5.
- Porcentaje de selección univariante: 80 %.

La optimización utilizará F1 macro como criterio principal.

#### Evaluación

Se calcularán accuracy, precision de la clase adversa, recall de la clase adversa, F1 adverso, F1 macro, F1 weighted y ROC-AUC.

También se registrarán:

- Matrices de confusión.
- Hiperparámetros seleccionados.
- Brechas entre entrenamiento y validación interna.
- Diferencias pareadas entre los experimentos B y A.
- Predicciones y probabilidades externas para análisis posteriores.

#### Hipótesis evaluada

Se investigará si las características longitudinales incrementan de manera consistente el rendimiento de Random Forest respecto de las características estáticas.

#### Consideraciones metodológicas

Las transformaciones y la selección de características se ajustarán dentro de cada fold de entrenamiento.

Las diez evaluaciones externas presentan dependencia estadística y se interpretarán inicialmente mediante estadísticas descriptivas de estabilidad.

El conjunto de prueba independiente permanecerá reservado.

Esta actividad no permitirá por sí sola confirmar la hipótesis general del proyecto, dado que todavía faltan las ventanas de las semanas 4, 6 y 8 y la evaluación final.


In [ ]:

# ============================================================
# PROYECTO FINAL DE MACHINE LEARNING
# ACTIVIDAD 3.9 - VALIDACIÓN CRUZADA ANIDADA
# RANDOM FOREST - SEMANA 2
# ============================================================

# Asistencia IA: ChatGPT, octubre de 2026.
# Prompt:
# "Realizar validación cruzada anidada y repetida de
# Random Forest para los experimentos A y B de OULAD,
# utilizando las mismas particiones externas e internas
# que Regresión Logística en semana 2. Optimizar
# hiperparámetros, medir F1 macro, recall y ROC-AUC,
# analizar sobreajuste y conservar predicciones sin
# utilizar el conjunto de prueba independiente."

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time

from sklearn.base import clone

from sklearn.model_selection import (
    StratifiedGroupKFold,
    GridSearchCV
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

from IPython.display import display

# ------------------------------------------------------------
# 1. CONFIGURACIÓN
# ------------------------------------------------------------

SEMANA_RF = 2
ALGORITMO_RF = "RandomForest"

N_FOLDS_EXTERNOS_RF = 5
N_REPETICIONES_RF = 2
N_FOLDS_INTERNOS_RF = 3

SEMILLA_INTERNA_RF = 137

GRID_RF = {
    "modelo__n_estimators": [100],
    "modelo__max_depth": [12, None],
    "modelo__min_samples_leaf": [2, 5],
    "seleccion__percentile": [80]
}

datos_rf = experimentos_oulad[SEMANA_RF]

X_A_rf = (
    datos_rf["X_A_train"]
    .reset_index(drop=True)
)

X_B_rf = (
    datos_rf["X_B_train"]
    .reset_index(drop=True)
)

y_rf = (
    datos_rf["y_train"]
    .reset_index(drop=True)
)

grupos_rf = (
    datos_rf["grupos_train"]
    .reset_index(drop=True)
)

claves_rf = (
    datos_rf["claves_train"]
    .reset_index(drop=True)
)

splits_externos_rf = (
    cv_splits_oulad[SEMANA_RF]
)

assert len(splits_externos_rf) == 10

assert (
    len(X_A_rf)
    == len(X_B_rf)
    == len(y_rf)
    == len(grupos_rf)
    == len(claves_rf)
)

print("=== 3.9.1 CONFIGURACIÓN ===")

print("Semana:", SEMANA_RF)
print("Algoritmo:", ALGORITMO_RF)
print("Matrículas:", len(y_rf))
print("Folds externos:", len(splits_externos_rf))
print("Folds internos:", N_FOLDS_INTERNOS_RF)
print("Experimentos:", ["A", "B"])
print("Métrica principal: F1 macro")
print("Grid:", GRID_RF)

# ------------------------------------------------------------
# 2. FUNCIÓN DE MÉTRICAS
# ------------------------------------------------------------

def calcular_metricas_rf(y_real, y_pred, probabilidades):

    return {
        "Accuracy": accuracy_score(
            y_real, y_pred
        ),
        "Precision_adverso": precision_score(
            y_real, y_pred,
            pos_label=1,
            zero_division=0
        ),
        "Recall_adverso": recall_score(
            y_real, y_pred,
            pos_label=1,
            zero_division=0
        ),
        "F1_adverso": f1_score(
            y_real, y_pred,
            pos_label=1,
            zero_division=0
        ),
        "F1_macro": f1_score(
            y_real, y_pred,
            average="macro",
            zero_division=0
        ),
        "F1_weighted": f1_score(
            y_real, y_pred,
            average="weighted",
            zero_division=0
        ),
        "ROC_AUC": roc_auc_score(
            y_real, probabilidades
        )
    }

# ------------------------------------------------------------
# 3. PREPARAR RESULTADOS
# ------------------------------------------------------------

resultados_rf_cv = []
parametros_rf_cv = []
predicciones_rf_cv = []

inicio_global_rf = time.perf_counter()

# ------------------------------------------------------------
# 4. VALIDACIÓN CRUZADA ANIDADA
# ------------------------------------------------------------

print("\n=== 3.9.2 VALIDACIÓN CRUZADA ANIDADA ===")

for numero_split, (idx_train, idx_val) in enumerate(
    splits_externos_rf,
    start=1
):

    repeticion = (
        (numero_split - 1) // N_FOLDS_EXTERNOS_RF + 1
    )

    fold = (
        (numero_split - 1) % N_FOLDS_EXTERNOS_RF + 1
    )

    print(
        f"\nRepetición {repeticion} | "
        f"Fold externo {fold}",
        flush=True
    )

    # --------------------------------------------------------
    # A. ETIQUETAS Y GRUPOS DEL FOLD
    # --------------------------------------------------------

    y_train_fold = (
        y_rf.iloc[idx_train]
        .reset_index(drop=True)
    )

    y_val_fold = (
        y_rf.iloc[idx_val]
        .reset_index(drop=True)
    )

    grupos_train_fold = (
        grupos_rf.iloc[idx_train]
        .reset_index(drop=True)
    )

    grupos_val_fold = (
        grupos_rf.iloc[idx_val]
        .reset_index(drop=True)
    )

    assert set(grupos_train_fold).isdisjoint(
        set(grupos_val_fold)
    )

    assert set(y_train_fold.unique()) == {0, 1}
    assert set(y_val_fold.unique()) == {0, 1}

    # --------------------------------------------------------
    # B. REPRODUCIR LA VALIDACIÓN INTERNA DE 3.8
    # --------------------------------------------------------

    splitter_interno = StratifiedGroupKFold(
        n_splits=N_FOLDS_INTERNOS_RF,
        shuffle=True,
        random_state=(
            SEMILLA_INTERNA_RF + numero_split - 1
        )
    )

    XA_train_fold = (
        X_A_rf.iloc[idx_train]
        .reset_index(drop=True)
    )

    splits_internos = list(
        splitter_interno.split(
            XA_train_fold,
            y_train_fold,
            groups=grupos_train_fold
        )
    )

    for idx_it, idx_iv in splits_internos:

        assert set(
            grupos_train_fold.iloc[idx_it]
        ).isdisjoint(
            set(grupos_train_fold.iloc[idx_iv])
        )

        assert set(
            y_train_fold.iloc[idx_it].unique()
        ) == {0, 1}

        assert set(
            y_train_fold.iloc[idx_iv].unique()
        ) == {0, 1}

    # --------------------------------------------------------
    # C. ENTRENAR EXPERIMENTOS A Y B
    # --------------------------------------------------------

    for experimento, X_completo in [
        ("A", X_A_rf),
        ("B", X_B_rf)
    ]:

        inicio_modelo = time.perf_counter()

        X_train_fold = (
            X_completo.iloc[idx_train]
            .reset_index(drop=True)
        )

        X_val_fold = (
            X_completo.iloc[idx_val]
            .reset_index(drop=True)
        )

        pipeline = clone(
            pipelines_oulad[
                (
                    SEMANA_RF,
                    experimento,
                    ALGORITMO_RF
                )
            ]
        )

        # El pipeline incluye imputación,
        # codificación, escalamiento y selección.
        #
        # n_jobs=1 evita combinar procesos paralelos
        # externos con los propios hilos del bosque.

        busqueda = GridSearchCV(
            estimator=pipeline,
            param_grid=GRID_RF,
            scoring="f1_macro",
            cv=splits_internos,
            refit=True,
            n_jobs=1,
            verbose=0,
            return_train_score=True,
            error_score="raise"
        )

        # Exclusivamente entrenamiento externo.
        busqueda.fit(
            X_train_fold,
            y_train_fold
        )

        mejor_modelo = busqueda.best_estimator_

        # ----------------------------------------------------
        # D. EVALUACIÓN EXTERNA
        # ----------------------------------------------------

        y_pred = mejor_modelo.predict(
            X_val_fold
        )

        indice_adverso = list(
            mejor_modelo.classes_
        ).index(1)

        prob_adverso = (
            mejor_modelo.predict_proba(
                X_val_fold
            )[:, indice_adverso]
        )

        metricas = calcular_metricas_rf(
            y_val_fold,
            y_pred,
            prob_adverso
        )

        cm = confusion_matrix(
            y_val_fold,
            y_pred,
            labels=[0, 1]
        )

        duracion = (
            time.perf_counter() - inicio_modelo
        )

        # ----------------------------------------------------
        # E. REGISTRAR MÉTRICAS
        # ----------------------------------------------------

        resultados_rf_cv.append({
            "Repeticion": repeticion,
            "Fold": fold,
            "Experimento": experimento,
            "N_train": len(idx_train),
            "N_validacion": len(idx_val),
            "F1_CV_interno": busqueda.best_score_,
            **metricas,
            "VN": int(cm[0, 0]),
            "FP": int(cm[0, 1]),
            "FN": int(cm[1, 0]),
            "VP": int(cm[1, 1]),
            "Tiempo_segundos": duracion
        })

        # ----------------------------------------------------
        # F. REGISTRAR HIPERPARÁMETROS
        # ----------------------------------------------------

        mejor_idx = busqueda.best_index_

        f1_train_interno = (
            busqueda.cv_results_[
                "mean_train_score"
            ][mejor_idx]
        )

        f1_val_interno = (
            busqueda.cv_results_[
                "mean_test_score"
            ][mejor_idx]
        )

        parametros_rf_cv.append({
            "Repeticion": repeticion,
            "Fold": fold,
            "Experimento": experimento,
            "Max_depth": busqueda.best_params_[
                "modelo__max_depth"
            ],
            "Min_samples_leaf": busqueda.best_params_[
                "modelo__min_samples_leaf"
            ],
            "N_estimators": busqueda.best_params_[
                "modelo__n_estimators"
            ],
            "Percentile": busqueda.best_params_[
                "seleccion__percentile"
            ],
            "F1_train_interno": f1_train_interno,
            "F1_val_interno": f1_val_interno,
            "Brecha_train_val": (
                f1_train_interno - f1_val_interno
            )
        })

        # ----------------------------------------------------
        # G. CONSERVAR PREDICCIONES Y CLAVES COMPUESTAS
        # ----------------------------------------------------

        identificadores_val = (
            claves_rf.iloc[idx_val]
            .reset_index(drop=True)
        )

        df_predicciones = identificadores_val.copy()

        df_predicciones["Repeticion"] = repeticion
        df_predicciones["Fold"] = fold
        df_predicciones["Experimento"] = experimento

        df_predicciones["y_real"] = (
            y_val_fold.to_numpy()
        )

        df_predicciones["y_pred"] = y_pred

        df_predicciones["prob_adverso"] = (
            prob_adverso
        )

        predicciones_rf_cv.append(
            df_predicciones
        )

        print(
            f"  {experimento}: "
            f"F1={metricas['F1_macro']:.4f} | "
            f"Recall={metricas['Recall_adverso']:.4f} | "
            f"AUC={metricas['ROC_AUC']:.4f} | "
            f"{duracion:.1f}s",
            flush=True
        )

# ------------------------------------------------------------
# 5. CONSOLIDAR TABLAS
# ------------------------------------------------------------

tabla_rf_cv = pd.DataFrame(
    resultados_rf_cv
)

tabla_parametros_rf_cv = pd.DataFrame(
    parametros_rf_cv
)

tabla_predicciones_rf_cv = pd.concat(
    predicciones_rf_cv,
    ignore_index=True
)

duracion_global_rf = (
    time.perf_counter() - inicio_global_rf
)

assert len(tabla_rf_cv) == 20

assert len(tabla_parametros_rf_cv) == 20

print("\n=== 3.9.3 RESULTADOS POR FOLD ===")

display(
    tabla_rf_cv[
        [
            "Repeticion",
            "Fold",
            "Experimento",
            "F1_macro",
            "Recall_adverso",
            "ROC_AUC",
            "FN",
            "FP"
        ]
    ].round(4)
)

# ------------------------------------------------------------
# 6. ESTADÍSTICAS POR EXPERIMENTO
# ------------------------------------------------------------

print("\n=== 3.9.4 RESUMEN POR EXPERIMENTO ===")

metricas_resumen_rf = [
    "Accuracy",
    "Precision_adverso",
    "Recall_adverso",
    "F1_adverso",
    "F1_macro",
    "F1_weighted",
    "ROC_AUC"
]

resumen_rf = []

for experimento in ["A", "B"]:

    sub = tabla_rf_cv.loc[
        tabla_rf_cv["Experimento"] == experimento
    ]

    fila = {"Experimento": experimento}

    for metrica in metricas_resumen_rf:

        fila[f"{metrica}_media"] = (
            sub[metrica].mean()
        )

        fila[f"{metrica}_std"] = (
            sub[metrica].std(ddof=1)
        )

    resumen_rf.append(fila)

tabla_resumen_rf_cv = pd.DataFrame(resumen_rf)

display(
    tabla_resumen_rf_cv[
        [
            "Experimento",
            "F1_macro_media",
            "F1_macro_std",
            "Recall_adverso_media",
            "Recall_adverso_std",
            "ROC_AUC_media",
            "ROC_AUC_std",
            "Accuracy_media"
        ]
    ].round(4)
)

# ------------------------------------------------------------
# 7. COMPARACIÓN PAREADA B - A
# ------------------------------------------------------------

print("\n=== 3.9.5 DIFERENCIAS PAREADAS B - A ===")

metricas_pares_rf = [
    "F1_macro",
    "Recall_adverso",
    "ROC_AUC",
    "Accuracy"
]

columnas_pares_rf = (
    ["Repeticion", "Fold"] + metricas_pares_rf
)

pares_A_rf = tabla_rf_cv.loc[
    tabla_rf_cv["Experimento"] == "A",
    columnas_pares_rf
].copy()

pares_B_rf = tabla_rf_cv.loc[
    tabla_rf_cv["Experimento"] == "B",
    columnas_pares_rf
].copy()

pares_A_rf = pares_A_rf.rename(
    columns={
        metrica: f"{metrica}_A"
        for metrica in metricas_pares_rf
    }
)

pares_B_rf = pares_B_rf.rename(
    columns={
        metrica: f"{metrica}_B"
        for metrica in metricas_pares_rf
    }
)

tabla_pares_rf = pares_A_rf.merge(
    pares_B_rf,
    on=["Repeticion", "Fold"],
    how="inner",
    validate="one_to_one"
)

for metrica in metricas_pares_rf:

    tabla_pares_rf[f"Delta_{metrica}"] = (
        tabla_pares_rf[f"{metrica}_B"]
        - tabla_pares_rf[f"{metrica}_A"]
    )

assert len(tabla_pares_rf) == 10

display(
    tabla_pares_rf[
        [
            "Repeticion",
            "Fold",
            "F1_macro_A",
            "F1_macro_B",
            "Delta_F1_macro",
            "Delta_Recall_adverso",
            "Delta_ROC_AUC"
        ]
    ].round(4)
)

# ------------------------------------------------------------
# 8. ESTABILIDAD
# ------------------------------------------------------------

print("\n=== 3.9.6 ESTABILIDAD DE LAS DIFERENCIAS ===")

estabilidad_rf = []

for metrica in metricas_pares_rf:

    delta = tabla_pares_rf[
        f"Delta_{metrica}"
    ]

    estabilidad_rf.append({
        "Metrica": metrica,
        "Delta_medio_B_menos_A": delta.mean(),
        "Desviacion_deltas": delta.std(ddof=1),
        "Delta_minimo": delta.min(),
        "Delta_maximo": delta.max(),
        "Folds_B_supera_A": int(
            (delta > 0).sum()
        ),
        "Folds_empatados": int(
            np.isclose(delta, 0).sum()
        ),
        "Folds_B_inferior_A": int(
            (delta < 0).sum()
        )
    })

tabla_estabilidad_rf = pd.DataFrame(
    estabilidad_rf
)

display(tabla_estabilidad_rf.round(4))

# ------------------------------------------------------------
# 9. HIPERPARÁMETROS Y SOBREAJUSTE
# ------------------------------------------------------------

print("\n=== 3.9.7 HIPERPARÁMETROS POR FOLD ===")

display(
    tabla_parametros_rf_cv.round(4)
)

print("\nBrecha media entrenamiento - validación:")

brechas_rf = (
    tabla_parametros_rf_cv.groupby("Experimento")
    .agg(
        Brecha_media=("Brecha_train_val", "mean"),
        Brecha_maxima=("Brecha_train_val", "max"),
        F1_train_medio=("F1_train_interno", "mean"),
        F1_validacion_medio=("F1_val_interno", "mean")
    )
    .reset_index()
)

display(brechas_rf.round(4))

print("\nFrecuencia de hiperparámetros seleccionados:")

frecuencia_rf = (
    tabla_parametros_rf_cv
    .assign(
        Max_depth=lambda d: (
            d["Max_depth"].astype(str)
        )
    )
    .groupby(
        [
            "Experimento",
            "Max_depth",
            "Min_samples_leaf",
            "Percentile"
        ]
    )
    .size()
    .reset_index(name="Numero_de_folds")
)

display(frecuencia_rf)

# ------------------------------------------------------------
# 10. VISUALIZACIONES
# ------------------------------------------------------------

print("\n=== 3.9.8 GRÁFICOS ===")

grafico_rf = tabla_pares_rf.copy()

grafico_rf["Numero_fold"] = np.arange(
    1, len(grafico_rf) + 1
)

plt.figure(figsize=(10, 4))

plt.plot(
    grafico_rf["Numero_fold"],
    grafico_rf["F1_macro_A"],
    marker="o",
    label="A - Estático"
)

plt.plot(
    grafico_rf["Numero_fold"],
    grafico_rf["F1_macro_B"],
    marker="s",
    label="B - Longitudinal"
)

plt.title(
    "F1 macro por fold externo - Random Forest"
)

plt.xlabel("Fold externo (dos repeticiones)")
plt.ylabel("F1 macro")
plt.xticks(range(1, 11))
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 4))

plt.bar(
    grafico_rf["Numero_fold"],
    grafico_rf["Delta_F1_macro"]
)

plt.axhline(
    0,
    color="black",
    linestyle="--"
)

plt.title(
    "Diferencia de F1 macro entre B y A - Random Forest"
)

plt.xlabel("Fold externo")
plt.ylabel("F1 macro B - A")
plt.xticks(range(1, 11))
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 11. CONTROLES FINALES
# ------------------------------------------------------------

print("\n=== 3.9.9 CONTROLES FINALES ===")

assert (
    tabla_rf_cv.groupby("Experimento")
    .size().eq(10).all()
)

assert (
    tabla_rf_cv[
        ["F1_macro", "Recall_adverso", "ROC_AUC"]
    ].notna().all().all()
)

assert (
    tabla_predicciones_rf_cv.groupby(
        ["Repeticion", "Experimento"]
    ).size().eq(len(y_rf)).all()
)

assert not (
    tabla_predicciones_rf_cv.duplicated(
        subset=[
            "Repeticion",
            "Experimento",
            "code_module",
            "code_presentation",
            "id_student"
        ]
    ).any()
)

assert set(
    datos_rf["grupos_train"]
).isdisjoint(
    set(datos_rf["grupos_test"])
)

print("Evaluaciones externas completadas:", len(tabla_rf_cv))

print(
    "Tiempo total:",
    round(duracion_global_rf, 1),
    "segundos"
)

print(
    "Máximo de estudiantes compartidos "
    "entre entrenamiento y validación: 0"
)

print("Conjunto de prueba independiente: NO utilizado")

print(
    "Resultados disponibles para "
    "comparación posterior con Regresión Logística."
)

print("\n=== ACTIVIDAD 3.9 FINALIZADA ===")



### 3.9.10. Resultados e interpretación

Se completó la validación cruzada anidada y repetida de Random Forest para la semana 2 de OULAD, utilizando cinco folds externos, dos repeticiones y tres folds internos de optimización.

Se evaluaron los experimentos A (características estáticas) y B (características estáticas y longitudinales), manteniendo las mismas matrículas, etiquetas y particiones de estudiantes.

**Resultados promedio**

| Métrica | Experimento A | Experimento B | Diferencia B - A |
|---|---:|---:|---:|
| F1 macro | 0,5986 | 0,6566 | +0,0580 |
| Recall adverso | 0,4576 | 0,5417 | +0,0842 |
| ROC-AUC | 0,6522 | 0,7204 | +0,0682 |
| Accuracy | 0,6152 | 0,6676 | +0,0525 |

El incremento relativo de F1 macro fue aproximadamente del 9,7 % respecto al modelo estático.

**Estabilidad de la mejora**

El Experimento B superó al Experimento A en los diez folds externos para F1 macro, recall adverso, ROC-AUC y accuracy.

La mejora media de F1 macro fue 0,0580, con desviación estándar de 0,0100 y valores comprendidos entre 0,0365 y 0,0713.

La consistencia del resultado en las diez particiones proporciona evidencia descriptiva favorable al aporte de las características longitudinales.

**Análisis de sobreajuste**

El F1 macro medio de entrenamiento interno fue de 0,6811 para A y de 0,8099 para B, frente a valores de validación interna de 0,5996 y 0,6549, respectivamente.

Las brechas medias entre entrenamiento y validación fueron de 0,0814 en A y 0,1550 en B.

Estos resultados sugieren sobreajuste, especialmente en el Experimento B, y justifican examinar la regularización y complejidad del modelo en etapas posteriores.

**Hiperparámetros**

La optimización seleccionó mayoritariamente un mínimo de cinco muestras por hoja.

En el Experimento B, siete de los diez folds seleccionaron árboles sin un límite explícito de profundidad, mientras que tres seleccionaron una profundidad máxima de 12.

Estos resultados reflejan las mejores configuraciones dentro del espacio explorado y no garantizan optimalidad global.

**Comparación preliminar entre algoritmos**

La regresión logística obtuvo valores medios superiores a Random Forest tanto en el Experimento A como en el B.

En el Experimento B, la regresión logística alcanzó un F1 macro de 0,6656 y ROC-AUC de 0,7315, frente a 0,6566 y 0,7204 de Random Forest.

La comparación formal de las diferencias pareadas entre algoritmos se realizará en la siguiente actividad.

### 3.9.11. Limitaciones

La validación corresponde exclusivamente a la semana 2.

Los diez resultados externos presentan dependencia estadística por el solapamiento de las muestras de entrenamiento.

Por esta razón, la consistencia de las mejoras no debe confundirse con una prueba formal de significancia estadística.

Además, la variable objetivo combina reprobación y retiro de un módulo, por lo que todavía es necesario examinar el comportamiento de los clasificadores frente a cada resultado académico específico.

### 3.9.12. Conclusión

Random Forest presentó mejoras consistentes de rendimiento al incorporar características longitudinales, reproduciendo el patrón previamente observado con regresión logística.

No obstante, mostró mayores brechas entre entrenamiento y validación, especialmente en su configuración longitudinal.

**Conclusión principal:** la incorporación de características longitudinales mejora de manera consistente el rendimiento de Random Forest en la semana 2, aunque la regresión logística presenta mejores promedios de generalización y menores señales de sobreajuste bajo las configuraciones evaluadas.


In [ ]:

# ============================================================
# ACTIVIDAD 3.9.13
# VISUALIZACIÓN DEL SOBREAJUSTE - RANDOM FOREST B
# ============================================================

# Objetivo:
# Comparar F1 macro de entrenamiento interno,
# validación interna y validación externa en los
# diez folds de Random Forest B.
#
# No se vuelven a entrenar modelos.
# Se utilizan los resultados de la Actividad 3.9.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

# ------------------------------------------------------------
# 1. RECUPERAR RESULTADOS DE RANDOM FOREST B
# ------------------------------------------------------------

datos_internos = (
    tabla_parametros_rf_cv.loc[
        tabla_parametros_rf_cv["Experimento"] == "B",
        [
            "Repeticion",
            "Fold",
            "F1_train_interno",
            "F1_val_interno",
            "Brecha_train_val"
        ]
    ]
    .copy()
)

datos_externos = (
    tabla_rf_cv.loc[
        tabla_rf_cv["Experimento"] == "B",
        [
            "Repeticion",
            "Fold",
            "F1_macro"
        ]
    ]
    .rename(
        columns={
            "F1_macro": "F1_val_externa"
        }
    )
    .copy()
)

# Unir por repetición y fold.
tabla_sobreajuste_rf_B = (
    datos_internos.merge(
        datos_externos,
        on=["Repeticion", "Fold"],
        how="inner",
        validate="one_to_one"
    )
    .sort_values(
        ["Repeticion", "Fold"]
    )
    .reset_index(drop=True)
)

assert len(tabla_sobreajuste_rf_B) == 10

assert tabla_sobreajuste_rf_B[
    [
        "F1_train_interno",
        "F1_val_interno",
        "F1_val_externa"
    ]
].notna().all().all()

# ------------------------------------------------------------
# 2. CALCULAR PROMEDIOS
# ------------------------------------------------------------

media_train = (
    tabla_sobreajuste_rf_B[
        "F1_train_interno"
    ].mean()
)

media_val_interna = (
    tabla_sobreajuste_rf_B[
        "F1_val_interno"
    ].mean()
)

media_val_externa = (
    tabla_sobreajuste_rf_B[
        "F1_val_externa"
    ].mean()
)

brecha_media = (
    media_train - media_val_interna
)

print("=== SOBREAJUSTE RANDOM FOREST B ===")
print(f"F1 entrenamiento interno: {media_train:.4f}")
print(f"F1 validación interna:    {media_val_interna:.4f}")
print(f"F1 validación externa:    {media_val_externa:.4f}")
print(f"Brecha media:             {brecha_media:.4f}")

# ------------------------------------------------------------
# 3. PREPARAR EJE HORIZONTAL
# ------------------------------------------------------------

x = np.arange(1, 11)

etiquetas = [
    f"R{int(r)}-F{int(f)}"
    for r, f in zip(
        tabla_sobreajuste_rf_B["Repeticion"],
        tabla_sobreajuste_rf_B["Fold"]
    )
]

train = tabla_sobreajuste_rf_B[
    "F1_train_interno"
].to_numpy()

val_interna = tabla_sobreajuste_rf_B[
    "F1_val_interno"
].to_numpy()

val_externa = tabla_sobreajuste_rf_B[
    "F1_val_externa"
].to_numpy()

# ------------------------------------------------------------
# 4. GRAFICAR SOBREAJUSTE
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(12, 5.5))

# Entrenamiento
ax.plot(
    x,
    train,
    marker="o",
    markersize=7,
    linewidth=2.5,
    label=f"Entrenamiento interno (media={media_train:.4f})"
)

# Validación interna
ax.plot(
    x,
    val_interna,
    marker="s",
    markersize=7,
    linewidth=2.5,
    label=f"Validación interna (media={media_val_interna:.4f})"
)

# Validación externa
ax.plot(
    x,
    val_externa,
    marker="^",
    markersize=7,
    linewidth=2,
    linestyle="--",
    label=f"Validación externa (media={media_val_externa:.4f})"
)

# Sombrear la brecha entrenamiento-validación interna
ax.fill_between(
    x,
    val_interna,
    train,
    alpha=0.12,
    label="Brecha de generalización"
)

# Línea horizontal de referencia
ax.axhline(
    media_val_interna,
    linestyle=":",
    linewidth=1,
    alpha=0.7
)

# ------------------------------------------------------------
# 5. CONFIGURACIÓN VISUAL
# ------------------------------------------------------------

ax.set_title(
    "Evidencia de sobreajuste - Random Forest B\n"
    "Validación cruzada anidada, semana 2",
    fontsize=14,
    fontweight="bold",
    pad=15
)

ax.set_xlabel(
    "Fold externo (R = repetición, F = fold)",
    fontsize=11
)

ax.set_ylabel(
    "F1 macro",
    fontsize=11
)

ax.set_xticks(x)
ax.set_xticklabels(etiquetas)

ax.set_ylim(0.50, 0.90)

ax.grid(
    axis="y",
    linestyle="--",
    alpha=0.25
)

ax.legend(
    loc="upper right",
    fontsize=9
)

# Anotación del resultado central
ax.text(
    0.02,
    0.04,
    f"Brecha media entrenamiento - validación interna: "
    f"{brecha_media:.4f}",
    transform=ax.transAxes,
    fontsize=10,
    fontweight="bold"
)

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 6. MOSTRAR TABLA DE AUDITORÍA
# ------------------------------------------------------------

print("\n=== DETALLE DEL SOBREAJUSTE POR FOLD ===")

display(
    tabla_sobreajuste_rf_B.round(4)
)

print("\n=== INTERPRETACIÓN AUTOMÁTICA ===")

if brecha_media >= 0.10:
    print(
        "Se observa una brecha considerable entre "
        "entrenamiento y validación interna."
    )
    print(
        "Este resultado es compatible con sobreajuste."
    )
else:
    print(
        "La brecha promedio es relativamente reducida."
    )

print(
    "\nNota: este gráfico NO es una curva de aprendizaje."
)
print(
    "Compara el rendimiento entre entrenamiento y "
    "validación en diez folds anidados."
)



### 3.9.14. Análisis gráfico del sobreajuste de Random Forest B

Se construyó un gráfico comparativo del F1 macro obtenido durante el entrenamiento interno, la validación interna y la validación externa de Random Forest B, utilizando los diez folds de la validación cruzada anidada.

Los resultados medios fueron:

| Evaluación | F1 macro medio |
|---|---:|
| Entrenamiento interno | 0,8099 |
| Validación interna | 0,6549 |
| Validación externa | 0,6566 |
| Brecha entrenamiento-validación interna | 0,1550 |

Se observó una separación considerable entre el rendimiento obtenido durante el entrenamiento y la validación interna.

Esta diferencia constituye evidencia descriptiva compatible con sobreajuste: el modelo presenta una mayor capacidad para clasificar observaciones utilizadas en su ajuste que matrículas correspondientes a estudiantes no vistos.

El F1 macro de validación externa fue cercano al obtenido en validación interna, lo que sugiere consistencia entre ambas estimaciones de generalización bajo el procedimiento utilizado.

Sin embargo, el sobreajuste no invalida las mejoras observadas al incorporar las características longitudinales. Random Forest B superó al experimento estático A en los diez folds externos.

**Conclusión:** Random Forest B presenta una brecha de generalización considerable, por lo que resulta pertinente investigar configuraciones de regularización más estrictas. Cualquier modificación deberá optimizarse dentro de los folds de entrenamiento, evitando adaptar los hiperparámetros directamente a los resultados de validación externa.



### 3.9.15. Interpretación del gráfico de sobreajuste

El gráfico evidencia una diferencia sistemática entre el rendimiento de entrenamiento y validación de Random Forest B durante los diez folds de validación cruzada anidada.

El F1 macro medio de entrenamiento fue de 0,8099, mientras que la validación interna alcanzó 0,6549 y la validación externa 0,6566. La brecha media entre entrenamiento y validación interna fue de 0,1550.

Esta diferencia constituye evidencia descriptiva compatible con sobreajuste, dado que el modelo obtiene un rendimiento considerablemente superior sobre las observaciones utilizadas durante el entrenamiento.

Sin embargo, los valores de validación interna y externa se mantuvieron próximos, lo que sugiere una estimación relativamente consistente de su capacidad de generalización dentro del procedimiento experimental.

Adicionalmente, se observó que las configuraciones con profundidad máxima restringida a 12 niveles presentaron menores valores de F1 durante el entrenamiento que varias configuraciones sin límite explícito de profundidad, sin diferencias proporcionales evidentes en validación.

Este comportamiento sugiere que la complejidad del bosque contribuye a la brecha de generalización y justifica un análisis posterior de regularización.

**Conclusión:** Random Forest B presenta señales claras de sobreajuste relativo, aunque conserva un rendimiento externo superior al de Random Forest A. Por tanto, el sobreajuste observado no invalida el aporte predictivo de las características longitudinales, pero evidencia la necesidad de investigar configuraciones que equilibren mejor el ajuste y la generalización.



### 3.10. Comparación integrada de algoritmos y análisis de errores

#### Objetivo

Comparar los resultados de Regresión Logística y Random Forest bajo los mismos folds externos de la semana 2, y estudiar por separado su comportamiento frente a reprobación y retiro futuro del módulo.

#### Comparación entre algoritmos

Se utilizarán las métricas externas generadas mediante validación cruzada anidada en las Actividades 3.8 y 3.9.

Para cada experimento se calcularán las diferencias pareadas entre Regresión Logística y Random Forest en:

- F1 macro.
- Recall de la clase adversa.
- ROC-AUC.
- Accuracy.

También se examinará en cuántos folds cada algoritmo supera al otro.

#### Análisis de errores

Se compararán las predicciones de los experimentos A y B sobre las mismas matrículas, identificando:

- Casos adversos recuperados por B que A no identificó.
- Casos adversos identificados por A que B deja de identificar.
- Falsas alertas evitadas por B.
- Nuevas falsas alertas introducidas por B.

Se mantendrá la unidad de análisis como matrícula académica.

#### Resultados académicos específicos

La etiqueta adversa incluye dos resultados:

- `Fail`: reprobación del módulo.
- `Withdrawn`: retiro del módulo.

Se analizará la tasa de identificación de cada resultado utilizando exclusivamente las predicciones externas generadas durante la validación cruzada.

Para los resultados favorables (`Pass` y `Distinction`) se calculará la proporción de falsas alertas.

#### Anticipación de retiros

En los casos `Withdrawn`, se examinará el tiempo transcurrido entre el momento de predicción (día 14) y la fecha real de desregistro.

Esta información se utilizará exclusivamente para evaluar anticipación y nunca para construir características del modelo.

La anticipación observada no equivale a la demostración de una intervención temprana eficaz. Para evaluarla deberán considerarse también la precisión de las alertas y la utilidad práctica del sistema.

#### Precauciones metodológicas

Se reconstruirán las claves compuestas de matrícula para evitar emparejamientos ambiguos entre estudiantes que tengan más de una inscripción.

Cada repetición externa aporta una predicción fuera de muestra por matrícula y configuración. Las dos repeticiones corresponden a observaciones relacionadas y no deberán tratarse como muestras independientes.

No se volverán a entrenar clasificadores y no se utilizará el conjunto de prueba independiente.


In [ ]:

# ============================================================
# PROYECTO FINAL DE MACHINE LEARNING
# ACTIVIDAD 3.10 - COMPARACIÓN INTEGRADA Y ANÁLISIS DE ERRORES
# REGRESIÓN LOGÍSTICA VS RANDOM FOREST - SEMANA 2
# ============================================================

# Asistencia IA: ChatGPT, octubre de 2026.
# Prompt:
# "Comparar de forma pareada los resultados de Regresión
# Logística y Random Forest obtenidos en validación
# cruzada anidada de OULAD. Examinar casos adversos
# recuperados, falsas alertas, reprobaciones y retiros
# futuros, reconstruyendo claves compuestas de matrícula
# para evitar emparejamientos incorrectos. Evaluar
# anticipación de retiro sin volver a entrenar modelos
# ni utilizar el conjunto de prueba independiente."

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

# ------------------------------------------------------------
# 1. CONFIGURACIÓN
# ------------------------------------------------------------

SEMANA_ANALISIS = 2
DIA_CORTE = 14
N_FOLDS = 5

CLAVES = [
    "code_module",
    "code_presentation",
    "id_student"
]

COLUMNAS_METRICAS = [
    "F1_macro",
    "Recall_adverso",
    "ROC_AUC",
    "Accuracy"
]

datos_analisis = experimentos_oulad[SEMANA_ANALISIS]

claves_analisis = (
    datos_analisis["claves_train"]
    .reset_index(drop=True)
)

y_analisis = (
    datos_analisis["y_train"]
    .reset_index(drop=True)
)

splits_analisis = cv_splits_oulad[SEMANA_ANALISIS]

assert len(splits_analisis) == 10

print("=== 3.10.1 CONFIGURACIÓN ===")
print("Semana:", SEMANA_ANALISIS)
print("Día de predicción:", DIA_CORTE)
print("Folds externos:", len(splits_analisis))
print("Algoritmos: Regresión Logística y Random Forest")
print("Experimentos: A y B")
print("Reentrenamiento: NO")

# ------------------------------------------------------------
# 2. CONSOLIDAR MÉTRICAS EXTERNAS
# ------------------------------------------------------------

resultados_lr = tabla_lr_cv.copy()
resultados_rf = tabla_rf_cv.copy()

resultados_lr["Algoritmo"] = "LogisticRegression"
resultados_rf["Algoritmo"] = "RandomForest"

tabla_conjunta_s2 = pd.concat(
    [resultados_lr, resultados_rf],
    ignore_index=True
)

assert len(tabla_conjunta_s2) == 40

assert not tabla_conjunta_s2.duplicated(
    subset=[
        "Algoritmo",
        "Experimento",
        "Repeticion",
        "Fold"
    ]
).any()

print("\n=== 3.10.2 RESUMEN DE LOS CUATRO MODELOS ===")

tabla_resumen_conjunto = (
    tabla_conjunta_s2.groupby(
        ["Algoritmo", "Experimento"]
    )
    .agg(
        F1_macro_medio=("F1_macro", "mean"),
        F1_macro_std=("F1_macro", "std"),
        Recall_adverso_medio=("Recall_adverso", "mean"),
        ROC_AUC_medio=("ROC_AUC", "mean"),
        Accuracy_media=("Accuracy", "mean"),
        Folds=("Fold", "size")
    )
    .reset_index()
)

display(tabla_resumen_conjunto.round(4))

# ------------------------------------------------------------
# 3. COMPARAR LR FRENTE A RF POR FOLD
# ------------------------------------------------------------

print("\n=== 3.10.3 COMPARACIÓN PAREADA LR - RF ===")

columnas_comparacion = (
    ["Repeticion", "Fold", "Experimento"]
    + COLUMNAS_METRICAS
)

lr = resultados_lr[
    columnas_comparacion
].rename(
    columns={
        m: f"{m}_LR"
        for m in COLUMNAS_METRICAS
    }
)

rf = resultados_rf[
    columnas_comparacion
].rename(
    columns={
        m: f"{m}_RF"
        for m in COLUMNAS_METRICAS
    }
)

tabla_pares_algoritmos = lr.merge(
    rf,
    on=["Repeticion", "Fold", "Experimento"],
    validate="one_to_one"
)

assert len(tabla_pares_algoritmos) == 20

for metrica in COLUMNAS_METRICAS:

    tabla_pares_algoritmos[f"Delta_{metrica}_LR_RF"] = (
        tabla_pares_algoritmos[f"{metrica}_LR"]
        - tabla_pares_algoritmos[f"{metrica}_RF"]
    )

display(
    tabla_pares_algoritmos[
        [
            "Repeticion",
            "Fold",
            "Experimento",
            "F1_macro_LR",
            "F1_macro_RF",
            "Delta_F1_macro_LR_RF",
            "Delta_Recall_adverso_LR_RF",
            "Delta_ROC_AUC_LR_RF"
        ]
    ].round(4)
)

# ------------------------------------------------------------
# 4. ESTABILIDAD DE DIFERENCIAS ENTRE ALGORITMOS
# ------------------------------------------------------------

print("\n=== 3.10.4 ESTABILIDAD LR VS RF ===")

resumen_diferencias_algoritmos = []

for experimento in ["A", "B"]:

    sub = tabla_pares_algoritmos.loc[
        tabla_pares_algoritmos["Experimento"] == experimento
    ]

    for metrica in COLUMNAS_METRICAS:

        delta = sub[f"Delta_{metrica}_LR_RF"]

        resumen_diferencias_algoritmos.append({
            "Experimento": experimento,
            "Metrica": metrica,
            "Delta_medio_LR_RF": delta.mean(),
            "Delta_std": delta.std(ddof=1),
            "LR_supera_RF": int((delta > 0).sum()),
            "RF_supera_LR": int((delta < 0).sum()),
            "Empates": int(np.isclose(delta, 0).sum())
        })

tabla_diferencias_algoritmos = pd.DataFrame(
    resumen_diferencias_algoritmos
)

display(tabla_diferencias_algoritmos.round(4))

# ------------------------------------------------------------
# 5. RECONSTRUIR CLAVES DE PREDICCIONES LR
# ------------------------------------------------------------

# En 3.8 se guardó id_student, pero no las claves
# completas de la matrícula. Se reconstruyen usando
# los índices originales de validación.
#
# Es indispensable verificar orden, id_student
# y etiqueta antes de añadir las claves.

print("\n=== 3.10.5 INTEGRIDAD DE PREDICCIONES ===")

pred_lr_original = tabla_predicciones_lr_cv.copy()

pred_lr_completas = []

for numero_split, (_, idx_val) in enumerate(
    splits_analisis,
    start=1
):

    repeticion = (numero_split - 1) // N_FOLDS + 1
    fold = (numero_split - 1) % N_FOLDS + 1

    claves_esperadas = (
        claves_analisis.iloc[idx_val]
        .reset_index(drop=True)
    )

    y_esperada = (
        y_analisis.iloc[idx_val]
        .to_numpy()
    )

    for experimento in ["A", "B"]:

        mascara = (
            pred_lr_original["Repeticion"].eq(repeticion)
            & pred_lr_original["Fold"].eq(fold)
            & pred_lr_original["Experimento"].eq(experimento)
        )

        bloque = (
            pred_lr_original.loc[mascara]
            .reset_index(drop=True)
        )

        assert len(bloque) == len(idx_val), (
            "Cantidad inesperada de predicciones LR."
        )

        assert np.array_equal(
            bloque["id_student"].to_numpy(),
            claves_esperadas["id_student"].to_numpy()
        ), "No coincide el orden de estudiantes en LR."

        assert np.array_equal(
            bloque["y_real"].to_numpy(),
            y_esperada
        ), "No coinciden las etiquetas en LR."

        bloque["code_module"] = (
            claves_esperadas["code_module"].to_numpy()
        )

        bloque["code_presentation"] = (
            claves_esperadas["code_presentation"].to_numpy()
        )

        bloque["Algoritmo"] = "LogisticRegression"

        pred_lr_completas.append(bloque)

pred_lr_completas = pd.concat(
    pred_lr_completas,
    ignore_index=True
)

# Random Forest ya conserva las claves completas.
pred_rf_completas = tabla_predicciones_rf_cv.copy()

pred_rf_completas["Algoritmo"] = "RandomForest"

# ------------------------------------------------------------
# 6. CONSOLIDAR PREDICCIONES
# ------------------------------------------------------------

columnas_predicciones = (
    CLAVES
    + [
        "Repeticion",
        "Fold",
        "Experimento",
        "Algoritmo",
        "y_real",
        "y_pred",
        "prob_adverso"
    ]
)

predicciones_conjuntas_s2 = pd.concat(
    [
        pred_lr_completas[columnas_predicciones],
        pred_rf_completas[columnas_predicciones]
    ],
    ignore_index=True
)

# Una predicción por matrícula, repetición,
# experimento y algoritmo.
CLAVE_PREDICCION = [
    "Repeticion",
    "Algoritmo",
    "Experimento"
] + CLAVES

assert not predicciones_conjuntas_s2.duplicated(
    subset=CLAVE_PREDICCION
).any()

conteos_predicciones = (
    predicciones_conjuntas_s2.groupby(
        ["Repeticion", "Algoritmo", "Experimento"]
    ).size()
)

assert conteos_predicciones.eq(
    len(y_analisis)
).all()

assert predicciones_conjuntas_s2[
    "prob_adverso"
].between(0, 1).all()

print(
    "Total de predicciones reconstruidas:",
    len(predicciones_conjuntas_s2)
)

print(
    "Matrículas por repetición, algoritmo y experimento:",
    len(y_analisis)
)

print("Claves compuestas: verificadas")
print("Duplicados de predicciones: 0")

# ------------------------------------------------------------
# 7. ADJUNTAR RESULTADO ORIGINAL SOLO PARA EVALUACIÓN
# ------------------------------------------------------------

# ATENCIÓN:
# final_result y date_unregistration se incorporan
# DESPUÉS del entrenamiento y las predicciones.
# Se utilizan exclusivamente para auditoría de errores.

cohorte_meta = cohortes_oulad.loc[
    (
        cohortes_oulad["semana_prediccion"]
        == SEMANA_ANALISIS
    )
    &
    (
        cohortes_oulad["conjunto"]
        == "Entrenamiento"
    ),
    CLAVES + [
        "final_result",
        "date_unregistration"
    ]
].copy()

assert not cohorte_meta.duplicated(
    subset=CLAVES
).any()

assert len(cohorte_meta) == len(y_analisis)

predicciones_conjuntas_s2 = (
    predicciones_conjuntas_s2.merge(
        cohorte_meta,
        on=CLAVES,
        how="left",
        validate="many_to_one",
        indicator=True
    )
)

assert predicciones_conjuntas_s2["_merge"].eq(
    "both"
).all()

predicciones_conjuntas_s2 = (
    predicciones_conjuntas_s2.drop(
        columns="_merge"
    )
)

mapa_adverso = {
    "Pass": 0,
    "Distinction": 0,
    "Fail": 1,
    "Withdrawn": 1
}

assert np.array_equal(
    predicciones_conjuntas_s2[
        "final_result"
    ].map(mapa_adverso).to_numpy(),
    predicciones_conjuntas_s2[
        "y_real"
    ].to_numpy()
)

# ------------------------------------------------------------
# 8. ANÁLISIS POR RESULTADO ACADÉMICO
# ------------------------------------------------------------

print("\n=== 3.10.6 IDENTIFICACIÓN POR RESULTADO FINAL ===")

tasas_por_resultado = (
    predicciones_conjuntas_s2.groupby(
        [
            "Algoritmo",
            "Experimento",
            "Repeticion",
            "final_result"
        ]
    )
    .agg(
        Matriculas=("y_pred", "size"),
        Alertas=("y_pred", "sum")
    )
    .reset_index()
)

tasas_por_resultado["Tasa_alerta_pct"] = (
    100
    * tasas_por_resultado["Alertas"]
    / tasas_por_resultado["Matriculas"]
)

# En Fail y Withdrawn esta tasa es recall del subtipo.
# En Pass y Distinction es tasa de falsas alertas.

tabla_resultados_especificos = (
    tasas_por_resultado.groupby(
        [
            "Algoritmo",
            "Experimento",
            "final_result"
        ]
    )
    .agg(
        Matriculas_por_repeticion=("Matriculas", "first"),
        Tasa_alerta_media_pct=("Tasa_alerta_pct", "mean"),
        Tasa_alerta_std_pct=("Tasa_alerta_pct", "std")
    )
    .reset_index()
)

tabla_resultados_especificos[
    "Interpretacion"
] = np.where(
    tabla_resultados_especificos[
        "final_result"
    ].isin(["Fail", "Withdrawn"]),
    "Recall del subtipo adverso",
    "Tasa de falsas alertas"
)

display(
    tabla_resultados_especificos.round(2)
)

# ------------------------------------------------------------
# 9. COMPARAR CASOS RECUPERADOS ENTRE B Y A
# ------------------------------------------------------------

print("\n=== 3.10.7 CAMBIOS INDIVIDUALES A VS B ===")

cambios_individuales = []

for algoritmo in [
    "LogisticRegression",
    "RandomForest"
]:

    for repeticion in [1, 2]:

        sub = predicciones_conjuntas_s2.loc[
            (
                predicciones_conjuntas_s2[
                    "Algoritmo"
                ] == algoritmo
            )
            &
            (
                predicciones_conjuntas_s2[
                    "Repeticion"
                ] == repeticion
            )
        ]

        ancho = sub.pivot(
            index=CLAVES,
            columns="Experimento",
            values="y_pred"
        )

        etiquetas = (
            sub.loc[
                sub["Experimento"] == "A"
            ]
            .set_index(CLAVES)["y_real"]
        )

        etiquetas_B = (
            sub.loc[
                sub["Experimento"] == "B"
            ]
            .set_index(CLAVES)["y_real"]
        )

        assert etiquetas.sort_index().equals(
            etiquetas_B.sort_index()
        )

        conjunto = ancho.join(
            etiquetas.rename("y_real")
        )

        assert len(conjunto) == len(y_analisis)

        adversos = conjunto["y_real"] == 1
        favorables = conjunto["y_real"] == 0

        recuperados = (
            adversos
            & (conjunto["A"] == 0)
            & (conjunto["B"] == 1)
        ).sum()

        perdidos = (
            adversos
            & (conjunto["A"] == 1)
            & (conjunto["B"] == 0)
        ).sum()

        falsas_alertas_evitadas = (
            favorables
            & (conjunto["A"] == 1)
            & (conjunto["B"] == 0)
        ).sum()

        falsas_alertas_nuevas = (
            favorables
            & (conjunto["A"] == 0)
            & (conjunto["B"] == 1)
        ).sum()

        cambios_individuales.append({
            "Algoritmo": algoritmo,
            "Repeticion": repeticion,
            "Adversos_recuperados_B": int(
                recuperados
            ),
            "Adversos_perdidos_B": int(
                perdidos
            ),
            "Ganancia_neta_VP": int(
                recuperados - perdidos
            ),
            "Falsas_alertas_evitadas_B": int(
                falsas_alertas_evitadas
            ),
            "Falsas_alertas_nuevas_B": int(
                falsas_alertas_nuevas
            ),
            "Reduccion_neta_FP": int(
                falsas_alertas_evitadas
                - falsas_alertas_nuevas
            )
        })

tabla_cambios_individuales = pd.DataFrame(
    cambios_individuales
)

display(tabla_cambios_individuales)

# ------------------------------------------------------------
# 10. ANTICIPACIÓN DE RETIROS FUTUROS
# ------------------------------------------------------------

print("\n=== 3.10.8 ANTICIPACIÓN DE RETIROS ===")

retiros = predicciones_conjuntas_s2.loc[
    predicciones_conjuntas_s2[
        "final_result"
    ] == "Withdrawn"
].copy()

retiros["dias_hasta_retiro"] = (
    retiros["date_unregistration"]
    - DIA_CORTE
)

assert retiros[
    "dias_hasta_retiro"
].gt(0).all()

resultados_anticipacion = []

for (
    algoritmo,
    experimento,
    repeticion
), sub in retiros.groupby(
    ["Algoritmo", "Experimento", "Repeticion"]
):

    detectados = sub.loc[
        sub["y_pred"] == 1
    ]

    resultados_anticipacion.append({
        "Algoritmo": algoritmo,
        "Experimento": experimento,
        "Repeticion": repeticion,
        "Retiros_futuros": len(sub),
        "Retiros_detectados": len(detectados),
        "Recall_retiro": (
            len(detectados) / len(sub)
        ),
        "Anticipacion_mediana_dias": (
            detectados["dias_hasta_retiro"].median()
            if len(detectados) else np.nan
        ),
        "Alertas_con_14dias_o_mas": int(
            (
                detectados["dias_hasta_retiro"]
                >= 14
            ).sum()
        ),
        "Alertas_con_28dias_o_mas": int(
            (
                detectados["dias_hasta_retiro"]
                >= 28
            ).sum()
        )
    })

tabla_anticipacion_retiros = pd.DataFrame(
    resultados_anticipacion
)

display(
    tabla_anticipacion_retiros.round(4)
)

print(
    "\nNota: la anticipación se calcula únicamente "
    "para retiros efectivamente identificados."
)
print(
    "No representa el tiempo de anticipación "
    "de todos los estudiantes ni una intervención real."
)

# ------------------------------------------------------------
# 11. RESUMEN VISUAL DE LOS CUATRO MODELOS
# ------------------------------------------------------------

print("\n=== 3.10.9 GRÁFICOS COMPARATIVOS ===")

etiquetas_modelos = [
    f"{fila.Algoritmo.replace('LogisticRegression', 'LR').replace('RandomForest', 'RF')} {fila.Experimento}"
    for fila in tabla_resumen_conjunto.itertuples()
]

plt.figure(figsize=(9, 4))

plt.bar(
    etiquetas_modelos,
    tabla_resumen_conjunto["F1_macro_medio"]
)

plt.title("F1 macro medio - Semana 2")
plt.ylabel("F1 macro")
plt.ylim(0, 0.8)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

plt.figure(figsize=(9, 4))

plt.bar(
    etiquetas_modelos,
    tabla_resumen_conjunto["Recall_adverso_medio"]
)

plt.title("Recall de la clase adversa - Semana 2")
plt.ylabel("Recall")
plt.ylim(0, 0.8)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 12. CONTROLES FINALES
# ------------------------------------------------------------

print("\n=== 3.10.10 CONTROLES FINALES ===")

assert len(tabla_resumen_conjunto) == 4

assert len(tabla_pares_algoritmos) == 20

assert len(tabla_cambios_individuales) == 4

assert not predicciones_conjuntas_s2[
    "final_result"
].isna().any()

for _, sub in predicciones_conjuntas_s2.groupby(
    ["Repeticion", "Algoritmo", "Experimento"]
):
    assert len(sub) == len(y_analisis)

print("Cuatro modelos comparados correctamente.")
print("Predicciones emparejadas por matrícula.")
print("Resultados originales conservados para auditoría.")
print("Análisis de reprobación y retiro preparado.")
print("No se han entrenado nuevos modelos.")
print("Conjunto de prueba independiente: NO utilizado.")

print("\n=== ACTIVIDAD 3.10 FINALIZADA ===")


In [ ]:

# ============================================================
# ACTIVIDAD 3.10 - RESUMEN PARA INTERPRETACIÓN
# No vuelve a entrenar modelos.
# ============================================================

from IPython.display import display

print("=== 3.10.4 COMPARACIÓN LR VS RF ===")
display(tabla_diferencias_algoritmos.round(4))

print("\n=== 3.10.6 IDENTIFICACIÓN POR RESULTADO ===")
display(tabla_resultados_especificos.round(2))

print("\n=== 3.10.7 CAMBIOS INDIVIDUALES A VS B ===")
display(tabla_cambios_individuales)

print("\n=== 3.10.8 ANTICIPACIÓN DE RETIROS ===")
display(tabla_anticipacion_retiros.round(4))



### 3.10.11. Discusión integrada de resultados

La Actividad 3.10 permitió comparar los cuatro modelos de clasificación de la semana 2 mediante sus resultados de validación cruzada anidada y repetida, así como analizar individualmente las predicciones frente a reprobaciones y retiros futuros.

**Comparación entre algoritmos**

Regresión Logística presentó una ventaja media de 0,0090 puntos de F1 macro respecto a Random Forest, tanto en el Experimento A como en el Experimento B.

En el Experimento A, Regresión Logística superó a Random Forest en F1 macro y ROC-AUC en los diez folds externos.

En el Experimento B, Regresión Logística obtuvo mayor F1 macro en nueve de diez folds, y mayor recall adverso y ROC-AUC en los diez folds.

La ventaja de Regresión Logística fue consistente, aunque de magnitud relativamente reducida.

**Identificación de resultados académicos específicos**

El análisis por subtipo de resultado permitió comprobar que la mejora longitudinal se presenta tanto en reprobaciones como en retiros futuros.

Para Regresión Logística:

- Recall de `Fail`: 46,23 % en A y 57,76 % en B.
- Recall de `Withdrawn`: 49,16 % en A y 57,38 % en B.

Para Random Forest:

- Recall de `Fail`: 44,18 % en A y 54,82 % en B.
- Recall de `Withdrawn`: 47,71 % en A y 53,36 % en B.

Estos resultados muestran que las características longitudinales proporcionan información predictiva adicional para ambos subtipos de resultados adversos.

**Cambios individuales en las predicciones**

Durante la primera repetición, Regresión Logística B obtuvo una ganancia neta de 1.009 casos adversos correctamente identificados y redujo en 154 las falsas alertas respecto de A.

En la segunda repetición, la ganancia neta fue de 1.024 casos adversos, con una reducción de 110 falsas alertas.

Random Forest B también presentó ganancias netas positivas en ambas repeticiones, recuperando 790 y 911 casos adversos adicionales, respectivamente, y reduciendo las falsas alertas en 351 y 300 matrículas.

Las cifras de distintas repeticiones no representan poblaciones independientes y, por tanto, no deben sumarse.

### 3.10.12. Análisis de anticipación de retiros

La cohorte de entrenamiento de la semana 2 contiene 4.538 matrículas con retiro posterior al día de predicción.

En la primera repetición, Regresión Logística B identificó 2.609 de estos retiros, frente a 2.230 identificados por Regresión Logística A.

Además, 1.989 retiros identificados por B contaban con al menos 28 días de anticipación, frente a 1.738 de A.

La mediana de anticipación entre los retiros correctamente identificados fue de 75 días para Regresión Logística B y 74 días para Random Forest B.

Las medianas correspondientes a los modelos estáticos fueron de 81 y 80 días, respectivamente.

La menor mediana observada en los modelos longitudinales debe interpretarse considerando que estos identificaron un conjunto más amplio de retiros, posiblemente incorporando eventos próximos al momento de predicción.

La anticipación debe evaluarse conjuntamente con la proporción de retiros detectados y la precisión de las alertas, evitando interpretarla aisladamente.

### 3.10.13. Limitaciones metodológicas

La clasificación principal es binaria y combina reprobación y retiro del módulo académico.

Por tanto, el análisis de identificación por subtipo no corresponde a modelos independientes de predicción de reprobación o retiro.

Los resultados provienen de validación cruzada anidada y repetida sobre el conjunto de desarrollo, con dependencia entre las observaciones de distintas repeticiones.

Las diferencias de rendimiento no constituyen todavía evidencia confirmatoria de significancia estadística.

La medición de anticipación corresponde exclusivamente a retiros futuros correctamente identificados y no demuestra la efectividad de una intervención académica.

El conjunto de prueba independiente permaneció reservado durante todo el análisis.

### 3.10.14. Conclusión general

Los resultados de la semana 2 proporcionan evidencia empírica consistente de que las características longitudinales incrementan la capacidad predictiva de Regresión Logística y Random Forest respecto de sus versiones estáticas.

La mejora se observa en el F1 macro, recall adverso y ROC-AUC, así como en la identificación específica de reprobaciones y retiros futuros.

Regresión Logística B obtuvo el mejor rendimiento promedio entre los modelos examinados, mientras que Random Forest presentó mayores señales de sobreajuste.

El análisis de retiros muestra que una parte de los eventos futuros puede identificarse con varias semanas de anticipación, aunque la utilidad institucional de estas alertas todavía debe evaluarse con criterios específicos.

**Conclusión principal:** los resultados respaldan provisionalmente la hipótesis de que el comportamiento longitudinal del estudiante aporta información predictiva adicional desde las primeras dos semanas del módulo. La evaluación de las semanas 4, 6 y 8 será necesaria para determinar cómo evoluciona esa contribución conforme aumenta la información académica disponible.



### 3.11. Validación cruzada anidada — Semana 4

#### Objetivo

Evaluar la capacidad predictiva de Regresión Logística y Random Forest sobre la cohorte de la semana 4, comparando características estáticas y longitudinales mediante validación cruzada repetida y anidada.

#### Cohorte experimental

La predicción se realizará al inicio del día 28, considerando únicamente eventos registrados entre los días 0 y 27.

La cohorte contiene 27.422 matrículas elegibles, distribuidas en:

- Entrenamiento y desarrollo: 21.984 matrículas.
- Prueba independiente: 5.438 matrículas.

El Experimento A utilizará 12 características estáticas.

El Experimento B utilizará 36 características, compuestas por 12 estáticas y 24 longitudinales.

#### Protocolo de validación

Se reutilizarán las diez particiones externas agrupadas por estudiante, correspondientes a cinco folds y dos repeticiones.

En cada fold externo se realizará una optimización interna mediante tres folds agrupados por estudiante.

El preprocesamiento y la selección de características se ajustarán exclusivamente dentro de cada fold de entrenamiento.

#### Algoritmos

Se evaluarán Regresión Logística y Random Forest, manteniendo los mismos espacios de hiperparámetros de las actividades anteriores.

Esto permitirá comparar el comportamiento de los algoritmos y estudiar el valor incremental de las características longitudinales sin modificar simultáneamente el protocolo de optimización.

#### Métrica principal

F1 macro será la métrica principal. Se reportarán adicionalmente recall adverso, ROC-AUC, accuracy y matrices de confusión.

#### Análisis

Se calcularán diferencias pareadas de B respecto a A utilizando las mismas particiones.

Los resultados se compararán descriptivamente con los de la semana 2.

Las diferencias entre ventanas no se interpretarán como efectos exclusivos del aumento de información, porque las cohortes elegibles cambian.

#### Alcance

El procedimiento será reutilizable para las semanas 6 y 8, conservando las mismas reglas de entrenamiento y evaluación.

El conjunto de prueba independiente continuará reservado.

No se realizarán pruebas de significancia estadística en esta actividad.

**Referencias metodológicas:** la optimización y evaluación por validación cruzada se apoyan en [2], [10]. La evaluación por F1, precisión y recall es especialmente útil para interpretar los resultados por clase [12].


In [ ]:

# ============================================================
# PROYECTO FINAL DE MACHINE LEARNING
# ACTIVIDAD 3.11 - VALIDACIÓN ANIDADA MULTIVENTANA
# EJECUCIÓN INICIAL: SEMANA 4
# ============================================================

# Asistencia IA: ChatGPT, octubre de 2026.
# Prompt:
# "Generalizar la validación cruzada anidada y repetida
# de Regresión Logística y Random Forest a otras semanas
# de OULAD. Conservar los mismos grids, folds agrupados
# y métricas; guardar predicciones y diferencias pareadas.
# No utilizar el conjunto de prueba independiente."

import time
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.model_selection import (
    StratifiedGroupKFold,
    GridSearchCV
)
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)
from IPython.display import display

# ------------------------------------------------------------
# 1. CONFIGURACIÓN
# ------------------------------------------------------------

SEMANA_EJECUTAR = 4

# Para ejecutar ambos algoritmos:
ALGORITMOS_EJECUTAR = [
    "LogisticRegression",
    "RandomForest"
]

# Si se desea ejecutar uno por vez, usar por ejemplo:
# ALGORITMOS_EJECUTAR = ["LogisticRegression"]
# Después cambiar a ["RandomForest"] y repetir la celda.

GRIDS_ANIDADOS = {
    "LogisticRegression": {
        "modelo__C": [0.1, 1.0],
        "seleccion__percentile": [80, 100]
    },
    "RandomForest": {
        "modelo__n_estimators": [100],
        "modelo__max_depth": [12, None],
        "modelo__min_samples_leaf": [2, 5],
        "seleccion__percentile": [80]
    }
}

METRICAS_PAREADAS = [
    "F1_macro",
    "Recall_adverso",
    "ROC_AUC",
    "Accuracy"
]

CLAVES_MATRICULA = [
    "code_module",
    "code_presentation",
    "id_student"
]

if "resultados_cv_ventanas" not in globals():
    resultados_cv_ventanas = {}

# ------------------------------------------------------------
# 2. FUNCIÓN DE EVALUACIÓN
# ------------------------------------------------------------

def metricas_modelo_oulad(y_real, pred, prob):

    cm = confusion_matrix(
        y_real, pred, labels=[0, 1]
    )

    return {
        "Accuracy": accuracy_score(y_real, pred),
        "Precision_adverso": precision_score(
            y_real, pred, zero_division=0
        ),
        "Recall_adverso": recall_score(
            y_real, pred, zero_division=0
        ),
        "F1_macro": f1_score(
            y_real, pred,
            average="macro",
            zero_division=0
        ),
        "F1_weighted": f1_score(
            y_real, pred,
            average="weighted",
            zero_division=0
        ),
        "ROC_AUC": roc_auc_score(y_real, prob),
        "VN": int(cm[0, 0]),
        "FP": int(cm[0, 1]),
        "FN": int(cm[1, 0]),
        "VP": int(cm[1, 1])
    }


# ------------------------------------------------------------
# 3. FUNCIÓN GENERAL PARA UNA SEMANA Y ALGORITMO
# ------------------------------------------------------------

def evaluar_cv_anidada_oulad(semana, algoritmo):

    datos = experimentos_oulad[semana]

    X = {
        "A": datos["X_A_train"].reset_index(drop=True),
        "B": datos["X_B_train"].reset_index(drop=True)
    }

    y = datos["y_train"].reset_index(drop=True)

    grupos = (
        datos["grupos_train"].reset_index(drop=True)
    )

    claves = (
        datos["claves_train"].reset_index(drop=True)
    )

    splits = cv_splits_oulad[semana]

    assert len(splits) == 10

    assert all(
        len(matriz) == len(y)
        for matriz in X.values()
    )

    assert len(grupos) == len(claves) == len(y)

    resultados = []
    parametros = []
    predicciones = []

    inicio_total = time.perf_counter()

    for n_split, (idx_tr, idx_val) in enumerate(
        splits, start=1
    ):

        repeticion = (n_split - 1) // 5 + 1
        fold = (n_split - 1) % 5 + 1

        y_tr = y.iloc[idx_tr].reset_index(drop=True)
        y_val = y.iloc[idx_val].reset_index(drop=True)

        g_tr = (
            grupos.iloc[idx_tr].reset_index(drop=True)
        )

        g_val = (
            grupos.iloc[idx_val].reset_index(drop=True)
        )

        assert set(g_tr).isdisjoint(set(g_val))
        assert set(y_tr.unique()) == {0, 1}
        assert set(y_val.unique()) == {0, 1}

        # Reconstruir exactamente el mismo protocolo
        # interno utilizado en semanas anteriores.
        cv_interna = StratifiedGroupKFold(
            n_splits=3,
            shuffle=True,
            random_state=137 + n_split - 1
        )

        XA_tr = (
            X["A"].iloc[idx_tr]
            .reset_index(drop=True)
        )

        indices_internos = list(
            cv_interna.split(
                XA_tr,
                y_tr,
                groups=g_tr
            )
        )

        for it, iv in indices_internos:
            assert set(
                g_tr.iloc[it]
            ).isdisjoint(
                set(g_tr.iloc[iv])
            )

        print(
            f"\nSemana {semana} | {algoritmo} | "
            f"R{repeticion}-F{fold}",
            flush=True
        )

        for exp in ["A", "B"]:

            inicio_modelo = time.perf_counter()

            X_tr = (
                X[exp].iloc[idx_tr]
                .reset_index(drop=True)
            )

            X_val = (
                X[exp].iloc[idx_val]
                .reset_index(drop=True)
            )

            pipeline = clone(
                pipelines_oulad[
                    (semana, exp, algoritmo)
                ]
            )

            busqueda = GridSearchCV(
                estimator=pipeline,
                param_grid=GRIDS_ANIDADOS[algoritmo],
                cv=indices_internos,
                scoring="f1_macro",
                refit=True,
                n_jobs=1,
                return_train_score=True,
                error_score="raise"
            )

            # Exclusivamente entrenamiento externo.
            busqueda.fit(X_tr, y_tr)

            modelo = busqueda.best_estimator_

            pred = modelo.predict(X_val)

            indice_positivo = list(
                modelo.classes_
            ).index(1)

            prob = modelo.predict_proba(
                X_val
            )[:, indice_positivo]

            metricas = metricas_modelo_oulad(
                y_val, pred, prob
            )

            duracion = (
                time.perf_counter() - inicio_modelo
            )

            resultados.append({
                "Semana": semana,
                "Algoritmo": algoritmo,
                "Experimento": exp,
                "Repeticion": repeticion,
                "Fold": fold,
                "N_train": len(idx_tr),
                "N_validacion": len(idx_val),
                "F1_CV_interno": busqueda.best_score_,
                **metricas,
                "Tiempo_segundos": duracion
            })

            best_i = busqueda.best_index_

            f1_train = busqueda.cv_results_[
                "mean_train_score"
            ][best_i]

            f1_val = busqueda.cv_results_[
                "mean_test_score"
            ][best_i]

            parametros.append({
                "Semana": semana,
                "Algoritmo": algoritmo,
                "Experimento": exp,
                "Repeticion": repeticion,
                "Fold": fold,
                "Parametros": str(
                    busqueda.best_params_
                ),
                "F1_train_interno": f1_train,
                "F1_val_interno": f1_val,
                "Brecha_train_val": (
                    f1_train - f1_val
                )
            })

            # Guardar claves completas para permitir
            # análisis posteriores de retiro y errores.
            p = (
                claves.iloc[idx_val]
                .reset_index(drop=True)
                .copy()
            )

            p["Semana"] = semana
            p["Algoritmo"] = algoritmo
            p["Experimento"] = exp
            p["Repeticion"] = repeticion
            p["Fold"] = fold
            p["y_real"] = y_val.to_numpy()
            p["y_pred"] = pred
            p["prob_adverso"] = prob

            predicciones.append(p)

            print(
                f"  {exp}: "
                f"F1={metricas['F1_macro']:.4f} | "
                f"Recall={metricas['Recall_adverso']:.4f} | "
                f"AUC={metricas['ROC_AUC']:.4f} | "
                f"{duracion:.1f}s",
                flush=True
            )

    # --------------------------------------------------------
    # 4. CONSOLIDAR
    # --------------------------------------------------------

    tabla = pd.DataFrame(resultados)

    tabla_params = pd.DataFrame(parametros)

    tabla_pred = pd.concat(
        predicciones, ignore_index=True
    )

    assert len(tabla) == 20
    assert len(tabla_params) == 20

    assert not tabla_pred.duplicated(
        subset=[
            "Experimento",
            "Repeticion"
        ] + CLAVES_MATRICULA
    ).any()

    assert (
        tabla_pred.groupby(
            ["Experimento", "Repeticion"]
        ).size().eq(len(y)).all()
    )

    # --------------------------------------------------------
    # 5. CONSTRUIR DIFERENCIAS PAREADAS
    # --------------------------------------------------------

    columnas = (
        ["Repeticion", "Fold"]
        + METRICAS_PAREADAS
    )

    pa = (
        tabla.loc[
            tabla["Experimento"] == "A",
            columnas
        ]
        .rename(columns={
            m: f"{m}_A"
            for m in METRICAS_PAREADAS
        })
    )

    pb = (
        tabla.loc[
            tabla["Experimento"] == "B",
            columnas
        ]
        .rename(columns={
            m: f"{m}_B"
            for m in METRICAS_PAREADAS
        })
    )

    pares = pa.merge(
        pb,
        on=["Repeticion", "Fold"],
        validate="one_to_one"
    )

    assert len(pares) == 10

    for m in METRICAS_PAREADAS:

        pares[f"Delta_{m}"] = (
            pares[f"{m}_B"]
            - pares[f"{m}_A"]
        )

    duracion_total = (
        time.perf_counter() - inicio_total
    )

    return {
        "resultados": tabla,
        "parametros": tabla_params,
        "predicciones": tabla_pred,
        "pares": pares,
        "duracion_segundos": duracion_total
    }


# ------------------------------------------------------------
# 6. EJECUCIÓN DE LA SEMANA SELECCIONADA
# ------------------------------------------------------------

print("=== 3.11.1 CONFIGURACIÓN ===")
print("Semana:", SEMANA_EJECUTAR)
print("Último día observable:", 7 * SEMANA_EJECUTAR - 1)
print(
    "Matrículas entrenamiento:",
    len(
        experimentos_oulad[
            SEMANA_EJECUTAR
        ]["y_train"]
    )
)
print("Algoritmos:", ALGORITMOS_EJECUTAR)

for algoritmo in ALGORITMOS_EJECUTAR:

    clave = (SEMANA_EJECUTAR, algoritmo)

    resultados_cv_ventanas[clave] = (
        evaluar_cv_anidada_oulad(
            SEMANA_EJECUTAR,
            algoritmo
        )
    )

    print(
        f"\n{algoritmo} completado y guardado.",
        flush=True
    )

# ------------------------------------------------------------
# 7. RESUMEN GENERAL
# ------------------------------------------------------------

print("\n=== 3.11.2 RENDIMIENTO MEDIO ===")

tablas_semana = [
    resultados_cv_ventanas[
        (SEMANA_EJECUTAR, a)
    ]["resultados"]
    for a in ALGORITMOS_EJECUTAR
]

tabla_semana_cv = pd.concat(
    tablas_semana,
    ignore_index=True
)

resumen_semana_cv = (
    tabla_semana_cv.groupby(
        ["Algoritmo", "Experimento"]
    )
    .agg(
        F1_macro_medio=("F1_macro", "mean"),
        F1_macro_std=("F1_macro", "std"),
        Recall_adverso_medio=("Recall_adverso", "mean"),
        ROC_AUC_medio=("ROC_AUC", "mean"),
        Accuracy_media=("Accuracy", "mean")
    )
    .reset_index()
)

display(resumen_semana_cv.round(4))

# ------------------------------------------------------------
# 8. COMPARACIÓN PAREADA
# ------------------------------------------------------------

print("\n=== 3.11.3 DIFERENCIAS B - A ===")

filas_diferencias = []

for algoritmo in ALGORITMOS_EJECUTAR:

    pares = resultados_cv_ventanas[
        (SEMANA_EJECUTAR, algoritmo)
    ]["pares"]

    for metrica in METRICAS_PAREADAS:

        delta = pares[f"Delta_{metrica}"]

        filas_diferencias.append({
            "Algoritmo": algoritmo,
            "Metrica": metrica,
            "Delta_medio_B_A": delta.mean(),
            "Desviacion": delta.std(ddof=1),
            "Delta_minimo": delta.min(),
            "Delta_maximo": delta.max(),
            "Folds_B_supera_A": int(
                (delta > 0).sum()
            )
        })

tabla_diferencias_semana = pd.DataFrame(
    filas_diferencias
)

display(tabla_diferencias_semana.round(4))

# ------------------------------------------------------------
# 9. BRECHAS DE GENERALIZACIÓN
# ------------------------------------------------------------

print("\n=== 3.11.4 SOBREAJUSTE ===")

tablas_parametros = [
    resultados_cv_ventanas[
        (SEMANA_EJECUTAR, a)
    ]["parametros"]
    for a in ALGORITMOS_EJECUTAR
]

tabla_parametros_semana = pd.concat(
    tablas_parametros,
    ignore_index=True
)

brechas_semana = (
    tabla_parametros_semana.groupby(
        ["Algoritmo", "Experimento"]
    )
    .agg(
        F1_train_medio=("F1_train_interno", "mean"),
        F1_validacion_medio=("F1_val_interno", "mean"),
        Brecha_media=("Brecha_train_val", "mean")
    )
    .reset_index()
)

display(brechas_semana.round(4))

# ------------------------------------------------------------
# 10. COMPROBACIONES FINALES
# ------------------------------------------------------------

print("\n=== 3.11.5 CONTROLES FINALES ===")

for algoritmo in ALGORITMOS_EJECUTAR:

    r = resultados_cv_ventanas[
        (SEMANA_EJECUTAR, algoritmo)
    ]

    assert len(r["resultados"]) == 20
    assert len(r["pares"]) == 10

    assert r["resultados"][
        ["F1_macro", "Recall_adverso", "ROC_AUC"]
    ].notna().all().all()

    print(
        f"{algoritmo}: "
        "10 folds por experimento y "
        "predicciones conservadas."
    )

print("Conjunto de prueba independiente: NO utilizado.")
print("Protocolo comparable con semanas anteriores.")

print("\n=== ACTIVIDAD 3.11 FINALIZADA ===")



### 3.11.6. Resultados e interpretación

Se completó la validación cruzada anidada y repetida de los experimentos A y B para la semana 4 de OULAD, utilizando cinco folds externos, dos repeticiones y tres folds internos para optimización.

Se evaluaron 21.984 matrículas del conjunto de desarrollo, manteniendo la separación por estudiante y reservando el conjunto de prueba independiente.

**Rendimiento predictivo**

| Modelo | F1 macro | Recall adverso | ROC-AUC |
|---|---:|---:|---:|
| Regresión Logística A | 0,6040 | 0,4428 | 0,6579 |
| Regresión Logística B | 0,6855 | 0,5739 | 0,7560 |
| Random Forest A | 0,5945 | 0,4376 | 0,6474 |
| Random Forest B | 0,6706 | 0,5238 | 0,7447 |

Regresión Logística B alcanzó el mayor rendimiento promedio en las tres métricas principales.

**Aporte de las características longitudinales**

En Regresión Logística, la diferencia media B-A fue de 0,0815 en F1 macro, 0,1311 en recall adverso y 0,0981 en ROC-AUC.

En Random Forest, la diferencia media fue de 0,0761 en F1 macro, 0,0862 en recall adverso y 0,0974 en ROC-AUC.

Los dos algoritmos presentaron mejoras positivas en los diez folds externos para F1 macro, recall adverso, ROC-AUC y accuracy.

Esto representa evidencia descriptiva consistente a favor del aporte predictivo de las características longitudinales.

**Comparación temporal con la semana 2**

El F1 macro medio de Regresión Logística B aumentó de 0,6656 a 0,6855, mientras que Random Forest B pasó de 0,6566 a 0,6706.

Asimismo, el ROC-AUC aumentó de 0,7315 a 0,7560 en Regresión Logística B y de 0,7204 a 0,7447 en Random Forest B.

Sin embargo, el recall adverso disminuyó ligeramente: de 0,5759 a 0,5739 en Regresión Logística B y de 0,5417 a 0,5238 en Random Forest B.

Por tanto, una mejora en F1 macro y ROC-AUC no implica necesariamente una mejora simultánea de la sensibilidad para detectar resultados adversos con el umbral de clasificación utilizado.

Las diferencias entre semanas también pueden estar afectadas por cambios en la composición de las cohortes elegibles. Por ello, será necesario realizar posteriormente una comparación temporal con población fija.

**Análisis de sobreajuste**

Regresión Logística presentó brechas medias de entrenamiento-validación interna de 0,0037 en A y 0,0040 en B.

Random Forest presentó brechas superiores: 0,0952 en A y 0,1960 en B.

La brecha de Random Forest B aumentó respecto a la semana 2, cuando fue de 0,1550.

Estos resultados sugieren un sobreajuste relativo más pronunciado en Random Forest B y justifican posteriores análisis de regularización y curvas de aprendizaje.

### 3.11.7. Limitaciones

Los resultados corresponden a validación cruzada repetida y anidada sobre el conjunto de desarrollo, cuyas particiones no son estadísticamente independientes.

Las diferencias entre semanas no pueden atribuirse exclusivamente al incremento de información longitudinal, porque la población elegible y la distribución de resultados también cambian.

No se ha realizado todavía una prueba confirmatoria sobre el conjunto independiente ni una evaluación específica de precisión de alertas para retiro futuro.

### 3.11.8. Conclusión

La evaluación de la semana 4 confirmó que incorporar características longitudinales mejora consistentemente el rendimiento de Regresión Logística y Random Forest frente a sus versiones estáticas.

El aporte longitudinal al F1 macro y ROC-AUC fue mayor que el observado en la semana 2, aunque no se registró una mejora paralela en recall adverso entre ambas ventanas.

Regresión Logística B obtuvo el mejor rendimiento promedio y mantuvo una brecha reducida entre entrenamiento y validación.

**Conclusión principal:** las características longitudinales continúan aportando valor predictivo durante la cuarta semana del módulo. Sin embargo, la evolución temporal del rendimiento debe interpretarse considerando el balance entre métricas, el sobreajuste y los cambios en las cohortes de estudiantes.



### 3.12. Validación cruzada anidada — Semana 6

#### Objetivo

Evaluar la capacidad predictiva de Regresión Logística y Random Forest en la semana 6, comparando características estáticas y longitudinales bajo el mismo procedimiento de validación utilizado en las semanas 2 y 4.

#### Diseño experimental

La predicción se realizará al inicio del día 42, utilizando exclusivamente información disponible hasta el día 41.

La cohorte está compuesta por 21.568 matrículas de desarrollo y 5.344 matrículas reservadas para prueba independiente.

El Experimento A utilizará 12 características estáticas y el Experimento B utilizará 42 características estáticas y longitudinales.

Se emplearán cinco folds externos con dos repeticiones y tres folds internos para optimización de hiperparámetros.

Se conservarán los espacios de búsqueda previamente establecidos y F1 macro como métrica principal.

#### Evaluación

Se reportarán F1 macro, recall de la clase adversa, ROC-AUC y accuracy.

Se calcularán diferencias pareadas entre B y A sobre las mismas particiones de estudiantes.

También se analizarán las brechas de rendimiento entre entrenamiento y validación interna, particularmente en Random Forest.

#### Comparación temporal

Se contrastarán descriptivamente los resultados de las semanas 2, 4 y 6.

Las diferencias temporales no se atribuirán exclusivamente al incremento de información, debido a los cambios en las cohortes elegibles.

Posteriormente se realizará una evaluación de sensibilidad con una población fija para investigar esta limitación.

#### Prevención de fuga de información

Todos los procedimientos de preprocesamiento, selección de características y optimización se ajustarán exclusivamente con los datos de entrenamiento correspondientes a cada fold.

El conjunto de prueba independiente no será utilizado.


In [ ]:

# ============================================================
# PROYECTO FINAL DE MACHINE LEARNING
# ACTIVIDAD 3.12 - VALIDACIÓN ANIDADA, SEMANA 6
# ============================================================

# Asistencia IA: ChatGPT, octubre de 2026.
# Prompt:
# "Ejecutar la validación anidada de Regresión Logística
# y Random Forest en la semana 6 de OULAD. Reutilizar
# la función y los hiperparámetros de la semana 4,
# comparar A vs B y consolidar los resultados de las
# semanas 2, 4 y 6, sin utilizar la prueba independiente."

import numpy as np
import pandas as pd
from IPython.display import display

# ------------------------------------------------------------
# 1. CONFIGURACIÓN
# ------------------------------------------------------------

SEMANA_312 = 6

ALGORITMOS_312 = [
    "LogisticRegression",
    "RandomForest"
]

METRICAS_312 = [
    "F1_macro",
    "Recall_adverso",
    "ROC_AUC",
    "Accuracy"
]

assert "evaluar_cv_anidada_oulad" in globals()
assert "resultados_cv_ventanas" in globals()

print("=== 3.12.1 CONFIGURACIÓN ===")

print("Semana:", SEMANA_312)
print("Último día observable:", 41)

print(
    "Matrículas de desarrollo:",
    len(experimentos_oulad[6]["y_train"])
)

print(
    "Matrículas de prueba reservada:",
    len(experimentos_oulad[6]["y_test"])
)

print("Algoritmos:", ALGORITMOS_312)

# ------------------------------------------------------------
# 2. EJECUTAR LOS DOS ALGORITMOS
# ------------------------------------------------------------

for algoritmo in ALGORITMOS_312:

    clave = (SEMANA_312, algoritmo)

    if clave in resultados_cv_ventanas:
        print(
            f"\n{algoritmo}: resultados existentes "
            "reutilizados."
        )

    else:
        resultados_cv_ventanas[clave] = (
            evaluar_cv_anidada_oulad(
                SEMANA_312,
                algoritmo
            )
        )

        print(
            f"\n{algoritmo}: evaluación completada.",
            flush=True
        )

# ------------------------------------------------------------
# 3. CONSOLIDAR RESULTADOS DE SEMANA 6
# ------------------------------------------------------------

tabla_s6 = pd.concat(
    [
        resultados_cv_ventanas[
            (SEMANA_312, algoritmo)
        ]["resultados"]

        for algoritmo in ALGORITMOS_312
    ],
    ignore_index=True
)

assert len(tabla_s6) == 40

print("\n=== 3.12.2 RENDIMIENTO MEDIO ===")

resumen_s6 = (
    tabla_s6.groupby(
        ["Algoritmo", "Experimento"]
    )
    .agg(
        F1_macro_medio=("F1_macro", "mean"),
        F1_macro_std=("F1_macro", "std"),
        Recall_adverso_medio=(
            "Recall_adverso", "mean"
        ),
        ROC_AUC_medio=("ROC_AUC", "mean"),
        Accuracy_media=("Accuracy", "mean")
    )
    .reset_index()
)

display(resumen_s6.round(4))

# ------------------------------------------------------------
# 4. COMPARACIÓN B - A
# ------------------------------------------------------------

print("\n=== 3.12.3 DIFERENCIAS PAREADAS B - A ===")

diferencias_s6 = []

for algoritmo in ALGORITMOS_312:

    pares = resultados_cv_ventanas[
        (SEMANA_312, algoritmo)
    ]["pares"]

    for metrica in METRICAS_312:

        delta = pares[f"Delta_{metrica}"]

        diferencias_s6.append({
            "Algoritmo": algoritmo,
            "Metrica": metrica,
            "Delta_medio_B_A": delta.mean(),
            "Delta_std": delta.std(ddof=1),
            "Delta_minimo": delta.min(),
            "Delta_maximo": delta.max(),
            "Folds_B_supera_A": int(
                (delta > 0).sum()
            ),
            "Folds_B_inferior_A": int(
                (delta < 0).sum()
            )
        })

tabla_diferencias_s6 = pd.DataFrame(
    diferencias_s6
)

display(tabla_diferencias_s6.round(4))

# ------------------------------------------------------------
# 5. AUDITORÍA DE SOBREAJUSTE
# ------------------------------------------------------------

print("\n=== 3.12.4 SOBREAJUSTE ===")

tabla_parametros_s6 = pd.concat(
    [
        resultados_cv_ventanas[
            (SEMANA_312, algoritmo)
        ]["parametros"]

        for algoritmo in ALGORITMOS_312
    ],
    ignore_index=True
)

brechas_s6 = (
    tabla_parametros_s6.groupby(
        ["Algoritmo", "Experimento"]
    )
    .agg(
        F1_train_medio=(
            "F1_train_interno", "mean"
        ),
        F1_validacion_medio=(
            "F1_val_interno", "mean"
        ),
        Brecha_media=(
            "Brecha_train_val", "mean"
        )
    )
    .reset_index()
)

display(brechas_s6.round(4))

# ------------------------------------------------------------
# 6. CONSOLIDAR SEMANAS 2, 4 Y 6
# ------------------------------------------------------------

print("\n=== 3.12.5 EVOLUCIÓN TEMPORAL ===")

# Recuperar resultados ya evaluados de semana 2.
tabla_s2_lr = tabla_lr_cv.copy()
tabla_s2_lr["Semana"] = 2
tabla_s2_lr["Algoritmo"] = "LogisticRegression"

tabla_s2_rf = tabla_rf_cv.copy()
tabla_s2_rf["Semana"] = 2
tabla_s2_rf["Algoritmo"] = "RandomForest"

# Recuperar las tablas de semana 4.
tabla_s4_lr = resultados_cv_ventanas[
    (4, "LogisticRegression")
]["resultados"].copy()

tabla_s4_rf = resultados_cv_ventanas[
    (4, "RandomForest")
]["resultados"].copy()

# Combinar con semana 6.
tabla_evolucion_312 = pd.concat(
    [
        tabla_s2_lr,
        tabla_s2_rf,
        tabla_s4_lr,
        tabla_s4_rf,
        tabla_s6
    ],
    ignore_index=True
)

resumen_temporal_312 = (
    tabla_evolucion_312.groupby(
        ["Semana", "Algoritmo", "Experimento"]
    )
    .agg(
        F1_macro=("F1_macro", "mean"),
        Recall_adverso=("Recall_adverso", "mean"),
        ROC_AUC=("ROC_AUC", "mean"),
        Accuracy=("Accuracy", "mean")
    )
    .reset_index()
)

display(resumen_temporal_312.round(4))

# ------------------------------------------------------------
# 7. MEJORA LONGITUDINAL POR SEMANA
# ------------------------------------------------------------

print("\n=== 3.12.6 EVOLUCIÓN DEL APORTE LONGITUDINAL ===")

tabla_deltas_temporales = []

for (semana, algoritmo), grupo in (
    resumen_temporal_312.groupby(
        ["Semana", "Algoritmo"]
    )
):

    a = grupo.loc[
        grupo["Experimento"] == "A"
    ].iloc[0]

    b = grupo.loc[
        grupo["Experimento"] == "B"
    ].iloc[0]

    tabla_deltas_temporales.append({
        "Semana": semana,
        "Algoritmo": algoritmo,
        "Delta_F1_macro": (
            b["F1_macro"] - a["F1_macro"]
        ),
        "Delta_Recall": (
            b["Recall_adverso"]
            - a["Recall_adverso"]
        ),
        "Delta_ROC_AUC": (
            b["ROC_AUC"] - a["ROC_AUC"]
        ),
        "Delta_Accuracy": (
            b["Accuracy"] - a["Accuracy"]
        )
    })

tabla_deltas_temporales = pd.DataFrame(
    tabla_deltas_temporales
)

display(
    tabla_deltas_temporales.sort_values(
        ["Algoritmo", "Semana"]
    ).round(4)
)

# ------------------------------------------------------------
# 8. CONTROLES FINALES
# ------------------------------------------------------------

print("\n=== 3.12.7 CONTROLES FINALES ===")

for algoritmo in ALGORITMOS_312:

    resultado = resultados_cv_ventanas[
        (SEMANA_312, algoritmo)
    ]

    assert len(resultado["resultados"]) == 20
    assert len(resultado["pares"]) == 10

    assert resultado["resultados"][
        ["F1_macro", "Recall_adverso", "ROC_AUC"]
    ].notna().all().all()

    pred = resultado["predicciones"]

    assert not pred.duplicated(
        subset=[
            "Experimento",
            "Repeticion",
            "code_module",
            "code_presentation",
            "id_student"
        ]
    ).any()

    print(
        algoritmo,
        "- evaluación y predicciones verificadas"
    )

assert (
    tabla_evolucion_312.groupby(
        ["Semana", "Algoritmo", "Experimento"]
    ).size() == 10
).all()

print(
    "\nResultados temporales: "
    "semanas 2, 4 y 6 consolidadas."
)

print("Conjunto de prueba independiente: NO utilizado.")
print("No se realizaron pruebas de significancia.")

print("\n=== ACTIVIDAD 3.12 FINALIZADA ===")



### 3.12.8. Resultados e interpretación

Se completó la validación cruzada anidada y repetida correspondiente a la semana 6, con cinco folds externos, dos repeticiones y tres folds internos de optimización.

La cohorte de desarrollo estuvo compuesta por 21.568 matrículas. Se mantuvieron las mismas reglas de partición por estudiante, preprocesamiento y selección de características establecidas en las ventanas anteriores.

**Rendimiento predictivo**

| Modelo | F1 macro | Recall adverso | ROC-AUC |
|---|---:|---:|---:|
| Regresión Logística A | 0,5976 | 0,4097 | 0,6571 |
| Regresión Logística B | 0,6997 | 0,5802 | 0,7741 |
| Random Forest A | 0,5879 | 0,4044 | 0,6460 |
| Random Forest B | 0,6869 | 0,5328 | 0,7625 |

Regresión Logística B obtuvo el mayor rendimiento promedio en las métricas principales.

**Aporte de las características longitudinales**

La diferencia media B-A fue de 0,1021 en F1 macro, 0,1705 en recall adverso y 0,1171 en ROC-AUC para Regresión Logística.

Para Random Forest, las diferencias correspondientes fueron de 0,0990, 0,1284 y 0,1165.

En ambos algoritmos, el Experimento B superó al A en los diez folds externos para F1 macro, recall adverso, ROC-AUC y accuracy.

**Evolución temporal**

Entre las semanas 2 y 6, el F1 macro de Regresión Logística B aumentó de 0,6656 a 0,6997, mientras que Random Forest B pasó de 0,6566 a 0,6869.

Durante el mismo período, los modelos estáticos presentaron una ligera disminución del F1 macro promedio.

La ventaja longitudinal B-A aumentó del 0,0580 al 0,1021 en Regresión Logística y del 0,0580 al 0,0990 en Random Forest.

Estos resultados son compatibles con un aumento del valor predictivo de la información longitudinal conforme avanza el módulo.

Sin embargo, las cohortes elegibles cambian entre semanas. Por tanto, no puede atribuirse la totalidad de la mejora al incremento de información temporal sin efectuar un análisis adicional con población fija.

**Evolución del recall adverso**

El recall de Regresión Logística B pasó de 0,5759 en la semana 2 a 0,5739 en la semana 4 y 0,5802 en la semana 6.

Random Forest B presentó valores de 0,5417, 0,5238 y 0,5328, respectivamente.

Por tanto, el incremento del F1 macro y ROC-AUC no se tradujo en un aumento monotónico del recall adverso.

**Análisis de sobreajuste**

La brecha media entre entrenamiento y validación interna de Random Forest B alcanzó 0,2376 en la semana 6, frente a 0,1550 en la semana 2 y 0,1960 en la semana 4.

Regresión Logística B mantuvo una brecha reducida de 0,0043 en la semana 6.

Esto sugiere que Random Forest B presenta sobreajuste relativo creciente, por lo que será importante examinar posteriormente su regularización mediante curvas de validación y aprendizaje.

### 3.12.9. Limitaciones

La evaluación se realizó sobre el conjunto de desarrollo mediante validación cruzada anidada y repetida.

Las particiones no son estadísticamente independientes debido al solapamiento entre muestras de entrenamiento.

La comparación temporal utiliza cohortes elegibles diferentes en cada semana.

No se han realizado pruebas confirmatorias sobre la muestra independiente.

### 3.12.10. Conclusión

Los resultados de la semana 6 refuerzan la evidencia empírica favorable al aporte predictivo de las características longitudinales.

Ambos algoritmos mostraron mejoras consistentes de F1 macro, recall adverso y ROC-AUC frente a sus versiones estáticas.

Regresión Logística B obtuvo el mejor desempeño promedio y una menor brecha de generalización que Random Forest B.

**Conclusión principal:** la información longitudinal presenta una ventaja predictiva creciente durante las primeras seis semanas evaluadas, aunque todavía es necesario controlar los cambios en la composición de las cohortes y estudiar el sobreajuste de Random Forest antes de establecer conclusiones generales.




### 3.13. Validación cruzada anidada — Semana 8

#### Objetivo

Completar la evaluación temporal de los experimentos A y B utilizando la información académica disponible durante las primeras ocho semanas del módulo.

#### Diseño experimental

La predicción se realizará al inicio del día 56, utilizando exclusivamente registros anteriores a ese momento.

La cohorte de desarrollo contiene 21.164 matrículas y el conjunto de prueba independiente contiene 5.249 matrículas.

Se evaluarán:

- Regresión Logística A y B.
- Random Forest A y B.

Se utilizarán cinco folds externos, dos repeticiones y tres folds internos para optimización, manteniendo la agrupación por identificador de estudiante.

Los espacios de búsqueda, procedimientos de preprocesamiento y métodos de selección de características serán los mismos de las actividades anteriores.

#### Métricas

Se utilizará F1 macro como métrica principal. También se evaluarán recall adverso, ROC-AUC y accuracy.

Se calcularán diferencias pareadas entre los experimentos longitudinal y estático dentro de cada algoritmo.

#### Evaluación de sobreajuste

Se compararán los promedios de F1 macro de entrenamiento y validación interna, prestando especial atención a la evolución de Random Forest B.

#### Integración temporal

Al finalizar se consolidarán los resultados de las semanas 2, 4, 6 y 8.

Esta integración permitirá describir:

1. La evolución del rendimiento de los cuatro modelos.
2. La evolución del aporte longitudinal B-A.
3. El comportamiento de las métricas de identificación adversa.
4. La evolución de las brechas de generalización.

#### Limitaciones

Las cohortes contienen diferentes matrículas elegibles y sus proporciones de resultados adversos también varían.

Por tanto, las diferencias entre semanas serán consideradas descriptivas y no se atribuirán exclusivamente al incremento de información temporal.

La siguiente fase incorporará un análisis de sensibilidad sobre una población común a las cuatro ventanas, además de las curvas de aprendizaje y los contrastes estadísticos.

El conjunto de prueba independiente continuará reservado.


In [ ]:

# ============================================================
# PROYECTO FINAL DE MACHINE LEARNING
# ACTIVIDAD 3.13 - SEMANA 8 Y COMPARACIÓN TEMPORAL
# ============================================================

# Asistencia IA: ChatGPT, octubre de 2026.
# Prompt:
# "Completar validación cruzada anidada de la semana 8
# de OULAD y consolidar los resultados de semanas 2,
# 4, 6 y 8. Comparar F1 macro, recall y ROC-AUC,
# calcular diferencias longitudinales y brechas de
# generalización sin utilizar prueba independiente."

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

# ------------------------------------------------------------
# 1. CONFIGURACIÓN
# ------------------------------------------------------------

SEMANA_313 = 8

ALGORITMOS_313 = [
    "LogisticRegression",
    "RandomForest"
]

METRICAS_313 = [
    "F1_macro",
    "Recall_adverso",
    "ROC_AUC",
    "Accuracy"
]

assert "evaluar_cv_anidada_oulad" in globals()
assert "resultados_cv_ventanas" in globals()

print("=== 3.13.1 CONFIGURACIÓN ===")

print("Semana:", SEMANA_313)
print("Último día observable:", 55)

print(
    "Matrículas desarrollo:",
    len(experimentos_oulad[8]["y_train"])
)

print(
    "Matrículas prueba reservada:",
    len(experimentos_oulad[8]["y_test"])
)

# ------------------------------------------------------------
# 2. EJECUTAR VALIDACIÓN ANIDADA
# ------------------------------------------------------------

for algoritmo in ALGORITMOS_313:

    clave = (SEMANA_313, algoritmo)

    if clave not in resultados_cv_ventanas:

        resultados_cv_ventanas[clave] = (
            evaluar_cv_anidada_oulad(
                SEMANA_313,
                algoritmo
            )
        )

        print(
            f"\n{algoritmo}: evaluación completada.",
            flush=True
        )

    else:

        print(
            f"\n{algoritmo}: resultados ya disponibles."
        )

# ------------------------------------------------------------
# 3. RENDIMIENTO MEDIO DE SEMANA 8
# ------------------------------------------------------------

print("\n=== 3.13.2 RENDIMIENTO MEDIO ===")

tabla_s8 = pd.concat(
    [
        resultados_cv_ventanas[
            (8, algoritmo)
        ]["resultados"]

        for algoritmo in ALGORITMOS_313
    ],
    ignore_index=True
)

assert len(tabla_s8) == 40

resumen_s8 = (
    tabla_s8.groupby(
        ["Algoritmo", "Experimento"]
    )
    .agg(
        F1_macro_medio=("F1_macro", "mean"),
        F1_macro_std=("F1_macro", "std"),
        Recall_adverso_medio=(
            "Recall_adverso", "mean"
        ),
        ROC_AUC_medio=("ROC_AUC", "mean"),
        Accuracy_media=("Accuracy", "mean")
    )
    .reset_index()
)

display(resumen_s8.round(4))

# ------------------------------------------------------------
# 4. DIFERENCIAS PAREADAS B - A
# ------------------------------------------------------------

print("\n=== 3.13.3 DIFERENCIAS PAREADAS B - A ===")

filas_deltas_s8 = []

for algoritmo in ALGORITMOS_313:

    pares = resultados_cv_ventanas[
        (8, algoritmo)
    ]["pares"]

    assert len(pares) == 10

    for metrica in METRICAS_313:

        delta = pares[
            f"Delta_{metrica}"
        ]

        filas_deltas_s8.append({
            "Algoritmo": algoritmo,
            "Metrica": metrica,
            "Delta_medio_B_A": delta.mean(),
            "Delta_std": delta.std(ddof=1),
            "Delta_minimo": delta.min(),
            "Delta_maximo": delta.max(),
            "Folds_B_supera_A": int(
                (delta > 0).sum()
            ),
            "Folds_B_inferior_A": int(
                (delta < 0).sum()
            )
        })

tabla_deltas_s8 = pd.DataFrame(
    filas_deltas_s8
)

display(tabla_deltas_s8.round(4))

# ------------------------------------------------------------
# 5. SOBREAJUSTE DE SEMANA 8
# ------------------------------------------------------------

print("\n=== 3.13.4 SOBREAJUSTE ===")

params_s8 = pd.concat(
    [
        resultados_cv_ventanas[
            (8, algoritmo)
        ]["parametros"]

        for algoritmo in ALGORITMOS_313
    ],
    ignore_index=True
)

brechas_s8 = (
    params_s8.groupby(
        ["Algoritmo", "Experimento"]
    )
    .agg(
        F1_train_medio=(
            "F1_train_interno", "mean"
        ),
        F1_validacion_medio=(
            "F1_val_interno", "mean"
        ),
        Brecha_media=(
            "Brecha_train_val", "mean"
        )
    )
    .reset_index()
)

display(brechas_s8.round(4))

# ------------------------------------------------------------
# 6. CONSOLIDAR LAS CUATRO SEMANAS
# ------------------------------------------------------------

print("\n=== 3.13.5 EVOLUCIÓN TEMPORAL COMPLETA ===")

# Semana 2
s2_lr = tabla_lr_cv.copy()
s2_lr["Semana"] = 2
s2_lr["Algoritmo"] = "LogisticRegression"

s2_rf = tabla_rf_cv.copy()
s2_rf["Semana"] = 2
s2_rf["Algoritmo"] = "RandomForest"

tablas = [s2_lr, s2_rf]

# Semanas 4, 6 y 8
for semana in [4, 6, 8]:

    for algoritmo in ALGORITMOS_313:

        clave = (semana, algoritmo)

        if clave not in resultados_cv_ventanas:
            raise ValueError(
                f"Faltan resultados para {clave}"
            )

        tablas.append(
            resultados_cv_ventanas[
                clave
            ]["resultados"].copy()
        )

tabla_temporal_completa = pd.concat(
    tablas,
    ignore_index=True
)

# 4 semanas x 2 algoritmos x 2 experimentos
# x 10 folds = 160 evaluaciones.
assert len(tabla_temporal_completa) == 160

assert (
    tabla_temporal_completa.groupby(
        ["Semana", "Algoritmo", "Experimento"]
    ).size().eq(10).all()
)

resumen_temporal_completo = (
    tabla_temporal_completa.groupby(
        ["Semana", "Algoritmo", "Experimento"]
    )
    .agg(
        F1_macro=("F1_macro", "mean"),
        Recall_adverso=("Recall_adverso", "mean"),
        ROC_AUC=("ROC_AUC", "mean"),
        Accuracy=("Accuracy", "mean")
    )
    .reset_index()
)

display(
    resumen_temporal_completo.sort_values(
        ["Algoritmo", "Experimento", "Semana"]
    ).round(4)
)

# ------------------------------------------------------------
# 7. APORTE LONGITUDINAL EN CADA SEMANA
# ------------------------------------------------------------

print("\n=== 3.13.6 EVOLUCIÓN DEL APORTE LONGITUDINAL ===")

deltas_temporales = []

for (semana, algoritmo), grupo in (
    resumen_temporal_completo.groupby(
        ["Semana", "Algoritmo"]
    )
):

    a = grupo.loc[
        grupo["Experimento"] == "A"
    ].iloc[0]

    b = grupo.loc[
        grupo["Experimento"] == "B"
    ].iloc[0]

    deltas_temporales.append({
        "Semana": semana,
        "Algoritmo": algoritmo,
        "Delta_F1_macro": (
            b["F1_macro"] - a["F1_macro"]
        ),
        "Delta_Recall": (
            b["Recall_adverso"]
            - a["Recall_adverso"]
        ),
        "Delta_ROC_AUC": (
            b["ROC_AUC"] - a["ROC_AUC"]
        ),
        "Delta_Accuracy": (
            b["Accuracy"] - a["Accuracy"]
        )
    })

tabla_deltas_temporales_completa = (
    pd.DataFrame(deltas_temporales)
    .sort_values(["Algoritmo", "Semana"])
    .reset_index(drop=True)
)

display(
    tabla_deltas_temporales_completa.round(4)
)

# ------------------------------------------------------------
# 8. GRÁFICO F1 MACRO POR SEMANA
# ------------------------------------------------------------

print("\n=== 3.13.7 GRÁFICOS TEMPORALES ===")

plt.figure(figsize=(10, 5))

for algoritmo in ALGORITMOS_313:

    for experimento in ["A", "B"]:

        sub = resumen_temporal_completo.loc[
            (
                resumen_temporal_completo[
                    "Algoritmo"
                ] == algoritmo
            )
            &
            (
                resumen_temporal_completo[
                    "Experimento"
                ] == experimento
            )
        ].sort_values("Semana")

        nombre = (
            ("LR" if algoritmo == "LogisticRegression"
             else "RF")
            + " "
            + experimento
        )

        plt.plot(
            sub["Semana"],
            sub["F1_macro"],
            marker="o",
            linewidth=2,
            label=nombre
        )

plt.title(
    "Evolución del F1 macro - Semanas 2, 4, 6 y 8"
)

plt.xlabel("Semana de predicción")
plt.ylabel("F1 macro")
plt.xticks([2, 4, 6, 8])
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 9. GRÁFICO DEL APORTE LONGITUDINAL
# ------------------------------------------------------------

plt.figure(figsize=(10, 5))

for algoritmo in ALGORITMOS_313:

    sub = tabla_deltas_temporales_completa.loc[
        tabla_deltas_temporales_completa[
            "Algoritmo"
        ] == algoritmo
    ].sort_values("Semana")

    plt.plot(
        sub["Semana"],
        sub["Delta_F1_macro"],
        marker="o",
        linewidth=2,
        label=algoritmo
    )

plt.axhline(
    0,
    linestyle="--",
    linewidth=1
)

plt.title(
    "Aporte longitudinal: diferencia de F1 macro B - A"
)

plt.xlabel("Semana de predicción")
plt.ylabel("Diferencia de F1 macro")
plt.xticks([2, 4, 6, 8])
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 10. CONTROL FINAL
# ------------------------------------------------------------

print("\n=== 3.13.8 CONTROLES FINALES ===")

for algoritmo in ALGORITMOS_313:

    resultado = resultados_cv_ventanas[
        (8, algoritmo)
    ]

    assert len(resultado["resultados"]) == 20
    assert len(resultado["pares"]) == 10

    pred = resultado["predicciones"]

    assert not pred.duplicated(
        subset=[
            "Experimento",
            "Repeticion",
            "code_module",
            "code_presentation",
            "id_student"
        ]
    ).any()

    esperado = (
        4 * len(experimentos_oulad[8]["y_train"])
    )

    assert len(pred) == esperado

    print(
        algoritmo,
        "- métricas y predicciones verificadas."
    )

print(
    "Total evaluaciones externas consolidadas:",
    len(tabla_temporal_completa)
)

print("Semanas analizadas: 2, 4, 6 y 8")
print("Conjunto de prueba independiente: NO utilizado.")
print("No se realizaron pruebas de significancia.")

print("\n=== ACTIVIDAD 3.13 FINALIZADA ===")



### 3.13.9. Resultados e interpretación

Se completó la validación cruzada anidada y repetida de la semana 8, finalizando la evaluación inicial de las cuatro ventanas temporales propuestas.

La cohorte de desarrollo de la semana 8 contiene 21.164 matrículas, mientras que 5.249 matrículas permanecen reservadas para prueba independiente.

Se consolidaron 160 evaluaciones externas correspondientes a cuatro semanas, dos algoritmos, dos experimentos y diez folds externos por configuración.

**Resultados de la semana 8**

| Modelo | F1 macro | Recall adverso | ROC-AUC |
|---|---:|---:|---:|
| Regresión Logística A | 0,5907 | 0,3779 | 0,6561 |
| Regresión Logística B | 0,7228 | 0,5908 | 0,7995 |
| Random Forest A | 0,5817 | 0,3804 | 0,6432 |
| Random Forest B | 0,7006 | 0,5258 | 0,7837 |

Regresión Logística B presentó el mejor desempeño promedio entre los cuatro modelos.

**Aporte de las características longitudinales**

La mejora media B-A de F1 macro fue de 0,1321 para Regresión Logística y 0,1189 para Random Forest.

Las mejoras de recall adverso fueron de 0,2128 y 0,1454, respectivamente.

Asimismo, las diferencias de ROC-AUC fueron de 0,1433 para Regresión Logística y 0,1405 para Random Forest.

En ambos algoritmos, el Experimento B superó al A en los diez folds externos para todas las métricas evaluadas.

**Evolución durante las primeras ocho semanas**

Los modelos estáticos presentaron una reducción progresiva de F1 macro entre las semanas 2 y 8.

Regresión Logística A disminuyó de 0,6076 a 0,5907, mientras que Random Forest A pasó de 0,5986 a 0,5817.

Los modelos longitudinales mostraron la tendencia contraria.

Regresión Logística B aumentó de 0,6656 a 0,7228, mientras que Random Forest B pasó de 0,6566 a 0,7006.

La ventaja longitudinal B-A aumentó de 0,0580 a 0,1321 para Regresión Logística y de 0,0580 a 0,1189 para Random Forest.

Este comportamiento es consistente con un aporte predictivo creciente de las características longitudinales en las cohortes observadas.

No obstante, las cohortes elegibles cambian entre semanas y el número de características longitudinales también aumenta. Por tanto, estas diferencias no permiten atribuir causalmente la totalidad de la mejora al incremento de semanas de observación.

**Análisis de sobreajuste**

En la semana 8, Random Forest B presentó un F1 macro de entrenamiento interno de 0,9214 y de validación interna de 0,7003, con una brecha media de 0,2211.

Esta brecha fue inferior a la de la semana 6 (0,2376), aunque continuó siendo considerable.

Regresión Logística B mantuvo una brecha reducida de 0,0042.

Estos hallazgos justifican realizar análisis adicionales de regularización y curvas de aprendizaje.

### 3.13.10. Limitaciones

Las evaluaciones se realizaron sobre conjuntos de desarrollo, utilizando validación cruzada anidada y agrupada por estudiante.

Los folds repetidos no representan observaciones estadísticamente independientes.

Las poblaciones elegibles difieren entre ventanas temporales debido principalmente a cambios en la matrícula activa.

La comparación temporal presentada es descriptiva y requiere un análisis adicional de sensibilidad sobre una población común.

La variable objetivo combina reprobación y retiro del módulo, sin representar directamente deserción institucional.

El conjunto de prueba independiente permanece reservado.

### 3.13.11. Conclusión

Los resultados de las cuatro ventanas temporales proporcionan evidencia empírica consistente a favor del valor predictivo de la información longitudinal del comportamiento estudiantil.

La ventaja del Experimento B sobre el A se observó en los dos algoritmos y fue positiva en todos los folds externos de las cuatro semanas.

Regresión Logística B obtuvo el mejor rendimiento promedio y presentó menores señales de sobreajuste que Random Forest B.

**Conclusión principal:** la incorporación de trayectorias longitudinales permite mejorar la clasificación temprana de resultados académicos adversos en OULAD. La magnitud de esta ventaja aumenta descriptivamente conforme avanza el curso, aunque será necesario controlar la composición de las cohortes y realizar una evaluación estadística e independiente antes de establecer conclusiones definitivas.



### 3.14. Análisis de sensibilidad temporal con población común

#### Objetivo

Examinar si la tendencia de mejora de los modelos longitudinales se mantiene al evaluar las cuatro ventanas sobre una misma población de matrículas.

#### Justificación

Las cohortes utilizadas en las semanas 2, 4, 6 y 8 presentan diferentes tamaños debido a los cambios en la elegibilidad temporal.

Estas diferencias pueden influir en las métricas y dificultar la atribución del cambio de rendimiento exclusivamente a la información longitudinal adicional.

#### Procedimiento

Se identificará la intersección de las matrículas de desarrollo elegibles en las cuatro ventanas.

Para estas matrículas se recuperarán las predicciones fuera de muestra generadas mediante validación cruzada anidada y repetida.

Las predicciones se compararán utilizando la misma población en cada semana, sin entrenar nuevamente los clasificadores.

La comparación incluirá:

- F1 macro.
- Recall de la clase adversa.
- ROC-AUC.
- Accuracy.
- Diferencias entre los experimentos B y A.

Se conservarán las dos repeticiones por separado, sin tratarlas como observaciones independientes.

#### Alcance de la comparación

Este procedimiento mantiene fija la población de evaluación, pero no la población de entrenamiento utilizada originalmente en cada semana.

Los modelos también fueron ajustados mediante particiones de validación generadas independientemente para cada ventana.

Por tanto, se trata de un análisis de sensibilidad basado en predicciones fuera de muestra, no de un experimento completamente controlado con entrenamiento y particiones idénticos entre semanas.

#### Limitación por selección retrospectiva

La pertenencia a la cohorte común depende de permanecer elegible hasta la semana 8.

Esta selección utiliza información futura con respecto a la predicción de la semana 2 y excluye retiros tempranos.

Como consecuencia, la cohorte común no representa a todos los estudiantes a quienes se habría podido emitir una alerta temprana en la semana 2.

Los resultados se interpretarán únicamente como un análisis retrospectivo de robustez frente a cambios en la composición de la población.

El conjunto de prueba independiente permanecerá reservado.


In [ ]:

# ============================================================
# PROYECTO FINAL DE MACHINE LEARNING
# ACTIVIDAD 3.14 - SENSIBILIDAD CON COHORTE COMÚN
# SIN REENTRENAR MODELOS
# ============================================================

# Asistencia IA: ChatGPT, octubre de 2026.
# Prompt:
# "Identificar las matrículas de OULAD elegibles en
# semanas 2, 4, 6 y 8; recuperar las predicciones
# fuera de muestra de validación cruzada anidada;
# recalcular métricas sobre la misma cohorte para
# comparar la evolución temporal sin reentrenar
# modelos ni acceder al conjunto de prueba."

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    f1_score,
    recall_score,
    roc_auc_score,
    accuracy_score
)

from IPython.display import display

SEMANAS_314 = [2, 4, 6, 8]

ALGORITMOS_314 = [
    "LogisticRegression",
    "RandomForest"
]

CLAVES_314 = [
    "code_module",
    "code_presentation",
    "id_student"
]

# ------------------------------------------------------------
# 1. IDENTIFICAR MATRÍCULAS COMUNES
# ------------------------------------------------------------

print("=== 3.14.1 COHORTE COMÚN ===")

desarrollo = cohortes_oulad.loc[
    cohortes_oulad["conjunto"] == "Entrenamiento"
].copy()

assert not desarrollo.duplicated(
    subset=["semana_prediccion"] + CLAVES_314
).any()

control_matriculas = (
    desarrollo.groupby(CLAVES_314)
    .agg(
        Ventanas_elegibles=(
            "semana_prediccion", "nunique"
        ),
        Etiquetas_distintas=(
            "target_adverso", "nunique"
        )
    )
    .reset_index()
)

assert control_matriculas[
    "Etiquetas_distintas"
].eq(1).all()

cohorte_comun_314 = control_matriculas.loc[
    control_matriculas["Ventanas_elegibles"]
    == len(SEMANAS_314),
    CLAVES_314
].copy()

N_COMUN = len(cohorte_comun_314)

assert N_COMUN > 0

print("Matrículas comunes:", N_COMUN)

print(
    "% respecto a semana 2:",
    round(
        100 * N_COMUN / 22412,
        2
    )
)

# ------------------------------------------------------------
# 2. DESCRIBIR LA COMPOSICIÓN DE LA COHORTE
# ------------------------------------------------------------

print("\n=== 3.14.2 COMPOSICIÓN ACADÉMICA ===")

meta_comun = (
    desarrollo.loc[
        desarrollo["semana_prediccion"] == 2,
        CLAVES_314 + [
            "final_result",
            "target_adverso"
        ]
    ]
    .merge(
        cohorte_comun_314,
        on=CLAVES_314,
        how="inner",
        validate="one_to_one"
    )
)

tabla_composicion_314 = (
    meta_comun["final_result"]
    .value_counts()
    .rename_axis("Resultado")
    .reset_index(name="Matriculas")
)

tabla_composicion_314["Porcentaje"] = (
    100
    * tabla_composicion_314["Matriculas"]
    / N_COMUN
).round(2)

display(tabla_composicion_314)

prevalencia_comun = (
    100 * meta_comun["target_adverso"].mean()
)

print(
    "Prevalencia adversa de la cohorte común:",
    f"{prevalencia_comun:.2f}%"
)

# ------------------------------------------------------------
# 3. RECUPERAR PREDICCIONES DE SEMANA 2
# ------------------------------------------------------------

print("\n=== 3.14.3 RECUPERACIÓN DE PREDICCIONES ===")

COLUMNAS_PRED_314 = (
    CLAVES_314
    + [
        "Semana",
        "Repeticion",
        "Algoritmo",
        "Experimento",
        "y_real",
        "y_pred",
        "prob_adverso"
    ]
)

pred_s2 = predicciones_conjuntas_s2[
    [
        c for c in COLUMNAS_PRED_314
        if c != "Semana"
    ]
].copy()

pred_s2["Semana"] = 2

pred_s2 = pred_s2[
    COLUMNAS_PRED_314
]

fuentes = [pred_s2]

# ------------------------------------------------------------
# 4. RECUPERAR PREDICCIONES DE SEMANAS 4, 6 Y 8
# ------------------------------------------------------------

for semana in [4, 6, 8]:

    for algoritmo in ALGORITMOS_314:

        clave = (semana, algoritmo)

        assert clave in resultados_cv_ventanas, (
            f"Faltan resultados para {clave}."
        )

        pred = resultados_cv_ventanas[
            clave
        ]["predicciones"].copy()

        fuentes.append(
            pred[COLUMNAS_PRED_314]
        )

predicciones_todas_314 = pd.concat(
    fuentes,
    ignore_index=True
)

# Cada matrícula se predice una vez por repetición
# dentro de cada configuración.

CLAVE_OOF = [
    "Semana",
    "Repeticion",
    "Algoritmo",
    "Experimento"
] + CLAVES_314

assert not predicciones_todas_314.duplicated(
    subset=CLAVE_OOF
).any()

# ------------------------------------------------------------
# 5. FILTRAR PREDICCIONES A LA POBLACIÓN COMÚN
# ------------------------------------------------------------

predicciones_comunes_314 = (
    predicciones_todas_314.merge(
        cohorte_comun_314,
        on=CLAVES_314,
        how="inner",
        validate="many_to_one"
    )
)

conteo = (
    predicciones_comunes_314.groupby(
        [
            "Semana",
            "Algoritmo",
            "Experimento",
            "Repeticion"
        ]
    )
    .size()
)

assert len(conteo) == 32
assert conteo.eq(N_COMUN).all()

assert predicciones_comunes_314[
    "prob_adverso"
].between(0, 1).all()

print(
    "Predicciones comunes recuperadas:",
    len(predicciones_comunes_314)
)

print(
    "Matrículas por configuración y repetición:",
    N_COMUN
)

# ------------------------------------------------------------
# 6. CALCULAR MÉTRICAS SOBRE LA COHORTE COMÚN
# ------------------------------------------------------------

print("\n=== 3.14.4 MÉTRICAS SOBRE POBLACIÓN FIJA ===")

filas_metricas = []

for (
    semana,
    algoritmo,
    experimento,
    repeticion
), sub in predicciones_comunes_314.groupby(
    [
        "Semana",
        "Algoritmo",
        "Experimento",
        "Repeticion"
    ]
):

    y_real = sub["y_real"].to_numpy()
    y_pred = sub["y_pred"].to_numpy()
    prob = sub["prob_adverso"].to_numpy()

    assert set(np.unique(y_real)) == {0, 1}

    filas_metricas.append({
        "Semana": semana,
        "Algoritmo": algoritmo,
        "Experimento": experimento,
        "Repeticion": repeticion,
        "Matriculas": len(sub),
        "F1_macro": f1_score(
            y_real,
            y_pred,
            average="macro",
            zero_division=0
        ),
        "Recall_adverso": recall_score(
            y_real,
            y_pred,
            zero_division=0
        ),
        "ROC_AUC": roc_auc_score(
            y_real,
            prob
        ),
        "Accuracy": accuracy_score(
            y_real,
            y_pred
        )
    })

tabla_metricas_comunes_314 = pd.DataFrame(
    filas_metricas
)

resumen_comun_314 = (
    tabla_metricas_comunes_314.groupby(
        ["Semana", "Algoritmo", "Experimento"]
    )
    .agg(
        F1_macro=("F1_macro", "mean"),
        Recall_adverso=("Recall_adverso", "mean"),
        ROC_AUC=("ROC_AUC", "mean"),
        Accuracy=("Accuracy", "mean")
    )
    .reset_index()
)

display(
    resumen_comun_314.sort_values(
        ["Algoritmo", "Experimento", "Semana"]
    ).round(4)
)

# ------------------------------------------------------------
# 7. CALCULAR DIFERENCIAS PAREADAS B - A
# ------------------------------------------------------------

print("\n=== 3.14.5 APORTE LONGITUDINAL EN COHORTE COMÚN ===")

columnas_metrica = [
    "F1_macro",
    "Recall_adverso",
    "ROC_AUC",
    "Accuracy"
]

pares_comunes_314 = (
    tabla_metricas_comunes_314.pivot(
        index=[
            "Semana",
            "Algoritmo",
            "Repeticion"
        ],
        columns="Experimento",
        values=columnas_metrica
    )
)

pares_comunes_314.columns = [
    f"{metrica}_{experimento}"
    for metrica, experimento
    in pares_comunes_314.columns
]

pares_comunes_314 = (
    pares_comunes_314.reset_index()
)

for metrica in columnas_metrica:

    pares_comunes_314[f"Delta_{metrica}"] = (
        pares_comunes_314[f"{metrica}_B"]
        - pares_comunes_314[f"{metrica}_A"]
    )

resumen_deltas_comunes_314 = (
    pares_comunes_314.groupby(
        ["Semana", "Algoritmo"]
    )
    .agg(
        Delta_F1_macro=(
            "Delta_F1_macro", "mean"
        ),
        Delta_Recall=(
            "Delta_Recall_adverso", "mean"
        ),
        Delta_ROC_AUC=(
            "Delta_ROC_AUC", "mean"
        ),
        Delta_Accuracy=(
            "Delta_Accuracy", "mean"
        )
    )
    .reset_index()
)

display(
    resumen_deltas_comunes_314.sort_values(
        ["Algoritmo", "Semana"]
    ).round(4)
)

# ------------------------------------------------------------
# 8. COMPARACIÓN ORIGINAL VS COHORTE COMÚN
# ------------------------------------------------------------

print("\n=== 3.14.6 COMPARACIÓN DE SENSIBILIDAD ===")

original = tabla_deltas_temporales_completa[
    [
        "Semana",
        "Algoritmo",
        "Delta_F1_macro"
    ]
].rename(
    columns={
        "Delta_F1_macro":
        "Delta_F1_cohorte_original"
    }
)

comun = resumen_deltas_comunes_314[
    [
        "Semana",
        "Algoritmo",
        "Delta_F1_macro"
    ]
].rename(
    columns={
        "Delta_F1_macro":
        "Delta_F1_cohorte_comun"
    }
)

comparacion_sensibilidad_314 = (
    original.merge(
        comun,
        on=["Semana", "Algoritmo"],
        validate="one_to_one"
    )
)

comparacion_sensibilidad_314[
    "Diferencia_comun_menos_original"
] = (
    comparacion_sensibilidad_314[
        "Delta_F1_cohorte_comun"
    ]
    - comparacion_sensibilidad_314[
        "Delta_F1_cohorte_original"
    ]
)

display(
    comparacion_sensibilidad_314.sort_values(
        ["Algoritmo", "Semana"]
    ).round(4)
)

# ------------------------------------------------------------
# 9. GRAFICAR F1 MACRO CON POBLACIÓN COMÚN
# ------------------------------------------------------------

print("\n=== 3.14.7 GRÁFICOS ===")

plt.figure(figsize=(10, 5))

for algoritmo in ALGORITMOS_314:

    for experimento in ["A", "B"]:

        sub = resumen_comun_314.loc[
            (
                resumen_comun_314[
                    "Algoritmo"
                ] == algoritmo
            )
            &
            (
                resumen_comun_314[
                    "Experimento"
                ] == experimento
            )
        ].sort_values("Semana")

        etiqueta = (
            ("LR" if algoritmo == "LogisticRegression"
             else "RF")
            + " "
            + experimento
        )

        plt.plot(
            sub["Semana"],
            sub["F1_macro"],
            marker="o",
            linewidth=2,
            label=etiqueta
        )

plt.title(
    "Evolución del F1 macro - Cohorte común"
)
plt.xlabel("Semana de predicción")
plt.ylabel("F1 macro")
plt.xticks(SEMANAS_314)
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 10. GRAFICAR APORTE LONGITUDINAL
# ------------------------------------------------------------

plt.figure(figsize=(10, 5))

for algoritmo in ALGORITMOS_314:

    sub = resumen_deltas_comunes_314.loc[
        resumen_deltas_comunes_314[
            "Algoritmo"
        ] == algoritmo
    ].sort_values("Semana")

    plt.plot(
        sub["Semana"],
        sub["Delta_F1_macro"],
        marker="o",
        linewidth=2,
        label=algoritmo
    )

plt.axhline(
    0,
    linestyle="--",
    linewidth=1
)

plt.title(
    "Aporte longitudinal B - A en la cohorte común"
)
plt.xlabel("Semana de predicción")
plt.ylabel("Diferencia de F1 macro")
plt.xticks(SEMANAS_314)
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 11. CONTROLES FINALES
# ------------------------------------------------------------

print("\n=== 3.14.8 CONTROLES FINALES ===")

assert len(tabla_metricas_comunes_314) == 32
assert len(resumen_comun_314) == 16
assert len(resumen_deltas_comunes_314) == 8

assert not predicciones_comunes_314.duplicated(
    subset=CLAVE_OOF
).any()

print("Matrículas comunes:", N_COMUN)
print("Modelos reentrenados: 0")
print("Predicciones utilizadas: fuera de muestra")
print("Conjunto de prueba independiente: NO utilizado")

print(
    "Limitación: selección retrospectiva por "
    "elegibilidad hasta semana 8."
)

print(
    "Las poblaciones de entrenamiento y los folds "
    "originales todavía varían entre semanas."
)

print("\n=== ACTIVIDAD 3.14 FINALIZADA ===")



### 3.14.9. Resultados e interpretación

Se realizó un análisis retrospectivo de sensibilidad para examinar la evolución del rendimiento predictivo sobre una población común a las cuatro ventanas temporales.

La intersección de las cohortes de desarrollo permitió identificar 21.126 matrículas elegibles en las semanas 2, 4, 6 y 8, equivalentes al 94,26 % de las matrículas del conjunto de desarrollo de la semana 2.

La prevalencia de resultados adversos en esta población fue del 41,82 %.

Se reutilizaron las predicciones fuera de muestra previamente generadas mediante validación cruzada anidada, sin volver a entrenar clasificadores ni utilizar el conjunto de prueba independiente.

**Rendimiento sobre la cohorte común**

| Semana | F1 macro LR A | F1 macro LR B | F1 macro RF A | F1 macro RF B |
|---|---:|---:|---:|---:|
| 2 | 0,6043 | 0,6587 | 0,5954 | 0,6509 |
| 4 | 0,6028 | 0,6808 | 0,5941 | 0,6670 |
| 6 | 0,5977 | 0,6974 | 0,5879 | 0,6851 |
| 8 | 0,5907 | 0,7229 | 0,5817 | 0,7006 |

Los modelos longitudinales mantuvieron una tendencia ascendente de F1 macro, mientras que los modelos estáticos presentaron una ligera disminución.

**Aporte longitudinal con población fija**

La mejora de F1 macro del Experimento B respecto al A aumentó progresivamente:

- Regresión Logística: de 0,0544 en semana 2 a 0,1322 en semana 8.
- Random Forest: de 0,0555 en semana 2 a 0,1189 en semana 8.

Las diferencias de la ventaja longitudinal respecto a las cohortes originales fueron pequeñas, con una variación máxima absoluta de 0,0036 puntos de F1 macro.

Esto indica que la tendencia observada es robusta frente a la restricción de la población de evaluación.

**Análisis de resultados adversos**

La cohorte común contiene 5.582 matrículas con reprobación y 3.252 con retiro del módulo.

La restricción retrospectiva excluyó retiros tempranos que estaban presentes en la cohorte de la semana 2 y dejaron de ser elegibles antes de la semana 8.

Por esta razón, la cohorte común no representa la población completa disponible para intervención temprana.

### 3.14.10. Limitaciones

El análisis mantuvo fija la población de evaluación, pero no la población de entrenamiento ni las particiones originales de validación entre semanas.

Los modelos y sus hiperparámetros fueron ajustados separadamente para cada ventana temporal.

La selección de matrículas elegibles hasta la semana 8 utiliza información futura respecto a los cortes anteriores y genera un posible sesgo de supervivencia.

La comparación es retrospectiva y descriptiva; no identifica causalmente el efecto de incorporar semanas adicionales de información.

Las dos repeticiones corresponden a las mismas matrículas y no constituyen observaciones independientes.

### 3.14.11. Conclusión

El análisis de sensibilidad confirmó que la mejora progresiva de los modelos longitudinales persiste al evaluar las cuatro ventanas sobre una población común de 21.126 matrículas.

La magnitud de las diferencias B-A fue similar a la observada en las cohortes originales.

**Conclusión principal:** la tendencia de incremento del aporte predictivo longitudinal no desaparece al controlar la composición de la población de evaluación. Sin embargo, su interpretación continúa limitada por diferencias en los conjuntos de entrenamiento, los procedimientos de validación y la selección retrospectiva de estudiantes elegibles.



### 3.15. Curvas de aprendizaje y validación de Random Forest B

#### Objetivo

Investigar el sobreajuste observado en Random Forest B mediante el análisis de su rendimiento en entrenamiento y validación, considerando el tamaño del conjunto de entrenamiento y la profundidad máxima de los árboles.

#### Contexto

Durante la semana 8, Random Forest B presentó un F1 macro medio de entrenamiento interno de 0,9214 y de validación interna de 0,7003, con una brecha de generalización de 0,2211.

Esta diferencia sugiere sobreajuste y justifica evaluar si el problema se relaciona con la cantidad de datos utilizados o con la complejidad del modelo.

#### Curva de aprendizaje

Se construirá una curva con diferentes proporciones de estudiantes disponibles para entrenamiento.

El muestreo se realizará por identificador de estudiante, conservando juntas todas sus matrículas académicas.

Se evaluarán cinco proporciones de grupos de entrenamiento: 20 %, 40 %, 60 %, 80 % y 100 %.

Para cada tamaño se ajustará nuevamente el pipeline completo y se calculará F1 macro sobre entrenamiento y validación.

La curva permitirá observar si el rendimiento de validación mejora conforme aumenta la cantidad de estudiantes utilizados durante el ajuste.

#### Curva de validación

Se evaluarán distintas profundidades máximas para Random Forest:

- 4 niveles.
- 8 niveles.
- 12 niveles.
- 16 niveles.
- Sin límite explícito de profundidad.

Para aislar el efecto de este hiperparámetro se mantendrán constantes:

- `n_estimators = 100`.
- `min_samples_leaf = 5`.
- `seleccion__percentile = 80`.

Estos valores conforman una configuración diagnóstica fija y no representan una nueva optimización definitiva.

#### Validación

Se utilizarán los cinco folds agrupados por estudiante de la primera repetición correspondiente a la semana 8.

En cada ajuste se volverán a entrenar el preprocesamiento, los filtros, el selector de características y el clasificador.

El conjunto de prueba independiente permanecerá reservado.

#### Limitaciones

Las curvas representan un diagnóstico exploratorio sobre el conjunto de desarrollo.

No deben utilizarse por sí solas para afirmar que una configuración de profundidad es estadísticamente superior ni para reemplazar la optimización de hiperparámetros anidada.

Cualquier cambio definitivo de configuración deberá evaluarse con procedimientos de selección internos, evitando optimizar directamente sobre las evaluaciones externas ya observadas.


In [ ]:

# ============================================================
# PROYECTO FINAL DE MACHINE LEARNING
# ACTIVIDAD 3.15 - CURVAS DE APRENDIZAJE Y VALIDACIÓN
# RANDOM FOREST B - SEMANA 8
# ============================================================

# Asistencia IA: ChatGPT, octubre de 2026.
# Prompt:
# "Construir curvas de aprendizaje y validación
# de Random Forest B sobre OULAD, utilizando
# particiones agrupadas por estudiante, selección
# y preprocesamiento dentro del pipeline, cinco
# tamaños de entrenamiento y cinco profundidades.
# No utilizar el conjunto de prueba independiente."

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.metrics import f1_score
from IPython.display import display

# ------------------------------------------------------------
# 1. CONFIGURACIÓN
# ------------------------------------------------------------

SEMANA_315 = 8
EXPERIMENTO_315 = "B"
ALGORITMO_315 = "RandomForest"

FRACCIONES_315 = [0.20, 0.40, 0.60, 0.80, 1.00]
PROFUNDIDADES_315 = [4, 8, 12, 16, None]

SEMILLA_315 = 2030

datos_315 = experimentos_oulad[SEMANA_315]

X_315 = (
    datos_315["X_B_train"]
    .reset_index(drop=True)
)

y_315 = (
    datos_315["y_train"]
    .reset_index(drop=True)
)

grupos_315 = (
    datos_315["grupos_train"]
    .reset_index(drop=True)
)

# Utilizar los 5 folds de la primera repetición.
folds_315 = cv_splits_oulad[SEMANA_315][:5]

assert len(folds_315) == 5

assert len(X_315) == len(y_315) == len(grupos_315)

print("=== 3.15.1 CONFIGURACIÓN ===")
print("Semana:", SEMANA_315)
print("Experimento:", EXPERIMENTO_315)
print("Algoritmo:", ALGORITMO_315)
print("Matrículas:", len(X_315))
print("Folds:", len(folds_315))
print("Fracciones:", FRACCIONES_315)
print("Profundidades:", PROFUNDIDADES_315)

# ------------------------------------------------------------
# 2. PIPELINE DIAGNÓSTICO
# ------------------------------------------------------------

def construir_rf_diagnostico(profundidad=None):

    modelo = clone(
        pipelines_oulad[
            (
                SEMANA_315,
                EXPERIMENTO_315,
                ALGORITMO_315
            )
        ]
    )

    modelo.set_params(
        modelo__n_estimators=100,
        modelo__max_depth=profundidad,
        modelo__min_samples_leaf=5,
        modelo__random_state=42,
        modelo__n_jobs=-1,
        seleccion__percentile=80
    )

    return modelo


def evaluar_f1(modelo, X, y):

    pred = modelo.predict(X)

    return f1_score(
        y,
        pred,
        average="macro",
        zero_division=0
    )


# ------------------------------------------------------------
# 3. CURVA DE APRENDIZAJE
# ------------------------------------------------------------

print("\n=== 3.15.2 CURVA DE APRENDIZAJE ===")

resultados_aprendizaje_315 = []
inicio_315 = time.perf_counter()

for numero_fold, (idx_tr, idx_val) in enumerate(
    folds_315,
    start=1
):

    grupos_train = grupos_315.iloc[idx_tr].to_numpy()
    grupos_val = grupos_315.iloc[idx_val].to_numpy()

    assert set(grupos_train).isdisjoint(
        set(grupos_val)
    )

    # Seleccionar grupos completos de estudiantes.
    # La permutación permanece fija para cada fold;
    # los conjuntos pequeños están contenidos
    # dentro de los grandes.
    grupos_unicos = np.unique(grupos_train)

    rng = np.random.default_rng(
        SEMILLA_315 + numero_fold
    )

    grupos_permutados = rng.permutation(
        grupos_unicos
    )

    for fraccion in FRACCIONES_315:

        n_grupos = max(
            1,
            int(
                np.ceil(
                    len(grupos_permutados) * fraccion
                )
            )
        )

        grupos_seleccionados = (
            grupos_permutados[:n_grupos]
        )

        mascara = np.isin(
            grupos_train,
            grupos_seleccionados
        )

        idx_subtrain = idx_tr[mascara]

        X_sub = X_315.iloc[idx_subtrain]
        y_sub = y_315.iloc[idx_subtrain]

        X_val = X_315.iloc[idx_val]
        y_val = y_315.iloc[idx_val]

        assert set(y_sub.unique()) == {0, 1}

        assert set(
            grupos_315.iloc[idx_subtrain]
        ).isdisjoint(
            set(grupos_val)
        )

        # Configuración fija sin límite de profundidad.
        modelo = construir_rf_diagnostico(
            profundidad=None
        )

        modelo.fit(X_sub, y_sub)

        f1_train = evaluar_f1(
            modelo, X_sub, y_sub
        )

        f1_val = evaluar_f1(
            modelo, X_val, y_val
        )

        resultados_aprendizaje_315.append({
            "Fold": numero_fold,
            "Fraccion_grupos": fraccion,
            "N_estudiantes_train": n_grupos,
            "N_matriculas_train": len(idx_subtrain),
            "F1_train": f1_train,
            "F1_validacion": f1_val,
            "Brecha": f1_train - f1_val
        })

        print(
            f"Fold {numero_fold} | "
            f"{fraccion:.0%} grupos | "
            f"F1 train={f1_train:.4f} | "
            f"F1 val={f1_val:.4f}",
            flush=True
        )

tabla_aprendizaje_315 = pd.DataFrame(
    resultados_aprendizaje_315
)

resumen_aprendizaje_315 = (
    tabla_aprendizaje_315.groupby(
        "Fraccion_grupos"
    )
    .agg(
        Estudiantes_train_medios=(
            "N_estudiantes_train", "mean"
        ),
        Matriculas_train_medias=(
            "N_matriculas_train", "mean"
        ),
        F1_train_medio=("F1_train", "mean"),
        F1_train_std=("F1_train", "std"),
        F1_val_medio=("F1_validacion", "mean"),
        F1_val_std=("F1_validacion", "std"),
        Brecha_media=("Brecha", "mean")
    )
    .reset_index()
)

display(resumen_aprendizaje_315.round(4))

# ------------------------------------------------------------
# 4. GRAFICAR CURVA DE APRENDIZAJE
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(10, 5))

x = (
    100 * resumen_aprendizaje_315[
        "Fraccion_grupos"
    ]
)

ax.errorbar(
    x,
    resumen_aprendizaje_315["F1_train_medio"],
    yerr=resumen_aprendizaje_315["F1_train_std"],
    marker="o",
    linewidth=2,
    capsize=3,
    label="Entrenamiento"
)

ax.errorbar(
    x,
    resumen_aprendizaje_315["F1_val_medio"],
    yerr=resumen_aprendizaje_315["F1_val_std"],
    marker="s",
    linewidth=2,
    capsize=3,
    label="Validación"
)

ax.set_title(
    "Curva de aprendizaje - Random Forest B, semana 8"
)

ax.set_xlabel(
    "Porcentaje de estudiantes de entrenamiento"
)

ax.set_ylabel("F1 macro")

ax.set_xticks([20, 40, 60, 80, 100])

ax.set_ylim(0, 1)

ax.grid(alpha=0.3)
ax.legend()

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 5. CURVA DE VALIDACIÓN DE MAX_DEPTH
# ------------------------------------------------------------

print("\n=== 3.15.3 CURVA DE VALIDACIÓN ===")

resultados_profundidad_315 = []

for numero_fold, (idx_tr, idx_val) in enumerate(
    folds_315,
    start=1
):

    X_tr = X_315.iloc[idx_tr]
    X_val = X_315.iloc[idx_val]

    y_tr = y_315.iloc[idx_tr]
    y_val = y_315.iloc[idx_val]

    assert set(
        grupos_315.iloc[idx_tr]
    ).isdisjoint(
        set(grupos_315.iloc[idx_val])
    )

    for profundidad in PROFUNDIDADES_315:

        modelo = construir_rf_diagnostico(
            profundidad=profundidad
        )

        modelo.fit(X_tr, y_tr)

        f1_train = evaluar_f1(
            modelo, X_tr, y_tr
        )

        f1_val = evaluar_f1(
            modelo, X_val, y_val
        )

        resultados_profundidad_315.append({
            "Fold": numero_fold,
            "Max_depth": (
                "Sin limite"
                if profundidad is None
                else str(profundidad)
            ),
            "F1_train": f1_train,
            "F1_validacion": f1_val,
            "Brecha": f1_train - f1_val
        })

        print(
            f"Fold {numero_fold} | "
            f"depth={profundidad} | "
            f"F1 train={f1_train:.4f} | "
            f"F1 val={f1_val:.4f}",
            flush=True
        )

tabla_profundidad_315 = pd.DataFrame(
    resultados_profundidad_315
)

orden_profundidad = [
    "4", "8", "12", "16", "Sin limite"
]

resumen_profundidad_315 = (
    tabla_profundidad_315.groupby("Max_depth")
    .agg(
        F1_train_medio=("F1_train", "mean"),
        F1_train_std=("F1_train", "std"),
        F1_val_medio=("F1_validacion", "mean"),
        F1_val_std=("F1_validacion", "std"),
        Brecha_media=("Brecha", "mean")
    )
    .reindex(orden_profundidad)
    .reset_index()
)

display(resumen_profundidad_315.round(4))

# ------------------------------------------------------------
# 6. GRAFICAR CURVA DE VALIDACIÓN
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(10, 5))

posiciones = np.arange(
    len(orden_profundidad)
)

ax.errorbar(
    posiciones,
    resumen_profundidad_315["F1_train_medio"],
    yerr=resumen_profundidad_315["F1_train_std"],
    marker="o",
    linewidth=2,
    capsize=3,
    label="Entrenamiento"
)

ax.errorbar(
    posiciones,
    resumen_profundidad_315["F1_val_medio"],
    yerr=resumen_profundidad_315["F1_val_std"],
    marker="s",
    linewidth=2,
    capsize=3,
    label="Validación"
)

ax.set_title(
    "Curva de validación - Random Forest B, semana 8"
)

ax.set_xlabel("Profundidad máxima de los árboles")
ax.set_ylabel("F1 macro")

ax.set_xticks(posiciones)
ax.set_xticklabels(orden_profundidad)

ax.set_ylim(0, 1)

ax.grid(alpha=0.3)
ax.legend()

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 7. RESUMEN DE REGULARIZACIÓN
# ------------------------------------------------------------

print("\n=== 3.15.4 RESUMEN DE REGULARIZACIÓN ===")

# El mayor rendimiento aquí es exclusivamente
# exploratorio, NO una selección definitiva.

fila_mayor_validacion = (
    resumen_profundidad_315.loc[
        resumen_profundidad_315[
            "F1_val_medio"
        ].idxmax()
    ]
)

print(
    "Profundidad con mayor F1 de validación "
    "en este diagnóstico:",
    fila_mayor_validacion["Max_depth"]
)

print(
    "F1 de validación observado:",
    round(
        fila_mayor_validacion["F1_val_medio"],
        4
    )
)

print(
    "Brecha observada:",
    round(
        fila_mayor_validacion["Brecha_media"],
        4
    )
)

print(
    "\nEste resultado es exploratorio y no "
    "modifica los hiperparámetros finales."
)

# ------------------------------------------------------------
# 8. CONTROLES FINALES
# ------------------------------------------------------------

print("\n=== 3.15.5 CONTROLES FINALES ===")

assert len(tabla_aprendizaje_315) == (
    5 * len(FRACCIONES_315)
)

assert len(tabla_profundidad_315) == (
    5 * len(PROFUNDIDADES_315)
)

assert tabla_aprendizaje_315[
    ["F1_train", "F1_validacion"]
].notna().all().all()

assert tabla_profundidad_315[
    ["F1_train", "F1_validacion"]
].notna().all().all()

print(
    "Ajustes curva de aprendizaje:",
    len(tabla_aprendizaje_315)
)

print(
    "Ajustes curva de validación:",
    len(tabla_profundidad_315)
)

print(
    "Tiempo total:",
    round(
        time.perf_counter() - inicio_315,
        1
    ),
    "segundos"
)

print(
    "Todos los subconjuntos respetaron "
    "las agrupaciones por estudiante."
)

print("Conjunto de prueba independiente: NO utilizado.")
print("Resultados previos: sin modificaciones.")

print("\n=== ACTIVIDAD 3.15 FINALIZADA ===")



### 3.15.6. Resultados e interpretación

Se construyeron curvas de aprendizaje y de validación para Random Forest B correspondiente a la semana 8 de OULAD.

El análisis se realizó mediante cinco folds agrupados por estudiante, utilizando exclusivamente el conjunto de desarrollo y conservando el preprocesamiento y la selección de características dentro del pipeline.

#### Curva de aprendizaje

Se evaluaron cinco proporciones de estudiantes de entrenamiento: 20 %, 40 %, 60 %, 80 % y 100 %.

El F1 macro medio de validación aumentó de 0,6857 con el 20 % de los estudiantes a 0,7016 con el 100 %.

Durante el mismo período, la brecha entre entrenamiento y validación disminuyó de 0,2027 a 0,1834.

El rendimiento de validación presentó una estabilización aproximada entre el 80 % y el 100 % de los estudiantes disponibles.

Estos resultados indican que incrementar el tamaño de entrenamiento mejora parcialmente la generalización, pero no elimina el sobreajuste observado bajo la configuración evaluada.

La estabilización no demuestra que mayores cantidades de datos sean incapaces de generar mejoras futuras.

#### Curva de validación

Se evaluaron profundidades máximas de 4, 8, 12, 16 y sin límite explícito, manteniendo constantes 100 árboles, cinco muestras mínimas por hoja y selección del 80 % de las características.

| Profundidad | F1 entrenamiento | F1 validación | Brecha |
|---|---:|---:|---:|
| 4 | 0,6631 | 0,6584 | 0,0047 |
| 8 | 0,7297 | 0,6905 | 0,0392 |
| 12 | 0,8103 | 0,6975 | 0,1128 |
| 16 | 0,8632 | 0,7010 | 0,1622 |
| Sin límite | 0,8850 | 0,7016 | 0,1834 |

La profundidad máxima de 4 produjo una brecha reducida, pero también el menor F1 macro de validación.

A medida que aumentó la profundidad, el rendimiento de entrenamiento creció de forma considerable.

La validación mejoró inicialmente y después presentó una estabilización alrededor de 0,70.

La configuración sin límite obtuvo el mayor F1 macro medio de validación (0,7016), aunque su diferencia respecto a la profundidad 16 fue de apenas 0,0006.

Este resultado no permite afirmar que la configuración sin límite sea estadísticamente superior ni que la profundidad 16 sea equivalente.

#### Relación entre sobreajuste y rendimiento

Las curvas evidencian que reducir la diferencia entre entrenamiento y validación no garantiza obtener mejores predicciones.

Una restricción excesiva de la profundidad puede reducir el sobreajuste aparente, pero también limitar la capacidad del modelo para representar patrones relevantes.

Las configuraciones más profundas obtuvieron mejores rendimientos de validación, aunque conservaron brechas considerables.

### 3.15.7. Limitaciones

Las curvas corresponden a una configuración diagnóstica específica de Random Forest B y solamente a la primera repetición de cinco folds de la semana 8.

Los resultados no sustituyen a la optimización anidada realizada anteriormente.

La curva de aprendizaje se construyó mediante submuestras de estudiantes completos, manteniendo la separación por identificador.

Los resultados de validación utilizados para estudiar la profundidad tienen carácter exploratorio y no deben emplearse directamente como prueba confirmatoria de superioridad entre configuraciones.

El conjunto de prueba independiente no fue utilizado.

### 3.15.8. Conclusión

Las curvas de aprendizaje y validación muestran que el sobreajuste de Random Forest B está asociado con una diferencia considerable entre el rendimiento alcanzado durante el entrenamiento y la capacidad de generalización.

El aumento de estudiantes mejora parcialmente el F1 macro de validación, pero no elimina la brecha.

La reducción de la profundidad disminuye el sobreajuste aparente, aunque una restricción excesiva perjudica el rendimiento predictivo.

**Conclusión principal:** la selección de la complejidad de Random Forest debe priorizar el rendimiento de validación y no únicamente la reducción de la brecha entrenamiento-validación. Los resultados justifican mantener la optimización de hiperparámetros dentro del procedimiento de validación cruzada y conservar el conjunto de prueba independiente para una evaluación posterior.



### 3.16. Comparación estadística de los experimentos y algoritmos

#### Objetivo

Evaluar estadísticamente las diferencias de rendimiento obtenidas durante la validación cruzada anidada y repetida, prestando especial atención al aporte de las características longitudinales.

#### Hipótesis principal

Para cada algoritmo y semana se contrastará:

- H0: la diferencia media de F1 macro entre los experimentos B y A es igual a cero.
- H1: la diferencia media de F1 macro entre los experimentos B y A es distinta de cero.

La hipótesis científica anticipa una mejora del experimento longitudinal, pero se utilizarán contrastes bilaterales.

#### Comparación complementaria

Se contrastarán las diferencias medias de F1 macro entre Regresión Logística y Random Forest dentro de cada experimento y ventana temporal.

#### Procedimiento estadístico

Las comparaciones utilizarán diferencias pareadas sobre los mismos folds externos y las mismas matrículas dentro de cada semana.

Debido a la dependencia entre folds de validación cruzada repetida, se empleará una prueba t corregida que incorpora la proporción entre tamaños de validación y entrenamiento.

Se calcularán intervalos aproximados de confianza del 95 % y valores p bilaterales.

#### Comparaciones múltiples

Se aplicará la corrección de Holm con un nivel de significancia de 0,05.

Se definirán dos familias de contrastes:

1. Ocho comparaciones principales entre B y A: dos algoritmos por cuatro semanas.
2. Ocho comparaciones complementarias entre Regresión Logística y Random Forest: dos experimentos por cuatro semanas.

#### Métricas secundarias

Las diferencias de recall adverso, ROC-AUC y accuracy se mantendrán como resultados descriptivos complementarios.

La métrica principal para los contrastes estadísticos será F1 macro.

#### Limitaciones

La corrección utilizada constituye una aproximación para resultados correlacionados de validación cruzada repetida.

La distribución de las diferencias, la reutilización de estudiantes y la selección previa de decisiones metodológicas pueden afectar la calibración de los valores p.

La prueba no elimina las limitaciones de generalización a otras instituciones o períodos académicos.

Los resultados deberán interpretarse junto con la magnitud de las diferencias, los intervalos de incertidumbre y la futura evaluación independiente.

No se entrenarán nuevos modelos ni se utilizará el conjunto de prueba reservado.


In [ ]:

# ============================================================
# PROYECTO FINAL DE MACHINE LEARNING
# ACTIVIDAD 3.16 - CONTRASTES ESTADÍSTICOS CORREGIDOS
# ============================================================

# Asistencia IA: ChatGPT, octubre de 2026.
# Prompt:
# "Comparar estadísticamente los experimentos A y B,
# y Regresión Logística frente a Random Forest,
# usando los diez folds pareados de validación cruzada
# repetida y anidada de OULAD.
# Aplicar una prueba t corregida por dependencia,
# intervalos de confianza y corrección de Holm.
# No entrenar nuevos modelos ni usar datos de prueba."

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import t as distribucion_t
from IPython.display import display

SEMANAS_316 = [2, 4, 6, 8]

ALGORITMOS_316 = [
    "LogisticRegression",
    "RandomForest"
]

ALFA_316 = 0.05

# ------------------------------------------------------------
# 1. RECUPERAR RESULTADOS EXTERNOS
# ------------------------------------------------------------

print("=== 3.16.1 PREPARACIÓN ===")

tablas_316 = []

# Semana 2
lr_s2 = tabla_lr_cv.copy()
lr_s2["Semana"] = 2
lr_s2["Algoritmo"] = "LogisticRegression"

rf_s2 = tabla_rf_cv.copy()
rf_s2["Semana"] = 2
rf_s2["Algoritmo"] = "RandomForest"

tablas_316.extend([lr_s2, rf_s2])

# Semanas 4, 6 y 8
for semana in [4, 6, 8]:

    for algoritmo in ALGORITMOS_316:

        clave = (semana, algoritmo)

        assert clave in resultados_cv_ventanas

        tablas_316.append(
            resultados_cv_ventanas[
                clave
            ]["resultados"].copy()
        )

datos_estadisticos_316 = pd.concat(
    tablas_316,
    ignore_index=True
)

assert len(datos_estadisticos_316) == 160

identificadores = [
    "Semana",
    "Algoritmo",
    "Experimento",
    "Repeticion",
    "Fold"
]

assert not datos_estadisticos_316.duplicated(
    subset=identificadores
).any()

assert (
    datos_estadisticos_316.groupby(
        ["Semana", "Algoritmo", "Experimento"]
    ).size().eq(10).all()
)

print("Evaluaciones externas recuperadas:", 160)
print("Modelos reentrenados: 0")
print("Métrica principal: F1 macro")

# ------------------------------------------------------------
# 2. PROPORCIÓN VALIDACIÓN / ENTRENAMIENTO
# ------------------------------------------------------------

razones_316 = {}

for semana in SEMANAS_316:

    splits = cv_splits_oulad[semana]

    assert len(splits) == 10

    razones = [
        len(idx_val) / len(idx_train)
        for idx_train, idx_val in splits
    ]

    razones_316[semana] = float(
        np.mean(razones)
    )

print("\nProporción media validación/entrenamiento:")

display(
    pd.DataFrame([
        {
            "Semana": semana,
            "Ratio_val_train": razones_316[semana]
        }
        for semana in SEMANAS_316
    ]).round(4)
)

# ------------------------------------------------------------
# 3. PRUEBA T CORREGIDA
# ------------------------------------------------------------

def prueba_t_corregida(diferencias, ratio, alfa=0.05):

    d = np.asarray(diferencias, dtype=float)

    assert len(d) == 10
    assert np.isfinite(d).all()

    n = len(d)
    grados_libertad = n - 1

    media = float(d.mean())
    desviacion = float(d.std(ddof=1))

    # Corrección aproximada para validación cruzada
    # repetida: (1/n + n_val/n_train) * s_d^2.

    factor = (1.0 / n) + ratio

    error_estandar = (
        np.sqrt(factor) * desviacion
    )

    if error_estandar == 0:

        estadistico_t = (
            np.inf if media > 0
            else -np.inf if media < 0
            else 0.0
        )

        p_valor = (
            0.0 if media != 0 else 1.0
        )

        limite_inferior = media
        limite_superior = media

    else:

        estadistico_t = (
            media / error_estandar
        )

        p_valor = float(
            2 * distribucion_t.sf(
                abs(estadistico_t),
                df=grados_libertad
            )
        )

        valor_critico = distribucion_t.ppf(
            1 - alfa / 2,
            df=grados_libertad
        )

        margen = (
            valor_critico * error_estandar
        )

        limite_inferior = media - margen
        limite_superior = media + margen

    return {
        "Delta_medio": media,
        "Delta_std": desviacion,
        "Ratio_val_train": ratio,
        "Factor_correccion": factor,
        "Error_estandar_corregido": error_estandar,
        "t_corregida": estadistico_t,
        "Grados_libertad": grados_libertad,
        "p_sin_ajustar": p_valor,
        "IC95_inferior": limite_inferior,
        "IC95_superior": limite_superior,
        "Comparaciones_positivas": int(
            (d > 0).sum()
        ),
        "Comparaciones_negativas": int(
            (d < 0).sum()
        )
    }


# ------------------------------------------------------------
# 4. CORRECCIÓN DE HOLM
# ------------------------------------------------------------

def ajustar_holm(p_valores):

    p = np.asarray(p_valores, dtype=float)

    assert np.isfinite(p).all()

    m = len(p)
    orden = np.argsort(p)

    p_ajustados = np.empty(m)

    maximo_previo = 0.0

    for rango, indice in enumerate(orden):

        valor = (
            (m - rango) * p[indice]
        )

        maximo_previo = max(
            maximo_previo,
            valor
        )

        p_ajustados[indice] = min(
            maximo_previo,
            1.0
        )

    return p_ajustados


# ------------------------------------------------------------
# 5. FUNCIÓN PARA FORMAR PARES
# ------------------------------------------------------------

def diferencias_pareadas(
    tabla,
    columna_grupo,
    valor_primero,
    valor_segundo
):

    columnas = [
        "Repeticion",
        "Fold",
        "F1_macro"
    ]

    primero = (
        tabla.loc[
            tabla[columna_grupo] == valor_primero,
            columnas
        ]
        .rename(
            columns={"F1_macro": "F1_primero"}
        )
    )

    segundo = (
        tabla.loc[
            tabla[columna_grupo] == valor_segundo,
            columnas
        ]
        .rename(
            columns={"F1_macro": "F1_segundo"}
        )
    )

    pares = primero.merge(
        segundo,
        on=["Repeticion", "Fold"],
        how="inner",
        validate="one_to_one"
    )

    assert len(pares) == 10

    return (
        pares["F1_primero"]
        - pares["F1_segundo"]
    ).to_numpy()


# ------------------------------------------------------------
# 6. CONTRASTES PRINCIPALES: B - A
# ------------------------------------------------------------

print("\n=== 3.16.2 CONTRASTES B - A ===")

contrastes_principales = []

for semana in SEMANAS_316:

    for algoritmo in ALGORITMOS_316:

        sub = datos_estadisticos_316.loc[
            (datos_estadisticos_316["Semana"] == semana)
            &
            (
                datos_estadisticos_316[
                    "Algoritmo"
                ] == algoritmo
            )
        ]

        diferencias = diferencias_pareadas(
            sub,
            columna_grupo="Experimento",
            valor_primero="B",
            valor_segundo="A"
        )

        resultado = prueba_t_corregida(
            diferencias,
            ratio=razones_316[semana],
            alfa=ALFA_316
        )

        contrastes_principales.append({
            "Semana": semana,
            "Algoritmo": algoritmo,
            "Comparacion": "B - A",
            **resultado
        })

tabla_contrastes_BA_316 = pd.DataFrame(
    contrastes_principales
)

tabla_contrastes_BA_316["p_Holm"] = ajustar_holm(
    tabla_contrastes_BA_316[
        "p_sin_ajustar"
    ].to_numpy()
)

tabla_contrastes_BA_316["Rechazar_H0_Holm"] = (
    tabla_contrastes_BA_316["p_Holm"]
    < ALFA_316
)

display(
    tabla_contrastes_BA_316[
        [
            "Semana",
            "Algoritmo",
            "Delta_medio",
            "Delta_std",
            "IC95_inferior",
            "IC95_superior",
            "t_corregida",
            "p_sin_ajustar",
            "p_Holm",
            "Rechazar_H0_Holm"
        ]
    ].round(6)
)

# ------------------------------------------------------------
# 7. CONTRASTES COMPLEMENTARIOS: LR - RF
# ------------------------------------------------------------

print("\n=== 3.16.3 CONTRASTES LR - RF ===")

contrastes_algoritmos = []

for semana in SEMANAS_316:

    for experimento in ["A", "B"]:

        sub = datos_estadisticos_316.loc[
            (datos_estadisticos_316["Semana"] == semana)
            &
            (
                datos_estadisticos_316[
                    "Experimento"
                ] == experimento
            )
        ]

        diferencias = diferencias_pareadas(
            sub,
            columna_grupo="Algoritmo",
            valor_primero="LogisticRegression",
            valor_segundo="RandomForest"
        )

        resultado = prueba_t_corregida(
            diferencias,
            ratio=razones_316[semana],
            alfa=ALFA_316
        )

        contrastes_algoritmos.append({
            "Semana": semana,
            "Experimento": experimento,
            "Comparacion": "LR - RF",
            **resultado
        })

tabla_contrastes_LR_RF_316 = pd.DataFrame(
    contrastes_algoritmos
)

tabla_contrastes_LR_RF_316["p_Holm"] = ajustar_holm(
    tabla_contrastes_LR_RF_316[
        "p_sin_ajustar"
    ].to_numpy()
)

tabla_contrastes_LR_RF_316["Rechazar_H0_Holm"] = (
    tabla_contrastes_LR_RF_316["p_Holm"]
    < ALFA_316
)

display(
    tabla_contrastes_LR_RF_316[
        [
            "Semana",
            "Experimento",
            "Delta_medio",
            "Delta_std",
            "IC95_inferior",
            "IC95_superior",
            "t_corregida",
            "p_sin_ajustar",
            "p_Holm",
            "Rechazar_H0_Holm"
        ]
    ].round(6)
)

# ------------------------------------------------------------
# 8. MÉTRICAS SECUNDARIAS DESCRIPTIVAS
# ------------------------------------------------------------

print("\n=== 3.16.4 MÉTRICAS COMPLEMENTARIAS ===")

filas_secundarias = []

for semana in SEMANAS_316:

    for algoritmo in ALGORITMOS_316:

        sub = datos_estadisticos_316.loc[
            (datos_estadisticos_316["Semana"] == semana)
            &
            (
                datos_estadisticos_316[
                    "Algoritmo"
                ] == algoritmo
            )
        ]

        for metrica in [
            "Recall_adverso",
            "ROC_AUC",
            "Accuracy"
        ]:

            primero = sub.loc[
                sub["Experimento"] == "B",
                ["Repeticion", "Fold", metrica]
            ].rename(
                columns={metrica: "Valor_B"}
            )

            segundo = sub.loc[
                sub["Experimento"] == "A",
                ["Repeticion", "Fold", metrica]
            ].rename(
                columns={metrica: "Valor_A"}
            )

            pares = primero.merge(
                segundo,
                on=["Repeticion", "Fold"],
                validate="one_to_one"
            )

            delta = (
                pares["Valor_B"] - pares["Valor_A"]
            )

            filas_secundarias.append({
                "Semana": semana,
                "Algoritmo": algoritmo,
                "Metrica": metrica,
                "Delta_medio_B_A": delta.mean(),
                "Delta_std": delta.std(ddof=1),
                "Folds_B_supera_A": int(
                    (delta > 0).sum()
                )
            })

tabla_secundarias_316 = pd.DataFrame(
    filas_secundarias
)

display(tabla_secundarias_316.round(4))

# ------------------------------------------------------------
# 9. VISUALIZACIÓN DE EFECTOS E INTERVALOS
# ------------------------------------------------------------

print("\n=== 3.16.5 GRÁFICO DE CONTRASTES ===")

fig, ax = plt.subplots(figsize=(10, 5))

for algoritmo in ALGORITMOS_316:

    sub = tabla_contrastes_BA_316.loc[
        tabla_contrastes_BA_316[
            "Algoritmo"
        ] == algoritmo
    ].sort_values("Semana")

    margen_inferior = (
        sub["Delta_medio"]
        - sub["IC95_inferior"]
    )

    margen_superior = (
        sub["IC95_superior"]
        - sub["Delta_medio"]
    )

    etiqueta = (
        "Regresión Logística"
        if algoritmo == "LogisticRegression"
        else "Random Forest"
    )

    ax.errorbar(
        sub["Semana"],
        sub["Delta_medio"],
        yerr=np.vstack([
            margen_inferior.to_numpy(),
            margen_superior.to_numpy()
        ]),
        marker="o",
        linewidth=2,
        capsize=4,
        label=etiqueta
    )

ax.axhline(
    0,
    linestyle="--",
    linewidth=1
)

ax.set_title(
    "Aporte longitudinal B - A e intervalos corregidos"
)

ax.set_xlabel("Semana de predicción")
ax.set_ylabel("Diferencia de F1 macro")
ax.set_xticks(SEMANAS_316)
ax.grid(alpha=0.3)
ax.legend()

plt.tight_layout()
plt.show()

print(
    "Nota: intervalos aproximados al 95 %, "
    "no ajustados por multiplicidad."
)
print(
    "Los valores p sí se ajustaron mediante Holm."
)

# ------------------------------------------------------------
# 10. RESUMEN Y CONTROLES
# ------------------------------------------------------------

print("\n=== 3.16.6 CONTROLES FINALES ===")

assert len(tabla_contrastes_BA_316) == 8
assert len(tabla_contrastes_LR_RF_316) == 8
assert len(tabla_secundarias_316) == 24

assert tabla_contrastes_BA_316[
    ["Delta_medio", "p_Holm"]
].notna().all().all()

assert tabla_contrastes_LR_RF_316[
    ["Delta_medio", "p_Holm"]
].notna().all().all()

print(
    "Contrastes principales B - A:",
    len(tabla_contrastes_BA_316)
)

print(
    "Contrastes complementarios LR - RF:",
    len(tabla_contrastes_LR_RF_316)
)

print(
    "Contrastes B - A con p_Holm < 0.05:",
    int(
        tabla_contrastes_BA_316[
            "Rechazar_H0_Holm"
        ].sum()
    )
)

print(
    "Contrastes LR - RF con p_Holm < 0.05:",
    int(
        tabla_contrastes_LR_RF_316[
            "Rechazar_H0_Holm"
        ].sum()
    )
)

print("Modelos nuevos entrenados: 0")
print("Conjunto de prueba independiente: NO utilizado")
print(
    "Prueba estadística: corregida, aproximada "
    "y sujeta a los supuestos del procedimiento."
)

print("\n=== ACTIVIDAD 3.16 FINALIZADA ===")



### 3.16.7. Resultados e interpretación estadística

Se realizó una comparación estadística de los modelos utilizando los resultados de 160 evaluaciones externas obtenidas mediante validación cruzada anidada y repetida.

Se empleó una prueba t corregida para resultados correlacionados de validación cruzada, utilizando una corrección aproximada de la varianza y contrastes bilaterales con nivel de significancia de 0,05.

Para controlar las comparaciones múltiples, se aplicó la corrección de Holm por separado a dos familias de ocho contrastes.

#### Comparación principal: características longitudinales frente a estáticas

Los ocho contrastes de F1 macro entre los experimentos B y A resultaron estadísticamente significativos bajo el procedimiento utilizado, después de la corrección de Holm.

Todas las diferencias medias fueron positivas.

La mejora de F1 macro aumentó descriptivamente entre las semanas 2 y 8:

- Regresión Logística: de 0,0580 a 0,1321.
- Random Forest: de 0,0580 a 0,1189.

Los intervalos individuales corregidos por dependencia de folds se encontraron por encima de cero en las ocho comparaciones.

Estos resultados proporcionan evidencia estadística favorable al valor predictivo incremental de las características longitudinales, condicionada a los supuestos del método aplicado.

#### Comparación complementaria: Regresión Logística frente a Random Forest

Regresión Logística presentó una ventaja media positiva de F1 macro frente a Random Forest en los ocho escenarios evaluados.

Sin embargo, solamente cuatro contrastes fueron estadísticamente significativos después de aplicar Holm:

- Semana 4, Experimento B: p ajustado = 0,005969.
- Semana 6, Experimento A: p ajustado = 0,033021.
- Semana 6, Experimento B: p ajustado = 0,013257.
- Semana 8, Experimento B: p ajustado = 0,000575.

Los demás contrastes no alcanzaron significancia después del ajuste.

Por tanto, los resultados favorecen descriptivamente a Regresión Logística, pero no permiten afirmar superioridad estadística universal frente a Random Forest.

#### Métricas secundarias

Las diferencias de recall adverso, ROC-AUC y accuracy favorecieron al Experimento B en las diez particiones externas de todos los escenarios.

Estos resultados se consideran evidencia descriptiva complementaria, debido a que los contrastes estadísticos se concentraron en la métrica principal F1 macro.

#### Evolución temporal

Los resultados presentan una tendencia ascendente de la ventaja longitudinal B-A entre las semanas 2 y 8.

Esta tendencia también se observó en el análisis de sensibilidad con población común.

Sin embargo, no se realizó un contraste estadístico específico de tendencia temporal. Además, las cohortes de entrenamiento y el número de características disponibles varían entre ventanas.

### 3.16.8. Limitaciones estadísticas

La prueba t corregida constituye una aproximación para la dependencia entre particiones de validación cruzada repetida.

Los diez valores utilizados en cada contraste no representan diez muestras independientes de estudiantes.

Los intervalos de confianza presentados fueron corregidos aproximadamente por dependencia de folds, pero no ajustados simultáneamente por multiplicidad.

Los valores p se obtuvieron exclusivamente sobre el conjunto de desarrollo, después de varias actividades de exploración y evaluación de modelos.

Por tanto, no deben interpretarse como evidencia confirmatoria completamente independiente.

Los resultados tampoco permiten establecer una relación causal entre actividad longitudinal y resultados académicos.

Será necesario complementar el análisis mediante remuestreo agrupado por estudiante y realizar posteriormente una evaluación independiente.

### 3.16.9. Conclusión

Los contrastes realizados aportan respaldo estadístico, bajo los supuestos de la prueba corregida, a la mejora de F1 macro obtenida al incorporar características longitudinales.

La ventaja fue positiva y estadísticamente significativa en los ocho escenarios de comparación B-A.

Regresión Logística obtuvo mejores promedios que Random Forest, aunque solamente cuatro de las ocho comparaciones entre algoritmos alcanzaron significancia tras la corrección de Holm.

**Conclusión principal:** la evidencia acumulada de validación cruzada, análisis de sensibilidad y contrastes corregidos respalda el valor predictivo adicional de las trayectorias longitudinales del comportamiento estudiantil en OULAD. No obstante, la interpretación definitiva requiere cuantificar la incertidumbre agrupada por estudiante y contrastar los modelos sobre el conjunto de prueba independiente.



### 3.17. Bootstrap pareado agrupado por estudiante

#### Objetivo

Estimar la incertidumbre de las diferencias de rendimiento entre los experimentos longitudinal y estático mediante remuestreo agrupado por identificador de estudiante.

#### Justificación

La validación cruzada repetida genera resultados correlacionados debido al solapamiento entre conjuntos de entrenamiento.

Además, algunos estudiantes de OULAD presentan múltiples matrículas académicas, por lo que las observaciones individuales no pueden considerarse completamente independientes.

Como complemento de los contrastes realizados en la Actividad 3.16, se implementará un bootstrap por conglomerados utilizando `id_student` como unidad de remuestreo.

#### Procedimiento

Para cada ventana temporal:

1. Se recuperarán las predicciones fuera de muestra de Regresión Logística y Random Forest.
2. Se comprobará el emparejamiento entre los experimentos A y B.
3. Se agruparán las matrículas de cada estudiante.
4. Se generarán 1.000 muestras bootstrap de estudiantes con reemplazo.
5. Para cada muestra se reconstruirán las matrices de confusión de ambas repeticiones.
6. Se calculará el F1 macro y el recall adverso para A y B.
7. Se obtendrán las diferencias B-A.
8. Se calcularán intervalos percentiles del 95 %.

Los mismos estudiantes remuestreados se utilizarán para los dos experimentos y algoritmos de una misma ventana.

#### Interpretación

Se estudiará la magnitud de las diferencias y la proporción de muestras bootstrap con una mejora positiva.

Los intervalos tendrán carácter complementario y describirán la variabilidad de las métricas condicionada a los modelos y predicciones fuera de muestra ya obtenidos.

No incluirán la incertidumbre completa asociada a volver a entrenar los modelos y seleccionar sus hiperparámetros.

#### Comparaciones múltiples

Los intervalos bootstrap serán individuales y no estarán ajustados por comparaciones múltiples.

No se utilizarán como sustituto de los contrastes corregidos de la actividad anterior.

#### Prevención de fuga de información

No se volverán a entrenar clasificadores ni se utilizará el conjunto de prueba independiente.

El análisis utilizará únicamente predicciones externas previamente generadas sobre el conjunto de desarrollo.


In [ ]:

# ============================================================
# PROYECTO FINAL DE MACHINE LEARNING
# ACTIVIDAD 3.17 - CLUSTER BOOTSTRAP PAREADO
# ============================================================

# Asistencia IA: ChatGPT, octubre de 2026.
# Prompt:
# "Realizar 1.000 remuestreos bootstrap agrupados por
# id_student sobre las predicciones externas de OULAD.
# Mantener emparejados A y B, conservar las dos
# repeticiones y calcular intervalos percentiles de
# diferencias de F1 macro y recall adverso.
# No reentrenar modelos ni utilizar prueba independiente."

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

# ------------------------------------------------------------
# 1. CONFIGURACIÓN
# ------------------------------------------------------------

SEMANAS_317 = [2, 4, 6, 8]

ALGORITMOS_317 = [
    "LogisticRegression",
    "RandomForest"
]

EXPERIMENTOS_317 = ["A", "B"]

CLAVES_317 = [
    "code_module",
    "code_presentation",
    "id_student"
]

CONTEOS_317 = ["TN", "FP", "FN", "TP"]

N_BOOTSTRAP_317 = 1000
SEMILLA_317 = 2031

rng_317 = np.random.default_rng(
    SEMILLA_317
)

print("=== 3.17.1 CONFIGURACIÓN ===")

print("Semanas:", SEMANAS_317)
print("Algoritmos:", ALGORITMOS_317)
print("Remuestreos:", N_BOOTSTRAP_317)
print("Unidad de remuestreo: id_student")
print("Métricas: F1 macro y recall adverso")

# ------------------------------------------------------------
# 2. MÉTRICAS A PARTIR DE MATRICES DE CONFUSIÓN
# ------------------------------------------------------------

# Orden de las columnas:
# TN, FP, FN, TP

def f1_macro_desde_conteos(c):

    tn = c[..., 0]
    fp = c[..., 1]
    fn = c[..., 2]
    tp = c[..., 3]

    denominador_positivo = (
        2 * tp + fp + fn
    )

    denominador_negativo = (
        2 * tn + fp + fn
    )

    f1_positivo = np.divide(
        2 * tp,
        denominador_positivo,
        out=np.zeros_like(tp, dtype=float),
        where=denominador_positivo > 0
    )

    f1_negativo = np.divide(
        2 * tn,
        denominador_negativo,
        out=np.zeros_like(tn, dtype=float),
        where=denominador_negativo > 0
    )

    return (
        f1_positivo + f1_negativo
    ) / 2


def recall_desde_conteos(c):

    tp = c[..., 3]
    fn = c[..., 2]

    denominador = tp + fn

    return np.divide(
        tp,
        denominador,
        out=np.zeros_like(tp, dtype=float),
        where=denominador > 0
    )


# ------------------------------------------------------------
# 3. RECUPERAR PREDICCIONES DE CADA SEMANA
# ------------------------------------------------------------

def recuperar_predicciones_317(semana, algoritmo):

    if semana == 2:

        resultado = (
            predicciones_conjuntas_s2.loc[
                predicciones_conjuntas_s2[
                    "Algoritmo"
                ] == algoritmo
            ]
            .copy()
        )

    else:

        resultado = (
            resultados_cv_ventanas[
                (semana, algoritmo)
            ]["predicciones"].copy()
        )

    columnas_necesarias = (
        CLAVES_317
        + [
            "Repeticion",
            "Experimento",
            "y_real",
            "y_pred"
        ]
    )

    assert all(
        col in resultado.columns
        for col in columnas_necesarias
    )

    return resultado


# ------------------------------------------------------------
# 4. MATRICES DE CONFUSIÓN POR ESTUDIANTE
# ------------------------------------------------------------

def preparar_matrices_317(semana):

    estudiantes = np.sort(
        experimentos_oulad[
            semana
        ]["grupos_train"].unique()
    )

    n_estudiantes = len(estudiantes)

    n_matriculas = len(
        experimentos_oulad[
            semana
        ]["y_train"]
    )

    # Dimensiones:
    # algoritmo, experimento, repetición,
    # estudiante, tipo de resultado (TN, FP, FN, TP)

    matrices = np.zeros(
        (
            2,
            2,
            2,
            n_estudiantes,
            4
        ),
        dtype=np.int32
    )

    referencia_etiquetas = None

    for ai, algoritmo in enumerate(
        ALGORITMOS_317
    ):

        predicciones = recuperar_predicciones_317(
            semana,
            algoritmo
        )

        for ri, repeticion in enumerate([1, 2]):

            for ei, experimento in enumerate(
                EXPERIMENTOS_317
            ):

                sub = predicciones.loc[
                    (
                        predicciones[
                            "Repeticion"
                        ] == repeticion
                    )
                    &
                    (
                        predicciones[
                            "Experimento"
                        ] == experimento
                    )
                ].copy()

                assert len(sub) == n_matriculas

                assert not sub.duplicated(
                    subset=CLAVES_317
                ).any()

                # Verificar que todos los modelos
                # evalúen las mismas etiquetas.
                etiquetas = (
                    sub.set_index(
                        CLAVES_317
                    )["y_real"].sort_index()
                )

                if referencia_etiquetas is None:

                    referencia_etiquetas = (
                        etiquetas
                    )

                else:

                    assert etiquetas.equals(
                        referencia_etiquetas
                    ), (
                        "No coinciden las matrículas "
                        "o sus etiquetas."
                    )

                # Indicadores de clasificación.
                y = sub["y_real"]
                pred = sub["y_pred"]

                cuentas = pd.DataFrame({
                    "id_student": (
                        sub["id_student"].to_numpy()
                    ),
                    "TN": (
                        (y == 0) & (pred == 0)
                    ).astype(np.int32).to_numpy(),
                    "FP": (
                        (y == 0) & (pred == 1)
                    ).astype(np.int32).to_numpy(),
                    "FN": (
                        (y == 1) & (pred == 0)
                    ).astype(np.int32).to_numpy(),
                    "TP": (
                        (y == 1) & (pred == 1)
                    ).astype(np.int32).to_numpy()
                })

                # Todas las matrículas del mismo
                # estudiante permanecen agrupadas.
                agrupado = (
                    cuentas.groupby(
                        "id_student"
                    )[CONTEOS_317]
                    .sum()
                    .reindex(
                        estudiantes,
                        fill_value=0
                    )
                )

                matrices[
                    ai, ei, ri, :, :
                ] = agrupado.to_numpy(
                    dtype=np.int32
                )

    # Todas las configuraciones deben evaluar
    # exactamente el mismo número de matrículas.
    assert np.all(
        matrices.sum(axis=(3, 4))
        == n_matriculas
    )

    return estudiantes, matrices


# ------------------------------------------------------------
# 5. FUNCIÓN DE EVALUACIÓN DE UNA MUESTRA
# ------------------------------------------------------------

def evaluar_conteos_317(conteos):

    # conteos:
    # algoritmos x experimentos x repeticiones x 4

    f1 = f1_macro_desde_conteos(
        conteos
    )

    recall = recall_desde_conteos(
        conteos
    )

    # Primero calcular las métricas de cada
    # repetición, después promediar.
    #
    # Evita interpretar las dos repeticiones
    # como matrículas independientes.

    f1_medio = f1.mean(axis=2)
    recall_medio = recall.mean(axis=2)

    delta_f1 = (
        f1_medio[:, 1]
        - f1_medio[:, 0]
    )

    delta_recall = (
        recall_medio[:, 1]
        - recall_medio[:, 0]
    )

    return (
        f1_medio,
        recall_medio,
        delta_f1,
        delta_recall
    )


# ------------------------------------------------------------
# 6. EJECUTAR BOOTSTRAP PARA LAS CUATRO SEMANAS
# ------------------------------------------------------------

print("\n=== 3.17.2 BOOTSTRAP POR ESTUDIANTE ===")

resultados_bootstrap_317 = []
distribuciones_bootstrap_317 = []

for semana in SEMANAS_317:

    estudiantes, matrices = (
        preparar_matrices_317(semana)
    )

    n = len(estudiantes)

    print(
        f"\nSemana {semana} | "
        f"Estudiantes únicos: {n}",
        flush=True
    )

    # Métricas con todas las matrículas,
    # sin remuestreo.
    conteos_originales = matrices.sum(
        axis=3,
        dtype=np.int64
    )

    (
        f1_original,
        recall_original,
        delta_f1_original,
        delta_recall_original
    ) = evaluar_conteos_317(
        conteos_originales
    )

    boot_delta_f1 = np.empty(
        (N_BOOTSTRAP_317, 2),
        dtype=float
    )

    boot_delta_recall = np.empty(
        (N_BOOTSTRAP_317, 2),
        dtype=float
    )

    for b in range(N_BOOTSTRAP_317):

        # Selección de estudiantes con reemplazo.
        indices = rng_317.integers(
            low=0,
            high=n,
            size=n
        )

        # Cuántas veces aparece cada estudiante.
        pesos = np.bincount(
            indices,
            minlength=n
        ).astype(np.int64)

        # Aplicar los mismos pesos a A, B,
        # ambos algoritmos y ambas repeticiones.
        conteos_remuestreados = np.einsum(
            "g,aergk->aerk",
            pesos,
            matrices,
            dtype=np.int64
        )

        (
            _,
            _,
            delta_f1,
            delta_recall
        ) = evaluar_conteos_317(
            conteos_remuestreados
        )

        boot_delta_f1[b, :] = delta_f1

        boot_delta_recall[b, :] = delta_recall

    # --------------------------------------------------------
    # 7. INTERVALOS PERCENTILES
    # --------------------------------------------------------

    for ai, algoritmo in enumerate(
        ALGORITMOS_317
    ):

        muestras_f1 = (
            boot_delta_f1[:, ai]
        )

        muestras_recall = (
            boot_delta_recall[:, ai]
        )

        ic_f1 = np.quantile(
            muestras_f1,
            [0.025, 0.975]
        )

        ic_recall = np.quantile(
            muestras_recall,
            [0.025, 0.975]
        )

        resultados_bootstrap_317.append({
            "Semana": semana,
            "Algoritmo": algoritmo,
            "Estudiantes": n,
            "F1_A_OOF": f1_original[ai, 0],
            "F1_B_OOF": f1_original[ai, 1],
            "Delta_F1": delta_f1_original[ai],
            "IC95_F1_inferior": ic_f1[0],
            "IC95_F1_superior": ic_f1[1],
            "Fraccion_boot_F1_positiva": (
                np.mean(muestras_f1 > 0)
            ),
            "Delta_Recall": (
                delta_recall_original[ai]
            ),
            "IC95_Recall_inferior": ic_recall[0],
            "IC95_Recall_superior": ic_recall[1]
        })

        for b in range(N_BOOTSTRAP_317):

            distribuciones_bootstrap_317.append({
                "Semana": semana,
                "Algoritmo": algoritmo,
                "Bootstrap": b + 1,
                "Delta_F1": muestras_f1[b],
                "Delta_Recall": muestras_recall[b]
            })

        print(
            f"  {algoritmo}: "
            f"Delta F1={delta_f1_original[ai]:.4f} | "
            f"IC95=[{ic_f1[0]:.4f}, "
            f"{ic_f1[1]:.4f}]",
            flush=True
        )

# ------------------------------------------------------------
# 8. PRESENTAR RESULTADOS
# ------------------------------------------------------------

tabla_bootstrap_317 = pd.DataFrame(
    resultados_bootstrap_317
)

tabla_distribuciones_317 = pd.DataFrame(
    distribuciones_bootstrap_317
)

print("\n=== 3.17.3 RESULTADOS F1 MACRO ===")

display(
    tabla_bootstrap_317[
        [
            "Semana",
            "Algoritmo",
            "Estudiantes",
            "F1_A_OOF",
            "F1_B_OOF",
            "Delta_F1",
            "IC95_F1_inferior",
            "IC95_F1_superior",
            "Fraccion_boot_F1_positiva"
        ]
    ].round(4)
)

print("\n=== 3.17.4 RESULTADOS RECALL ADVERSO ===")

display(
    tabla_bootstrap_317[
        [
            "Semana",
            "Algoritmo",
            "Delta_Recall",
            "IC95_Recall_inferior",
            "IC95_Recall_superior"
        ]
    ].round(4)
)

# ------------------------------------------------------------
# 9. COMPARAR CON LA VALIDACIÓN ANIDADA
# ------------------------------------------------------------

print("\n=== 3.17.5 COMPARACIÓN DE ESTIMADORES ===")

comparacion_bootstrap_317 = (
    tabla_bootstrap_317[
        [
            "Semana",
            "Algoritmo",
            "Delta_F1"
        ]
    ]
    .merge(
        tabla_deltas_temporales_completa[
            [
                "Semana",
                "Algoritmo",
                "Delta_F1_macro"
            ]
        ],
        on=["Semana", "Algoritmo"],
        validate="one_to_one"
    )
)

comparacion_bootstrap_317[
    "Diferencia_estimadores"
] = (
    comparacion_bootstrap_317[
        "Delta_F1"
    ]
    - comparacion_bootstrap_317[
        "Delta_F1_macro"
    ]
)

display(
    comparacion_bootstrap_317.round(5)
)

print(
    "\nLas diferencias pueden variar ligeramente: "
    "el bootstrap utiliza el F1 de las predicciones "
    "fuera de muestra de cada repetición, mientras "
    "que la validación anidada promedia el F1 "
    "calculado por fold."
)

# ------------------------------------------------------------
# 10. GRÁFICO DE INTERVALOS BOOTSTRAP
# ------------------------------------------------------------

print("\n=== 3.17.6 GRÁFICO BOOTSTRAP ===")

fig, ax = plt.subplots(figsize=(10, 5))

for algoritmo in ALGORITMOS_317:

    sub = tabla_bootstrap_317.loc[
        tabla_bootstrap_317[
            "Algoritmo"
        ] == algoritmo
    ].sort_values("Semana")

    error_inf = (
        sub["Delta_F1"]
        - sub["IC95_F1_inferior"]
    ).clip(lower=0)

    error_sup = (
        sub["IC95_F1_superior"]
        - sub["Delta_F1"]
    ).clip(lower=0)

    ax.errorbar(
        sub["Semana"],
        sub["Delta_F1"],
        yerr=np.vstack([
            error_inf.to_numpy(),
            error_sup.to_numpy()
        ]),
        marker="o",
        capsize=4,
        linewidth=2,
        label=algoritmo
    )

ax.axhline(
    0,
    linestyle="--",
    linewidth=1
)

ax.set_title(
    "Aporte longitudinal B - A\n"
    "Bootstrap agrupado por estudiante"
)

ax.set_xlabel("Semana de predicción")
ax.set_ylabel("Diferencia de F1 macro")

ax.set_xticks(SEMANAS_317)
ax.grid(alpha=0.3)
ax.legend()

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 11. CONTROLES FINALES
# ------------------------------------------------------------

print("\n=== 3.17.7 CONTROLES FINALES ===")

assert len(tabla_bootstrap_317) == 8

assert len(tabla_distribuciones_317) == (
    8 * N_BOOTSTRAP_317
)

assert tabla_bootstrap_317[
    [
        "Delta_F1",
        "IC95_F1_inferior",
        "IC95_F1_superior"
    ]
].notna().all().all()

print(
    "Comparaciones completadas:",
    len(tabla_bootstrap_317)
)

print(
    "Muestras por comparación:",
    N_BOOTSTRAP_317
)

print(
    "Remuestreo realizado por estudiante."
)

print(
    "Predicciones de A y B mantenidas emparejadas."
)

print(
    "Las dos repeticiones se evaluaron "
    "por separado y después se promediaron."
)

print(
    "Modelos reentrenados: 0"
)

print(
    "Conjunto de prueba independiente: NO utilizado"
)

print(
    "Intervalos: percentiles descriptivos, "
    "condicionados a predicciones OOF existentes."
)

print("\n=== ACTIVIDAD 3.17 FINALIZADA ===")



### 3.17.8. Resultados e interpretación

Se realizó un análisis de incertidumbre mediante bootstrap pareado agrupado por estudiante, utilizando las predicciones fuera de muestra obtenidas durante la validación cruzada anidada de los experimentos A y B.

Para cada semana se generaron 1.000 muestras bootstrap seleccionando estudiantes completos con reemplazo y conservando todas sus matrículas académicas.

Las predicciones de los experimentos A y B permanecieron emparejadas. Las métricas de las dos repeticiones se calcularon por separado y posteriormente se promediaron.

#### Resultados de F1 macro

| Semana | Algoritmo | Diferencia B-A | IC percentil 95 % |
|---|---|---:|---|
| 2 | Regresión Logística | 0,0580 | [0,0519; 0,0641] |
| 2 | Random Forest | 0,0579 | [0,0506; 0,0643] |
| 4 | Regresión Logística | 0,0815 | [0,0749; 0,0889] |
| 4 | Random Forest | 0,0760 | [0,0683; 0,0836] |
| 6 | Regresión Logística | 0,1022 | [0,0948; 0,1096] |
| 6 | Random Forest | 0,0990 | [0,0916; 0,1067] |
| 8 | Regresión Logística | 0,1322 | [0,1243; 0,1396] |
| 8 | Random Forest | 0,1190 | [0,1104; 0,1273] |

Los ocho intervalos percentiles se encontraron por encima de cero.

La proporción de muestras bootstrap con diferencias positivas fue de 1,0 en las ocho comparaciones, lo que expresa consistencia dentro de los 1.000 remuestreos realizados y no una probabilidad absoluta de que la hipótesis sea verdadera.

#### Resultados de recall adverso

Las diferencias de recall también fueron positivas en todas las ventanas.

En Regresión Logística, la ganancia aumentó de 0,1004 en semana 2 a 0,2131 en semana 8.

En Random Forest, aumentó de 0,0840 a 0,1457 durante el mismo período.

Los ocho intervalos bootstrap correspondientes al recall adverso también se encontraron completamente por encima de cero.

#### Consistencia con la validación cruzada anidada

Las diferencias puntuales de F1 macro calculadas mediante las predicciones agregadas fueron prácticamente iguales a las obtenidas al promediar los folds externos.

La mayor discrepancia absoluta entre ambos estimadores fue aproximadamente 0,00015.

Este resultado demuestra consistencia numérica entre las formas de agregar las predicciones existentes, sin constituir una validación independiente adicional.

### 3.17.9. Limitaciones

Los intervalos bootstrap se calcularon condicionados a las predicciones fuera de muestra y a los modelos ya entrenados.

El procedimiento no incorporó la incertidumbre completa de repetir el entrenamiento, la selección de variables y la optimización de hiperparámetros.

Las repeticiones de validación utilizaron las mismas matrículas y no constituyen observaciones independientes.

Los intervalos percentiles fueron individuales y no se ajustaron por comparaciones múltiples.

El bootstrap también utiliza el conjunto de desarrollo previamente explorado, por lo que no sustituye una evaluación final independiente.

### 3.17.10. Conclusión

El bootstrap agrupado por estudiante proporciona evidencia complementaria de estabilidad de la ventaja predictiva longitudinal.

Las diferencias de F1 macro y recall adverso entre los experimentos B y A fueron positivas en las cuatro semanas y los dos algoritmos, con intervalos percentiles del 95 % completamente superiores a cero.

La magnitud de la mejora longitudinal aumentó descriptivamente conforme avanzaron las ventanas de observación.

**Conclusión principal:** la información longitudinal del comportamiento estudiantil aporta una ventaja predictiva consistente bajo las diferentes estrategias de evaluación realizadas en el conjunto de desarrollo. Esta evidencia debe interpretarse considerando las limitaciones de dependencia, remuestreo y selección de modelos, y contrastarse posteriormente sobre el conjunto de prueba independiente.



### 3.18. Preparación de datos secuenciales para el Experimento C

#### Objetivo

Construir representaciones temporales semanales del comportamiento académico de los estudiantes de OULAD para su posterior procesamiento mediante un Transformer Encoder.

#### Diseño del Experimento C

El modelo utilizará dos fuentes de información:

**Rama estática:** las mismas 12 características iniciales del Experimento A.

**Rama temporal:** una secuencia de observaciones semanales del comportamiento académico.

Cada semana temporal incluirá cuatro canales:

1. Número de clics registrados en el VLE.
2. Número de días con actividad VLE.
3. Número de entregas registradas.
4. Número de días de exposición desde la inscripción durante la semana.

El último canal permite distinguir la ausencia de actividad de las semanas en las que el estudiante todavía no estaba inscrito.

#### Ventanas de predicción

| Corte | Último día observable | Longitud de secuencia |
|---|---:|---:|
| Semana 2 | 13 | 2 |
| Semana 4 | 27 | 4 |
| Semana 6 | 41 | 6 |
| Semana 8 | 55 | 8 |

#### Prevención de fuga de información

Se utilizarán exclusivamente eventos registrados antes del día de predicción.

Se excluirán entregas marcadas como heredadas (`is_banked = 1`) y actividades producidas fuera del período válido de inscripción.

No se incorporarán calificaciones, resultado final, fecha de retiro ni días hasta el retiro como características predictivas.

La construcción de los canales consistirá en agregaciones deterministas de eventos. Las transformaciones estadísticas, incluida la normalización, deberán aprenderse posteriormente solo con los datos de entrenamiento de cada fold.

#### Unidad de análisis y validación

Se mantendrá la matrícula académica identificada por la combinación de módulo, presentación y estudiante.

Las secuencias conservarán el mismo orden de las matrículas y las mismas particiones agrupadas por estudiante utilizadas anteriormente.

El Experimento C empleará las mismas etiquetas binarias y el mismo conjunto independiente reservado.

#### Alcance

Esta actividad se limitará a construir y auditar los tensores temporales del conjunto de desarrollo.

La definición, entrenamiento y optimización de la arquitectura Transformer se realizará en la siguiente actividad.


In [ ]:

# ============================================================
# PROYECTO FINAL DE MACHINE LEARNING
# ACTIVIDAD 3.18 - SECUENCIAS PARA TRANSFORMER C
# ============================================================

# Asistencia IA: ChatGPT, octubre de 2026.
# Prompt:
# "Construir tensores temporales semanales para un
# Transformer de OULAD en semanas 2, 4, 6 y 8.
# Usar clics VLE, días activos, entregas y exposición.
# Mantener claves compuestas y partición por estudiante,
# evitar fuga temporal y no entrenar el modelo."

import numpy as np
import pandas as pd

from IPython.display import display

# ------------------------------------------------------------
# 1. CONFIGURACIÓN
# ------------------------------------------------------------

SEMANAS_318 = [2, 4, 6, 8]

CLAVES_318 = [
    "code_module",
    "code_presentation",
    "id_student"
]

CANALES_318 = [
    "vle_clics",
    "vle_dias_activos",
    "entregas",
    "dias_exposicion"
]

DIAS_MAX_318 = 56

print("=== 3.18.1 CONFIGURACIÓN ===")
print("Semanas:", SEMANAS_318)
print("Canales:", CANALES_318)

# ------------------------------------------------------------
# 2. REGISTROS DE INSCRIPCIÓN
# ------------------------------------------------------------

registro_318 = (
    oulad_tablas["studentRegistration.csv"][
        CLAVES_318 + [
            "date_registration",
            "date_unregistration"
        ]
    ].copy()
)

assert not registro_318.duplicated(
    subset=CLAVES_318
).any()

# ------------------------------------------------------------
# 3. PREPARAR ACTIVIDAD VLE SEMANAL
# ------------------------------------------------------------

print("\n=== 3.18.2 PREPARACIÓN VLE ===")

vle_diario_318 = actividad_diaria.copy()

assert all(
    c in vle_diario_318.columns
    for c in CLAVES_318 + ["date", "clics"]
)

vle_diario_318 = vle_diario_318.merge(
    registro_318,
    on=CLAVES_318,
    how="left",
    validate="many_to_one",
    indicator=True
)

assert vle_diario_318["_merge"].eq(
    "both"
).all()

# Conservar únicamente eventos temporalmente válidos.
fecha = vle_diario_318["date"]

mascara_vle = (
    fecha.ge(0)
    & fecha.lt(DIAS_MAX_318)
    & vle_diario_318[
        "date_registration"
    ].notna()
    & fecha.ge(
        vle_diario_318["date_registration"]
    )
    & (
        vle_diario_318[
            "date_unregistration"
        ].isna()
        | fecha.lt(
            vle_diario_318[
                "date_unregistration"
            ]
        )
    )
)

vle_valido_318 = (
    vle_diario_318.loc[
        mascara_vle,
        CLAVES_318 + ["date", "clics"]
    ].copy()
)

vle_valido_318["semana_indice"] = (
    vle_valido_318["date"] // 7
).astype(int)

vle_semanal_318 = (
    vle_valido_318.groupby(
        CLAVES_318 + ["semana_indice"],
        as_index=False
    )
    .agg(
        vle_clics=("clics", "sum"),
        vle_dias_activos=("date", "nunique")
    )
)

assert not vle_semanal_318.duplicated(
    subset=CLAVES_318 + ["semana_indice"]
).any()

print(
    "Observaciones estudiante-módulo-día válidas:",
    len(vle_valido_318)
)

print(
    "Registros matrícula-semana con VLE:",
    len(vle_semanal_318)
)

# ------------------------------------------------------------
# 4. PREPARAR ENTREGAS SEMANALES
# ------------------------------------------------------------

print("\n=== 3.18.3 PREPARACIÓN DE ENTREGAS ===")

evaluaciones_318 = (
    oulad_tablas["assessments.csv"][
        [
            "id_assessment",
            "code_module",
            "code_presentation"
        ]
    ].copy()
)

assert not evaluaciones_318.duplicated(
    subset=["id_assessment"]
).any()

entregas_318 = (
    oulad_tablas["studentAssessment.csv"][
        [
            "id_assessment",
            "id_student",
            "date_submitted",
            "is_banked"
        ]
    ].copy()
)

# Las entregas heredadas no representan una
# entrega realizada en la ventana actual.
entregas_318 = entregas_318.loc[
    entregas_318["is_banked"].eq(0)
].copy()

entregas_318 = entregas_318.merge(
    evaluaciones_318,
    on="id_assessment",
    how="inner",
    validate="many_to_one"
)

entregas_318 = entregas_318.merge(
    registro_318,
    on=CLAVES_318,
    how="inner",
    validate="many_to_one"
)

entregas_318["date_submitted"] = pd.to_numeric(
    entregas_318["date_submitted"],
    errors="coerce"
)

fecha_entrega = entregas_318["date_submitted"]

mascara_entregas = (
    fecha_entrega.ge(0)
    & fecha_entrega.lt(DIAS_MAX_318)
    & entregas_318[
        "date_registration"
    ].notna()
    & fecha_entrega.ge(
        entregas_318["date_registration"]
    )
    & (
        entregas_318[
            "date_unregistration"
        ].isna()
        | fecha_entrega.lt(
            entregas_318[
                "date_unregistration"
            ]
        )
    )
)

entregas_validas_318 = (
    entregas_318.loc[
        mascara_entregas,
        CLAVES_318 + ["date_submitted"]
    ].copy()
)

entregas_validas_318["semana_indice"] = (
    entregas_validas_318[
        "date_submitted"
    ] // 7
).astype(int)

entregas_semanales_318 = (
    entregas_validas_318.groupby(
        CLAVES_318 + ["semana_indice"],
        as_index=False
    )
    .size()
    .rename(columns={"size": "entregas"})
)

assert not entregas_semanales_318.duplicated(
    subset=CLAVES_318 + ["semana_indice"]
).any()

print(
    "Entregas ordinarias temporalmente válidas:",
    len(entregas_validas_318)
)

print(
    "Registros matrícula-semana con entregas:",
    len(entregas_semanales_318)
)

# ------------------------------------------------------------
# 5. FUNCIÓN PARA CONSTRUIR TENSORES
# ------------------------------------------------------------

def construir_tensor_C_318(semana):

    datos = experimentos_oulad[semana]

    claves = (
        datos["claves_train"]
        .reset_index(drop=True)
        .copy()
    )

    estaticas = (
        datos["X_A_train"]
        .reset_index(drop=True)
        .copy()
    )

    etiquetas = (
        datos["y_train"]
        .reset_index(drop=True)
    )

    grupos = (
        datos["grupos_train"]
        .reset_index(drop=True)
    )

    n = len(claves)

    assert (
        len(estaticas)
        == len(etiquetas)
        == len(grupos)
        == n
    )

    assert not claves.duplicated().any()

    assert (
        claves["id_student"]
        .to_numpy()
        == grupos.to_numpy()
    ).all()

    # Generar n filas por cada semana observable.
    grilla = pd.DataFrame({
        c: np.repeat(
            claves[c].to_numpy(),
            semana
        )
        for c in CLAVES_318
    })

    grilla["semana_indice"] = np.tile(
        np.arange(semana),
        n
    )

    grilla = grilla.merge(
        vle_semanal_318,
        on=CLAVES_318 + ["semana_indice"],
        how="left",
        validate="one_to_one"
    )

    grilla = grilla.merge(
        entregas_semanales_318,
        on=CLAVES_318 + ["semana_indice"],
        how="left",
        validate="one_to_one"
    )

    # Ausencia de eventos: recuento cero.
    for canal in [
        "vle_clics",
        "vle_dias_activos",
        "entregas"
    ]:
        grilla[canal] = (
            grilla[canal].fillna(0)
        )

    # --------------------------------------------------------
    # 6. CALCULAR DÍAS DE EXPOSICIÓN
    # --------------------------------------------------------

    fecha_registro = pd.to_numeric(
        estaticas["date_registration"],
        errors="coerce"
    ).to_numpy(dtype=float)

    assert np.isfinite(
        fecha_registro
    ).all()

    registro_repetido = np.repeat(
        fecha_registro,
        semana
    )

    comienzo_semana = (
        grilla["semana_indice"]
        .to_numpy(dtype=int) * 7
    )

    final_exclusivo = comienzo_semana + 7

    comienzo_exposicion = np.maximum(
        comienzo_semana,
        np.maximum(registro_repetido, 0)
    )

    dias_exposicion = np.clip(
        final_exclusivo - comienzo_exposicion,
        0,
        7
    )

    grilla["dias_exposicion"] = (
        dias_exposicion
    )

    # --------------------------------------------------------
    # 7. CONSTRUIR TENSOR Y AUDITAR
    # --------------------------------------------------------

    secuencia = (
        grilla[CANALES_318]
        .to_numpy(dtype=np.float32)
        .reshape(
            n,
            semana,
            len(CANALES_318)
        )
    )

    assert secuencia.shape == (
        n,
        semana,
        4
    )

    assert np.isfinite(
        secuencia
    ).all()

    assert (secuencia >= 0).all()

    assert (
        secuencia[:, :, 1]
        <= secuencia[:, :, 3]
    ).all(), (
        "Se encontraron más días activos "
        "que días de exposición."
    )

    assert (
        secuencia[:, :, 3] <= 7
    ).all()

    # Comprobar que ninguna observación con
    # cero exposición contiene eventos.
    sin_exposicion = (
        secuencia[:, :, 3] == 0
    )

    assert (
        secuencia[:, :, :3][
            sin_exposicion
        ] == 0
    ).all(), (
        "Hay actividad en semanas sin exposición."
    )

    # --------------------------------------------------------
    # 8. COMPROBAR CONSISTENCIA CON EXPERIMENTO B
    # --------------------------------------------------------

    nombre_clics = f"vle_clics_{semana * 7}d"

    if nombre_clics in datos["X_B_train"].columns:

        clics_existentes = (
            datos["X_B_train"][
                nombre_clics
            ]
            .reset_index(drop=True)
            .to_numpy(dtype=float)
        )

        clics_secuencia = (
            secuencia[:, :, 0].sum(axis=1)
        )

        diferencias = ~np.isclose(
            clics_existentes,
            clics_secuencia,
            atol=0.01,
            rtol=0
        )

        if diferencias.any():
            raise ValueError(
                f"Semana {semana}: "
                f"{diferencias.sum()} matrículas "
                "tienen recuentos VLE distintos "
                "entre B y C. Revisar antes "
                "de entrenar el Transformer."
            )

    # Las transformaciones estadísticas de los
    # canales se ajustarán dentro de cada fold.
    # Aquí se conservan recuentos originales.

    return {
        "X_secuencia_train": secuencia,
        "X_estatica_train": estaticas,
        "y_train": etiquetas,
        "grupos_train": grupos,
        "claves_train": claves,
        "canales": CANALES_318.copy(),
        "semana_prediccion": semana
    }


# ------------------------------------------------------------
# 9. CONSTRUIR LAS CUATRO VENTANAS
# ------------------------------------------------------------

print("\n=== 3.18.4 CONSTRUCCIÓN DE TENSORES ===")

secuencias_oulad_C = {}

resumen_secuencias_318 = []

for semana in SEMANAS_318:

    resultado = construir_tensor_C_318(
        semana
    )

    secuencias_oulad_C[semana] = resultado

    X_seq = resultado[
        "X_secuencia_train"
    ]

    idx_clics = CANALES_318.index("vle_clics")
    idx_entregas = CANALES_318.index("entregas")
    idx_exposicion = CANALES_318.index("dias_exposicion")

    clics_totales = X_seq[:, :, idx_clics].sum(
        axis=1
    )

    entregas_totales = X_seq[
        :, :, idx_entregas
    ].sum(axis=1)

    exposicion_total = X_seq[
        :, :, idx_exposicion
    ].sum(axis=1)

    resumen_secuencias_318.append({
        "Semana": semana,
        "Matriculas": X_seq.shape[0],
        "Semanas_temporales": X_seq.shape[1],
        "Canales": X_seq.shape[2],
        "Estudiantes_unicos": resultado[
            "grupos_train"
        ].nunique(),
        "%_con_VLE": (
            100 * np.mean(clics_totales > 0)
        ),
        "%_con_entregas": (
            100 * np.mean(entregas_totales > 0)
        ),
        "Mediana_clics": np.median(clics_totales),
        "Mediana_entregas": np.median(
            entregas_totales
        ),
        "Mediana_dias_exposicion": np.median(
            exposicion_total
        ),
        "Nulos_o_infinitos": int(
            (~np.isfinite(X_seq)).sum()
        )
    })

    print(
        f"Semana {semana}: "
        f"tensor {X_seq.shape}"
    )

tabla_secuencias_318 = pd.DataFrame(
    resumen_secuencias_318
)

display(
    tabla_secuencias_318.round(2)
)

# ------------------------------------------------------------
# 10. AUDITORÍA DE LOS FOLDS
# ------------------------------------------------------------

print("\n=== 3.18.5 AUDITORÍA DE PARTICIONES ===")

auditoria_318 = []

for semana in SEMANAS_318:

    grupos = secuencias_oulad_C[
        semana
    ]["grupos_train"]

    splits = cv_splits_oulad[semana]

    for numero, (it, iv) in enumerate(
        splits,
        start=1
    ):

        solapamiento = len(
            set(grupos.iloc[it])
            & set(grupos.iloc[iv])
        )

        assert solapamiento == 0

        auditoria_318.append({
            "Semana": semana,
            "Fold_externo": numero,
            "Matriculas_train": len(it),
            "Matriculas_validacion": len(iv),
            "Grupos_compartidos": solapamiento
        })

tabla_auditoria_C_318 = pd.DataFrame(
    auditoria_318
)

display(
    tabla_auditoria_C_318.groupby("Semana")
    .agg(
        Folds=("Fold_externo", "size"),
        Solapamiento_maximo=(
            "Grupos_compartidos", "max"
        )
    )
    .reset_index()
)

# ------------------------------------------------------------
# 11. CONTROLES FINALES
# ------------------------------------------------------------

print("\n=== 3.18.6 CONTROLES FINALES ===")

assert len(secuencias_oulad_C) == 4

assert (
    tabla_auditoria_C_318[
        "Grupos_compartidos"
    ] == 0
).all()

assert (
    tabla_secuencias_318[
        "Nulos_o_infinitos"
    ] == 0
).all()

print("Ventanas preparadas: 4")
print("Canales temporales: 4")
print("Secuencias ordenadas cronológicamente.")
print("Datos posteriores al corte: no utilizados.")
print("Calificaciones: no utilizadas.")
print("Resultado final: no utilizado como predictor.")
print("Transformaciones ajustadas: ninguna.")
print("Redes neuronales entrenadas: ninguna.")
print("Conjunto de prueba independiente: reservado.")

print("\n=== ACTIVIDAD 3.18 FINALIZADA ===")



### 3.18.7. Resultados e interpretación

Se completó la construcción de las secuencias temporales necesarias para el Experimento C, basado en un Transformer Encoder.

Se utilizaron cuatro canales semanales: clics VLE, días activos, entregas y días de exposición desde la inscripción.

**Dimensiones de los tensores**

| Semana | Matrículas de desarrollo | Dimensiones |
|---|---:|---|
| 2 | 22.412 | (22412, 2, 4) |
| 4 | 21.984 | (21984, 4, 4) |
| 6 | 21.568 | (21568, 6, 4) |
| 8 | 21.164 | (21164, 8, 4) |

La cobertura VLE aumentó del 90,59 % en la semana 2 al 97,21 % en la semana 8.

La proporción de matrículas con entregas aumentó del 10,33 % al 85,95 % durante el mismo período.

Estos resultados evidencian una disponibilidad creciente de información académica susceptible de representarse mediante secuencias temporales.

**Auditoría de integridad**

La construcción superó los controles de dimensiones, finitud de valores, orden temporal y coherencia de los clics acumulados con el Experimento B.

Las diez particiones de cada ventana mantuvieron cero estudiantes compartidos entre entrenamiento y validación.

Los tensores se construyeron sin incorporar calificaciones, resultados finales o variables explícitas de retiro como predictores.

No se ajustaron transformaciones estadísticas ni se entrenaron redes neuronales.

### 3.18.8. Consideraciones metodológicas

La preparación utiliza agregaciones temporales deterministas, pero la normalización de los canales y el preprocesamiento de las variables estáticas deberán realizarse exclusivamente con las observaciones de entrenamiento correspondientes a cada partición.

La cobertura creciente de entregas y actividad no demuestra por sí sola una mejora predictiva; su contribución deberá cuantificarse mediante evaluación fuera de muestra.

El Experimento C utilizará las mismas etiquetas y la misma separación agrupada por estudiante que los experimentos clásicos.

### 3.18.9. Conclusión

Se construyeron satisfactoriamente las representaciones secuenciales correspondientes a las cuatro ventanas temporales del estudio.

Las estructuras obtenidas permiten implementar un Transformer Encoder que aprenda relaciones entre observaciones semanales, complementando el enfoque basado en características longitudinales agregadas del Experimento B.

**Conclusión principal:** los datos están preparados para iniciar el desarrollo del modelo neuronal, manteniendo los controles fundamentales de consistencia temporal, agrupación por estudiante y separación de entrenamiento y evaluación.



### 3.19. Diseño y entrenamiento piloto del Transformer C

#### Objetivo

Implementar y verificar una arquitectura Transformer Encoder para la predicción de resultados académicos adversos a partir de trayectorias semanales de comportamiento estudiantil y características estáticas.

#### Arquitectura

El modelo utilizará dos ramas de procesamiento.

**Rama temporal**

- Cuatro canales por semana: clics VLE, días activos, entregas y días de exposición.
- Proyección lineal a 32 dimensiones.
- Codificación posicional aprendida.
- Token de clasificación (`CLS`) para representar la secuencia.
- Dos capas Transformer Encoder.
- Cuatro cabezas de atención.
- Dropout de 0,15.

Las semanas sin exposición serán identificadas mediante una máscara de atención.

**Rama estática**

Las 12 variables del Experimento A serán preprocesadas mediante imputación, escalamiento numérico y codificación One-Hot para variables categóricas.

La representación estática se proyectará a 32 dimensiones y se combinará con la representación temporal.

**Clasificación**

La representación conjunta se procesará mediante capas densas y una salida binaria con función sigmoide durante la evaluación.

La pérdida de entrenamiento será `BCEWithLogitsLoss`.

#### Diseño del piloto

Se utilizará la cohorte de desarrollo de la semana 8.

Se tomará el primer fold externo únicamente para delimitar los datos disponibles durante el piloto.

Dentro de su entrenamiento se generará una partición adicional mediante `StratifiedGroupKFold`, agrupando matrículas por estudiante.

El fold externo de validación no participará en el entrenamiento, la normalización, la selección de épocas ni el cálculo de métricas de esta actividad.

#### Preprocesamiento sin fuga

Los parámetros de imputación, codificación y escalamiento se aprenderán exclusivamente sobre la partición interna de entrenamiento.

Los canales temporales se transformarán mediante `log1p` para los recuentos de actividad, seguido de estandarización ajustada exclusivamente sobre las observaciones internas de entrenamiento con exposición válida.

#### Entrenamiento

Se utilizará AdamW, una tasa de aprendizaje inicial de 0,0007, un máximo de 20 épocas y parada temprana con paciencia de cinco épocas.

La época se seleccionará minimizando la pérdida de validación interna.

Se calcularán F1 macro, recall adverso, ROC-AUC y accuracy.

#### Alcance y limitaciones

Esta actividad constituye un piloto técnico y metodológico.

Los resultados de validación interna no son directamente comparables con los promedios de validación externa anidada de los experimentos A y B.

La arquitectura deberá fijarse antes de ejecutar una evaluación externa completa del Transformer.

El conjunto de prueba independiente permanecerá reservado.

**Referencias metodológicas:** el experimento Transformer toma como antecedente la arquitectura de atención de Vaswani *et al.* [8]; la implementación modular con PyTorch se relaciona con [13]. Esta adaptación a OULAD no reproduce íntegramente el Transformer original.


In [ ]:

# ============================================================
# PROYECTO FINAL DE MACHINE LEARNING
# ACTIVIDAD 3.19 - TRANSFORMER C
# ARQUITECTURA Y ENTRENAMIENTO PILOTO
# ============================================================

# Asistencia IA: ChatGPT, octubre de 2026.
# Prompt:
# "Implementar un Transformer Encoder en PyTorch
# para secuencias semanales de OULAD, combinando
# características estáticas y temporales.
# Realizar un piloto en semana 8 con validación
# interna agrupada por estudiante, preprocesamiento
# sin fuga, early stopping y métricas reproducibles.
# No evaluar el fold externo ni la prueba independiente."

import copy
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn

from torch.utils.data import (
    TensorDataset,
    DataLoader
)

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder
)
from sklearn.impute import SimpleImputer

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

from IPython.display import display

# ------------------------------------------------------------
# 1. CONFIGURACIÓN GENERAL
# ------------------------------------------------------------

SEMANA_319 = 8
SEMILLA_319 = 42

BATCH_SIZE_319 = 256
MAX_EPOCAS_319 = 20
PACIENCIA_319 = 5
LR_319 = 0.0007
WEIGHT_DECAY_319 = 0.001

random.seed(SEMILLA_319)
np.random.seed(SEMILLA_319)
torch.manual_seed(SEMILLA_319)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEMILLA_319)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

dispositivo_319 = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

datos_319 = secuencias_oulad_C[SEMANA_319]

X_seq_original = datos_319[
    "X_secuencia_train"
].copy()

X_est_original = (
    datos_319["X_estatica_train"]
    .reset_index(drop=True)
    .copy()
)

y_original = (
    datos_319["y_train"]
    .reset_index(drop=True)
)

grupos_original = (
    datos_319["grupos_train"]
    .reset_index(drop=True)
)

print("=== 3.19.1 CONFIGURACIÓN ===")

print("PyTorch:", torch.__version__)
print("Dispositivo:", dispositivo_319)
print("Semana:", SEMANA_319)
print("Secuencias:", X_seq_original.shape)
print("Características estáticas:", X_est_original.shape)
print("Épocas máximas:", MAX_EPOCAS_319)

assert X_seq_original.shape == (
    len(y_original), 8, 4
)

# ------------------------------------------------------------
# 2. PARTICIÓN INTERNA AGRUPADA
# ------------------------------------------------------------

# Se toma el primer fold externo solamente para
# delimitar el conjunto permitido durante el piloto.

idx_externo_train, idx_externo_val = (
    cv_splits_oulad[SEMANA_319][0]
)

# Importante: idx_externo_val no se utiliza para
# preprocesar, entrenar, elegir épocas ni evaluar.

assert set(
    grupos_original.iloc[idx_externo_train]
).isdisjoint(
    set(grupos_original.iloc[idx_externo_val])
)

cv_interno_319 = StratifiedGroupKFold(
    n_splits=3,
    shuffle=True,
    random_state=137
)

indices_internos = list(
    cv_interno_319.split(
        X_est_original.iloc[idx_externo_train],
        y_original.iloc[idx_externo_train],
        groups=grupos_original.iloc[idx_externo_train]
    )
)

idx_interno_train_rel, idx_interno_val_rel = (
    indices_internos[0]
)

idx_train_319 = idx_externo_train[
    idx_interno_train_rel
]

idx_val_319 = idx_externo_train[
    idx_interno_val_rel
]

assert set(
    grupos_original.iloc[idx_train_319]
).isdisjoint(
    set(grupos_original.iloc[idx_val_319])
)

assert set(
    grupos_original.iloc[idx_train_319]
).isdisjoint(
    set(grupos_original.iloc[idx_externo_val])
)

assert set(
    grupos_original.iloc[idx_val_319]
).isdisjoint(
    set(grupos_original.iloc[idx_externo_val])
)

y_train_319 = (
    y_original.iloc[idx_train_319]
    .to_numpy(dtype=np.float32)
)

y_val_319 = (
    y_original.iloc[idx_val_319]
    .to_numpy(dtype=np.float32)
)

print("\n=== 3.19.2 PARTICIÓN INTERNA ===")

print("Matrículas entrenamiento:", len(idx_train_319))
print("Matrículas validación:", len(idx_val_319))
print("Adversos entrenamiento (%):", round(
    100 * y_train_319.mean(), 2
))
print("Adversos validación (%):", round(
    100 * y_val_319.mean(), 2
))
print("Estudiantes compartidos: 0")
print("Fold externo de validación: reservado")

# ------------------------------------------------------------
# 3. PREPROCESAMIENTO ESTÁTICO
# ------------------------------------------------------------

X_est_train = X_est_original.iloc[
    idx_train_319
].copy()

X_est_val = X_est_original.iloc[
    idx_val_319
].copy()

columnas_numericas = (
    X_est_train.select_dtypes(
        include=[np.number]
    ).columns.tolist()
)

columnas_categoricas = [
    c for c in X_est_train.columns
    if c not in columnas_numericas
]

pipeline_num = Pipeline([
    (
        "imputacion",
        SimpleImputer(strategy="median")
    ),
    (
        "escalamiento",
        StandardScaler()
    )
])

pipeline_cat = Pipeline([
    (
        "imputacion",
        SimpleImputer(strategy="most_frequent")
    ),
    (
        "onehot",
        OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        )
    )
])

preprocesador_estatico_319 = ColumnTransformer(
    transformers=[
        ("num", pipeline_num, columnas_numericas),
        ("cat", pipeline_cat, columnas_categoricas)
    ],
    remainder="drop",
    sparse_threshold=0
)

# Fit exclusivamente en entrenamiento interno.
X_est_train_proc = (
    preprocesador_estatico_319.fit_transform(
        X_est_train
    )
    .astype(np.float32)
)

X_est_val_proc = (
    preprocesador_estatico_319.transform(
        X_est_val
    )
    .astype(np.float32)
)

assert X_est_train_proc.shape[1] == (
    X_est_val_proc.shape[1]
)

assert np.isfinite(X_est_train_proc).all()
assert np.isfinite(X_est_val_proc).all()

# ------------------------------------------------------------
# 4. PREPROCESAMIENTO TEMPORAL
# ------------------------------------------------------------

X_seq_train_raw = X_seq_original[
    idx_train_319
].copy()

X_seq_val_raw = X_seq_original[
    idx_val_319
].copy()


def preparar_temporales_319(train, valid):

    tr = train.astype(np.float32).copy()
    va = valid.astype(np.float32).copy()

    # Las tres primeras columnas son recuentos.
    # La cuarta es exposición semanal de 0 a 7.

    tr[:, :, :3] = np.log1p(
        tr[:, :, :3]
    )

    va[:, :, :3] = np.log1p(
        va[:, :, :3]
    )

    tr[:, :, 3] = tr[:, :, 3] / 7.0
    va[:, :, 3] = va[:, :, 3] / 7.0

    mascara_train = train[:, :, 3] <= 0
    mascara_valid = valid[:, :, 3] <= 0

    # Ajustar el escalador solo sobre semanas
    # realmente observables del entrenamiento.

    filas_validas_train = tr[
        ~mascara_train
    ]

    assert len(filas_validas_train) > 0

    escalador = StandardScaler()

    escalador.fit(
        filas_validas_train
    )

    tr = escalador.transform(
        tr.reshape(-1, 4)
    ).reshape(tr.shape).astype(np.float32)

    va = escalador.transform(
        va.reshape(-1, 4)
    ).reshape(va.shape).astype(np.float32)

    # Los pasos sin exposición quedan en cero.
    # El Transformer los ignorará con su máscara.

    tr[mascara_train] = 0.0
    va[mascara_valid] = 0.0

    assert np.isfinite(tr).all()
    assert np.isfinite(va).all()

    return (
        tr,
        va,
        mascara_train,
        mascara_valid,
        escalador
    )


(
    X_seq_train_proc,
    X_seq_val_proc,
    mask_train_319,
    mask_val_319,
    escalador_temporal_319
) = preparar_temporales_319(
    X_seq_train_raw,
    X_seq_val_raw
)

print("\n=== 3.19.3 PREPROCESAMIENTO ===")

print(
    "Dimensiones estáticas codificadas:",
    X_est_train_proc.shape[1]
)

print(
    "Tensor temporal entrenamiento:",
    X_seq_train_proc.shape
)

print(
    "Tensor temporal validación:",
    X_seq_val_proc.shape
)

print(
    "Semanas sin exposición en entrenamiento:",
    int(mask_train_319.sum())
)

print("Escaladores ajustados: solo entrenamiento interno")

# ------------------------------------------------------------
# 5. CONSTRUIR DATASETS DE PYTORCH
# ------------------------------------------------------------

def crear_dataset_319(
    secuencias,
    estaticas,
    mascara,
    etiquetas
):

    return TensorDataset(
        torch.tensor(
            secuencias,
            dtype=torch.float32
        ),
        torch.tensor(
            estaticas,
            dtype=torch.float32
        ),
        torch.tensor(
            mascara,
            dtype=torch.bool
        ),
        torch.tensor(
            etiquetas,
            dtype=torch.float32
        )
    )


dataset_train_319 = crear_dataset_319(
    X_seq_train_proc,
    X_est_train_proc,
    mask_train_319,
    y_train_319
)

dataset_val_319 = crear_dataset_319(
    X_seq_val_proc,
    X_est_val_proc,
    mask_val_319,
    y_val_319
)

generador_319 = torch.Generator()
generador_319.manual_seed(SEMILLA_319)

loader_train_319 = DataLoader(
    dataset_train_319,
    batch_size=BATCH_SIZE_319,
    shuffle=True,
    generator=generador_319,
    num_workers=0
)

loader_val_319 = DataLoader(
    dataset_val_319,
    batch_size=BATCH_SIZE_319,
    shuffle=False,
    num_workers=0
)

# ------------------------------------------------------------
# 6. DEFINIR TRANSFORMER ENCODER
# ------------------------------------------------------------

class TransformerOULAD(nn.Module):

    def __init__(
        self,
        n_static,
        n_weeks,
        n_channels=4,
        d_model=32,
        n_heads=4,
        n_layers=2,
        dropout=0.15
    ):

        super().__init__()

        self.proyeccion_temporal = nn.Linear(
            n_channels,
            d_model
        )

        # Token para resumir la secuencia.
        self.cls_token = nn.Parameter(
            torch.zeros(1, 1, d_model)
        )

        # Posiciones aprendidas para CLS y semanas.
        self.posiciones = nn.Parameter(
            torch.zeros(
                1,
                n_weeks + 1,
                d_model
            )
        )

        nn.init.normal_(
            self.cls_token,
            std=0.02
        )

        nn.init.normal_(
            self.posiciones,
            std=0.02
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=64,
            dropout=dropout,
            activation="gelu",
            batch_first=True
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=n_layers,
            enable_nested_tensor=False
        )

        self.normalizacion_temporal = nn.LayerNorm(
            d_model
        )

        self.rama_estatica = nn.Sequential(
            nn.Linear(n_static, 32),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        self.clasificador = nn.Sequential(
            nn.Linear(d_model + 32, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1)
        )

    def forward(
        self,
        x_seq,
        x_static,
        mask_temporal
    ):

        batch_size = x_seq.shape[0]

        representacion = self.proyeccion_temporal(
            x_seq
        )

        cls = self.cls_token.expand(
            batch_size,
            -1,
            -1
        )

        tokens = torch.cat(
            [cls, representacion],
            dim=1
        )

        tokens = tokens + self.posiciones[
            :, :tokens.shape[1], :
        ]

        # CLS nunca queda enmascarado.
        mascara_cls = torch.zeros(
            (batch_size, 1),
            device=x_seq.device,
            dtype=torch.bool
        )

        mascara_completa = torch.cat(
            [mascara_cls, mask_temporal],
            dim=1
        )

        salida = self.encoder(
            tokens,
            src_key_padding_mask=mascara_completa
        )

        embedding_temporal = (
            self.normalizacion_temporal(
                salida[:, 0, :]
            )
        )

        embedding_estatico = self.rama_estatica(
            x_static
        )

        fusion = torch.cat(
            [
                embedding_temporal,
                embedding_estatico
            ],
            dim=1
        )

        logits = self.clasificador(
            fusion
        ).squeeze(-1)

        return logits


modelo_C_319 = TransformerOULAD(
    n_static=X_est_train_proc.shape[1],
    n_weeks=SEMANA_319
).to(dispositivo_319)

numero_parametros_319 = sum(
    p.numel()
    for p in modelo_C_319.parameters()
    if p.requires_grad
)

print("\n=== 3.19.4 ARQUITECTURA ===")

print("Parámetros entrenables:", numero_parametros_319)
print("Capas Transformer:", 2)
print("Cabezas de atención:", 4)
print("Dimensión temporal:", 32)
print("Dimensión rama estática:", 32)

# ------------------------------------------------------------
# 7. FUNCIÓN DE EVALUACIÓN INTERNA
# ------------------------------------------------------------

criterio_319 = nn.BCEWithLogitsLoss()


@torch.no_grad()
def evaluar_loader_319(modelo, loader):

    modelo.eval()

    perdidas = []
    reales = []
    probabilidades = []

    total_observaciones = 0
    suma_perdida = 0.0

    for x_seq, x_static, mask, y_batch in loader:

        x_seq = x_seq.to(dispositivo_319)
        x_static = x_static.to(dispositivo_319)
        mask = mask.to(dispositivo_319)
        y_batch = y_batch.to(dispositivo_319)

        logits = modelo(
            x_seq,
            x_static,
            mask
        )

        loss = criterio_319(
            logits,
            y_batch
        )

        n_batch = len(y_batch)

        suma_perdida += loss.item() * n_batch
        total_observaciones += n_batch

        prob = torch.sigmoid(
            logits
        )

        reales.append(
            y_batch.cpu().numpy()
        )

        probabilidades.append(
            prob.cpu().numpy()
        )

    y_real = np.concatenate(
        reales
    ).astype(int)

    prob = np.concatenate(
        probabilidades
    )

    pred = (
        prob >= 0.5
    ).astype(int)

    return {
        "Loss": suma_perdida / total_observaciones,
        "F1_macro": f1_score(
            y_real,
            pred,
            average="macro",
            zero_division=0
        ),
        "Recall_adverso": recall_score(
            y_real,
            pred,
            zero_division=0
        ),
        "Precision_adverso": precision_score(
            y_real,
            pred,
            zero_division=0
        ),
        "Accuracy": accuracy_score(
            y_real,
            pred
        ),
        "ROC_AUC": roc_auc_score(
            y_real,
            prob
        )
    }


# ------------------------------------------------------------
# 8. OPTIMIZADOR Y ENTRENAMIENTO
# ------------------------------------------------------------

optimizador_319 = torch.optim.AdamW(
    modelo_C_319.parameters(),
    lr=LR_319,
    weight_decay=WEIGHT_DECAY_319
)

historial_319 = []

mejor_loss_319 = np.inf
mejor_epoca_319 = 0
mejor_estado_319 = None

epocas_sin_mejora_319 = 0

print("\n=== 3.19.5 ENTRENAMIENTO PILOTO ===")

for epoca in range(1, MAX_EPOCAS_319 + 1):

    modelo_C_319.train()

    suma_loss_train = 0.0
    n_train = 0

    for x_seq, x_static, mask, y_batch in (
        loader_train_319
    ):

        x_seq = x_seq.to(dispositivo_319)
        x_static = x_static.to(dispositivo_319)
        mask = mask.to(dispositivo_319)
        y_batch = y_batch.to(dispositivo_319)

        optimizador_319.zero_grad(
            set_to_none=True
        )

        logits = modelo_C_319(
            x_seq,
            x_static,
            mask
        )

        loss = criterio_319(
            logits,
            y_batch
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            modelo_C_319.parameters(),
            max_norm=1.0
        )

        optimizador_319.step()

        suma_loss_train += (
            loss.item() * len(y_batch)
        )

        n_train += len(y_batch)

    loss_train = (
        suma_loss_train / n_train
    )

    # Exclusivamente validación interna.
    metricas_val = evaluar_loader_319(
        modelo_C_319,
        loader_val_319
    )

    historial_319.append({
        "Epoca": epoca,
        "Loss_train": loss_train,
        "Loss_validacion": metricas_val["Loss"],
        "F1_macro_validacion": (
            metricas_val["F1_macro"]
        ),
        "Recall_validacion": (
            metricas_val["Recall_adverso"]
        ),
        "ROC_AUC_validacion": (
            metricas_val["ROC_AUC"]
        )
    })

    print(
        f"Época {epoca:02d} | "
        f"Loss train={loss_train:.4f} | "
        f"Loss val={metricas_val['Loss']:.4f} | "
        f"F1={metricas_val['F1_macro']:.4f} | "
        f"AUC={metricas_val['ROC_AUC']:.4f}",
        flush=True
    )

    # Seleccionar usando SOLO pérdida interna.
    if metricas_val["Loss"] < (
        mejor_loss_319 - 1e-4
    ):

        mejor_loss_319 = metricas_val["Loss"]
        mejor_epoca_319 = epoca

        mejor_estado_319 = copy.deepcopy(
            modelo_C_319.state_dict()
        )

        epocas_sin_mejora_319 = 0

    else:

        epocas_sin_mejora_319 += 1

        if epocas_sin_mejora_319 >= PACIENCIA_319:

            print(
                "Early stopping activado."
            )

            break

assert mejor_estado_319 is not None

# Recuperar los pesos de la mejor época.
modelo_C_319.load_state_dict(
    mejor_estado_319
)

# ------------------------------------------------------------
# 9. RESULTADOS DEL MEJOR MODELO INTERNO
# ------------------------------------------------------------

metricas_finales_319 = evaluar_loader_319(
    modelo_C_319,
    loader_val_319
)

tabla_historial_319 = pd.DataFrame(
    historial_319
)

print("\n=== 3.19.6 RESULTADOS INTERNOS ===")

print("Mejor época:", mejor_epoca_319)

display(
    pd.DataFrame([{
        "Metrica": nombre,
        "Validacion_interna": valor
    } for nombre, valor in metricas_finales_319.items()]
    ).round(4)
)

print(
    "\nEstas métricas corresponden únicamente "
    "a validación interna del piloto."
)

# ------------------------------------------------------------
# 10. GRÁFICOS DE ENTRENAMIENTO
# ------------------------------------------------------------

print("\n=== 3.19.7 CURVAS DE ENTRENAMIENTO ===")

plt.figure(figsize=(10, 5))

plt.plot(
    tabla_historial_319["Epoca"],
    tabla_historial_319["Loss_train"],
    marker="o",
    label="Entrenamiento"
)

plt.plot(
    tabla_historial_319["Epoca"],
    tabla_historial_319["Loss_validacion"],
    marker="s",
    label="Validación interna"
)

plt.axvline(
    mejor_epoca_319,
    linestyle="--",
    label="Mejor época"
)

plt.title(
    "Transformer C - Evolución de la pérdida"
)

plt.xlabel("Época")
plt.ylabel("BCEWithLogitsLoss")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 5))

plt.plot(
    tabla_historial_319["Epoca"],
    tabla_historial_319["F1_macro_validacion"],
    marker="o",
    label="F1 macro"
)

plt.plot(
    tabla_historial_319["Epoca"],
    tabla_historial_319["ROC_AUC_validacion"],
    marker="s",
    label="ROC-AUC"
)

plt.axvline(
    mejor_epoca_319,
    linestyle="--",
    label="Mejor época"
)

plt.title(
    "Transformer C - Rendimiento de validación interna"
)

plt.xlabel("Época")
plt.ylabel("Métrica")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 11. CONTROLES FINALES
# ------------------------------------------------------------

print("\n=== 3.19.8 CONTROLES FINALES ===")

assert np.isfinite(
    tabla_historial_319[
        [
            "Loss_train",
            "Loss_validacion",
            "F1_macro_validacion",
            "ROC_AUC_validacion"
        ]
    ].to_numpy()
).all()

assert 1 <= mejor_epoca_319 <= MAX_EPOCAS_319

assert 0 <= metricas_finales_319["F1_macro"] <= 1
assert 0 <= metricas_finales_319["ROC_AUC"] <= 1

print("Arquitectura Transformer: construida")
print("Preprocesamiento: solo entrenamiento interno")
print("Máscara temporal: implementada")
print("Parada temprana: implementada")
print("Estudiantes compartidos: 0")
print("Fold externo de validación: NO evaluado")
print("Conjunto de prueba independiente: NO utilizado")

print(
    "\nEl piloto no sustituye a la validación "
    "cruzada anidada del Experimento C."
)

print("\n=== ACTIVIDAD 3.19 FINALIZADA ===")



### 3.19.9. Resultados e interpretación

Se implementó y entrenó un Transformer Encoder de dos capas y cuatro cabezas de atención para predecir resultados académicos adversos a partir de trayectorias semanales del comportamiento estudiantil y características estáticas.

El modelo contó con 21.377 parámetros entrenables y utilizó una representación temporal de ocho semanas con cuatro canales por semana.

El piloto se desarrolló sobre una partición interna agrupada por estudiante, con 11.255 matrículas de entrenamiento y 5.643 de validación.

El fold externo de validación y el conjunto de prueba independiente permanecieron reservados.

#### Resultados de validación interna

| Métrica | Resultado |
|---|---:|
| F1 macro | 0,7011 |
| Recall adverso | 0,5307 |
| Precisión adversa | 0,7434 |
| Accuracy | 0,7236 |
| ROC-AUC | 0,7813 |
| Pérdida de validación | 0,5431 |

Los resultados indican que la arquitectura logró aprender patrones útiles para clasificar resultados académicos adversos sobre estudiantes no utilizados durante el ajuste.

#### Evolución del entrenamiento

La pérdida de entrenamiento disminuyó de 0,6217 a 0,5215 durante veinte épocas.

La pérdida de validación disminuyó de 0,6126 a 0,5431, con oscilaciones durante las épocas finales.

El ROC-AUC de validación aumentó de 0,7128 a 0,7813 y el F1 macro pasó de 0,6450 a 0,7011.

El máximo F1 macro observado fue de 0,7078 en la época 9, mientras que la menor pérdida de validación se obtuvo en la época 20.

Esta diferencia se explica porque la optimización de probabilidades mediante la función de pérdida y la clasificación mediante un umbral fijo no necesariamente producen máximos coincidentes.

#### Regularización y selección de épocas

Se incorporaron dropout, AdamW, clipping de gradientes y un mecanismo de parada temprana basado en la pérdida de validación.

Sin embargo, el entrenamiento alcanzó las veinte épocas máximas sin activar una parada anticipada.

Por tanto, no es posible concluir que se haya identificado la duración óptima del entrenamiento.

Se recomienda realizar una selección de épocas utilizando varias particiones internas agrupadas por estudiante, sin consultar resultados de validación externa.

#### Comportamiento de las alertas

El modelo alcanzó una precisión adversa de 0,7434 y un recall adverso de 0,5307.

Estos resultados indican que el clasificador identificó una parte importante de los casos adversos, aunque todavía dejó sin detectar aproximadamente el 46,93 % de ellos.

La eventual optimización del umbral de alerta deberá realizarse exclusivamente dentro de validación interna y mediante un protocolo comparable con los demás modelos.

### 3.19.10. Limitaciones

La evaluación corresponde exclusivamente a una partición interna de la semana 8.

Los resultados no son directamente comparables con los promedios externos de los experimentos A y B.

La arquitectura se evaluó con una única configuración inicial de hiperparámetros y una semilla de entrenamiento.

El entrenamiento se ejecutó sobre CPU, lo que puede incrementar el costo computacional de evaluaciones repetidas.

No se ha realizado una evaluación externa del Transformer ni una evaluación sobre el conjunto independiente reservado.

### 3.19.11. Conclusión

El piloto confirmó la viabilidad técnica del Transformer Encoder para procesar secuencias temporales de OULAD y combinar información estática y longitudinal.

La arquitectura mostró una reducción sostenida de la pérdida y mejoras en las métricas de validación interna.

No se observaron señales concluyentes de sobreajuste severo, aunque el entrenamiento finalizó en el máximo de épocas permitido.

**Conclusión principal:** el Experimento C dispone de una arquitectura funcional y de un procedimiento de preprocesamiento temporal compatible con el protocolo de prevención de fuga de información. Antes de comparar su rendimiento con los modelos clásicos, será necesario seleccionar la duración del entrenamiento mediante validación interna y efectuar una evaluación externa agrupada por estudiante.



### 3.20. Primera evaluación externa anidada del Transformer C

#### Objetivo

Evaluar por primera vez la capacidad de generalización del Transformer C utilizando una partición externa de la semana 8 y un procedimiento interno de selección de la duración del entrenamiento.

#### Protocolo

Se utilizará el primer fold externo de la primera repetición de la semana 8, conservando las mismas matrículas y la misma separación por estudiante empleadas en los experimentos clásicos.

Dentro del conjunto de entrenamiento externo se generarán tres folds internos mediante `StratifiedGroupKFold`.

Cada fold interno ajustará desde cero el preprocesamiento estático, la normalización temporal y los parámetros de la red neuronal.

Se utilizará la arquitectura definida en la Actividad 3.19, conservando la tasa de aprendizaje, dropout, optimizador y demás parámetros.

El máximo de épocas se ampliará de 20 a 30 para permitir estudiar la convergencia que no se alcanzó completamente durante el piloto.

La duración final del entrenamiento será la mediana de las épocas con menor pérdida de validación interna.

#### Reentrenamiento y evaluación

Una vez seleccionada la duración del entrenamiento, se ajustará una nueva instancia del Transformer utilizando todo el conjunto de entrenamiento externo.

El fold externo de validación se evaluará una sola vez y no participará en ninguna decisión de ajuste.

Se reportarán F1 macro, recall adverso, precisión adversa, ROC-AUC, accuracy y matriz de confusión.

#### Comparación

Los resultados se compararán descriptivamente con Regresión Logística B y Random Forest B sobre exactamente el mismo fold externo.

Esta comparación corresponde a una evaluación externa preliminar. No sustituye la evaluación repetida de todos los folds.

#### Limitaciones

La duración seleccionada y los resultados pueden depender de las semillas utilizadas durante el entrenamiento neuronal.

Una sola partición externa no permite establecer superioridad estadística.

Los resultados externos no deberán utilizarse para rediseñar la arquitectura ni optimizar sus hiperparámetros.

El conjunto de prueba independiente continuará reservado.


In [ ]:

# ============================================================
# ACTIVIDAD 3.20 - PRIMER FOLD EXTERNO DEL TRANSFORMER C
# SEMANA 8 - VALIDACIÓN INTERNA EN 3 FOLDS
# ============================================================

# Asistencia IA: ChatGPT, octubre de 2026.
# Prompt:
# "Evaluar Transformer C sobre el primer fold externo
# de la semana 8 con selección de épocas mediante
# tres folds internos agrupados por estudiante.
# Reentrenar desde cero sobre el entrenamiento externo
# y comparar con LR B y RF B en el mismo fold.
# Evitar toda fuga de información y reservar el test."

import copy
import random
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn

from torch.utils.data import TensorDataset, DataLoader
from sklearn.base import clone
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    accuracy_score,
    roc_auc_score,
    confusion_matrix
)

from IPython.display import display

# ------------------------------------------------------------
# 1. CONFIGURACIÓN
# ------------------------------------------------------------

SEMANA_320 = 8
SEMILLA_320 = 42
MAX_EPOCAS_320 = 30
PACIENCIA_320 = 6
BATCH_320 = 256
LR_320 = 0.0007
WD_320 = 0.001

dispositivo_320 = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

assert "TransformerOULAD" in globals()
assert "preparar_temporales_319" in globals()
assert "preprocesador_estatico_319" in globals()

datos_320 = secuencias_oulad_C[SEMANA_320]

X_seq_320 = datos_320["X_secuencia_train"]
X_est_320 = (
    datos_320["X_estatica_train"]
    .reset_index(drop=True)
)

y_320 = (
    datos_320["y_train"]
    .reset_index(drop=True)
)

grupos_320 = (
    datos_320["grupos_train"]
    .reset_index(drop=True)
)

claves_320 = (
    datos_320["claves_train"]
    .reset_index(drop=True)
)

idx_ext_train_320, idx_ext_val_320 = (
    cv_splits_oulad[SEMANA_320][0]
)

assert set(
    grupos_320.iloc[idx_ext_train_320]
).isdisjoint(
    set(grupos_320.iloc[idx_ext_val_320])
)

print("=== 3.20.1 CONFIGURACIÓN ===")
print("Dispositivo:", dispositivo_320)
print("Semana:", SEMANA_320)
print("Fold externo: R1-F1")
print("Entrenamiento externo:", len(idx_ext_train_320))
print("Validación externa:", len(idx_ext_val_320))
print("Folds internos:", 3)
print("Máximo de épocas:", MAX_EPOCAS_320)

# ------------------------------------------------------------
# 2. FUNCIONES AUXILIARES
# ------------------------------------------------------------

def fijar_semilla_320(semilla):
    random.seed(semilla)
    np.random.seed(semilla)
    torch.manual_seed(semilla)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(semilla)


def preparar_datos_320(idx_fit, idx_eval):

    # El preprocesador se clona sin estadísticas
    # previamente ajustadas.
    preprocesador = clone(
        preprocesador_estatico_319
    )

    est_train = (
        preprocesador.fit_transform(
            X_est_320.iloc[idx_fit]
        ).astype(np.float32)
    )

    est_eval = (
        preprocesador.transform(
            X_est_320.iloc[idx_eval]
        ).astype(np.float32)
    )

    (
        seq_train,
        seq_eval,
        mask_train,
        mask_eval,
        escalador_temporal
    ) = preparar_temporales_319(
        X_seq_320[idx_fit],
        X_seq_320[idx_eval]
    )

    assert est_train.shape[1] == est_eval.shape[1]

    return {
        "est_train": est_train,
        "est_eval": est_eval,
        "seq_train": seq_train,
        "seq_eval": seq_eval,
        "mask_train": mask_train,
        "mask_eval": mask_eval,
        "preprocesador": preprocesador,
        "escalador": escalador_temporal
    }


def construir_loader_320(
    secuencias,
    estaticas,
    mascara,
    etiquetas,
    shuffle=False,
    semilla=42
):

    dataset = TensorDataset(
        torch.tensor(
            secuencias, dtype=torch.float32
        ),
        torch.tensor(
            estaticas, dtype=torch.float32
        ),
        torch.tensor(
            mascara, dtype=torch.bool
        ),
        torch.tensor(
            etiquetas, dtype=torch.float32
        )
    )

    generador = torch.Generator()
    generador.manual_seed(semilla)

    return DataLoader(
        dataset,
        batch_size=BATCH_320,
        shuffle=shuffle,
        generator=generador,
        num_workers=0
    )


def nuevo_transformer_320(n_estaticas, semilla):

    fijar_semilla_320(semilla)

    return TransformerOULAD(
        n_static=n_estaticas,
        n_weeks=SEMANA_320,
        n_channels=4,
        d_model=32,
        n_heads=4,
        n_layers=2,
        dropout=0.15
    ).to(dispositivo_320)


@torch.no_grad()
def predecir_loader_320(modelo, loader):

    modelo.eval()

    probas = []
    reales = []
    perdida_acumulada = 0.0
    cantidad = 0

    criterio = nn.BCEWithLogitsLoss()

    for seq, est, mask, y_batch in loader:

        seq = seq.to(dispositivo_320)
        est = est.to(dispositivo_320)
        mask = mask.to(dispositivo_320)
        y_batch = y_batch.to(dispositivo_320)

        logits = modelo(seq, est, mask)

        loss = criterio(logits, y_batch)

        perdida_acumulada += (
            loss.item() * len(y_batch)
        )

        cantidad += len(y_batch)

        probas.append(
            torch.sigmoid(logits)
            .cpu().numpy()
        )

        reales.append(
            y_batch.cpu().numpy()
        )

    return {
        "loss": perdida_acumulada / cantidad,
        "prob": np.concatenate(probas),
        "y": np.concatenate(reales).astype(int)
    }


def entrenar_transformer_320(
    modelo,
    loader_train,
    loader_val=None,
    max_epocas=30,
    paciencia=6
):

    optimizador = torch.optim.AdamW(
        modelo.parameters(),
        lr=LR_320,
        weight_decay=WD_320
    )

    criterio = nn.BCEWithLogitsLoss()

    mejor_loss = np.inf
    mejor_epoca = 0
    mejor_estado = None
    sin_mejora = 0

    historia = []

    for epoca in range(1, max_epocas + 1):

        modelo.train()

        suma_loss = 0.0
        n_total = 0

        for seq, est, mask, y_batch in loader_train:

            seq = seq.to(dispositivo_320)
            est = est.to(dispositivo_320)
            mask = mask.to(dispositivo_320)
            y_batch = y_batch.to(dispositivo_320)

            optimizador.zero_grad(
                set_to_none=True
            )

            logits = modelo(seq, est, mask)

            loss = criterio(logits, y_batch)

            loss.backward()

            nn.utils.clip_grad_norm_(
                modelo.parameters(),
                max_norm=1.0
            )

            optimizador.step()

            suma_loss += loss.item() * len(y_batch)
            n_total += len(y_batch)

        loss_train = suma_loss / n_total

        fila = {
            "Epoca": epoca,
            "Loss_train": loss_train
        }

        if loader_val is not None:

            evaluacion = predecir_loader_320(
                modelo, loader_val
            )

            loss_val = evaluacion["loss"]

            fila["Loss_validacion"] = loss_val

            if loss_val < mejor_loss - 1e-4:

                mejor_loss = loss_val
                mejor_epoca = epoca

                mejor_estado = copy.deepcopy(
                    modelo.state_dict()
                )

                sin_mejora = 0

            else:

                sin_mejora += 1

        historia.append(fila)

        if loader_val is not None:

            if (
                epoca == 1
                or epoca % 5 == 0
                or epoca == max_epocas
            ):

                print(
                    f"    Época {epoca:02d}: "
                    f"train={loss_train:.4f} | "
                    f"val={loss_val:.4f}",
                    flush=True
                )

            if sin_mejora >= paciencia:
                print(
                    f"    Early stopping: época {epoca}",
                    flush=True
                )
                break

    if loader_val is not None:

        assert mejor_estado is not None

        modelo.load_state_dict(
            mejor_estado
        )

    return {
        "modelo": modelo,
        "historia": pd.DataFrame(historia),
        "mejor_epoca": mejor_epoca,
        "mejor_loss": mejor_loss
    }


# ------------------------------------------------------------
# 3. CONSTRUIR LOS FOLDS INTERNOS
# ------------------------------------------------------------

print("\n=== 3.20.2 SELECCIÓN INTERNA DE ÉPOCAS ===")

cv_interna_320 = StratifiedGroupKFold(
    n_splits=3,
    shuffle=True,
    random_state=137
)

splits_internos_320 = list(
    cv_interna_320.split(
        X_est_320.iloc[idx_ext_train_320],
        y_320.iloc[idx_ext_train_320],
        groups=grupos_320.iloc[idx_ext_train_320]
    )
)

tabla_epocas_320 = []
historias_internas_320 = []

inicio_total_320 = time.perf_counter()

for numero_fold, (
    idx_train_rel,
    idx_val_rel
) in enumerate(
    splits_internos_320,
    start=1
):

    idx_train = idx_ext_train_320[
        idx_train_rel
    ]

    idx_val = idx_ext_train_320[
        idx_val_rel
    ]

    assert set(
        grupos_320.iloc[idx_train]
    ).isdisjoint(
        set(grupos_320.iloc[idx_val])
    )

    print(
        f"\nFold interno {numero_fold}/3",
        flush=True
    )

    preparados = preparar_datos_320(
        idx_train, idx_val
    )

    loader_train = construir_loader_320(
        preparados["seq_train"],
        preparados["est_train"],
        preparados["mask_train"],
        y_320.iloc[idx_train].to_numpy(),
        shuffle=True,
        semilla=SEMILLA_320 + numero_fold
    )

    loader_val = construir_loader_320(
        preparados["seq_eval"],
        preparados["est_eval"],
        preparados["mask_eval"],
        y_320.iloc[idx_val].to_numpy()
    )

    modelo = nuevo_transformer_320(
        preparados["est_train"].shape[1],
        semilla=SEMILLA_320 + numero_fold
    )

    entrenado = entrenar_transformer_320(
        modelo,
        loader_train,
        loader_val=loader_val,
        max_epocas=MAX_EPOCAS_320,
        paciencia=PACIENCIA_320
    )

    predicciones_internas = predecir_loader_320(
        entrenado["modelo"],
        loader_val
    )

    y_real = predicciones_internas["y"]
    prob = predicciones_internas["prob"]

    pred = (prob >= 0.5).astype(int)

    tabla_epocas_320.append({
        "Fold_interno": numero_fold,
        "Matriculas_train": len(idx_train),
        "Matriculas_val": len(idx_val),
        "Mejor_epoca": entrenado["mejor_epoca"],
        "Loss_validacion": entrenado["mejor_loss"],
        "F1_macro": f1_score(
            y_real, pred,
            average="macro",
            zero_division=0
        ),
        "ROC_AUC": roc_auc_score(
            y_real, prob
        )
    })

    historia = entrenado["historia"].copy()
    historia["Fold_interno"] = numero_fold

    historias_internas_320.append(historia)

tabla_epocas_320 = pd.DataFrame(
    tabla_epocas_320
)

display(tabla_epocas_320.round(4))

# ------------------------------------------------------------
# 4. ELEGIR DURACIÓN SIN CONSULTAR VALIDACIÓN EXTERNA
# ------------------------------------------------------------

EPOCAS_FINALES_320 = int(
    np.median(
        tabla_epocas_320["Mejor_epoca"]
    )
)

assert 1 <= EPOCAS_FINALES_320 <= MAX_EPOCAS_320

print("\n=== 3.20.3 ÉPOCAS SELECCIONADAS ===")

print(
    "Épocas óptimas internas:",
    tabla_epocas_320["Mejor_epoca"].tolist()
)

print(
    "Duración final seleccionada:",
    EPOCAS_FINALES_320
)

if (
    tabla_epocas_320["Mejor_epoca"]
    == MAX_EPOCAS_320
).sum() >= 2:

    print(
        "ADVERTENCIA: la mayoría de folds internos "
        "alcanzó el máximo de épocas."
    )
    print(
        "Esto indica una limitación del presupuesto "
        "de entrenamiento; la duración seleccionada "
        "no necesariamente es óptima."
    )

# ------------------------------------------------------------
# 5. REENTRENAR SOBRE TODO EL ENTRENAMIENTO EXTERNO
# ------------------------------------------------------------

print("\n=== 3.20.4 REENTRENAMIENTO EXTERNO ===")

preparados_finales_320 = preparar_datos_320(
    idx_ext_train_320,
    idx_ext_val_320
)

# El preprocesamiento se ajusta únicamente al
# entrenamiento externo. La validación externa
# no aporta estadísticas ni parámetros.

loader_train_final = construir_loader_320(
    preparados_finales_320["seq_train"],
    preparados_finales_320["est_train"],
    preparados_finales_320["mask_train"],
    y_320.iloc[idx_ext_train_320].to_numpy(),
    shuffle=True,
    semilla=SEMILLA_320 + 100
)

loader_val_externo = construir_loader_320(
    preparados_finales_320["seq_eval"],
    preparados_finales_320["est_eval"],
    preparados_finales_320["mask_eval"],
    y_320.iloc[idx_ext_val_320].to_numpy()
)

modelo_final_320 = nuevo_transformer_320(
    preparados_finales_320["est_train"].shape[1],
    semilla=SEMILLA_320 + 100
)

entrenamiento_final_320 = entrenar_transformer_320(
    modelo_final_320,
    loader_train_final,
    loader_val=None,
    max_epocas=EPOCAS_FINALES_320
)

print(
    "Entrenamiento final completado:",
    EPOCAS_FINALES_320,
    "épocas"
)

# ------------------------------------------------------------
# 6. EVALUAR UNA SOLA VEZ EL FOLD EXTERNO
# ------------------------------------------------------------

print("\n=== 3.20.5 EVALUACIÓN EXTERNA ===")

evaluacion_ext_320 = predecir_loader_320(
    modelo_final_320,
    loader_val_externo
)

y_real_320 = evaluacion_ext_320["y"]
prob_320 = evaluacion_ext_320["prob"]
pred_320 = (prob_320 >= 0.5).astype(int)

vn, fp, fn, vp = confusion_matrix(
    y_real_320, pred_320,
    labels=[0, 1]
).ravel()

metricas_C_320 = {
    "Algoritmo": "Transformer C",
    "F1_macro": f1_score(
        y_real_320, pred_320,
        average="macro",
        zero_division=0
    ),
    "Recall_adverso": recall_score(
        y_real_320, pred_320,
        zero_division=0
    ),
    "Precision_adverso": precision_score(
        y_real_320, pred_320,
        zero_division=0
    ),
    "ROC_AUC": roc_auc_score(
        y_real_320, prob_320
    ),
    "Accuracy": accuracy_score(
        y_real_320, pred_320
    ),
    "VN": int(vn),
    "FP": int(fp),
    "FN": int(fn),
    "VP": int(vp)
}

display(
    pd.DataFrame([metricas_C_320]).round(4)
)

# ------------------------------------------------------------
# 7. COMPARAR CON LR B Y RF B EN EL MISMO FOLD
# ------------------------------------------------------------

print("\n=== 3.20.6 COMPARACIÓN EN R1-F1 ===")

comparaciones_320 = []

for algoritmo in [
    "LogisticRegression",
    "RandomForest"
]:

    tabla = resultados_cv_ventanas[
        (8, algoritmo)
    ]["resultados"]

    fila = tabla.loc[
        tabla["Repeticion"].eq(1)
        & tabla["Fold"].eq(1)
        & tabla["Experimento"].eq("B")
    ]

    assert len(fila) == 1

    comparaciones_320.append({
        "Modelo": (
            "LR B"
            if algoritmo == "LogisticRegression"
            else "RF B"
        ),
        "F1_macro": fila["F1_macro"].iloc[0],
        "Recall_adverso": (
            fila["Recall_adverso"].iloc[0]
        ),
        "ROC_AUC": fila["ROC_AUC"].iloc[0],
        "Accuracy": fila["Accuracy"].iloc[0]
    })

comparaciones_320.append({
    "Modelo": "Transformer C",
    "F1_macro": metricas_C_320["F1_macro"],
    "Recall_adverso": metricas_C_320["Recall_adverso"],
    "ROC_AUC": metricas_C_320["ROC_AUC"],
    "Accuracy": metricas_C_320["Accuracy"]
})

tabla_comparativa_C_320 = pd.DataFrame(
    comparaciones_320
)

display(tabla_comparativa_C_320.round(4))

print(
    "\nComparación preliminar: un único fold externo."
)
print(
    "No utilizar estos resultados para cambiar "
    "la arquitectura del Transformer."
)

# ------------------------------------------------------------
# 8. GUARDAR RESULTADOS Y PREDICCIONES
# ------------------------------------------------------------

predicciones_C_320 = (
    claves_320.iloc[idx_ext_val_320]
    .reset_index(drop=True)
    .copy()
)

predicciones_C_320["Semana"] = 8
predicciones_C_320["Repeticion"] = 1
predicciones_C_320["Fold"] = 1
predicciones_C_320["Experimento"] = "C"
predicciones_C_320["Algoritmo"] = "Transformer"
predicciones_C_320["y_real"] = y_real_320
predicciones_C_320["y_pred"] = pred_320
predicciones_C_320["prob_adverso"] = prob_320

assert not predicciones_C_320.duplicated(
    subset=[
        "code_module",
        "code_presentation",
        "id_student"
    ]
).any()

resultado_transformer_C_320 = {
    "epocas_internas": tabla_epocas_320,
    "epocas_finales": EPOCAS_FINALES_320,
    "metricas": metricas_C_320,
    "predicciones": predicciones_C_320,
    "comparacion": tabla_comparativa_C_320,
    "historias_internas": pd.concat(
        historias_internas_320,
        ignore_index=True
    ),
    "historia_final": entrenamiento_final_320[
        "historia"
    ]
}

# ------------------------------------------------------------
# 9. GRÁFICO DE VALIDACIÓN INTERNA
# ------------------------------------------------------------

print("\n=== 3.20.7 GRÁFICO DE PÉRDIDA INTERNA ===")

plt.figure(figsize=(10, 5))

for fold, grupo in (
    resultado_transformer_C_320[
        "historias_internas"
    ].groupby("Fold_interno")
):

    plt.plot(
        grupo["Epoca"],
        grupo["Loss_validacion"],
        marker=".",
        label=f"Fold interno {fold}"
    )

plt.title(
    "Transformer C - Selección de épocas mediante "
    "validación interna"
)

plt.xlabel("Época")
plt.ylabel("Pérdida de validación")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 10. CONTROLES FINALES
# ------------------------------------------------------------

print("\n=== 3.20.8 CONTROLES FINALES ===")

assert len(tabla_epocas_320) == 3

assert len(predicciones_C_320) == len(
    idx_ext_val_320
)

assert predicciones_C_320[
    "prob_adverso"
].between(0, 1).all()

assert (
    metricas_C_320["VN"]
    + metricas_C_320["FP"]
    + metricas_C_320["FN"]
    + metricas_C_320["VP"]
) == len(idx_ext_val_320)

assert set(
    grupos_320.iloc[idx_ext_train_320]
).isdisjoint(
    set(grupos_320.iloc[idx_ext_val_320])
)

print("Folds internos evaluados: 3")
print("Duración elegida solo con validación interna.")
print("Transformaciones ajustadas solo en entrenamiento.")
print("Fold externo evaluado: R1-F1, semana 8.")
print("Predicciones guardadas correctamente.")
print("Conjunto de prueba independiente: NO utilizado.")

print(
    "Tiempo total:",
    round(
        time.perf_counter() - inicio_total_320,
        1
    ),
    "segundos"
)

print("\n=== ACTIVIDAD 3.20 FINALIZADA ===")



### 3.20.9. Resultados e interpretación

Se completó la primera evaluación externa del Experimento C utilizando un Transformer Encoder con características estáticas y secuencias semanales del comportamiento estudiantil.

La evaluación se realizó sobre el primer fold externo de la semana 8, con 16.898 matrículas de entrenamiento y 4.266 de validación externa.

La duración del entrenamiento se seleccionó mediante tres folds internos agrupados por estudiante, ajustando el preprocesamiento y la normalización exclusivamente con las matrículas de entrenamiento correspondientes a cada fold.

#### Selección de épocas

Las mejores épocas internas fueron 26, 22 y 16, con pérdidas de validación de 0,5423, 0,5198 y 0,5292.

Se seleccionó una duración final de 22 épocas mediante la mediana de las épocas óptimas internas.

A diferencia del piloto anterior, ninguno de los tres folds internos seleccionó el límite máximo de 30 épocas, lo que permitió identificar un rango de estabilización dentro del presupuesto de entrenamiento establecido.

#### Evaluación externa

El Transformer obtuvo los siguientes resultados:

| Métrica | Resultado |
|---|---:|
| F1 macro | 0,6946 |
| Recall adverso | 0,4980 |
| Precisión adversa | 0,7581 |
| ROC-AUC | 0,7898 |
| Accuracy | 0,7232 |

La matriz de confusión presentó 2.195 verdaderos negativos, 284 falsos positivos, 897 falsos negativos y 890 verdaderos positivos.

El modelo mostró una precisión relativamente elevada entre las alertas emitidas, aunque identificó menos de la mitad de los casos adversos adicionales necesarios para alcanzar una sensibilidad superior al 50 %.

#### Comparación con los modelos clásicos

Sobre el mismo fold externo:

| Modelo | F1 macro | Recall adverso | ROC-AUC |
|---|---:|---:|---:|
| Regresión Logística B | 0,7206 | 0,5904 | 0,8001 |
| Random Forest B | 0,6868 | 0,4952 | 0,7802 |
| Transformer C | 0,6946 | 0,4980 | 0,7898 |

El Transformer superó ligeramente a Random Forest B en F1 macro y ROC-AUC, pero obtuvo un rendimiento inferior al de Regresión Logística B.

Estos resultados no permiten establecer superioridad estadística entre arquitecturas, ya que corresponden a una única partición externa.

#### Consideraciones metodológicas

La selección de épocas se realizó sin utilizar la validación externa y se mantuvo el umbral de clasificación de 0,5.

El Transformer utiliza secuencias temporales semanales y características estáticas, mientras que los modelos B utilizan características longitudinales diseñadas explícitamente.

Por tanto, sus diferencias de rendimiento deben interpretarse como una comparación entre estrategias de representación y aprendizaje, no como una prueba aislada del efecto de la atención neuronal.

### 3.20.10. Limitaciones

La evaluación corresponde a un único fold externo.

La arquitectura neuronal se mantiene fija tras el piloto técnico, pero todavía no se ha estimado la variabilidad entre folds y repeticiones.

El criterio de selección de épocas minimiza la pérdida de validación interna, mientras que los modelos clásicos optimizaron F1 macro mediante búsqueda de hiperparámetros. Esta diferencia debe considerarse al interpretar la comparación.

No se ha utilizado el conjunto de prueba independiente.

### 3.20.11. Conclusión

La evaluación confirmó que el Transformer C puede generalizar a matrículas de estudiantes no utilizadas durante el entrenamiento.

El modelo obtuvo un F1 macro externo de 0,6946 y un ROC-AUC de 0,7898, superando ligeramente a Random Forest B, pero sin alcanzar el rendimiento de Regresión Logística B en el mismo fold.

**Conclusión principal:** el aprendizaje secuencial mediante Transformer constituye una alternativa viable para representar el comportamiento estudiantil, aunque los resultados iniciales no demuestran una ventaja frente a la regresión logística con características longitudinales agregadas. Es necesario completar una evaluación externa repetida antes de establecer conclusiones comparativas.



### 3.21. Evaluación externa repetida del Transformer C — Semana 8

#### Objetivo

Estimar el rendimiento y la variabilidad del Transformer C mediante validación cruzada externa agrupada por estudiante, manteniendo el protocolo de selección interna de épocas establecido en la Actividad 3.20.

#### Protocolo

Se utilizarán diez particiones externas correspondientes a cinco folds y dos repeticiones de la semana 8.

En cada partición externa:

1. Se dividirá el entrenamiento externo en tres folds internos agrupados por estudiante.
2. Se ajustarán desde cero el preprocesamiento estático, la normalización temporal y los parámetros del Transformer para cada fold interno.
3. Se seleccionará la mejor época de cada fold interno utilizando la pérdida de validación.
4. Se calculará la mediana de las tres mejores épocas.
5. Se entrenará una nueva instancia del Transformer sobre todo el entrenamiento externo durante esa cantidad de épocas.
6. Se obtendrán predicciones sobre la validación externa, sin utilizarla para modificar los hiperparámetros.

#### Configuración congelada

Se conservará la arquitectura y los hiperparámetros empleados en la Actividad 3.20.

No se realizará una búsqueda adicional de configuraciones basada en los resultados externos.

#### Comparación

Se compararán F1 macro, recall adverso, ROC-AUC y accuracy frente a Regresión Logística B y Random Forest B, utilizando las mismas particiones externas.

El resultado R1-F1, previamente obtenido en la Actividad 3.20, será reutilizado sin reentrenamiento.

#### Ejecución incremental

Para reducir el riesgo de perder resultados por interrupciones del entorno de Google Colab, se almacenará un archivo de control después de cada fold.

La ejecución podrá reanudarse mientras los resultados permanezcan disponibles.

Los resultados agregados se considerarán completos únicamente después de evaluar los diez folds.

#### Limitaciones

La arquitectura fue desarrollada mediante una fase piloto, por lo que los resultados externos deben interpretarse como evaluación de desarrollo y no como validación confirmatoria completamente independiente.

Las particiones repetidas comparten matrículas entre conjuntos de entrenamiento y no constituyen evaluaciones estadísticamente independientes.

El conjunto de prueba independiente continuará reservado.


In [ ]:

# ============================================================
# PROYECTO FINAL - ACTIVIDAD 3.21
# TRANSFORMER C - VALIDACIÓN EXTERNA REPETIDA
# SEMANA 8 - EJECUCIÓN REANUDABLE
# ============================================================

# Asistencia IA: ChatGPT, octubre de 2026.
# Prompt:
# "Completar los 10 folds externos del Transformer C
# en semana 8, reutilizando el piloto R1-F1 y todas
# las funciones de la Actividad 3.20.
# Seleccionar épocas con 3 folds internos agrupados.
# Guardar resultados incrementalmente y comparar
# con LR B y RF B en las mismas particiones.
# No utilizar el conjunto de prueba independiente."

import os
import pickle
import time
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    f1_score,
    recall_score,
    precision_score,
    accuracy_score,
    roc_auc_score,
    confusion_matrix
)

from IPython.display import display

# ------------------------------------------------------------
# 1. CONFIGURACIÓN
# ------------------------------------------------------------

SEMANA_321 = 8

# Ajustar a 1 si se desea ejecutar un fold por vez.
# Ajustar a 9 para procesar todos los pendientes.
NUEVOS_FOLDS_POR_EJECUCION = 2

RUTA_CHECKPOINT_321 = (
    "/content/transformer_C_semana8_checkpoint.pkl"
)

METRICAS_321 = [
    "F1_macro",
    "Recall_adverso",
    "Precision_adverso",
    "ROC_AUC",
    "Accuracy"
]

FUNCIONES_REQUERIDAS_321 = [
    "preparar_datos_320",
    "construir_loader_320",
    "nuevo_transformer_320",
    "entrenar_transformer_320",
    "predecir_loader_320"
]

for nombre in FUNCIONES_REQUERIDAS_321:
    assert nombre in globals(), (
        f"Falta {nombre}. Ejecuta primero "
        "la Actividad 3.20."
    )

assert MAX_EPOCAS_320 == 30
assert PACIENCIA_320 == 6
assert SEMANA_320 == 8
assert len(cv_splits_oulad[8]) == 10

print("=== 3.21.1 CONFIGURACIÓN ===")
print("Semana:", SEMANA_321)
print("Folds externos:", 10)
print("Folds internos:", 3)
print("Máximo de épocas:", MAX_EPOCAS_320)
print("Paciencia:", PACIENCIA_320)
print(
    "Folds nuevos por ejecución:",
    NUEVOS_FOLDS_POR_EJECUCION
)

# ------------------------------------------------------------
# 2. RECUPERAR RESULTADOS EXISTENTES
# ------------------------------------------------------------

if (
    "resultados_transformer_C_s8" not in globals()
):

    if os.path.exists(RUTA_CHECKPOINT_321):

        with open(
            RUTA_CHECKPOINT_321, "rb"
        ) as archivo:

            resultados_transformer_C_s8 = (
                pickle.load(archivo)
            )

        print("Checkpoint anterior recuperado.")

    else:

        resultados_transformer_C_s8 = {}

# Reutilizar el fold externo de la Actividad 3.20.
if (1, 1) not in resultados_transformer_C_s8:

    assert "resultado_transformer_C_320" in globals()
    assert "predicciones_C_320" in globals()

    resumen_piloto = {
        "Semana": 8,
        "Repeticion": 1,
        "Fold": 1,
        "Epocas_finales": (
            resultado_transformer_C_320[
                "epocas_finales"
            ]
        ),
        "N_validacion": len(predicciones_C_320)
    }

    for metrica in METRICAS_321:
        resumen_piloto[metrica] = (
            resultado_transformer_C_320[
                "metricas"
            ][metrica]
        )

    for metrica in ["VN", "FP", "FN", "VP"]:
        resumen_piloto[metrica] = (
            resultado_transformer_C_320[
                "metricas"
            ][metrica]
        )

    resultados_transformer_C_s8[(1, 1)] = {
        "resumen": resumen_piloto,
        "predicciones": predicciones_C_320.copy(),
        "epocas_internas": (
            resultado_transformer_C_320[
                "epocas_internas"
            ].copy()
        )
    }

    print("R1-F1 recuperado de la Actividad 3.20.")

# ------------------------------------------------------------
# 3. FUNCIÓN DE ENTRENAMIENTO DE UN FOLD EXTERNO
# ------------------------------------------------------------

def ejecutar_fold_transformer_321(numero_split):

    repeticion = numero_split // 5 + 1
    fold = numero_split % 5 + 1

    idx_ext_train, idx_ext_val = (
        cv_splits_oulad[8][numero_split]
    )

    assert set(
        grupos_320.iloc[idx_ext_train]
    ).isdisjoint(
        set(grupos_320.iloc[idx_ext_val])
    )

    print(
        f"\n=== SEMANA 8 | R{repeticion}-F{fold} ===",
        flush=True
    )

    inicio = time.perf_counter()

    # --------------------------------------------------------
    # 3.1. DEFINIR LOS TRES FOLDS INTERNOS
    # --------------------------------------------------------

    cv_interna = StratifiedGroupKFold(
        n_splits=3,
        shuffle=True,
        random_state=137 + numero_split
    )

    splits_internos = list(
        cv_interna.split(
            X_est_320.iloc[idx_ext_train],
            y_320.iloc[idx_ext_train],
            groups=grupos_320.iloc[idx_ext_train]
        )
    )

    epocas_internas = []

    for f_interno, (
        idx_train_rel,
        idx_val_rel
    ) in enumerate(
        splits_internos,
        start=1
    ):

        idx_train = idx_ext_train[idx_train_rel]
        idx_val = idx_ext_train[idx_val_rel]

        assert set(
            grupos_320.iloc[idx_train]
        ).isdisjoint(
            set(grupos_320.iloc[idx_val])
        )

        datos = preparar_datos_320(
            idx_train,
            idx_val
        )

        semilla = (
            SEMILLA_320
            + 1000 * numero_split
            + f_interno
        )

        loader_train = construir_loader_320(
            datos["seq_train"],
            datos["est_train"],
            datos["mask_train"],
            y_320.iloc[idx_train].to_numpy(),
            shuffle=True,
            semilla=semilla
        )

        loader_val = construir_loader_320(
            datos["seq_eval"],
            datos["est_eval"],
            datos["mask_eval"],
            y_320.iloc[idx_val].to_numpy()
        )

        modelo = nuevo_transformer_320(
            datos["est_train"].shape[1],
            semilla=semilla
        )

        ajustado = entrenar_transformer_320(
            modelo,
            loader_train,
            loader_val=loader_val,
            max_epocas=MAX_EPOCAS_320,
            paciencia=PACIENCIA_320
        )

        evaluacion_interna = predecir_loader_320(
            ajustado["modelo"],
            loader_val
        )

        y_interno = evaluacion_interna["y"]
        prob_interno = evaluacion_interna["prob"]

        epocas_internas.append({
            "Fold_interno": f_interno,
            "Mejor_epoca": ajustado["mejor_epoca"],
            "Loss_validacion": ajustado["mejor_loss"],
            "F1_macro": f1_score(
                y_interno,
                (prob_interno >= 0.5).astype(int),
                average="macro",
                zero_division=0
            ),
            "ROC_AUC": roc_auc_score(
                y_interno,
                prob_interno
            )
        })

        print(
            f"Fold interno {f_interno}: "
            f"época={ajustado['mejor_epoca']} | "
            f"loss={ajustado['mejor_loss']:.4f}",
            flush=True
        )

        del modelo, ajustado

    tabla_epocas = pd.DataFrame(
        epocas_internas
    )

    # --------------------------------------------------------
    # 3.2. SELECCIONAR ÉPOCAS
    # --------------------------------------------------------

    epocas_finales = int(
        np.median(
            tabla_epocas["Mejor_epoca"]
        )
    )

    assert 1 <= epocas_finales <= MAX_EPOCAS_320

    print(
        "Épocas internas:",
        tabla_epocas["Mejor_epoca"].tolist()
    )

    print(
        "Épocas finales:",
        epocas_finales
    )

    # --------------------------------------------------------
    # 3.3. REENTRENAMIENTO EXTERNO
    # --------------------------------------------------------

    datos_finales = preparar_datos_320(
        idx_ext_train,
        idx_ext_val
    )

    semilla_final = (
        SEMILLA_320
        + 100
        + 1000 * numero_split
    )

    loader_train_final = construir_loader_320(
        datos_finales["seq_train"],
        datos_finales["est_train"],
        datos_finales["mask_train"],
        y_320.iloc[idx_ext_train].to_numpy(),
        shuffle=True,
        semilla=semilla_final
    )

    loader_val_externo = construir_loader_320(
        datos_finales["seq_eval"],
        datos_finales["est_eval"],
        datos_finales["mask_eval"],
        y_320.iloc[idx_ext_val].to_numpy()
    )

    modelo_final = nuevo_transformer_320(
        datos_finales["est_train"].shape[1],
        semilla=semilla_final
    )

    entrenado_final = entrenar_transformer_320(
        modelo_final,
        loader_train_final,
        loader_val=None,
        max_epocas=epocas_finales
    )

    # --------------------------------------------------------
    # 3.4. UNA SOLA EVALUACIÓN EXTERNA
    # --------------------------------------------------------

    evaluacion = predecir_loader_320(
        entrenado_final["modelo"],
        loader_val_externo
    )

    y_real = evaluacion["y"]
    prob = evaluacion["prob"]
    pred = (prob >= 0.5).astype(int)

    assert np.array_equal(
        y_real,
        y_320.iloc[idx_ext_val].to_numpy()
    )

    vn, fp, fn, vp = confusion_matrix(
        y_real,
        pred,
        labels=[0, 1]
    ).ravel()

    resumen = {
        "Semana": 8,
        "Repeticion": repeticion,
        "Fold": fold,
        "Epocas_finales": epocas_finales,
        "N_validacion": len(y_real),
        "F1_macro": f1_score(
            y_real, pred,
            average="macro",
            zero_division=0
        ),
        "Recall_adverso": recall_score(
            y_real, pred,
            zero_division=0
        ),
        "Precision_adverso": precision_score(
            y_real, pred,
            zero_division=0
        ),
        "ROC_AUC": roc_auc_score(
            y_real, prob
        ),
        "Accuracy": accuracy_score(
            y_real, pred
        ),
        "VN": int(vn),
        "FP": int(fp),
        "FN": int(fn),
        "VP": int(vp),
        "Tiempo_segundos": (
            time.perf_counter() - inicio
        )
    }

    predicciones = (
        claves_320.iloc[idx_ext_val]
        .reset_index(drop=True)
        .copy()
    )

    predicciones["Semana"] = 8
    predicciones["Repeticion"] = repeticion
    predicciones["Fold"] = fold
    predicciones["Experimento"] = "C"
    predicciones["Algoritmo"] = "Transformer"
    predicciones["y_real"] = y_real
    predicciones["y_pred"] = pred
    predicciones["prob_adverso"] = prob

    assert not predicciones.duplicated(
        subset=[
            "code_module",
            "code_presentation",
            "id_student"
        ]
    ).any()

    assert len(predicciones) == len(idx_ext_val)

    print(
        f"F1 externo={resumen['F1_macro']:.4f} | "
        f"Recall={resumen['Recall_adverso']:.4f} | "
        f"AUC={resumen['ROC_AUC']:.4f}",
        flush=True
    )

    return {
        "resumen": resumen,
        "predicciones": predicciones,
        "epocas_internas": tabla_epocas
    }


# ------------------------------------------------------------
# 4. IDENTIFICAR FOLDS PENDIENTES
# ------------------------------------------------------------

pendientes_321 = []

for numero_split in range(10):

    clave = (
        numero_split // 5 + 1,
        numero_split % 5 + 1
    )

    if clave not in resultados_transformer_C_s8:
        pendientes_321.append(numero_split)

print("\n=== 3.21.2 ESTADO DE EJECUCIÓN ===")
print(
    "Folds completados:",
    len(resultados_transformer_C_s8)
)
print("Folds pendientes:", len(pendientes_321))

# ------------------------------------------------------------
# 5. EJECUTAR EL SIGUIENTE BLOQUE
# ------------------------------------------------------------

for numero_split in pendientes_321[
    :NUEVOS_FOLDS_POR_EJECUCION
]:

    repeticion = numero_split // 5 + 1
    fold = numero_split % 5 + 1

    clave = (repeticion, fold)

    resultado = ejecutar_fold_transformer_321(
        numero_split
    )

    resultados_transformer_C_s8[clave] = resultado

    # Guardar inmediatamente al finalizar el fold.
    with open(
        RUTA_CHECKPOINT_321,
        "wb"
    ) as archivo:

        pickle.dump(
            resultados_transformer_C_s8,
            archivo
        )

    # Copiar a Drive inmediatamente después de cerrar el archivo del fold.
    respaldar_checkpoint_oulad(RUTA_CHECKPOINT_321)

    print(
        "Resultado guardado:",
        clave,
        flush=True
    )

# ------------------------------------------------------------
# 6. TABLA DE RESULTADOS ACUMULADOS
# ------------------------------------------------------------

print("\n=== 3.21.3 RESULTADOS DISPONIBLES ===")

tabla_transformer_C_s8 = pd.DataFrame(
    [
        resultado["resumen"]
        for resultado in resultados_transformer_C_s8.values()
    ]
).sort_values(
    ["Repeticion", "Fold"]
).reset_index(drop=True)

display(
    tabla_transformer_C_s8.round(4)
)

# ------------------------------------------------------------
# 7. COMPARACIÓN CON LR B Y RF B
# ------------------------------------------------------------

print("\n=== 3.21.4 COMPARACIÓN PAREADA ===")

comparacion_321 = (
    tabla_transformer_C_s8[
        [
            "Repeticion",
            "Fold",
            "F1_macro",
            "Recall_adverso",
            "ROC_AUC"
        ]
    ].rename(
        columns={
            "F1_macro": "F1_C",
            "Recall_adverso": "Recall_C",
            "ROC_AUC": "AUC_C"
        }
    )
)

for algoritmo, sufijo in [
    ("LogisticRegression", "LR_B"),
    ("RandomForest", "RF_B")
]:

    clasico = resultados_cv_ventanas[
        (8, algoritmo)
    ]["resultados"]

    clasico = clasico.loc[
        clasico["Experimento"] == "B",
        [
            "Repeticion",
            "Fold",
            "F1_macro",
            "Recall_adverso",
            "ROC_AUC"
        ]
    ].rename(
        columns={
            "F1_macro": f"F1_{sufijo}",
            "Recall_adverso": f"Recall_{sufijo}",
            "ROC_AUC": f"AUC_{sufijo}"
        }
    )

    comparacion_321 = comparacion_321.merge(
        clasico,
        on=["Repeticion", "Fold"],
        validate="one_to_one"
    )

comparacion_321["Delta_F1_C_menos_LR_B"] = (
    comparacion_321["F1_C"]
    - comparacion_321["F1_LR_B"]
)

comparacion_321["Delta_F1_C_menos_RF_B"] = (
    comparacion_321["F1_C"]
    - comparacion_321["F1_RF_B"]
)

display(comparacion_321.round(4))

# ------------------------------------------------------------
# 8. RESUMEN PROVISIONAL O FINAL
# ------------------------------------------------------------

print("\n=== 3.21.5 RESUMEN ===")

print(
    "Folds evaluados:",
    len(tabla_transformer_C_s8),
    "de 10"
)

print(
    "F1 medio disponible:",
    round(
        tabla_transformer_C_s8[
            "F1_macro"
        ].mean(),
        4
    )
)

print(
    "ROC-AUC medio disponible:",
    round(
        tabla_transformer_C_s8[
            "ROC_AUC"
        ].mean(),
        4
    )
)

if len(tabla_transformer_C_s8) == 10:

    print(
        "\nVALIDACIÓN EXTERNA REPETIDA COMPLETA."
    )

    print(
        "Se pueden calcular comparaciones "
        "pareadas de los 10 folds."
    )

else:

    print(
        "\nRESULTADOS PARCIALES."
    )

    print(
        "No deben interpretarse todavía como "
        "el rendimiento medio de la semana 8."
    )

    print(
        "Vuelve a ejecutar esta misma celda "
        "para procesar los siguientes folds."
    )

# ------------------------------------------------------------
# 9. CONTROLES
# ------------------------------------------------------------

print("\n=== 3.21.6 CONTROLES ===")

assert len(resultados_transformer_C_s8) <= 10

assert not tabla_transformer_C_s8.duplicated(
    subset=["Repeticion", "Fold"]
).any()

for resultado in resultados_transformer_C_s8.values():

    assert resultado["predicciones"][
        "prob_adverso"
    ].between(0, 1).all()

    assert not resultado["predicciones"].duplicated(
        subset=[
            "code_module",
            "code_presentation",
            "id_student"
        ]
    ).any()

print("Particiones agrupadas por estudiante.")
print("Preprocesamiento ajustado dentro de cada fold.")
print("Selección de épocas exclusivamente interna.")
print("Arquitectura e hiperparámetros congelados.")
print("Conjunto de prueba independiente: NO utilizado.")

print("\nCheckpoint:", RUTA_CHECKPOINT_321)
print("\n=== EJECUCIÓN DE ACTIVIDAD 3.21 FINALIZADA ===")



### 3.21.7. Seguimiento parcial de la validación externa

Se completaron tres de las diez particiones externas previstas para el Transformer C en la semana 8.

La primera partición se recuperó de la Actividad 3.20, mientras que las dos siguientes fueron entrenadas y almacenadas mediante el procedimiento incremental.

**Resultados preliminares**

| Modelo | F1 macro medio | Recall adverso medio | ROC-AUC medio |
|---|---:|---:|---:|
| Regresión Logística B | 0,7257 | 0,5938 | 0,8059 |
| Transformer C | 0,7111 | 0,5576 | 0,7980 |
| Random Forest B | 0,6996 | 0,5207 | 0,7898 |

En los tres folds disponibles, el Transformer obtuvo un F1 macro superior al de Random Forest B e inferior al de Regresión Logística B.

La duración final de entrenamiento se seleccionó mediante tres folds internos agrupados por estudiante, obteniéndose 22, 21 y 19 épocas para las primeras tres particiones externas.

Los resultados son preliminares y no deben interpretarse como evidencia de superioridad estadística ni como rendimiento definitivo del Experimento C.

El procedimiento conserva la independencia entre entrenamiento y validación por estudiante y no utiliza el conjunto de prueba independiente.

**Estado:** tres folds externos completados; siete pendientes. Se ejecutará el mismo codigo 4 veces más para completar los 10 Folds.


In [ ]:

# ============================================================
# PROYECTO FINAL - ACTIVIDAD 3.21
# TRANSFORMER C - VALIDACIÓN EXTERNA REPETIDA
# SEMANA 8 - EJECUCIÓN REANUDABLE
# ============================================================

# Asistencia IA: ChatGPT, octubre de 2026.
# Prompt:
# "Completar los 10 folds externos del Transformer C
# en semana 8, reutilizando el piloto R1-F1 y todas
# las funciones de la Actividad 3.20.
# Seleccionar épocas con 3 folds internos agrupados.
# Guardar resultados incrementalmente y comparar
# con LR B y RF B en las mismas particiones.
# No utilizar el conjunto de prueba independiente."

import os
import pickle
import time
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    f1_score,
    recall_score,
    precision_score,
    accuracy_score,
    roc_auc_score,
    confusion_matrix
)

from IPython.display import display

# ------------------------------------------------------------
# 1. CONFIGURACIÓN
# ------------------------------------------------------------

SEMANA_321 = 8

# Ajustar a 1 si se desea ejecutar un fold por vez.
# Ajustar a 9 para procesar todos los pendientes.
NUEVOS_FOLDS_POR_EJECUCION = 2

RUTA_CHECKPOINT_321 = (
    "/content/transformer_C_semana8_checkpoint.pkl"
)

METRICAS_321 = [
    "F1_macro",
    "Recall_adverso",
    "Precision_adverso",
    "ROC_AUC",
    "Accuracy"
]

FUNCIONES_REQUERIDAS_321 = [
    "preparar_datos_320",
    "construir_loader_320",
    "nuevo_transformer_320",
    "entrenar_transformer_320",
    "predecir_loader_320"
]

for nombre in FUNCIONES_REQUERIDAS_321:
    assert nombre in globals(), (
        f"Falta {nombre}. Ejecuta primero "
        "la Actividad 3.20."
    )

assert MAX_EPOCAS_320 == 30
assert PACIENCIA_320 == 6
assert SEMANA_320 == 8
assert len(cv_splits_oulad[8]) == 10

print("=== 3.21.1 CONFIGURACIÓN ===")
print("Semana:", SEMANA_321)
print("Folds externos:", 10)
print("Folds internos:", 3)
print("Máximo de épocas:", MAX_EPOCAS_320)
print("Paciencia:", PACIENCIA_320)
print(
    "Folds nuevos por ejecución:",
    NUEVOS_FOLDS_POR_EJECUCION
)

# ------------------------------------------------------------
# 2. RECUPERAR RESULTADOS EXISTENTES
# ------------------------------------------------------------

if (
    "resultados_transformer_C_s8" not in globals()
):

    if os.path.exists(RUTA_CHECKPOINT_321):

        with open(
            RUTA_CHECKPOINT_321, "rb"
        ) as archivo:

            resultados_transformer_C_s8 = (
                pickle.load(archivo)
            )

        print("Checkpoint anterior recuperado.")

    else:

        resultados_transformer_C_s8 = {}

# Reutilizar el fold externo de la Actividad 3.20.
if (1, 1) not in resultados_transformer_C_s8:

    assert "resultado_transformer_C_320" in globals()
    assert "predicciones_C_320" in globals()

    resumen_piloto = {
        "Semana": 8,
        "Repeticion": 1,
        "Fold": 1,
        "Epocas_finales": (
            resultado_transformer_C_320[
                "epocas_finales"
            ]
        ),
        "N_validacion": len(predicciones_C_320)
    }

    for metrica in METRICAS_321:
        resumen_piloto[metrica] = (
            resultado_transformer_C_320[
                "metricas"
            ][metrica]
        )

    for metrica in ["VN", "FP", "FN", "VP"]:
        resumen_piloto[metrica] = (
            resultado_transformer_C_320[
                "metricas"
            ][metrica]
        )

    resultados_transformer_C_s8[(1, 1)] = {
        "resumen": resumen_piloto,
        "predicciones": predicciones_C_320.copy(),
        "epocas_internas": (
            resultado_transformer_C_320[
                "epocas_internas"
            ].copy()
        )
    }

    print("R1-F1 recuperado de la Actividad 3.20.")

# ------------------------------------------------------------
# 3. FUNCIÓN DE ENTRENAMIENTO DE UN FOLD EXTERNO
# ------------------------------------------------------------

def ejecutar_fold_transformer_321(numero_split):

    repeticion = numero_split // 5 + 1
    fold = numero_split % 5 + 1

    idx_ext_train, idx_ext_val = (
        cv_splits_oulad[8][numero_split]
    )

    assert set(
        grupos_320.iloc[idx_ext_train]
    ).isdisjoint(
        set(grupos_320.iloc[idx_ext_val])
    )

    print(
        f"\n=== SEMANA 8 | R{repeticion}-F{fold} ===",
        flush=True
    )

    inicio = time.perf_counter()

    # --------------------------------------------------------
    # 3.1. DEFINIR LOS TRES FOLDS INTERNOS
    # --------------------------------------------------------

    cv_interna = StratifiedGroupKFold(
        n_splits=3,
        shuffle=True,
        random_state=137 + numero_split
    )

    splits_internos = list(
        cv_interna.split(
            X_est_320.iloc[idx_ext_train],
            y_320.iloc[idx_ext_train],
            groups=grupos_320.iloc[idx_ext_train]
        )
    )

    epocas_internas = []

    for f_interno, (
        idx_train_rel,
        idx_val_rel
    ) in enumerate(
        splits_internos,
        start=1
    ):

        idx_train = idx_ext_train[idx_train_rel]
        idx_val = idx_ext_train[idx_val_rel]

        assert set(
            grupos_320.iloc[idx_train]
        ).isdisjoint(
            set(grupos_320.iloc[idx_val])
        )

        datos = preparar_datos_320(
            idx_train,
            idx_val
        )

        semilla = (
            SEMILLA_320
            + 1000 * numero_split
            + f_interno
        )

        loader_train = construir_loader_320(
            datos["seq_train"],
            datos["est_train"],
            datos["mask_train"],
            y_320.iloc[idx_train].to_numpy(),
            shuffle=True,
            semilla=semilla
        )

        loader_val = construir_loader_320(
            datos["seq_eval"],
            datos["est_eval"],
            datos["mask_eval"],
            y_320.iloc[idx_val].to_numpy()
        )

        modelo = nuevo_transformer_320(
            datos["est_train"].shape[1],
            semilla=semilla
        )

        ajustado = entrenar_transformer_320(
            modelo,
            loader_train,
            loader_val=loader_val,
            max_epocas=MAX_EPOCAS_320,
            paciencia=PACIENCIA_320
        )

        evaluacion_interna = predecir_loader_320(
            ajustado["modelo"],
            loader_val
        )

        y_interno = evaluacion_interna["y"]
        prob_interno = evaluacion_interna["prob"]

        epocas_internas.append({
            "Fold_interno": f_interno,
            "Mejor_epoca": ajustado["mejor_epoca"],
            "Loss_validacion": ajustado["mejor_loss"],
            "F1_macro": f1_score(
                y_interno,
                (prob_interno >= 0.5).astype(int),
                average="macro",
                zero_division=0
            ),
            "ROC_AUC": roc_auc_score(
                y_interno,
                prob_interno
            )
        })

        print(
            f"Fold interno {f_interno}: "
            f"época={ajustado['mejor_epoca']} | "
            f"loss={ajustado['mejor_loss']:.4f}",
            flush=True
        )

        del modelo, ajustado

    tabla_epocas = pd.DataFrame(
        epocas_internas
    )

    # --------------------------------------------------------
    # 3.2. SELECCIONAR ÉPOCAS
    # --------------------------------------------------------

    epocas_finales = int(
        np.median(
            tabla_epocas["Mejor_epoca"]
        )
    )

    assert 1 <= epocas_finales <= MAX_EPOCAS_320

    print(
        "Épocas internas:",
        tabla_epocas["Mejor_epoca"].tolist()
    )

    print(
        "Épocas finales:",
        epocas_finales
    )

    # --------------------------------------------------------
    # 3.3. REENTRENAMIENTO EXTERNO
    # --------------------------------------------------------

    datos_finales = preparar_datos_320(
        idx_ext_train,
        idx_ext_val
    )

    semilla_final = (
        SEMILLA_320
        + 100
        + 1000 * numero_split
    )

    loader_train_final = construir_loader_320(
        datos_finales["seq_train"],
        datos_finales["est_train"],
        datos_finales["mask_train"],
        y_320.iloc[idx_ext_train].to_numpy(),
        shuffle=True,
        semilla=semilla_final
    )

    loader_val_externo = construir_loader_320(
        datos_finales["seq_eval"],
        datos_finales["est_eval"],
        datos_finales["mask_eval"],
        y_320.iloc[idx_ext_val].to_numpy()
    )

    modelo_final = nuevo_transformer_320(
        datos_finales["est_train"].shape[1],
        semilla=semilla_final
    )

    entrenado_final = entrenar_transformer_320(
        modelo_final,
        loader_train_final,
        loader_val=None,
        max_epocas=epocas_finales
    )

    # --------------------------------------------------------
    # 3.4. UNA SOLA EVALUACIÓN EXTERNA
    # --------------------------------------------------------

    evaluacion = predecir_loader_320(
        entrenado_final["modelo"],
        loader_val_externo
    )

    y_real = evaluacion["y"]
    prob = evaluacion["prob"]
    pred = (prob >= 0.5).astype(int)

    assert np.array_equal(
        y_real,
        y_320.iloc[idx_ext_val].to_numpy()
    )

    vn, fp, fn, vp = confusion_matrix(
        y_real,
        pred,
        labels=[0, 1]
    ).ravel()

    resumen = {
        "Semana": 8,
        "Repeticion": repeticion,
        "Fold": fold,
        "Epocas_finales": epocas_finales,
        "N_validacion": len(y_real),
        "F1_macro": f1_score(
            y_real, pred,
            average="macro",
            zero_division=0
        ),
        "Recall_adverso": recall_score(
            y_real, pred,
            zero_division=0
        ),
        "Precision_adverso": precision_score(
            y_real, pred,
            zero_division=0
        ),
        "ROC_AUC": roc_auc_score(
            y_real, prob
        ),
        "Accuracy": accuracy_score(
            y_real, pred
        ),
        "VN": int(vn),
        "FP": int(fp),
        "FN": int(fn),
        "VP": int(vp),
        "Tiempo_segundos": (
            time.perf_counter() - inicio
        )
    }

    predicciones = (
        claves_320.iloc[idx_ext_val]
        .reset_index(drop=True)
        .copy()
    )

    predicciones["Semana"] = 8
    predicciones["Repeticion"] = repeticion
    predicciones["Fold"] = fold
    predicciones["Experimento"] = "C"
    predicciones["Algoritmo"] = "Transformer"
    predicciones["y_real"] = y_real
    predicciones["y_pred"] = pred
    predicciones["prob_adverso"] = prob

    assert not predicciones.duplicated(
        subset=[
            "code_module",
            "code_presentation",
            "id_student"
        ]
    ).any()

    assert len(predicciones) == len(idx_ext_val)

    print(
        f"F1 externo={resumen['F1_macro']:.4f} | "
        f"Recall={resumen['Recall_adverso']:.4f} | "
        f"AUC={resumen['ROC_AUC']:.4f}",
        flush=True
    )

    return {
        "resumen": resumen,
        "predicciones": predicciones,
        "epocas_internas": tabla_epocas
    }


# ------------------------------------------------------------
# 4. IDENTIFICAR FOLDS PENDIENTES
# ------------------------------------------------------------

pendientes_321 = []

for numero_split in range(10):

    clave = (
        numero_split // 5 + 1,
        numero_split % 5 + 1
    )

    if clave not in resultados_transformer_C_s8:
        pendientes_321.append(numero_split)

print("\n=== 3.21.2 ESTADO DE EJECUCIÓN ===")
print(
    "Folds completados:",
    len(resultados_transformer_C_s8)
)
print("Folds pendientes:", len(pendientes_321))

# ------------------------------------------------------------
# 5. EJECUTAR EL SIGUIENTE BLOQUE
# ------------------------------------------------------------

for numero_split in pendientes_321[
    :NUEVOS_FOLDS_POR_EJECUCION
]:

    repeticion = numero_split // 5 + 1
    fold = numero_split % 5 + 1

    clave = (repeticion, fold)

    resultado = ejecutar_fold_transformer_321(
        numero_split
    )

    resultados_transformer_C_s8[clave] = resultado

    # Guardar inmediatamente al finalizar el fold.
    with open(
        RUTA_CHECKPOINT_321,
        "wb"
    ) as archivo:

        pickle.dump(
            resultados_transformer_C_s8,
            archivo
        )

    # Copiar a Drive inmediatamente después de cerrar el archivo del fold.
    respaldar_checkpoint_oulad(RUTA_CHECKPOINT_321)

    print(
        "Resultado guardado:",
        clave,
        flush=True
    )

# ------------------------------------------------------------
# 6. TABLA DE RESULTADOS ACUMULADOS
# ------------------------------------------------------------

print("\n=== 3.21.3 RESULTADOS DISPONIBLES ===")

tabla_transformer_C_s8 = pd.DataFrame(
    [
        resultado["resumen"]
        for resultado in resultados_transformer_C_s8.values()
    ]
).sort_values(
    ["Repeticion", "Fold"]
).reset_index(drop=True)

display(
    tabla_transformer_C_s8.round(4)
)

# ------------------------------------------------------------
# 7. COMPARACIÓN CON LR B Y RF B
# ------------------------------------------------------------

print("\n=== 3.21.4 COMPARACIÓN PAREADA ===")

comparacion_321 = (
    tabla_transformer_C_s8[
        [
            "Repeticion",
            "Fold",
            "F1_macro",
            "Recall_adverso",
            "ROC_AUC"
        ]
    ].rename(
        columns={
            "F1_macro": "F1_C",
            "Recall_adverso": "Recall_C",
            "ROC_AUC": "AUC_C"
        }
    )
)

for algoritmo, sufijo in [
    ("LogisticRegression", "LR_B"),
    ("RandomForest", "RF_B")
]:

    clasico = resultados_cv_ventanas[
        (8, algoritmo)
    ]["resultados"]

    clasico = clasico.loc[
        clasico["Experimento"] == "B",
        [
            "Repeticion",
            "Fold",
            "F1_macro",
            "Recall_adverso",
            "ROC_AUC"
        ]
    ].rename(
        columns={
            "F1_macro": f"F1_{sufijo}",
            "Recall_adverso": f"Recall_{sufijo}",
            "ROC_AUC": f"AUC_{sufijo}"
        }
    )

    comparacion_321 = comparacion_321.merge(
        clasico,
        on=["Repeticion", "Fold"],
        validate="one_to_one"
    )

comparacion_321["Delta_F1_C_menos_LR_B"] = (
    comparacion_321["F1_C"]
    - comparacion_321["F1_LR_B"]
)

comparacion_321["Delta_F1_C_menos_RF_B"] = (
    comparacion_321["F1_C"]
    - comparacion_321["F1_RF_B"]
)

display(comparacion_321.round(4))

# ------------------------------------------------------------
# 8. RESUMEN PROVISIONAL O FINAL
# ------------------------------------------------------------

print("\n=== 3.21.5 RESUMEN ===")

print(
    "Folds evaluados:",
    len(tabla_transformer_C_s8),
    "de 10"
)

print(
    "F1 medio disponible:",
    round(
        tabla_transformer_C_s8[
            "F1_macro"
        ].mean(),
        4
    )
)

print(
    "ROC-AUC medio disponible:",
    round(
        tabla_transformer_C_s8[
            "ROC_AUC"
        ].mean(),
        4
    )
)

if len(tabla_transformer_C_s8) == 10:

    print(
        "\nVALIDACIÓN EXTERNA REPETIDA COMPLETA."
    )

    print(
        "Se pueden calcular comparaciones "
        "pareadas de los 10 folds."
    )

else:

    print(
        "\nRESULTADOS PARCIALES."
    )

    print(
        "No deben interpretarse todavía como "
        "el rendimiento medio de la semana 8."
    )

    print(
        "Vuelve a ejecutar esta misma celda "
        "para procesar los siguientes folds."
    )

# ------------------------------------------------------------
# 9. CONTROLES
# ------------------------------------------------------------

print("\n=== 3.21.6 CONTROLES ===")

assert len(resultados_transformer_C_s8) <= 10

assert not tabla_transformer_C_s8.duplicated(
    subset=["Repeticion", "Fold"]
).any()

for resultado in resultados_transformer_C_s8.values():

    assert resultado["predicciones"][
        "prob_adverso"
    ].between(0, 1).all()

    assert not resultado["predicciones"].duplicated(
        subset=[
            "code_module",
            "code_presentation",
            "id_student"
        ]
    ).any()

print("Particiones agrupadas por estudiante.")
print("Preprocesamiento ajustado dentro de cada fold.")
print("Selección de épocas exclusivamente interna.")
print("Arquitectura e hiperparámetros congelados.")
print("Conjunto de prueba independiente: NO utilizado.")

print("\nCheckpoint:", RUTA_CHECKPOINT_321)
print("\n=== EJECUCIÓN DE ACTIVIDAD 3.21 FINALIZADA ===")



### 3.21.7. Resultados e interpretación

Se completó la evaluación externa repetida del Transformer C para la semana 8 de OULAD mediante cinco folds y dos repeticiones, agrupando las matrículas por identificador de estudiante.

En cada partición externa se utilizaron tres folds internos para seleccionar la duración del entrenamiento. Los procedimientos de preprocesamiento estático, normalización temporal y ajuste de la red neuronal se ejecutaron sin acceder a los datos de validación externa durante la selección.

Se completaron diez evaluaciones externas, manteniendo la arquitectura y los hiperparámetros definidos anteriormente.

#### Rendimiento del Transformer C

| Métrica | Media | Desviación estándar |
|---|---:|---:|
| F1 macro | 0,7095 | 0,0099 |
| Recall adverso | 0,5515 | 0,0401 |
| ROC-AUC | 0,7924 | 0,0073 |
| Accuracy | 0,7310 | 0,0049 |

La duración final de entrenamiento varió entre 19 y 29 épocas, con una mediana de 24.

#### Comparación con los modelos longitudinales clásicos

| Modelo | F1 macro | Recall adverso | ROC-AUC | Accuracy |
|---|---:|---:|---:|---:|
| Regresión Logística B | 0,7228 | 0,5908 | 0,7995 | 0,7395 |
| Transformer C | 0,7095 | 0,5515 | 0,7924 | 0,7310 |
| Random Forest B | 0,7006 | 0,5258 | 0,7837 | 0,7249 |

El Transformer C presentó una diferencia media de F1 macro de -0,0133 respecto a Regresión Logística B y de +0,0089 respecto a Random Forest B.

Su F1 macro fue inferior al de Regresión Logística B en las diez particiones externas, mientras que superó al de Random Forest B en nueve de las diez.

En ROC-AUC, el Transformer obtuvo valores superiores a Random Forest B en los diez folds, aunque inferiores a Regresión Logística B en los diez.

#### Interpretación

La arquitectura Transformer demostró capacidad para aprender patrones predictivos a partir de observaciones semanales y variables estáticas.

Sin embargo, los resultados no muestran una ventaja consistente frente a Regresión Logística B, que emplea características longitudinales diseñadas explícitamente y un modelo de menor complejidad.

El Transformer presentó mejor rendimiento promedio que Random Forest B, aunque la magnitud de esta diferencia fue reducida.

Estos hallazgos sugieren que, bajo la representación temporal y la arquitectura evaluadas, el incremento de complejidad neuronal no garantiza una mejora en el rendimiento predictivo.

### 3.21.8. Limitaciones

La evaluación corresponde exclusivamente a la semana 8. Todavía no se ha determinado cómo se comporta el Transformer en las ventanas más tempranas.

Las diez particiones externas reutilizan estudiantes entre repeticiones y no pueden tratarse como diez muestras independientes.

El Transformer y los modelos clásicos utilizan procedimientos diferentes para seleccionar sus parámetros de entrenamiento.

Aunque el Transformer fue evaluado externamente mediante validación interna de épocas, la arquitectura se desarrolló durante una fase exploratoria anterior.

Las diferencias entre modelos todavía requieren una evaluación estadística pareada que considere la dependencia de las particiones.

El conjunto de prueba independiente permaneció reservado.

### 3.21.9. Conclusión

Se completó satisfactoriamente la validación cruzada externa repetida del Transformer C en la semana 8.

El Transformer alcanzó un F1 macro medio de 0,7095 y un ROC-AUC medio de 0,7924, situándose entre Regresión Logística B y Random Forest B.

**Conclusión principal:** el aprendizaje secuencial mediante Transformer es una alternativa viable para la predicción de resultados académicos adversos, pero los resultados de la semana 8 no evidencian una mejora frente a la regresión logística con características longitudinales agregadas.

El Experimento C se mantiene como una comparación complementaria destinada a estudiar el valor del aprendizaje directo de secuencias temporales.



### 3.22. Comparación estadística del Transformer C — Semana 8

#### Objetivo

Cuantificar la magnitud y la incertidumbre de las diferencias de rendimiento entre el Transformer C, Regresión Logística B y Random Forest B sobre las mismas diez particiones externas de la semana 8.

#### Comparaciones

Se evaluarán dos contrastes principales de carácter complementario:

- Transformer C frente a Regresión Logística B.
- Transformer C frente a Random Forest B.

La métrica principal será F1 macro.

También se presentarán descriptivamente las diferencias de recall adverso, ROC-AUC y accuracy.

#### Contraste estadístico

Se utilizará una prueba t corregida para validación cruzada repetida, incorporando la proporción entre los tamaños de validación y entrenamiento.

Los valores p de las dos comparaciones se ajustarán mediante Holm.

Se reportarán diferencias medias, desviaciones, intervalos individuales aproximados del 95 % y valores p ajustados.

#### Bootstrap agrupado por estudiante

Como análisis complementario, se realizarán 1.000 remuestreos de estudiantes con reemplazo.

Cada estudiante conservará sus matrículas, sus dos predicciones de validación externa repetida y el emparejamiento entre los tres modelos.

En cada muestra se calculará F1 macro para cada repetición y después se promediarán ambas.

Se construirán intervalos percentiles del 95 % para las diferencias de F1 macro entre Transformer C y los modelos clásicos.

#### Limitaciones

Los contrastes se realizan sobre modelos ya explorados y utilizan particiones correlacionadas de validación cruzada.

Los intervalos bootstrap se condicionan a las predicciones fuera de muestra existentes y no incluyen la incertidumbre completa de reentrenar las arquitecturas.

Los intervalos no se ajustarán por multiplicidad. La corrección de Holm se aplicará exclusivamente a los dos valores p de F1 macro incluidos en esta actividad.

El análisis tendrá carácter exploratorio y no sustituirá la evaluación independiente.

No se entrenarán nuevos clasificadores ni se utilizará el conjunto de prueba reservado.


In [ ]:

# ============================================================
# PROYECTO FINAL DE MACHINE LEARNING
# ACTIVIDAD 3.22
# COMPARACIÓN ESTADÍSTICA DEL TRANSFORMER C
# SEMANA 8 - SIN REENTRENAMIENTO
# ============================================================

# Asistencia IA: ChatGPT, octubre de 2026.
# Prompt:
# "Comparar Transformer C con Regresión Logística B y
# Random Forest B sobre los mismos diez folds externos
# de OULAD. Utilizar una prueba t corregida y Holm.
# Complementar con bootstrap pareado agrupado por
# id_student, utilizando predicciones OOF existentes.
# No entrenar modelos ni acceder al test independiente."

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import t as distribucion_t
from IPython.display import display

SEMANA_322 = 8
N_BOOTSTRAP_322 = 1000
SEMILLA_322 = 2032

CLAVES_322 = [
    "code_module",
    "code_presentation",
    "id_student",
    "Repeticion"
]

MODELOS_322 = ["C", "LR", "RF"]

# ------------------------------------------------------------
# 1. CONSOLIDAR MÉTRICAS DE LOS DIEZ FOLDS
# ------------------------------------------------------------

print("=== 3.22.1 CONFIGURACIÓN ===")

assert len(tabla_transformer_C_s8) == 10
assert len(resultados_transformer_C_s8) == 10

tabla_322 = (
    tabla_transformer_C_s8[
        [
            "Repeticion",
            "Fold",
            "F1_macro",
            "Recall_adverso",
            "ROC_AUC",
            "Accuracy",
            "Epocas_finales"
        ]
    ]
    .rename(columns={
        "F1_macro": "F1_C",
        "Recall_adverso": "Recall_C",
        "ROC_AUC": "AUC_C",
        "Accuracy": "Accuracy_C"
    })
    .copy()
)

for algoritmo, sufijo in [
    ("LogisticRegression", "LR"),
    ("RandomForest", "RF")
]:

    clasico = resultados_cv_ventanas[
        (8, algoritmo)
    ]["resultados"]

    clasico = clasico.loc[
        clasico["Experimento"].eq("B"),
        [
            "Repeticion",
            "Fold",
            "F1_macro",
            "Recall_adverso",
            "ROC_AUC",
            "Accuracy"
        ]
    ].rename(columns={
        "F1_macro": f"F1_{sufijo}",
        "Recall_adverso": f"Recall_{sufijo}",
        "ROC_AUC": f"AUC_{sufijo}",
        "Accuracy": f"Accuracy_{sufijo}"
    })

    tabla_322 = tabla_322.merge(
        clasico,
        on=["Repeticion", "Fold"],
        how="inner",
        validate="one_to_one"
    )

assert len(tabla_322) == 10

assert not tabla_322.duplicated(
    subset=["Repeticion", "Fold"]
).any()

print("Semana:", SEMANA_322)
print("Folds emparejados:", len(tabla_322))
print("Bootstrap:", N_BOOTSTRAP_322)
print("Modelos reentrenados: 0")

# ------------------------------------------------------------
# 2. RENDIMIENTO DESCRIPTIVO
# ------------------------------------------------------------

print("\n=== 3.22.2 RENDIMIENTO DE LOS TRES MODELOS ===")

filas_resumen_322 = []

for sufijo, nombre in [
    ("LR", "Regresión Logística B"),
    ("C", "Transformer C"),
    ("RF", "Random Forest B")
]:

    fila = {"Modelo": nombre}

    for metrica, prefijo in [
        ("F1_macro", "F1"),
        ("Recall_adverso", "Recall"),
        ("ROC_AUC", "AUC"),
        ("Accuracy", "Accuracy")
    ]:

        valores = tabla_322[
            f"{prefijo}_{sufijo}"
        ]

        fila[f"{metrica}_media"] = valores.mean()
        fila[f"{metrica}_std"] = valores.std(ddof=1)

    filas_resumen_322.append(fila)

resumen_modelos_322 = pd.DataFrame(
    filas_resumen_322
)

display(resumen_modelos_322.round(4))

# ------------------------------------------------------------
# 3. DIFERENCIAS PAREADAS
# ------------------------------------------------------------

print("\n=== 3.22.3 DIFERENCIAS PAREADAS ===")

filas_diferencias_322 = []

for referencia in ["LR", "RF"]:

    for metrica, prefijo in [
        ("F1_macro", "F1"),
        ("Recall_adverso", "Recall"),
        ("ROC_AUC", "AUC"),
        ("Accuracy", "Accuracy")
    ]:

        delta = (
            tabla_322[f"{prefijo}_C"]
            - tabla_322[f"{prefijo}_{referencia}"]
        )

        filas_diferencias_322.append({
            "Comparacion": f"C - {referencia} B",
            "Metrica": metrica,
            "Delta_medio": delta.mean(),
            "Delta_std": delta.std(ddof=1),
            "Folds_C_supera": int((delta > 0).sum()),
            "Folds_C_inferior": int((delta < 0).sum()),
            "Folds_empatados": int((delta == 0).sum())
        })

tabla_diferencias_322 = pd.DataFrame(
    filas_diferencias_322
)

display(tabla_diferencias_322.round(4))

# ------------------------------------------------------------
# 4. PRUEBA T CORREGIDA PARA F1 MACRO
# ------------------------------------------------------------

print("\n=== 3.22.4 CONTRASTES CORREGIDOS ===")

splits_322 = cv_splits_oulad[8]

assert len(splits_322) == 10

ratio_322 = np.mean([
    len(idx_val) / len(idx_train)
    for idx_train, idx_val in splits_322
])

n_322 = 10
factor_322 = (1.0 / n_322) + ratio_322

contrastes_322 = []

for referencia in ["LR", "RF"]:

    delta = (
        tabla_322["F1_C"]
        - tabla_322[f"F1_{referencia}"]
    ).to_numpy(dtype=float)

    media = float(delta.mean())
    std = float(delta.std(ddof=1))

    error = np.sqrt(factor_322) * std

    assert error > 0

    t_corregida = media / error
    grados = n_322 - 1

    p = 2 * distribucion_t.sf(
        abs(t_corregida),
        df=grados
    )

    critico = distribucion_t.ppf(
        0.975,
        df=grados
    )

    contrastes_322.append({
        "Comparacion": f"C - {referencia} B",
        "Delta_F1": media,
        "Desviacion_delta": std,
        "t_corregida": t_corregida,
        "p_sin_ajustar": p,
        "IC95_inferior": media - critico * error,
        "IC95_superior": media + critico * error
    })

tabla_contrastes_322 = pd.DataFrame(
    contrastes_322
)

# Holm sobre las dos comparaciones F1 de esta actividad.
p_originales = tabla_contrastes_322[
    "p_sin_ajustar"
].to_numpy()

orden = np.argsort(p_originales)

p_ordenados = p_originales[orden]
m = len(p_ordenados)

p_ajustados_ordenados = np.maximum.accumulate(
    p_ordenados * (m - np.arange(m))
)

p_ajustados = np.empty(m)

p_ajustados[orden] = np.minimum(
    p_ajustados_ordenados,
    1.0
)

tabla_contrastes_322["p_Holm"] = p_ajustados

tabla_contrastes_322["Significativo_Holm_005"] = (
    tabla_contrastes_322["p_Holm"] < 0.05
)

print(
    "Proporción media validación / entrenamiento:",
    round(ratio_322, 4)
)

display(tabla_contrastes_322.round(6))

# ------------------------------------------------------------
# 5. RECUPERAR PREDICCIONES EXTERNAS INDIVIDUALES
# ------------------------------------------------------------

print("\n=== 3.22.5 INTEGRIDAD DE PREDICCIONES ===")

pred_C_322 = pd.concat(
    [
        resultado["predicciones"]
        for resultado in resultados_transformer_C_s8.values()
    ],
    ignore_index=True
)

pred_C_322 = pred_C_322[
    CLAVES_322 + ["y_real", "y_pred"]
].rename(columns={
    "y_pred": "pred_C"
})

n_matriculas_322 = len(
    experimentos_oulad[8]["y_train"]
)

assert len(pred_C_322) == (
    2 * n_matriculas_322
)

assert not pred_C_322.duplicated(
    subset=CLAVES_322
).any()

pred_emparejadas_322 = pred_C_322.copy()

for algoritmo, sufijo in [
    ("LogisticRegression", "LR"),
    ("RandomForest", "RF")
]:

    pred_clasicas = resultados_cv_ventanas[
        (8, algoritmo)
    ]["predicciones"]

    pred_clasicas = (
        pred_clasicas.loc[
            pred_clasicas["Experimento"].eq("B"),
            CLAVES_322 + ["y_real", "y_pred"]
        ]
        .rename(columns={
            "y_real": f"y_{sufijo}",
            "y_pred": f"pred_{sufijo}"
        })
    )

    assert not pred_clasicas.duplicated(
        subset=CLAVES_322
    ).any()

    pred_emparejadas_322 = (
        pred_emparejadas_322.merge(
            pred_clasicas,
            on=CLAVES_322,
            how="inner",
            validate="one_to_one"
        )
    )

    assert len(pred_emparejadas_322) == (
        2 * n_matriculas_322
    )

    assert np.array_equal(
        pred_emparejadas_322["y_real"].to_numpy(),
        pred_emparejadas_322[
            f"y_{sufijo}"
        ].to_numpy()
    )

assert pred_emparejadas_322.groupby(
    "Repeticion"
).size().eq(n_matriculas_322).all()

estudiantes_322 = np.sort(
    pred_emparejadas_322[
        "id_student"
    ].unique()
)

N_ESTUDIANTES_322 = len(estudiantes_322)

print(
    "Predicciones emparejadas:",
    len(pred_emparejadas_322)
)

print(
    "Estudiantes únicos:",
    N_ESTUDIANTES_322
)

print(
    "Matrículas por repetición:",
    n_matriculas_322
)

# ------------------------------------------------------------
# 6. PREPARAR MATRICES POR ESTUDIANTE
# ------------------------------------------------------------

# Forma:
# modelo, repetición, estudiante, TN/FP/FN/TP

matrices_322 = np.zeros(
    (3, 2, N_ESTUDIANTES_322, 4),
    dtype=np.int32
)

for mi, sufijo in enumerate(MODELOS_322):

    for ri, repeticion in enumerate([1, 2]):

        sub = pred_emparejadas_322.loc[
            pred_emparejadas_322[
                "Repeticion"
            ].eq(repeticion)
        ]

        reales = sub["y_real"].to_numpy(
            dtype=int
        )

        predichos = sub[
            f"pred_{sufijo}"
        ].to_numpy(dtype=int)

        cuentas = pd.DataFrame({
            "id_student": sub["id_student"].to_numpy(),
            "TN": (
                (reales == 0) & (predichos == 0)
            ).astype(int),
            "FP": (
                (reales == 0) & (predichos == 1)
            ).astype(int),
            "FN": (
                (reales == 1) & (predichos == 0)
            ).astype(int),
            "TP": (
                (reales == 1) & (predichos == 1)
            ).astype(int)
        })

        agrupado = (
            cuentas.groupby("id_student")[
                ["TN", "FP", "FN", "TP"]
            ]
            .sum()
            .reindex(
                estudiantes_322,
                fill_value=0
            )
        )

        matrices_322[mi, ri] = agrupado.to_numpy(
            dtype=np.int32
        )

assert np.all(
    matrices_322.sum(axis=(2, 3))
    == n_matriculas_322
)

# ------------------------------------------------------------
# 7. F1 MACRO A PARTIR DE CONTEOS
# ------------------------------------------------------------

def f1_desde_conteos_322(c):

    tn = c[..., 0]
    fp = c[..., 1]
    fn = c[..., 2]
    tp = c[..., 3]

    f1_pos = np.divide(
        2 * tp,
        2 * tp + fp + fn,
        out=np.zeros_like(tp, dtype=float),
        where=(2 * tp + fp + fn) > 0
    )

    f1_neg = np.divide(
        2 * tn,
        2 * tn + fp + fn,
        out=np.zeros_like(tn, dtype=float),
        where=(2 * tn + fp + fn) > 0
    )

    return (f1_pos + f1_neg) / 2


# ------------------------------------------------------------
# 8. BOOTSTRAP AGRUPADO POR ESTUDIANTE
# ------------------------------------------------------------

print("\n=== 3.22.6 BOOTSTRAP PAREADO ===")

rng_322 = np.random.default_rng(
    SEMILLA_322
)

distribucion_boot_322 = np.empty(
    (N_BOOTSTRAP_322, 2),
    dtype=float
)

conteos_originales = matrices_322.sum(
    axis=2,
    dtype=np.int64
)

f1_originales = f1_desde_conteos_322(
    conteos_originales
).mean(axis=1)

deltas_originales = np.array([
    f1_originales[0] - f1_originales[1],
    f1_originales[0] - f1_originales[2]
])

for b in range(N_BOOTSTRAP_322):

    indices = rng_322.integers(
        0,
        N_ESTUDIANTES_322,
        size=N_ESTUDIANTES_322
    )

    pesos = np.bincount(
        indices,
        minlength=N_ESTUDIANTES_322
    ).astype(np.int64)

    conteos = np.einsum(
        "g,mrgk->mrk",
        pesos,
        matrices_322,
        dtype=np.int64
    )

    f1_remuestreo = f1_desde_conteos_322(
        conteos
    ).mean(axis=1)

    distribucion_boot_322[b, 0] = (
        f1_remuestreo[0] - f1_remuestreo[1]
    )

    distribucion_boot_322[b, 1] = (
        f1_remuestreo[0] - f1_remuestreo[2]
    )

resultados_boot_322 = []

for i, referencia in enumerate(["LR", "RF"]):

    valores = distribucion_boot_322[:, i]

    ic = np.quantile(
        valores,
        [0.025, 0.975]
    )

    resultados_boot_322.append({
        "Comparacion": f"C - {referencia} B",
        "Delta_F1_OOF": deltas_originales[i],
        "IC95_inferior": ic[0],
        "IC95_superior": ic[1],
        "Fraccion_boot_delta_positivo": (
            valores > 0
        ).mean()
    })

tabla_bootstrap_C_322 = pd.DataFrame(
    resultados_boot_322
)

display(tabla_bootstrap_C_322.round(5))

# ------------------------------------------------------------
# 9. GRÁFICO COMPARATIVO
# ------------------------------------------------------------

print("\n=== 3.22.7 GRÁFICO DE DIFERENCIAS ===")

fig, ax = plt.subplots(figsize=(9, 4.5))

posiciones = np.arange(2)

valores = tabla_bootstrap_C_322[
    "Delta_F1_OOF"
].to_numpy()

errores_inf = (
    valores
    - tabla_bootstrap_C_322[
        "IC95_inferior"
    ].to_numpy()
)

errores_sup = (
    tabla_bootstrap_C_322[
        "IC95_superior"
    ].to_numpy()
    - valores
)

ax.errorbar(
    posiciones,
    valores,
    yerr=np.vstack([
        errores_inf,
        errores_sup
    ]),
    fmt="o",
    capsize=6,
    linewidth=2
)

ax.axhline(
    0,
    linestyle="--",
    linewidth=1
)

ax.set_xticks(posiciones)

ax.set_xticklabels([
    "Transformer C - LR B",
    "Transformer C - RF B"
])

ax.set_ylabel("Diferencia de F1 macro")

ax.set_title(
    "Semana 8 - Comparación pareada del Transformer\n"
    "Intervalos bootstrap agrupados por estudiante"
)

ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 10. CONTROLES FINALES
# ------------------------------------------------------------

print("\n=== 3.22.8 CONTROLES FINALES ===")

assert len(tabla_contrastes_322) == 2

assert len(tabla_bootstrap_C_322) == 2

assert np.isfinite(
    distribucion_boot_322
).all()

assert len(pred_emparejadas_322) == (
    2 * n_matriculas_322
)

assert N_ESTUDIANTES_322 == (
    experimentos_oulad[8][
        "grupos_train"
    ].nunique()
)

print("Comparaciones estadísticas: 2")
print("Folds externos emparejados: 10")

print(
    "Bootstrap por comparación:",
    N_BOOTSTRAP_322
)

print(
    "Emparejamiento verificado por matrícula "
    "y estudiante."
)

print(
    "Dos repeticiones conservadas "
    "sin tratarlas como muestras independientes."
)

print("Modelos entrenados nuevamente: 0")

print(
    "Conjunto de prueba independiente: NO utilizado"
)

print(
    "Interpretación: análisis exploratorio "
    "de resultados de desarrollo."
)

print("\n=== ACTIVIDAD 3.22 FINALIZADA ===")



### 3.22.9. Resultados e interpretación estadística

Se completó la comparación estadística del Transformer C frente a Regresión Logística B y Random Forest B sobre las diez particiones externas de la semana 8.

El análisis utilizó diferencias pareadas de F1 macro, una prueba t corregida para validación cruzada repetida, ajuste de Holm y 1.000 remuestreos bootstrap agrupados por estudiante.

No se entrenaron nuevos modelos y el conjunto de prueba independiente permaneció reservado.

#### Rendimiento comparativo

| Modelo | F1 macro | Recall adverso | ROC-AUC | Accuracy |
|---|---:|---:|---:|---:|
| Regresión Logística B | 0,7228 | 0,5908 | 0,7995 | 0,7395 |
| Transformer C | 0,7095 | 0,5515 | 0,7924 | 0,7310 |
| Random Forest B | 0,7006 | 0,5258 | 0,7837 | 0,7249 |

Regresión Logística B presentó el mejor rendimiento medio en las cuatro métricas.

El Transformer C superó a Random Forest B en F1 macro, recall adverso, ROC-AUC y accuracy medios, pero obtuvo resultados inferiores a Regresión Logística B.

#### Diferencias pareadas

La diferencia media de F1 macro entre Transformer C y Regresión Logística B fue de -0,0133, con una desviación estándar de 0,0088.

La diferencia media entre Transformer C y Random Forest B fue de +0,0089, con una desviación estándar de 0,0076.

El Transformer obtuvo un F1 macro inferior al de Regresión Logística B en los diez folds y superior al de Random Forest B en nueve.

#### Contrastes estadísticos corregidos

Se utilizó una prueba t corregida por dependencia de particiones y se aplicó Holm a las dos comparaciones de F1 macro.

| Comparación | p sin ajustar | p ajustado Holm |
|---|---:|---:|
| Transformer C - LR B | 0,030804 | 0,061607 |
| Transformer C - RF B | 0,078297 | 0,078297 |

Ninguno de los contrastes alcanzó significancia estadística al nivel de 0,05 después del ajuste de Holm.

Por tanto, el procedimiento corregido no proporciona evidencia suficiente para rechazar las hipótesis nulas de ausencia de diferencia media en las dos comparaciones.

La ausencia de significancia no demuestra equivalencia entre modelos.

#### Bootstrap agrupado por estudiante

Se recuperaron 42.328 predicciones emparejadas, correspondientes a dos repeticiones de 21.164 matrículas y 19.233 estudiantes únicos.

Los resultados del bootstrap fueron:

| Comparación | Diferencia F1 | IC percentil 95 % |
|---|---:|---|
| Transformer C - LR B | -0,01295 | [-0,01697; -0,00922] |
| Transformer C - RF B | +0,00931 | [0,00449; 0,01383] |

Los intervalos bootstrap excluyeron cero, indicando estabilidad de las diferencias bajo el remuestreo de estudiantes y condicionada a las predicciones fuera de muestra disponibles.

La diferencia respecto a los resultados del contraste t se explica por los distintos procedimientos de estimación de incertidumbre.

El bootstrap no incorpora toda la variabilidad asociada al reentrenamiento de modelos, sus inicializaciones, la selección de épocas y la optimización de hiperparámetros.

### 3.22.10. Limitaciones

Los contrastes se realizaron sobre modelos previamente explorados y sobre particiones correlacionadas de validación cruzada.

La prueba t corregida constituye una aproximación estadística y no garantiza la independencia entre evaluaciones.

Los intervalos bootstrap se condicionan a los modelos entrenados y no representan una repetición independiente del procedimiento completo.

Las comparaciones corresponden exclusivamente a la semana 8.

Además, el Experimento B y el Transformer C utilizan representaciones distintas de la información longitudinal. El primero incorpora características de actividad, tendencias y evaluaciones diseñadas explícitamente, mientras que el segundo utiliza secuencias semanales de cuatro canales. En consecuencia, la comparación no permite aislar únicamente el efecto de la arquitectura Transformer.

No se ha utilizado el conjunto de prueba independiente.

### 3.22.11. Conclusión

Regresión Logística B obtuvo el mejor rendimiento promedio durante la semana 8, seguida del Transformer C y Random Forest B.

El Transformer presentó ventajas descriptivas frente a Random Forest B, pero no superó a Regresión Logística B.

Las diferencias fueron consistentes bajo el bootstrap agrupado por estudiante, aunque los contrastes t corregidos y ajustados por Holm no alcanzaron significancia al 5 %.

**Conclusión principal:** el Transformer constituye una alternativa viable de aprendizaje secuencial, pero la evidencia disponible no demuestra una superioridad estadística frente a los modelos clásicos longitudinales. La mayor complejidad de la arquitectura neuronal no garantiza una mejora predictiva en las condiciones evaluadas.

La evaluación temporal del Experimento C deberá extenderse a las semanas 2, 4 y 6 antes de formular una conclusión global sobre las arquitecturas.



### 3.23. Evaluación temporal del Transformer C

#### Objetivo

Evaluar la evolución del rendimiento predictivo del Transformer C en las semanas 2, 4 y 6, complementando los resultados previamente obtenidos en la semana 8.

#### Diseño experimental

Se mantendrá la arquitectura Transformer Encoder definida durante las actividades anteriores:

- Dos capas de atención.
- Cuatro cabezas de atención.
- Dimensión de representación temporal de 32.
- Rama estática de 32 dimensiones.
- Dropout de 0,15.
- Optimizador AdamW.
- Tasa de aprendizaje de 0,0007.
- Weight decay de 0,001.
- Máximo de 30 épocas.
- Early stopping con paciencia de seis.

La única adaptación estructural será la longitud de las secuencias semanales de entrada.

#### Ventanas

| Semana | Longitud de secuencia |
|---|---:|
| 2 | 2 observaciones |
| 4 | 4 observaciones |
| 6 | 6 observaciones |
| 8 | 8 observaciones, evaluadas anteriormente |

#### Procedimiento

En cada semana se utilizarán cinco folds externos y dos repeticiones, conservando las mismas particiones agrupadas por estudiante empleadas en los experimentos A y B.

Dentro de cada entrenamiento externo se generarán tres folds internos para seleccionar la duración del entrenamiento mediante la pérdida de validación.

Posteriormente, el Transformer se reentrenará sobre el conjunto completo de entrenamiento externo y se evaluará una sola vez sobre la validación externa correspondiente.

Los procedimientos de imputación, codificación y escalamiento se ajustarán exclusivamente con los datos de entrenamiento de cada partición.

#### Ejecución incremental

Los resultados y predicciones se guardarán después de cada fold.

Se procesarán inicialmente dos folds por ejecución, permitiendo continuar posteriormente sin repetir entrenamientos finalizados.

#### Métricas

Se reportarán F1 macro, recall adverso, precisión adversa, ROC-AUC y accuracy.

Se conservarán las predicciones individuales para realizar comparaciones pareadas con Regresión Logística B y Random Forest B.

#### Limitaciones

La arquitectura fue definida durante el desarrollo de la semana 8; mantenerla congelada permite evaluar su transferencia a ventanas más tempranas, pero no garantiza que sea la arquitectura óptima para cada longitud de secuencia.

Los modelos clásicos y el Transformer utilizan representaciones longitudinales distintas.

El Experimento C se mantiene como análisis complementario.

El conjunto de prueba independiente continuará reservado.


In [ ]:

# ============================================================
# PROYECTO FINAL DE MACHINE LEARNING
# ACTIVIDAD 3.23 - EVALUACIÓN TEMPORAL TRANSFORMER C
# SEMANAS 2, 4 Y 6
# ============================================================

# Asistencia IA: ChatGPT, octubre de 2026.
# Prompt:
# "Extender el Transformer C de OULAD a las semanas
# 2, 4 y 6, manteniendo la arquitectura y el protocolo
# anidado ya utilizados en semana 8. Crear checkpoints
# incrementales, conservar las particiones agrupadas
# por estudiante y no utilizar el test independiente."

import os
import pickle
import random
import time
import numpy as np
import pandas as pd
import torch

from sklearn.base import clone
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    recall_score,
    precision_score,
    roc_auc_score,
    confusion_matrix
)

from IPython.display import display

# ------------------------------------------------------------
# 1. CONFIGURACIÓN
# ------------------------------------------------------------

SEMANAS_323 = [2, 4, 6]
NUEVOS_FOLDS_323 = 2

RUTA_323 = (
    "/content/transformer_C_temporal_323.pkl"
)

# Reutilizar exactamente los parámetros de 3.20.
assert MAX_EPOCAS_320 == 30
assert PACIENCIA_320 == 6
assert LR_320 == 0.0007
assert WD_320 == 0.001

funciones_necesarias_323 = [
    "TransformerOULAD",
    "preparar_temporales_319",
    "preprocesador_estatico_319",
    "construir_loader_320",
    "entrenar_transformer_320",
    "predecir_loader_320",
    "secuencias_oulad_C",
    "cv_splits_oulad"
]

for nombre in funciones_necesarias_323:
    assert nombre in globals(), (
        f"Falta {nombre}. Ejecuta primero "
        "las actividades 3.19 y 3.20."
    )

# Las funciones existentes utilizan esta variable.
dispositivo_320 = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=== 3.23.1 CONFIGURACIÓN ===")
print("Semanas pendientes:", SEMANAS_323)
print("Dispositivo:", dispositivo_320)
print("Folds externos por semana:", 10)
print("Folds internos:", 3)
print("Folds nuevos por ejecución:", NUEVOS_FOLDS_323)

# ------------------------------------------------------------
# 2. CHECKPOINT REANUDABLE
# ------------------------------------------------------------

if os.path.exists(RUTA_323):

    with open(RUTA_323, "rb") as archivo:
        resultados_transformer_C_323 = (
            pickle.load(archivo)
        )

    print("Checkpoint recuperado.")

elif "resultados_transformer_C_323" not in globals():

    resultados_transformer_C_323 = {}

# Validar las claves recuperadas.
for clave in resultados_transformer_C_323:
    assert len(clave) == 3
    assert clave[0] in SEMANAS_323
    assert clave[1] in [1, 2]
    assert 1 <= clave[2] <= 5

# ------------------------------------------------------------
# 3. PREPROCESAMIENTO POR FOLD
# ------------------------------------------------------------

def preparar_fold_C_323(semana, idx_train, idx_val):

    datos = secuencias_oulad_C[semana]

    X_est = datos["X_estatica_train"]
    X_seq = datos["X_secuencia_train"]

    # Clonar preprocesador sin parámetros aprendidos.
    preprocesador = clone(
        preprocesador_estatico_319
    )

    est_train = (
        preprocesador.fit_transform(
            X_est.iloc[idx_train]
        ).astype(np.float32)
    )

    est_val = (
        preprocesador.transform(
            X_est.iloc[idx_val]
        ).astype(np.float32)
    )

    (
        seq_train,
        seq_val,
        mask_train,
        mask_val,
        escalador
    ) = preparar_temporales_319(
        X_seq[idx_train],
        X_seq[idx_val]
    )

    assert seq_train.shape[1] == semana
    assert seq_val.shape[1] == semana

    assert est_train.shape[1] == est_val.shape[1]

    assert np.isfinite(est_train).all()
    assert np.isfinite(est_val).all()
    assert np.isfinite(seq_train).all()
    assert np.isfinite(seq_val).all()

    return {
        "est_train": est_train,
        "est_val": est_val,
        "seq_train": seq_train,
        "seq_val": seq_val,
        "mask_train": mask_train,
        "mask_val": mask_val
    }


# ------------------------------------------------------------
# 4. CREAR MODELO CON LONGITUD CORRECTA
# ------------------------------------------------------------

def nuevo_transformer_323(
    semana,
    n_estaticas,
    semilla
):

    random.seed(semilla)
    np.random.seed(semilla)
    torch.manual_seed(semilla)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(semilla)

    modelo = TransformerOULAD(
        n_static=n_estaticas,
        n_weeks=semana,
        n_channels=4,
        d_model=32,
        n_heads=4,
        n_layers=2,
        dropout=0.15
    )

    return modelo.to(dispositivo_320)


# ------------------------------------------------------------
# 5. EJECUTAR UN FOLD EXTERNO
# ------------------------------------------------------------

def ejecutar_fold_C_323(semana, numero_split):

    inicio = time.perf_counter()

    repeticion = numero_split // 5 + 1
    fold = numero_split % 5 + 1

    datos = secuencias_oulad_C[semana]

    X_est = datos["X_estatica_train"]
    y = datos["y_train"]
    grupos = datos["grupos_train"]
    claves = datos["claves_train"]

    idx_ext_train, idx_ext_val = (
        cv_splits_oulad[semana][numero_split]
    )

    assert set(
        grupos.iloc[idx_ext_train]
    ).isdisjoint(
        set(grupos.iloc[idx_ext_val])
    )

    print(
        f"\nSEMANA {semana} | R{repeticion}-F{fold}",
        flush=True
    )

    # --------------------------------------------------------
    # 5.1 FOLDS INTERNOS
    # --------------------------------------------------------

    cv_interna = StratifiedGroupKFold(
        n_splits=3,
        shuffle=True,
        random_state=137 + numero_split
    )

    splits_internos = list(
        cv_interna.split(
            X_est.iloc[idx_ext_train],
            y.iloc[idx_ext_train],
            groups=grupos.iloc[idx_ext_train]
        )
    )

    resultados_internos = []

    for numero_interno, (
        idx_train_rel,
        idx_val_rel
    ) in enumerate(
        splits_internos,
        start=1
    ):

        idx_train = idx_ext_train[idx_train_rel]
        idx_val = idx_ext_train[idx_val_rel]

        assert set(
            grupos.iloc[idx_train]
        ).isdisjoint(
            set(grupos.iloc[idx_val])
        )

        preparados = preparar_fold_C_323(
            semana,
            idx_train,
            idx_val
        )

        semilla = (
            42
            + 10000 * semana
            + 1000 * numero_split
            + numero_interno
        )

        loader_train = construir_loader_320(
            preparados["seq_train"],
            preparados["est_train"],
            preparados["mask_train"],
            y.iloc[idx_train].to_numpy(),
            shuffle=True,
            semilla=semilla
        )

        loader_val = construir_loader_320(
            preparados["seq_val"],
            preparados["est_val"],
            preparados["mask_val"],
            y.iloc[idx_val].to_numpy()
        )

        modelo = nuevo_transformer_323(
            semana,
            preparados["est_train"].shape[1],
            semilla
        )

        entrenado = entrenar_transformer_320(
            modelo,
            loader_train,
            loader_val=loader_val,
            max_epocas=MAX_EPOCAS_320,
            paciencia=PACIENCIA_320
        )

        resultados_internos.append({
            "Fold_interno": numero_interno,
            "Mejor_epoca": entrenado[
                "mejor_epoca"
            ],
            "Loss_validacion": entrenado[
                "mejor_loss"
            ]
        })

        print(
            f"  Fold interno {numero_interno}: "
            f"época={entrenado['mejor_epoca']} | "
            f"loss={entrenado['mejor_loss']:.4f}",
            flush=True
        )

        del modelo, entrenado

    tabla_interna = pd.DataFrame(
        resultados_internos
    )

    epocas_finales = int(
        np.median(
            tabla_interna["Mejor_epoca"]
        )
    )

    assert 1 <= epocas_finales <= 30

    print(
        "Épocas internas:",
        tabla_interna["Mejor_epoca"].tolist()
    )

    print(
        "Épocas finales:",
        epocas_finales
    )

    # --------------------------------------------------------
    # 5.2 REENTRENAR SOBRE TODO EL FOLD EXTERNO
    # --------------------------------------------------------

    preparados_finales = preparar_fold_C_323(
        semana,
        idx_ext_train,
        idx_ext_val
    )

    semilla_final = (
        42
        + 10000 * semana
        + 1000 * numero_split
        + 100
    )

    loader_train_final = construir_loader_320(
        preparados_finales["seq_train"],
        preparados_finales["est_train"],
        preparados_finales["mask_train"],
        y.iloc[idx_ext_train].to_numpy(),
        shuffle=True,
        semilla=semilla_final
    )

    loader_val_externo = construir_loader_320(
        preparados_finales["seq_val"],
        preparados_finales["est_val"],
        preparados_finales["mask_val"],
        y.iloc[idx_ext_val].to_numpy()
    )

    modelo_final = nuevo_transformer_323(
        semana,
        preparados_finales["est_train"].shape[1],
        semilla_final
    )

    entrenado_final = entrenar_transformer_320(
        modelo_final,
        loader_train_final,
        loader_val=None,
        max_epocas=epocas_finales
    )

    # --------------------------------------------------------
    # 5.3 EVALUACIÓN EXTERNA
    # --------------------------------------------------------

    evaluacion = predecir_loader_320(
        entrenado_final["modelo"],
        loader_val_externo
    )

    reales = evaluacion["y"]
    prob = evaluacion["prob"]
    pred = (prob >= 0.5).astype(int)

    assert np.array_equal(
        reales,
        y.iloc[idx_ext_val].to_numpy()
    )

    vn, fp, fn, vp = confusion_matrix(
        reales,
        pred,
        labels=[0, 1]
    ).ravel()

    resumen = {
        "Semana": semana,
        "Repeticion": repeticion,
        "Fold": fold,
        "Epocas_finales": epocas_finales,
        "N_validacion": len(reales),
        "F1_macro": f1_score(
            reales,
            pred,
            average="macro",
            zero_division=0
        ),
        "Recall_adverso": recall_score(
            reales,
            pred,
            zero_division=0
        ),
        "Precision_adverso": precision_score(
            reales,
            pred,
            zero_division=0
        ),
        "ROC_AUC": roc_auc_score(
            reales,
            prob
        ),
        "Accuracy": accuracy_score(
            reales,
            pred
        ),
        "VN": int(vn),
        "FP": int(fp),
        "FN": int(fn),
        "VP": int(vp),
        "Tiempo_segundos": (
            time.perf_counter() - inicio
        )
    }

    predicciones = (
        claves.iloc[idx_ext_val]
        .reset_index(drop=True)
        .copy()
    )

    predicciones["Semana"] = semana
    predicciones["Repeticion"] = repeticion
    predicciones["Fold"] = fold
    predicciones["Experimento"] = "C"
    predicciones["Algoritmo"] = "Transformer"
    predicciones["y_real"] = reales
    predicciones["y_pred"] = pred
    predicciones["prob_adverso"] = prob

    assert not predicciones.duplicated(
        subset=[
            "code_module",
            "code_presentation",
            "id_student"
        ]
    ).any()

    print(
        f"F1={resumen['F1_macro']:.4f} | "
        f"Recall={resumen['Recall_adverso']:.4f} | "
        f"AUC={resumen['ROC_AUC']:.4f}",
        flush=True
    )

    return {
        "resumen": resumen,
        "predicciones": predicciones,
        "epocas_internas": tabla_interna
    }


# ------------------------------------------------------------
# 6. IDENTIFICAR FOLDS PENDIENTES
# ------------------------------------------------------------

pendientes_323 = []

for semana in SEMANAS_323:

    assert len(cv_splits_oulad[semana]) == 10

    for numero_split in range(10):

        repeticion = numero_split // 5 + 1
        fold = numero_split % 5 + 1

        clave = (semana, repeticion, fold)

        if clave not in resultados_transformer_C_323:
            pendientes_323.append(
                (semana, numero_split)
            )

print("\n=== 3.23.2 ESTADO DE EJECUCIÓN ===")

print(
    "Folds ya completados:",
    len(resultados_transformer_C_323)
)

print(
    "Folds pendientes:",
    len(pendientes_323)
)

# ------------------------------------------------------------
# 7. EJECUCIÓN INCREMENTAL
# ------------------------------------------------------------

for semana, numero_split in (
    pendientes_323[:NUEVOS_FOLDS_323]
):

    repeticion = numero_split // 5 + 1
    fold = numero_split % 5 + 1

    clave = (semana, repeticion, fold)

    resultado = ejecutar_fold_C_323(
        semana,
        numero_split
    )

    resultados_transformer_C_323[clave] = resultado

    # Guardar de forma atómica para reducir
    # el riesgo de perder el checkpoint.
    ruta_temporal = RUTA_323 + ".tmp"

    with open(ruta_temporal, "wb") as archivo:
        pickle.dump(
            resultados_transformer_C_323,
            archivo
        )

    os.replace(
        ruta_temporal,
        RUTA_323
    )

    respaldar_checkpoint_oulad(RUTA_323)

    print(
        "Fold guardado:",
        clave,
        flush=True
    )

# ------------------------------------------------------------
# 8. TABLA DE RESULTADOS
# ------------------------------------------------------------

print("\n=== 3.23.3 RESULTADOS ACUMULADOS ===")

if resultados_transformer_C_323:

    tabla_transformer_C_323 = pd.DataFrame([
        resultado["resumen"]
        for resultado in (
            resultados_transformer_C_323.values()
        )
    ]).sort_values(
        ["Semana", "Repeticion", "Fold"]
    ).reset_index(drop=True)

    display(
        tabla_transformer_C_323.round(4)
    )

else:

    tabla_transformer_C_323 = pd.DataFrame()

# ------------------------------------------------------------
# 9. PROGRESO POR SEMANA
# ------------------------------------------------------------

print("\n=== 3.23.4 RESUMEN POR SEMANA ===")

filas_323 = []

for semana in SEMANAS_323:

    if tabla_transformer_C_323.empty:
        sub = pd.DataFrame()
    else:
        sub = tabla_transformer_C_323.loc[
            tabla_transformer_C_323[
                "Semana"
            ].eq(semana)
        ]

    filas_323.append({
        "Semana": semana,
        "Folds_completados": len(sub),
        "Folds_pendientes": 10 - len(sub),
        "F1_macro_provisional": (
            sub["F1_macro"].mean()
            if len(sub) else np.nan
        ),
        "Recall_provisional": (
            sub["Recall_adverso"].mean()
            if len(sub) else np.nan
        ),
        "ROC_AUC_provisional": (
            sub["ROC_AUC"].mean()
            if len(sub) else np.nan
        )
    })

tabla_progreso_C_323 = pd.DataFrame(
    filas_323
)

display(tabla_progreso_C_323.round(4))

# ------------------------------------------------------------
# 10. CONTROLES FINALES
# ------------------------------------------------------------

print("\n=== 3.23.5 CONTROLES FINALES ===")

assert len(resultados_transformer_C_323) <= 30

if not tabla_transformer_C_323.empty:

    assert not tabla_transformer_C_323.duplicated(
        subset=[
            "Semana",
            "Repeticion",
            "Fold"
        ]
    ).any()

    for clave, resultado in (
        resultados_transformer_C_323.items()
    ):

        pred = resultado["predicciones"]

        assert pred["prob_adverso"].between(
            0, 1
        ).all()

        assert not pred.duplicated(
            subset=[
                "code_module",
                "code_presentation",
                "id_student"
            ]
        ).any()

        assert (
            resultado["resumen"]["VN"]
            + resultado["resumen"]["FP"]
            + resultado["resumen"]["FN"]
            + resultado["resumen"]["VP"]
        ) == len(pred)

print(
    "Folds completados:",
    len(resultados_transformer_C_323),
    "de 30"
)

print(
    "Arquitectura e hiperparámetros congelados."
)

print(
    "Selección de épocas mediante CV interna."
)

print(
    "Preprocesamiento ajustado dentro de cada fold."
)

print(
    "Agrupación por estudiante conservada."
)

print(
    "Conjunto de prueba independiente: NO utilizado."
)

print("Checkpoint:", RUTA_323)

if len(resultados_transformer_C_323) == 30:

    print(
        "\nACTIVIDAD 3.23 COMPLETADA."
    )

else:

    print(
        "\nRESULTADOS PARCIALES."
    )

    print(
        "Vuelve a ejecutar la celda para "
        "procesar los siguientes folds."
    )



### 3.23.6. Seguimiento parcial: semana 2

Se completaron las primeras dos particiones externas del Transformer C en la semana 2, correspondientes a R1-F1 y R1-F2.

La arquitectura fue adaptada a secuencias de dos observaciones semanales, manteniendo los hiperparámetros y el procedimiento de entrenamiento definidos anteriormente.

En cada fold externo se seleccionó la duración del entrenamiento mediante tres particiones internas agrupadas por estudiante.

Las épocas finales seleccionadas fueron 20 y 13.

#### Resultados preliminares

| Modelo | F1 macro medio | Recall adverso medio | ROC-AUC medio |
|---|---:|---:|---:|
| Regresión Logística B | 0,6653 | 0,5665 | 0,7285 |
| Transformer C | 0,6653 | 0,5485 | 0,7280 |
| Random Forest B | 0,6511 | 0,5241 | 0,7164 |

El Transformer obtuvo un rendimiento de F1 macro prácticamente idéntico al de Regresión Logística B y superior al de Random Forest B en las dos particiones disponibles.

Sin embargo, estos resultados son preliminares y no permiten establecer conclusiones sobre superioridad estadística o comportamiento temporal.

Los controles de separación por estudiante, preprocesamiento y selección interna de épocas se completaron correctamente.

El conjunto de prueba independiente permaneció reservado.

**Estado del experimento:** 2 de 30 folds externos completados. La semana 2 tiene ocho folds pendientes; las semanas 4 y 6 todavía no han comenzado. Ejecutaremos la celda, varias veces hasta completar los 30 folds.


In [ ]:

# ============================================================
# PROYECTO FINAL DE MACHINE LEARNING
# ACTIVIDAD 3.23 - EVALUACIÓN TEMPORAL TRANSFORMER C
# SEMANAS 2, 4 Y 6
# ============================================================

# Asistencia IA: ChatGPT, octubre de 2026.
# Prompt:
# "Extender el Transformer C de OULAD a las semanas
# 2, 4 y 6, manteniendo la arquitectura y el protocolo
# anidado ya utilizados en semana 8. Crear checkpoints
# incrementales, conservar las particiones agrupadas
# por estudiante y no utilizar el test independiente."

import os
import pickle
import random
import time
import numpy as np
import pandas as pd
import torch

from sklearn.base import clone
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    recall_score,
    precision_score,
    roc_auc_score,
    confusion_matrix
)

from IPython.display import display

# ------------------------------------------------------------
# 1. CONFIGURACIÓN
# ------------------------------------------------------------

SEMANAS_323 = [2, 4, 6]
NUEVOS_FOLDS_323 = 5

RUTA_323 = (
    "/content/transformer_C_temporal_323.pkl"
)

# Reutilizar exactamente los parámetros de 3.20.
assert MAX_EPOCAS_320 == 30
assert PACIENCIA_320 == 6
assert LR_320 == 0.0007
assert WD_320 == 0.001

funciones_necesarias_323 = [
    "TransformerOULAD",
    "preparar_temporales_319",
    "preprocesador_estatico_319",
    "construir_loader_320",
    "entrenar_transformer_320",
    "predecir_loader_320",
    "secuencias_oulad_C",
    "cv_splits_oulad"
]

for nombre in funciones_necesarias_323:
    assert nombre in globals(), (
        f"Falta {nombre}. Ejecuta primero "
        "las actividades 3.19 y 3.20."
    )

# Las funciones existentes utilizan esta variable.
dispositivo_320 = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=== 3.23.1 CONFIGURACIÓN ===")
print("Semanas pendientes:", SEMANAS_323)
print("Dispositivo:", dispositivo_320)
print("Folds externos por semana:", 10)
print("Folds internos:", 3)
print("Folds nuevos por ejecución:", NUEVOS_FOLDS_323)

# ------------------------------------------------------------
# 2. CHECKPOINT REANUDABLE
# ------------------------------------------------------------

if os.path.exists(RUTA_323):

    with open(RUTA_323, "rb") as archivo:
        resultados_transformer_C_323 = (
            pickle.load(archivo)
        )

    print("Checkpoint recuperado.")

elif "resultados_transformer_C_323" not in globals():

    resultados_transformer_C_323 = {}

# Validar las claves recuperadas.
for clave in resultados_transformer_C_323:
    assert len(clave) == 3
    assert clave[0] in SEMANAS_323
    assert clave[1] in [1, 2]
    assert 1 <= clave[2] <= 5

# ------------------------------------------------------------
# 3. PREPROCESAMIENTO POR FOLD
# ------------------------------------------------------------

def preparar_fold_C_323(semana, idx_train, idx_val):

    datos = secuencias_oulad_C[semana]

    X_est = datos["X_estatica_train"]
    X_seq = datos["X_secuencia_train"]

    # Clonar preprocesador sin parámetros aprendidos.
    preprocesador = clone(
        preprocesador_estatico_319
    )

    est_train = (
        preprocesador.fit_transform(
            X_est.iloc[idx_train]
        ).astype(np.float32)
    )

    est_val = (
        preprocesador.transform(
            X_est.iloc[idx_val]
        ).astype(np.float32)
    )

    (
        seq_train,
        seq_val,
        mask_train,
        mask_val,
        escalador
    ) = preparar_temporales_319(
        X_seq[idx_train],
        X_seq[idx_val]
    )

    assert seq_train.shape[1] == semana
    assert seq_val.shape[1] == semana

    assert est_train.shape[1] == est_val.shape[1]

    assert np.isfinite(est_train).all()
    assert np.isfinite(est_val).all()
    assert np.isfinite(seq_train).all()
    assert np.isfinite(seq_val).all()

    return {
        "est_train": est_train,
        "est_val": est_val,
        "seq_train": seq_train,
        "seq_val": seq_val,
        "mask_train": mask_train,
        "mask_val": mask_val
    }


# ------------------------------------------------------------
# 4. CREAR MODELO CON LONGITUD CORRECTA
# ------------------------------------------------------------

def nuevo_transformer_323(
    semana,
    n_estaticas,
    semilla
):

    random.seed(semilla)
    np.random.seed(semilla)
    torch.manual_seed(semilla)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(semilla)

    modelo = TransformerOULAD(
        n_static=n_estaticas,
        n_weeks=semana,
        n_channels=4,
        d_model=32,
        n_heads=4,
        n_layers=2,
        dropout=0.15
    )

    return modelo.to(dispositivo_320)


# ------------------------------------------------------------
# 5. EJECUTAR UN FOLD EXTERNO
# ------------------------------------------------------------

def ejecutar_fold_C_323(semana, numero_split):

    inicio = time.perf_counter()

    repeticion = numero_split // 5 + 1
    fold = numero_split % 5 + 1

    datos = secuencias_oulad_C[semana]

    X_est = datos["X_estatica_train"]
    y = datos["y_train"]
    grupos = datos["grupos_train"]
    claves = datos["claves_train"]

    idx_ext_train, idx_ext_val = (
        cv_splits_oulad[semana][numero_split]
    )

    assert set(
        grupos.iloc[idx_ext_train]
    ).isdisjoint(
        set(grupos.iloc[idx_ext_val])
    )

    print(
        f"\nSEMANA {semana} | R{repeticion}-F{fold}",
        flush=True
    )

    # --------------------------------------------------------
    # 5.1 FOLDS INTERNOS
    # --------------------------------------------------------

    cv_interna = StratifiedGroupKFold(
        n_splits=3,
        shuffle=True,
        random_state=137 + numero_split
    )

    splits_internos = list(
        cv_interna.split(
            X_est.iloc[idx_ext_train],
            y.iloc[idx_ext_train],
            groups=grupos.iloc[idx_ext_train]
        )
    )

    resultados_internos = []

    for numero_interno, (
        idx_train_rel,
        idx_val_rel
    ) in enumerate(
        splits_internos,
        start=1
    ):

        idx_train = idx_ext_train[idx_train_rel]
        idx_val = idx_ext_train[idx_val_rel]

        assert set(
            grupos.iloc[idx_train]
        ).isdisjoint(
            set(grupos.iloc[idx_val])
        )

        preparados = preparar_fold_C_323(
            semana,
            idx_train,
            idx_val
        )

        semilla = (
            42
            + 10000 * semana
            + 1000 * numero_split
            + numero_interno
        )

        loader_train = construir_loader_320(
            preparados["seq_train"],
            preparados["est_train"],
            preparados["mask_train"],
            y.iloc[idx_train].to_numpy(),
            shuffle=True,
            semilla=semilla
        )

        loader_val = construir_loader_320(
            preparados["seq_val"],
            preparados["est_val"],
            preparados["mask_val"],
            y.iloc[idx_val].to_numpy()
        )

        modelo = nuevo_transformer_323(
            semana,
            preparados["est_train"].shape[1],
            semilla
        )

        entrenado = entrenar_transformer_320(
            modelo,
            loader_train,
            loader_val=loader_val,
            max_epocas=MAX_EPOCAS_320,
            paciencia=PACIENCIA_320
        )

        resultados_internos.append({
            "Fold_interno": numero_interno,
            "Mejor_epoca": entrenado[
                "mejor_epoca"
            ],
            "Loss_validacion": entrenado[
                "mejor_loss"
            ]
        })

        print(
            f"  Fold interno {numero_interno}: "
            f"época={entrenado['mejor_epoca']} | "
            f"loss={entrenado['mejor_loss']:.4f}",
            flush=True
        )

        del modelo, entrenado

    tabla_interna = pd.DataFrame(
        resultados_internos
    )

    epocas_finales = int(
        np.median(
            tabla_interna["Mejor_epoca"]
        )
    )

    assert 1 <= epocas_finales <= 30

    print(
        "Épocas internas:",
        tabla_interna["Mejor_epoca"].tolist()
    )

    print(
        "Épocas finales:",
        epocas_finales
    )

    # --------------------------------------------------------
    # 5.2 REENTRENAR SOBRE TODO EL FOLD EXTERNO
    # --------------------------------------------------------

    preparados_finales = preparar_fold_C_323(
        semana,
        idx_ext_train,
        idx_ext_val
    )

    semilla_final = (
        42
        + 10000 * semana
        + 1000 * numero_split
        + 100
    )

    loader_train_final = construir_loader_320(
        preparados_finales["seq_train"],
        preparados_finales["est_train"],
        preparados_finales["mask_train"],
        y.iloc[idx_ext_train].to_numpy(),
        shuffle=True,
        semilla=semilla_final
    )

    loader_val_externo = construir_loader_320(
        preparados_finales["seq_val"],
        preparados_finales["est_val"],
        preparados_finales["mask_val"],
        y.iloc[idx_ext_val].to_numpy()
    )

    modelo_final = nuevo_transformer_323(
        semana,
        preparados_finales["est_train"].shape[1],
        semilla_final
    )

    entrenado_final = entrenar_transformer_320(
        modelo_final,
        loader_train_final,
        loader_val=None,
        max_epocas=epocas_finales
    )

    # --------------------------------------------------------
    # 5.3 EVALUACIÓN EXTERNA
    # --------------------------------------------------------

    evaluacion = predecir_loader_320(
        entrenado_final["modelo"],
        loader_val_externo
    )

    reales = evaluacion["y"]
    prob = evaluacion["prob"]
    pred = (prob >= 0.5).astype(int)

    assert np.array_equal(
        reales,
        y.iloc[idx_ext_val].to_numpy()
    )

    vn, fp, fn, vp = confusion_matrix(
        reales,
        pred,
        labels=[0, 1]
    ).ravel()

    resumen = {
        "Semana": semana,
        "Repeticion": repeticion,
        "Fold": fold,
        "Epocas_finales": epocas_finales,
        "N_validacion": len(reales),
        "F1_macro": f1_score(
            reales,
            pred,
            average="macro",
            zero_division=0
        ),
        "Recall_adverso": recall_score(
            reales,
            pred,
            zero_division=0
        ),
        "Precision_adverso": precision_score(
            reales,
            pred,
            zero_division=0
        ),
        "ROC_AUC": roc_auc_score(
            reales,
            prob
        ),
        "Accuracy": accuracy_score(
            reales,
            pred
        ),
        "VN": int(vn),
        "FP": int(fp),
        "FN": int(fn),
        "VP": int(vp),
        "Tiempo_segundos": (
            time.perf_counter() - inicio
        )
    }

    predicciones = (
        claves.iloc[idx_ext_val]
        .reset_index(drop=True)
        .copy()
    )

    predicciones["Semana"] = semana
    predicciones["Repeticion"] = repeticion
    predicciones["Fold"] = fold
    predicciones["Experimento"] = "C"
    predicciones["Algoritmo"] = "Transformer"
    predicciones["y_real"] = reales
    predicciones["y_pred"] = pred
    predicciones["prob_adverso"] = prob

    assert not predicciones.duplicated(
        subset=[
            "code_module",
            "code_presentation",
            "id_student"
        ]
    ).any()

    print(
        f"F1={resumen['F1_macro']:.4f} | "
        f"Recall={resumen['Recall_adverso']:.4f} | "
        f"AUC={resumen['ROC_AUC']:.4f}",
        flush=True
    )

    return {
        "resumen": resumen,
        "predicciones": predicciones,
        "epocas_internas": tabla_interna
    }


# ------------------------------------------------------------
# 6. IDENTIFICAR FOLDS PENDIENTES
# ------------------------------------------------------------

pendientes_323 = []

for semana in SEMANAS_323:

    assert len(cv_splits_oulad[semana]) == 10

    for numero_split in range(10):

        repeticion = numero_split // 5 + 1
        fold = numero_split % 5 + 1

        clave = (semana, repeticion, fold)

        if clave not in resultados_transformer_C_323:
            pendientes_323.append(
                (semana, numero_split)
            )

print("\n=== 3.23.2 ESTADO DE EJECUCIÓN ===")

print(
    "Folds ya completados:",
    len(resultados_transformer_C_323)
)

print(
    "Folds pendientes:",
    len(pendientes_323)
)

# ------------------------------------------------------------
# 7. EJECUCIÓN INCREMENTAL
# ------------------------------------------------------------

for semana, numero_split in (
    pendientes_323[:NUEVOS_FOLDS_323]
):

    repeticion = numero_split // 5 + 1
    fold = numero_split % 5 + 1

    clave = (semana, repeticion, fold)

    resultado = ejecutar_fold_C_323(
        semana,
        numero_split
    )

    resultados_transformer_C_323[clave] = resultado

    # Guardar de forma atómica para reducir
    # el riesgo de perder el checkpoint.
    ruta_temporal = RUTA_323 + ".tmp"

    with open(ruta_temporal, "wb") as archivo:
        pickle.dump(
            resultados_transformer_C_323,
            archivo
        )

    os.replace(
        ruta_temporal,
        RUTA_323
    )

    respaldar_checkpoint_oulad(RUTA_323)

    print(
        "Fold guardado:",
        clave,
        flush=True
    )

# ------------------------------------------------------------
# 8. TABLA DE RESULTADOS
# ------------------------------------------------------------

print("\n=== 3.23.3 RESULTADOS ACUMULADOS ===")

if resultados_transformer_C_323:

    tabla_transformer_C_323 = pd.DataFrame([
        resultado["resumen"]
        for resultado in (
            resultados_transformer_C_323.values()
        )
    ]).sort_values(
        ["Semana", "Repeticion", "Fold"]
    ).reset_index(drop=True)

    display(
        tabla_transformer_C_323.round(4)
    )

else:

    tabla_transformer_C_323 = pd.DataFrame()

# ------------------------------------------------------------
# 9. PROGRESO POR SEMANA
# ------------------------------------------------------------

print("\n=== 3.23.4 RESUMEN POR SEMANA ===")

filas_323 = []

for semana in SEMANAS_323:

    if tabla_transformer_C_323.empty:
        sub = pd.DataFrame()
    else:
        sub = tabla_transformer_C_323.loc[
            tabla_transformer_C_323[
                "Semana"
            ].eq(semana)
        ]

    filas_323.append({
        "Semana": semana,
        "Folds_completados": len(sub),
        "Folds_pendientes": 10 - len(sub),
        "F1_macro_provisional": (
            sub["F1_macro"].mean()
            if len(sub) else np.nan
        ),
        "Recall_provisional": (
            sub["Recall_adverso"].mean()
            if len(sub) else np.nan
        ),
        "ROC_AUC_provisional": (
            sub["ROC_AUC"].mean()
            if len(sub) else np.nan
        )
    })

tabla_progreso_C_323 = pd.DataFrame(
    filas_323
)

display(tabla_progreso_C_323.round(4))

# ------------------------------------------------------------
# 10. CONTROLES FINALES
# ------------------------------------------------------------

print("\n=== 3.23.5 CONTROLES FINALES ===")

assert len(resultados_transformer_C_323) <= 30

if not tabla_transformer_C_323.empty:

    assert not tabla_transformer_C_323.duplicated(
        subset=[
            "Semana",
            "Repeticion",
            "Fold"
        ]
    ).any()

    for clave, resultado in (
        resultados_transformer_C_323.items()
    ):

        pred = resultado["predicciones"]

        assert pred["prob_adverso"].between(
            0, 1
        ).all()

        assert not pred.duplicated(
            subset=[
                "code_module",
                "code_presentation",
                "id_student"
            ]
        ).any()

        assert (
            resultado["resumen"]["VN"]
            + resultado["resumen"]["FP"]
            + resultado["resumen"]["FN"]
            + resultado["resumen"]["VP"]
        ) == len(pred)

print(
    "Folds completados:",
    len(resultados_transformer_C_323),
    "de 30"
)

print(
    "Arquitectura e hiperparámetros congelados."
)

print(
    "Selección de épocas mediante CV interna."
)

print(
    "Preprocesamiento ajustado dentro de cada fold."
)

print(
    "Agrupación por estudiante conservada."
)

print(
    "Conjunto de prueba independiente: NO utilizado."
)

print("Checkpoint:", RUTA_323)

if len(resultados_transformer_C_323) == 30:

    print(
        "\nACTIVIDAD 3.23 COMPLETADA."
    )

else:

    print(
        "\nRESULTADOS PARCIALES."
    )

    print(
        "Vuelve a ejecutar la celda para "
        "procesar los siguientes folds."
    )



### 3.23.6. Resultados e interpretación

Se completó la evaluación temporal del Transformer C en las semanas 2, 4 y 6 de OULAD, utilizando diez particiones externas por ventana, correspondientes a cinco folds y dos repeticiones.

Los modelos se entrenaron con secuencias de dos, cuatro y seis observaciones semanales, respectivamente, conservando la arquitectura Transformer Encoder y sus hiperparámetros.

En cada partición externa se utilizó validación cruzada interna agrupada por estudiante para seleccionar la duración del entrenamiento.

Los procedimientos de preprocesamiento estático y normalización temporal se ajustaron exclusivamente sobre los datos de entrenamiento correspondientes.

No se utilizó el conjunto de prueba independiente.

#### Rendimiento del Transformer C

| Semana | F1 macro | Recall adverso | ROC-AUC | Accuracy |
|---|---:|---:|---:|---:|
| 2 | 0,6654 | 0,5682 | 0,7320 | 0,6745 |
| 4 | 0,6825 | 0,5779 | 0,7546 | 0,6939 |
| 6 | 0,6953 | 0,5648 | 0,7709 | 0,7110 |
| 8 | 0,7095 | 0,5515 | 0,7924 | 0,7310 |

Los resultados de la semana 8 provienen de la Actividad 3.21.

El F1 macro aumentó de 0,6654 en la semana 2 a 0,7095 en la semana 8, lo que representa un incremento descriptivo de aproximadamente 0,0441 puntos.

El ROC-AUC también aumentó progresivamente, pasando de 0,7320 a 0,7924.

El recall adverso no mostró una tendencia monotónica: alcanzó su mayor promedio en la semana 4, con 0,5779, y descendió a 0,5515 en la semana 8.

Esta diferencia entre la evolución del ROC-AUC y del recall indica que una mejora de discriminación no necesariamente incrementa la sensibilidad cuando se mantiene un umbral fijo de clasificación de 0,5.

#### Comparación con los modelos clásicos longitudinales

| Semana | Regresión Logística B | Transformer C | Random Forest B |
|---|---:|---:|---:|
| 2 | 0,6656 | 0,6654 | 0,6566 |
| 4 | 0,6855 | 0,6825 | 0,6706 |
| 6 | 0,6997 | 0,6953 | 0,6869 |
| 8 | 0,7228 | 0,7095 | 0,7006 |

Las cifras representan promedios de F1 macro obtenidos mediante validación cruzada externa sobre las mismas particiones agrupadas por estudiante.

Regresión Logística B obtuvo el mayor F1 macro medio en las cuatro semanas.

El Transformer C presentó resultados muy próximos a Regresión Logística B en las semanas 2, 4 y 6, y una diferencia más amplia en la semana 8.

Frente a Random Forest B, el Transformer obtuvo mejores promedios de F1 macro en las cuatro ventanas.

En la semana 2, el Transformer superó a Random Forest B en los diez folds, pero únicamente superó a Regresión Logística B en cuatro.

En la semana 4, superó a Random Forest B en diez folds y a Regresión Logística B en cinco.

En la semana 6, superó a Random Forest B en diez folds y a Regresión Logística B en dos.

En la semana 8, superó a Random Forest B en nueve folds y no superó a Regresión Logística B en ninguno.

#### Interpretación del aprendizaje secuencial

El Transformer demostró capacidad para aprender patrones predictivos a partir de secuencias semanales de distinta longitud.

Sin embargo, la utilización de una arquitectura neuronal más compleja no produjo una mejora sistemática frente a Regresión Logística B.

Los resultados sugieren que las características longitudinales explícitamente diseñadas pueden constituir una representación altamente competitiva del comportamiento estudiantil, incluso frente a mecanismos de atención temporal.

La comparación no permite atribuir las diferencias exclusivamente a la arquitectura, porque los modelos utilizan representaciones de información diferentes.

### 3.23.7. Limitaciones

La evaluación temporal se realizó sobre cohortes cuya composición cambia entre semanas.

Los conjuntos de entrenamiento también varían entre ventanas, por lo que la evolución observada no constituye una estimación causal del efecto de añadir semanas de información.

Aunque los folds externos están agrupados por estudiante, las repeticiones reutilizan matrículas y sus resultados son estadísticamente dependientes.

El Transformer utiliza cuatro canales semanales, mientras que el Experimento B incorpora características longitudinales agregadas, tendencias y otras variables construidas explícitamente.

Los resultados todavía requieren consolidación sobre una cohorte común y comparaciones estadísticas pareadas entre las tres estrategias.

No se ha utilizado el conjunto de prueba independiente.

### 3.23.8. Conclusión

Se completaron satisfactoriamente las 30 evaluaciones externas del Transformer C correspondientes a las semanas 2, 4 y 6, que, junto con las diez evaluaciones de la semana 8, permiten disponer de una trayectoria temporal completa.

El F1 macro y el ROC-AUC del Transformer aumentaron conforme avanzaron las ventanas de predicción.

La arquitectura produjo resultados competitivos frente a los modelos clásicos longitudinales, superando descriptivamente a Random Forest B en las cuatro semanas, pero sin superar en promedio a Regresión Logística B.

**Conclusión principal:** el aprendizaje secuencial mediante atención permite representar de forma efectiva las trayectorias semanales de actividad estudiantil. No obstante, la mayor complejidad del Transformer no garantiza una mejora frente a la regresión logística alimentada con características longitudinales diseñadas explícitamente.

El Experimento C aporta evidencia complementaria de la utilidad de la información temporal y permite estudiar las diferencias entre estrategias de representación del comportamiento estudiantil.



### 3.24. Comparación integral de los experimentos A, B y C

#### Objetivo

Consolidar el rendimiento de los cinco modelos desarrollados y analizar su evolución temporal sobre una población común a las semanas 2, 4, 6 y 8.

Los modelos considerados serán:

- Regresión Logística A: características estáticas.
- Random Forest A: características estáticas.
- Regresión Logística B: características estáticas y longitudinales.
- Random Forest B: características estáticas y longitudinales.
- Transformer C: características estáticas y secuencias temporales semanales.

#### Primera comparación: cohortes originales

Se integrarán los resultados de validación cruzada externa obtenidos anteriormente.

Para cada modelo y semana se calcularán medias y desviaciones estándar de F1 macro, recall adverso, ROC-AUC y accuracy.

También se calcularán las diferencias de F1 macro del Transformer frente a Regresión Logística B y Random Forest B utilizando las mismas particiones.

#### Segunda comparación: cohorte común

Se identificarán las matrículas presentes en las cuatro cohortes de desarrollo.

Se recuperarán exclusivamente las predicciones fuera de muestra generadas previamente para los cinco modelos.

Las evaluaciones se restringirán a la intersección de matrículas, manteniendo sus etiquetas reales y el emparejamiento entre los modelos.

Las métricas se calcularán separadamente para cada repetición y después se promediarán, evitando contar las dos predicciones de una misma matrícula como observaciones independientes.

#### Evaluación de sensibilidad

Se compararán los resultados de las cohortes originales con los obtenidos sobre la cohorte común.

Se examinará si:

1. La ventaja longitudinal B-A se mantiene.
2. El Transformer continúa superando descriptivamente a Random Forest B.
3. Regresión Logística B conserva su ventaja media frente al Transformer.
4. La evolución temporal de F1 macro persiste cuando la población evaluada es fija.

#### Limitaciones

La cohorte común se selecciona retrospectivamente según la elegibilidad hasta la semana 8.

Esta restricción excluye a estudiantes retirados tempranamente y puede producir sesgo de supervivencia.

El análisis mantiene fija la población de evaluación, pero no las poblaciones de entrenamiento ni los modelos ajustados entre semanas.

Por tanto, constituye un análisis de sensibilidad y no una estimación causal de los beneficios de incorporar información temporal.

No se entrenarán nuevos modelos ni se utilizará el conjunto de prueba independiente.


In [ ]:

# ============================================================
# PROYECTO FINAL DE MACHINE LEARNING
# ACTIVIDAD 3.24 - COMPARACIÓN INTEGRAL A, B Y C
# COHORTES ORIGINALES Y COHORTE COMÚN
# ============================================================

# Asistencia IA: ChatGPT, octubre de 2026.
# Prompt:
# "Consolidar los resultados de validación cruzada
# de Regresión Logística A/B, Random Forest A/B y
# Transformer C para las semanas 2, 4, 6 y 8.
# Comparar los modelos sobre sus cohortes originales
# y sobre una cohorte común, reutilizando predicciones
# fuera de muestra. No reentrenar ni utilizar test."

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    f1_score,
    recall_score,
    accuracy_score
)

from IPython.display import display

SEMANAS_324 = [2, 4, 6, 8]

MODELOS_324 = [
    "LR A",
    "RF A",
    "LR B",
    "RF B",
    "Transformer C"
]

CLAVES_324 = [
    "code_module",
    "code_presentation",
    "id_student"
]

METRICAS_324 = [
    "F1_macro",
    "Recall_adverso",
    "ROC_AUC",
    "Accuracy"
]

# ------------------------------------------------------------
# 1. VERIFICAR RESULTADOS EXISTENTES
# ------------------------------------------------------------

print("=== 3.24.1 VERIFICACIÓN INICIAL ===")

variables_requeridas = [
    "tabla_lr_cv",
    "tabla_rf_cv",
    "resultados_cv_ventanas",
    "tabla_transformer_C_323",
    "tabla_transformer_C_s8",
    "resultados_transformer_C_323",
    "resultados_transformer_C_s8",
    "predicciones_conjuntas_s2",
    "experimentos_oulad"
]

faltantes = [
    nombre
    for nombre in variables_requeridas
    if nombre not in globals()
]

assert not faltantes, (
    "Faltan variables del notebook: "
    + ", ".join(faltantes)
)

assert len(resultados_transformer_C_323) == 30
assert len(resultados_transformer_C_s8) == 10

print("Resultados Transformer S2/S4/S6: 30 folds")
print("Resultados Transformer S8: 10 folds")
print("Modelos nuevos a entrenar: 0")

# ------------------------------------------------------------
# 2. CONSOLIDAR MÉTRICAS DE LOS CINCO MODELOS
# ------------------------------------------------------------

print("\n=== 3.24.2 CONSOLIDACIÓN DE MÉTRICAS ===")

tablas = []

for semana in SEMANAS_324:

    for algoritmo, prefijo in [
        ("LogisticRegression", "LR"),
        ("RandomForest", "RF")
    ]:

        if semana == 2:

            origen = (
                tabla_lr_cv
                if prefijo == "LR"
                else tabla_rf_cv
            )

        else:

            origen = resultados_cv_ventanas[
                (semana, algoritmo)
            ]["resultados"]

        for experimento in ["A", "B"]:

            sub = origen.loc[
                origen["Experimento"].eq(
                    experimento
                ),
                ["Repeticion", "Fold"] + METRICAS_324
            ].copy()

            sub["Semana"] = semana
            sub["Modelo"] = (
                f"{prefijo} {experimento}"
            )

            assert len(sub) == 10

            tablas.append(sub)

    # Recuperar Transformer C
    if semana == 8:

        transformer = (
            tabla_transformer_C_s8.copy()
        )

    else:

        transformer = (
            tabla_transformer_C_323.loc[
                tabla_transformer_C_323[
                    "Semana"
                ].eq(semana)
            ].copy()
        )

    sub_C = transformer[
        ["Repeticion", "Fold"] + METRICAS_324
    ].copy()

    sub_C["Semana"] = semana
    sub_C["Modelo"] = "Transformer C"

    assert len(sub_C) == 10

    tablas.append(sub_C)

tabla_integral_324 = pd.concat(
    tablas,
    ignore_index=True
)

assert len(tabla_integral_324) == 200

assert not tabla_integral_324.duplicated(
    subset=[
        "Semana",
        "Modelo",
        "Repeticion",
        "Fold"
    ]
).any()

tabla_resumen_integral_324 = (
    tabla_integral_324.groupby(
        ["Semana", "Modelo"]
    )
    .agg(
        F1_macro=("F1_macro", "mean"),
        F1_std=("F1_macro", "std"),
        Recall_adverso=("Recall_adverso", "mean"),
        ROC_AUC=("ROC_AUC", "mean"),
        Accuracy=("Accuracy", "mean"),
        Folds=("Fold", "size")
    )
    .reset_index()
)

display(
    tabla_resumen_integral_324.round(4)
)

# ------------------------------------------------------------
# 3. COMPARACIONES PAREADAS POR FOLD
# ------------------------------------------------------------

print("\n=== 3.24.3 COMPARACIONES PAREADAS ===")

pares_324 = tabla_integral_324.pivot(
    index=["Semana", "Repeticion", "Fold"],
    columns="Modelo",
    values="F1_macro"
)

assert len(pares_324) == 40
assert not pares_324.isna().any().any()

comparaciones = [
    ("LR B", "LR A", "B - A (LR)"),
    ("RF B", "RF A", "B - A (RF)"),
    ("Transformer C", "LR B", "C - LR B"),
    ("Transformer C", "RF B", "C - RF B")
]

filas_pares = []

for semana in SEMANAS_324:

    sub = pares_324.loc[semana]

    for primero, segundo, descripcion in comparaciones:

        delta = sub[primero] - sub[segundo]

        filas_pares.append({
            "Semana": semana,
            "Comparacion": descripcion,
            "Delta_F1_medio": delta.mean(),
            "Delta_std": delta.std(ddof=1),
            "Folds_positivos": int((delta > 0).sum()),
            "Folds_negativos": int((delta < 0).sum()),
            "Folds_empatados": int((delta == 0).sum())
        })

tabla_pares_324 = pd.DataFrame(
    filas_pares
)

display(
    tabla_pares_324.round(5)
)

# ------------------------------------------------------------
# 4. CONSTRUIR LA COHORTE COMÚN DE DESARROLLO
# ------------------------------------------------------------

print("\n=== 3.24.4 COHORTE COMÚN ===")

conjuntos_claves = []

for semana in SEMANAS_324:

    claves = (
        experimentos_oulad[semana][
            "claves_train"
        ][CLAVES_324]
        .reset_index(drop=True)
    )

    assert not claves.duplicated().any()

    conjuntos_claves.append(
        set(
            claves.itertuples(
                index=False,
                name=None
            )
        )
    )

claves_comunes_324 = set.intersection(
    *conjuntos_claves
)

cohorte_comun_324 = pd.DataFrame(
    list(claves_comunes_324),
    columns=CLAVES_324
)

n_comun_324 = len(cohorte_comun_324)

print("Matrículas comunes:", n_comun_324)

# Comprobar consistencia con la Actividad 3.14.
assert n_comun_324 == 21126, (
    "La cohorte común no coincide con la "
    "Actividad 3.14. Revisar las claves "
    "y particiones antes de continuar."
)

# ------------------------------------------------------------
# 5. FUNCIÓN PARA RECUPERAR PREDICCIONES OOF
# ------------------------------------------------------------

def obtener_predicciones_324(semana, modelo):

    if modelo == "Transformer C":

        if semana == 8:

            resultados = [
                r["predicciones"]
                for r in (
                    resultados_transformer_C_s8.values()
                )
            ]

        else:

            resultados = [
                r["predicciones"]
                for clave, r in (
                    resultados_transformer_C_323.items()
                )
                if clave[0] == semana
            ]

        assert len(resultados) == 10

        pred = pd.concat(
            resultados,
            ignore_index=True
        )

    else:

        prefijo, experimento = modelo.split()

        algoritmo = (
            "LogisticRegression"
            if prefijo == "LR"
            else "RandomForest"
        )

        if semana == 2:

            fuente = predicciones_conjuntas_s2

        else:

            fuente = resultados_cv_ventanas[
                (semana, algoritmo)
            ]["predicciones"]

        pred = fuente.loc[
            fuente["Algoritmo"].eq(algoritmo)
            & fuente["Experimento"].eq(
                experimento
            )
        ].copy()

    columnas = (
        CLAVES_324
        + ["Repeticion", "y_real", "y_pred"]
    )

    assert all(
        c in pred.columns
        for c in columnas
    )

    pred = pred[columnas].copy()

    assert not pred.duplicated(
        subset=CLAVES_324 + ["Repeticion"]
    ).any()

    n_esperado = (
        2
        * len(
            experimentos_oulad[semana][
                "y_train"
            ]
        )
    )

    assert len(pred) == n_esperado

    return pred

# ------------------------------------------------------------
# 6. EVALUACIÓN SOBRE COHORTE COMÚN
# ------------------------------------------------------------

print("\n=== 3.24.5 MÉTRICAS EN POBLACIÓN FIJA ===")

filas_comunes = []

referencia_etiquetas = None

for semana in SEMANAS_324:

    for modelo in MODELOS_324:

        pred = obtener_predicciones_324(
            semana,
            modelo
        )

        pred = pred.merge(
            cohorte_comun_324,
            on=CLAVES_324,
            how="inner",
            validate="many_to_one"
        )

        assert len(pred) == 2 * n_comun_324

        assert pred.groupby(
            "Repeticion"
        ).size().eq(n_comun_324).all()

        etiquetas = (
            pred[
                CLAVES_324 + ["y_real"]
            ]
            .drop_duplicates()
            .sort_values(CLAVES_324)
            .reset_index(drop=True)
        )

        assert len(etiquetas) == n_comun_324

        if referencia_etiquetas is None:
            referencia_etiquetas = etiquetas
        else:
            assert etiquetas.equals(
                referencia_etiquetas
            ), (
                "Se detectaron etiquetas "
                "inconsistentes entre modelos "
                "o semanas."
            )

        for repeticion, sub in pred.groupby(
            "Repeticion"
        ):

            y_real = sub["y_real"].to_numpy()
            y_pred = sub["y_pred"].to_numpy()

            filas_comunes.append({
                "Semana": semana,
                "Modelo": modelo,
                "Repeticion": repeticion,
                "Matriculas": len(sub),
                "F1_macro": f1_score(
                    y_real,
                    y_pred,
                    average="macro",
                    zero_division=0
                ),
                "Recall_adverso": recall_score(
                    y_real,
                    y_pred,
                    zero_division=0
                ),
                "Accuracy": accuracy_score(
                    y_real,
                    y_pred
                )
            })

tabla_repeticiones_comunes_324 = pd.DataFrame(
    filas_comunes
)

assert len(tabla_repeticiones_comunes_324) == 40

tabla_resumen_comun_324 = (
    tabla_repeticiones_comunes_324.groupby(
        ["Semana", "Modelo"]
    )
    .agg(
        F1_macro=("F1_macro", "mean"),
        Recall_adverso=("Recall_adverso", "mean"),
        Accuracy=("Accuracy", "mean"),
        Matriculas=("Matriculas", "first"),
        Repeticiones=("Repeticion", "nunique")
    )
    .reset_index()
)

display(
    tabla_resumen_comun_324.round(4)
)

# ------------------------------------------------------------
# 7. DIFERENCIAS SOBRE LA COHORTE COMÚN
# ------------------------------------------------------------

print("\n=== 3.24.6 DIFERENCIAS EN COHORTE COMÚN ===")

pivot_comun_324 = (
    tabla_resumen_comun_324.pivot(
        index="Semana",
        columns="Modelo",
        values="F1_macro"
    )
)

tabla_delta_comun_324 = pd.DataFrame({
    "Semana": pivot_comun_324.index,
    "Delta_LR_B_A": (
        pivot_comun_324["LR B"]
        - pivot_comun_324["LR A"]
    ).to_numpy(),
    "Delta_RF_B_A": (
        pivot_comun_324["RF B"]
        - pivot_comun_324["RF A"]
    ).to_numpy(),
    "Delta_C_LR_B": (
        pivot_comun_324["Transformer C"]
        - pivot_comun_324["LR B"]
    ).to_numpy(),
    "Delta_C_RF_B": (
        pivot_comun_324["Transformer C"]
        - pivot_comun_324["RF B"]
    ).to_numpy()
})

display(
    tabla_delta_comun_324.round(5)
)

# ------------------------------------------------------------
# 8. COMPARAR COHORTE ORIGINAL Y COMÚN
# ------------------------------------------------------------

print("\n=== 3.24.7 ANÁLISIS DE SENSIBILIDAD ===")

tabla_sensibilidad_324 = (
    tabla_resumen_integral_324[
        ["Semana", "Modelo", "F1_macro"]
    ]
    .rename(columns={
        "F1_macro": "F1_cohorte_original"
    })
    .merge(
        tabla_resumen_comun_324[
            ["Semana", "Modelo", "F1_macro"]
        ].rename(columns={
            "F1_macro": "F1_cohorte_comun"
        }),
        on=["Semana", "Modelo"],
        validate="one_to_one"
    )
)

tabla_sensibilidad_324[
    "Diferencia_comun_original"
] = (
    tabla_sensibilidad_324[
        "F1_cohorte_comun"
    ]
    - tabla_sensibilidad_324[
        "F1_cohorte_original"
    ]
)

display(
    tabla_sensibilidad_324.round(5)
)

# ------------------------------------------------------------
# 9. GRÁFICOS TEMPORALES
# ------------------------------------------------------------

print("\n=== 3.24.8 GRÁFICOS ===")

fig, ax = plt.subplots(figsize=(10, 5))

for modelo in MODELOS_324:

    sub = tabla_resumen_integral_324.loc[
        tabla_resumen_integral_324[
            "Modelo"
        ].eq(modelo)
    ].sort_values("Semana")

    ax.plot(
        sub["Semana"],
        sub["F1_macro"],
        marker="o",
        linewidth=2,
        label=modelo
    )

ax.set_title(
    "F1 macro por semana - Cohortes originales"
)
ax.set_xlabel("Semana de predicción")
ax.set_ylabel("F1 macro")
ax.set_xticks(SEMANAS_324)
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(10, 5))

for modelo in MODELOS_324:

    sub = tabla_resumen_comun_324.loc[
        tabla_resumen_comun_324[
            "Modelo"
        ].eq(modelo)
    ].sort_values("Semana")

    ax.plot(
        sub["Semana"],
        sub["F1_macro"],
        marker="o",
        linewidth=2,
        label=modelo
    )

ax.set_title(
    "F1 macro por semana - Cohorte común"
)
ax.set_xlabel("Semana de predicción")
ax.set_ylabel("F1 macro")
ax.set_xticks(SEMANAS_324)
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 10. CONTROLES FINALES
# ------------------------------------------------------------

print("\n=== 3.24.9 CONTROLES FINALES ===")

assert len(tabla_resumen_integral_324) == 20
assert len(tabla_resumen_comun_324) == 20
assert len(tabla_pares_324) == 16
assert len(tabla_delta_comun_324) == 4

assert tabla_resumen_integral_324[
    "Folds"
].eq(10).all()

assert tabla_resumen_comun_324[
    "Repeticiones"
].eq(2).all()

assert tabla_resumen_comun_324[
    "Matriculas"
].eq(n_comun_324).all()

print("Modelos evaluados: 5")
print("Semanas evaluadas: 4")
print("Evaluaciones externas originales: 200")
print("Matrículas comunes:", n_comun_324)
print("Predicciones utilizadas: fuera de muestra.")
print("Comparaciones emparejadas: verificadas.")
print("Modelos reentrenados: 0")
print("Conjunto de prueba independiente: NO utilizado")

print(
    "\nLimitación: cohorte común retrospectiva. "
    "Los modelos y sus poblaciones de entrenamiento "
    "siguen variando entre semanas."
)

print("\n=== ACTIVIDAD 3.24 FINALIZADA ===")



### 3.24.10. Resultados e interpretación

Se completó la comparación integral de los experimentos A, B y C en las semanas 2, 4, 6 y 8.

El análisis consolidó 200 evaluaciones externas correspondientes a cinco modelos, cuatro ventanas temporales y diez particiones por configuración.

Se incluyeron dos algoritmos con características estáticas, dos algoritmos con características estáticas y longitudinales agregadas y un Transformer Encoder con información estática y secuencias semanales.

#### Evaluación sobre cohorte común

Se identificaron 21.126 matrículas presentes en las cuatro cohortes de desarrollo.

Se recuperaron sus predicciones fuera de muestra y se recalcularon F1 macro, recall adverso y accuracy sobre una población de evaluación fija.

| Modelo | Semana 2 | Semana 4 | Semana 6 | Semana 8 |
|---|---:|---:|---:|---:|
| Regresión Logística A | 0,6043 | 0,6028 | 0,5977 | 0,5907 |
| Random Forest A | 0,5954 | 0,5941 | 0,5879 | 0,5817 |
| Regresión Logística B | 0,6587 | 0,6808 | 0,6974 | 0,7229 |
| Random Forest B | 0,6509 | 0,6670 | 0,6851 | 0,7006 |
| Transformer C | 0,6592 | 0,6787 | 0,6936 | 0,7102 |

Los modelos estáticos presentaron una disminución descriptiva del F1 macro conforme avanzaron las ventanas de predicción.

Por el contrario, los tres modelos que incorporaron información longitudinal presentaron una evolución creciente.

#### Aporte longitudinal B-A

En la cohorte común, la ventaja de Regresión Logística B sobre su versión estática aumentó de 0,0544 en la semana 2 a 0,1322 en la semana 8.

En Random Forest, la ventaja aumentó de 0,0555 a 0,1189.

Estas diferencias permanecieron positivas en las cuatro ventanas.

#### Comportamiento del Transformer C

El Transformer presentó un rendimiento competitivo frente a los modelos clásicos longitudinales.

En la semana 2 obtuvo un F1 macro de 0,6592, prácticamente equivalente al de Regresión Logística B, que alcanzó 0,6587.

En las semanas 4, 6 y 8, Regresión Logística B presentó mejores promedios.

El Transformer superó descriptivamente a Random Forest B en las cuatro ventanas.

Estos resultados indican que la arquitectura secuencial puede aprender señales predictivas relevantes, pero no demuestran una ventaja sistemática frente a la regresión logística con características longitudinales diseñadas explícitamente.

#### Sensibilidad a la composición de la población

Las diferencias entre las métricas de las cohortes originales y de la cohorte común fueron reducidas.

La ventaja longitudinal B-A y la tendencia temporal creciente persistieron después de restringir la evaluación a las mismas matrículas.

Este resultado proporciona evidencia complementaria de robustez descriptiva frente a cambios de composición de la población evaluada.

### 3.24.11. Limitaciones

La cohorte común se seleccionó retrospectivamente según elegibilidad hasta la semana 8, lo que puede introducir sesgo de supervivencia y excluir matrículas correspondientes a retiros tempranos.

La restricción mantiene fija la población de evaluación, pero los modelos fueron entrenados sobre cohortes que varían entre semanas.

Los resultados no permiten establecer relaciones causales entre el tiempo de observación y la mejora predictiva.

El ROC-AUC no se recalculó sobre la cohorte común; sus valores originales corresponden a las cohortes completas de cada semana.

Las comparaciones del Transformer frente a los modelos clásicos todavía requieren contrastes estadísticos que contemplen la dependencia entre particiones y las comparaciones múltiples.

El conjunto de prueba independiente permaneció reservado.

### 3.24.12. Conclusión

La comparación integral muestra que los modelos con información longitudinal superan consistentemente a los modelos estáticos en F1 macro.

La tendencia se conserva al restringir la evaluación a una cohorte común de 21.126 matrículas.

Regresión Logística B obtuvo el mejor F1 macro medio en las semanas 4, 6 y 8, mientras que en la semana 2 el Transformer C y Regresión Logística B presentaron resultados prácticamente equivalentes.

**Conclusión principal:** la información longitudinal del comportamiento estudiantil aporta valor predictivo adicional y consistente frente a la información exclusivamente estática. El Transformer constituye una alternativa competitiva, aunque su mayor complejidad no se traduce en una superioridad sistemática frente a Regresión Logística B.

Estos hallazgos respaldan la hipótesis principal en el conjunto de desarrollo, con las limitaciones propias de la validación cruzada repetida y la selección retrospectiva de la cohorte común.



### 3.25. Comparación estadística temporal del Transformer C

#### Objetivo

Cuantificar las diferencias de rendimiento entre Transformer C y los modelos longitudinales clásicos durante las semanas 2, 4, 6 y 8.

Se realizarán dos análisis complementarios:

1. Contrastes estadísticos corregidos sobre las diez particiones externas de cada semana.
2. Bootstrap pareado agrupado por estudiante sobre la cohorte común de 21.126 matrículas.

#### Hipótesis

Para cada semana se contrastarán las siguientes hipótesis:

**Comparación C frente a Regresión Logística B**

- H0: la diferencia media de F1 macro es igual a cero.
- H1: la diferencia media de F1 macro es diferente de cero.

**Comparación C frente a Random Forest B**

- H0: la diferencia media de F1 macro es igual a cero.
- H1: la diferencia media de F1 macro es diferente de cero.

#### Contrastes de validación cruzada

Se utilizará una prueba t corregida para particiones correlacionadas de validación cruzada repetida.

Se calcularán ocho contrastes: dos comparaciones por cada una de las cuatro semanas.

Los valores p se ajustarán conjuntamente mediante Holm para controlar la multiplicidad de esta familia de contrastes.

Se reportarán diferencias de F1 macro, desviaciones estándar, intervalos individuales aproximados del 95 % y valores p ajustados.

#### Bootstrap de cohorte común

Se utilizarán las predicciones fuera de muestra recuperadas en la Actividad 3.24.

Se realizarán 1.000 remuestreos de estudiantes completos con reemplazo.

Los mismos estudiantes remuestreados se utilizarán en las cuatro semanas y en los tres modelos para conservar el emparejamiento temporal.

En cada remuestreo:

1. Se reconstruirán las matrices de confusión.
2. Se calculará el F1 macro de cada repetición.
3. Se promediarán las dos repeticiones.
4. Se calcularán las diferencias Transformer C - Regresión Logística B y Transformer C - Random Forest B.

Se obtendrán intervalos percentiles individuales del 95 %.

También se analizará el cambio de estas diferencias entre las semanas 2 y 8.

#### Interpretación

Los contrastes corregidos y el bootstrap estiman incertidumbres distintas.

La prueba t incorpora una corrección aproximada por dependencia entre particiones.

El bootstrap considera el agrupamiento por estudiante, pero se condiciona a los modelos y predicciones ya obtenidos.

El bootstrap no incorpora la incertidumbre completa del reentrenamiento ni de la selección de hiperparámetros.

Las comparaciones tienen carácter exploratorio, porque se realizan después de haber inspeccionado los resultados del conjunto de desarrollo.

#### Prevención de fuga de información

No se entrenarán nuevos modelos.

No se optimizarán hiperparámetros ni umbrales de clasificación.

No se utilizará el conjunto de prueba independiente.

**Referencias metodológicas:** para comparar estimaciones sobre particiones repetidas se consideran las limitaciones de dependencia descritas por Nadeau y Bengio [5]. El ajuste de múltiples contrastes utiliza el procedimiento de Holm [6].


In [ ]:

# ============================================================
# PROYECTO FINAL DE MACHINE LEARNING
# ACTIVIDAD 3.25 - COMPARACIÓN ESTADÍSTICA TEMPORAL
# TRANSFORMER C VS LR B Y RF B
# ============================================================

# Asistencia IA: ChatGPT, octubre de 2026.
# Prompt:
# "Extender los contrastes estadísticos del Transformer
# a las semanas 2, 4, 6 y 8. Emplear prueba t corregida
# y ajuste Holm para ocho comparaciones de F1 macro.
# Complementar con bootstrap pareado de 1.000 muestras
# sobre una cohorte común, remuestreando estudiantes
# completos y utilizando los mismos pesos en las
# cuatro ventanas temporales.
# Reutilizar predicciones OOF. No entrenar modelos
# ni abrir el conjunto de prueba independiente."

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import t as distribucion_t
from IPython.display import display

# ------------------------------------------------------------
# 1. CONFIGURACIÓN
# ------------------------------------------------------------

SEMANAS_325 = [2, 4, 6, 8]

MODELOS_325 = [
    "LR B",
    "RF B",
    "Transformer C"
]

CLAVES_325 = [
    "code_module",
    "code_presentation",
    "id_student"
]

N_BOOTSTRAP_325 = 1000
SEMILLA_325 = 2033
ALFA_325 = 0.05

variables_325 = [
    "tabla_integral_324",
    "tabla_resumen_comun_324",
    "cohorte_comun_324",
    "obtener_predicciones_324",
    "cv_splits_oulad"
]

faltantes_325 = [
    v for v in variables_325
    if v not in globals()
]

assert not faltantes_325, (
    "Faltan resultados de la Actividad 3.24: "
    + ", ".join(faltantes_325)
)

print("=== 3.25.1 CONFIGURACIÓN ===")

print("Semanas:", SEMANAS_325)
print("Modelos:", MODELOS_325)
print("Contrastes previstos:", 8)
print("Remuestreos bootstrap:", N_BOOTSTRAP_325)
print("Alfa:", ALFA_325)

# ------------------------------------------------------------
# 2. RECUPERAR MÉTRICAS DE FOLDS EXTERNOS
# ------------------------------------------------------------

print("\n=== 3.25.2 MÉTRICAS EMPAREJADAS ===")

original_325 = tabla_integral_324.loc[
    tabla_integral_324[
        "Modelo"
    ].isin(MODELOS_325)
].copy()

assert len(original_325) == 120

pivot_325 = original_325.pivot(
    index=["Semana", "Repeticion", "Fold"],
    columns="Modelo",
    values="F1_macro"
)

assert pivot_325.shape == (40, 3)

assert not pivot_325.isna().any().any()

display(
    pivot_325.groupby(level="Semana")
    .mean()
    .round(4)
)

# ------------------------------------------------------------
# 3. CONTRASTES T CORREGIDOS
# ------------------------------------------------------------

print("\n=== 3.25.3 CONTRASTES CORREGIDOS ===")

filas_contrastes_325 = []

for semana in SEMANAS_325:

    datos_semana = pivot_325.loc[semana]

    splits = cv_splits_oulad[semana]

    assert len(splits) == 10
    assert len(datos_semana) == 10

    ratio_val_train = np.mean([
        len(idx_val) / len(idx_train)
        for idx_train, idx_val in splits
    ])

    n = len(datos_semana)

    # Corrección aproximada para folds
    # correlacionados de validación repetida.
    factor_correccion = (
        (1.0 / n)
        + ratio_val_train
    )

    for referencia in ["LR B", "RF B"]:

        delta = (
            datos_semana["Transformer C"]
            - datos_semana[referencia]
        ).to_numpy(dtype=float)

        media = delta.mean()
        desviacion = delta.std(ddof=1)

        error_corregido = (
            np.sqrt(factor_correccion)
            * desviacion
        )

        grados_libertad = n - 1

        if error_corregido > 0:

            t_stat = media / error_corregido

            p_valor = 2 * distribucion_t.sf(
                abs(t_stat),
                df=grados_libertad
            )

            t_critico = distribucion_t.ppf(
                1 - ALFA_325 / 2,
                df=grados_libertad
            )

            ic_inf = (
                media - t_critico * error_corregido
            )

            ic_sup = (
                media + t_critico * error_corregido
            )

        else:

            t_stat = np.nan

            p_valor = (
                1.0 if np.isclose(media, 0)
                else 0.0
            )

            ic_inf = media
            ic_sup = media

        filas_contrastes_325.append({
            "Semana": semana,
            "Comparacion": (
                f"Transformer C - {referencia}"
            ),
            "Delta_F1_medio": media,
            "Delta_std": desviacion,
            "Folds_C_supera": int(
                (delta > 0).sum()
            ),
            "Folds_C_inferior": int(
                (delta < 0).sum()
            ),
            "Ratio_val_train": ratio_val_train,
            "t_corregida": t_stat,
            "IC95_inferior": ic_inf,
            "IC95_superior": ic_sup,
            "p_sin_ajustar": p_valor
        })

tabla_contrastes_325 = pd.DataFrame(
    filas_contrastes_325
)

assert len(tabla_contrastes_325) == 8

# ------------------------------------------------------------
# 4. AJUSTE DE HOLM SOBRE LOS OCHO CONTRASTES
# ------------------------------------------------------------

p_originales = tabla_contrastes_325[
    "p_sin_ajustar"
].to_numpy(dtype=float)

orden = np.argsort(p_originales)

p_ordenados = p_originales[orden]

m = len(p_ordenados)

p_holm_ordenados = np.maximum.accumulate(
    p_ordenados * (
        m - np.arange(m)
    )
)

p_holm = np.empty(m, dtype=float)

p_holm[orden] = np.minimum(
    p_holm_ordenados,
    1.0
)

tabla_contrastes_325["p_Holm"] = p_holm

tabla_contrastes_325[
    "Significativo_Holm"
] = (
    tabla_contrastes_325["p_Holm"]
    < ALFA_325
)

display(
    tabla_contrastes_325.round(6)
)

print(
    "Contrastes significativos después de Holm:",
    int(
        tabla_contrastes_325[
            "Significativo_Holm"
        ].sum()
    ),
    "de 8"
)

print(
    "Nota: intervalos individuales, "
    "no ajustados por multiplicidad."
)

# ------------------------------------------------------------
# 5. PREPARAR COHORTE COMÚN
# ------------------------------------------------------------

print("\n=== 3.25.4 COHORTE COMÚN ===")

cohorte_325 = (
    cohorte_comun_324[
        CLAVES_325
    ]
    .copy()
)

assert not cohorte_325.duplicated().any()

assert len(cohorte_325) == 21126

estudiantes_325 = np.sort(
    cohorte_325[
        "id_student"
    ].unique()
)

n_estudiantes_325 = len(
    estudiantes_325
)

n_matriculas_325 = len(
    cohorte_325
)

print(
    "Matrículas comunes:",
    n_matriculas_325
)

print(
    "Estudiantes únicos:",
    n_estudiantes_325
)

# ------------------------------------------------------------
# 6. MATRICES DE CONFUSIÓN POR ESTUDIANTE
# ------------------------------------------------------------

print("\n=== 3.25.5 PREPARACIÓN BOOTSTRAP ===")

# Dimensiones:
# semana, modelo, repetición,
# estudiante, TN/FP/FN/TP

matrices_325 = np.zeros(
    (
        4,
        3,
        2,
        n_estudiantes_325,
        4
    ),
    dtype=np.int32
)

conteos_columnas_325 = [
    "TN",
    "FP",
    "FN",
    "TP"
]

referencia_etiquetas_325 = None

for si, semana in enumerate(SEMANAS_325):

    for mi, modelo in enumerate(MODELOS_325):

        pred = obtener_predicciones_324(
            semana,
            modelo
        )

        pred = pred.merge(
            cohorte_325,
            on=CLAVES_325,
            how="inner",
            validate="many_to_one"
        )

        assert len(pred) == (
            2 * n_matriculas_325
        )

        assert not pred.duplicated(
            subset=CLAVES_325 + ["Repeticion"]
        ).any()

        for ri, repeticion in enumerate([1, 2]):

            sub = pred.loc[
                pred[
                    "Repeticion"
                ].eq(repeticion)
            ].copy()

            assert len(sub) == (
                n_matriculas_325
            )

            # Comprobar las mismas etiquetas
            # y matrículas en todas las semanas.
            etiquetas_ordenadas = (
                sub[
                    CLAVES_325 + ["y_real"]
                ]
                .sort_values(CLAVES_325)
                .reset_index(drop=True)
            )

            if referencia_etiquetas_325 is None:

                referencia_etiquetas_325 = (
                    etiquetas_ordenadas
                )

            else:

                assert etiquetas_ordenadas.equals(
                    referencia_etiquetas_325
                ), (
                    "Etiquetas o matrículas "
                    "no coinciden entre modelos."
                )

            y = sub["y_real"].to_numpy(
                dtype=int
            )

            p = sub["y_pred"].to_numpy(
                dtype=int
            )

            cuentas = pd.DataFrame({
                "id_student": (
                    sub["id_student"].to_numpy()
                ),
                "TN": (
                    ((y == 0) & (p == 0))
                    .astype(np.int32)
                ),
                "FP": (
                    ((y == 0) & (p == 1))
                    .astype(np.int32)
                ),
                "FN": (
                    ((y == 1) & (p == 0))
                    .astype(np.int32)
                ),
                "TP": (
                    ((y == 1) & (p == 1))
                    .astype(np.int32)
                )
            })

            agrupado = (
                cuentas.groupby(
                    "id_student"
                )[conteos_columnas_325]
                .sum()
                .reindex(
                    estudiantes_325,
                    fill_value=0
                )
            )

            matrices_325[
                si, mi, ri, :, :
            ] = agrupado.to_numpy(
                dtype=np.int32
            )

assert np.all(
    matrices_325.sum(axis=(3, 4))
    == n_matriculas_325
)

print(
    "Matrices agrupadas construidas:",
    matrices_325.shape
)

# ------------------------------------------------------------
# 7. MÉTRICA F1 MACRO DESDE CONTEOS
# ------------------------------------------------------------

def f1_macro_conteos_325(c):

    tn = c[..., 0]
    fp = c[..., 1]
    fn = c[..., 2]
    tp = c[..., 3]

    f1_positivo = np.divide(
        2 * tp,
        2 * tp + fp + fn,
        out=np.zeros_like(
            tp,
            dtype=float
        ),
        where=(2 * tp + fp + fn) > 0
    )

    f1_negativo = np.divide(
        2 * tn,
        2 * tn + fp + fn,
        out=np.zeros_like(
            tn,
            dtype=float
        ),
        where=(2 * tn + fp + fn) > 0
    )

    return (
        f1_positivo + f1_negativo
    ) / 2

# ------------------------------------------------------------
# 8. ESTIMACIONES EN COHORTE COMÚN
# ------------------------------------------------------------

conteos_originales_325 = (
    matrices_325.sum(
        axis=3,
        dtype=np.int64
    )
)

f1_original_325 = (
    f1_macro_conteos_325(
        conteos_originales_325
    )
    .mean(axis=2)
)

# Columnas de modelo:
# 0 = LR B
# 1 = RF B
# 2 = Transformer C

deltas_originales_325 = np.column_stack([
    f1_original_325[:, 2]
    - f1_original_325[:, 0],

    f1_original_325[:, 2]
    - f1_original_325[:, 1]
])

# Comprobar consistencia con 3.24.
for si, semana in enumerate(SEMANAS_325):

    for mi, modelo in enumerate(MODELOS_325):

        referencia = (
            tabla_resumen_comun_324.loc[
                (
                    tabla_resumen_comun_324[
                        "Semana"
                    ].eq(semana)
                )
                &
                (
                    tabla_resumen_comun_324[
                        "Modelo"
                    ].eq(modelo)
                ),
                "F1_macro"
            ]
        )

        assert len(referencia) == 1

        assert np.isclose(
            f1_original_325[si, mi],
            referencia.iloc[0],
            atol=1e-8
        ), (
            "F1 no coincide con la "
            "Actividad 3.24."
        )

# ------------------------------------------------------------
# 9. BOOTSTRAP COMPARTIDO ENTRE SEMANAS
# ------------------------------------------------------------

print("\n=== 3.25.6 BOOTSTRAP COHORTE COMÚN ===")

rng_325 = np.random.default_rng(
    SEMILLA_325
)

# Dimensiones:
# bootstrap, semana, comparación

deltas_bootstrap_325 = np.empty(
    (N_BOOTSTRAP_325, 4, 2),
    dtype=float
)

for b in range(N_BOOTSTRAP_325):

    # Una muestra de estudiantes compartida
    # por las cuatro semanas.
    indices = rng_325.integers(
        0,
        n_estudiantes_325,
        size=n_estudiantes_325
    )

    pesos = np.bincount(
        indices,
        minlength=n_estudiantes_325
    ).astype(np.int64)

    # Los pesos son iguales para todas
    # las semanas, modelos y repeticiones.
    conteos_boot = np.einsum(
        "g,smrgk->smrk",
        pesos,
        matrices_325,
        dtype=np.int64
    )

    f1_boot = (
        f1_macro_conteos_325(
            conteos_boot
        )
        .mean(axis=2)
    )

    deltas_bootstrap_325[b, :, 0] = (
        f1_boot[:, 2]
        - f1_boot[:, 0]
    )

    deltas_bootstrap_325[b, :, 1] = (
        f1_boot[:, 2]
        - f1_boot[:, 1]
    )

assert np.isfinite(
    deltas_bootstrap_325
).all()

# ------------------------------------------------------------
# 10. INTERVALOS BOOTSTRAP POR SEMANA
# ------------------------------------------------------------

print("\n=== 3.25.7 INTERVALOS BOOTSTRAP ===")

filas_boot_325 = []

nombres_comparacion_325 = [
    "Transformer C - LR B",
    "Transformer C - RF B"
]

for si, semana in enumerate(SEMANAS_325):

    for ci, comparacion in enumerate(
        nombres_comparacion_325
    ):

        valores = deltas_bootstrap_325[
            :, si, ci
        ]

        ic = np.quantile(
            valores,
            [0.025, 0.975]
        )

        filas_boot_325.append({
            "Semana": semana,
            "Comparacion": comparacion,
            "Delta_F1": (
                deltas_originales_325[si, ci]
            ),
            "IC95_inferior": ic[0],
            "IC95_superior": ic[1],
            "Fraccion_delta_positivo": (
                (valores > 0).mean()
            )
        })

tabla_bootstrap_325 = pd.DataFrame(
    filas_boot_325
)

display(
    tabla_bootstrap_325.round(5)
)

# ------------------------------------------------------------
# 11. CAMBIO DE LA DIFERENCIA ENTRE SEMANAS 2 Y 8
# ------------------------------------------------------------

print("\n=== 3.25.8 CAMBIO TEMPORAL DE LAS DIFERENCIAS ===")

filas_cambio_325 = []

for ci, comparacion in enumerate(
    nombres_comparacion_325
):

    cambio_bootstrap = (
        deltas_bootstrap_325[:, 3, ci]
        - deltas_bootstrap_325[:, 0, ci]
    )

    cambio_original = (
        deltas_originales_325[3, ci]
        - deltas_originales_325[0, ci]
    )

    ic = np.quantile(
        cambio_bootstrap,
        [0.025, 0.975]
    )

    filas_cambio_325.append({
        "Comparacion": comparacion,
        "Cambio_S8_menos_S2": cambio_original,
        "IC95_inferior": ic[0],
        "IC95_superior": ic[1],
        "Fraccion_cambio_positivo": (
            (cambio_bootstrap > 0).mean()
        )
    })

tabla_cambio_temporal_325 = pd.DataFrame(
    filas_cambio_325
)

display(
    tabla_cambio_temporal_325.round(5)
)

print(
    "Nota: cambios descriptivos condicionados "
    "a predicciones ya entrenadas."
)

# ------------------------------------------------------------
# 12. GRÁFICOS DE DIFERENCIAS
# ------------------------------------------------------------

print("\n=== 3.25.9 GRÁFICOS ===")

fig, ax = plt.subplots(figsize=(10, 5))

for comparacion in nombres_comparacion_325:

    sub = tabla_bootstrap_325.loc[
        tabla_bootstrap_325[
            "Comparacion"
        ].eq(comparacion)
    ].sort_values("Semana")

    error_inferior = (
        sub["Delta_F1"]
        - sub["IC95_inferior"]
    ).clip(lower=0)

    error_superior = (
        sub["IC95_superior"]
        - sub["Delta_F1"]
    ).clip(lower=0)

    ax.errorbar(
        sub["Semana"],
        sub["Delta_F1"],
        yerr=np.vstack([
            error_inferior.to_numpy(),
            error_superior.to_numpy()
        ]),
        marker="o",
        capsize=4,
        linewidth=2,
        label=comparacion
    )

ax.axhline(
    0,
    linestyle="--",
    linewidth=1
)

ax.set_title(
    "Transformer C frente a modelos clásicos\n"
    "Bootstrap pareado sobre cohorte común"
)

ax.set_xlabel("Semana de predicción")
ax.set_ylabel("Diferencia de F1 macro")

ax.set_xticks(SEMANAS_325)
ax.grid(alpha=0.3)
ax.legend()

plt.tight_layout()
plt.show()

# Gráfico de los contrastes t corregidos.
fig, ax = plt.subplots(figsize=(10, 5))

for comparacion in nombres_comparacion_325:

    sub = tabla_contrastes_325.loc[
        tabla_contrastes_325[
            "Comparacion"
        ].eq(comparacion)
    ].sort_values("Semana")

    error_inferior = (
        sub["Delta_F1_medio"]
        - sub["IC95_inferior"]
    ).clip(lower=0)

    error_superior = (
        sub["IC95_superior"]
        - sub["Delta_F1_medio"]
    ).clip(lower=0)

    ax.errorbar(
        sub["Semana"],
        sub["Delta_F1_medio"],
        yerr=np.vstack([
            error_inferior.to_numpy(),
            error_superior.to_numpy()
        ]),
        marker="s",
        capsize=4,
        linewidth=2,
        label=comparacion
    )

ax.axhline(
    0,
    linestyle="--",
    linewidth=1
)

ax.set_title(
    "Diferencias de F1 macro\n"
    "Intervalos t corregidos por dependencia de folds"
)

ax.set_xlabel("Semana de predicción")
ax.set_ylabel("Diferencia de F1 macro")

ax.set_xticks(SEMANAS_325)
ax.grid(alpha=0.3)
ax.legend()

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 13. CONTROLES FINALES
# ------------------------------------------------------------

print("\n=== 3.25.10 CONTROLES FINALES ===")

assert len(tabla_contrastes_325) == 8

assert len(tabla_bootstrap_325) == 8

assert len(tabla_cambio_temporal_325) == 2

assert np.all(
    matrices_325.sum(axis=(3, 4))
    == n_matriculas_325
)

assert np.isfinite(
    tabla_bootstrap_325[
        [
            "Delta_F1",
            "IC95_inferior",
            "IC95_superior"
        ]
    ].to_numpy()
).all()

print("Contrastes corregidos:", 8)
print("Ajuste Holm: aplicado a las 8 comparaciones")
print("Bootstrap por comparación:", N_BOOTSTRAP_325)
print("Unidad de remuestreo: estudiante")
print("Matrículas comunes:", n_matriculas_325)
print("Mismos pesos bootstrap en las cuatro semanas")
print("Modelos reentrenados: 0")
print("Conjunto de prueba independiente: NO utilizado")

print(
    "Limitación: intervalos bootstrap condicionados "
    "a las predicciones fuera de muestra disponibles."
)

print("\n=== ACTIVIDAD 3.25 FINALIZADA ===")



### 3.26. Identificación temprana de resultados adversos y anticipación de retiros

#### Objetivo

Evaluar la utilidad de los modelos A, B y C como posibles sistemas de alerta temprana, examinando la identificación de los dos subtipos adversos de OULAD: `Fail` y `Withdrawn`.

Se utilizarán las predicciones fuera de muestra de los cinco modelos en las semanas 2, 4, 6 y 8.

#### Evaluación por subtipo académico

Se calculará la proporción de alertas emitidas para cada resultado final:

- `Fail`: proporción de matrículas con reprobación posterior que recibieron una alerta.
- `Withdrawn`: proporción de matrículas con retiro posterior que recibieron una alerta.
- `Pass`: proporción de matrículas aprobadas que recibieron una alerta incorrecta.
- `Distinction`: proporción de matrículas con distinción que recibieron una alerta incorrecta.

Los subtipos se utilizarán únicamente para evaluación retrospectiva. El objetivo de entrenamiento original continuará siendo la clasificación binaria entre resultados favorables y adversos.

#### Anticipación de retiros

Para las matrículas con resultado `Withdrawn`, se calculará:

**Anticipación = fecha efectiva de retiro - día de predicción**

Los días de predicción serán 14, 28, 42 y 56.

Se analizarán las alertas emitidas al menos 7, 14 y 28 días antes de un retiro efectivo.

Se distinguirán dos denominadores:

1. Todos los retiros futuros de la cohorte, que permiten medir la cobertura total de alertas oportunas.
2. Los retiros que todavía disponían de al menos el horizonte solicitado, que permiten medir la sensibilidad entre los casos temporalmente elegibles para recibir una alerta con esa anticipación.

La mediana de anticipación se calculará exclusivamente entre los retiros correctamente identificados.

#### Alertas incorrectas

Se analizará la proporción de alertas emitidas sobre matrículas cuyo resultado final fue `Pass` o `Distinction`.

También se calculará la precisión global de las alertas, la proporción de estudiantes identificados como adversos y el recall adverso.

#### Metodología

Se conservarán las predicciones fuera de muestra y el umbral de clasificación original de 0,5.

Las métricas se calcularán por repetición y después se promediarán. No se tratarán las dos predicciones de una misma matrícula como observaciones independientes.

La fecha de retiro se utilizará exclusivamente para evaluación retrospectiva, nunca como característica predictiva.

#### Consideraciones

`Withdrawn` representa el retiro de una matrícula académica en OULAD, no necesariamente el abandono definitivo de una institución universitaria.

La anticipación de un retiro únicamente puede calcularse cuando se dispone de una fecha efectiva de desinscripción.

Para `Fail`, OULAD proporciona el resultado final, pero este experimento no establece una fecha individual verificable de ocurrencia de la reprobación.

Las alertas se consideran simulaciones retrospectivas. No se ha realizado ninguna intervención real sobre estudiantes.

#### Alcance

Esta actividad no entrenará modelos nuevos, no optimizará umbrales y no utilizará el conjunto de prueba independiente.


In [ ]:

# ============================================================
# PROYECTO FINAL DE MACHINE LEARNING
# ACTIVIDAD 3.26
# ALERTAS TEMPRANAS Y ANTICIPACIÓN DE RETIROS
# ============================================================

# Asistencia IA: ChatGPT, octubre de 2026.
# Prompt:
# "Evaluar las predicciones OOF de los cinco modelos
# de OULAD en semanas 2, 4, 6 y 8. Calcular recall
# retrospectivo por subtipo Fail y Withdrawn, falsas
# alertas para Pass y Distinction, y anticipación de
# retiros de 7, 14 y 28 días. No reentrenar modelos,
# no cambiar umbrales y no utilizar el test."

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

# ------------------------------------------------------------
# 1. CONFIGURACIÓN
# ------------------------------------------------------------

SEMANAS_326 = [2, 4, 6, 8]

MODELOS_326 = [
    "LR A",
    "RF A",
    "LR B",
    "RF B",
    "Transformer C"
]

RESULTADOS_326 = [
    "Pass",
    "Distinction",
    "Fail",
    "Withdrawn"
]

CLAVES_326 = [
    "code_module",
    "code_presentation",
    "id_student"
]

HORIZONTES_326 = [7, 14, 28]

assert "obtener_predicciones_324" in globals()
assert "oulad_tablas" in globals()
assert "experimentos_oulad" in globals()

def porcentaje_326(numerador, denominador):
    if denominador == 0:
        return np.nan
    return 100.0 * numerador / denominador

print("=== 3.26.1 CONFIGURACIÓN ===")
print("Semanas:", SEMANAS_326)
print("Modelos:", MODELOS_326)
print("Horizontes de retiro:", HORIZONTES_326)
print("Umbral: predicciones originales sin modificar")

# ------------------------------------------------------------
# 2. RECUPERAR METADATOS ACADÉMICOS
# ------------------------------------------------------------

info_326 = oulad_tablas["studentInfo.csv"]

registro_326 = oulad_tablas[
    "studentRegistration.csv"
]

metadatos_326 = (
    info_326[
        CLAVES_326 + ["final_result"]
    ]
    .merge(
        registro_326[
            CLAVES_326 + ["date_unregistration"]
        ],
        on=CLAVES_326,
        how="inner",
        validate="one_to_one"
    )
)

assert not metadatos_326.duplicated(
    subset=CLAVES_326
).any()

assert set(
    metadatos_326["final_result"].unique()
).issubset(set(RESULTADOS_326))

metadatos_326["date_unregistration"] = (
    pd.to_numeric(
        metadatos_326["date_unregistration"],
        errors="coerce"
    )
)

print("\n=== 3.26.2 METADATOS ===")
print("Matrículas con metadatos:", len(metadatos_326))
print("Etiquetas originales:", RESULTADOS_326)

# ------------------------------------------------------------
# 3. RECUPERAR PREDICCIONES POR MODELO Y SEMANA
# ------------------------------------------------------------

predicciones_326 = []

for semana in SEMANAS_326:

    dia_corte = 7 * semana

    n_desarrollo = len(
        experimentos_oulad[semana]["y_train"]
    )

    for modelo in MODELOS_326:

        pred = obtener_predicciones_324(
            semana,
            modelo
        )

        assert len(pred) == 2 * n_desarrollo

        assert set(
            pred["Repeticion"].unique()
        ) == {1, 2}

        pred = pred.merge(
            metadatos_326,
            on=CLAVES_326,
            how="left",
            validate="many_to_one"
        )

        assert pred["final_result"].notna().all()

        assert not pred.duplicated(
            subset=CLAVES_326 + ["Repeticion"]
        ).any()

        # La etiqueta final solo se usa para comprobar
        # que coincide con el objetivo binario anterior.
        objetivo_verificado = pred[
            "final_result"
        ].isin(["Fail", "Withdrawn"]).astype(int)

        assert np.array_equal(
            objetivo_verificado.to_numpy(),
            pred["y_real"].to_numpy(dtype=int)
        )

        # La cohorte de predicción solo debe incluir
        # retiros cuya fecha es posterior al corte.
        retiros = pred[
            "final_result"
        ].eq("Withdrawn")

        assert pred.loc[
            retiros,
            "date_unregistration"
        ].notna().all()

        assert (
            pred.loc[
                retiros,
                "date_unregistration"
            ] > dia_corte
        ).all(), (
            f"Semana {semana}: hay retiros "
            "no posteriores al corte."
        )

        pred["Semana"] = semana
        pred["Dia_prediccion"] = dia_corte
        pred["Modelo"] = modelo

        pred["dias_hasta_retiro"] = np.where(
            retiros,
            pred["date_unregistration"] - dia_corte,
            np.nan
        )

        predicciones_326.append(pred)

tabla_predicciones_326 = pd.concat(
    predicciones_326,
    ignore_index=True
)

print("\n=== 3.26.3 PREDICCIONES RECUPERADAS ===")

print(
    "Filas de predicción:",
    len(tabla_predicciones_326)
)

print(
    "Configuraciones modelo-semana:",
    len(MODELOS_326) * len(SEMANAS_326)
)

print(
    "Fecha de retiro incorporada "
    "exclusivamente para evaluación."
)

# ------------------------------------------------------------
# 4. RESULTADOS POR REPETICIÓN
# ------------------------------------------------------------

filas_subtipos_326 = []
filas_alertas_326 = []
filas_retiros_326 = []

for (semana, modelo, repeticion), grupo in (
    tabla_predicciones_326.groupby(
        ["Semana", "Modelo", "Repeticion"],
        sort=True
    )
):

    n_total = len(grupo)

    y_real = grupo["y_real"].to_numpy(dtype=int)
    y_pred = grupo["y_pred"].to_numpy(dtype=int)

    alertas = int(y_pred.sum())

    vp = int(
        ((y_real == 1) & (y_pred == 1)).sum()
    )

    fp = int(
        ((y_real == 0) & (y_pred == 1)).sum()
    )

    n_adversos = int((y_real == 1).sum())
    n_favorables = int((y_real == 0).sum())

    filas_alertas_326.append({
        "Semana": semana,
        "Modelo": modelo,
        "Repeticion": repeticion,
        "Matriculas": n_total,
        "Alertas_totales": alertas,
        "Alertas_pct": porcentaje_326(
            alertas, n_total
        ),
        "Precision_adversa_pct": porcentaje_326(
            vp, alertas
        ),
        "Recall_adverso_pct": porcentaje_326(
            vp, n_adversos
        ),
        "Falsas_alertas": fp,
        "Tasa_falsas_alertas_pct": porcentaje_326(
            fp, n_favorables
        )
    })

    # --------------------------------------------------------
    # 4.1 RECALL POR SUBTIPO
    # --------------------------------------------------------

    for resultado in RESULTADOS_326:

        sub = grupo.loc[
            grupo["final_result"].eq(resultado)
        ]

        n_subtipo = len(sub)

        n_alertas = int(
            sub["y_pred"].sum()
        )

        filas_subtipos_326.append({
            "Semana": semana,
            "Modelo": modelo,
            "Repeticion": repeticion,
            "Resultado": resultado,
            "Matriculas": n_subtipo,
            "Alertas": n_alertas,
            "Tasa_alerta_pct": porcentaje_326(
                n_alertas, n_subtipo
            )
        })

    # --------------------------------------------------------
    # 4.2 RETIROS FUTUROS Y ANTICIPACIÓN
    # --------------------------------------------------------

    futuros = grupo.loc[
        grupo["final_result"].eq("Withdrawn")
    ].copy()

    detectados = futuros.loc[
        futuros["y_pred"].eq(1)
    ]

    n_futuros = len(futuros)
    n_detectados = len(detectados)

    fila_retiro = {
        "Semana": semana,
        "Modelo": modelo,
        "Repeticion": repeticion,
        "Retiros_futuros": n_futuros,
        "Retiros_detectados": n_detectados,
        "Recall_retiro_pct": porcentaje_326(
            n_detectados, n_futuros
        ),
        "Mediana_dias_anticipacion_detectados": (
            detectados["dias_hasta_retiro"].median()
            if n_detectados > 0
            else np.nan
        )
    }

    for horizonte in HORIZONTES_326:

        # Casos que objetivamente disponían
        # de ese horizonte desde el día de corte.
        elegibles_h = int(
            (
                futuros["dias_hasta_retiro"]
                >= horizonte
            ).sum()
        )

        # Casos retirados que recibieron una alerta
        # con al menos ese número de días.
        detectados_h = int(
            (
                detectados["dias_hasta_retiro"]
                >= horizonte
            ).sum()
        )

        fila_retiro[
            f"Retiros_elegibles_{horizonte}d"
        ] = elegibles_h

        fila_retiro[
            f"Alertas_{horizonte}d"
        ] = detectados_h

        fila_retiro[
            f"Cobertura_total_{horizonte}d_pct"
        ] = porcentaje_326(
            detectados_h, n_futuros
        )

        fila_retiro[
            f"Recall_elegibles_{horizonte}d_pct"
        ] = porcentaje_326(
            detectados_h, elegibles_h
        )

    filas_retiros_326.append(fila_retiro)

tabla_subtipos_326 = pd.DataFrame(
    filas_subtipos_326
)

tabla_alertas_326 = pd.DataFrame(
    filas_alertas_326
)

tabla_retiros_326 = pd.DataFrame(
    filas_retiros_326
)

# ------------------------------------------------------------
# 5. COBERTURA DE COHORTES
# ------------------------------------------------------------

print("\n=== 3.26.4 COMPOSICIÓN ACADÉMICA ===")

tabla_composicion_326 = (
    tabla_subtipos_326.loc[
        tabla_subtipos_326["Modelo"].eq("LR B")
        & tabla_subtipos_326["Repeticion"].eq(1),
        ["Semana", "Resultado", "Matriculas"]
    ]
    .pivot(
        index="Semana",
        columns="Resultado",
        values="Matriculas"
    )
    .reset_index()
)

display(tabla_composicion_326)

# ------------------------------------------------------------
# 6. RESUMEN DE ALERTAS POR SUBTIPO
# ------------------------------------------------------------

print("\n=== 3.26.5 ALERTAS POR RESULTADO FINAL ===")

tabla_subtipos_resumen_326 = (
    tabla_subtipos_326.groupby(
        ["Semana", "Modelo", "Resultado"],
        as_index=False
    )
    .agg(
        Matriculas_por_repeticion=(
            "Matriculas", "first"
        ),
        Alertas_medias=("Alertas", "mean"),
        Tasa_alerta_media_pct=(
            "Tasa_alerta_pct", "mean"
        ),
        Tasa_alerta_std_pct=(
            "Tasa_alerta_pct", "std"
        )
    )
)

display(
    tabla_subtipos_resumen_326.round(2)
)

# ------------------------------------------------------------
# 7. COMPARACIÓN DEL RECALL POR SUBTIPO
# ------------------------------------------------------------

print("\n=== 3.26.6 DIFERENCIAS POR SUBTIPO ADVERSO ===")

subtipos_adversos_326 = (
    tabla_subtipos_resumen_326.loc[
        tabla_subtipos_resumen_326[
            "Resultado"
        ].isin(["Fail", "Withdrawn"])
    ]
)

pivot_subtipos_326 = (
    subtipos_adversos_326.pivot(
        index=["Semana", "Resultado"],
        columns="Modelo",
        values="Tasa_alerta_media_pct"
    )
)

tabla_delta_subtipos_326 = pd.DataFrame({
    "Delta_LR_B_A_pp": (
        pivot_subtipos_326["LR B"]
        - pivot_subtipos_326["LR A"]
    ),
    "Delta_RF_B_A_pp": (
        pivot_subtipos_326["RF B"]
        - pivot_subtipos_326["RF A"]
    ),
    "Delta_C_LR_B_pp": (
        pivot_subtipos_326["Transformer C"]
        - pivot_subtipos_326["LR B"]
    ),
    "Delta_C_RF_B_pp": (
        pivot_subtipos_326["Transformer C"]
        - pivot_subtipos_326["RF B"]
    )
}).reset_index()

display(
    tabla_delta_subtipos_326.round(2)
)

# ------------------------------------------------------------
# 8. ANTICIPACIÓN DE RETIROS
# ------------------------------------------------------------

print("\n=== 3.26.7 ANTICIPACIÓN DE RETIROS ===")

columnas_promediar_326 = [
    "Retiros_detectados",
    "Recall_retiro_pct",
    "Mediana_dias_anticipacion_detectados"
]

for horizonte in HORIZONTES_326:
    columnas_promediar_326.extend([
        f"Alertas_{horizonte}d",
        f"Cobertura_total_{horizonte}d_pct",
        f"Recall_elegibles_{horizonte}d_pct"
    ])

tabla_retiros_resumen_326 = (
    tabla_retiros_326.groupby(
        ["Semana", "Modelo"]
    )[columnas_promediar_326]
    .mean()
    .reset_index()
)

n_retiros_por_semana_326 = (
    tabla_retiros_326.groupby("Semana")[
        "Retiros_futuros"
    ]
    .first()
    .rename("Retiros_futuros")
    .reset_index()
)

tabla_retiros_resumen_326 = (
    tabla_retiros_resumen_326.merge(
        n_retiros_por_semana_326,
        on="Semana",
        validate="many_to_one"
    )
)

display(
    tabla_retiros_resumen_326[
        [
            "Semana",
            "Modelo",
            "Retiros_futuros",
            "Retiros_detectados",
            "Recall_retiro_pct",
            "Mediana_dias_anticipacion_detectados",
            "Alertas_14d",
            "Cobertura_total_14d_pct",
            "Recall_elegibles_14d_pct",
            "Alertas_28d",
            "Cobertura_total_28d_pct",
            "Recall_elegibles_28d_pct"
        ]
    ].round(2)
)

# ------------------------------------------------------------
# 9. PRECISIÓN Y CARGA DE ALERTAS
# ------------------------------------------------------------

print("\n=== 3.26.8 CARGA Y PRECISIÓN DE ALERTAS ===")

tabla_alertas_resumen_326 = (
    tabla_alertas_326.groupby(
        ["Semana", "Modelo"],
        as_index=False
    )
    .agg(
        Matriculas=("Matriculas", "first"),
        Alertas_medias=("Alertas_totales", "mean"),
        Alertas_pct=("Alertas_pct", "mean"),
        Precision_adversa_pct=(
            "Precision_adversa_pct", "mean"
        ),
        Recall_adverso_pct=(
            "Recall_adverso_pct", "mean"
        ),
        Tasa_falsas_alertas_pct=(
            "Tasa_falsas_alertas_pct", "mean"
        )
    )
)

display(
    tabla_alertas_resumen_326.round(2)
)

# ------------------------------------------------------------
# 10. GRÁFICOS POR SUBTIPO
# ------------------------------------------------------------

print("\n=== 3.26.9 GRÁFICOS ===")

for resultado in ["Fail", "Withdrawn"]:

    fig, ax = plt.subplots(figsize=(10, 5))

    for modelo in MODELOS_326:

        sub = tabla_subtipos_resumen_326.loc[
            tabla_subtipos_resumen_326[
                "Modelo"
            ].eq(modelo)
            &
            tabla_subtipos_resumen_326[
                "Resultado"
            ].eq(resultado)
        ].sort_values("Semana")

        ax.plot(
            sub["Semana"],
            sub["Tasa_alerta_media_pct"],
            marker="o",
            linewidth=2,
            label=modelo
        )

    ax.set_title(
        f"Detección retrospectiva de {resultado}"
    )

    ax.set_xlabel("Semana de predicción")

    ax.set_ylabel(
        "Matrículas del subtipo alertadas (%)"
    )

    ax.set_xticks(SEMANAS_326)
    ax.set_ylim(0, 100)
    ax.grid(alpha=0.3)
    ax.legend()

    plt.tight_layout()
    plt.show()

# ------------------------------------------------------------
# 11. GRÁFICO DE ALERTAS OPORTUNAS
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(10, 5))

for modelo in MODELOS_326:

    sub = tabla_retiros_resumen_326.loc[
        tabla_retiros_resumen_326[
            "Modelo"
        ].eq(modelo)
    ].sort_values("Semana")

    ax.plot(
        sub["Semana"],
        sub["Cobertura_total_14d_pct"],
        marker="o",
        linewidth=2,
        label=modelo
    )

ax.set_title(
    "Retiros detectados con al menos 14 días\n"
    "de anticipación"
)

ax.set_xlabel("Semana de predicción")

ax.set_ylabel(
    "Porcentaje de todos los retiros futuros"
)

ax.set_xticks(SEMANAS_326)
ax.set_ylim(0, 100)
ax.grid(alpha=0.3)
ax.legend()

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 12. CONTROLES FINALES
# ------------------------------------------------------------

print("\n=== 3.26.10 CONTROLES FINALES ===")

assert len(tabla_subtipos_326) == (
    4 * 5 * 2 * 4
)

assert len(tabla_alertas_326) == (
    4 * 5 * 2
)

assert len(tabla_retiros_326) == (
    4 * 5 * 2
)

assert len(tabla_delta_subtipos_326) == 8

assert tabla_retiros_326[
    "Recall_retiro_pct"
].between(0, 100).all()

for horizonte in HORIZONTES_326:

    assert (
        tabla_retiros_326[
            f"Alertas_{horizonte}d"
        ]
        <=
        tabla_retiros_326[
            "Retiros_detectados"
        ]
    ).all()

    assert (
        tabla_retiros_326[
            f"Alertas_{horizonte}d"
        ]
        <=
        tabla_retiros_326[
            f"Retiros_elegibles_{horizonte}d"
        ]
    ).all()

for (semana, modelo, repeticion), sub in (
    tabla_subtipos_326.groupby(
        ["Semana", "Modelo", "Repeticion"]
    )
):

    assert sub["Matriculas"].sum() == len(
        experimentos_oulad[semana]["y_train"]
    )

    assert sub["Alertas"].sum() == int(
        tabla_alertas_326.loc[
            tabla_alertas_326["Semana"].eq(semana)
            &
            tabla_alertas_326["Modelo"].eq(modelo)
            &
            tabla_alertas_326[
                "Repeticion"
            ].eq(repeticion),
            "Alertas_totales"
        ].iloc[0]
    )

print("Ventanas evaluadas: 4")
print("Modelos evaluados: 5")
print("Resultados académicos analizados: 4")
print("Horizontes de retiro: 7, 14 y 28 días")
print("Predicciones: fuera de muestra")
print("Las dos repeticiones se promediaron")
print("Fecha de retiro usada solo para evaluación")
print("Modelos reentrenados: 0")
print("Umbrales modificados: 0")
print("Conjunto de prueba independiente: NO utilizado")

print("\n=== ACTIVIDAD 3.26 FINALIZADA ===")



### 3.26.11. Resultados e interpretación

Se evaluó retrospectivamente la capacidad de los cinco modelos de Machine Learning para identificar resultados académicos adversos en las semanas 2, 4, 6 y 8 de OULAD.

El análisis utilizó exclusivamente predicciones fuera de muestra procedentes de validación cruzada agrupada por estudiante.

Se examinaron los resultados `Fail`, `Withdrawn`, `Pass` y `Distinction`, así como la anticipación temporal de alertas para matrículas con retiro posterior.

#### Identificación de retiros futuros

| Semana | LR A | RF A | LR B | RF B | Transformer C |
|---|---:|---:|---:|---:|---:|
| 2 | 49,16 % | 47,71 % | 57,38 % | 53,36 % | 56,63 % |
| 4 | 45,19 % | 44,75 % | 55,95 % | 50,26 % | 56,47 % |
| 6 | 41,09 % | 41,09 % | 55,26 % | 49,99 % | 53,27 % |
| 8 | 37,42 % | 37,66 % | 55,72 % | 48,99 % | 51,26 % |

Los resultados corresponden a la proporción de matrículas con resultado final `Withdrawn` que recibieron una alerta binaria de riesgo adverso.

Los modelos longitudinales presentaron una capacidad de detección superior a la de sus versiones estáticas.

Regresión Logística B mantuvo un recall de retiros futuros entre 55,26 % y 57,38 %, mientras que el Transformer C obtuvo resultados competitivos.

Las poblaciones elegibles son diferentes entre semanas, por lo que estos valores no representan necesariamente la evolución del recall sobre un mismo conjunto de retiros.

#### Anticipación desde la semana 2

En la semana 2 se analizaron 4.538 matrículas de desarrollo que se retiraron posteriormente.

Regresión Logística B detectó, en promedio, 2.604 de esos retiros, obteniendo un recall de 57,38 %.

El modelo identificó aproximadamente 2.268,5 retiros por repetición al menos 14 días antes de su fecha efectiva, equivalentes al 49,99 % de todos los retiros futuros.

Asimismo, identificó aproximadamente 1.985,5 retiros al menos 28 días antes, equivalentes al 43,75 % de todos los retiros futuros.

Entre los retiros que todavía disponían de al menos 28 días de anticipación desde el corte, el recall correspondiente fue de 54,13 %.

La mediana de anticipación entre los retiros efectivamente identificados fue de 75 días.

#### Reprobación futura

En la semana 8, Regresión Logística B incrementó la identificación retrospectiva de matrículas `Fail` en 23,06 puntos porcentuales respecto a Regresión Logística A.

Para `Withdrawn`, la mejora correspondiente fue de 18,30 puntos porcentuales.

Estos hallazgos sugieren que la incorporación del comportamiento longitudinal aporta información relevante para ambos subtipos adversos.

#### Precisión y carga de alertas

En la semana 2, Regresión Logística B generó alertas sobre el 39,50 % de las matrículas de desarrollo.

La precisión de esas alertas fue de 65,83 %, con una tasa de falsas alertas del 24,61 % entre las matrículas cuyo resultado final fue favorable.

En la semana 8, la precisión aumentó a 73,47 % y la tasa de falsas alertas disminuyó a 15,35 %.

Estas diferencias describen poblaciones temporales distintas y no deben interpretarse como mejoras causadas exclusivamente por el transcurso del tiempo.

El volumen de alertas revela que cualquier implementación práctica debería considerar la capacidad institucional disponible para atender los casos identificados.

### 3.26.12. Limitaciones

La evaluación es retrospectiva y no representa una intervención educativa real.

Los modelos predicen resultados adversos binarios; las alertas no distinguen explícitamente entre reprobación y retiro.

El resultado `Withdrawn` corresponde al retiro de una matrícula o módulo de OULAD y no necesariamente a la deserción definitiva de una institución.

La fecha de retiro se utilizó únicamente para evaluar la anticipación, no para entrenar modelos.

La mediana de anticipación se calcula sobre los retiros correctamente detectados y está condicionada al subconjunto identificado por cada modelo.

Las cohortes cambian entre semanas debido a los criterios de elegibilidad temporal.

La clasificación utiliza el umbral previamente establecido de 0,5, sin optimización específica para recursos institucionales ni costes de intervención.

No se utilizó el conjunto de prueba independiente.

### 3.26.13. Conclusión

La incorporación de información longitudinal permitió mejorar tanto la clasificación general como la detección retrospectiva de reprobaciones y retiros futuros.

Regresión Logística B obtuvo resultados especialmente competitivos para la identificación temprana, logrando alertar durante la semana 2 sobre aproximadamente el 43,75 % de los retiros futuros con al menos 28 días de anticipación.

Sin embargo, también generó alertas sobre una proporción importante de matrículas con resultados finalmente favorables.

**Conclusión principal:** los modelos longitudinales presentan potencial para apoyar sistemas de alerta académica temprana, pero su utilidad operativa debe evaluarse considerando simultáneamente anticipación, cobertura, precisión y carga de intervención.

Los resultados proporcionan evidencia retrospectiva de capacidad predictiva, no evidencia de que las alertas o las intervenciones reduzcan efectivamente los retiros o las reprobaciones.



### 3.27. Seguimiento longitudinal de alertas académicas

#### Objetivo

Evaluar retrospectivamente un sistema de alertas repetidas en las semanas 2, 4, 6 y 8, utilizando las predicciones fuera de muestra de los cinco modelos desarrollados.

#### Población de referencia

Se utilizará una cohorte inicial fija: las matrículas del conjunto de desarrollo elegibles durante la semana 2.

A diferencia de la cohorte común de la Actividad 3.24, esta población conservará a las matrículas que se retiren antes de la semana 8.

Así se evitará excluir retrospectivamente a parte de los retiros tempranos.

#### Primera alerta

Para cada matrícula, modelo y repetición se identificará la primera semana en la que se predijo un resultado adverso.

Las fechas posibles serán los días 14, 28, 42 y 56.

Las matrículas que nunca recibieron una alerta conservarán un valor faltante en la fecha de primera detección.

Solo se utilizarán predicciones correspondientes a ventanas en las que la matrícula era elegible.

#### Cobertura acumulada

Se calculará la proporción de matrículas con resultado final `Withdrawn` que hayan recibido al menos una alerta hasta cada semana.

El denominador será fijo: todos los retiros futuros existentes en la cohorte inicial de la semana 2.

También se calculará la cobertura acumulada de alertas emitidas con al menos 14 y 28 días de anticipación respecto a la fecha efectiva de retiro.

#### Carga institucional

Se calcularán las matrículas distintas alertadas acumulativamente y las falsas alertas acumuladas sobre resultados favorables.

Una matrícula alertada varias veces contará una sola vez para estimar la cantidad de casos distintos que habrían requerido atención inicial.

#### Limitaciones

El análisis representa una simulación retrospectiva de alertas repetidas. Los modelos de cada semana fueron entrenados por separado y no constituyen un único sistema actualizado en producción.

Los resultados no incorporan decisiones reales de intervención, costes institucionales ni cambios de comportamiento posteriores a una alerta.

Las matrículas que se retiraron dejan de ser elegibles para nuevas predicciones, pero permanecen en el denominador inicial de cobertura.

El análisis se realizará por repetición, sin considerar las dos predicciones de una matrícula como observaciones independientes.

No se entrenarán nuevos modelos, no se modificarán umbrales y no se utilizará el conjunto de prueba independiente.


In [ ]:

# ============================================================
# ACTIVIDAD 3.27
# PRIMERA ALERTA Y COBERTURA ACUMULADA
# COHORTE INICIAL FIJA DE SEMANA 2
# ============================================================

# Asistencia IA: ChatGPT, octubre de 2026.
# Prompt:
# "Reconstruir la primera alerta de cada matrícula
# en semanas 2, 4, 6 y 8 con las predicciones OOF
# existentes. Mantener una cohorte inicial fija,
# incluir retiros tempranos, evaluar anticipación
# y falsas alertas acumuladas. No reentrenar
# ni utilizar el conjunto de prueba independiente."

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

SEMANAS_327 = [2, 4, 6, 8]

MODELOS_327 = [
    "LR A",
    "RF A",
    "LR B",
    "RF B",
    "Transformer C"
]

CLAVES_327 = [
    "code_module",
    "code_presentation",
    "id_student"
]

assert "tabla_predicciones_326" in globals()
assert "tabla_retiros_resumen_326" in globals()

def porcentaje_327(a, b):
    return 100 * a / b if b else np.nan

# ------------------------------------------------------------
# 1. COHORTE INICIAL
# ------------------------------------------------------------

print("=== 3.27.1 COHORTE INICIAL ===")

predicciones_327 = tabla_predicciones_326.copy()

base_semana2_327 = predicciones_327.loc[
    predicciones_327["Semana"].eq(2)
].copy()

assert set(
    base_semana2_327["Modelo"].unique()
) == set(MODELOS_327)

assert set(
    base_semana2_327["Repeticion"].unique()
) == {1, 2}

print("Matrículas por repetición:", 22412)
print("Semanas de alerta:", SEMANAS_327)

# ------------------------------------------------------------
# 2. PRIMERA ALERTA POR MATRÍCULA
# ------------------------------------------------------------

print("\n=== 3.27.2 PRIMERA ALERTA ===")

episodios_327 = []
primera_alerta_resumen_327 = []
filas_acumuladas_327 = []

for modelo in MODELOS_327:

    for repeticion in [1, 2]:

        # Mantener la cohorte elegible en semana 2.
        base = base_semana2_327.loc[
            base_semana2_327["Modelo"].eq(modelo)
            &
            base_semana2_327["Repeticion"].eq(
                repeticion
            ),
            CLAVES_327 + [
                "final_result",
                "date_unregistration",
                "y_real"
            ]
        ].copy()

        assert len(base) == 22412
        assert not base.duplicated(
            subset=CLAVES_327
        ).any()

        # Recuperar todas las predicciones disponibles
        # para esas mismas matrículas.
        historia = predicciones_327.loc[
            predicciones_327["Modelo"].eq(modelo)
            &
            predicciones_327["Repeticion"].eq(
                repeticion
            ),
            CLAVES_327 + [
                "Semana",
                "Dia_prediccion",
                "y_pred"
            ]
        ].merge(
            base[CLAVES_327],
            on=CLAVES_327,
            how="inner",
            validate="many_to_one"
        )

        assert not historia.duplicated(
            subset=CLAVES_327 + ["Semana"]
        ).any()

        assert historia[
            "Dia_prediccion"
        ].isin([14, 28, 42, 56]).all()

        primera = (
            historia.loc[
                historia["y_pred"].eq(1)
            ]
            .groupby(CLAVES_327)[
                "Dia_prediccion"
            ]
            .min()
            .rename("Dia_primera_alerta")
            .reset_index()
        )

        episodios = base.merge(
            primera,
            on=CLAVES_327,
            how="left",
            validate="one_to_one"
        )

        episodios["Modelo"] = modelo
        episodios["Repeticion"] = repeticion

        es_retiro = episodios[
            "final_result"
        ].eq("Withdrawn")

        es_favorable = episodios[
            "final_result"
        ].isin(["Pass", "Distinction"])

        n_retiros = int(es_retiro.sum())
        n_favorables = int(es_favorable.sum())

        assert n_retiros == 4538

        assert episodios.loc[
            es_retiro,
            "date_unregistration"
        ].notna().all()

        # Ninguna primera alerta debe ser posterior
        # al retiro de la matrícula.
        detectados = episodios.loc[
            es_retiro
            &
            episodios[
                "Dia_primera_alerta"
            ].notna()
        ]

        assert (
            detectados["Dia_primera_alerta"]
            <
            detectados["date_unregistration"]
        ).all()

        episodios["Dias_anticipacion"] = np.where(
            es_retiro,
            (
                episodios["date_unregistration"]
                - episodios["Dia_primera_alerta"]
            ),
            np.nan
        )

        episodios_327.append(episodios)

        # Número de nuevas alertas en cada semana.
        for semana in SEMANAS_327:

            dia = semana * 7

            primeras_semana = (
                episodios["Dia_primera_alerta"]
                .eq(dia)
            )

            primera_alerta_resumen_327.append({
                "Modelo": modelo,
                "Repeticion": repeticion,
                "Semana": semana,
                "Nuevas_alertas": int(
                    primeras_semana.sum()
                ),
                "Nuevos_retiros_detectados": int(
                    (
                        primeras_semana
                        & es_retiro
                    ).sum()
                ),
                "Nuevas_falsas_alertas": int(
                    (
                        primeras_semana
                        & es_favorable
                    ).sum()
                )
            })

            # Todas las matrículas alertadas por
            # primera vez hasta ese día.
            ya_alertadas = (
                episodios["Dia_primera_alerta"]
                <= dia
            )

            retiros_alertados = int(
                (es_retiro & ya_alertadas).sum()
            )

            falsas_alertas = int(
                (
                    es_favorable
                    & ya_alertadas
                ).sum()
            )

            total_alertas = int(
                ya_alertadas.sum()
            )

            fila = {
                "Semana": semana,
                "Modelo": modelo,
                "Repeticion": repeticion,
                "Cohorte_inicial": len(episodios),
                "Retiros_iniciales": n_retiros,
                "Alertas_acumuladas": total_alertas,
                "Porcentaje_alertado": porcentaje_327(
                    total_alertas,
                    len(episodios)
                ),
                "Retiros_detectados_acumulados": (
                    retiros_alertados
                ),
                "Cobertura_retiros_pct": porcentaje_327(
                    retiros_alertados,
                    n_retiros
                ),
                "Falsas_alertas_acumuladas": (
                    falsas_alertas
                ),
                "Tasa_falsas_alertas_pct": porcentaje_327(
                    falsas_alertas,
                    n_favorables
                ),
                "Precision_alertas_pct": porcentaje_327(
                    total_alertas - falsas_alertas,
                    total_alertas
                )
            }

            for horizonte in [14, 28]:

                oportunos = int(
                    (
                        es_retiro
                        & ya_alertadas
                        & (
                            episodios[
                                "Dias_anticipacion"
                            ] >= horizonte
                        )
                    ).sum()
                )

                fila[
                    f"Retiros_oportunos_{horizonte}d"
                ] = oportunos

                fila[
                    f"Cobertura_{horizonte}d_pct"
                ] = porcentaje_327(
                    oportunos,
                    n_retiros
                )

            filas_acumuladas_327.append(fila)

tabla_episodios_327 = pd.concat(
    episodios_327,
    ignore_index=True
)

tabla_primeras_327 = pd.DataFrame(
    primera_alerta_resumen_327
)

tabla_acumulada_327 = pd.DataFrame(
    filas_acumuladas_327
)

# ------------------------------------------------------------
# 3. VALIDAR RESULTADOS DE SEMANA 2
# ------------------------------------------------------------

print("\n=== 3.27.3 CONSISTENCIA CON ACTIVIDAD 3.26 ===")

for modelo in MODELOS_327:

    valor_327 = (
        tabla_acumulada_327.loc[
            tabla_acumulada_327[
                "Modelo"
            ].eq(modelo)
            &
            tabla_acumulada_327[
                "Semana"
            ].eq(2),
            "Cobertura_retiros_pct"
        ].mean()
    )

    valor_326 = (
        tabla_retiros_resumen_326.loc[
            tabla_retiros_resumen_326[
                "Modelo"
            ].eq(modelo)
            &
            tabla_retiros_resumen_326[
                "Semana"
            ].eq(2),
            "Recall_retiro_pct"
        ].iloc[0]
    )

    assert np.isclose(
        valor_327,
        valor_326,
        atol=1e-8
    ), (
        f"Inconsistencia en semana 2: {modelo}"
    )

print(
    "Los resultados iniciales coinciden "
    "con la Actividad 3.26."
)

# ------------------------------------------------------------
# 4. NUEVAS ALERTAS POR SEMANA
# ------------------------------------------------------------

print("\n=== 3.27.4 NUEVAS ALERTAS ===")

tabla_primeras_resumen_327 = (
    tabla_primeras_327.groupby(
        ["Semana", "Modelo"],
        as_index=False
    )
    .agg(
        Nuevas_alertas=("Nuevas_alertas", "mean"),
        Nuevos_retiros_detectados=(
            "Nuevos_retiros_detectados", "mean"
        ),
        Nuevas_falsas_alertas=(
            "Nuevas_falsas_alertas", "mean"
        )
    )
)

display(
    tabla_primeras_resumen_327.round(2)
)

# ------------------------------------------------------------
# 5. COBERTURA ACUMULADA
# ------------------------------------------------------------

print("\n=== 3.27.5 COBERTURA ACUMULADA ===")

columnas_resumen_327 = [
    "Alertas_acumuladas",
    "Porcentaje_alertado",
    "Retiros_detectados_acumulados",
    "Cobertura_retiros_pct",
    "Falsas_alertas_acumuladas",
    "Tasa_falsas_alertas_pct",
    "Precision_alertas_pct",
    "Retiros_oportunos_14d",
    "Cobertura_14d_pct",
    "Retiros_oportunos_28d",
    "Cobertura_28d_pct"
]

tabla_resumen_acumulado_327 = (
    tabla_acumulada_327.groupby(
        ["Semana", "Modelo"],
        as_index=False
    )[columnas_resumen_327]
    .mean()
)

display(
    tabla_resumen_acumulado_327.round(2)
)

# ------------------------------------------------------------
# 6. GANANCIA DE EVALUACIONES SUCESIVAS
# ------------------------------------------------------------

print("\n=== 3.27.6 BENEFICIO DE VENTANAS ADICIONALES ===")

ganancia_327 = (
    tabla_resumen_acumulado_327.pivot(
        index="Modelo",
        columns="Semana",
        values="Cobertura_retiros_pct"
    )
)

ganancia_327["Ganancia_S8_menos_S2_pp"] = (
    ganancia_327[8]
    - ganancia_327[2]
)

display(
    ganancia_327.round(2)
)

# ------------------------------------------------------------
# 7. PRIMERA ALERTA EN RETIROS DETECTADOS
# ------------------------------------------------------------

print("\n=== 3.27.7 ANTICIPACIÓN DE PRIMERA ALERTA ===")

filas_anticipacion_327 = []

for (modelo, repeticion), sub in (
    tabla_episodios_327.groupby(
        ["Modelo", "Repeticion"]
    )
):

    retiros = sub.loc[
        sub["final_result"].eq("Withdrawn")
    ]

    detectados = retiros.loc[
        retiros["Dia_primera_alerta"].notna()
    ]

    filas_anticipacion_327.append({
        "Modelo": modelo,
        "Repeticion": repeticion,
        "Retiros_futuros_iniciales": len(retiros),
        "Retiros_detectados_alguna_vez": (
            len(detectados)
        ),
        "Mediana_anticipacion_detectados": (
            detectados["Dias_anticipacion"].median()
            if len(detectados)
            else np.nan
        )
    })

tabla_anticipacion_327 = pd.DataFrame(
    filas_anticipacion_327
)

display(
    tabla_anticipacion_327.groupby(
        "Modelo",
        as_index=False
    )
    .agg(
        Retiros_iniciales=(
            "Retiros_futuros_iniciales", "first"
        ),
        Retiros_detectados_media=(
            "Retiros_detectados_alguna_vez", "mean"
        ),
        Mediana_anticipacion_media=(
            "Mediana_anticipacion_detectados", "mean"
        )
    )
    .round(2)
)

# ------------------------------------------------------------
# 8. GRÁFICOS
# ------------------------------------------------------------

print("\n=== 3.27.8 GRÁFICOS ===")

for columna, titulo, eje in [
    (
        "Cobertura_retiros_pct",
        "Cobertura acumulada de retiros futuros",
        "Retiros iniciales alertados (%)"
    ),
    (
        "Cobertura_28d_pct",
        "Retiros alertados con al menos 28 días",
        "Retiros iniciales con alerta oportuna (%)"
    ),
    (
        "Porcentaje_alertado",
        "Carga acumulada de alertas",
        "Matrículas iniciales alertadas (%)"
    )
]:

    fig, ax = plt.subplots(figsize=(10, 5))

    for modelo in MODELOS_327:

        sub = tabla_resumen_acumulado_327.loc[
            tabla_resumen_acumulado_327[
                "Modelo"
            ].eq(modelo)
        ].sort_values("Semana")

        ax.plot(
            sub["Semana"],
            sub[columna],
            marker="o",
            linewidth=2,
            label=modelo
        )

    ax.set_title(titulo)
    ax.set_xlabel("Semana de predicción")
    ax.set_ylabel(eje)
    ax.set_xticks(SEMANAS_327)
    ax.set_ylim(0, 100)
    ax.grid(alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.show()

# ------------------------------------------------------------
# 9. CONTROLES FINALES
# ------------------------------------------------------------

print("\n=== 3.27.9 CONTROLES FINALES ===")

assert len(tabla_acumulada_327) == 40
assert len(tabla_resumen_acumulado_327) == 20
assert len(tabla_primeras_327) == 40

assert tabla_episodios_327.groupby(
    ["Modelo", "Repeticion"]
).size().eq(22412).all()

assert tabla_episodios_327[
    "Dia_primera_alerta"
].dropna().isin([14, 28, 42, 56]).all()

for modelo in MODELOS_327:

    sub = tabla_resumen_acumulado_327.loc[
        tabla_resumen_acumulado_327[
            "Modelo"
        ].eq(modelo)
    ].sort_values("Semana")

    assert (
        sub["Cobertura_retiros_pct"]
        .diff()
        .dropna()
        >= -1e-9
    ).all()

    assert (
        sub["Alertas_acumuladas"]
        .diff()
        .dropna()
        >= -1e-9
    ).all()

    assert (
        sub["Cobertura_28d_pct"]
        <= sub["Cobertura_14d_pct"]
    ).all()

    assert (
        sub["Cobertura_14d_pct"]
        <= sub["Cobertura_retiros_pct"]
    ).all()

print("Modelos analizados: 5")
print("Repeticiones por modelo: 2")
print("Cohorte inicial fija: 22412 matrículas")
print("Retiros futuros iniciales: 4538")
print("Semanas evaluadas: 2, 4, 6, 8")
print("Primera alerta reconstruida correctamente.")
print("Retiros tempranos conservados en la población.")
print("Modelos reentrenados: 0")
print("Umbrales modificados: 0")
print("Conjunto de prueba independiente: NO utilizado")

print("\n=== ACTIVIDAD 3.27 FINALIZADA ===")



### 3.27.10. Resultados e interpretación

Se completó la simulación retrospectiva de un sistema de alertas académicas repetidas en las semanas 2, 4, 6 y 8.

El análisis utilizó una cohorte inicial fija de 22.412 matrículas de desarrollo, que incluía 4.538 matrículas cuyo resultado final fue `Withdrawn`.

A diferencia de la comparación de cohortes comunes, las matrículas que se retiraron antes de la semana 8 permanecieron en el denominador de evaluación.

Para cada modelo y repetición se identificó la primera alerta de riesgo adverso mediante predicciones fuera de muestra previamente generadas.

#### Cobertura acumulada de retiros

| Modelo | Semana 2 | Semana 8 | Ganancia |
|---|---:|---:|---:|
| Regresión Logística A | 49,16 % | 50,13 % | +0,97 pp |
| Random Forest A | 47,71 % | 54,25 % | +6,54 pp |
| Regresión Logística B | 57,38 % | 72,29 % | +14,91 pp |
| Random Forest B | 53,36 % | 69,06 % | +15,70 pp |
| Transformer C | 56,63 % | 71,47 % | +14,84 pp |

Los modelos longitudinales presentaron mayores ganancias de cobertura acumulada de retiros futuros que los modelos estáticos.

Regresión Logística B alcanzó la mayor cobertura final, con 72,29 %, seguida del Transformer C, con 71,47 %, y Random Forest B, con 69,06 %.

#### Identificación de retiros adicionales

Regresión Logística B pasó de identificar 2.604 retiros futuros durante la semana 2 a un promedio acumulado de 3.280,5 durante la semana 8.

Esto representa aproximadamente 676,5 retiros adicionales identificados mediante las evaluaciones posteriores.

El Transformer C alcanzó una cobertura acumulada del 71,47 %, equivalente a 3.243,5 retiros identificados por repetición.

#### Anticipación de la primera alerta

Al finalizar la semana 8, Regresión Logística B había identificado el 54,81 % de todos los retiros iniciales con al menos 28 días de anticipación.

El Transformer C alcanzó una cobertura equivalente del 54,37 %, mientras que Random Forest B obtuvo 51,94 %.

La mediana de anticipación entre los retiros correctamente detectados fue de 73 días para Regresión Logística B, 72 para Transformer C y 73 para Random Forest B.

Estas medianas están condicionadas a los casos identificados por cada modelo y no deben interpretarse como tiempos de anticipación de la población completa.

#### Carga acumulada de alertas

Regresión Logística B incrementó sus alertas acumuladas desde 8.853 matrículas en la semana 2 hasta 11.648,5 al finalizar la semana 8.

Las falsas alertas acumuladas aumentaron de 3.025 a 4.128, mientras que la precisión de las alertas pasó de 65,83 % a 64,56 %.

La tasa de falsas alertas entre las matrículas con resultados favorables aumentó de 24,61 % a 33,58 %.

Estos resultados muestran que la cobertura adicional de resultados adversos tiene un costo operativo asociado a un mayor volumen de alertas, incluidas alertas incorrectas.

### 3.27.11. Limitaciones

La evaluación constituye una simulación retrospectiva basada en predicciones fuera de muestra, no una intervención educativa real.

Cada ventana utiliza modelos entrenados por separado. Por tanto, el análisis no equivale a evaluar longitudinalmente un único modelo actualizado en producción.

La simulación supone que las alertas no modifican el comportamiento académico posterior de los estudiantes.

Las matrículas con retiro temprano permanecen en el denominador inicial, pero dejan de generar nuevas predicciones una vez que no son elegibles.

Las medianas de anticipación están condicionadas a los retiros correctamente detectados.

Las métricas representan promedios de dos repeticiones de validación cruzada y no deben tratarse como observaciones estadísticamente independientes.

Los modelos conservan el umbral original de clasificación de 0,5, sin optimización según la capacidad institucional de intervención.

El conjunto de prueba independiente permaneció reservado.

### 3.27.12. Conclusión

Las evaluaciones sucesivas permiten aumentar la identificación retrospectiva de retiros futuros en comparación con una única predicción durante la semana 2.

Los modelos longitudinales obtienen las mayores ganancias de cobertura acumulada.

Regresión Logística B alcanzó una cobertura de retiros futuros del 72,29 % al finalizar la semana 8 y una cobertura del 54,81 % para retiros identificados al menos 28 días antes de ocurrir.

Sin embargo, este aumento de cobertura estuvo acompañado por un incremento de falsas alertas y del número de matrículas que habrían requerido seguimiento.

**Conclusión principal:** un sistema longitudinal de alertas repetidas presenta mayor potencial predictivo que una evaluación estática aislada, pero debe diseñarse considerando simultáneamente la oportunidad de detección y la capacidad institucional para atender las alertas.

Los resultados respaldan la utilidad de la información longitudinal y de las evaluaciones periódicas, sin demostrar todavía que las intervenciones derivadas de esas alertas mejoren los resultados educativos.



### 3.28. Eficiencia marginal de las alertas y escenarios de capacidad institucional

#### Objetivo

Evaluar la eficiencia de cada nueva ronda de alertas académicas y estudiar el compromiso entre cobertura de retiros futuros y cantidad de matrículas que recibirían seguimiento.

#### Evaluación marginal

Para cada modelo y semana se identificarán las matrículas que reciben su primera alerta.

Las nuevas alertas se clasificarán retrospectivamente según su resultado final:

- `Fail`: alerta correcta de un resultado adverso.
- `Withdrawn`: alerta correcta de un resultado adverso.
- `Pass` y `Distinction`: alertas incorrectas según la etiqueta final.

Se calcularán:

1. Nuevas alertas por ronda.
2. Nuevos retiros futuros identificados.
3. Nuevas reprobaciones futuras identificadas.
4. Nuevas falsas alertas.
5. Precisión de las nuevas alertas.
6. Porcentaje de nuevas alertas que corresponden a retiros identificados con al menos 28 días de anticipación.

#### Evaluación de estrategias acumuladas

Se compararán cuatro estrategias de seguimiento:

- Evaluar únicamente durante la semana 2.
- Evaluar en las semanas 2 y 4.
- Evaluar en las semanas 2, 4 y 6.
- Evaluar en las semanas 2, 4, 6 y 8.

Todas las estrategias conservarán como denominador la cohorte inicial de 22.412 matrículas.

#### Escenarios ilustrativos de capacidad

Se explorarán capacidades acumuladas hipotéticas equivalentes al 40 %, 45 % y 50 % de las matrículas iniciales.

Para cada escenario se identificarán las combinaciones de modelo y estrategia que no superen la capacidad establecida.

Se comparará su cobertura de retiros identificados al menos 28 días antes de ocurrir.

El análisis permitirá representar la frontera entre carga de alertas y cobertura temprana.

#### Consideraciones metodológicas

La selección de escenarios se realizará sobre resultados del conjunto de desarrollo y será exploratoria.

No se interpretará como una política institucional validada ni como una optimización de utilidad clínica o educativa.

La precisión de nuevas alertas corresponde al objetivo binario adverso. La proporción de retiros entre nuevas alertas se reportará por separado y no debe confundirse con la precisión binaria.

Las métricas se calcularán dentro de cada repetición y posteriormente se promediarán.

No se reentrenarán modelos, no se modificarán umbrales y el conjunto de prueba independiente permanecerá reservado.


In [ ]:

# ============================================================
# ACTIVIDAD 3.28
# EFICIENCIA MARGINAL DE ALERTAS Y CAPACIDAD INSTITUCIONAL
# ============================================================

# Asistencia IA: ChatGPT, octubre de 2026.
# Prompt:
# "Evaluar nuevas alertas por semana, separando Fail,
# Withdrawn y falsas alertas. Calcular precisión
# marginal y detección oportuna de retiros.
# Analizar políticas acumuladas y capacidades
# hipotéticas de atención con resultados OOF existentes.
# No reentrenar ni acceder al conjunto de prueba."

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

SEMANAS_328 = [2, 4, 6, 8]

MODELOS_328 = [
    "LR A",
    "RF A",
    "LR B",
    "RF B",
    "Transformer C"
]

N_INICIAL_328 = 22412
N_RETIROS_328 = 4538

CAPACIDADES_328 = [40, 45, 50]

assert "tabla_episodios_327" in globals()
assert "tabla_primeras_327" in globals()
assert "tabla_acumulada_327" in globals()
assert "tabla_resumen_acumulado_327" in globals()

def pct_328(a, b):
    return 100.0 * a / b if b else np.nan

# ------------------------------------------------------------
# 1. AUDITORÍA INICIAL
# ------------------------------------------------------------

print("=== 3.28.1 CONFIGURACIÓN ===")

episodios_328 = tabla_episodios_327.copy()

claves_328 = [
    "code_module",
    "code_presentation",
    "id_student"
]

assert episodios_328.groupby(
    ["Modelo", "Repeticion"]
).size().eq(N_INICIAL_328).all()

assert not episodios_328.duplicated(
    subset=claves_328 + ["Modelo", "Repeticion"]
).any()

print("Modelos:", MODELOS_328)
print("Semanas:", SEMANAS_328)
print("Matrículas iniciales:", N_INICIAL_328)
print("Retiros futuros iniciales:", N_RETIROS_328)

# ------------------------------------------------------------
# 2. EFICIENCIA DE CADA RONDA DE ALERTAS
# ------------------------------------------------------------

print("\n=== 3.28.2 NUEVAS ALERTAS POR RONDA ===")

filas_328 = []

for (modelo, repeticion), grupo in (
    episodios_328.groupby(
        ["Modelo", "Repeticion"]
    )
):

    assert len(grupo) == N_INICIAL_328

    for semana in SEMANAS_328:

        dia = semana * 7

        nuevas = grupo.loc[
            grupo["Dia_primera_alerta"].eq(dia)
        ]

        n_nuevas = len(nuevas)

        n_fail = int(
            nuevas["final_result"].eq("Fail").sum()
        )

        n_retiro = int(
            nuevas["final_result"].eq(
                "Withdrawn"
            ).sum()
        )

        n_falsas = int(
            nuevas["final_result"].isin(
                ["Pass", "Distinction"]
            ).sum()
        )

        n_correctas = n_fail + n_retiro

        n_retiro_28 = int(
            (
                nuevas["final_result"].eq(
                    "Withdrawn"
                )
                &
                nuevas["Dias_anticipacion"].ge(28)
            ).sum()
        )

        assert (
            n_correctas + n_falsas
            == n_nuevas
        )

        filas_328.append({
            "Semana": semana,
            "Modelo": modelo,
            "Repeticion": repeticion,
            "Nuevas_alertas": n_nuevas,
            "Nuevas_fail": n_fail,
            "Nuevos_retiros": n_retiro,
            "Nuevas_falsas": n_falsas,
            "Nuevas_correctas": n_correctas,
            "Nuevos_retiros_28d": n_retiro_28,
            "Precision_nuevas_pct": pct_328(
                n_correctas,
                n_nuevas
            ),
            "Retiros_por_100_alertas": pct_328(
                n_retiro,
                n_nuevas
            ),
            "Retiros_28d_por_100_alertas": pct_328(
                n_retiro_28,
                n_nuevas
            )
        })

tabla_marginal_328 = pd.DataFrame(
    filas_328
)

assert len(tabla_marginal_328) == 40

tabla_marginal_resumen_328 = (
    tabla_marginal_328.groupby(
        ["Semana", "Modelo"],
        as_index=False
    )
    .agg(
        Nuevas_alertas=("Nuevas_alertas", "mean"),
        Nuevas_fail=("Nuevas_fail", "mean"),
        Nuevos_retiros=("Nuevos_retiros", "mean"),
        Nuevas_falsas=("Nuevas_falsas", "mean"),
        Nuevos_retiros_28d=(
            "Nuevos_retiros_28d", "mean"
        ),
        Precision_nuevas_pct=(
            "Precision_nuevas_pct", "mean"
        ),
        Retiros_por_100_alertas=(
            "Retiros_por_100_alertas", "mean"
        ),
        Retiros_28d_por_100_alertas=(
            "Retiros_28d_por_100_alertas", "mean"
        )
    )
)

display(
    tabla_marginal_resumen_328.round(2)
)

# ------------------------------------------------------------
# 3. VERIFICAR CONSISTENCIA CON ACTIVIDAD 3.27
# ------------------------------------------------------------

print("\n=== 3.28.3 CONTROL DE NUEVAS ALERTAS ===")

comparacion_328 = tabla_marginal_resumen_328.merge(
    tabla_primeras_resumen_327[
        [
            "Semana",
            "Modelo",
            "Nuevas_alertas",
            "Nuevos_retiros_detectados",
            "Nuevas_falsas_alertas"
        ]
    ],
    on=["Semana", "Modelo"],
    validate="one_to_one",
    suffixes=("_328", "_327")
)

assert np.allclose(
    comparacion_328["Nuevas_alertas_328"],
    comparacion_328["Nuevas_alertas_327"]
)

assert np.allclose(
    comparacion_328["Nuevos_retiros"],
    comparacion_328["Nuevos_retiros_detectados"]
)

assert np.allclose(
    comparacion_328["Nuevas_falsas"],
    comparacion_328["Nuevas_falsas_alertas"]
)

print(
    "Los conteos coinciden con la Actividad 3.27."
)

# ------------------------------------------------------------
# 4. ESTRATEGIAS DE EVALUACIÓN ACUMULADA
# ------------------------------------------------------------

print("\n=== 3.28.4 ESTRATEGIAS ACUMULADAS ===")

tabla_politicas_328 = (
    tabla_resumen_acumulado_327[
        [
            "Semana",
            "Modelo",
            "Alertas_acumuladas",
            "Porcentaje_alertado",
            "Retiros_detectados_acumulados",
            "Cobertura_retiros_pct",
            "Falsas_alertas_acumuladas",
            "Precision_alertas_pct",
            "Cobertura_14d_pct",
            "Cobertura_28d_pct"
        ]
    ]
    .copy()
)

n_rondas_328 = {
    2: 1,
    4: 2,
    6: 3,
    8: 4
}

tabla_politicas_328["Rondas"] = (
    tabla_politicas_328["Semana"]
    .map(n_rondas_328)
)

display(
    tabla_politicas_328.round(2)
)

# ------------------------------------------------------------
# 5. BENEFICIO DESPUÉS DE LA PRIMERA RONDA
# ------------------------------------------------------------

print("\n=== 3.28.5 APORTE ADICIONAL POSTERIOR A S2 ===")

filas_incrementales_328 = []

for modelo in MODELOS_328:

    inicial = tabla_politicas_328.loc[
        tabla_politicas_328["Modelo"].eq(modelo)
        &
        tabla_politicas_328["Semana"].eq(2)
    ].iloc[0]

    final = tabla_politicas_328.loc[
        tabla_politicas_328["Modelo"].eq(modelo)
        &
        tabla_politicas_328["Semana"].eq(8)
    ].iloc[0]

    nuevas = (
        final["Alertas_acumuladas"]
        - inicial["Alertas_acumuladas"]
    )

    nuevos_retiros = (
        final["Retiros_detectados_acumulados"]
        - inicial["Retiros_detectados_acumulados"]
    )

    nuevas_falsas = (
        final["Falsas_alertas_acumuladas"]
        - inicial["Falsas_alertas_acumuladas"]
    )

    nuevos_correctos = nuevas - nuevas_falsas

    filas_incrementales_328.append({
        "Modelo": modelo,
        "Alertas_adicionales": nuevas,
        "Retiros_adicionales": nuevos_retiros,
        "Falsas_adicionales": nuevas_falsas,
        "Alertas_correctas_adicionales": (
            nuevos_correctos
        ),
        "Precision_incremental_pct": pct_328(
            nuevos_correctos, nuevas
        ),
        "Ganancia_cobertura_retiros_pp": (
            final["Cobertura_retiros_pct"]
            - inicial["Cobertura_retiros_pct"]
        ),
        "Ganancia_cobertura_28d_pp": (
            final["Cobertura_28d_pct"]
            - inicial["Cobertura_28d_pct"]
        )
    })

tabla_incremental_328 = pd.DataFrame(
    filas_incrementales_328
)

display(
    tabla_incremental_328.round(2)
)

# ------------------------------------------------------------
# 6. FRONTERA DE COBERTURA Y CARGA
# ------------------------------------------------------------

print("\n=== 3.28.6 FRONTERA OPERATIVA ===")

# Una estrategia es dominada si otra produce:
# - una carga menor o igual
# - una cobertura a 28 días mayor o igual
# con al menos una desigualdad estricta.

carga_328 = tabla_politicas_328[
    "Porcentaje_alertado"
].to_numpy()

cobertura_328 = tabla_politicas_328[
    "Cobertura_28d_pct"
].to_numpy()

dominada_328 = np.zeros(
    len(tabla_politicas_328),
    dtype=bool
)

for i in range(len(tabla_politicas_328)):

    for j in range(len(tabla_politicas_328)):

        if i == j:
            continue

        menor_carga = (
            carga_328[j] <= carga_328[i]
        )

        mayor_cobertura = (
            cobertura_328[j] >= cobertura_328[i]
        )

        mejora_estricta = (
            (carga_328[j] < carga_328[i])
            or
            (cobertura_328[j] > cobertura_328[i])
        )

        if (
            menor_carga
            and mayor_cobertura
            and mejora_estricta
        ):
            dominada_328[i] = True
            break

tabla_politicas_328["Dominada"] = dominada_328

tabla_frontera_328 = (
    tabla_politicas_328.loc[
        ~tabla_politicas_328["Dominada"],
        [
            "Modelo",
            "Semana",
            "Rondas",
            "Porcentaje_alertado",
            "Cobertura_28d_pct",
            "Cobertura_retiros_pct",
            "Precision_alertas_pct"
        ]
    ]
    .sort_values("Porcentaje_alertado")
    .reset_index(drop=True)
)

display(
    tabla_frontera_328.round(2)
)

# ------------------------------------------------------------
# 7. ESCENARIOS HIPOTÉTICOS DE CAPACIDAD
# ------------------------------------------------------------

print("\n=== 3.28.7 ESCENARIOS ILUSTRATIVOS ===")

filas_capacidad_328 = []

for capacidad in CAPACIDADES_328:

    posibles = (
        tabla_politicas_328.loc[
            tabla_politicas_328[
                "Porcentaje_alertado"
            ] <= capacidad
        ]
        .sort_values(
            [
                "Cobertura_28d_pct",
                "Porcentaje_alertado"
            ],
            ascending=[False, True]
        )
    )

    if posibles.empty:

        filas_capacidad_328.append({
            "Capacidad_pct": capacidad,
            "Modelo": "Ninguno",
            "Ultima_semana": np.nan,
            "Rondas": np.nan,
            "Carga_pct": np.nan,
            "Cobertura_28d_pct": np.nan
        })

    else:

        candidato = posibles.iloc[0]

        filas_capacidad_328.append({
            "Capacidad_pct": capacidad,
            "Modelo": candidato["Modelo"],
            "Ultima_semana": candidato["Semana"],
            "Rondas": candidato["Rondas"],
            "Carga_pct": candidato[
                "Porcentaje_alertado"
            ],
            "Cobertura_28d_pct": candidato[
                "Cobertura_28d_pct"
            ]
        })

tabla_capacidad_328 = pd.DataFrame(
    filas_capacidad_328
)

display(
    tabla_capacidad_328.round(2)
)

print(
    "Escenarios ilustrativos: NO son una "
    "selección final validada."
)

# ------------------------------------------------------------
# 8. GRÁFICO DE PRECISIÓN MARGINAL
# ------------------------------------------------------------

print("\n=== 3.28.8 GRÁFICOS ===")

modelos_long_328 = [
    "LR B",
    "RF B",
    "Transformer C"
]

fig, ax = plt.subplots(figsize=(10, 5))

for modelo in modelos_long_328:

    sub = tabla_marginal_resumen_328.loc[
        tabla_marginal_resumen_328[
            "Modelo"
        ].eq(modelo)
    ].sort_values("Semana")

    ax.plot(
        sub["Semana"],
        sub["Precision_nuevas_pct"],
        marker="o",
        linewidth=2,
        label=modelo
    )

ax.set_title(
    "Precisión de las nuevas alertas por ronda"
)
ax.set_xlabel("Semana")
ax.set_ylabel("Alertas nuevas correctas (%)")
ax.set_xticks(SEMANAS_328)
ax.set_ylim(0, 100)
ax.grid(alpha=0.3)
ax.legend()

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 9. GRÁFICO DE FRONTERA OPERATIVA
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(10, 5))

for modelo in MODELOS_328:

    sub = tabla_politicas_328.loc[
        tabla_politicas_328["Modelo"].eq(modelo)
    ].sort_values("Porcentaje_alertado")

    ax.plot(
        sub["Porcentaje_alertado"],
        sub["Cobertura_28d_pct"],
        marker="o",
        linewidth=1.5,
        label=modelo
    )

frontera = tabla_frontera_328.sort_values(
    "Porcentaje_alertado"
)

ax.plot(
    frontera["Porcentaje_alertado"],
    frontera["Cobertura_28d_pct"],
    linestyle="--",
    linewidth=2,
    label="Frontera no dominada"
)

ax.set_title(
    "Carga acumulada frente a cobertura oportuna\n"
    "Cohorte inicial de semana 2"
)
ax.set_xlabel(
    "Matrículas iniciales alertadas (%)"
)
ax.set_ylabel(
    "Retiros iniciales detectados ≥28 días antes (%)"
)

ax.grid(alpha=0.3)
ax.legend()

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 10. CONTROLES FINALES
# ------------------------------------------------------------

print("\n=== 3.28.9 CONTROLES FINALES ===")

assert len(tabla_marginal_328) == 40
assert len(tabla_marginal_resumen_328) == 20
assert len(tabla_politicas_328) == 20
assert len(tabla_incremental_328) == 5

assert tabla_marginal_328[
    "Precision_nuevas_pct"
].dropna().between(0, 100).all()

assert tabla_marginal_328[
    "Nuevos_retiros_28d"
].le(
    tabla_marginal_328["Nuevos_retiros"]
).all()

assert tabla_politicas_328[
    "Porcentaje_alertado"
].between(0, 100).all()

assert tabla_politicas_328[
    "Cobertura_28d_pct"
].between(0, 100).all()

assert (
    tabla_incremental_328[
        "Alertas_adicionales"
    ] >= 0
).all()

print("Modelos analizados: 5")
print("Rondas evaluadas: 4")
print("Estrategias acumuladas: 20")
print("Escenarios hipotéticos de capacidad: 3")
print("Primera alerta considerada una sola vez")
print("Etiquetas finales usadas solo para evaluación")
print("Modelos reentrenados: 0")
print("Umbrales modificados: 0")
print("Conjunto de prueba independiente: NO utilizado")

print("\n=== ACTIVIDAD 3.28 FINALIZADA ===")



### 3.28.10. Resultados e interpretación

Se completó el análisis de eficiencia marginal de las alertas y de escenarios hipotéticos de capacidad institucional.

Se utilizaron las predicciones fuera de muestra de los cinco modelos en las semanas 2, 4, 6 y 8, conservando una cohorte inicial de 22.412 matrículas, de las cuales 4.538 presentaron posteriormente el resultado `Withdrawn`.

No se entrenaron nuevos modelos ni se modificaron los umbrales originales de clasificación.

#### Eficiencia marginal de las evaluaciones posteriores

Después de la semana 2, Regresión Logística B generó un promedio de 2.795,5 primeras alertas adicionales e identificó 676,5 retiros futuros adicionales.

Estas alertas incluyeron 1.103 falsas alertas y presentaron una precisión incremental del 60,54 % para la clasificación binaria de resultados adversos.

Random Forest B identificó 712,5 retiros adicionales mediante 3.150 nuevas alertas, con una precisión incremental del 56,97 %.

El Transformer C identificó 673,5 retiros adicionales mediante 2.868 nuevas alertas, con una precisión incremental del 58,46 %.

Por tanto, la mayor cobertura adicional de retiros no siempre estuvo asociada con una mayor precisión de las alertas.

#### Precisión por ronda

En los modelos longitudinales, la precisión de las nuevas alertas fue mayor durante la primera evaluación que durante las rondas intermedias.

Regresión Logística B presentó precisiones de 65,83 %, 59,74 %, 59,90 % y 63,04 % en las semanas 2, 4, 6 y 8, respectivamente.

El Transformer C alcanzó 64,71 % en las nuevas alertas de la semana 8, mientras que Random Forest B obtuvo 58,11 %.

Estos resultados evidencian que las nuevas alertas emitidas durante diferentes momentos del curso no presentan la misma precisión retrospectiva.

#### Capacidad hipotética de intervención

Se analizaron capacidades acumuladas máximas equivalentes al 40 %, 45 % y 50 % de las matrículas iniciales.

Entre las veinte estrategias discretas evaluadas:

| Capacidad máxima | Estrategia | Carga observada | Cobertura de retiros ≥28 días |
|---|---|---:|---:|
| 40 % | Regresión Logística B, solo semana 2 | 39,50 % | 43,75 % |
| 45 % | Random Forest B, semanas 2 y 4 | 43,58 % | 45,82 % |
| 50 % | Regresión Logística B, semanas 2, 4 y 6 | 49,11 % | 52,35 % |

La estrategia con mayor cobertura oportuna depende de la restricción de capacidad considerada.

Se construyó adicionalmente una frontera de estrategias no dominadas, considerando simultáneamente el porcentaje de matrículas inicialmente alertadas y la cobertura acumulada de retiros identificados con al menos 28 días de anticipación.

#### Interpretación operativa

Los resultados muestran que las evaluaciones sucesivas permiten recuperar matrículas con resultados adversos que no fueron identificadas durante la primera ronda.

Sin embargo, estos beneficios están asociados con un incremento en la cantidad de primeras alertas emitidas y de falsas alertas.

Por tanto, la selección de un sistema de alerta temprana debería considerar tanto el rendimiento predictivo como la capacidad institucional para ofrecer seguimiento oportuno.

### 3.28.11. Limitaciones

Los escenarios de capacidad son hipotéticos y no representan presupuestos reales de una universidad.

La capacidad fue expresada como porcentaje acumulado de matrículas con al menos una alerta, no como número de estudiantes atendidos simultáneamente.

La frontera de eficiencia se construyó sobre veinte estrategias discretas, sin modificar los umbrales de clasificación y sin incorporar costos institucionales.

Los resultados proceden del conjunto de desarrollo y fueron examinados durante la selección exploratoria de estrategias.

El resultado `Withdrawn` identifica retiros de matrículas académicas y no necesariamente abandono universitario definitivo.

Las alertas son simulaciones retrospectivas y no intervenciones educativas reales.

Las dos repeticiones de validación cruzada no constituyen muestras independientes.

El conjunto de prueba independiente permaneció reservado.

### 3.28.12. Conclusión

Los modelos longitudinales permitieron incrementar la cobertura de retiros futuros mediante evaluaciones sucesivas, aunque este beneficio estuvo acompañado de un aumento de falsas alertas.

Regresión Logística B presentó una combinación competitiva de precisión incremental y cobertura oportuna, pero no fue la alternativa seleccionada para todos los escenarios hipotéticos de capacidad.

**Conclusión principal:** la evaluación de sistemas de alerta académica temprana debe ir más allá de las métricas tradicionales de Machine Learning e incorporar criterios de oportunidad de detección, precisión marginal y capacidad de atención.

Los resultados respaldan el potencial de los modelos longitudinales para apoyar la priorización de casos, pero no permiten todavía recomendar una política institucional definitiva.



### 3.29. Auditoría de calibración probabilística

#### Objetivo

Evaluar la calidad de las probabilidades de riesgo adverso generadas por los cinco modelos en las semanas 2, 4, 6 y 8.

La calibración se refiere a la correspondencia entre las probabilidades predichas y las frecuencias observadas de resultados adversos.

Por ejemplo, un modelo está bien calibrado alrededor del valor 0,70 si las matrículas a las que asigna aproximadamente ese riesgo presentan resultados adversos con una frecuencia cercana al 70 %.

#### Métricas de evaluación

**Brier Score**

Mide el error cuadrático medio entre la probabilidad predicha y el resultado binario real.

Valores menores indican un mejor desempeño probabilístico, combinando aspectos de calibración y discriminación.

**Log Loss**

Evalúa la calidad de las probabilidades mediante pérdida logarítmica. Penaliza especialmente las predicciones incorrectas realizadas con gran confianza.

Valores menores indican mejor desempeño.

**Expected Calibration Error (ECE)**

Compara la probabilidad media predicha con la proporción observada de resultados adversos dentro de intervalos de probabilidad.

Se utilizarán diez intervalos de amplitud uniforme.

El ECE constituye un indicador descriptivo aproximado y depende del número y construcción de los intervalos.

**Sesgo medio de probabilidad**

Se calculará la diferencia entre la probabilidad media predicha y la prevalencia observada de resultados adversos.

Un valor positivo indica sobreestimación media del riesgo; uno negativo indica subestimación media.

#### Gráficos de calibración

Se construirán diagramas de confiabilidad para comparar las probabilidades predichas con las frecuencias observadas.

Se prestará especial atención a Regresión Logística B, Random Forest B y Transformer C en las semanas 2 y 8.

#### Procedimiento

Se recuperarán exclusivamente las probabilidades fuera de muestra generadas durante las evaluaciones anteriores.

Las métricas se calcularán por repetición y después se promediarán.

No se ajustarán calibradores, no se modificarán probabilidades y no se entrenarán nuevos modelos.

#### Limitaciones

El Brier Score y la pérdida logarítmica no miden exclusivamente calibración.

El ECE depende de la discretización y puede ocultar diferencias locales.

Los resultados corresponden a predicciones fuera de muestra del conjunto de desarrollo y a modelos ajustados sobre diferentes folds.

La auditoría no sustituye la evaluación probabilística del conjunto de prueba independiente, que permanecerá reservado.


In [ ]:

# ============================================================
# ACTIVIDAD 3.29 - AUDITORÍA DE CALIBRACIÓN PROBABILÍSTICA
# ============================================================

# Asistencia IA: ChatGPT, octubre de 2026.
# Prompt:
# "Evaluar calibración de probabilidades OOF de los cinco
# modelos OULAD en semanas 2, 4, 6 y 8. Calcular Brier,
# Log Loss, ECE, sesgo medio y curvas de confiabilidad.
# Verificar correspondencia con resultados anteriores.
# No modificar modelos, probabilidades o test."

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    brier_score_loss,
    log_loss
)
from IPython.display import display

# ------------------------------------------------------------
# 1. CONFIGURACIÓN
# ------------------------------------------------------------

SEMANAS_329 = [2, 4, 6, 8]

MODELOS_329 = [
    "LR A",
    "RF A",
    "LR B",
    "RF B",
    "Transformer C"
]

CLAVES_329 = [
    "code_module",
    "code_presentation",
    "id_student"
]

N_BINS_329 = 10
EPS_329 = 1e-12

REQUERIDAS_329 = [
    "predicciones_conjuntas_s2",
    "resultados_cv_ventanas",
    "resultados_transformer_C_323",
    "resultados_transformer_C_s8",
    "obtener_predicciones_324"
]

faltantes_329 = [
    nombre for nombre in REQUERIDAS_329
    if nombre not in globals()
]

assert not faltantes_329, (
    "Faltan variables del notebook: "
    + ", ".join(faltantes_329)
)

print("=== 3.29.1 CONFIGURACIÓN ===")
print("Semanas:", SEMANAS_329)
print("Modelos:", MODELOS_329)
print("Intervalos de calibración:", N_BINS_329)
print("Reentrenamiento: NO")

# ------------------------------------------------------------
# 2. RECUPERACIÓN DE PROBABILIDADES
# ------------------------------------------------------------

CANDIDATAS_PROB_329 = [
    "prob_adverso",
    "proba_adverso",
    "probabilidad_adverso",
    "probabilidad_adversa",
    "probabilidad_riesgo",
    "y_prob",
    "y_proba",
    "y_score",
    "prob_1",
    "proba_1",
    "probabilidad_1",
    "probabilidad_clase_1"
]

def recuperar_probabilidades_329(semana, modelo):

    if modelo == "Transformer C":

        if semana == 8:
            resultados = (
                resultados_transformer_C_s8
            )
        else:
            resultados = {
                clave: resultado
                for clave, resultado in (
                    resultados_transformer_C_323.items()
                )
                if clave[0] == semana
            }

        assert len(resultados) == 10

        fuente = pd.concat(
            [
                resultado["predicciones"]
                for resultado in resultados.values()
            ],
            ignore_index=True
        )

    else:

        prefijo, experimento = modelo.split()

        algoritmo = (
            "LogisticRegression"
            if prefijo == "LR"
            else "RandomForest"
        )

        if semana == 2:
            origen = predicciones_conjuntas_s2
        else:
            origen = resultados_cv_ventanas[
                (semana, algoritmo)
            ]["predicciones"]

        fuente = origen.loc[
            origen["Algoritmo"].eq(algoritmo)
            &
            origen["Experimento"].eq(experimento)
        ].copy()

    disponibles = [
        nombre
        for nombre in CANDIDATAS_PROB_329
        if nombre in fuente.columns
    ]

    if not disponibles:
        raise ValueError(
            f"No se encontró la columna de probabilidad "
            f"para {modelo}, semana {semana}. "
            f"Columnas disponibles: "
            f"{fuente.columns.tolist()}. "
            "Es necesario identificar explícitamente "
            "la probabilidad de la clase adversa."
        )

    # Prioridad establecida en la lista anterior.
    columna_prob = disponibles[0]

    columnas = (
        CLAVES_329
        + ["Repeticion", "y_real", "y_pred"]
        + [columna_prob]
    )

    assert all(
        c in fuente.columns
        for c in columnas
    )

    pred = fuente[columnas].copy()

    pred = pred.rename(
        columns={columna_prob: "prob_adverso_329"}
    )

    pred["prob_adverso_329"] = pd.to_numeric(
        pred["prob_adverso_329"],
        errors="raise"
    )

    assert pred["prob_adverso_329"].notna().all()

    assert np.isfinite(
        pred["prob_adverso_329"].to_numpy()
    ).all()

    assert pred["prob_adverso_329"].between(
        0, 1
    ).all()

    assert not pred.duplicated(
        subset=CLAVES_329 + ["Repeticion"]
    ).any()

    assert set(
        pred["Repeticion"].unique()
    ) == {1, 2}

    # Verificar contra las predicciones OOF
    # ya utilizadas en la Actividad 3.24.
    referencia = obtener_predicciones_324(
        semana, modelo
    )

    auditoria = pred.merge(
        referencia,
        on=CLAVES_329 + ["Repeticion"],
        how="inner",
        validate="one_to_one",
        suffixes=("", "_referencia")
    )

    assert len(auditoria) == len(pred)
    assert len(auditoria) == len(referencia)

    assert (
        auditoria["y_real"]
        == auditoria["y_real_referencia"]
    ).all()

    assert (
        auditoria["y_pred"]
        == auditoria["y_pred_referencia"]
    ).all()

    # El experimento usó umbral 0.5.
    prob = pred["prob_adverso_329"].to_numpy()
    clase = pred["y_pred"].to_numpy(dtype=int)

    discordancias = (
        clase != (prob >= 0.5).astype(int)
    )

    # Admitir únicamente posibles empates
    # numéricos exactamente en el umbral.
    assert not (
        discordancias
        &
        ~np.isclose(prob, 0.5, atol=1e-8)
    ).any(), (
        f"Las probabilidades de {modelo}, "
        f"semana {semana} no corresponden "
        "a las clases predichas. "
        "Revisar si la columna representa "
        "realmente la clase adversa."
    )

    pred["Semana"] = semana
    pred["Modelo"] = modelo

    return pred

# ------------------------------------------------------------
# 3. UNIFICAR PREDICCIONES
# ------------------------------------------------------------

print("\n=== 3.29.2 PROBABILIDADES RECUPERADAS ===")

partes_329 = []

for semana in SEMANAS_329:

    for modelo in MODELOS_329:

        pred = recuperar_probabilidades_329(
            semana, modelo
        )

        partes_329.append(pred)

        print(
            f"Semana {semana} | {modelo}: "
            f"{len(pred)} predicciones verificadas"
        )

predicciones_prob_329 = pd.concat(
    partes_329,
    ignore_index=True
)

assert not predicciones_prob_329.duplicated(
    subset=(
        CLAVES_329
        + ["Semana", "Modelo", "Repeticion"]
    )
).any()

print(
    "Total de predicciones:",
    len(predicciones_prob_329)
)

# ------------------------------------------------------------
# 4. FUNCIÓN PARA ERROR DE CALIBRACIÓN
# ------------------------------------------------------------

def confiabilidad_329(y, probabilidades):

    y = np.asarray(y, dtype=int)

    p = np.asarray(
        probabilidades,
        dtype=float
    )

    indices = np.minimum(
        np.floor(p * N_BINS_329).astype(int),
        N_BINS_329 - 1
    )

    tabla = pd.DataFrame({
        "bin": indices,
        "prob": p,
        "real": y
    })

    resumen = (
        tabla.groupby("bin")
        .agg(
            n=("real", "size"),
            prob_media=("prob", "mean"),
            frecuencia_observada=("real", "mean")
        )
        .reset_index()
    )

    resumen["error_absoluto"] = (
        resumen["prob_media"]
        - resumen["frecuencia_observada"]
    ).abs()

    ece = (
        resumen["n"]
        / len(tabla)
        * resumen["error_absoluto"]
    ).sum()

    return float(ece), resumen

# ------------------------------------------------------------
# 5. MÉTRICAS POR REPETICIÓN
# ------------------------------------------------------------

print("\n=== 3.29.3 MÉTRICAS DE CALIBRACIÓN ===")

filas_metricas_329 = []
filas_bins_329 = []

for (semana, modelo, repeticion), grupo in (
    predicciones_prob_329.groupby(
        ["Semana", "Modelo", "Repeticion"]
    )
):

    y = grupo["y_real"].to_numpy(dtype=int)

    p = grupo[
        "prob_adverso_329"
    ].to_numpy(dtype=float)

    prevalencia = y.mean()

    brier = brier_score_loss(y, p)

    perdida = log_loss(
        y,
        np.clip(p, EPS_329, 1 - EPS_329),
        labels=[0, 1]
    )

    ece, bins = confiabilidad_329(y, p)

    # Referencia: probabilidad constante
    # equivalente a la prevalencia del conjunto.
    brier_referencia = np.mean(
        (y - prevalencia) ** 2
    )

    filas_metricas_329.append({
        "Semana": semana,
        "Modelo": modelo,
        "Repeticion": repeticion,
        "Matriculas": len(y),
        "Prevalencia": prevalencia,
        "Probabilidad_media": p.mean(),
        "Sesgo_medio_pp": (
            100 * (p.mean() - prevalencia)
        ),
        "Brier": brier,
        "Brier_referencia": brier_referencia,
        "Brier_skill": (
            1 - brier / brier_referencia
        ),
        "Log_Loss": perdida,
        "ECE_10_bins": ece
    })

    bins["Semana"] = semana
    bins["Modelo"] = modelo
    bins["Repeticion"] = repeticion

    filas_bins_329.append(bins)

tabla_metricas_cal_329 = pd.DataFrame(
    filas_metricas_329
)

tabla_bins_329 = pd.concat(
    filas_bins_329,
    ignore_index=True
)

assert len(tabla_metricas_cal_329) == 40

# ------------------------------------------------------------
# 6. RESUMEN POR MODELO Y SEMANA
# ------------------------------------------------------------

tabla_resumen_cal_329 = (
    tabla_metricas_cal_329.groupby(
        ["Semana", "Modelo"],
        as_index=False
    )
    .agg(
        Matriculas_por_repeticion=(
            "Matriculas", "first"
        ),
        Prevalencia=("Prevalencia", "mean"),
        Probabilidad_media=(
            "Probabilidad_media", "mean"
        ),
        Sesgo_medio_pp=("Sesgo_medio_pp", "mean"),
        Brier=("Brier", "mean"),
        Brier_skill=("Brier_skill", "mean"),
        Log_Loss=("Log_Loss", "mean"),
        ECE_10_bins=("ECE_10_bins", "mean")
    )
)

display(
    tabla_resumen_cal_329.round(5)
)

# ------------------------------------------------------------
# 7. COMPARACIÓN ENTRE MODELOS LONGITUDINALES
# ------------------------------------------------------------

print("\n=== 3.29.4 COMPARACIÓN LONGITUDINAL ===")

tabla_cal_long_329 = (
    tabla_resumen_cal_329.loc[
        tabla_resumen_cal_329["Modelo"].isin(
            ["LR B", "RF B", "Transformer C"]
        ),
        [
            "Semana",
            "Modelo",
            "Brier",
            "Log_Loss",
            "ECE_10_bins",
            "Sesgo_medio_pp"
        ]
    ]
    .sort_values(["Semana", "Modelo"])
)

display(
    tabla_cal_long_329.round(5)
)

# ------------------------------------------------------------
# 8. TABLAS DE CONFIABILIDAD
# ------------------------------------------------------------

print("\n=== 3.29.5 CONFIABILIDAD POR INTERVALO ===")

tabla_confiabilidad_329 = (
    tabla_bins_329.groupby(
        ["Semana", "Modelo", "bin"],
        as_index=False
    )
    .agg(
        N_medio=("n", "mean"),
        Probabilidad_media=("prob_media", "mean"),
        Frecuencia_observada=(
            "frecuencia_observada", "mean"
        )
    )
)

tabla_confiabilidad_329["Diferencia"] = (
    tabla_confiabilidad_329[
        "Frecuencia_observada"
    ]
    -
    tabla_confiabilidad_329[
        "Probabilidad_media"
    ]
)

display(
    tabla_confiabilidad_329.loc[
        tabla_confiabilidad_329["Semana"].eq(2)
        &
        tabla_confiabilidad_329["Modelo"].eq("LR B")
    ].round(4)
)

# ------------------------------------------------------------
# 9. DIAGRAMAS DE CONFIABILIDAD
# ------------------------------------------------------------

print("\n=== 3.29.6 GRÁFICOS DE CALIBRACIÓN ===")

for semana in [2, 8]:

    fig, ax = plt.subplots(figsize=(8, 7))

    ax.plot(
        [0, 1],
        [0, 1],
        linestyle="--",
        linewidth=1.5,
        label="Calibración ideal"
    )

    for modelo in [
        "LR B",
        "RF B",
        "Transformer C"
    ]:

        sub = tabla_confiabilidad_329.loc[
            tabla_confiabilidad_329[
                "Semana"
            ].eq(semana)
            &
            tabla_confiabilidad_329[
                "Modelo"
            ].eq(modelo)
            &
            tabla_confiabilidad_329[
                "N_medio"
            ].ge(30)
        ].sort_values("bin")

        ax.plot(
            sub["Probabilidad_media"],
            sub["Frecuencia_observada"],
            marker="o",
            linewidth=2,
            label=modelo
        )

    ax.set_title(
        "Diagrama de confiabilidad\n"
        f"Semana {semana}"
    )

    ax.set_xlabel(
        "Probabilidad adversa media predicha"
    )

    ax.set_ylabel(
        "Frecuencia observada de resultados adversos"
    )

    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)

    ax.grid(alpha=0.3)
    ax.legend()

    plt.tight_layout()
    plt.show()

# ------------------------------------------------------------
# 10. EVOLUCIÓN DEL BRIER SCORE
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(10, 5))

for modelo in MODELOS_329:

    sub = tabla_resumen_cal_329.loc[
        tabla_resumen_cal_329[
            "Modelo"
        ].eq(modelo)
    ].sort_values("Semana")

    ax.plot(
        sub["Semana"],
        sub["Brier"],
        marker="o",
        linewidth=2,
        label=modelo
    )

ax.set_title(
    "Brier Score por semana\n"
    "Valores menores indican mejor desempeño probabilístico"
)

ax.set_xlabel("Semana de predicción")
ax.set_ylabel("Brier Score")

ax.set_xticks(SEMANAS_329)
ax.grid(alpha=0.3)
ax.legend()

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 11. CONTROLES FINALES
# ------------------------------------------------------------

print("\n=== 3.29.7 CONTROLES FINALES ===")

assert len(tabla_resumen_cal_329) == 20

assert tabla_metricas_cal_329[
    "Brier"
].between(0, 1).all()

assert tabla_metricas_cal_329[
    "ECE_10_bins"
].between(0, 1).all()

assert (
    tabla_metricas_cal_329[
        "Log_Loss"
    ] >= 0
).all()

assert (
    tabla_metricas_cal_329[
        "Brier_referencia"
    ] > 0
).all()

assert np.isfinite(
    tabla_resumen_cal_329[
        [
            "Brier",
            "Log_Loss",
            "ECE_10_bins"
        ]
    ].to_numpy()
).all()

print("Modelos evaluados: 5")
print("Semanas evaluadas: 4")
print("Evaluaciones por repetición: 40")
print("Intervalos de calibración: 10")
print("Probabilidades originales verificadas")
print("Modelos reentrenados: 0")
print("Calibradores ajustados: 0")
print("Umbrales modificados: 0")
print("Conjunto de prueba independiente: NO utilizado")

print("\n=== ACTIVIDAD 3.29 FINALIZADA ===")



### 3.29.8. Resultados e interpretación

Se completó la auditoría de calibración probabilística de los cinco modelos en las semanas 2, 4, 6 y 8.

Se verificaron 871.280 predicciones fuera de muestra correspondientes a Regresión Logística A, Random Forest A, Regresión Logística B, Random Forest B y Transformer C.

Se utilizaron cuatro indicadores principales: Brier Score, Log Loss, Expected Calibration Error (ECE) con diez intervalos y sesgo medio de probabilidad.

#### Calidad probabilística

Regresión Logística B obtuvo el menor Brier Score y Log Loss de los cinco modelos en las cuatro semanas.

Su Brier Score disminuyó de 0,20732 en la semana 2 a 0,17533 en la semana 8, mientras que su Log Loss disminuyó de 0,60161 a 0,52515.

En la semana 8, el Brier Score de Regresión Logística B fue inferior al de Transformer C (0,17880) y Random Forest B (0,18220).

Estas métricas reflejan el desempeño de las predicciones probabilísticas en su conjunto, no exclusivamente la calibración.

#### Calibración global

Los diagramas de confiabilidad mostraron una correspondencia relativamente cercana entre probabilidades predichas y frecuencias observadas de resultados adversos.

Regresión Logística B presentó un ECE de 0,00930 en la semana 2 y de 0,01075 en la semana 8.

En la semana 8, Transformer C obtuvo un ECE de 0,01112 y Random Forest B de 0,01985.

Estos resultados sugieren una calibración global razonable, especialmente para Regresión Logística B y Transformer C.

No obstante, el ECE depende del esquema de discretización y puede ocultar desviaciones relevantes en determinados rangos de probabilidad.

#### Sesgo medio de probabilidad

La probabilidad media estimada por Regresión Logística B fue prácticamente igual a la prevalencia adversa observada en las cuatro ventanas.

En la semana 8, la prevalencia fue de 0,41840 y la probabilidad media predicha de 0,41845.

El sesgo medio de probabilidad fue de aproximadamente +0,005 puntos porcentuales.

La proximidad de ambas medias no garantiza calibración perfecta en todos los intervalos ni en los diferentes grupos estudiantiles.

### 3.29.9. Limitaciones

Las métricas se calcularon sobre predicciones fuera de muestra del conjunto de desarrollo.

Las cohortes cambian entre ventanas, de modo que la evolución temporal de los indicadores no puede atribuirse exclusivamente al aumento de información longitudinal.

El Brier Score y el Log Loss evalúan la calidad probabilística global, combinando discriminación y calibración.

El ECE es un indicador aproximado que depende de la cantidad y amplitud de los intervalos utilizados.

Los diagramas de confiabilidad no incluyeron intervalos de incertidumbre para las frecuencias observadas.

La calibración global no garantiza calibración equivalente entre grupos demográficos, socioeconómicos o académicos.

No se ajustaron calibradores ni se modificaron las probabilidades originales.

El conjunto de prueba independiente permaneció reservado.

### 3.29.10. Conclusión

Los modelos longitudinales mostraron mejor desempeño probabilístico que los modelos estáticos.

Regresión Logística B obtuvo consistentemente el menor Brier Score y Log Loss, además de presentar una calibración global razonable.

El Transformer C también produjo probabilidades competitivas, especialmente en las primeras ventanas.

**Conclusión principal:** Regresión Logística B presenta una combinación favorable de rendimiento predictivo, calidad probabilística e interpretabilidad relativa, lo que respalda su consideración como modelo principal candidato para un sistema de alerta temprana.

Antes de proponer una aplicación institucional, resulta necesario examinar la consistencia de estas métricas entre subgrupos estudiantiles y evaluar posibles diferencias de rendimiento.



### 3.30. Auditoría de equidad y rendimiento entre subgrupos estudiantiles

#### Objetivo

Evaluar si los modelos de Machine Learning presentan diferencias descriptivas de rendimiento entre subgrupos de estudiantes de OULAD.

La auditoría incluirá los cinco modelos en las semanas 2, 4, 6 y 8, utilizando exclusivamente predicciones fuera de muestra.

#### Variables de agrupación

Se analizarán siete dimensiones:

1. Género (`gender`).
2. Grupo etario (`age_band`).
3. Indicador socioeconómico (`imd_band`).
4. Discapacidad declarada (`disability`).
5. Educación previa (`highest_education`).
6. Región (`region`).
7. Módulo académico (`code_module`).

Las primeras seis dimensiones permiten examinar heterogeneidad demográfica y socioeconómica. El módulo académico permitirá analizar heterogeneidad contextual y no se interpretará como una característica demográfica protegida.

#### Métricas por subgrupo

Para cada grupo se calcularán:

- Número de matrículas y prevalencia de resultados adversos.
- Recall de la clase adversa o tasa de verdaderos positivos (TPR).
- Tasa de falsos positivos (FPR).
- Precisión de las alertas.
- Porcentaje de matrículas que reciben alertas.
- Brier Score.
- Diferencia entre probabilidad media estimada y prevalencia observada.

#### Brechas entre grupos

Se calculará la diferencia entre el valor máximo y mínimo de recall y FPR dentro de cada dimensión.

Las brechas de recall y FPR se expresarán en puntos porcentuales.

Se empleará un criterio descriptivo de soporte mínimo de 150 matrículas, 50 resultados adversos y 50 favorables por grupo y repetición. Los grupos con menor soporte permanecerán visibles, pero no participarán en las comparaciones de brechas principales.

Estos umbrales constituyen criterios prácticos de estabilidad descriptiva, no pruebas de suficiencia estadística.

#### Interpretación responsable

Las diferencias observadas entre subgrupos no demostrarán por sí mismas discriminación algorítmica.

El rendimiento puede variar por diferencias en prevalencia, composición académica, disponibilidad de datos, patrones de comportamiento y otros factores.

Asimismo, la igualdad de una métrica no garantiza equidad en todas sus dimensiones.

Los resultados se interpretarán como una auditoría exploratoria para identificar aspectos que requieren investigación adicional.

#### Prevención de fuga de información

Las características demográficas se utilizarán para auditoría sobre predicciones ya generadas.

No se reentrenarán clasificadores, no se modificarán umbrales y no se utilizará el conjunto de prueba independiente.


In [ ]:

# ============================================================
# PROYECTO FINAL DE MACHINE LEARNING
# ACTIVIDAD 3.30 - AUDITORÍA DE EQUIDAD Y SUBGRUPOS
# ============================================================

# Asistencia IA: ChatGPT, octubre de 2026.
# Prompt:
# "Auditar cinco modelos OULAD en semanas 2,4,6,8
# por género, edad, IMD, discapacidad, educación,
# región y módulo. Calcular recall, FPR, precisión,
# tasa de alertas, Brier y sesgo probabilístico.
# Respetar soporte mínimo y evitar afirmaciones
# causales. No entrenar modelos ni usar test."

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

# ------------------------------------------------------------
# 1. CONFIGURACIÓN
# ------------------------------------------------------------

SEMANAS_330 = [2, 4, 6, 8]

MODELOS_330 = [
    "LR A",
    "RF A",
    "LR B",
    "RF B",
    "Transformer C"
]

CLAVES_330 = [
    "code_module",
    "code_presentation",
    "id_student"
]

DIMENSIONES_330 = [
    "gender",
    "age_band",
    "imd_band",
    "disability",
    "highest_education",
    "region",
    "code_module"
]

MIN_MATRICULAS_330 = 150
MIN_ADVERSOS_330 = 50
MIN_FAVORABLES_330 = 50

assert "predicciones_prob_329" in globals()
assert "oulad_tablas" in globals()

print("=== 3.30.1 CONFIGURACIÓN ===")
print("Semanas:", SEMANAS_330)
print("Modelos:", MODELOS_330)
print("Dimensiones:", DIMENSIONES_330)
print("Mínimo de matrículas:", MIN_MATRICULAS_330)
print("Mínimo de adversos:", MIN_ADVERSOS_330)
print("Mínimo de favorables:", MIN_FAVORABLES_330)

# ------------------------------------------------------------
# 2. METADATOS DE SUBGRUPOS
# ------------------------------------------------------------

info_330 = oulad_tablas["studentInfo.csv"]

metadatos_330 = info_330[
    CLAVES_330
    + [d for d in DIMENSIONES_330
       if d not in CLAVES_330]
].copy()

assert not metadatos_330.duplicated(
    subset=CLAVES_330
).any()

for dimension in DIMENSIONES_330:

    if dimension not in CLAVES_330:
        metadatos_330[dimension] = (
            metadatos_330[dimension]
            .fillna("No informado")
            .astype(str)
        )

datos_330 = predicciones_prob_329.merge(
    metadatos_330,
    on=CLAVES_330,
    how="left",
    validate="many_to_one"
)

assert len(datos_330) == len(
    predicciones_prob_329
)

for dimension in DIMENSIONES_330:
    assert datos_330[dimension].notna().all()

assert datos_330["y_real"].isin([0, 1]).all()
assert datos_330["y_pred"].isin([0, 1]).all()

assert datos_330[
    "prob_adverso_329"
].between(0, 1).all()

print("\n=== 3.30.2 DATOS DISPONIBLES ===")
print("Predicciones analizadas:", len(datos_330))
print("Matrículas agrupadas por módulo y estudiante")
print("Metadatos vinculados sin duplicaciones")

# ------------------------------------------------------------
# 3. INDICADORES INDIVIDUALES
# ------------------------------------------------------------

y_330 = datos_330["y_real"].to_numpy(
    dtype=int
)

p_330 = datos_330["y_pred"].to_numpy(
    dtype=int
)

proba_330 = datos_330[
    "prob_adverso_329"
].to_numpy(dtype=float)

datos_330["TP_330"] = (
    (y_330 == 1) & (p_330 == 1)
).astype(np.int8)

datos_330["TN_330"] = (
    (y_330 == 0) & (p_330 == 0)
).astype(np.int8)

datos_330["FP_330"] = (
    (y_330 == 0) & (p_330 == 1)
).astype(np.int8)

datos_330["FN_330"] = (
    (y_330 == 1) & (p_330 == 0)
).astype(np.int8)

datos_330["Brier_individual_330"] = (
    proba_330 - y_330
) ** 2

# ------------------------------------------------------------
# 4. AGRUPACIÓN POR DIMENSIÓN
# ------------------------------------------------------------

print("\n=== 3.30.3 CÁLCULO POR SUBGRUPO ===")

partes_grupos_330 = []

for dimension in DIMENSIONES_330:

    agrupacion = (
        datos_330.groupby(
            [
                "Semana",
                "Modelo",
                "Repeticion",
                dimension
            ],
            as_index=False,
            dropna=False
        )
        .agg(
            Matriculas=("y_real", "size"),
            Adversos=("y_real", "sum"),
            Alertas=("y_pred", "sum"),
            TP=("TP_330", "sum"),
            TN=("TN_330", "sum"),
            FP=("FP_330", "sum"),
            FN=("FN_330", "sum"),
            Brier=(
                "Brier_individual_330", "mean"
            ),
            Probabilidad_media=(
                "prob_adverso_329", "mean"
            )
        )
    )

    agrupacion = agrupacion.rename(
        columns={dimension: "Grupo"}
    )

    agrupacion["Dimension"] = dimension

    agrupacion["Favorables"] = (
        agrupacion["Matriculas"]
        - agrupacion["Adversos"]
    )

    partes_grupos_330.append(agrupacion)

tabla_grupos_rep_330 = pd.concat(
    partes_grupos_330,
    ignore_index=True
)

# ------------------------------------------------------------
# 5. MÉTRICAS POR GRUPO
# ------------------------------------------------------------

def division_pct_330(numerador, denominador):

    a = np.asarray(numerador, dtype=float)
    b = np.asarray(denominador, dtype=float)

    return np.divide(
        100 * a,
        b,
        out=np.full(a.shape, np.nan),
        where=b > 0
    )

t = tabla_grupos_rep_330

t["Prevalencia_pct"] = division_pct_330(
    t["Adversos"], t["Matriculas"]
)

t["Recall_adverso_pct"] = division_pct_330(
    t["TP"], t["Adversos"]
)

t["FPR_pct"] = division_pct_330(
    t["FP"], t["Favorables"]
)

t["Precision_pct"] = division_pct_330(
    t["TP"], t["Alertas"]
)

t["Tasa_alertas_pct"] = division_pct_330(
    t["Alertas"], t["Matriculas"]
)

t["Sesgo_probabilidad_pp"] = (
    100 * t["Probabilidad_media"]
    - t["Prevalencia_pct"]
)

assert (
    t["TP"] + t["TN"] + t["FP"] + t["FN"]
    == t["Matriculas"]
).all()

# ------------------------------------------------------------
# 6. RESUMEN DE DOS REPETICIONES
# ------------------------------------------------------------

tabla_grupos_resumen_330 = (
    tabla_grupos_rep_330.groupby(
        [
            "Semana",
            "Modelo",
            "Dimension",
            "Grupo"
        ],
        as_index=False,
        dropna=False
    )
    .agg(
        Matriculas=("Matriculas", "first"),
        Adversos=("Adversos", "first"),
        Favorables=("Favorables", "first"),
        Prevalencia_pct=(
            "Prevalencia_pct", "mean"
        ),
        Recall_adverso_pct=(
            "Recall_adverso_pct", "mean"
        ),
        Recall_std=(
            "Recall_adverso_pct", "std"
        ),
        FPR_pct=("FPR_pct", "mean"),
        Precision_pct=("Precision_pct", "mean"),
        Tasa_alertas_pct=(
            "Tasa_alertas_pct", "mean"
        ),
        Brier=("Brier", "mean"),
        Sesgo_probabilidad_pp=(
            "Sesgo_probabilidad_pp", "mean"
        ),
        Repeticiones=("Repeticion", "nunique")
    )
)

assert tabla_grupos_resumen_330[
    "Repeticiones"
].eq(2).all()

tabla_grupos_resumen_330[
    "Soporte_suficiente"
] = (
    tabla_grupos_resumen_330[
        "Matriculas"
    ].ge(MIN_MATRICULAS_330)
    &
    tabla_grupos_resumen_330[
        "Adversos"
    ].ge(MIN_ADVERSOS_330)
    &
    tabla_grupos_resumen_330[
        "Favorables"
    ].ge(MIN_FAVORABLES_330)
)

# ------------------------------------------------------------
# 7. EJEMPLOS: LR B EN SEMANAS 2 Y 8
# ------------------------------------------------------------

print("\n=== 3.30.4 AUDITORÍA LR B ===")

ejemplo_330 = tabla_grupos_resumen_330.loc[
    tabla_grupos_resumen_330[
        "Modelo"
    ].eq("LR B")
    &
    tabla_grupos_resumen_330[
        "Semana"
    ].isin([2, 8])
    &
    tabla_grupos_resumen_330[
        "Dimension"
    ].isin([
        "gender",
        "disability",
        "imd_band"
    ]),
    [
        "Semana",
        "Dimension",
        "Grupo",
        "Matriculas",
        "Prevalencia_pct",
        "Recall_adverso_pct",
        "FPR_pct",
        "Precision_pct",
        "Brier",
        "Sesgo_probabilidad_pp",
        "Soporte_suficiente"
    ]
].sort_values(
    ["Semana", "Dimension", "Grupo"]
)

display(ejemplo_330.round(3))

# ------------------------------------------------------------
# 8. BRECHAS ENTRE GRUPOS
# ------------------------------------------------------------

print("\n=== 3.30.5 BRECHAS DESCRIPTIVAS ===")

filas_brechas_330 = []

for (semana, modelo, dimension), grupo in (
    tabla_grupos_resumen_330.groupby(
        ["Semana", "Modelo", "Dimension"]
    )
):

    validos = grupo.loc[
        grupo["Soporte_suficiente"]
    ]

    def brecha_330(columna):

        valores = validos[columna].dropna()

        if len(valores) < 2:
            return np.nan

        return valores.max() - valores.min()

    filas_brechas_330.append({
        "Semana": semana,
        "Modelo": modelo,
        "Dimension": dimension,
        "Grupos_totales": len(grupo),
        "Grupos_con_soporte": len(validos),
        "Brecha_recall_pp": brecha_330(
            "Recall_adverso_pct"
        ),
        "Brecha_FPR_pp": brecha_330(
            "FPR_pct"
        ),
        "Brecha_Brier": brecha_330(
            "Brier"
        ),
        "Brecha_prevalencia_pp": brecha_330(
            "Prevalencia_pct"
        )
    })

tabla_brechas_330 = pd.DataFrame(
    filas_brechas_330
)

tabla_brechas_prioritarias_330 = (
    tabla_brechas_330.loc[
        tabla_brechas_330["Semana"].isin([2, 8])
        &
        tabla_brechas_330["Modelo"].isin([
            "LR B",
            "RF B",
            "Transformer C"
        ]),
        [
            "Semana",
            "Modelo",
            "Dimension",
            "Grupos_con_soporte",
            "Brecha_recall_pp",
            "Brecha_FPR_pp",
            "Brecha_Brier",
            "Brecha_prevalencia_pp"
        ]
    ]
    .sort_values(
        ["Semana", "Dimension", "Modelo"]
    )
)

display(
    tabla_brechas_prioritarias_330.round(3)
)

# ------------------------------------------------------------
# 9. COMPARAR BRECHAS A VS B
# ------------------------------------------------------------

print("\n=== 3.30.6 SENSIBILIDAD A VS B ===")

comparaciones_330 = []

for algoritmo in ["LR", "RF"]:

    a = tabla_brechas_330.loc[
        tabla_brechas_330[
            "Modelo"
        ].eq(f"{algoritmo} A"),
        [
            "Semana",
            "Dimension",
            "Brecha_recall_pp",
            "Brecha_FPR_pp"
        ]
    ].rename(columns={
        "Brecha_recall_pp": "Recall_A",
        "Brecha_FPR_pp": "FPR_A"
    })

    b = tabla_brechas_330.loc[
        tabla_brechas_330[
            "Modelo"
        ].eq(f"{algoritmo} B"),
        [
            "Semana",
            "Dimension",
            "Brecha_recall_pp",
            "Brecha_FPR_pp"
        ]
    ].rename(columns={
        "Brecha_recall_pp": "Recall_B",
        "Brecha_FPR_pp": "FPR_B"
    })

    ab = a.merge(
        b,
        on=["Semana", "Dimension"],
        validate="one_to_one"
    )

    ab["Algoritmo"] = algoritmo

    ab["Cambio_brecha_recall_pp"] = (
        ab["Recall_B"] - ab["Recall_A"]
    )

    ab["Cambio_brecha_FPR_pp"] = (
        ab["FPR_B"] - ab["FPR_A"]
    )

    comparaciones_330.append(ab)

tabla_brechas_AB_330 = pd.concat(
    comparaciones_330,
    ignore_index=True
)

display(
    tabla_brechas_AB_330.loc[
        tabla_brechas_AB_330[
            "Semana"
        ].isin([2, 8])
    ].round(3)
)

print(
    "Una brecha menor no demuestra por sí sola "
    "que el modelo sea justo."
)

# ------------------------------------------------------------
# 10. GRÁFICOS DE RECALL POR SUBGRUPO
# ------------------------------------------------------------

print("\n=== 3.30.7 GRÁFICOS DE SUBGRUPOS ===")

for dimension in ["gender", "disability"]:

    fig, ax = plt.subplots(figsize=(10, 5))

    sub = tabla_grupos_resumen_330.loc[
        tabla_grupos_resumen_330[
            "Dimension"
        ].eq(dimension)
        &
        tabla_grupos_resumen_330[
            "Modelo"
        ].isin(["LR B", "Transformer C"])
        &
        tabla_grupos_resumen_330[
            "Soporte_suficiente"
        ]
    ]

    for (modelo, grupo), datos in sub.groupby(
        ["Modelo", "Grupo"]
    ):

        datos = datos.sort_values("Semana")

        ax.plot(
            datos["Semana"],
            datos["Recall_adverso_pct"],
            marker="o",
            linewidth=2,
            label=f"{modelo} | {grupo}"
        )

    ax.set_title(
        "Recall adverso por subgrupo\n"
        f"Dimensión: {dimension}"
    )

    ax.set_xlabel("Semana de predicción")
    ax.set_ylabel("Recall adverso (%)")

    ax.set_xticks(SEMANAS_330)
    ax.set_ylim(0, 100)
    ax.grid(alpha=0.3)
    ax.legend()

    plt.tight_layout()
    plt.show()

# ------------------------------------------------------------
# 11. GRÁFICO SOCIOECONÓMICO, SEMANA 8
# ------------------------------------------------------------

sub_imd_330 = tabla_grupos_resumen_330.loc[
    tabla_grupos_resumen_330[
        "Semana"
    ].eq(8)
    &
    tabla_grupos_resumen_330[
        "Dimension"
    ].eq("imd_band")
    &
    tabla_grupos_resumen_330[
        "Modelo"
    ].isin(["LR B", "Transformer C"])
    &
    tabla_grupos_resumen_330[
        "Soporte_suficiente"
    ]
].copy()

orden_imd_330 = [
    "0-10%",
    "10-20%",
    "20-30%",
    "30-40%",
    "40-50%",
    "50-60%",
    "60-70%",
    "70-80%",
    "80-90%",
    "90-100%",
    "No informado"
]

fig, ax = plt.subplots(figsize=(11, 5))

for modelo in ["LR B", "Transformer C"]:

    sub = sub_imd_330.loc[
        sub_imd_330["Modelo"].eq(modelo)
    ]

    disponibles = [
        x for x in orden_imd_330
        if x in sub["Grupo"].values
    ]

    sub = (
        sub.set_index("Grupo")
        .reindex(disponibles)
    )

    ax.plot(
        sub.index,
        sub["Recall_adverso_pct"],
        marker="o",
        linewidth=2,
        label=modelo
    )

ax.set_title(
    "Recall adverso por categoría IMD - Semana 8"
)

ax.set_xlabel("Categoría socioeconómica IMD")
ax.set_ylabel("Recall adverso (%)")
ax.tick_params(axis="x", rotation=40)
ax.set_ylim(0, 100)
ax.grid(alpha=0.3)
ax.legend()

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 12. CONTROLES FINALES
# ------------------------------------------------------------

print("\n=== 3.30.8 CONTROLES FINALES ===")

assert tabla_grupos_rep_330[
    "Recall_adverso_pct"
].dropna().between(0, 100).all()

assert tabla_grupos_rep_330[
    "FPR_pct"
].dropna().between(0, 100).all()

assert tabla_grupos_rep_330[
    "Brier"
].between(0, 1).all()

# Para cada modelo y semana, todos los grupos
# de cada dimensión deben reconstruir el total.
for dimension in DIMENSIONES_330:

    sub = tabla_grupos_rep_330.loc[
        tabla_grupos_rep_330[
            "Dimension"
        ].eq(dimension)
    ]

    totales = sub.groupby(
        ["Semana", "Modelo", "Repeticion"]
    )["Matriculas"].sum()

    originales = datos_330.groupby(
        ["Semana", "Modelo", "Repeticion"]
    ).size()

    assert totales.equals(originales)

assert len(tabla_brechas_330) == (
    4 * 5 * len(DIMENSIONES_330)
)

assert not tabla_grupos_resumen_330[
    "Repeticiones"
].ne(2).any()

print("Modelos auditados: 5")
print("Semanas auditadas: 4")
print("Dimensiones de agrupación: 7")
print("Métricas calculadas por repetición")
print("Grupos pequeños conservados y señalados")
print("Brechas descriptivas verificadas")
print("Modelos reentrenados: 0")
print("Probabilidades modificadas: 0")
print("Umbrales modificados: 0")
print("Conjunto de prueba independiente: NO utilizado")

print("\n=== ACTIVIDAD 3.30 FINALIZADA ===")


In [ ]:

# ============================================================
# ACTIVIDAD 3.30 - CORRECCIÓN DEL GRÁFICO IMD
# Solo visualización. No modifica modelos ni métricas.
# ============================================================

import pandas as pd
import matplotlib.pyplot as plt

orden_imd_corregido = [
    "0-10%",
    "10-20%",
    "20-30%",
    "30-40%",
    "40-50%",
    "50-60%",
    "60-70%",
    "70-80%",
    "80-90%",
    "90-100%",
    "No informado"
]

datos_imd = tabla_grupos_resumen_330.loc[
    (tabla_grupos_resumen_330["Semana"] == 8)
    & (tabla_grupos_resumen_330["Dimension"] == "imd_band")
    & (tabla_grupos_resumen_330["Modelo"].isin(
        ["LR B", "Transformer C"]
    ))
    & tabla_grupos_resumen_330["Soporte_suficiente"]
].copy()

# Normalización exclusivamente de la etiqueta visible.
datos_imd["Grupo_grafico"] = (
    datos_imd["Grupo"]
    .replace({"10-20": "10-20%"})
)

for modelo in ["LR B", "Transformer C"]:
    sub = datos_imd.loc[
        datos_imd["Modelo"].eq(modelo)
    ]

    faltantes = (
        set(orden_imd_corregido)
        - set(sub["Grupo_grafico"])
    )

    assert not faltantes, (
        f"Categorías ausentes para {modelo}: {faltantes}"
    )

    assert not sub["Grupo_grafico"].duplicated().any()

fig, ax = plt.subplots(figsize=(12, 5))

for modelo in ["LR B", "Transformer C"]:
    sub = (
        datos_imd.loc[datos_imd["Modelo"].eq(modelo)]
        .set_index("Grupo_grafico")
        .reindex(orden_imd_corregido)
    )

    ax.plot(
        sub.index,
        sub["Recall_adverso_pct"],
        marker="o",
        linewidth=2,
        label=modelo
    )

ax.set_title(
    "Recall adverso por categoría IMD - Semana 8\n"
    "Todas las categorías incluidas"
)
ax.set_xlabel("Categoría IMD")
ax.set_ylabel("Recall adverso (%)")
ax.set_ylim(0, 100)
ax.grid(alpha=0.3)
ax.tick_params(axis="x", rotation=40)
ax.legend()

plt.tight_layout()
plt.show()

print("Categorías representadas:", len(orden_imd_corregido))
print("Categoría 10-20 incluida correctamente.")
print("Modelos reentrenados: 0")
print("Datos originales modificados: NO")



### 3.30.9. Resultados e interpretación

Se completó una auditoría exploratoria del rendimiento por subgrupos estudiantiles utilizando predicciones fuera de muestra de cinco modelos y cuatro ventanas temporales.

Se examinaron siete dimensiones: género, edad, indicador socioeconómico IMD, discapacidad declarada, educación previa, región y módulo académico.

Se calcularon recall adverso, tasa de falsos positivos, precisión, prevalencia adversa, Brier Score y sesgo medio de probabilidad. Las brechas se calcularon exclusivamente entre grupos que superaron los criterios mínimos de soporte establecidos.

#### Diferencias por género

En la semana 8, Regresión Logística B obtuvo un recall adverso del 55,12 % en el grupo F y del 62,14 % en el grupo M, con una brecha de 7,02 puntos porcentuales.

Las tasas de falsos positivos fueron del 14,36 % y 16,21 %, respectivamente.

En la semana 2, la brecha de recall había sido de 9,86 puntos porcentuales.

#### Diferencias por discapacidad declarada

En la semana 8, Regresión Logística B presentó un recall adverso del 57,64 % para el grupo N y del 70,41 % para el grupo Y.

La brecha de recall fue de 12,77 puntos porcentuales.

Las tasas de falsos positivos fueron del 14,46 % y 25,86 %, respectivamente, con una diferencia de 11,40 puntos.

Por tanto, el grupo Y presentó tanto una mayor proporción de resultados adversos correctamente identificados como una mayor proporción de alertas incorrectas entre resultados favorables.

#### Diferencias socioeconómicas

El gráfico corregido permite observar el rendimiento de Regresión Logística B y Transformer C en las diez bandas del índice de privación socioeconómica (IMD), además de las matrículas sin información disponible.

En la semana 8, Regresión Logística B alcanzó un recall adverso de 71,90 % en la banda 0–10 %, frente a 46,56 % en la banda 90–100 %, lo que representa una diferencia de 25,35 puntos porcentuales.

Ambos modelos mostraron una tendencia general de menor sensibilidad en las categorías con menor privación socioeconómica, aunque el comportamiento no fue estrictamente monotónico.

Estas diferencias deben interpretarse considerando las variaciones de prevalencia adversa, composición académica y tasas de falsos positivos entre grupos.

Por tanto, los resultados constituyen evidencia descriptiva de heterogeneidad predictiva, pero no permiten establecer por sí solos la existencia de discriminación algorítmica.

#### Efecto de la incorporación longitudinal sobre las brechas

En la semana 8, Regresión Logística B presentó brechas de recall menores que Regresión Logística A en las dimensiones de género, discapacidad, IMD, educación previa, región y módulo académico.

La brecha de recall socioeconómica disminuyó de 63,01 a 32,79 puntos porcentuales.

La brecha por discapacidad disminuyó de 27,01 a 12,77 puntos porcentuales.

No obstante, la reducción de una brecha descriptiva no constituye evidencia suficiente de equidad algorítmica.

También se observaron diferencias relevantes por educación previa y módulo académico, que requieren una investigación adicional de la composición y el rendimiento de cada grupo.

### 3.30.10. Limitaciones

Las diferencias observadas son descriptivas y no constituyen pruebas causales de discriminación.

La prevalencia adversa, el contexto académico y otras características pueden variar entre grupos.

Los indicadores se calcularon por matrícula, por lo que un estudiante puede contribuir con más de una observación.

Las dos repeticiones de validación cruzada no representan muestras independientes.

Los criterios mínimos de soporte no sustituyen intervalos de confianza ni análisis estadísticos específicos por grupo.

Algunas dimensiones cambian en el número de categorías con soporte suficiente entre semanas, por lo que sus brechas no siempre son directamente comparables a lo largo del tiempo.

Las diferencias máximas se calcularon entre categorías agregadas y no consideran todavía combinaciones de atributos ni ajustes por composición académica.

Las variables sensibles incluidas originalmente en los modelos no fueron eliminadas ni modificadas durante esta auditoría.

El conjunto de prueba independiente permaneció reservado.

### 3.30.11. Conclusión

La auditoría identificó diferencias relevantes de recall y tasas de falsos positivos entre diversos subgrupos estudiantiles.

La incorporación de características longitudinales estuvo acompañada por reducciones descriptivas de varias brechas observadas en los modelos estáticos.

Sin embargo, persistieron diferencias considerables por educación previa, contexto académico, indicador socioeconómico y discapacidad declarada.

**Conclusión principal:** la mejora del rendimiento predictivo global no garantiza un desempeño uniforme entre subgrupos. Antes de considerar una implementación institucional, será necesario examinar las causas posibles de estas diferencias, su incertidumbre estadística y las consecuencias de las alertas para cada población.

La evidencia disponible respalda una auditoría de equidad más profunda, pero no permite afirmar discriminación causal ni certificar equidad algorítmica.


#Actividad 3.31 — Preparación de Optuna para optimizar los modelos longitudinales
Incorporaremos Optuna como una extensión metodológica, antes de abrir el conjunto de prueba independiente.
El objetivo será investigar si una optimización adaptativa consigue mejorar los modelos B que originalmente ajustamos con GridSearchCV.
Comenzaremos con Regresión Logística B y Random Forest B en la semana 8. Conservaremos los resultados anteriores y las particiones agrupadas por estudiante.
Para construir el código, vamos primero a comprobar cómo quedaron guardadas las matrices y los pipelines, de tal manera de evitar reconstruir incorrectamente los datos o introducir una fuga de información.

In [ ]:

# ============================================================
# ACTIVIDAD 3.31
# PREPARACIÓN DE OPTUNA Y AUDITORÍA DE OBJETOS
# ============================================================

# Instalación en Google Colab
%pip -q install optuna

import optuna
import sklearn
import pandas as pd
import numpy as np

print("=== 3.31.1 ENTORNO ===")
print("Optuna:", optuna.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Semana piloto: 8")
print("Experimentos: LR B y RF B")

# ------------------------------------------------------------
# 1. OBJETOS NECESARIOS
# ------------------------------------------------------------

requeridos = [
    "experimentos_oulad",
    "cv_splits_oulad"
]

for nombre in requeridos:
    assert nombre in globals(), (
        f"No está disponible: {nombre}"
    )

print("\n=== 3.31.2 MATRICES DE LA SEMANA 8 ===")

exp = experimentos_oulad[8]

print("Tipo de objeto:", type(exp).__name__)

if isinstance(exp, dict):

    for clave, valor in exp.items():

        # No examinar valores del test reservado.
        if any(
            palabra in str(clave).lower()
            for palabra in ["test", "prueba"]
        ):
            print(
                f"{clave}: RESERVADO - no inspeccionado"
            )
            continue

        forma = getattr(valor, "shape", None)

        print(
            f"{clave}: "
            f"tipo={type(valor).__name__}; "
            f"dimensiones={forma}"
        )

else:
    print(
        "El objeto no es un diccionario; "
        "revisaremos su estructura antes de usarlo."
    )

# ------------------------------------------------------------
# 2. PARTICIONES AGRUPADAS
# ------------------------------------------------------------

print("\n=== 3.31.3 PARTICIONES ORIGINALES ===")

splits = cv_splits_oulad[8]

print("Tipo:", type(splits).__name__)
print("Número de entradas:", len(splits))

if isinstance(splits, (list, tuple)) and len(splits):

    primera = splits[0]

    print(
        "Tipo de primera entrada:",
        type(primera).__name__
    )

    if isinstance(primera, dict):

        print(
            "Campos disponibles:",
            list(primera.keys())
        )

    elif isinstance(primera, (list, tuple)):

        print(
            "Número de componentes:",
            len(primera)
        )

        for i, componente in enumerate(primera):

            print(
                f"Componente {i}: "
                f"tipo={type(componente).__name__}, "
                f"longitud={len(componente) if hasattr(componente, '__len__') else 'N/A'}"
            )

elif isinstance(splits, dict):

    print(
        "Claves iniciales:",
        list(splits.keys())[:10]
    )

# ------------------------------------------------------------
# 3. LOCALIZAR PIPELINES PREEXISTENTES
# ------------------------------------------------------------

print("\n=== 3.31.4 OBJETOS DE PIPELINE ===")

from sklearn.pipeline import Pipeline

encontrados = []

for nombre, valor in list(globals().items()):

    if nombre.startswith("_"):
        continue

    if isinstance(valor, Pipeline):
        encontrados.append(
            (nombre, "Pipeline")
        )

    elif (
        isinstance(valor, dict)
        and "pipeline" in nombre.lower()
    ):
        encontrados.append(
            (nombre, "Diccionario de pipelines")
        )

for nombre, tipo in encontrados[:20]:
    print(f"{nombre}: {tipo}")

if not encontrados:
    print(
        "No se encontraron pipelines globales "
        "por nombre. Se podrán reconstruir "
        "desde las matrices de entrenamiento."
    )

# ------------------------------------------------------------
# 4. ESTADO DE SEGURIDAD
# ------------------------------------------------------------

print("\n=== 3.31.5 CONTROLES ===")

print("Optuna instalado o actualizado.")
print("No se han realizado búsquedas.")
print("No se han ajustado modelos.")
print("No se han accedido valores de prueba.")
print("Resultados de GridSearchCV conservados.")

print(
    "\n=== PREPARACIÓN DE ACTIVIDAD 3.31 COMPLETADA ==="
)



### 3.32. Optimización adaptativa de hiperparámetros con Optuna

#### Objetivo

Evaluar el potencial de Optuna para mejorar los modelos longitudinales de Regresión Logística y Random Forest, originalmente optimizados mediante GridSearchCV.

Se utilizará la semana 8 como experimento piloto, considerando que dispone de la mayor ventana de información temporal analizada.

#### Metodología

Se empleará el algoritmo TPE (Tree-structured Parzen Estimator) para buscar configuraciones que maximicen el F1 macro.

Los espacios de búsqueda contemplarán:

**Regresión Logística B**
- Intensidad de regularización `C`.
- Regularización L1 o L2.
- Ponderación de clases.
- Porcentaje de características seleccionadas.

**Random Forest B**
- Número de árboles.
- Profundidad máxima.
- Cantidad mínima de muestras por hoja.
- Cantidad mínima de muestras por división.
- Número de características consideradas por división.
- Ponderación de clases.
- Porcentaje de características seleccionadas.

#### Protocolo experimental

Se utilizará el primer fold externo de la semana 8, manteniendo intacta la partición agrupada por estudiante.

Dentro del entrenamiento externo se realizarán tres folds internos mediante `StratifiedGroupKFold`.

Cada intento de Optuna ajustará el pipeline completo, incluyendo imputación, escalamiento, codificación y selección de características.

La métrica objetivo será F1 macro.

Se incluirán configuraciones originales de GridSearchCV como puntos de referencia iniciales para la búsqueda.

#### Comparación

Se contrastarán los resultados obtenidos con Optuna frente a los resultados originales de GridSearchCV en el mismo fold externo.

Se reportarán:
- Mejor F1 macro de validación interna.
- Hiperparámetros seleccionados.
- F1 macro externo.
- Recall de resultados adversos.
- ROC-AUC.
- Brecha de rendimiento entre entrenamiento y validación interna.
- Tiempo de ejecución.

#### Precaución metodológica

El piloto tiene carácter exploratorio. Los resultados no permiten afirmar superioridad estadística de Optuna.

El espacio de búsqueda y el presupuesto deberán quedar congelados antes de iniciar la comparación anidada completa.

Las mejoras observadas pueden deberse al espacio de hiperparámetros, al mayor presupuesto de evaluación o a la estrategia de búsqueda; este piloto no permite aislar esos efectos.

No se accederá al conjunto de prueba independiente.

**Referencias metodológicas:** el espacio de búsqueda adaptativa se estudia con Optuna [7]. Como antecedente de la discusión sobre estrategias de búsqueda de hiperparámetros se considera [11].


In [ ]:

# ============================================================
# ACTIVIDAD 3.32
# OPTUNA - PILOTO SEMANA 8
# REGRESIÓN LOGÍSTICA B Y RANDOM FOREST B
# ============================================================

# Asistencia IA: ChatGPT, octubre de 2026.
# Prompt:
# "Implementar optimización Optuna TPE para LR B y RF B,
# preservando pipelines existentes, validación interna
# agrupada por estudiante y fold externo independiente.
# Comparar con GridSearchCV sin utilizar el test final."

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import optuna

from sklearn.base import clone
from sklearn.model_selection import (
    StratifiedGroupKFold,
    cross_validate
)
from sklearn.metrics import (
    f1_score,
    recall_score,
    roc_auc_score,
    accuracy_score
)
from optuna.trial import TrialState
from IPython.display import display

# ------------------------------------------------------------
# 1. CONFIGURACIÓN
# ------------------------------------------------------------

SEMANA_332 = 8
FOLD_EXTERNO_332 = 0

N_INTERNOS_332 = 3
SEMILLA_332 = 42
SEMILLA_CV_332 = 137

PRESUPUESTO_332 = {
    "LogisticRegression": 24,
    "RandomForest": 20
}

ALGORITMOS_332 = [
    "LogisticRegression",
    "RandomForest"
]

print("=== 3.32.1 CONFIGURACIÓN ===")

print("Optuna:", optuna.__version__)
print("Semana:", SEMANA_332)
print("Fold externo:", FOLD_EXTERNO_332 + 1)
print("Folds internos:", N_INTERNOS_332)
print("Presupuesto:", PRESUPUESTO_332)
print("Métrica objetivo: F1 macro")

# ------------------------------------------------------------
# 2. RECUPERAR MATRICES EXISTENTES
# ------------------------------------------------------------

datos_332 = experimentos_oulad[SEMANA_332]

X_332 = (
    datos_332["X_B_train"]
    .reset_index(drop=True)
)

XA_332 = (
    datos_332["X_A_train"]
    .reset_index(drop=True)
)

y_332 = (
    datos_332["y_train"]
    .reset_index(drop=True)
)

grupos_332 = (
    datos_332["grupos_train"]
    .reset_index(drop=True)
)

idx_tr_332, idx_val_332 = (
    cv_splits_oulad[SEMANA_332][FOLD_EXTERNO_332]
)

X_tr_332 = (
    X_332.iloc[idx_tr_332]
    .reset_index(drop=True)
)

X_val_332 = (
    X_332.iloc[idx_val_332]
    .reset_index(drop=True)
)

XA_tr_332 = (
    XA_332.iloc[idx_tr_332]
    .reset_index(drop=True)
)

y_tr_332 = (
    y_332.iloc[idx_tr_332]
    .reset_index(drop=True)
)

y_val_332 = (
    y_332.iloc[idx_val_332]
    .reset_index(drop=True)
)

g_tr_332 = (
    grupos_332.iloc[idx_tr_332]
    .reset_index(drop=True)
)

g_val_332 = (
    grupos_332.iloc[idx_val_332]
    .reset_index(drop=True)
)

assert set(g_tr_332).isdisjoint(set(g_val_332))
assert len(X_tr_332) == len(y_tr_332)
assert len(X_val_332) == len(y_val_332)

print("\n=== 3.32.2 PARTICIÓN EXTERNA ===")

print("Entrenamiento externo:", len(X_tr_332))
print("Validación externa:", len(X_val_332))
print("Variables B:", X_tr_332.shape[1])
print("Estudiantes compartidos:", 0)

# ------------------------------------------------------------
# 3. PARTICIONES INTERNAS
# ------------------------------------------------------------

splitter_332 = StratifiedGroupKFold(
    n_splits=N_INTERNOS_332,
    shuffle=True,
    random_state=SEMILLA_CV_332
)

splits_internos_332 = list(
    splitter_332.split(
        XA_tr_332,
        y_tr_332,
        groups=g_tr_332
    )
)

for idx_it, idx_iv in splits_internos_332:

    assert set(
        g_tr_332.iloc[idx_it]
    ).isdisjoint(
        set(g_tr_332.iloc[idx_iv])
    )

    assert set(y_tr_332.iloc[idx_it].unique()) == {0, 1}
    assert set(y_tr_332.iloc[idx_iv].unique()) == {0, 1}

print("\n=== 3.32.3 VALIDACIÓN INTERNA ===")

tabla_splits_332 = pd.DataFrame([
    {
        "Fold_interno": i,
        "Matriculas_train": len(it),
        "Matriculas_validacion": len(iv),
        "Adversos_validacion_pct": (
            100 * y_tr_332.iloc[iv].mean()
        ),
        "Estudiantes_compartidos": 0
    }
    for i, (it, iv) in enumerate(
        splits_internos_332, start=1
    )
])

display(tabla_splits_332.round(2))

# ------------------------------------------------------------
# 4. DEFINIR ESPACIOS DE BÚSQUEDA
# ------------------------------------------------------------

def parametros_optuna_332(trial, algoritmo):

    porcentaje = trial.suggest_categorical(
        "seleccion__percentile",
        [60, 80, 100]
    )

    if algoritmo == "LogisticRegression":

        params = {
            "seleccion__percentile": porcentaje,
            "modelo__C": trial.suggest_float(
                "modelo__C",
                0.001,
                100.0,
                log=True
            ),
            "modelo__penalty": trial.suggest_categorical(
                "modelo__penalty",
                ["l1", "l2"]
            ),
            "modelo__class_weight": (
                trial.suggest_categorical(
                    "modelo__class_weight",
                    [None, "balanced"]
                )
            )
        }

    elif algoritmo == "RandomForest":

        params = {
            "seleccion__percentile": porcentaje,
            "modelo__n_estimators": (
                trial.suggest_categorical(
                    "modelo__n_estimators",
                    [100, 150, 250]
                )
            ),
            "modelo__max_depth": (
                trial.suggest_categorical(
                    "modelo__max_depth",
                    [6, 10, 14, 20, None]
                )
            ),
            "modelo__min_samples_leaf": (
                trial.suggest_categorical(
                    "modelo__min_samples_leaf",
                    [2, 4, 6, 10, 16]
                )
            ),
            "modelo__min_samples_split": (
                trial.suggest_categorical(
                    "modelo__min_samples_split",
                    [2, 5, 10, 20]
                )
            ),
            "modelo__max_features": (
                trial.suggest_categorical(
                    "modelo__max_features",
                    ["sqrt", "log2", 0.5, 0.8]
                )
            ),
            "modelo__class_weight": (
                trial.suggest_categorical(
                    "modelo__class_weight",
                    [None, "balanced_subsample"]
                )
            )
        }

    else:
        raise ValueError(
            f"Algoritmo desconocido: {algoritmo}"
        )

    return params

# ------------------------------------------------------------
# 5. CONFIGURACIONES ORIGINALES DE REFERENCIA
# ------------------------------------------------------------

ANCLAS_332 = {
    "LogisticRegression": [
        {
            "seleccion__percentile": 100,
            "modelo__C": 1.0,
            "modelo__penalty": "l2",
            "modelo__class_weight": None
        },
        {
            "seleccion__percentile": 80,
            "modelo__C": 0.1,
            "modelo__penalty": "l2",
            "modelo__class_weight": None
        }
    ],
    "RandomForest": [
        {
            "seleccion__percentile": 80,
            "modelo__n_estimators": 100,
            "modelo__max_depth": None,
            "modelo__min_samples_leaf": 5,
            "modelo__min_samples_split": 2,
            "modelo__max_features": "sqrt",
            "modelo__class_weight": None
        },
        {
            "seleccion__percentile": 80,
            "modelo__n_estimators": 100,
            "modelo__max_depth": 14,
            "modelo__min_samples_leaf": 4,
            "modelo__min_samples_split": 2,
            "modelo__max_features": "sqrt",
            "modelo__class_weight": None
        }
    ]
}

# Los valores deben estar incluidos en la búsqueda.
# RF original leaf=5, de modo que lo agregamos
# explícitamente a la distribución RF.

# ------------------------------------------------------------
# 6. OBJETIVO DE OPTIMIZACIÓN
# ------------------------------------------------------------

def funcion_objetivo_332(trial, algoritmo):

    params = parametros_optuna_332(
        trial, algoritmo
    )

    pipeline = clone(
        pipelines_oulad[
            (SEMANA_332, "B", algoritmo)
        ]
    )

    pipeline.set_params(**params)

    cv_resultado = cross_validate(
        estimator=pipeline,
        X=X_tr_332,
        y=y_tr_332,
        cv=splits_internos_332,
        scoring="f1_macro",
        n_jobs=1,
        return_train_score=True,
        error_score="raise"
    )

    scores_val = cv_resultado["test_score"]
    scores_train = cv_resultado["train_score"]

    media_val = float(np.mean(scores_val))
    media_train = float(np.mean(scores_train))

    trial.set_user_attr(
        "F1_train_interno", media_train
    )
    trial.set_user_attr(
        "F1_val_interno", media_val
    )
    trial.set_user_attr(
        "F1_std_interno",
        float(np.std(scores_val, ddof=1))
    )
    trial.set_user_attr(
        "Brecha_train_val",
        media_train - media_val
    )

    return media_val

# ------------------------------------------------------------
# 7. OPTIMIZACIÓN
# ------------------------------------------------------------

print("\n=== 3.32.4 OPTIMIZACIÓN OPTUNA ===")

estudios_optuna_332 = {}
resultados_piloto_optuna_332 = []
parametros_optuna_finales_332 = {}

for algoritmo in ALGORITMOS_332:

    print("\n" + "=" * 60)
    print("OPTUNA:", algoritmo)
    print("=" * 60)

    inicio = time.perf_counter()

    nombre_estudio = (
        f"oulad_s8_r1f1_B_{algoritmo}_v1"
    )

    ruta_db = (
        f"/content/{nombre_estudio}.db"
    )

    estudio = optuna.create_study(
        study_name=nombre_estudio,
        direction="maximize",
        sampler=optuna.samplers.TPESampler(
            seed=SEMILLA_332
        ),
        storage=f"sqlite:///{ruta_db}",
        load_if_exists=True
    )

    # Corregir la lista de leaf para incluir el
    # hiperparámetro original del GridSearchCV.
    # La definición del espacio es fija por ensayo.
    # La lista se modifica en la función en la celda
    # siguiente solamente si aún no se lanzó ningún
    # ensayo; aquí se verifica el estado.
    #
    # IMPORTANTE: para RF se utilizará el espacio
    # efectivo definido en la función de búsqueda.

    if len(estudio.trials) == 0:

        # Incluir solamente anclas compatibles con
        # el espacio definido.
        for ancla in ANCLAS_332[algoritmo]:

            if (
                algoritmo == "RandomForest"
                and ancla["modelo__min_samples_leaf"]
                not in [2, 4, 5, 6, 10, 16]
            ):
                continue

            estudio.enqueue_trial(ancla)

    completados = sum(
        t.state == TrialState.COMPLETE
        for t in estudio.trials
    )

    pendientes = max(
        0,
        PRESUPUESTO_332[algoritmo] - completados
    )

    if pendientes:

        estudio.optimize(
            lambda trial: funcion_objetivo_332(
                trial, algoritmo
            ),
            n_trials=pendientes,
            n_jobs=1,
            show_progress_bar=True
        )

    estudios_optuna_332[algoritmo] = estudio

    mejor_trial = estudio.best_trial

    mejores_params = dict(
        mejor_trial.params
    )

    parametros_optuna_finales_332[
        algoritmo
    ] = mejores_params

    print(
        "\nMejor F1 macro CV interno:",
        round(estudio.best_value, 5)
    )

    print("Mejores hiperparámetros:")
    print(mejores_params)

    # --------------------------------------------
    # 8. REENTRENAMIENTO EXTERNO
    # --------------------------------------------

    modelo_opt = clone(
        pipelines_oulad[
            (SEMANA_332, "B", algoritmo)
        ]
    )

    modelo_opt.set_params(**mejores_params)

    # Ajuste con el entrenamiento externo completo.
    modelo_opt.fit(
        X_tr_332,
        y_tr_332
    )

    pred = modelo_opt.predict(X_val_332)

    indice_positivo = (
        list(modelo_opt.classes_).index(1)
    )

    prob = modelo_opt.predict_proba(
        X_val_332
    )[:, indice_positivo]

    f1_externo = f1_score(
        y_val_332, pred,
        average="macro"
    )

    recall_externo = recall_score(
        y_val_332, pred,
        zero_division=0
    )

    auc_externo = roc_auc_score(
        y_val_332, prob
    )

    accuracy_externa = accuracy_score(
        y_val_332, pred
    )

    # --------------------------------------------
    # 9. RECUPERAR GRIDSEARCH ORIGINAL
    # --------------------------------------------

    referencia = resultados_cv_ventanas[
        (SEMANA_332, algoritmo)
    ]["resultados"]

    referencia = referencia.loc[
        (referencia["Experimento"] == "B")
        & (referencia["Repeticion"] == 1)
        & (referencia["Fold"] == 1)
    ]

    assert len(referencia) == 1

    referencia = referencia.iloc[0]

    duracion = time.perf_counter() - inicio

    resultados_piloto_optuna_332.append({
        "Algoritmo": algoritmo,
        "Trials_completados": sum(
            t.state == TrialState.COMPLETE
            for t in estudio.trials
        ),
        "F1_CV_Optuna": estudio.best_value,
        "F1_CV_Grid": referencia["F1_CV_interno"],
        "F1_externo_Optuna": f1_externo,
        "F1_externo_Grid": referencia["F1_macro"],
        "Delta_F1_externo": (
            f1_externo - referencia["F1_macro"]
        ),
        "Recall_Optuna": recall_externo,
        "Recall_Grid": referencia["Recall_adverso"],
        "AUC_Optuna": auc_externo,
        "AUC_Grid": referencia["ROC_AUC"],
        "Accuracy_Optuna": accuracy_externa,
        "Brecha_interna_Optuna": (
            mejor_trial.user_attrs[
                "Brecha_train_val"
            ]
        ),
        "Tiempo_segundos": duracion
    })

    print(
        f"F1 externo Optuna={f1_externo:.4f} | "
        f"Grid={referencia['F1_macro']:.4f}"
    )

    print(
        f"Recall Optuna={recall_externo:.4f} | "
        f"AUC Optuna={auc_externo:.4f}"
    )

tabla_optuna_piloto_332 = pd.DataFrame(
    resultados_piloto_optuna_332
)

print("\n=== 3.32.5 COMPARACIÓN PRELIMINAR ===")

display(
    tabla_optuna_piloto_332.round(5)
)

# ------------------------------------------------------------
# 10. CURVAS DE OPTIMIZACIÓN
# ------------------------------------------------------------

print("\n=== 3.32.6 CURVAS DE OPTIMIZACIÓN ===")

for algoritmo, estudio in estudios_optuna_332.items():

    trials = [
        t for t in estudio.trials
        if t.state == TrialState.COMPLETE
    ]

    historial = pd.DataFrame({
        "Trial": [
            t.number + 1 for t in trials
        ],
        "F1_CV": [
            t.value for t in trials
        ]
    }).sort_values("Trial")

    historial["Mejor_acumulado"] = (
        historial["F1_CV"].cummax()
    )

    fig, ax = plt.subplots(figsize=(9, 4.5))

    ax.scatter(
        historial["Trial"],
        historial["F1_CV"],
        label="F1 de cada intento"
    )

    ax.plot(
        historial["Trial"],
        historial["Mejor_acumulado"],
        linewidth=2,
        label="Mejor F1 acumulado"
    )

    ax.set_title(
        f"Convergencia de Optuna - {algoritmo}"
    )
    ax.set_xlabel("Intento de optimización")
    ax.set_ylabel("F1 macro CV interna")
    ax.grid(alpha=0.3)
    ax.legend()

    plt.tight_layout()
    plt.show()

# ------------------------------------------------------------
# 11. IMPORTANCIA EXPLORATORIA DE HIPERPARÁMETROS
# ------------------------------------------------------------

print("\n=== 3.32.7 IMPORTANCIA DE HIPERPARÁMETROS ===")

for algoritmo, estudio in estudios_optuna_332.items():

    try:

        importancias = (
            optuna.importance.get_param_importances(
                estudio
            )
        )

        tabla_imp = pd.DataFrame({
            "Hiperparametro": list(importancias),
            "Importancia": list(
                importancias.values()
            )
        })

        print("\n", algoritmo)

        display(tabla_imp.round(4))

        fig, ax = plt.subplots(figsize=(9, 4.5))

        ordenada = tabla_imp.sort_values(
            "Importancia"
        )

        ax.barh(
            ordenada["Hiperparametro"],
            ordenada["Importancia"]
        )

        ax.set_title(
            f"Importancia exploratoria - {algoritmo}"
        )
        ax.set_xlabel(
            "Importancia relativa en la búsqueda"
        )
        ax.grid(axis="x", alpha=0.3)

        plt.tight_layout()
        plt.show()

    except Exception as exc:

        print(
            f"No se pudo estimar la importancia "
            f"de {algoritmo}: {exc}"
        )

# ------------------------------------------------------------
# 12. CONTROLES FINALES
# ------------------------------------------------------------

print("\n=== 3.32.8 CONTROLES FINALES ===")

assert len(tabla_optuna_piloto_332) == 2

assert all(
    len(estudio.best_trial.params) > 0
    for estudio in estudios_optuna_332.values()
)

assert set(g_tr_332).isdisjoint(
    set(g_val_332)
)

print("Algoritmos optimizados: 2")
print("Semana evaluada: 8")
print("Folds internos por intento: 3")
print("Fold externo evaluado: R1-F1")
print("Pipeline original conservado")
print("Resultados GridSearchCV originales conservados")
print("Resultados Optuna guardados")
print("Conjunto de prueba independiente: NO utilizado")
print("Comparación: PILOTO EXPLORATORIO")

print("\n=== ACTIVIDAD 3.32 FINALIZADA ===")



### 3.32.9. Resultados e interpretación

Se completó un experimento piloto de optimización adaptativa con Optuna en la semana 8, utilizando los modelos longitudinales de Regresión Logística y Random Forest.

Se empleó el algoritmo TPE para explorar hiperparámetros mediante validación cruzada interna de tres folds, preservando la agrupación por estudiante.

El piloto comprendió 24 intentos para Regresión Logística y 20 intentos para Random Forest.

#### Regresión Logística B

Optuna obtuvo un F1 macro interno de 0,72361 y seleccionó regularización L2, C=0,05872, ponderación de clases uniforme y selección del 100 % de características.

El F1 macro externo fue de 0,71889, frente a 0,72057 obtenido previamente con GridSearchCV.

La diferencia externa fue de -0,00169, indicando un rendimiento muy similar en el fold evaluado.

El ROC-AUC externo alcanzó 0,80048.

#### Random Forest B

Optuna obtuvo un F1 macro interno de 0,71720.

La configuración seleccionada incluyó 250 árboles, profundidad máxima de 20, un mínimo de 10 muestras por hoja, max_features=0,8, ponderación balanced_subsample y selección del 100 % de características.

El F1 macro externo aumentó de 0,68675 con GridSearchCV a 0,71614 con Optuna.

La mejora observada fue de 0,02938 puntos de F1 macro.

El recall adverso aumentó de 0,49524 a 0,61388, y el ROC-AUC pasó de 0,78024 a 0,79106.

La brecha media entre entrenamiento y validación interna fue de 0,16109, por lo que persisten indicios de sobreajuste.

#### Convergencia e importancia de hiperparámetros

La búsqueda alcanzó mejores configuraciones internas conforme avanzaron los intentos, aunque las curvas también mostraron configuraciones individuales con rendimientos inferiores.

El análisis exploratorio de importancia señaló a C como el parámetro predominante en Regresión Logística.

En Random Forest destacaron el número de árboles, el mínimo de muestras por hoja y la profundidad máxima.

Estas importancias describen la búsqueda realizada y no deben interpretarse como importancia predictiva de las variables académicas.

### Limitaciones

La evaluación externa corresponde únicamente al primer fold de la semana 8.

No existen todavía resultados suficientes para afirmar que Optuna produce una mejora estable o estadísticamente significativa.

La búsqueda de Optuna utilizó un espacio de hiperparámetros más amplio que GridSearchCV, por lo que las diferencias observadas no pueden atribuirse exclusivamente al algoritmo TPE.

El tiempo registrado no representa necesariamente el costo total de búsqueda, debido a la recuperación de estudios persistidos.

El conjunto de prueba independiente no fue utilizado.

### Conclusión

Optuna mostró potencial para mejorar Random Forest B, especialmente en el recall de resultados académicos adversos.

Para Regresión Logística B, la optimización adicional produjo un rendimiento externo prácticamente equivalente al obtenido mediante GridSearchCV.

Se justifica extender la comparación a los diez folds externos originales, manteniendo las particiones y el protocolo de selección interna para evaluar estabilidad y capacidad de generalización.



### 3.33. Evaluación anidada de Optuna en la semana 8

#### Objetivo

Determinar si las mejoras observadas durante el piloto de optimización con Optuna se mantienen al evaluar múltiples particiones externas agrupadas por estudiante.

#### Diseño experimental

Se utilizarán las diez particiones externas predefinidas para la semana 8, correspondientes a cinco folds y dos repeticiones.

Para cada combinación de algoritmo y fold externo:

1. Se reservará la validación externa original.
2. Se construirán tres folds internos agrupados por estudiante.
3. Se optimizarán los hiperparámetros mediante Optuna TPE.
4. Se seleccionará la configuración con mejor F1 macro interno.
5. Se ajustará el pipeline completo sobre el entrenamiento externo.
6. Se evaluará exclusivamente sobre la validación externa reservada.

Se conservarán las mismas variables longitudinales y la arquitectura de preprocesamiento de los experimentos originales.

#### Hiperparámetros y presupuesto

- Regresión Logística: 24 intentos por fold externo.
- Random Forest: 20 intentos por fold externo.
- Métrica objetivo: F1 macro.
- Folds internos: 3.
- Folds externos: 10.

La optimización será independiente para cada entrenamiento externo.

#### Comparación con GridSearchCV

Se reutilizarán las métricas originales de GridSearchCV en los mismos folds externos, sin reentrenar sus modelos.

Se compararán F1 macro, recall adverso, ROC-AUC y brechas internas de generalización.

El primer fold externo fue utilizado previamente como piloto exploratorio. Por transparencia, el análisis posterior deberá presentar también los resultados de los nueve folds restantes, además del resumen completo de diez folds.

#### Reproducibilidad

Los estudios de Optuna se guardarán en SQLite y las predicciones externas en un checkpoint.

Esto permitirá reanudar el cálculo sin repetir los folds completados.

#### Restricciones

No se modificarán los resultados previos, las variables de entrada ni la arquitectura de los pipelines.

No se utilizarán las etiquetas del conjunto de prueba independiente.

Los resultados seguirán perteneciendo a la fase de desarrollo y no constituirán todavía la evaluación final del modelo.


In [ ]:

# ============================================================
# ACTIVIDAD 3.33
# VALIDACIÓN ANIDADA COMPLETA - OPTUNA, SEMANA 8
# ============================================================

# Asistencia IA: ChatGPT, octubre de 2026.
# Prompt:
# "Extender la optimización Optuna a diez folds externos,
# con tres folds internos agrupados por estudiante,
# espacios congelados, checkpoints y comparación
# pareada contra GridSearchCV. No usar el test final."

import os
import pickle
import time

import numpy as np
import pandas as pd
import optuna

from sklearn.base import clone
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.model_selection import cross_validate
from sklearn.metrics import (
    f1_score,
    recall_score,
    precision_score,
    roc_auc_score,
    accuracy_score,
    confusion_matrix
)
from IPython.display import display
from optuna.trial import TrialState

# ------------------------------------------------------------
# 1. CONFIGURACIÓN CONGELADA
# ------------------------------------------------------------

SEMANA_333 = 8
N_FOLDS_EXTERNOS_333 = 10
N_FOLDS_INTERNOS_333 = 3

FOLDS_NUEVOS_POR_EJECUCION_333 = 2

PRESUPUESTO_333 = {
    "LogisticRegression": 24,
    "RandomForest": 20
}

ALGORITMOS_333 = [
    "LogisticRegression",
    "RandomForest"
]

RUTA_CHECKPOINT_333 = (
    "/content/optuna_oulad_semana8_333.pkl"
)

print("=== 3.33.1 CONFIGURACIÓN ===")
print("Semana:", SEMANA_333)
print("Folds externos:", N_FOLDS_EXTERNOS_333)
print("Folds internos:", N_FOLDS_INTERNOS_333)
print("Presupuestos:", PRESUPUESTO_333)
print(
    "Folds nuevos por ejecución:",
    FOLDS_NUEVOS_POR_EJECUCION_333
)

# ------------------------------------------------------------
# 2. DATOS ORIGINALES DE DESARROLLO
# ------------------------------------------------------------

origen_333 = experimentos_oulad[SEMANA_333]

X_333 = (
    origen_333["X_B_train"]
    .reset_index(drop=True)
)

y_333 = (
    origen_333["y_train"]
    .reset_index(drop=True)
)

grupos_333 = (
    origen_333["grupos_train"]
    .reset_index(drop=True)
)

splits_externos_333 = (
    cv_splits_oulad[SEMANA_333]
)

assert len(splits_externos_333) == 10

assert len(X_333) == len(y_333)
assert len(X_333) == len(grupos_333)

assert X_333.shape[1] == 48

assert set(y_333.unique()) == {0, 1}

# ------------------------------------------------------------
# 3. ESPACIO DE BÚSQUEDA
# ------------------------------------------------------------

# Mantener idéntico al piloto, incluida la
# categoría min_samples_leaf = 5.

def sugerir_parametros_333(trial, algoritmo):

    percentile = trial.suggest_categorical(
        "seleccion__percentile",
        [60, 80, 100]
    )

    if algoritmo == "LogisticRegression":

        return {
            "seleccion__percentile": percentile,
            "modelo__C": trial.suggest_float(
                "modelo__C",
                0.001,
                100.0,
                log=True
            ),
            "modelo__penalty": (
                trial.suggest_categorical(
                    "modelo__penalty",
                    ["l1", "l2"]
                )
            ),
            "modelo__class_weight": (
                trial.suggest_categorical(
                    "modelo__class_weight",
                    [None, "balanced"]
                )
            )
        }

    if algoritmo == "RandomForest":

        return {
            "seleccion__percentile": percentile,
            "modelo__n_estimators": (
                trial.suggest_categorical(
                    "modelo__n_estimators",
                    [100, 150, 250]
                )
            ),
            "modelo__max_depth": (
                trial.suggest_categorical(
                    "modelo__max_depth",
                    [6, 10, 14, 20, None]
                )
            ),
            "modelo__min_samples_leaf": (
                trial.suggest_categorical(
                    "modelo__min_samples_leaf",
                    [2, 4, 5, 6, 10, 16]
                )
            ),
            "modelo__min_samples_split": (
                trial.suggest_categorical(
                    "modelo__min_samples_split",
                    [2, 5, 10, 20]
                )
            ),
            "modelo__max_features": (
                trial.suggest_categorical(
                    "modelo__max_features",
                    ["sqrt", "log2", 0.5, 0.8]
                )
            ),
            "modelo__class_weight": (
                trial.suggest_categorical(
                    "modelo__class_weight",
                    [None, "balanced_subsample"]
                )
            )
        }

    raise ValueError(
        f"Algoritmo inesperado: {algoritmo}"
    )

# ------------------------------------------------------------
# 4. ANCLAS INICIALES
# ------------------------------------------------------------

ANCLAS_333 = {
    "LogisticRegression": [
        {
            "seleccion__percentile": 100,
            "modelo__C": 1.0,
            "modelo__penalty": "l2",
            "modelo__class_weight": None
        },
        {
            "seleccion__percentile": 80,
            "modelo__C": 0.1,
            "modelo__penalty": "l2",
            "modelo__class_weight": None
        }
    ],
    "RandomForest": [
        {
            "seleccion__percentile": 80,
            "modelo__n_estimators": 100,
            "modelo__max_depth": None,
            "modelo__min_samples_leaf": 5,
            "modelo__min_samples_split": 2,
            "modelo__max_features": "sqrt",
            "modelo__class_weight": None
        },
        {
            "seleccion__percentile": 80,
            "modelo__n_estimators": 100,
            "modelo__max_depth": 14,
            "modelo__min_samples_leaf": 4,
            "modelo__min_samples_split": 2,
            "modelo__max_features": "sqrt",
            "modelo__class_weight": None
        }
    ]
}

# ------------------------------------------------------------
# 5. RECUPERAR CHECKPOINT
# ------------------------------------------------------------

if os.path.exists(RUTA_CHECKPOINT_333):

    with open(RUTA_CHECKPOINT_333, "rb") as f:
        checkpoint_333 = pickle.load(f)

    print("\nCheckpoint recuperado.")

else:

    checkpoint_333 = {
        "resultados": {},
        "predicciones": {}
    }

    print("\nCheckpoint nuevo.")

assert set(checkpoint_333) == {
    "resultados",
    "predicciones"
}

# ------------------------------------------------------------
# 6. FUNCIÓN DE GUARDADO ATÓMICO
# ------------------------------------------------------------

def guardar_checkpoint_333():

    temporal = RUTA_CHECKPOINT_333 + ".tmp"

    with open(temporal, "wb") as f:
        pickle.dump(
            checkpoint_333,
            f,
            protocol=pickle.HIGHEST_PROTOCOL
        )

    os.replace(
        temporal,
        RUTA_CHECKPOINT_333
    )

    respaldar_checkpoint_oulad(RUTA_CHECKPOINT_333)

# ------------------------------------------------------------
# 7. FUNCIÓN PARA EVALUAR UN ALGORITMO
# ------------------------------------------------------------

def evaluar_optuna_333(
    indice_externo,
    algoritmo
):

    repeticion = indice_externo // 5 + 1
    fold = indice_externo % 5 + 1

    idx_tr, idx_val = splits_externos_333[
        indice_externo
    ]

    X_tr = (
        X_333.iloc[idx_tr]
        .reset_index(drop=True)
    )

    X_val = (
        X_333.iloc[idx_val]
        .reset_index(drop=True)
    )

    y_tr = (
        y_333.iloc[idx_tr]
        .reset_index(drop=True)
    )

    y_val = (
        y_333.iloc[idx_val]
        .reset_index(drop=True)
    )

    g_tr = (
        grupos_333.iloc[idx_tr]
        .reset_index(drop=True)
    )

    g_val = (
        grupos_333.iloc[idx_val]
        .reset_index(drop=True)
    )

    assert set(g_tr).isdisjoint(set(g_val))

    # Construcción de folds internos.
    splitter = StratifiedGroupKFold(
        n_splits=N_FOLDS_INTERNOS_333,
        shuffle=True,
        random_state=137
    )

    inner_splits = list(
        splitter.split(
            X_tr,
            y_tr,
            groups=g_tr
        )
    )

    for it, iv in inner_splits:

        assert set(
            g_tr.iloc[it]
        ).isdisjoint(
            set(g_tr.iloc[iv])
        )

    # Nombre compatible con el estudio del piloto.
    if repeticion == 1 and fold == 1:

        nombre = (
            f"oulad_s8_r1f1_B_{algoritmo}_v1"
        )

    else:

        nombre = (
            f"oulad_s8_r{repeticion}f{fold}"
            f"_B_{algoritmo}_v1"
        )

    base_sqlite = f"/content/{nombre}.db"

    estudio = optuna.create_study(
        study_name=nombre,
        direction="maximize",
        sampler=optuna.samplers.TPESampler(
            seed=42
        ),
        storage=f"sqlite:///{base_sqlite}",
        load_if_exists=True
    )

    # Las anclas se agregan solo a estudios nuevos.
    if len(estudio.trials) == 0:

        for parametros in ANCLAS_333[
            algoritmo
        ]:

            estudio.enqueue_trial(
                parametros
            )

    def objetivo(trial):

        parametros = sugerir_parametros_333(
            trial, algoritmo
        )

        pipe = clone(
            pipelines_oulad[
                (SEMANA_333, "B", algoritmo)
            ]
        )

        pipe.set_params(**parametros)

        scores = cross_validate(
            pipe,
            X_tr,
            y_tr,
            cv=inner_splits,
            scoring="f1_macro",
            n_jobs=1,
            return_train_score=True,
            error_score="raise"
        )

        f1_val = float(
            np.mean(scores["test_score"])
        )

        f1_train = float(
            np.mean(scores["train_score"])
        )

        trial.set_user_attr(
            "F1_train_interno",
            f1_train
        )

        trial.set_user_attr(
            "F1_val_interno",
            f1_val
        )

        trial.set_user_attr(
            "Brecha_train_val",
            f1_train - f1_val
        )

        return f1_val

    completados = sum(
        trial.state == TrialState.COMPLETE
        for trial in estudio.trials
    )

    pendientes = max(
        0,
        PRESUPUESTO_333[algoritmo] - completados
    )

    inicio = time.perf_counter()

    print(
        f"\nSemana 8 | {algoritmo} | "
        f"R{repeticion}-F{fold}"
    )

    print(
        f"Trials existentes: {completados}; "
        f"pendientes: {pendientes}"
    )

    if pendientes > 0:

        estudio.optimize(
            objetivo,
            n_trials=pendientes,
            n_jobs=1,
            show_progress_bar=False
        )

    mejores_params = dict(
        estudio.best_trial.params
    )

    # Ajuste final SOLO en entrenamiento externo.
    pipe_final = clone(
        pipelines_oulad[
            (SEMANA_333, "B", algoritmo)
        ]
    )

    pipe_final.set_params(
        **mejores_params
    )

    pipe_final.fit(
        X_tr,
        y_tr
    )

    y_pred = pipe_final.predict(
        X_val
    )

    pos = list(
        pipe_final.classes_
    ).index(1)

    y_prob = (
        pipe_final.predict_proba(X_val)[:, pos]
    )

    vn, fp, fn, vp = (
        confusion_matrix(
            y_val,
            y_pred,
            labels=[0, 1]
        ).ravel()
    )

    duracion = time.perf_counter() - inicio

    # Recuperar evaluación GridSearchCV del mismo fold.
    tabla_grid = resultados_cv_ventanas[
        (SEMANA_333, algoritmo)
    ]["resultados"]

    ref = tabla_grid.loc[
        (tabla_grid["Experimento"] == "B")
        &
        (tabla_grid["Repeticion"] == repeticion)
        &
        (tabla_grid["Fold"] == fold)
    ]

    assert len(ref) == 1

    ref = ref.iloc[0]

    f1_opt = f1_score(
        y_val,
        y_pred,
        average="macro"
    )

    recall_opt = recall_score(
        y_val,
        y_pred,
        zero_division=0
    )

    auc_opt = roc_auc_score(
        y_val,
        y_prob
    )

    resultado = {
        "Semana": 8,
        "Repeticion": repeticion,
        "Fold": fold,
        "Algoritmo": algoritmo,
        "N_validacion": len(y_val),
        "F1_Optuna": f1_opt,
        "F1_Grid": float(ref["F1_macro"]),
        "Delta_F1": (
            f1_opt - float(ref["F1_macro"])
        ),
        "Recall_Optuna": recall_opt,
        "Recall_Grid": float(
            ref["Recall_adverso"]
        ),
        "AUC_Optuna": auc_opt,
        "AUC_Grid": float(
            ref["ROC_AUC"]
        ),
        "Accuracy_Optuna": accuracy_score(
            y_val, y_pred
        ),
        "Precision_Optuna": precision_score(
            y_val, y_pred,
            zero_division=0
        ),
        "VN": int(vn),
        "FP": int(fp),
        "FN": int(fn),
        "VP": int(vp),
        "Mejor_F1_interno": (
            float(estudio.best_value)
        ),
        "Brecha_interna": (
            estudio.best_trial.user_attrs.get(
                "Brecha_train_val",
                np.nan
            )
        ),
        "Epoca_de_busqueda": "TPE",
        "Trials_completos": sum(
            t.state == TrialState.COMPLETE
            for t in estudio.trials
        ),
        "Hiperparametros": mejores_params,
        "Tiempo_ejecucion_seg": duracion,
        "Es_fold_piloto": (
            repeticion == 1 and fold == 1
        )
    }

    predicciones = pd.DataFrame({
        "Semana": 8,
        "Repeticion": repeticion,
        "Fold": fold,
        "Algoritmo": algoritmo,
        "Experimento": "B",
        "code_module": (
            X_val["code_module"].to_numpy()
        ),
        "code_presentation": (
            X_val["code_presentation"].to_numpy()
        ),
        "id_student": g_val.to_numpy(),
        "y_real": y_val.to_numpy(),
        "y_pred": y_pred,
        "prob_adverso": y_prob
    })

    assert not predicciones.duplicated(
        [
            "code_module",
            "code_presentation",
            "id_student"
        ]
    ).any()

    assert predicciones[
        "prob_adverso"
    ].between(0, 1).all()

    return resultado, predicciones

# ------------------------------------------------------------
# 8. PROCESAMIENTO INCREMENTAL
# ------------------------------------------------------------

print("\n=== 3.33.2 EJECUCIÓN ANIDADA ===")

nuevos_folds = 0

for indice_externo in range(
    N_FOLDS_EXTERNOS_333
):

    repeticion = indice_externo // 5 + 1
    fold = indice_externo % 5 + 1

    claves = [
        (repeticion, fold, algoritmo)
        for algoritmo in ALGORITMOS_333
    ]

    if all(
        clave in checkpoint_333["resultados"]
        for clave in claves
    ):
        continue

    if nuevos_folds >= (
        FOLDS_NUEVOS_POR_EJECUCION_333
    ):
        break

    for algoritmo in ALGORITMOS_333:

        clave = (
            repeticion, fold, algoritmo
        )

        if clave in checkpoint_333[
            "resultados"
        ]:
            continue

        resultado, predicciones = (
            evaluar_optuna_333(
                indice_externo,
                algoritmo
            )
        )

        checkpoint_333[
            "resultados"
        ][clave] = resultado

        checkpoint_333[
            "predicciones"
        ][clave] = predicciones

        guardar_checkpoint_333()

        print(
            f"Guardado R{repeticion}-F{fold}"
            f" | {algoritmo}"
            f" | F1={resultado['F1_Optuna']:.4f}"
            f" | Delta={resultado['Delta_F1']:+.4f}"
            f" | Recall={resultado['Recall_Optuna']:.4f}"
        )

    nuevos_folds += 1

# ------------------------------------------------------------
# 9. TABLA DE RESULTADOS
# ------------------------------------------------------------

tabla_resultados_optuna_333 = pd.DataFrame(
    checkpoint_333["resultados"].values()
)

tabla_resultados_optuna_333 = (
    tabla_resultados_optuna_333.sort_values(
        ["Repeticion", "Fold", "Algoritmo"]
    ).reset_index(drop=True)
)

print("\n=== 3.33.3 RESULTADOS ACUMULADOS ===")

columnas_resumen = [
    "Repeticion",
    "Fold",
    "Algoritmo",
    "F1_Optuna",
    "F1_Grid",
    "Delta_F1",
    "Recall_Optuna",
    "Recall_Grid",
    "AUC_Optuna",
    "AUC_Grid",
    "Brecha_interna",
    "Es_fold_piloto"
]

display(
    tabla_resultados_optuna_333[
        columnas_resumen
    ].round(5)
)

# ------------------------------------------------------------
# 10. RESUMEN PROVISIONAL
# ------------------------------------------------------------

print("\n=== 3.33.4 RESUMEN POR ALGORITMO ===")

resumen_optuna_333 = (
    tabla_resultados_optuna_333.groupby(
        "Algoritmo"
    )
    .agg(
        Folds=("Fold", "size"),
        F1_Optuna=("F1_Optuna", "mean"),
        F1_Grid=("F1_Grid", "mean"),
        Delta_F1=("Delta_F1", "mean"),
        Recall_Optuna=("Recall_Optuna", "mean"),
        Recall_Grid=("Recall_Grid", "mean"),
        AUC_Optuna=("AUC_Optuna", "mean"),
        AUC_Grid=("AUC_Grid", "mean"),
        Brecha_interna=("Brecha_interna", "mean")
    )
    .reset_index()
)

display(resumen_optuna_333.round(5))

# ------------------------------------------------------------
# 11. ESTADO DE AVANCE
# ------------------------------------------------------------

print("\n=== 3.33.5 ESTADO ===")

completos_333 = sum(
    all(
        (r, f, a) in checkpoint_333[
            "resultados"
        ]
        for a in ALGORITMOS_333
    )
    for r in [1, 2]
    for f in [1, 2, 3, 4, 5]
)

pendientes_333 = (
    N_FOLDS_EXTERNOS_333 - completos_333
)

print("Folds externos completos:", completos_333)
print("Folds pendientes:", pendientes_333)

print(
    "Evaluaciones externas guardadas:",
    len(checkpoint_333["resultados"])
)

print("Checkpoint:", RUTA_CHECKPOINT_333)

# ------------------------------------------------------------
# 12. COMPROBACIONES FINALES
# ------------------------------------------------------------

assert len(
    checkpoint_333["resultados"]
) == len(
    checkpoint_333["predicciones"]
)

for clave, preds in checkpoint_333[
    "predicciones"
].items():

    assert preds[
        "prob_adverso"
    ].between(0, 1).all()

    assert not preds.duplicated(
        [
            "code_module",
            "code_presentation",
            "id_student"
        ]
    ).any()

print("\n=== 3.33.6 CONTROLES ===")
print("Particiones agrupadas por estudiante.")
print("Optimización exclusivamente interna.")
print("Modelos ajustados dentro del fold externo.")
print("Predicciones externas conservadas.")
print("Resultados GridSearchCV sin modificaciones.")
print("Conjunto de prueba independiente: NO utilizado.")

if pendientes_333 == 0:

    print("\nACTIVIDAD 3.33 COMPLETADA.")

else:

    print(
        "\nRESULTADOS PARCIALES. "
        "Vuelve a ejecutar la misma celda "
        "para continuar."
    )



### 3.34. Evaluación estadística y operativa de la optimización con Optuna

#### Objetivo

Determinar si Optuna mejora consistentemente el rendimiento de los modelos longitudinales de Regresión Logística y Random Forest respecto a GridSearchCV, utilizando las diez particiones externas previamente definidas.

#### Diseño

Se compararán los resultados de ambas estrategias de optimización sobre los mismos folds externos y las mismas matrículas.

La métrica principal será F1 macro. Como métricas complementarias se analizarán recall adverso, ROC-AUC, falsas alertas, precisión y tasa de falsos positivos.

#### Análisis estadístico

Se realizará una prueba t pareada corregida aproximadamente por la dependencia de las particiones de validación cruzada repetida.

Se utilizará la corrección de Nadeau-Bengio, incorporando la relación entre el tamaño de validación y entrenamiento.

Se corregirán los valores p de las dos comparaciones principales mediante Holm, con alfa = 0,05.

Los intervalos t serán individuales, sin ajuste simultáneo por multiplicidad.

También se reportarán las diferencias calculadas excluyendo el fold utilizado inicialmente como piloto.

#### Bootstrap agrupado

Se utilizarán predicciones fuera de muestra emparejadas por matrícula y estudiante.

Se realizará un bootstrap con 1.000 remuestreos de estudiantes, conservando juntas todas sus matrículas y utilizando los mismos estudiantes remuestreados en ambas repeticiones.

Este análisis producirá intervalos descriptivos para la diferencia de F1 macro entre Optuna y GridSearchCV.

Los intervalos bootstrap estarán condicionados a los modelos previamente ajustados y no incorporarán la variabilidad completa del proceso de selección de hiperparámetros.

#### Evaluación operativa

Se calcularán las variaciones en:

- Verdaderos positivos.
- Falsos negativos.
- Falsos positivos.
- Precisión de alertas.
- Porcentaje de matrículas alertadas.
- Recall adverso.

Esto permitirá establecer si las mejoras de recall implican una mayor carga de falsas alertas.

#### Limitaciones

La comparación utiliza diferentes espacios y presupuestos de búsqueda, por lo que no identifica aisladamente el efecto del algoritmo TPE.

El primer fold se utilizó previamente como piloto exploratorio.

La prueba t corregida es aproximada y depende de supuestos sobre la estructura de correlación entre los folds.

No se utilizará el conjunto de prueba independiente.


In [ ]:

# ============================================================
# ACTIVIDAD 3.34
# COMPARACIÓN OPTUNA VS GRIDSEARCHCV
# ============================================================

# Asistencia IA: ChatGPT, octubre de 2026.
# Prompt:
# "Comparar Optuna y GridSearchCV sobre 10 folds
# emparejados de OULAD semana 8. Aplicar t corregida,
# Holm, sensibilidad sin piloto, bootstrap agrupado
# por estudiante y auditoría de falsas alertas."

import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import t as distribucion_t
from sklearn.metrics import f1_score
from IPython.display import display

# ------------------------------------------------------------
# 1. RECUPERACIÓN
# ------------------------------------------------------------

RUTA_334 = "/content/optuna_oulad_semana8_333.pkl"
N_BOOT_334 = 1000
ALFA_334 = 0.05
SEMILLA_334 = 2026

assert os.path.exists(RUTA_334), (
    "No se encuentra el checkpoint de la Actividad 3.33."
)

assert "predicciones_prob_329" in globals(), (
    "Se necesitan las predicciones originales "
    "de GridSearchCV de la Actividad 3.29."
)

with open(RUTA_334, "rb") as archivo:
    checkpoint_334 = pickle.load(archivo)

resultados_334 = pd.DataFrame(
    checkpoint_334["resultados"].values()
)

assert len(resultados_334) == 20
assert len(checkpoint_334["predicciones"]) == 20

print("=== 3.34.1 CONFIGURACIÓN ===")
print("Comparaciones principales: 2")
print("Folds externos: 10")
print("Bootstrap:", N_BOOT_334)
print("Modelos reentrenados: 0")

# ------------------------------------------------------------
# 2. DIFERENCIAS PAREADAS
# ------------------------------------------------------------

print("\n=== 3.34.2 DIFERENCIAS POR FOLD ===")

resultados_334["Delta_recall"] = (
    resultados_334["Recall_Optuna"]
    - resultados_334["Recall_Grid"]
)

resultados_334["Delta_AUC"] = (
    resultados_334["AUC_Optuna"]
    - resultados_334["AUC_Grid"]
)

display(
    resultados_334[
        [
            "Repeticion",
            "Fold",
            "Algoritmo",
            "F1_Optuna",
            "F1_Grid",
            "Delta_F1",
            "Delta_recall",
            "Delta_AUC",
            "Es_fold_piloto"
        ]
    ].sort_values(
        ["Algoritmo", "Repeticion", "Fold"]
    ).round(5)
)

# ------------------------------------------------------------
# 3. T CORREGIDA POR DEPENDENCIA DE FOLDS
# ------------------------------------------------------------

ratios_334 = []

for tr, val in cv_splits_oulad[8]:
    ratios_334.append(len(val) / len(tr))

ratio_334 = float(np.mean(ratios_334))

def contraste_corregido_334(deltas, ratio):

    d = np.asarray(deltas, dtype=float)

    n = len(d)
    media = float(np.mean(d))
    desviacion = float(np.std(d, ddof=1))

    error = desviacion * np.sqrt(
        1 / n + ratio
    )

    if error == 0:
        if media == 0:
            return media, desviacion, 0, 1, media, media

        return (
            media, desviacion,
            np.sign(media) * np.inf,
            0, media, media
        )

    estadistico = media / error

    p_valor = 2 * distribucion_t.sf(
        abs(estadistico), df=n - 1
    )

    margen = distribucion_t.ppf(
        1 - ALFA_334 / 2,
        df=n - 1
    ) * error

    return (
        media,
        desviacion,
        estadistico,
        p_valor,
        media - margen,
        media + margen
    )

filas_contrastes_334 = []

for algoritmo in [
    "LogisticRegression",
    "RandomForest"
]:

    sub = resultados_334.loc[
        resultados_334["Algoritmo"].eq(algoritmo)
    ]

    delta = sub["Delta_F1"].to_numpy()

    (
        media, desv, t_corr, p,
        ic_inf, ic_sup
    ) = contraste_corregido_334(
        delta, ratio_334
    )

    filas_contrastes_334.append({
        "Algoritmo": algoritmo,
        "Delta_F1": media,
        "Desviacion": desv,
        "t_corregida": t_corr,
        "p_sin_ajustar": p,
        "IC95_inferior": ic_inf,
        "IC95_superior": ic_sup,
        "Folds_positivos": int(
            (delta > 0).sum()
        ),
        "Folds_negativos": int(
            (delta < 0).sum()
        )
    })

tabla_contrastes_334 = pd.DataFrame(
    filas_contrastes_334
)

# Corrección de Holm para dos hipótesis.
p_originales = tabla_contrastes_334[
    "p_sin_ajustar"
].to_numpy()

orden = np.argsort(p_originales)

ajustados_ordenados = np.maximum.accumulate(
    (len(p_originales) - np.arange(
        len(p_originales)
    )) * p_originales[orden]
)

p_holm = np.empty_like(p_originales)

p_holm[orden] = np.minimum(
    ajustados_ordenados, 1.0
)

tabla_contrastes_334["p_Holm"] = p_holm

tabla_contrastes_334[
    "Significativo_Holm"
] = p_holm < ALFA_334

print("\n=== 3.34.3 CONTRASTES CORREGIDOS ===")
print(
    "Ratio medio validación/entrenamiento:",
    round(ratio_334, 6)
)

display(tabla_contrastes_334.round(6))

# ------------------------------------------------------------
# 4. SENSIBILIDAD SIN EL FOLD PILOTO
# ------------------------------------------------------------

filas_sensibilidad_334 = []

for algoritmo in [
    "LogisticRegression",
    "RandomForest"
]:

    sub = resultados_334.loc[
        resultados_334["Algoritmo"].eq(algoritmo)
    ]

    for excluir_piloto in [False, True]:

        if excluir_piloto:
            evaluado = sub.loc[
                ~sub["Es_fold_piloto"]
            ]
        else:
            evaluado = sub

        d = evaluado["Delta_F1"]

        filas_sensibilidad_334.append({
            "Algoritmo": algoritmo,
            "Escenario": (
                "Sin piloto"
                if excluir_piloto
                else "Todos los folds"
            ),
            "Folds": len(evaluado),
            "Delta_F1_medio": d.mean(),
            "Delta_F1_std": d.std(),
            "Folds_Optuna_supera": int(
                (d > 0).sum()
            ),
            "Folds_Optuna_inferior": int(
                (d < 0).sum()
            ),
            "Delta_Recall_medio": (
                evaluado["Delta_recall"].mean()
            )
        })

tabla_sensibilidad_334 = pd.DataFrame(
    filas_sensibilidad_334
)

print("\n=== 3.34.4 SENSIBILIDAD SIN PILOTO ===")
display(tabla_sensibilidad_334.round(5))

# ------------------------------------------------------------
# 5. EMPAREJAMIENTO DE PREDICCIONES
# ------------------------------------------------------------

print("\n=== 3.34.5 PREDICCIONES EMPAREJADAS ===")

CLAVES_334 = [
    "Repeticion",
    "Fold",
    "code_module",
    "code_presentation",
    "id_student"
]

parejas_334 = {}

for algoritmo, etiqueta in [
    ("LogisticRegression", "LR B"),
    ("RandomForest", "RF B")
]:

    pred_opt = pd.concat(
        [
            checkpoint_334["predicciones"][
                (rep, fold, algoritmo)
            ]
            for rep in [1, 2]
            for fold in [1, 2, 3, 4, 5]
        ],
        ignore_index=True
    )

    pred_grid = predicciones_prob_329.loc[
        (predicciones_prob_329["Semana"] == 8)
        &
        (predicciones_prob_329["Modelo"] == etiqueta)
    ].copy()

    columnas_grid = CLAVES_334 + [
        "y_real",
        "y_pred"
    ]

    assert len(pred_opt) == 42328
    assert len(pred_grid) == 42328

    assert not pred_opt.duplicated(
        CLAVES_334
    ).any()

    assert not pred_grid.duplicated(
        CLAVES_334
    ).any()

    pareado = pred_opt.merge(
        pred_grid[columnas_grid],
        on=CLAVES_334,
        how="inner",
        validate="one_to_one",
        suffixes=("_optuna", "_grid")
    )

    assert len(pareado) == 42328

    assert (
        pareado["y_real_optuna"].to_numpy()
        ==
        pareado["y_real_grid"].to_numpy()
    ).all()

    pareado["y_real"] = (
        pareado["y_real_optuna"].astype(int)
    )

    parejas_334[algoritmo] = pareado

    # Reproducir el rendimiento por fold desde
    # las predicciones emparejadas.
    for (rep, fold), bloque in pareado.groupby(
        ["Repeticion", "Fold"]
    ):

        fila = resultados_334.loc[
            resultados_334["Algoritmo"].eq(algoritmo)
            &
            resultados_334["Repeticion"].eq(rep)
            &
            resultados_334["Fold"].eq(fold)
        ].iloc[0]

        f1_opt = f1_score(
            bloque["y_real"],
            bloque["y_pred_optuna"],
            average="macro"
        )

        f1_grid = f1_score(
            bloque["y_real"],
            bloque["y_pred_grid"],
            average="macro"
        )

        assert abs(
            f1_opt - fila["F1_Optuna"]
        ) < 0.0001

        assert abs(
            f1_grid - fila["F1_Grid"]
        ) < 0.0001

    print(
        algoritmo,
        "| predicciones:", len(pareado),
        "| estudiantes:",
        pareado["id_student"].nunique()
    )

# ------------------------------------------------------------
# 6. CARGA OPERATIVA DE ALERTAS
# ------------------------------------------------------------

print("\n=== 3.34.6 EFECTO SOBRE LAS ALERTAS ===")

resumen_alertas_334 = []

for algoritmo, tabla in parejas_334.items():

    filas_repeticion = []

    for rep, sub in tabla.groupby("Repeticion"):

        fila = {"Repeticion": rep}

        y = sub["y_real"].to_numpy()

        for nombre, columna in [
            ("Grid", "y_pred_grid"),
            ("Optuna", "y_pred_optuna")
        ]:

            pred = sub[columna].to_numpy()

            tp = int(((y == 1) & (pred == 1)).sum())
            fp = int(((y == 0) & (pred == 1)).sum())
            fn = int(((y == 1) & (pred == 0)).sum())
            tn = int(((y == 0) & (pred == 0)).sum())

            fila[f"TP_{nombre}"] = tp
            fila[f"FP_{nombre}"] = fp
            fila[f"FN_{nombre}"] = fn
            fila[f"TN_{nombre}"] = tn
            fila[f"Alertas_{nombre}"] = tp + fp

            fila[f"Precision_{nombre}_pct"] = (
                100 * tp / max(tp + fp, 1)
            )

            fila[f"Recall_{nombre}_pct"] = (
                100 * tp / max(tp + fn, 1)
            )

            fila[f"FPR_{nombre}_pct"] = (
                100 * fp / max(fp + tn, 1)
            )

        filas_repeticion.append(fila)

    por_repeticion = pd.DataFrame(
        filas_repeticion
    )

    promedio = por_repeticion.drop(
        columns="Repeticion"
    ).mean()

    resumen_alertas_334.append({
        "Algoritmo": algoritmo,
        "TP_adicionales": (
            promedio["TP_Optuna"]
            - promedio["TP_Grid"]
        ),
        "FP_adicionales": (
            promedio["FP_Optuna"]
            - promedio["FP_Grid"]
        ),
        "Alertas_adicionales": (
            promedio["Alertas_Optuna"]
            - promedio["Alertas_Grid"]
        ),
        "Precision_Grid_pct": (
            promedio["Precision_Grid_pct"]
        ),
        "Precision_Optuna_pct": (
            promedio["Precision_Optuna_pct"]
        ),
        "Recall_Grid_pct": (
            promedio["Recall_Grid_pct"]
        ),
        "Recall_Optuna_pct": (
            promedio["Recall_Optuna_pct"]
        ),
        "FPR_Grid_pct": (
            promedio["FPR_Grid_pct"]
        ),
        "FPR_Optuna_pct": (
            promedio["FPR_Optuna_pct"]
        )
    })

tabla_alertas_334 = pd.DataFrame(
    resumen_alertas_334
)

display(tabla_alertas_334.round(3))

print(
    "Los conteos son promedios de dos repeticiones, "
    "no matrículas adicionales independientes."
)

# ------------------------------------------------------------
# 7. BOOTSTRAP AGRUPADO POR ESTUDIANTE
# ------------------------------------------------------------

print("\n=== 3.34.7 BOOTSTRAP AGRUPADO ===")

def f1_desde_confusion_334(conf):
    """
    Orden de las columnas:
    TP, TN, FP, FN.
    """

    tp, tn, fp, fn = conf

    den_pos = 2 * tp + fp + fn
    den_neg = 2 * tn + fp + fn

    f1_pos = (
        2 * tp / den_pos
        if den_pos > 0 else 0.0
    )

    f1_neg = (
        2 * tn / den_neg
        if den_neg > 0 else 0.0
    )

    return 0.5 * (f1_pos + f1_neg)

filas_bootstrap_334 = []

for algoritmo, tabla in parejas_334.items():

    ids = np.sort(
        tabla["id_student"].unique()
    )

    cantidad = len(ids)

    matrices = {}

    for rep in [1, 2]:

        sub = tabla.loc[
            tabla["Repeticion"].eq(rep)
        ].copy()

        y = sub["y_real"].to_numpy()

        columnas = []

        for etiqueta, col_pred in [
            ("Grid", "y_pred_grid"),
            ("Optuna", "y_pred_optuna")
        ]:

            pred = sub[col_pred].to_numpy()

            for nombre, mask in [
                ("TP", (y == 1) & (pred == 1)),
                ("TN", (y == 0) & (pred == 0)),
                ("FP", (y == 0) & (pred == 1)),
                ("FN", (y == 1) & (pred == 0))
            ]:
                col = f"{etiqueta}_{nombre}"
                sub[col] = mask.astype(np.int16)
                columnas.append(col)

        agrupado = (
            sub.groupby("id_student")[columnas]
            .sum()
            .reindex(ids, fill_value=0)
        )

        matrices[rep] = agrupado.to_numpy(
            dtype=np.int32
        )

    rng = np.random.default_rng(
        SEMILLA_334
    )

    deltas_boot = np.empty(
        N_BOOT_334, dtype=float
    )

    for b in range(N_BOOT_334):

        seleccion = rng.integers(
            0, cantidad, size=cantidad
        )

        diferencias_reps = []

        for rep in [1, 2]:

            suma = matrices[rep][
                seleccion
            ].sum(axis=0)

            conf_grid = suma[0:4]
            conf_opt = suma[4:8]

            delta = (
                f1_desde_confusion_334(
                    conf_opt
                )
                -
                f1_desde_confusion_334(
                    conf_grid
                )
            )

            diferencias_reps.append(delta)

        deltas_boot[b] = np.mean(
            diferencias_reps
        )

    ic_inf, ic_sup = np.quantile(
        deltas_boot, [0.025, 0.975]
    )

    filas_bootstrap_334.append({
        "Algoritmo": algoritmo,
        "Estudiantes_unicos": cantidad,
        "Delta_F1_bootstrap_media": (
            float(np.mean(deltas_boot))
        ),
        "IC95_inferior": float(ic_inf),
        "IC95_superior": float(ic_sup),
        "Fraccion_delta_positivo": float(
            np.mean(deltas_boot > 0)
        )
    })

tabla_bootstrap_334 = pd.DataFrame(
    filas_bootstrap_334
)

display(tabla_bootstrap_334.round(5))

# ------------------------------------------------------------
# 8. VISUALIZACIONES
# ------------------------------------------------------------

print("\n=== 3.34.8 GRÁFICOS ===")

fig, ax = plt.subplots(figsize=(10, 5))

for algoritmo, sub in resultados_334.groupby(
    "Algoritmo"
):

    sub = sub.sort_values(
        ["Repeticion", "Fold"]
    )

    etiquetas = [
        f"R{r}-F{f}"
        for r, f in zip(
            sub["Repeticion"],
            sub["Fold"]
        )
    ]

    ax.plot(
        etiquetas,
        sub["Delta_F1"],
        marker="o",
        linewidth=2,
        label=algoritmo
    )

ax.axhline(
    0, linestyle="--", linewidth=1
)

ax.set_title(
    "Optuna frente a GridSearchCV\n"
    "Diferencias emparejadas de F1 macro"
)

ax.set_ylabel("F1 Optuna - F1 Grid")
ax.set_xlabel("Fold externo")
ax.grid(alpha=0.3)
ax.legend()

plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(8, 5))

for _, fila in tabla_contrastes_334.iterrows():

    ax.errorbar(
        x=fila["Delta_F1"],
        y=fila["Algoritmo"],
        xerr=np.array([[
            fila["Delta_F1"] - fila["IC95_inferior"],
            fila["IC95_superior"] - fila["Delta_F1"]
        ]]),
        fmt="o",
        capsize=5
    )

ax.axvline(
    0, linestyle="--", linewidth=1
)

ax.set_title(
    "Diferencias de F1 macro\n"
    "Intervalos t corregidos por dependencia"
)

ax.set_xlabel("Diferencia Optuna - GridSearchCV")
ax.grid(axis="x", alpha=0.3)

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 9. CONTROLES
# ------------------------------------------------------------

print("\n=== 3.34.9 CONTROLES FINALES ===")

assert len(tabla_contrastes_334) == 2
assert len(tabla_bootstrap_334) == 2
assert len(tabla_alertas_334) == 2

assert resultados_334.groupby(
    "Algoritmo"
).size().eq(10).all()

print("Folds emparejados: 10")
print("Sensibilidad sin piloto: calculada")
print("Contrastes corregidos: 2")
print("Corrección Holm: aplicada")
print("Bootstrap por estudiante: completado")
print("Alertas correctas e incorrectas: auditadas")
print("Modelos reentrenados: 0")
print("Conjunto de prueba independiente: NO utilizado")

print("\n=== ACTIVIDAD 3.34 FINALIZADA ===")


In [ ]:

# ============================================================
# ACTIVIDAD 3.34 - CORRECCIÓN Y CONTINUACIÓN
# Reconstrucción de Fold desde las particiones originales.
#
# No modifica las predicciones originales.
# No entrena modelos.
# No utiliza el test independiente.
# ============================================================

import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import f1_score
from IPython.display import display

SEMANA_334 = 8
N_BOOT_334 = 1000
SEMILLA_334 = 2026

CLAVES_334 = [
    "Repeticion",
    "Fold",
    "code_module",
    "code_presentation",
    "id_student"
]

# ------------------------------------------------------------
# 1. RECUPERAR RESULTADOS DE OPTUNA
# ------------------------------------------------------------

ruta = "/content/optuna_oulad_semana8_333.pkl"

assert os.path.exists(ruta), (
    "No se encuentra el checkpoint de Optuna."
)

with open(ruta, "rb") as archivo:
    checkpoint_334 = pickle.load(archivo)

resultados_334 = pd.DataFrame(
    checkpoint_334["resultados"].values()
)

assert len(resultados_334) == 20

# ------------------------------------------------------------
# 2. RECONSTRUIR MAPA REPETICIÓN-ESTUDIANTE-FOLD
# ------------------------------------------------------------

print("=== 3.34.5 RECONSTRUCCIÓN DE FOLDS ===")

grupos_desarrollo = (
    experimentos_oulad[8]["grupos_train"]
    .reset_index(drop=True)
)

particiones = cv_splits_oulad[8]

assert len(particiones) == 10

fragmentos_mapa = []

for indice, (idx_train, idx_validacion) in enumerate(
    particiones
):
    repeticion = indice // 5 + 1
    fold = indice % 5 + 1

    estudiantes_validacion = (
        grupos_desarrollo
        .iloc[idx_validacion]
        .drop_duplicates()
        .to_numpy()
    )

    fragmentos_mapa.append(
        pd.DataFrame({
            "Repeticion": repeticion,
            "id_student": estudiantes_validacion,
            "Fold": fold
        })
    )

mapa_folds_334 = pd.concat(
    fragmentos_mapa,
    ignore_index=True
)

# Cada estudiante pertenece a un solo fold
# externo por repetición.
assert not mapa_folds_334.duplicated(
    ["Repeticion", "id_student"]
).any()

print("Mapa reconstruido correctamente.")
print("Repeticiones:", sorted(
    mapa_folds_334["Repeticion"].unique()
))
print("Folds:", sorted(
    mapa_folds_334["Fold"].unique()
))

# ------------------------------------------------------------
# 3. EMPAREJAR PREDICCIONES ORIGINALES Y OPTUNA
# ------------------------------------------------------------

parejas_334 = {}

modelos = {
    "LogisticRegression": "LR B",
    "RandomForest": "RF B"
}

for algoritmo, etiqueta in modelos.items():

    # Predicciones Optuna.
    pred_opt = pd.concat(
        [
            checkpoint_334["predicciones"][
                (rep, fold, algoritmo)
            ]
            for rep in [1, 2]
            for fold in [1, 2, 3, 4, 5]
        ],
        ignore_index=True
    )

    # Predicciones originales GridSearchCV.
    pred_grid = predicciones_prob_329.loc[
        (predicciones_prob_329["Semana"] == 8)
        &
        (predicciones_prob_329["Modelo"] == etiqueta)
    ].copy()

    # Las claves deben compartir tipos.
    for df in [pred_grid, pred_opt]:
        df["Repeticion"] = pd.to_numeric(
            df["Repeticion"]
        ).astype(int)

        df["id_student"] = pd.to_numeric(
            df["id_student"]
        ).astype(int)

    # Reconstruir Fold exclusivamente para la copia
    # de las predicciones GridSearchCV.
    pred_grid = pred_grid.drop(
        columns=["Fold"],
        errors="ignore"
    )

    pred_grid = pred_grid.merge(
        mapa_folds_334,
        on=["Repeticion", "id_student"],
        how="left",
        validate="many_to_one"
    )

    assert pred_grid["Fold"].notna().all(), (
        f"No se pudieron asignar todos los folds "
        f"para {algoritmo}."
    )

    pred_grid["Fold"] = (
        pred_grid["Fold"].astype(int)
    )

    print(f"\n{algoritmo}")
    print("Columnas Grid:", pred_grid.columns.tolist())

    # Verificar columnas necesarias para comparar.
    requeridas = CLAVES_334 + [
        "y_real", "y_pred"
    ]

    faltantes = [
        c for c in requeridas
        if c not in pred_grid.columns
    ]

    if faltantes:
        raise ValueError(
            f"Faltan columnas en pred_grid: "
            f"{faltantes}. Revisa los nombres "
            f"mostrados arriba."
        )

    assert len(pred_grid) == 42328
    assert len(pred_opt) == 42328

    assert not pred_grid.duplicated(
        CLAVES_334
    ).any()

    assert not pred_opt.duplicated(
        CLAVES_334
    ).any()

    # Emparejamiento de una sola observación
    # de cada estrategia por matrícula y fold.
    pareado = pred_opt.merge(
        pred_grid[
            CLAVES_334 + ["y_real", "y_pred"]
        ],
        on=CLAVES_334,
        how="inner",
        validate="one_to_one",
        suffixes=("_optuna", "_grid")
    )

    assert len(pareado) == 42328

    assert (
        pareado["y_real_optuna"].to_numpy()
        ==
        pareado["y_real_grid"].to_numpy()
    ).all()

    pareado["y_real"] = (
        pareado["y_real_optuna"].astype(int)
    )

    # Reproducir las métricas originales
    # a partir de las predicciones emparejadas.
    for (rep, fold), bloque in pareado.groupby(
        ["Repeticion", "Fold"]
    ):

        referencia = resultados_334.loc[
            (resultados_334["Algoritmo"] == algoritmo)
            &
            (resultados_334["Repeticion"] == rep)
            &
            (resultados_334["Fold"] == fold)
        ]

        assert len(referencia) == 1
        referencia = referencia.iloc[0]

        f1_opt = f1_score(
            bloque["y_real"],
            bloque["y_pred_optuna"],
            average="macro"
        )

        f1_grid = f1_score(
            bloque["y_real"],
            bloque["y_pred_grid"],
            average="macro"
        )

        assert abs(
            f1_opt - referencia["F1_Optuna"]
        ) < 0.0001

        assert abs(
            f1_grid - referencia["F1_Grid"]
        ) < 0.0001

    parejas_334[algoritmo] = pareado

    print("Emparejadas:", len(pareado))
    print(
        "Estudiantes únicos:",
        pareado["id_student"].nunique()
    )
    print("Métricas originales reproducidas.")

# ------------------------------------------------------------
# 4. EFECTO OPERATIVO DE OPTUNA
# ------------------------------------------------------------

print("\n=== 3.34.6 EFECTO SOBRE LAS ALERTAS ===")

filas_alertas = []

for algoritmo, tabla in parejas_334.items():

    resumenes_repeticion = []

    for rep, sub in tabla.groupby("Repeticion"):

        y = sub["y_real"].to_numpy()

        fila = {"Repeticion": rep}

        for nombre, columna in [
            ("Grid", "y_pred_grid"),
            ("Optuna", "y_pred_optuna")
        ]:

            pred = sub[columna].to_numpy()

            tp = int(
                ((y == 1) & (pred == 1)).sum()
            )
            fp = int(
                ((y == 0) & (pred == 1)).sum()
            )
            fn = int(
                ((y == 1) & (pred == 0)).sum()
            )
            tn = int(
                ((y == 0) & (pred == 0)).sum()
            )

            fila[f"TP_{nombre}"] = tp
            fila[f"FP_{nombre}"] = fp
            fila[f"FN_{nombre}"] = fn
            fila[f"TN_{nombre}"] = tn

            fila[f"Alertas_{nombre}"] = tp + fp

            fila[f"Precision_{nombre}_pct"] = (
                100 * tp / max(tp + fp, 1)
            )

            fila[f"Recall_{nombre}_pct"] = (
                100 * tp / max(tp + fn, 1)
            )

            fila[f"FPR_{nombre}_pct"] = (
                100 * fp / max(fp + tn, 1)
            )

        resumenes_repeticion.append(fila)

    promedios = (
        pd.DataFrame(resumenes_repeticion)
        .drop(columns="Repeticion")
        .mean()
    )

    filas_alertas.append({
        "Algoritmo": algoritmo,
        "TP_adicionales": (
            promedios["TP_Optuna"]
            - promedios["TP_Grid"]
        ),
        "FP_adicionales": (
            promedios["FP_Optuna"]
            - promedios["FP_Grid"]
        ),
        "Alertas_adicionales": (
            promedios["Alertas_Optuna"]
            - promedios["Alertas_Grid"]
        ),
        "Precision_Grid_pct": (
            promedios["Precision_Grid_pct"]
        ),
        "Precision_Optuna_pct": (
            promedios["Precision_Optuna_pct"]
        ),
        "Recall_Grid_pct": (
            promedios["Recall_Grid_pct"]
        ),
        "Recall_Optuna_pct": (
            promedios["Recall_Optuna_pct"]
        ),
        "FPR_Grid_pct": (
            promedios["FPR_Grid_pct"]
        ),
        "FPR_Optuna_pct": (
            promedios["FPR_Optuna_pct"]
        )
    })

tabla_alertas_334 = pd.DataFrame(
    filas_alertas
)

display(tabla_alertas_334.round(3))

# ------------------------------------------------------------
# 5. BOOTSTRAP AGRUPADO POR ESTUDIANTE
# ------------------------------------------------------------

print("\n=== 3.34.7 BOOTSTRAP AGRUPADO ===")

def f1_confusion_334(conf):
    # Columnas: TP, TN, FP, FN
    tp, tn, fp, fn = conf

    den_pos = 2 * tp + fp + fn
    den_neg = 2 * tn + fp + fn

    f1_pos = (
        2 * tp / den_pos
        if den_pos else 0.0
    )

    f1_neg = (
        2 * tn / den_neg
        if den_neg else 0.0
    )

    return 0.5 * (f1_pos + f1_neg)

resultados_boot = []

for algoritmo, tabla in parejas_334.items():

    estudiantes = np.sort(
        tabla["id_student"].unique()
    )

    n_estudiantes = len(estudiantes)
    matrices = {}

    for rep in [1, 2]:

        sub = tabla.loc[
            tabla["Repeticion"] == rep
        ].copy()

        y = sub["y_real"].to_numpy()

        columnas_conf = []

        for estrategia, columna in [
            ("Grid", "y_pred_grid"),
            ("Optuna", "y_pred_optuna")
        ]:

            pred = sub[columna].to_numpy()

            condiciones = {
                "TP": (y == 1) & (pred == 1),
                "TN": (y == 0) & (pred == 0),
                "FP": (y == 0) & (pred == 1),
                "FN": (y == 1) & (pred == 0)
            }

            for tipo, condicion in condiciones.items():

                columna_conf = (
                    f"{estrategia}_{tipo}"
                )

                sub[columna_conf] = (
                    condicion.astype(np.int8)
                )

                columnas_conf.append(columna_conf)

        matrices[rep] = (
            sub.groupby("id_student")[columnas_conf]
            .sum()
            .reindex(estudiantes, fill_value=0)
            .to_numpy(dtype=np.int32)
        )

    rng = np.random.default_rng(
        SEMILLA_334
    )

    deltas = np.empty(
        N_BOOT_334, dtype=float
    )

    # Estimación puntual: media de las dos
    # repeticiones fuera de muestra.
    diferencias_originales = []

    for rep in [1, 2]:

        suma = matrices[rep].sum(axis=0)

        diferencias_originales.append(
            f1_confusion_334(suma[4:8])
            -
            f1_confusion_334(suma[0:4])
        )

    delta_observado = float(
        np.mean(diferencias_originales)
    )

    for b in range(N_BOOT_334):

        indices = rng.integers(
            0, n_estudiantes,
            size=n_estudiantes
        )

        diferencias = []

        for rep in [1, 2]:

            suma = matrices[rep][indices].sum(
                axis=0
            )

            diferencias.append(
                f1_confusion_334(suma[4:8])
                -
                f1_confusion_334(suma[0:4])
            )

        deltas[b] = np.mean(diferencias)

    limite_inf, limite_sup = np.quantile(
        deltas, [0.025, 0.975]
    )

    resultados_boot.append({
        "Algoritmo": algoritmo,
        "Estudiantes_unicos": n_estudiantes,
        "Delta_F1_OOF": delta_observado,
        "IC95_inferior": limite_inf,
        "IC95_superior": limite_sup,
        "Fraccion_delta_positivo": (
            np.mean(deltas > 0)
        )
    })

tabla_bootstrap_334 = pd.DataFrame(
    resultados_boot
)

display(tabla_bootstrap_334.round(5))

# ------------------------------------------------------------
# 6. GRÁFICOS
# ------------------------------------------------------------

print("\n=== 3.34.8 GRÁFICOS ===")

fig, ax = plt.subplots(figsize=(10, 5))

for algoritmo, sub in resultados_334.groupby(
    "Algoritmo"
):

    sub = sub.sort_values(
        ["Repeticion", "Fold"]
    )

    etiquetas = [
        f"R{r}-F{f}"
        for r, f in zip(
            sub["Repeticion"],
            sub["Fold"]
        )
    ]

    ax.plot(
        etiquetas,
        sub["Delta_F1"],
        marker="o",
        label=algoritmo
    )

ax.axhline(0, linestyle="--")
ax.set_title(
    "Optuna - GridSearchCV\n"
    "Diferencias de F1 macro por fold"
)
ax.set_xlabel("Fold externo")
ax.set_ylabel("Diferencia F1 macro")
ax.grid(alpha=0.3)
ax.legend()

plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(9, 4.5))

for _, fila in tabla_bootstrap_334.iterrows():

    delta = fila["Delta_F1_OOF"]

    ax.errorbar(
        delta,
        fila["Algoritmo"],
        xerr=[[
            delta - fila["IC95_inferior"],
            fila["IC95_superior"] - delta
        ]],
        fmt="o",
        capsize=5
    )

ax.axvline(0, linestyle="--")

ax.set_title(
    "Diferencia de F1 macro\n"
    "Bootstrap agrupado por estudiante"
)
ax.set_xlabel("Optuna - GridSearchCV")
ax.grid(axis="x", alpha=0.3)

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 7. CONTROLES FINALES
# ------------------------------------------------------------

print("\n=== 3.34.9 CONTROLES FINALES ===")

assert len(parejas_334) == 2
assert len(tabla_alertas_334) == 2
assert len(tabla_bootstrap_334) == 2

for algoritmo, tabla in parejas_334.items():
    assert len(tabla) == 42328

    assert not tabla.duplicated(
        CLAVES_334
    ).any()

print("Folds reconstruidos desde particiones originales.")
print("Predicciones emparejadas correctamente.")
print("Folds originales y F1 auditados.")
print("Análisis operativo: completado.")
print("Bootstrap agrupado: completado.")
print("Modelos reentrenados: 0")
print("Conjunto de prueba independiente: NO utilizado.")

print("\n=== ACTIVIDAD 3.34 COMPLETADA ===")


In [ ]:

# ============================================================
# ACTIVIDAD 3.34 - CORRECCIÓN FINAL DE GRÁFICOS
# No recalcula bootstrap ni reentrena modelos.
# ============================================================

import numpy as np
import matplotlib.pyplot as plt

assert "tabla_bootstrap_334" in globals()
assert "resultados_334" in globals()
assert "tabla_alertas_334" in globals()
assert "tabla_contrastes_334" in globals()
assert "parejas_334" in globals()

# ------------------------------------------------------------
# 3.34.8.1 DIFERENCIAS POR FOLD
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(11, 5))

for algoritmo, sub in resultados_334.groupby(
    "Algoritmo"
):
    sub = sub.sort_values(
        ["Repeticion", "Fold"]
    )

    posiciones = np.arange(len(sub))

    etiquetas = [
        f"R{int(r)}-F{int(f)}"
        for r, f in zip(
            sub["Repeticion"],
            sub["Fold"]
        )
    ]

    ax.plot(
        posiciones,
        sub["Delta_F1"],
        marker="o",
        linewidth=2,
        label=algoritmo
    )

ax.axhline(0, linestyle="--", linewidth=1)

ax.set_xticks(posiciones)
ax.set_xticklabels(etiquetas)

ax.set_title(
    "Optuna frente a GridSearchCV\n"
    "Diferencias emparejadas de F1 macro"
)
ax.set_xlabel("Fold externo")
ax.set_ylabel("Diferencia F1 macro (Optuna - Grid)")
ax.grid(alpha=0.3)
ax.legend()

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 3.34.8.2 INTERVALOS BOOTSTRAP CORREGIDOS
# ------------------------------------------------------------

tabla_grafico = tabla_bootstrap_334.reset_index(
    drop=True
).copy()

assert len(tabla_grafico) == 2

fig, ax = plt.subplots(figsize=(10, 4.5))

for i, fila in tabla_grafico.iterrows():

    centro = float(fila["Delta_F1_OOF"])
    inferior = float(fila["IC95_inferior"])
    superior = float(fila["IC95_superior"])

    assert inferior <= centro <= superior

    error_inferior = centro - inferior
    error_superior = superior - centro

    # Corrección: para un punto,
    # xerr debe tener dimensiones (2, 1).
    errores = np.array([
        [error_inferior],
        [error_superior]
    ], dtype=float)

    assert errores.shape == (2, 1)

    ax.errorbar(
        x=[centro],
        y=[i],
        xerr=errores,
        fmt="o",
        capsize=6,
        markersize=8,
        linewidth=2
    )

    ax.annotate(
        f"{centro:+.4f}",
        (centro, i),
        xytext=(0, 13),
        textcoords="offset points",
        ha="center"
    )

ax.set_yticks(
    np.arange(len(tabla_grafico))
)
ax.set_yticklabels(
    tabla_grafico["Algoritmo"]
)

ax.axvline(
    0,
    linestyle="--",
    linewidth=1,
    label="Sin diferencia"
)

ax.set_title(
    "Optuna frente a GridSearchCV - Semana 8\n"
    "IC 95 % bootstrap agrupado por estudiante"
)

ax.set_xlabel(
    "Diferencia F1 macro (Optuna - GridSearchCV)"
)

ax.set_ylabel("Algoritmo")
ax.grid(axis="x", alpha=0.3)
ax.legend()

ax.set_ylim(
    -0.6,
    len(tabla_grafico) - 0.4
)

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 3.34.9 CONTROLES FINALES
# ------------------------------------------------------------

print("=== 3.34.9 CONTROLES FINALES ===")

assert len(tabla_contrastes_334) == 2
assert len(tabla_bootstrap_334) == 2
assert len(tabla_alertas_334) == 2

for algoritmo, tabla in parejas_334.items():

    assert len(tabla) == 42328

    assert not tabla.duplicated(
        [
            "Repeticion",
            "Fold",
            "code_module",
            "code_presentation",
            "id_student"
        ]
    ).any()

    assert (
        tabla["y_real_optuna"].to_numpy()
        ==
        tabla["y_real_grid"].to_numpy()
    ).all()

print("Comparaciones principales: 2")
print("Folds externos por algoritmo: 10")
print("Predicciones emparejadas por algoritmo: 42328")
print("Bootstrap agrupado: 1000 remuestreos")
print("Intervalos de confianza: verificados")
print("Gráficos: generados correctamente")
print("Modelos reentrenados: 0")
print("Test independiente: NO utilizado")

print("\n=== ACTIVIDAD 3.34 COMPLETADA ===")



### 3.34.10. Conclusión final de la optimización con Optuna

La comparación entre Optuna y GridSearchCV se completó sobre diez folds externos emparejados de la semana 8, preservando la agrupación por estudiante y las predicciones fuera de muestra.

En Random Forest B, Optuna incrementó el F1 macro medio en 0,014005 puntos. La mejora se observó en los diez folds externos y persistió en los nueve folds restantes al excluir el utilizado previamente como piloto.

La prueba t corregida por dependencia de folds, con ajuste de Holm para las dos comparaciones principales, obtuvo un valor p ajustado de 0,009287. El bootstrap agrupado por estudiante produjo una diferencia de F1 macro de 0,01405, con un intervalo del 95 % entre 0,01062 y 0,01774.

En Regresión Logística B no se observó una mejora concluyente del F1 macro. El intervalo bootstrap incluyó el cero y la diferencia media entre los métodos de optimización fue prácticamente nula.

Sin embargo, Optuna incrementó el recall adverso de ambos algoritmos. En Regresión Logística pasó de 59,09 % a 63,78 %, mientras que en Random Forest aumentó de 52,58 % a 60,47 %.

Este incremento de sensibilidad estuvo acompañado de mayores tasas de falsos positivos y una reducción de la precisión de las alertas.

Los resultados permiten concluir que Optuna ofrece beneficios verificables para Random Forest dentro del protocolo de desarrollo, aunque no demuestra superioridad generalizada sobre GridSearchCV.

La diferencia observada tampoco puede atribuirse exclusivamente al algoritmo TPE, puesto que los espacios y presupuestos de búsqueda fueron distintos.

La comparación definitiva de configuraciones deberá considerar no solo F1 macro, sino también recall, calibración probabilística, carga de alertas, estabilidad y diferencias entre subgrupos.

El conjunto de prueba independiente permaneció completamente reservado.



### 3.35. Comparación integral de configuraciones candidatas

#### Objetivo

Comparar las configuraciones longitudinales obtenidas mediante GridSearchCV, Optuna y Transformer Encoder, considerando conjuntamente rendimiento predictivo, calidad probabilística, carga operativa y heterogeneidad entre subgrupos.

#### Configuraciones

1. Regresión Logística B — GridSearchCV.
2. Regresión Logística B — Optuna.
3. Random Forest B — GridSearchCV.
4. Random Forest B — Optuna.
5. Transformer C.

#### Protocolo

Se reutilizarán las predicciones fuera de muestra correspondientes a los mismos diez folds externos de la semana 8.

Las métricas se calcularán por repetición y posteriormente se promediarán, evitando interpretar las dos repeticiones como conjuntos independientes de estudiantes.

No se modificarán los umbrales de clasificación ni se realizarán nuevos entrenamientos.

#### Criterios de comparación

**Rendimiento:** F1 macro, recall adverso, precisión, ROC-AUC y accuracy.

**Calibración:** Brier Score y error de calibración esperado (ECE).

**Carga operativa:** proporción de matrículas alertadas, verdaderos positivos y falsos positivos.

**Heterogeneidad:** diferencias descriptivas de recall y tasa de falsos positivos por género, discapacidad declarada, IMD, edad, educación previa, región y módulo académico.

#### Limitaciones

Los resultados pertenecen a la fase de desarrollo.

La selección de configuraciones sobre estas predicciones puede introducir optimismo en el rendimiento estimado del modelo finalmente elegido.

Las diferencias entre subgrupos no constituyen pruebas causales de discriminación.

No se seleccionarán umbrales utilizando las predicciones externas de validación.

El conjunto de prueba independiente permanecerá reservado.


In [ ]:

# ============================================================
# ACTIVIDAD 3.35.0 - CORRECCIÓN DE PROBABILIDADES
#
# Recupera las probabilidades GridSearchCV originales
# de la Actividad 3.29 y las incorpora a parejas_334.
#
# No reentrena modelos.
# No modifica los valores predictivos.
# No utiliza el test independiente.
# ============================================================

import numpy as np
import pandas as pd

print("=== 3.35.0.1 VERIFICACIÓN DE OBJETOS ===")

assert "parejas_334" in globals(), (
    "No se encuentra parejas_334. "
    "Ejecuta primero la corrección de la Actividad 3.34."
)

assert "predicciones_prob_329" in globals(), (
    "No se encuentran las probabilidades originales "
    "de la Actividad 3.29."
)

claves_3350 = [
    "Repeticion",
    "code_module",
    "code_presentation",
    "id_student"
]

configuraciones_3350 = {
    "LogisticRegression": "LR B",
    "RandomForest": "RF B"
}

print("Tabla de probabilidades originales: disponible")
print("Predicciones emparejadas: disponibles")

# ------------------------------------------------------------
# 1. RECUPERAR PROBABILIDADES GRIDSEARCHCV
# ------------------------------------------------------------

print("\n=== 3.35.0.2 RECUPERACIÓN ===")

for algoritmo, nombre_modelo in configuraciones_3350.items():

    print(f"\nModelo: {nombre_modelo}")

    # Predicciones originales fuera de muestra.
    original = predicciones_prob_329.loc[
        (predicciones_prob_329["Semana"] == 8)
        &
        (predicciones_prob_329["Modelo"] == nombre_modelo)
    ].copy()

    columnas_necesarias = (
        claves_3350
        +
        [
            "y_real",
            "y_pred",
            "prob_adverso_329"
        ]
    )

    faltantes = [
        col for col in columnas_necesarias
        if col not in original.columns
    ]

    assert not faltantes, (
        f"Faltan columnas originales: {faltantes}"
    )

    original = original[
        columnas_necesarias
    ].copy()

    assert len(original) == 42328

    assert not original.duplicated(
        claves_3350
    ).any(), (
        "Existen matrículas duplicadas "
        "en las probabilidades originales."
    )

    # Copia de predicciones previamente emparejadas.
    base = parejas_334[algoritmo].copy()

    assert len(base) == 42328

    assert "prob_adverso" in base.columns, (
        "No se encontró la probabilidad de Optuna."
    )

    # Permite repetir esta corrección sin
    # generar columnas duplicadas.
    base = base.drop(
        columns=["prob_adverso_329"],
        errors="ignore"
    )

    # Columnas auxiliares para verificar identidad.
    original = original.rename(
        columns={
            "y_real": "y_real_verificacion",
            "y_pred": "y_pred_verificacion"
        }
    )

    # Incorporación usando una clave de matrícula.
    # Fold no es necesario: cada matrícula pertenece
    # a un solo fold externo por repetición.
    corregida = base.merge(
        original,
        on=claves_3350,
        how="left",
        validate="one_to_one"
    )

    # --------------------------------------------------------
    # 2. CONTROL DE INTEGRIDAD
    # --------------------------------------------------------

    assert len(corregida) == len(base), (
        "El emparejamiento cambió el número de filas."
    )

    assert corregida[
        "prob_adverso_329"
    ].notna().all(), (
        "Hay probabilidades sin correspondencia."
    )

    assert (
        corregida["y_real_grid"].to_numpy()
        ==
        corregida[
            "y_real_verificacion"
        ].to_numpy()
    ).all(), (
        "Las etiquetas originales no coinciden."
    )

    assert (
        corregida["y_pred_grid"].to_numpy()
        ==
        corregida[
            "y_pred_verificacion"
        ].to_numpy()
    ).all(), (
        "Las predicciones originales no coinciden."
    )

    probabilidades = pd.to_numeric(
        corregida["prob_adverso_329"]
    ).to_numpy(dtype=float)

    assert np.isfinite(probabilidades).all()

    assert (
        (probabilidades >= 0)
        &
        (probabilidades <= 1)
    ).all()

    # Eliminar columnas auxiliares.
    corregida = corregida.drop(
        columns=[
            "y_real_verificacion",
            "y_pred_verificacion"
        ]
    )

    # Actualizar solamente la tabla derivada.
    parejas_334[algoritmo] = corregida

    print("Matrículas emparejadas:", len(corregida))
    print(
        "Estudiantes únicos:",
        corregida["id_student"].nunique()
    )
    print(
        "Probabilidades Grid recuperadas:",
        corregida["prob_adverso_329"].notna().sum()
    )
    print(
        "Probabilidades Optuna conservadas:",
        corregida["prob_adverso"].notna().sum()
    )
    print("Etiquetas originales coinciden: SÍ")
    print("Predicciones originales coinciden: SÍ")

# ------------------------------------------------------------
# 3. CONTROLES FINALES
# ------------------------------------------------------------

print("\n=== 3.35.0.3 CONTROLES FINALES ===")

for algoritmo in configuraciones_3350:

    tabla = parejas_334[algoritmo]

    assert len(tabla) == 42328

    assert {
        "prob_adverso",
        "prob_adverso_329",
        "y_pred_optuna",
        "y_pred_grid"
    }.issubset(tabla.columns)

    assert not tabla.duplicated(
        claves_3350
    ).any()

print("Modelos verificados: 2")
print("Predicciones por modelo: 42328")
print("Probabilidades originales recuperadas")
print("Emparejamiento validado")
print("Valores predictivos sin modificaciones")
print("Modelos reentrenados: 0")
print("Conjunto de prueba independiente: NO utilizado")

print(
    "\n=== CORRECCIÓN 3.35.0 COMPLETADA ==="
)


In [ ]:


# ============================================================
# ACTIVIDAD 3.35
# COMPARACIÓN INTEGRAL DE CONFIGURACIONES
# SEMANA 8 - SIN REENTRENAMIENTO
# ============================================================

# Asistencia IA: ChatGPT, octubre de 2026.
# Prompt:
# "Comparar cinco configuraciones de OULAD sobre
# predicciones OOF emparejadas, incluyendo F1 macro,
# ROC-AUC, Brier, ECE, carga de alertas y diferencias
# descriptivas por subgrupo. No acceder al test."

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    accuracy_score,
    roc_auc_score,
    brier_score_loss
)
from IPython.display import display

SEMANA_335 = 8
N_REPS_335 = 2

CLAVES_335 = [
    "Repeticion",
    "Fold",
    "code_module",
    "code_presentation",
    "id_student"
]

NOMBRES_335 = [
    "LR B | Grid",
    "LR B | Optuna",
    "RF B | Grid",
    "RF B | Optuna",
    "Transformer C"
]

assert "parejas_334" in globals()
assert "predicciones_prob_329" in globals()
assert "oulad_tablas" in globals()

# ------------------------------------------------------------
# 1. NORMALIZACIÓN DE PREDICCIONES
# ------------------------------------------------------------

def extraer_parejas_335(
    algoritmo, estrategia, nombre
):

    base = parejas_334[algoritmo]

    columna_pred = (
        "y_pred_grid"
        if estrategia == "Grid"
        else "y_pred_optuna"
    )

    columna_prob = (
        "prob_adverso_329"
        if estrategia == "Grid"
        else "prob_adverso"
    )

    salida = base[
        CLAVES_335 + ["y_real"]
    ].copy()

    salida["y_pred"] = (
        base[columna_pred].to_numpy()
    )

    salida["prob_adverso"] = (
        base[columna_prob].to_numpy()
    )

    salida["Configuracion"] = nombre

    return salida

fragmentos_335 = [
    extraer_parejas_335(
        "LogisticRegression",
        "Grid",
        "LR B | Grid"
    ),
    extraer_parejas_335(
        "LogisticRegression",
        "Optuna",
        "LR B | Optuna"
    ),
    extraer_parejas_335(
        "RandomForest",
        "Grid",
        "RF B | Grid"
    ),
    extraer_parejas_335(
        "RandomForest",
        "Optuna",
        "RF B | Optuna"
    )
]

# Recuperar Transformer C de la Actividad 3.29.
transformer_335 = predicciones_prob_329.loc[
    (predicciones_prob_329["Semana"] == 8)
    &
    (
        predicciones_prob_329["Modelo"]
        == "Transformer C"
    )
].copy()

# Reconstruir Fold usando el emparejamiento
# previamente validado en la Actividad 3.34.
mapa_335 = (
    parejas_334["LogisticRegression"][
        CLAVES_335
    ]
    .drop_duplicates()
)

claves_sin_fold_335 = [
    c for c in CLAVES_335 if c != "Fold"
]

assert not mapa_335.duplicated(
    claves_sin_fold_335
).any()

transformer_335 = transformer_335.drop(
    columns="Fold",
    errors="ignore"
)

transformer_335 = transformer_335.merge(
    mapa_335,
    on=claves_sin_fold_335,
    how="left",
    validate="one_to_one"
)

assert transformer_335["Fold"].notna().all()

transformer_335 = transformer_335[
    CLAVES_335 + [
        "y_real",
        "y_pred",
        "prob_adverso_329"
    ]
].rename(
    columns={
        "prob_adverso_329": "prob_adverso"
    }
)

transformer_335["Configuracion"] = (
    "Transformer C"
)

fragmentos_335.append(transformer_335)

predicciones_335 = pd.concat(
    fragmentos_335,
    ignore_index=True
)

predicciones_335["y_real"] = (
    predicciones_335["y_real"].astype(int)
)

predicciones_335["y_pred"] = (
    predicciones_335["y_pred"].astype(int)
)

predicciones_335["prob_adverso"] = (
    pd.to_numeric(
        predicciones_335["prob_adverso"]
    )
)

assert not predicciones_335.duplicated(
    CLAVES_335 + ["Configuracion"]
).any()

assert predicciones_335[
    "prob_adverso"
].notna().all()

assert np.isfinite(
    predicciones_335["prob_adverso"]
).all()

assert predicciones_335[
    "prob_adverso"
].between(0, 1).all()

assert set(
    predicciones_335["Configuracion"]
) == set(NOMBRES_335)

conteos_335 = (
    predicciones_335.groupby(
        ["Configuracion", "Repeticion"]
    ).size()
)

assert conteos_335.eq(21164).all()

# Verificación de las mismas etiquetas para
# todas las configuraciones y observaciones.
referencia_335 = (
    predicciones_335.loc[
        predicciones_335["Configuracion"]
        == "LR B | Grid",
        CLAVES_335 + ["y_real"]
    ]
)

comprobacion_335 = predicciones_335.merge(
    referencia_335,
    on=CLAVES_335,
    how="inner",
    validate="many_to_one",
    suffixes=("", "_referencia")
)

assert len(comprobacion_335) == len(
    predicciones_335
)

assert (
    comprobacion_335["y_real"].to_numpy()
    ==
    comprobacion_335[
        "y_real_referencia"
    ].to_numpy()
).all()

print("=== 3.35.1 INTEGRIDAD ===")
print("Configuraciones:", len(NOMBRES_335))
print("Predicciones:", len(predicciones_335))
print("Repeticiones:", N_REPS_335)
print(
    "Estudiantes únicos:",
    predicciones_335["id_student"].nunique()
)
print("Mismas etiquetas y matrículas: SÍ")

# ------------------------------------------------------------
# 2. MÉTRICAS DE CALIBRACIÓN
# ------------------------------------------------------------

def ece_335(y, probabilidades, n_bins=10):

    y = np.asarray(y, dtype=int)
    p = np.asarray(
        probabilidades,
        dtype=float
    )

    indices = np.clip(
        (p * n_bins).astype(int),
        0,
        n_bins - 1
    )

    resultado = 0.0

    for bin_id in range(n_bins):

        mascara = indices == bin_id

        if not mascara.any():
            continue

        confianza = p[mascara].mean()
        frecuencia = y[mascara].mean()

        resultado += (
            mascara.mean()
            * abs(confianza - frecuencia)
        )

    return float(resultado)

# ------------------------------------------------------------
# 3. MÉTRICAS POR REPETICIÓN
# ------------------------------------------------------------

registros_metricas_335 = []

for (config, rep), sub in predicciones_335.groupby(
    ["Configuracion", "Repeticion"]
):

    y = sub["y_real"].to_numpy()
    pred = sub["y_pred"].to_numpy()
    p = sub["prob_adverso"].to_numpy()

    vn, fp, fn, vp = confusion_matrix(
        y,
        pred,
        labels=[0, 1]
    ).ravel()

    registros_metricas_335.append({
        "Configuracion": config,
        "Repeticion": int(rep),
        "Matriculas": len(sub),
        "F1_macro": f1_score(
            y, pred, average="macro"
        ),
        "Recall_pct": 100 * recall_score(
            y, pred, zero_division=0
        ),
        "Precision_pct": 100 * precision_score(
            y, pred, zero_division=0
        ),
        "FPR_pct": 100 * fp / max(fp + vn, 1),
        "ROC_AUC": roc_auc_score(y, p),
        "Accuracy": accuracy_score(y, pred),
        "Brier": brier_score_loss(y, p),
        "ECE": ece_335(y, p),
        "Alertadas_pct": (
            100 * (vp + fp) / len(sub)
        ),
        "VN": int(vn),
        "FP": int(fp),
        "FN": int(fn),
        "VP": int(vp)
    })

metricas_335 = pd.DataFrame(
    registros_metricas_335
)

resumen_335 = (
    metricas_335.groupby("Configuracion")
    .agg(
        F1_macro=("F1_macro", "mean"),
        Recall_pct=("Recall_pct", "mean"),
        Precision_pct=("Precision_pct", "mean"),
        FPR_pct=("FPR_pct", "mean"),
        ROC_AUC=("ROC_AUC", "mean"),
        Accuracy=("Accuracy", "mean"),
        Brier=("Brier", "mean"),
        ECE=("ECE", "mean"),
        Alertadas_pct=("Alertadas_pct", "mean"),
        VP_medios=("VP", "mean"),
        FP_medios=("FP", "mean"),
        FN_medios=("FN", "mean")
    )
    .reindex(NOMBRES_335)
    .reset_index()
)

print("\n=== 3.35.2 RENDIMIENTO INTEGRAL ===")

display(
    resumen_335[
        [
            "Configuracion",
            "F1_macro",
            "Recall_pct",
            "Precision_pct",
            "FPR_pct",
            "ROC_AUC",
            "Accuracy",
            "Brier",
            "ECE"
        ]
    ].round(5)
)

print("\n=== 3.35.3 CARGA OPERATIVA ===")

display(
    resumen_335[
        [
            "Configuracion",
            "Alertadas_pct",
            "VP_medios",
            "FP_medios",
            "FN_medios",
            "Precision_pct",
            "Recall_pct"
        ]
    ].round(2)
)

# ------------------------------------------------------------
# 4. AUDITORÍA DE SUBGRUPOS
# ------------------------------------------------------------

print("\n=== 3.35.4 AUDITORÍA DE SUBGRUPOS ===")

claves_academicas_335 = [
    "code_module",
    "code_presentation",
    "id_student"
]

dimensiones_335 = [
    "gender",
    "disability",
    "imd_band",
    "age_band",
    "highest_education",
    "region",
    "code_module"
]

info_335 = oulad_tablas[
    "studentInfo.csv"
][
    claves_academicas_335
    +
    [d for d in dimensiones_335
     if d != "code_module"]
].copy()

assert not info_335.duplicated(
    claves_academicas_335
).any()

datos_grupos_335 = predicciones_335.merge(
    info_335,
    on=claves_academicas_335,
    how="left",
    validate="many_to_one"
)

for dim in dimensiones_335:
    datos_grupos_335[dim] = (
        datos_grupos_335[dim]
        .fillna("No informado")
        .astype(str)
    )

datos_grupos_335["imd_band"] = (
    datos_grupos_335["imd_band"].replace({
        "10-20": "10-20%"
    })
)

filas_grupos_335 = []

for dimension in dimensiones_335:

    for (
        config,
        rep,
        grupo
    ), sub in datos_grupos_335.groupby(
        ["Configuracion", "Repeticion", dimension]
    ):

        y = sub["y_real"].to_numpy()
        pred = sub["y_pred"].to_numpy()

        adversos = int((y == 1).sum())
        favorables = int((y == 0).sum())

        soporte = (
            len(sub) >= 150
            and adversos >= 50
            and favorables >= 50
        )

        vn, fp, fn, vp = confusion_matrix(
            y, pred, labels=[0, 1]
        ).ravel()

        filas_grupos_335.append({
            "Configuracion": config,
            "Repeticion": int(rep),
            "Dimension": dimension,
            "Grupo": grupo,
            "Matriculas": len(sub),
            "Adversos": adversos,
            "Favorables": favorables,
            "Soporte": soporte,
            "Prevalencia_pct": (
                100 * adversos / len(sub)
            ),
            "Recall_pct": (
                100 * vp / adversos
                if adversos else np.nan
            ),
            "FPR_pct": (
                100 * fp / favorables
                if favorables else np.nan
            )
        })

tabla_grupos_335 = pd.DataFrame(
    filas_grupos_335
)

# Se incluyen en las brechas solamente
# categorías con soporte suficiente en ambas
# repeticiones de la misma configuración.
por_grupo_335 = (
    tabla_grupos_335.groupby(
        ["Configuracion", "Dimension", "Grupo"]
    )
    .agg(
        Repeticiones=("Repeticion", "nunique"),
        Soporte_ambas=("Soporte", "all"),
        Recall_pct=("Recall_pct", "mean"),
        FPR_pct=("FPR_pct", "mean"),
        Prevalencia_pct=("Prevalencia_pct", "mean"),
        Matriculas=("Matriculas", "mean")
    )
    .reset_index()
)

por_grupo_335 = por_grupo_335.loc[
    por_grupo_335["Soporte_ambas"]
    &
    por_grupo_335["Repeticiones"].eq(2)
].copy()

filas_brechas_335 = []

for (config, dim), sub in por_grupo_335.groupby(
    ["Configuracion", "Dimension"]
):

    if len(sub) < 2:
        continue

    filas_brechas_335.append({
        "Configuracion": config,
        "Dimension": dim,
        "Grupos": len(sub),
        "Brecha_recall_pp": (
            sub["Recall_pct"].max()
            - sub["Recall_pct"].min()
        ),
        "Brecha_FPR_pp": (
            sub["FPR_pct"].max()
            - sub["FPR_pct"].min()
        ),
        "Brecha_prevalencia_pp": (
            sub["Prevalencia_pct"].max()
            - sub["Prevalencia_pct"].min()
        )
    })

tabla_brechas_335 = pd.DataFrame(
    filas_brechas_335
)

display(
    tabla_brechas_335.loc[
        tabla_brechas_335["Dimension"].isin(
            ["gender", "disability", "imd_band"]
        )
    ].sort_values(
        ["Dimension", "Configuracion"]
    ).round(2)
)

print("\n=== 3.35.5 SUBGRUPOS DETALLADOS ===")

display(
    por_grupo_335.loc[
        (
            por_grupo_335["Dimension"]
            == "disability"
        )
        &
        (
            por_grupo_335["Configuracion"].isin([
                "LR B | Grid",
                "LR B | Optuna",
                "RF B | Optuna",
                "Transformer C"
            ])
        ),
        [
            "Configuracion",
            "Grupo",
            "Matriculas",
            "Prevalencia_pct",
            "Recall_pct",
            "FPR_pct"
        ]
    ].round(2)
)

# ------------------------------------------------------------
# 5. VISUALIZACIONES
# ------------------------------------------------------------

print("\n=== 3.35.6 GRÁFICOS COMPARATIVOS ===")

# Gráfico 1: F1 macro.
fig, ax = plt.subplots(figsize=(11, 5))

ax.bar(
    resumen_335["Configuracion"],
    resumen_335["F1_macro"]
)

ax.set_title(
    "Comparación de configuraciones - Semana 8"
)
ax.set_ylabel("F1 macro fuera de muestra")
ax.set_ylim(
    0,
    min(1, resumen_335["F1_macro"].max() + 0.12)
)
ax.tick_params(axis="x", rotation=25)
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

# Gráfico 2: intercambio entre sensibilidad
# y falsas alertas.
fig, ax = plt.subplots(figsize=(9, 6))

for _, fila in resumen_335.iterrows():

    ax.scatter(
        fila["FPR_pct"],
        fila["Recall_pct"],
        s=85
    )

    ax.annotate(
        fila["Configuracion"],
        (
            fila["FPR_pct"],
            fila["Recall_pct"]
        ),
        xytext=(5, 5),
        textcoords="offset points"
    )

ax.set_title(
    "Detección frente a falsas alertas - Semana 8"
)

ax.set_xlabel(
    "Tasa de falsos positivos (%)"
)
ax.set_ylabel(
    "Recall adverso (%)"
)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Gráfico 3: calidad de probabilidades.
fig, ax = plt.subplots(figsize=(11, 5))

ax.bar(
    resumen_335["Configuracion"],
    resumen_335["Brier"]
)

ax.set_title(
    "Brier Score por configuración - Semana 8\n"
    "Menor es mejor"
)
ax.set_ylabel("Brier Score")
ax.set_ylim(
    0,
    resumen_335["Brier"].max() * 1.2
)
ax.tick_params(axis="x", rotation=25)
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 6. CONTROLES FINALES
# ------------------------------------------------------------

print("\n=== 3.35.7 CONTROLES FINALES ===")

assert len(resumen_335) == 5

assert metricas_335.groupby(
    "Configuracion"
).size().eq(2).all()

assert predicciones_335.groupby(
    ["Configuracion", "Repeticion"]
).size().eq(21164).all()

assert len(predicciones_335) == (
    5 * 2 * 21164
)

print("Configuraciones evaluadas: 5")
print("Repeticiones: 2")
print("Folds originales: preservados")
print("Predicciones: emparejadas")
print("Métricas de calibración: calculadas")
print("Carga de alertas: calculada")
print("Auditoría descriptiva de subgrupos: calculada")
print("Umbrales modificados: 0")
print("Modelos reentrenados: 0")
print("Conjunto de prueba independiente: NO utilizado")

print("\n=== ACTIVIDAD 3.35 FINALIZADA ===")




### 3.35.8. Conclusión del análisis integral

La Actividad 3.35 permitió comparar cinco configuraciones predictivas sobre las mismas matrículas de desarrollo de la semana 8, utilizando predicciones fuera de muestra emparejadas.

Regresión Logística B con GridSearchCV obtuvo un F1 macro de 0,72289, un ROC-AUC de 0,79955 y un Brier Score de 0,17533, ofreciendo el mejor equilibrio general entre discriminación, calidad probabilística y carga operativa bajo los umbrales originales.

Regresión Logística B con Optuna alcanzó un F1 macro de 0,72332 y el mayor recall adverso, de 63,78 %. Sin embargo, su incremento de sensibilidad estuvo acompañado de una reducción de la precisión y un aumento de la tasa de falsos positivos de 15,35 % a 19,56 % respecto a GridSearchCV.

Random Forest B con Optuna alcanzó un F1 macro de 0,71462 y un recall de 60,47 %, confirmando mejoras respecto a su configuración original. No obstante, presentó una calidad de calibración menos favorable y una mayor carga de alertas que Regresión Logística B con GridSearchCV.

Transformer C obtuvo un F1 macro de 0,70993 y un ROC-AUC de 0,79133, demostrando la viabilidad de representar trayectorias estudiantiles mediante un modelo de atención temporal, aunque sin superar a la mejor configuración clásica.

La auditoría de subgrupos mostró diferencias descriptivas en recall y tasas de falsos positivos asociadas a variables demográficas y académicas. Estas diferencias requieren interpretación contextual y no permiten establecer por sí solas relaciones causales ni conclusiones definitivas de equidad.

Los resultados destacan que la optimización de hiperparámetros y la selección de modelos deben incorporar criterios operativos, además del rendimiento predictivo.

Se propone evaluar los cinco modelos bajo presupuestos idénticos de alertas, utilizando únicamente predicciones fuera de muestra de desarrollo, antes de seleccionar una configuración definitiva.

El conjunto de prueba independiente permaneció reservado.



### 3.36. Comparación de modelos bajo presupuestos equivalentes de alertas

#### Objetivo

Comparar las cinco configuraciones longitudinales de la semana 8 bajo presupuestos idénticos de alertas, controlando el efecto de las diferencias en las tasas de clasificación positiva.

#### Justificación

La comparación original utiliza los umbrales de clasificación de cada modelo. Esto produce diferentes volúmenes de alertas y dificulta distinguir entre una mejor capacidad de priorización y una mayor disposición a emitir alertas.

Se propone evaluar presupuestos hipotéticos del 25 %, 30 %, 35 %, 40 % y 45 % de las matrículas.

Estos porcentajes son escenarios ilustrativos y no representan capacidades institucionales verificadas.

#### Procedimiento

Para cada configuración, repetición y presupuesto:

1. Ordenar las matrículas por probabilidad predicha de resultado adverso, de mayor a menor.
2. Seleccionar exactamente el número de matrículas permitido por el presupuesto.
3. Calcular métricas de clasificación sobre esa selección.
4. Comparar recall adverso, precisión, falsos positivos, F1 macro y detección por subtipo académico.

La etiqueta real se utilizará exclusivamente para evaluar el resultado, nunca para determinar la posición de una matrícula en el ranking.

#### Comparabilidad

Todos los modelos utilizarán las mismas matrículas, el mismo presupuesto y las mismas repeticiones fuera de muestra.

Los conteos se reportarán como promedios por repetición y no como estudiantes independientes adicionales.

#### Precauciones metodológicas

El ranking se construye sobre predicciones externas de los folds de desarrollo. Sus probabilidades proceden de distintos modelos ajustados y pueden presentar diferencias de calibración entre folds.

La comparación será descriptiva y no implicará entrenamiento adicional.

La selección de un presupuesto o configuración a partir de estos resultados constituye una decisión de desarrollo que deberá congelarse antes de evaluar el test independiente.

No se utilizará el conjunto de prueba independiente.


In [ ]:

# ============================================================
# ACTIVIDAD 3.36
# COMPARACIÓN A IGUAL PRESUPUESTO DE ALERTAS
# OULAD - SEMANA 8
# ============================================================

# Asistencia IA: ChatGPT, octubre de 2026.
# Prompt:
# "Comparar cinco configuraciones longitudinales
# mediante top-K por probabilidad adversa, con
# presupuestos idénticos, predicciones OOF,
# métricas operativas y análisis de Fail/Withdrawn.
# No entrenar modelos ni usar el test independiente."

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix
)

from IPython.display import display

SEMANA_336 = 8

PRESUPUESTOS_336 = [
    25, 30, 35, 40, 45
]

PRESUPUESTO_REFERENCIA_336 = 35

CONFIGURACIONES_336 = [
    "LR B | Grid",
    "LR B | Optuna",
    "RF B | Grid",
    "RF B | Optuna",
    "Transformer C"
]

assert "predicciones_335" in globals()
assert "oulad_tablas" in globals()

# ------------------------------------------------------------
# 1. VERIFICACIÓN DE DATOS
# ------------------------------------------------------------

print("=== 3.36.1 CONFIGURACIÓN ===")

base_336 = predicciones_335.copy()

columnas_requeridas_336 = [
    "Configuracion",
    "Repeticion",
    "code_module",
    "code_presentation",
    "id_student",
    "y_real",
    "prob_adverso"
]

assert set(columnas_requeridas_336).issubset(
    base_336.columns
)

assert len(base_336) == 211640

assert base_336["prob_adverso"].between(
    0, 1
).all()

assert base_336["prob_adverso"].notna().all()

assert set(
    base_336["Configuracion"]
) == set(CONFIGURACIONES_336)

assert base_336.groupby(
    ["Configuracion", "Repeticion"]
).size().eq(21164).all()

# ------------------------------------------------------------
# 2. METADATOS PARA EVALUACIÓN DE SUBTIPOS
# ------------------------------------------------------------

claves_matricula_336 = [
    "code_module",
    "code_presentation",
    "id_student"
]

metadatos_336 = oulad_tablas[
    "studentInfo.csv"
][
    claves_matricula_336 + ["final_result"]
].copy()

assert not metadatos_336.duplicated(
    claves_matricula_336
).any()

base_336 = base_336.merge(
    metadatos_336,
    on=claves_matricula_336,
    how="left",
    validate="many_to_one"
)

assert base_336["final_result"].notna().all()

# Comprobar consistencia con el target adverso.
adverso_original_336 = (
    base_336["final_result"].isin(
        ["Fail", "Withdrawn"]
    )
).astype(int)

assert (
    adverso_original_336.to_numpy()
    ==
    base_336["y_real"].to_numpy()
).all()

print("Semana:", SEMANA_336)
print("Presupuestos (%):", PRESUPUESTOS_336)
print("Configuraciones:", len(CONFIGURACIONES_336))
print("Matrículas por repetición: 21164")
print("Metadatos de resultados: verificados")

# ------------------------------------------------------------
# 3. FUNCIÓN DE SELECCIÓN TOP-K
# ------------------------------------------------------------

def seleccionar_top_k_336(
    probabilidades,
    capacidad_pct
):
    """
    Selecciona exactamente K observaciones.

    K se determina solamente por el presupuesto.
    El orden depende únicamente de probabilidades.

    No utiliza etiquetas reales.
    """

    p = np.asarray(
        probabilidades,
        dtype=float
    )

    n = len(p)

    k = int(np.rint(
        n * capacidad_pct / 100
    ))

    k = max(0, min(k, n))

    # Orden estable para resolver empates.
    orden = np.argsort(
        -p,
        kind="stable"
    )

    alertas = np.zeros(
        n, dtype=int
    )

    alertas[orden[:k]] = 1

    assert alertas.sum() == k

    return alertas, k

# ------------------------------------------------------------
# 4. EVALUACIÓN DE TODOS LOS PRESUPUESTOS
# ------------------------------------------------------------

print("\n=== 3.36.2 EVALUACIÓN TOP-K ===")

filas_336 = []

for (
    configuracion,
    repeticion
), sub in base_336.groupby(
    ["Configuracion", "Repeticion"]
):

    # Orden fijo de matrícula para resolver
    # empates de forma reproducible.
    sub = sub.sort_values(
        claves_matricula_336,
        kind="mergesort"
    ).reset_index(drop=True)

    y = sub["y_real"].to_numpy(
        dtype=int
    )

    probabilidades = (
        sub["prob_adverso"].to_numpy(
            dtype=float
        )
    )

    resultado_academico = (
        sub["final_result"].to_numpy()
    )

    n = len(sub)

    assert n == 21164

    for capacidad in PRESUPUESTOS_336:

        pred, k = seleccionar_top_k_336(
            probabilidades,
            capacidad
        )

        vn, fp, fn, vp = confusion_matrix(
            y, pred, labels=[0, 1]
        ).ravel()

        mascara_fail = (
            resultado_academico == "Fail"
        )

        mascara_retiro = (
            resultado_academico == "Withdrawn"
        )

        detectados_fail = int(
            pred[mascara_fail].sum()
        )

        detectados_retiro = int(
            pred[mascara_retiro].sum()
        )

        n_fail = int(
            mascara_fail.sum()
        )

        n_retiro = int(
            mascara_retiro.sum()
        )

        filas_336.append({
            "Configuracion": configuracion,
            "Repeticion": int(repeticion),
            "Capacidad_pct": capacidad,
            "Matriculas": n,
            "Alertas": k,
            "Alertadas_pct": 100 * k / n,
            "F1_macro": f1_score(
                y, pred, average="macro"
            ),
            "Recall_adverso_pct": (
                100 * recall_score(
                    y, pred, zero_division=0
                )
            ),
            "Precision_pct": (
                100 * precision_score(
                    y, pred, zero_division=0
                )
            ),
            "FPR_pct": (
                100 * fp / max(fp + vn, 1)
            ),
            "VP": int(vp),
            "FP": int(fp),
            "FN": int(fn),
            "VN": int(vn),
            "Fail_detectados": detectados_fail,
            "Withdrawn_detectados": (
                detectados_retiro
            ),
            "Recall_Fail_pct": (
                100 * detectados_fail
                / max(n_fail, 1)
            ),
            "Recall_Withdrawn_pct": (
                100 * detectados_retiro
                / max(n_retiro, 1)
            )
        })

tabla_presupuestos_336 = pd.DataFrame(
    filas_336
)

assert len(tabla_presupuestos_336) == (
    5 * 2 * len(PRESUPUESTOS_336)
)

# ------------------------------------------------------------
# 5. RESULTADOS MEDIOS POR CONFIGURACIÓN
# ------------------------------------------------------------

resumen_presupuestos_336 = (
    tabla_presupuestos_336.groupby(
        ["Capacidad_pct", "Configuracion"]
    )
    .agg(
        Alertas=("Alertas", "mean"),
        F1_macro=("F1_macro", "mean"),
        Recall_adverso_pct=(
            "Recall_adverso_pct", "mean"
        ),
        Precision_pct=("Precision_pct", "mean"),
        FPR_pct=("FPR_pct", "mean"),
        VP=("VP", "mean"),
        FP=("FP", "mean"),
        Recall_Fail_pct=(
            "Recall_Fail_pct", "mean"
        ),
        Recall_Withdrawn_pct=(
            "Recall_Withdrawn_pct", "mean"
        )
    )
    .reset_index()
)

print(
    "\n=== 3.36.3 RESULTADOS CON "
    "PRESUPUESTO DE 35 % ==="
)

tabla_35_336 = (
    resumen_presupuestos_336.loc[
        resumen_presupuestos_336[
            "Capacidad_pct"
        ].eq(PRESUPUESTO_REFERENCIA_336)
    ]
    .set_index("Configuracion")
    .reindex(CONFIGURACIONES_336)
    .reset_index()
)

display(
    tabla_35_336.round(3)
)

# ------------------------------------------------------------
# 6. COMPARACIÓN CONTRA LR B GRID
# ------------------------------------------------------------

print(
    "\n=== 3.36.4 DIFERENCIAS A IGUAL CAPACIDAD ==="
)

comparaciones_336 = []

for capacidad in PRESUPUESTOS_336:

    sub = resumen_presupuestos_336.loc[
        resumen_presupuestos_336[
            "Capacidad_pct"
        ].eq(capacidad)
    ]

    ref = sub.loc[
        sub["Configuracion"].eq(
            "LR B | Grid"
        )
    ].iloc[0]

    for _, fila in sub.iterrows():

        comparaciones_336.append({
            "Capacidad_pct": capacidad,
            "Configuracion": fila[
                "Configuracion"
            ],
            "Delta_VP_vs_LR_Grid": (
                fila["VP"] - ref["VP"]
            ),
            "Delta_FP_vs_LR_Grid": (
                fila["FP"] - ref["FP"]
            ),
            "Delta_F1_vs_LR_Grid": (
                fila["F1_macro"]
                - ref["F1_macro"]
            ),
            "Delta_recall_pp": (
                fila["Recall_adverso_pct"]
                - ref["Recall_adverso_pct"]
            ),
            "Delta_precision_pp": (
                fila["Precision_pct"]
                - ref["Precision_pct"]
            )
        })

comparacion_equidad_336 = pd.DataFrame(
    comparaciones_336
)

display(
    comparacion_equidad_336.loc[
        comparacion_equidad_336[
            "Capacidad_pct"
        ].eq(35)
    ].round(4)
)

# Al tener el mismo número de alertas,
# cualquier aumento de VP debe corresponder
# exactamente a una reducción de FP.
assert np.allclose(
    comparacion_equidad_336[
        "Delta_VP_vs_LR_Grid"
    ].to_numpy()
    +
    comparacion_equidad_336[
        "Delta_FP_vs_LR_Grid"
    ].to_numpy(),
    0,
    atol=1e-8
)

# ------------------------------------------------------------
# 7. RESUMEN DE TODAS LAS CAPACIDADES
# ------------------------------------------------------------

print(
    "\n=== 3.36.5 EVOLUCIÓN "
    "SEGÚN CAPACIDAD ==="
)

display(
    resumen_presupuestos_336[
        [
            "Capacidad_pct",
            "Configuracion",
            "Recall_adverso_pct",
            "Precision_pct",
            "F1_macro",
            "VP",
            "FP"
        ]
    ].round(3)
)

# ------------------------------------------------------------
# 8. GRÁFICOS
# ------------------------------------------------------------

print("\n=== 3.36.6 GRÁFICOS ===")

fig, ax = plt.subplots(figsize=(10, 5))

for configuracion in CONFIGURACIONES_336:

    sub = resumen_presupuestos_336.loc[
        resumen_presupuestos_336[
            "Configuracion"
        ].eq(configuracion)
    ].sort_values("Capacidad_pct")

    ax.plot(
        sub["Capacidad_pct"],
        sub["Recall_adverso_pct"],
        marker="o",
        linewidth=2,
        label=configuracion
    )

ax.set_title(
    "Recall adverso bajo presupuestos equivalentes"
)

ax.set_xlabel(
    "Porcentaje de matrículas alertadas (%)"
)

ax.set_ylabel(
    "Recall adverso (%)"
)

ax.grid(alpha=0.3)
ax.legend()

plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(10, 5))

for configuracion in CONFIGURACIONES_336:

    sub = resumen_presupuestos_336.loc[
        resumen_presupuestos_336[
            "Configuracion"
        ].eq(configuracion)
    ].sort_values("Capacidad_pct")

    ax.plot(
        sub["Capacidad_pct"],
        sub["Precision_pct"],
        marker="o",
        linewidth=2,
        label=configuracion
    )

ax.set_title(
    "Precisión de alertas a igual presupuesto"
)

ax.set_xlabel(
    "Porcentaje de matrículas alertadas (%)"
)

ax.set_ylabel(
    "Precisión de alertas (%)"
)

ax.grid(alpha=0.3)
ax.legend()

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 9. CONTROL FINAL
# ------------------------------------------------------------

print("\n=== 3.36.7 CONTROLES FINALES ===")

assert len(CONFIGURACIONES_336) == 5
assert len(PRESUPUESTOS_336) == 5

for capacidad, sub in tabla_presupuestos_336.groupby(
    "Capacidad_pct"
):

    # Todos deben generar exactamente K alertas.
    assert sub["Alertas"].nunique() == 1

    assert np.all(
        sub["VP"] + sub["FP"]
        ==
        sub["Alertas"]
    )

    assert np.all(
        sub["VN"] + sub["FP"]
        + sub["FN"] + sub["VP"]
        ==
        sub["Matriculas"]
    )

print("Configuraciones evaluadas: 5")
print("Presupuestos evaluados: 5")
print("Repeticiones: 2")
print("Mismas matrículas: SÍ")
print("Mismo número de alertas por presupuesto: SÍ")
print("Ranking basado solo en probabilidades")
print("Resultados finales usados solo para evaluación")
print("Umbrales originales: sin modificaciones")
print("Modelos reentrenados: 0")
print("Test independiente: NO utilizado")

print("\n=== ACTIVIDAD 3.36 FINALIZADA ===")



### 3.36.8. Conclusión: comparación bajo presupuestos equivalentes de alertas

La Actividad 3.36 comparó cinco configuraciones longitudinales mediante selección top-K, utilizando presupuestos idénticos del 25 %, 30 %, 35 %, 40 % y 45 % de las matrículas.

El objetivo fue controlar las diferencias en el volumen de alertas y evaluar la capacidad de cada modelo para priorizar correctamente los casos de riesgo académico.

En el escenario ilustrativo del 35 %, cada modelo generó 7.407 alertas sobre 21.164 matrículas de desarrollo.

Regresión Logística B con GridSearchCV obtuvo el mayor recall adverso, con 60,58 %, y la mayor precisión, con 72,42 %. En promedio por repetición, identificó 5.364 verdaderos positivos y produjo 2.043 falsos positivos.

Bajo el mismo presupuesto, Regresión Logística B con Optuna identificó 24 casos adversos menos y produjo 24 falsas alertas adicionales.

Random Forest B con Optuna y Transformer C alcanzaron resultados competitivos, pero inferiores a Regresión Logística B con GridSearchCV en el escenario del 35 %.

La tendencia se mantuvo en los presupuestos del 30 %, 40 % y 45 %. En el presupuesto del 25 %, Optuna obtuvo una ventaja mínima para Regresión Logística.

Estos resultados indican que el mayor recall observado anteriormente con los umbrales originales de Optuna se debe, al menos parcialmente, a una mayor cantidad de matrículas clasificadas como adversas.

La comparación top-K permite distinguir entre sensibilidad obtenida mediante un mayor volumen de alertas y capacidad efectiva de priorización bajo recursos limitados.

No obstante, las diferencias entre configuraciones son descriptivas y no se han realizado contrastes de significancia específicos para sus rankings.

Los presupuestos considerados son hipotéticos, no capacidades institucionales verificadas.

En consecuencia, Regresión Logística B con GridSearchCV se considera la configuración candidata principal para la evaluación independiente, debido a su desempeño predictivo, calibración probabilística y resultados a igual capacidad.

La decisión deberá documentarse antes de acceder al conjunto de prueba independiente.



### 3.37. Selección de configuración y congelación del protocolo de evaluación independiente

#### Objetivo

Seleccionar y documentar la configuración principal del proyecto antes de realizar el entrenamiento definitivo y acceder al conjunto de prueba independiente.

#### Justificación de la selección

Regresión Logística longitudinal con GridSearchCV fue seleccionada como configuración principal por presentar un rendimiento competitivo en F1 macro, el mayor ROC-AUC y el menor Brier Score entre los cinco candidatos de la semana 8.

Adicionalmente, obtuvo el mejor resultado de priorización bajo presupuestos idénticos de alertas del 30 %, 35 %, 40 % y 45 %.

La diferencia de F1 macro frente a Regresión Logística optimizada con Optuna fue prácticamente nula, mientras que esta última presentó una mayor tasa de falsos positivos y un mayor error de calibración esperado.

La selección corresponde a una decisión metodológica basada exclusivamente en datos de desarrollo, no a una demostración de superioridad universal.

#### Hipótesis principal

La incorporación de información longitudinal sobre el comportamiento académico y la interacción con el entorno virtual permite mejorar la predicción temprana de resultados académicos adversos frente a un modelo que utiliza únicamente información estática.

#### Protocolo principal

Se entrenarán los modelos de Regresión Logística A y B para las semanas 2, 4, 6 y 8.

La selección de hiperparámetros se realizará mediante validación cruzada agrupada por estudiante, utilizando exclusivamente el conjunto de desarrollo.

El preprocesamiento y la selección de características permanecerán dentro de los pipelines de scikit-learn.

Una vez elegidos los hiperparámetros, los modelos se reajustarán sobre la totalidad de sus correspondientes cohortes de desarrollo.

Posteriormente se realizará una única evaluación independiente en cada ventana temporal, utilizando las particiones de prueba ya reservadas.

#### Métrica y umbral

La métrica primaria será F1 macro, utilizando el umbral de clasificación estándar de los modelos.

Se calcularán adicionalmente recall adverso, ROC-AUC, precisión, accuracy y Brier Score.

La selección top-K se conservará como análisis operativo complementario bajo los presupuestos ilustrativos del 25 %, 30 %, 35 %, 40 % y 45 %.

Ninguno de estos presupuestos representa una capacidad institucional real confirmada.

#### Análisis adicionales

Se examinarán retrospectivamente los resultados Fail y Withdrawn, incluyendo la anticipación de retiros cuando exista una fecha registrada.

Se describirán las diferencias de rendimiento entre subgrupos académicos y demográficos, sin atribuirles causalidad.

Optuna y Transformer C se conservarán como experimentos complementarios cuyos resultados de desarrollo ya están documentados.

#### Condiciones de validez

No se modificará el protocolo en función de los resultados del test.

No se reajustarán los umbrales utilizando el test.

No se utilizarán fechas de retiro, resultados finales ni información posterior al corte temporal como características predictivas.

La separación por estudiante permanecerá inalterada.

La selección de una configuración basada en su rendimiento de desarrollo puede introducir optimismo en las estimaciones de esa fase. Por ello, la evaluación independiente será la referencia principal para el rendimiento final.


In [ ]:
# Idempotencia: respetar el protocolo ya congelado.
from pathlib import Path
import hashlib
_protocolo_existente_337=Path('/content/protocolo_oulad_337.json')
if _protocolo_existente_337.exists():
    assert hashlib.sha256(_protocolo_existente_337.read_bytes()).hexdigest()==HASH_PROTOCOLO_337_ESPERADO
    print('=== 3.37 OMITIDA: protocolo existente con SHA correcto ===')
else:

    # ============================================================
    # ACTIVIDAD 3.37
    # CONGELACIÓN DEL PROTOCOLO Y AUDITORÍA PRE-TEST
    # ============================================================

    import json
    import hashlib
    from pathlib import Path
    import pandas as pd
    import numpy as np

    assert "resumen_335" in globals()
    assert "resumen_presupuestos_336" in globals()
    assert "experimentos_oulad" in globals()
    assert "cv_splits_oulad" in globals()

    SEMANAS_337 = [2, 4, 6, 8]

    protocolo_oulad_337 = {
        "actividad": "3.37",
        "estado": "protocolo_congelado_pre_test",
        "modelo_principal": "LogisticRegression",
        "experimento_principal": "B",
        "optimizador_principal": "GridSearchCV",
        "comparacion_principal": {
            "modelo_A": "LogisticRegression",
            "modelo_B": "LogisticRegression",
            "ventanas": SEMANAS_337,
            "metrica": "f1_macro"
        },
        "grupo_separacion": "id_student",
        "cv_interna_folds": 3,
        "grid_logistic_regression": {
            "modelo__C": [0.1, 1.0],
            "seleccion__percentile": [80, 100]
        },
        "semilla": 42,
        "umbral_clasificacion": 0.5,
        "presupuestos_top_k_exploratorios": [
            25, 30, 35, 40, 45
        ],
        "metricas_secundarias": [
            "recall_adverso",
            "precision_adversa",
            "roc_auc",
            "accuracy",
            "brier"
        ],
        "experimentos_complementarios": [
            "RandomForest",
            "Optuna",
            "Transformer_C"
        ],
        "test_independiente": "reservado",
        "permitir_ajustes_usando_test": False
    }

    # ------------------------------------------------------------
    # 1. VERIFICACIÓN DE RESULTADOS DE DESARROLLO
    # ------------------------------------------------------------

    print("=== 3.37.1 DECISIÓN DOCUMENTADA ===")

    tabla_candidatos_337 = (
        resumen_335.set_index("Configuracion")
    )

    principal_337 = tabla_candidatos_337.loc[
        "LR B | Grid"
    ]

    print("Modelo principal: LogisticRegression B")
    print("Optimización: GridSearchCV")
    print("Ventanas:", SEMANAS_337)

    print(
        "F1 macro en desarrollo (semana 8):",
        round(float(principal_337["F1_macro"]), 5)
    )

    print(
        "ROC-AUC en desarrollo:",
        round(float(principal_337["ROC_AUC"]), 5)
    )

    print(
        "Brier Score:",
        round(float(principal_337["Brier"]), 5)
    )

    # ------------------------------------------------------------
    # 2. AUDITORÍA DE PRESUPUESTOS
    # ------------------------------------------------------------

    print("\n=== 3.37.2 PRESUPUESTOS EVALUADOS ===")

    columnas_337 = [
        "Capacidad_pct",
        "Configuracion",
        "Recall_adverso_pct",
        "Precision_pct",
        "VP",
        "FP"
    ]

    tabla_capacidades_337 = (
        resumen_presupuestos_336[columnas_337]
        .copy()
    )

    display(
        tabla_capacidades_337.loc[
            tabla_capacidades_337[
                "Capacidad_pct"
            ].eq(35)
        ].round(3)
    )

    # No imponer aquí un presupuesto institucional
    # definitivo: no se dispone de su capacidad real.

    # ------------------------------------------------------------
    # 3. AUDITORÍA DE OBJETOS DE ENTRENAMIENTO
    # ------------------------------------------------------------

    print("\n=== 3.37.3 OBJETOS DE ENTRENAMIENTO ===")

    for semana in SEMANAS_337:

        assert semana in experimentos_oulad
        assert semana in cv_splits_oulad

        experimento = experimentos_oulad[semana]

        print(f"\nSEMANA {semana}")
        print("Tipo de experimento:", type(experimento).__name__)

        if isinstance(experimento, dict):

            print("Claves disponibles:")

            for clave, valor in experimento.items():

                forma = getattr(
                    valor, "shape", None
                )

                if forma is not None:
                    print(
                        "  ",
                        clave,
                        "| dimensiones:",
                        forma
                    )
                elif isinstance(valor, dict):
                    print(
                        "  ",
                        clave,
                        "| diccionario:",
                        list(valor.keys())[:15]
                    )
                else:
                    print(
                        "  ",
                        clave,
                        "| tipo:",
                        type(valor).__name__
                    )

        print(
            "Folds externos almacenados:",
            len(cv_splits_oulad[semana])
        )

    # Identificar objetos de pipelines que podrían
    # contener las configuraciones originales.
    print("\n=== 3.37.4 PIPELINES DISPONIBLES ===")

    nombres_pipelines_337 = sorted([
        nombre
        for nombre in globals()
        if (
            "pipeline" in nombre.lower()
            or "modelo_oulad" in nombre.lower()
        )
        and not nombre.startswith("_")
    ])

    print(
        "Variables candidatas:",
        nombres_pipelines_337[:35]
    )

    # ------------------------------------------------------------
    # 4. GUARDAR EL PROTOCOLO
    # ------------------------------------------------------------

    print("\n=== 3.37.5 REGISTRO REPRODUCIBLE ===")

    contenido_337 = json.dumps(
        protocolo_oulad_337,
        sort_keys=True,
        indent=2,
        ensure_ascii=False
    )

    huella_337 = hashlib.sha256(
        contenido_337.encode("utf-8")
    ).hexdigest()

    ruta_337 = Path(
        "/content/protocolo_oulad_337.json"
    )

    ruta_337.write_text(
        contenido_337,
        encoding="utf-8"
    )

    # Verificar lectura y coincidencia.
    contenido_recuperado_337 = (
        ruta_337.read_text(encoding="utf-8")
    )

    assert contenido_recuperado_337 == contenido_337

    assert (
        hashlib.sha256(
            contenido_recuperado_337.encode("utf-8")
        ).hexdigest()
        == huella_337
    )

    print("Archivo:", ruta_337)
    print("SHA-256:", huella_337)

    # ------------------------------------------------------------
    # 5. CONTROLES FINALES
    # ------------------------------------------------------------

    print("\n=== 3.37.6 CONTROLES FINALES ===")

    assert len(SEMANAS_337) == 4

    assert (
        protocolo_oulad_337[
            "permitir_ajustes_usando_test"
        ] is False
    )

    print("Modelo principal: documentado")
    print("Comparación A vs B: conservada")
    print("Cuatro ventanas temporales: conservadas")
    print("Grid de hiperparámetros: congelado")
    print("Protocolo JSON: guardado y verificado")
    print("Conjunto de prueba: NO utilizado")
    print("Modelos entrenados nuevamente: 0")

    print("\n=== ACTIVIDAD 3.37 FINALIZADA ===")



### 3.38. Selección final de hiperparámetros y entrenamiento sobre el conjunto de desarrollo

#### Objetivo

Entrenar los modelos definitivos de Regresión Logística para los experimentos A y B en las semanas 2, 4, 6 y 8, utilizando exclusivamente las matrículas de desarrollo y respetando el protocolo establecido en la Actividad 3.37.

#### Diseño experimental

- **Experimento A:** características estáticas disponibles en el momento de predicción.
- **Experimento B:** características estáticas y comportamiento longitudinal disponible hasta el corte temporal.
- **Algoritmo:** Regresión Logística.
- **Métrica de optimización:** F1 macro.
- **Optimización:** GridSearchCV.
- **Validación interna:** StratifiedGroupKFold con tres particiones, agrupadas por `id_student`.
- **Espacio de búsqueda:** parámetro `C` igual a 0,1 o 1,0; selección del 80 % o 100 % de las características no constantes.

El preprocesamiento, imputación, codificación, escalamiento y selección de variables se conservarán dentro de los pipelines originales. Sus parámetros se ajustarán exclusivamente sobre los datos de entrenamiento correspondientes a cada partición.

#### Procedimiento

1. Verificar la integridad del protocolo congelado en la Actividad 3.37.
2. Recuperar los pipelines originales y comprobar que coincidan con el algoritmo y los hiperparámetros previstos.
3. Construir tres folds internos agrupados por estudiante para cada semana.
4. Optimizar A y B sobre exactamente los mismos folds internos.
5. Reajustar automáticamente cada mejor configuración sobre todas las matrículas de desarrollo de la semana correspondiente.
6. Registrar hiperparámetros, puntuaciones internas, tiempos y número de características seleccionadas.
7. Guardar los ocho pipelines ajustados en un checkpoint reproducible.

#### Interpretación

Las puntuaciones de validación interna se utilizarán para seleccionar hiperparámetros y auditar el procedimiento de entrenamiento. No sustituirán las estimaciones obtenidas anteriormente mediante validación cruzada anidada.

No se calcularán métricas sobre el conjunto de prueba en esta actividad.

La evaluación independiente se realizará en una actividad posterior, una vez comprobada la integridad de los ocho modelos y del protocolo congelado.


In [ ]:
# Protección de modelos definitivos: no volver a ajustar si existen ocho.
from pathlib import Path
import joblib
import pandas as pd
_ruta_338=Path('/content/modelos_finales_oulad_338.joblib')
if _ruta_338.exists() and _modelo_338_completo(_ruta_338):
    modelos_finales_oulad_338=joblib.load(_ruta_338)
    _csv_338=Path('/content/resultados_entrenamiento_338.csv')
    resultados_entrenamiento_338=(pd.read_csv(_csv_338) if _csv_338.exists() else None)
    print('=== 3.38 REUTILIZADA: ocho modelos definitivos ya guardados ===')
    print('No se reentrena ni se modifica el checkpoint.')
else:

    # ============================================================
    # ACTIVIDAD 3.38
    # ENTRENAMIENTO DEFINITIVO - EXPERIMENTOS A Y B
    #
    # Regresión Logística
    # Semanas 2, 4, 6 y 8
    # GridSearchCV agrupado por estudiante
    #
    # No utiliza los conjuntos de prueba.
    # No modifica el protocolo de la Actividad 3.37.
    # ============================================================

    # Asistencia IA: ChatGPT, octubre de 2026.
    # Prompt:
    # "Entrenar los ocho modelos finales A/B de OULAD
    # utilizando los pipelines originales, GridSearchCV
    # con StratifiedGroupKFold y F1 macro, respetando
    # el protocolo congelado y sin evaluar el test."

    import re
    import json
    import time
    import hashlib
    import joblib
    import numpy as np
    import pandas as pd

    from pathlib import Path
    from sklearn.base import clone
    from sklearn.pipeline import Pipeline
    from sklearn.model_selection import (
        StratifiedGroupKFold,
        GridSearchCV
    )

    from IPython.display import display

    # ------------------------------------------------------------
    # 1. CONFIGURACIÓN CONGELADA
    # ------------------------------------------------------------

    SEMANAS_338 = [2, 4, 6, 8]
    EXPERIMENTOS_338 = ["A", "B"]

    GRID_338 = {
        "modelo__C": [0.1, 1.0],
        "seleccion__percentile": [80, 100]
    }

    N_FOLDS_338 = 3
    SEMILLA_338 = 42

    RUTA_PROTOCOLO_338 = Path(
        "/content/protocolo_oulad_337.json"
    )

    HASH_CONGELADO_338 = (
        "283bbddb0c56a615dde6a8a1201e149dfd7020f4672d7d18267e4c93ad880625"
    )

    RUTA_MODELOS_338 = Path(
        "/content/modelos_finales_oulad_338.joblib"
    )

    RUTA_RESULTADOS_338 = Path(
        "/content/resultados_entrenamiento_338.csv"
    )

    assert "experimentos_oulad" in globals()
    assert "pipelines_oulad" in globals()

    assert RUTA_PROTOCOLO_338.exists(), (
        "No se encuentra el protocolo de la Actividad 3.37."
    )

    hash_actual = hashlib.sha256(
        RUTA_PROTOCOLO_338.read_bytes()
    ).hexdigest()

    assert hash_actual == HASH_CONGELADO_338, (
        "El archivo del protocolo no coincide "
        "con la versión congelada en 3.37."
    )

    with open(
        RUTA_PROTOCOLO_338,
        "r",
        encoding="utf-8"
    ) as archivo:
        protocolo_338 = json.load(archivo)

    assert protocolo_338[
        "modelo_principal"
    ] == "LogisticRegression"

    assert protocolo_338[
        "comparacion_principal"
    ]["ventanas"] == SEMANAS_338

    assert protocolo_338[
        "grid_logistic_regression"
    ] == GRID_338

    assert protocolo_338[
        "permitir_ajustes_usando_test"
    ] is False

    print("=== 3.38.1 PROTOCOLO CONGELADO ===")

    print("SHA-256:", hash_actual)
    print("Semanas:", SEMANAS_338)
    print("Experimentos:", EXPERIMENTOS_338)
    print("CV interna:", N_FOLDS_338)
    print("Grid:", GRID_338)
    print("Protocolo: VERIFICADO")

    # ------------------------------------------------------------
    # 2. LOCALIZAR PIPELINES ORIGINALES
    # ------------------------------------------------------------

    print("\n=== 3.38.2 PIPELINES ORIGINALES ===")

    def buscar_pipelines_338(objeto, ruta=(), nivel=0):
        """
        Explora diccionarios anidados y recupera
        objetos Pipeline, junto con sus claves.
        """

        encontrados = []

        if isinstance(objeto, Pipeline):
            encontrados.append((ruta, objeto))

        elif isinstance(objeto, dict) and nivel < 8:

            for clave, valor in objeto.items():

                encontrados.extend(
                    buscar_pipelines_338(
                        valor,
                        ruta + (clave,),
                        nivel + 1
                    )
                )

        return encontrados


    def tokens_ruta_338(ruta):
        """
        Convierte rutas como:
        (8, 'B', 'LogisticRegression')
        o 'semana_8_B_LogisticRegression'
        en tokens comparables.
        """

        partes = []

        def agregar(elemento):

            if isinstance(
                elemento,
                (tuple, list)
            ):
                for valor in elemento:
                    agregar(valor)
            else:
                partes.extend(
                    re.findall(
                        r"[a-z0-9]+",
                        str(elemento).lower()
                    )
                )

        agregar(ruta)
        return partes


    candidatos_338 = buscar_pipelines_338(
        pipelines_oulad
    )

    print(
        "Pipelines encontrados:",
        len(candidatos_338)
    )

    print("\nEjemplos de identificadores:")

    for ruta, objeto in candidatos_338[:8]:
        print("  ", ruta)

    assert candidatos_338, (
        "No se encontraron objetos Pipeline "
        "en pipelines_oulad."
    )


    def localizar_pipeline_338(semana, experimento):

        coincidencias = []

        for ruta, pipeline in candidatos_338:

            tokens = tokens_ruta_338(ruta)

            coincide_semana = (
                str(semana) in tokens
            )

            coincide_experimento = (
                experimento.lower() in tokens
            )

            coincide_logistica = (
                "logisticregression" in tokens
                or "logistic" in tokens
                or "lr" in tokens
            )

            if (
                coincide_semana
                and coincide_experimento
                and coincide_logistica
            ):
                coincidencias.append(
                    (ruta, pipeline)
                )

        if len(coincidencias) != 1:

            print(
                "\nNo se identificó un pipeline "
                "único para:",
                semana,
                experimento
            )

            print("Rutas disponibles:")

            for ruta, _ in candidatos_338:
                print(" ", ruta)

            raise ValueError(
                "Se requiere verificar la estructura "
                "original de pipelines_oulad antes "
                "de continuar. No se ha sustituido "
                "ningún pipeline."
            )

        ruta, original = coincidencias[0]

        # clone() evita reutilizar modelos
        # o transformaciones previamente ajustadas.
        copia = clone(original)

        parametros = copia.get_params()

        requeridos = set(
            GRID_338.keys()
        )

        assert requeridos.issubset(
            parametros.keys()
        ), (
            "El pipeline original no dispone "
            "de los parámetros congelados."
        )

        assert {
            "preprocesamiento",
            "filtro_varianza",
            "seleccion",
            "modelo"
        }.issubset(
            copia.named_steps.keys()
        ), (
            "El pipeline no coincide con "
            "las etapas auditadas en 3.6."
        )

        modelo = copia.named_steps["modelo"]

        assert (
            type(modelo).__name__
            == "LogisticRegression"
        ), (
            "El pipeline seleccionado no "
            "corresponde a Regresión Logística."
        )

        return copia, ruta


    # Comprobar los ocho pipelines ANTES
    # de iniciar cualquier entrenamiento.

    plantillas_338 = {}
    rutas_pipelines_338 = {}

    for semana in SEMANAS_338:

        for experimento in EXPERIMENTOS_338:

            pipeline, ruta = localizar_pipeline_338(
                semana,
                experimento
            )

            clave = (semana, experimento)

            plantillas_338[clave] = pipeline
            rutas_pipelines_338[clave] = ruta

            print(
                f"Semana {semana} | {experimento}: "
                f"{ruta}"
            )

    assert len(plantillas_338) == 8

    print("\nPipelines verificados: 8")

    # ------------------------------------------------------------
    # 3. PREPARAR Y AUDITAR LA CV INTERNA
    # ------------------------------------------------------------

    print("\n=== 3.38.3 AUDITORÍA DE FOLDS ===")

    splits_internos_338 = {}
    filas_folds_338 = []

    for semana in SEMANAS_338:

        datos = experimentos_oulad[semana]

        X_A = datos["X_A_train"]
        X_B = datos["X_B_train"]

        y = np.asarray(
            datos["y_train"],
            dtype=int
        ).ravel()

        grupos = np.asarray(
            datos["grupos_train"]
        ).ravel()

        n = len(y)

        assert len(X_A) == n
        assert len(X_B) == n
        assert len(grupos) == n

        assert set(np.unique(y)) == {0, 1}

        assert not pd.isna(grupos).any()

        assert X_A.columns.is_unique
        assert X_B.columns.is_unique

        # Las matrices A y B deben compartir
        # exactamente filas, etiquetas y grupos.
        cv = StratifiedGroupKFold(
            n_splits=N_FOLDS_338,
            shuffle=True,
            random_state=SEMILLA_338
        )

        splits = list(
            cv.split(
                X_A,
                y,
                groups=grupos
            )
        )

        assert len(splits) == N_FOLDS_338

        vistos = []

        for numero, (
            idx_train,
            idx_val
        ) in enumerate(splits, start=1):

            grupos_train = set(
                grupos[idx_train]
            )

            grupos_val = set(
                grupos[idx_val]
            )

            compartidos = len(
                grupos_train.intersection(
                    grupos_val
                )
            )

            assert compartidos == 0

            assert (
                set(np.unique(y[idx_train]))
                == {0, 1}
            )

            assert (
                set(np.unique(y[idx_val]))
                == {0, 1}
            )

            vistos.extend(idx_val.tolist())

            filas_folds_338.append({
                "Semana": semana,
                "Fold": numero,
                "Matriculas_train": len(idx_train),
                "Matriculas_val": len(idx_val),
                "Estudiantes_train": len(
                    grupos_train
                ),
                "Estudiantes_val": len(
                    grupos_val
                ),
                "Adversos_val_pct": (
                    100 * y[idx_val].mean()
                ),
                "Estudiantes_compartidos": compartidos
            })

        # Todos los registros se validan exactamente
        # una vez en una CV de tres folds.
        assert len(vistos) == n
        assert len(set(vistos)) == n
        assert set(vistos) == set(range(n))

        splits_internos_338[semana] = splits

    tabla_folds_338 = pd.DataFrame(
        filas_folds_338
    )

    display(
        tabla_folds_338.round(2)
    )

    assert tabla_folds_338[
        "Estudiantes_compartidos"
    ].max() == 0

    # ------------------------------------------------------------
    # 4. ENTRENAMIENTO DEFINITIVO
    # ------------------------------------------------------------

    print("\n=== 3.38.4 ENTRENAMIENTO ===")

    # Restaurar checkpoint parcial si había entrenamientos interrumpidos.
    if RUTA_MODELOS_338.exists():
        modelos_finales_oulad_338 = joblib.load(RUTA_MODELOS_338)
    else:
        modelos_finales_oulad_338 = {}
    if RUTA_RESULTADOS_338.exists():
        filas_resultados_338 = pd.read_csv(RUTA_RESULTADOS_338).to_dict('records')
    else:
        filas_resultados_338 = []

    for semana in SEMANAS_338:

        datos = experimentos_oulad[semana]

        y = np.asarray(
            datos["y_train"],
            dtype=int
        ).ravel()

        grupos = np.asarray(
            datos["grupos_train"]
        ).ravel()

        cv_fija = splits_internos_338[semana]

        for experimento in EXPERIMENTOS_338:

            if (semana, experimento) in modelos_finales_oulad_338:
                print(f'[Checkpoint] Semana {semana} {experimento} ya entrenada; se reutiliza.')
                continue

            X = datos[
                f"X_{experimento}_train"
            ]

            plantilla = plantillas_338[
                (semana, experimento)
            ]

            print("\n" + "=" * 60)

            print(
                f"SEMANA {semana} | "
                f"EXPERIMENTO {experimento}"
            )

            print("=" * 60)

            print("Matrículas:", len(X))
            print("Variables originales:", X.shape[1])
            print(
                "Estudiantes únicos:",
                pd.Series(grupos).nunique()
            )

            inicio = time.perf_counter()

            busqueda = GridSearchCV(
                estimator=clone(plantilla),
                param_grid=GRID_338,
                scoring="f1_macro",
                cv=cv_fija,
                refit=True,
                return_train_score=True,
                n_jobs=1,
                error_score="raise"
            )

            # Solo se utilizan datos de desarrollo.
            # refit=True reajusta automáticamente
            # la mejor configuración sobre todo X.
            busqueda.fit(
                X,
                y,
                groups=grupos
            )

            duracion = (
                time.perf_counter() - inicio
            )

            mejor = busqueda.best_estimator_

            mejor_indice = busqueda.best_index_

            media_train = float(
                busqueda.cv_results_[
                    "mean_train_score"
                ][mejor_indice]
            )

            media_cv = float(
                busqueda.best_score_
            )

            desv_cv = float(
                busqueda.cv_results_[
                    "std_test_score"
                ][mejor_indice]
            )

            seleccion = mejor.named_steps[
                "seleccion"
            ]

            n_seleccionadas = int(
                seleccion.get_support().sum()
            )

            # Control de predicción en entrenamiento.
            # No es una estimación fuera de muestra.
            pred_train = mejor.predict(X)

            from sklearn.metrics import f1_score

            f1_reajuste = float(
                f1_score(
                    y,
                    pred_train,
                    average="macro"
                )
            )

            clave = (semana, experimento)

            modelos_finales_oulad_338[clave] = mejor

            fila = {
                "Semana": semana,
                "Experimento": experimento,
                "Matriculas": len(X),
                "Estudiantes": pd.Series(
                    grupos
                ).nunique(),
                "Variables_originales": X.shape[1],
                "Variables_seleccionadas": (
                    n_seleccionadas
                ),
                "C_optimo": (
                    busqueda.best_params_[
                        "modelo__C"
                    ]
                ),
                "Percentile_optimo": (
                    busqueda.best_params_[
                        "seleccion__percentile"
                    ]
                ),
                "F1_CV_interna": media_cv,
                "F1_CV_std": desv_cv,
                "F1_train_CV": media_train,
                "Brecha_CV": (
                    media_train - media_cv
                ),
                "F1_reajuste_train": f1_reajuste,
                "Tiempo_segundos": duracion
            }

            filas_resultados_338.append(fila)

            print(
                "Mejores hiperparámetros:",
                busqueda.best_params_
            )

            print(
                "F1 macro CV interna:",
                round(media_cv, 5)
            )

            print(
                "Desviación CV:",
                round(desv_cv, 5)
            )

            print(
                "F1 train CV:",
                round(media_train, 5)
            )

            print(
                "Características seleccionadas:",
                n_seleccionadas
            )

            print(
                "F1 de reajuste en train:",
                round(f1_reajuste, 5),
                "(optimista; no utilizar como test)"
            )

            print(
                "Tiempo:",
                round(duracion, 1),
                "segundos"
            )

            # Guardado incremental después de cada ajuste.
            joblib.dump(
                modelos_finales_oulad_338,
                RUTA_MODELOS_338
            )

            pd.DataFrame(
                filas_resultados_338
            ).to_csv(
                RUTA_RESULTADOS_338,
                index=False
            )

            # Guardado local rápido, luego copia a Drive tras CADA modelo.
            respaldar_checkpoint_oulad(RUTA_MODELOS_338)
            respaldar_checkpoint_oulad(RUTA_RESULTADOS_338)
            print("Modelo guardado.")

    # ------------------------------------------------------------
    # 5. TABLAS CONSOLIDADAS
    # ------------------------------------------------------------

    print("\n=== 3.38.5 RESULTADOS CONSOLIDADOS ===")

    tabla_resultados_338 = pd.DataFrame(
        filas_resultados_338
    ).sort_values(
        ["Semana", "Experimento"]
    ).reset_index(drop=True)

    display(
        tabla_resultados_338.round(5)
    )

    # Comparación descriptiva de la puntuación
    # interna del proceso de selección.
    comparacion_interna_338 = (
        tabla_resultados_338.pivot(
            index="Semana",
            columns="Experimento",
            values="F1_CV_interna"
        )
    )

    comparacion_interna_338[
        "Delta_B_menos_A"
    ] = (
        comparacion_interna_338["B"]
        -
        comparacion_interna_338["A"]
    )

    print(
        "\n=== 3.38.6 DIFERENCIA DE CV INTERNA ==="
    )

    display(
        comparacion_interna_338.round(5)
    )

    print(
        "\nNota: estas puntuaciones se utilizaron "
        "para optimización. La evaluación anidada "
        "anterior sigue siendo nuestra estimación "
        "de rendimiento en desarrollo."
    )

    # ------------------------------------------------------------
    # 6. CONTROLES FINALES
    # ------------------------------------------------------------

    print("\n=== 3.38.7 CONTROLES FINALES ===")

    assert len(
        modelos_finales_oulad_338
    ) == 8

    assert len(
        tabla_resultados_338
    ) == 8

    assert set(
        modelos_finales_oulad_338.keys()
    ) == {
        (semana, experimento)
        for semana in SEMANAS_338
        for experimento in EXPERIMENTOS_338
    }

    assert RUTA_MODELOS_338.exists()
    assert RUTA_RESULTADOS_338.exists()

    # Comprobar que se recuperan los ocho modelos.
    modelos_recuperados_338 = joblib.load(
        RUTA_MODELOS_338
    )

    assert len(
        modelos_recuperados_338
    ) == 8

    assert hashlib.sha256(
        RUTA_PROTOCOLO_338.read_bytes()
    ).hexdigest() == HASH_CONGELADO_338

    print("Modelos definitivos ajustados: 8")
    print("Semanas: 2, 4, 6, 8")
    print("Experimentos: A y B")
    print("Folds internos por modelo: 3")
    print("Grupos compartidos en CV: 0")
    print("GridSearchCV: completado")
    print("Pipelines originales: conservados")
    print("Protocolo: sin modificaciones")
    print("Test independiente: NO utilizado")
    print("Checkpoint:", RUTA_MODELOS_338)
    print("Resultados:", RUTA_RESULTADOS_338)

    print("\n=== ACTIVIDAD 3.38 FINALIZADA ===")



### 3.38.8. Conclusión del entrenamiento definitivo

Se completó el entrenamiento de ocho modelos de Regresión Logística correspondientes a los experimentos estático (A) y longitudinal (B) en las semanas 2, 4, 6 y 8.

La optimización se realizó mediante GridSearchCV con tres particiones internas agrupadas por estudiante. No se detectó solapamiento de estudiantes entre entrenamiento y validación.

Los pipelines conservaron el preprocesamiento, la selección de características y el espacio de hiperparámetros definidos en el protocolo congelado.

El F1 macro de validación interna del Experimento B fue de 0,66618, 0,68449, 0,70052 y 0,72251 para las semanas 2, 4, 6 y 8, respectivamente.

En las mismas ventanas, el Experimento A obtuvo valores de 0,61206, 0,60300, 0,59568 y 0,59151.

Las diferencias favorables al Experimento B fueron de 0,05413, 0,08150, 0,10483 y 0,13100 puntos de F1 macro.

Las brechas entre entrenamiento y validación interna fueron pequeñas para ambos experimentos, sin evidencia descriptiva importante de sobreajuste.

Los ocho modelos se reajustaron sobre sus respectivas cohortes completas de desarrollo y se guardaron en un checkpoint reproducible.

Estos resultados confirman la coherencia del procedimiento de entrenamiento definitivo con los hallazgos de la validación anidada. No constituyen una evaluación independiente adicional, puesto que la validación interna también intervino en la selección de hiperparámetros.

El protocolo congelado mantuvo su integridad y el conjunto de prueba independiente no fue utilizado.

Los modelos quedan preparados para la evaluación final sobre las particiones de prueba previamente reservadas.



### 3.39. Evaluación independiente de la hipótesis principal

#### Objetivo

Evaluar el rendimiento de los modelos definitivos de Regresión Logística estática (Experimento A) y longitudinal (Experimento B) sobre estudiantes que no participaron en el entrenamiento ni en la selección de hiperparámetros.

#### Diseño experimental

Se utilizarán los ocho pipelines ajustados en la Actividad 3.38, manteniendo las configuraciones y decisiones documentadas en el protocolo congelado de la Actividad 3.37.

Las ventanas de predicción corresponden a las semanas 2, 4, 6 y 8, utilizando exclusivamente información disponible antes de los días 14, 28, 42 y 56, respectivamente.

Los conjuntos independientes contienen:

| Semana | Matrículas de prueba |
|---|---:|
| 2 | 5.550 |
| 4 | 5.438 |
| 6 | 5.344 |
| 8 | 5.249 |

La partición se realizó mediante identificadores de estudiante, de manera que ningún estudiante de prueba aparece en los conjuntos de entrenamiento.

#### Métrica principal

La métrica primaria será el F1 macro.

Se calculará la diferencia emparejada:

Delta F1 = F1 longitudinal - F1 estático

Un valor positivo indica que el Experimento B supera al Experimento A sobre las mismas matrículas de prueba.

#### Métricas complementarias

Se calcularán:

- Accuracy.
- Precisión y recall de la clase adversa.
- F1 de la clase adversa.
- ROC-AUC.
- Brier Score.
- Matrices de confusión.
- Detección retrospectiva de Fail y Withdrawn.
- Anticipación de retiros con fechas disponibles.

#### Análisis de incertidumbre

Se utilizará bootstrap pareado agrupado por `id_student` para obtener intervalos descriptivos del 95 % de la diferencia de F1 macro en cada ventana.

El remuestreo conservará juntas todas las matrículas de un mismo estudiante.

Los intervalos se interpretarán como estimaciones condicionadas a los modelos entrenados. No incorporan toda la incertidumbre de selección de hiperparámetros ni constituyen intervalos simultáneos ajustados por multiplicidad.

#### Interpretación temporal

Las cohortes elegibles cambian entre semanas debido a retiros y criterios de elegibilidad. Por tanto, las diferencias de rendimiento entre semanas no deben atribuirse exclusivamente al paso del tiempo.

El contraste principal A frente a B se realizará dentro de cada semana, sobre una población de prueba idéntica para ambos experimentos.

#### Restricción fundamental

El conjunto independiente se utilizará únicamente para evaluar los modelos previamente congelados.

No se reentrenarán clasificadores, no se modificarán los umbrales y no se seleccionarán nuevos hiperparámetros después de conocer los resultados.

Los resultados de esta actividad se considerarán la evaluación final de la hipótesis principal.

**Referencias metodológicas:** la evaluación reservada corresponde a la estimación independiente del rendimiento predictivo [2], [10]. Se interpretan las curvas ROC y las métricas por clase junto con las precauciones sobre ROC en problemas desbalanceados [12].


In [ ]:
# TEST FINAL BLOQUEADO: se evaluó previamente en la Actividad 3.39.
# Una ejecución desde cero no debe volver a abrir el test para ajustar modelos.
from pathlib import Path
import pandas as pd
_archivos_finales_339=[
    'metricas_test_oulad_339.csv',
    'predicciones_test_oulad_339.csv',
    'bootstrap_test_oulad_339.csv',
    'diferencias_test_oulad_339.csv',
    'retiros_test_oulad_339.csv',
    'subtipos_test_oulad_339.csv',
]
faltantes_339=[x for x in _archivos_finales_339 if not (Path('/content')/x).is_file()]
if faltantes_339:
    raise RuntimeError(
        'La prueba independiente ya se realizó en una sesión anterior; '
        'no puede repetirse automáticamente. Recupera sus seis CSV de la sesión original/Drive. '
        f'Archivos no encontrados: {faltantes_339}'
    )
else:
    tabla_metricas_339=pd.read_csv('/content/metricas_test_oulad_339.csv')
    predicciones_test_339=pd.read_csv('/content/predicciones_test_oulad_339.csv')
    tabla_bootstrap_339=pd.read_csv('/content/bootstrap_test_oulad_339.csv')
    tabla_diferencias_339=pd.read_csv('/content/diferencias_test_oulad_339.csv')
    tabla_retiros_339=pd.read_csv('/content/retiros_test_oulad_339.csv')
    tabla_subtipos_339=pd.read_csv('/content/subtipos_test_oulad_339.csv')
    print('=== 3.39: RESULTADOS INDEPENDIENTES RECUPERADOS ===')
    print('No se vuelve a llamar a predict, predict_proba, fit ni GridSearchCV.')
    display(tabla_metricas_339.round(5))



### 3.39.12. Conclusión de la evaluación independiente

La Actividad 3.39 constituyó la primera evaluación independiente de los modelos definitivos de Regresión Logística estática (Experimento A) y longitudinal (Experimento B), después de congelar el protocolo y completar el entrenamiento sobre las cohortes de desarrollo.

La evaluación comprendió las semanas 2, 4, 6 y 8, con 5.550, 5.438, 5.344 y 5.249 matrículas de prueba, respectivamente. Se verificó la ausencia de estudiantes compartidos entre entrenamiento y prueba.

**Resultados de la hipótesis principal**

El Experimento B superó al Experimento A en F1 macro en las cuatro ventanas temporales.

Las diferencias observadas fueron de 0,06887 en la semana 2, 0,07645 en la semana 4, 0,09741 en la semana 6 y 0,11510 en la semana 8.

Los intervalos de confianza bootstrap del 95 %, obtenidos mediante remuestreo pareado por estudiante, fueron:

- Semana 2: [0,05521; 0,08248].
- Semana 4: [0,06163; 0,08980].
- Semana 6: [0,08201; 0,11238].
- Semana 8: [0,10013; 0,13133].

Todos los intervalos excluyeron el valor cero, proporcionando evidencia consistente de una mejora predictiva longitudinal. Estos intervalos son descriptivos, no están ajustados por multiplicidad y no incorporan toda la incertidumbre del entrenamiento y la selección de modelos.

**Identificación de resultados académicos adversos**

El Experimento B mejoró también la sensibilidad de la clase adversa en todas las ventanas.

En la semana 2 identificó 300 casos adversos adicionales y evitó 38 falsas alertas respecto al Experimento A. En la semana 8 identificó 430 casos adversos adicionales y evitó 58 falsas alertas.

El incremento del recall adverso fue de 12,09, 12,22, 16,42 y 19,75 puntos porcentuales en las semanas 2, 4, 6 y 8.

**Discriminación probabilística**

El ROC-AUC del Experimento B pasó de 0,73960 en la semana 2 a 0,79225 en la semana 8. En todas las ventanas fue superior al obtenido por el Experimento A.

Asimismo, el Brier Score de B fue menor en las cuatro semanas, lo que indica una reducción del error cuadrático de las probabilidades predichas.

**Anticipación de retiros**

En la semana 2, el Experimento B identificó 633 de los 1.042 retiros futuros registrados en la cohorte independiente, frente a 516 detectados por A.

De estos 633 retiros identificados, 545 fueron detectados con al menos 14 días y 472 con al menos 28 días de anticipación respecto a la fecha registrada de retiro.

La mediana de anticipación entre los retiros identificados por B fue de 72 días.

Estos resultados representan una capacidad predictiva retrospectiva. No demuestran que una intervención hubiera prevenido los retiros. Además, el resultado Withdrawn de OULAD corresponde al retiro de un módulo y no necesariamente a deserción universitaria.

**Generalización y reproducibilidad**

Las puntuaciones independientes fueron coherentes con las estimaciones obtenidas mediante validación cruzada anidada.

No se realizaron reajustes después de observar los resultados del conjunto de prueba. El protocolo y los modelos conservaron sus huellas de integridad.

**Conclusión general**

Los resultados independientes respaldan la hipótesis de investigación: incorporar trayectorias longitudinales de interacción y participación académica mejora la identificación temprana del riesgo académico en OULAD frente al uso exclusivo de características estáticas.

La mejora fue observable desde la segunda semana y se mantuvo en las cuatro ventanas evaluadas.

No obstante, las poblaciones elegibles cambian entre semanas; por ello, el incremento temporal del rendimiento no debe atribuirse exclusivamente a la acumulación de información sin realizar análisis de sensibilidad adicionales.

La evidencia corresponde a una validación independiente por estudiante dentro de OULAD y no constituye todavía una validación externa en otra universidad ni una demostración de efectividad de intervenciones.



### 3.40. Auditoría de generalización y análisis de sensibilidad temporal

#### Objetivo

Consolidar los resultados definitivos de los experimentos A y B, comparar su rendimiento entre desarrollo y prueba independiente, y examinar la sensibilidad de las conclusiones temporales ante los cambios de composición de las cohortes.

#### Procedimiento

1. Recuperar las métricas y predicciones independientes almacenadas en la Actividad 3.39, sin volver a ejecutar los modelos.
2. Comparar las estimaciones de F1 macro procedentes de la validación cruzada anidada con las obtenidas en el test.
3. Identificar las matrículas presentes en las cuatro cohortes de prueba.
4. Recalcular descriptivamente las métricas sobre esa población común utilizando únicamente las predicciones ya guardadas.
5. Comparar el aporte longitudinal sobre las cohortes originales y sobre la cohorte común.
6. Guardar las tablas de auditoría para su inclusión en el informe académico.

#### Consideraciones metodológicas

La cohorte común permite mantener constante la población evaluada, pero se define retrospectivamente mediante la elegibilidad hasta la semana 8.

Por tanto, puede introducir sesgo de supervivencia y no representa a todos los estudiantes observables inicialmente.

Los modelos fueron entrenados en distintas cohortes según la ventana temporal. El análisis controla la composición de la población de evaluación, pero no garantiza que las diferencias entre semanas correspondan exclusivamente al paso del tiempo.

Este procedimiento constituye un análisis de sensibilidad descriptivo, no una nueva selección de modelos.

No se modificarán umbrales, hiperparámetros ni características predictivas.


In [ ]:

# ============================================================
# ACTIVIDAD 3.40
# GENERALIZACIÓN Y SENSIBILIDAD TEMPORAL
#
# Solo lee resultados de la Actividad 3.39.
# NO carga clasificadores.
# NO ejecuta predict() ni fit().
# NO modifica el protocolo.
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from IPython.display import display

from sklearn.metrics import (
    f1_score,
    recall_score,
    roc_auc_score,
    brier_score_loss,
    accuracy_score
)

SEMANAS_340 = [2, 4, 6, 8]

CLAVES_340 = [
    "code_module",
    "code_presentation",
    "id_student"
]

DIRECTORIO_340 = Path("/content")

# ------------------------------------------------------------
# 1. RECUPERAR RESULTADOS PREVIAMENTE GUARDADOS
# ------------------------------------------------------------

print("=== 3.40.1 RECUPERACIÓN ===")

rutas_340 = {
    "metricas": DIRECTORIO_340 /
        "metricas_test_oulad_339.csv",
    "predicciones": DIRECTORIO_340 /
        "predicciones_test_oulad_339.csv",
    "bootstrap": DIRECTORIO_340 /
        "bootstrap_test_oulad_339.csv"
}

for nombre, ruta in rutas_340.items():
    assert ruta.exists(), (
        f"No se encontró el archivo {nombre}: {ruta}"
    )
    print(nombre, ":", ruta)

metricas_340 = pd.read_csv(
    rutas_340["metricas"]
)

predicciones_340 = pd.read_csv(
    rutas_340["predicciones"]
)

bootstrap_340 = pd.read_csv(
    rutas_340["bootstrap"]
)

assert len(metricas_340) == 8
assert len(bootstrap_340) == 4

assert set(
    predicciones_340["Semana"].unique()
) == set(SEMANAS_340)

assert set(
    predicciones_340["Experimento"].unique()
) == {"A", "B"}

assert not predicciones_340.duplicated(
    ["Semana", "Experimento"] + CLAVES_340
).any()

print(
    "Predicciones recuperadas:",
    len(predicciones_340)
)

print("No se han generado nuevas predicciones.")

# ------------------------------------------------------------
# 2. COMPARAR DESARROLLO VS TEST
# ------------------------------------------------------------

print("\n=== 3.40.2 GENERALIZACIÓN ===")

# Resultados previamente observados en la
# Actividad 3.24: medias de CV anidada.
# No son valores nuevos ni estimaciones del test.

desarrollo_340 = pd.DataFrame({
    "Semana": [2, 4, 6, 8],
    "F1_desarrollo_A": [
        0.60762, 0.60398, 0.59755, 0.59074
    ],
    "F1_desarrollo_B": [
        0.66560, 0.68547, 0.69968, 0.72280
    ]
})

test_f1_340 = metricas_340.pivot(
    index="Semana",
    columns="Experimento",
    values="F1_macro"
).reset_index().rename(
    columns={
        "A": "F1_test_A",
        "B": "F1_test_B"
    }
)

generalizacion_340 = desarrollo_340.merge(
    test_f1_340,
    on="Semana",
    validate="one_to_one"
)

generalizacion_340["Delta_desarrollo"] = (
    generalizacion_340["F1_desarrollo_B"]
    - generalizacion_340["F1_desarrollo_A"]
)

generalizacion_340["Delta_test"] = (
    generalizacion_340["F1_test_B"]
    - generalizacion_340["F1_test_A"]
)

generalizacion_340["Cambio_A_test_desarrollo"] = (
    generalizacion_340["F1_test_A"]
    - generalizacion_340["F1_desarrollo_A"]
)

generalizacion_340["Cambio_B_test_desarrollo"] = (
    generalizacion_340["F1_test_B"]
    - generalizacion_340["F1_desarrollo_B"]
)

display(
    generalizacion_340.round(5)
)

# ------------------------------------------------------------
# 3. CONSTRUIR COHORTE COMÚN DE PRUEBA
# ------------------------------------------------------------

print("\n=== 3.40.3 COHORTE COMÚN ===")

indices_por_semana_340 = []

for semana in SEMANAS_340:

    sub = predicciones_340.loc[
        predicciones_340["Semana"].eq(semana)
        & predicciones_340["Experimento"].eq("A")
    ]

    indice = pd.MultiIndex.from_frame(
        sub[CLAVES_340]
    )

    assert indice.is_unique

    indices_por_semana_340.append(
        indice
    )

comunes_340 = indices_por_semana_340[0]

for indice in indices_por_semana_340[1:]:
    comunes_340 = comunes_340.intersection(
        indice
    )

assert len(comunes_340) > 0

indice_predicciones_340 = (
    pd.MultiIndex.from_frame(
        predicciones_340[CLAVES_340]
    )
)

pred_comun_340 = predicciones_340.loc[
    indice_predicciones_340.isin(
        comunes_340
    )
].copy()

print(
    "Matrículas comunes:",
    len(comunes_340)
)

print(
    "Estudiantes únicos:",
    comunes_340.get_level_values(
        "id_student"
    ).nunique()
)

print(
    "Proporción de semana 2:",
    round(
        100 * len(comunes_340)
        / len(indices_por_semana_340[0]),
        2
    ),
    "%"
)

for semana in SEMANAS_340:
    for exp in ["A", "B"]:

        n = len(
            pred_comun_340.loc[
                pred_comun_340["Semana"].eq(semana)
                & pred_comun_340["Experimento"].eq(exp)
            ]
        )

        assert n == len(comunes_340)

print("Población común verificada.")

# ------------------------------------------------------------
# 4. MÉTRICAS SOBRE POBLACIÓN FIJA
# ------------------------------------------------------------

print("\n=== 3.40.4 MÉTRICAS COHORTE COMÚN ===")

filas_comunes_340 = []

for semana in SEMANAS_340:

    for exp in ["A", "B"]:

        sub = pred_comun_340.loc[
            pred_comun_340["Semana"].eq(semana)
            & pred_comun_340["Experimento"].eq(exp)
        ]

        y = sub["y_real"].to_numpy()
        pred = sub["y_pred"].to_numpy()
        p = sub["prob_adverso"].to_numpy()

        filas_comunes_340.append({
            "Semana": semana,
            "Experimento": exp,
            "Matriculas": len(sub),
            "Estudiantes": sub[
                "id_student"
            ].nunique(),
            "F1_macro": f1_score(
                y,
                pred,
                average="macro"
            ),
            "Recall_adverso": recall_score(
                y,
                pred
            ),
            "ROC_AUC": roc_auc_score(
                y,
                p
            ),
            "Accuracy": accuracy_score(
                y,
                pred
            ),
            "Brier": brier_score_loss(
                y,
                p
            )
        })

tabla_comun_340 = pd.DataFrame(
    filas_comunes_340
)

display(
    tabla_comun_340.round(5)
)

# ------------------------------------------------------------
# 5. DIFERENCIAS A VS B EN COHORTE COMÚN
# ------------------------------------------------------------

print("\n=== 3.40.5 SENSIBILIDAD TEMPORAL ===")

f1_comun_340 = tabla_comun_340.pivot(
    index="Semana",
    columns="Experimento",
    values="F1_macro"
).reset_index()

f1_comun_340["Delta_F1_comun"] = (
    f1_comun_340["B"]
    -
    f1_comun_340["A"]
)

comparacion_340 = generalizacion_340[
    [
        "Semana",
        "Delta_desarrollo",
        "Delta_test"
    ]
].merge(
    f1_comun_340[
        ["Semana", "Delta_F1_comun"]
    ],
    on="Semana",
    validate="one_to_one"
)

comparacion_340[
    "Cambio_comun_menos_test"
] = (
    comparacion_340["Delta_F1_comun"]
    -
    comparacion_340["Delta_test"]
)

display(
    comparacion_340.round(5)
)

# ------------------------------------------------------------
# 6. AUDITORÍA DE CONSISTENCIA
# ------------------------------------------------------------

print("\n=== 3.40.6 CONTROLES ===")

# Las predicciones originales deben reproducir
# exactamente las métricas almacenadas.

for _, fila in metricas_340.iterrows():

    semana = int(fila["Semana"])
    exp = fila["Experimento"]

    sub = predicciones_340.loc[
        predicciones_340["Semana"].eq(semana)
        & predicciones_340["Experimento"].eq(exp)
    ]

    f1 = f1_score(
        sub["y_real"],
        sub["y_pred"],
        average="macro"
    )

    assert np.isclose(
        f1,
        fila["F1_macro"],
        atol=1e-10
    )

# No permitir que el análisis de cohorte común
# introduzca registros nuevos ni predicciones nuevas.

assert len(pred_comun_340) == (
    len(comunes_340) * 8
)

assert set(
    comparacion_340["Semana"]
) == set(SEMANAS_340)

print("Métricas originales: reproducidas")
print("Matrículas comunes: verificadas")
print("Modelos reentrenados: 0")
print("Predicciones nuevas: 0")
print("Umbrales modificados: 0")

# ------------------------------------------------------------
# 7. GRÁFICOS DE ROBUSTEZ
# ------------------------------------------------------------

print("\n=== 3.40.7 GRÁFICOS ===")

fig, ax = plt.subplots(
    figsize=(9, 5)
)

ax.plot(
    comparacion_340["Semana"],
    comparacion_340["Delta_desarrollo"],
    marker="o",
    label="Desarrollo, CV anidada"
)

ax.plot(
    comparacion_340["Semana"],
    comparacion_340["Delta_test"],
    marker="s",
    label="Test original"
)

ax.plot(
    comparacion_340["Semana"],
    comparacion_340["Delta_F1_comun"],
    marker="^",
    label="Test, cohorte común"
)

ax.axhline(
    0,
    linestyle="--",
    linewidth=1
)

ax.set_title(
    "Robustez del aporte longitudinal"
)

ax.set_xlabel(
    "Semana de predicción"
)

ax.set_ylabel(
    "Diferencia F1 macro (B - A)"
)

ax.set_xticks(SEMANAS_340)
ax.grid(alpha=0.3)
ax.legend()

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 8. EXPORTACIÓN
# ------------------------------------------------------------

print("\n=== 3.40.8 EXPORTACIÓN ===")

archivos_salida_340 = {
    "generalizacion": DIRECTORIO_340 /
        "generalizacion_oulad_340.csv",
    "cohorte_comun": DIRECTORIO_340 /
        "cohorte_comun_oulad_340.csv",
    "sensibilidad": DIRECTORIO_340 /
        "sensibilidad_temporal_oulad_340.csv"
}

generalizacion_340.to_csv(
    archivos_salida_340["generalizacion"],
    index=False
)

tabla_comun_340.to_csv(
    archivos_salida_340["cohorte_comun"],
    index=False
)

comparacion_340.to_csv(
    archivos_salida_340["sensibilidad"],
    index=False
)

for nombre, ruta in archivos_salida_340.items():

    assert ruta.exists()

    print(nombre, ":", ruta)

print("\n=== ACTIVIDAD 3.40 FINALIZADA ===")
print("Auditoría realizada sin reentrenamiento.")
print("Los resultados de prueba originales no se alteraron.")



### 3.40.9. Conclusión de la auditoría de generalización y robustez temporal

La Actividad 3.40 evaluó la consistencia del aporte predictivo de las características longitudinales mediante tres escenarios: validación cruzada anidada sobre desarrollo, evaluación independiente sobre las cohortes originales y evaluación independiente sobre una cohorte común.

**Generalización a estudiantes no observados durante el entrenamiento**

La comparación entre desarrollo y prueba independiente mostró que el rendimiento del Experimento B se mantuvo próximo a las estimaciones de validación cruzada anidada.

Las diferencias del F1 macro entre test y desarrollo para B fueron de +0,01702, +0,00037, +0,00080 y -0,00615 en las semanas 2, 4, 6 y 8, respectivamente.

No se observó una degradación sistemática importante del rendimiento al evaluar estudiantes que no participaron en el entrenamiento ni en la selección de hiperparámetros.

**Análisis sobre población fija**

Se identificó una cohorte común de 5.243 matrículas, correspondientes a 4.789 estudiantes, presentes en las cuatro ventanas de predicción. Esta población representa el 94,47 % de las matrículas de la cohorte independiente de la semana 2.

Sobre esta cohorte fija, el Experimento B obtuvo F1 macro de 0,67414, 0,68148, 0,69721 y 0,71634 en las semanas 2, 4, 6 y 8, respectivamente.

El Experimento A obtuvo F1 macro de 0,61166, 0,60952, 0,60427 y 0,60137.

En consecuencia, las diferencias favorables al Experimento B fueron de 0,06248, 0,07197, 0,09294 y 0,11497.

**Sensibilidad a la composición de cohortes**

La diferencia entre el aporte longitudinal medido en las cohortes originales y en la cohorte común fue de -0,00639, -0,00448, -0,00447 y aproximadamente -0,00012 puntos de F1 macro.

La ventaja del modelo longitudinal permaneció positiva en las cuatro ventanas y conservó su tendencia creciente sobre la cohorte común.

Este resultado reduce la posibilidad de que el patrón observado se explique únicamente por cambios en la población evaluada.

**Limitaciones**

La cohorte común fue definida retrospectivamente mediante la elegibilidad hasta la semana 8, por lo que puede presentar sesgo de supervivencia.

Además, los modelos de cada semana fueron entrenados sobre cohortes diferentes y utilizaron cantidades crecientes de información disponible.

Por consiguiente, el análisis no permite atribuir causalmente toda la mejora al transcurso del tiempo.

La prueba independiente separó estudiantes, pero no constituyó una validación externa con datos de otra institución.

**Conclusión**

La auditoría respalda la robustez de la hipótesis principal: la incorporación de información longitudinal mejora la clasificación de resultados académicos adversos respecto a un modelo basado exclusivamente en características estáticas.

La mejora se mantuvo en desarrollo, en el conjunto independiente y en el análisis de sensibilidad sobre población fija.

El aporte longitudinal fue detectable desde la semana 2 y aumentó hasta la semana 8.

Todos los análisis utilizaron las predicciones previamente obtenidas. No se reentrenaron modelos, no se modificaron umbrales ni se alteró el protocolo experimental.



# Actividad 3.41. Consolidación final de la evidencia científica

## 3.41.1. Objetivo

Integrar los resultados de desarrollo, validación cruzada anidada, optimización de hiperparámetros y evaluación independiente para construir una síntesis reproducible de los hallazgos del proyecto.

Esta actividad no incorpora nuevos modelos ni modifica las decisiones experimentales. Su propósito es organizar e interpretar la evidencia ya obtenida y establecer qué conclusiones están respaldadas por evaluación independiente y cuáles corresponden a análisis exploratorios.

## 3.41.2. Hipótesis de investigación

La incorporación de características longitudinales del comportamiento estudiantil mejora el rendimiento predictivo para identificar resultados académicos adversos en comparación con modelos basados exclusivamente en características estáticas.

La evaluación se realizó en las semanas 2, 4, 6 y 8 de los módulos de OULAD.

Se utilizó F1 macro como métrica principal y se incluyeron recall adverso, ROC-AUC, Brier Score y anticipación de retiros como medidas complementarias.

## 3.41.3. Niveles de evidencia

### Evidencia principal: evaluación independiente

Corresponde a la comparación A frente a B con Regresión Logística, seleccionada y congelada antes de utilizar el conjunto de prueba.

Los estudiantes fueron separados entre entrenamiento y prueba mediante su identificador individual, impidiendo que distintas matrículas de un mismo estudiante aparecieran en ambos conjuntos.

### Evidencia complementaria

Incluye la detección de resultados adversos, el desempeño probabilístico, la identificación anticipada de retiros y el análisis de sensibilidad sobre una cohorte común.

Estos análisis permiten contextualizar la utilidad del modelo y sus limitaciones.

### Evidencia exploratoria

Comprende Random Forest, Transformer, optimización mediante Optuna, curvas de aprendizaje, pruebas de regularización, calibración, auditoría por subgrupos y estrategias operativas de alertas.

Estos resultados fueron utilizados para estudiar alternativas y comprender el comportamiento de los modelos durante el desarrollo.

No deben confundirse con evaluaciones independientes adicionales.

## 3.41.4. Criterios de reproducibilidad

La consolidación utilizará únicamente archivos previamente almacenados.

Se verificarán las dimensiones, los identificadores de semanas, la integridad de las métricas y la consistencia de las diferencias entre experimentos.

Se conservará una referencia explícita a la procedencia de los resultados.

No se realizarán nuevos ajustes, optimización de hiperparámetros, calibraciones, modificaciones de umbral ni predicciones sobre el conjunto independiente.

## 3.41.5. Productos esperados

- Tabla maestra de resultados definitivos por semana.
- Comparación entre desarrollo y prueba independiente.
- Auditoría de robustez sobre población fija.
- Resumen de la anticipación de retiros.
- Inventario de evidencia principal, complementaria y exploratoria.
- Figuras finales para el informe.
- Síntesis académica exportable en Markdown.

## 3.41.6. Alcance de las conclusiones

La evidencia permite evaluar la capacidad predictiva de resultados adversos en OULAD para estudiantes no incluidos en el entrenamiento.

No permite afirmar que las alertas hayan prevenido retiros, que los resultados se generalicen directamente a otras universidades ni que las diferencias observadas entre semanas tengan una interpretación causal.

Asimismo, el resultado Withdrawn corresponde al retiro de un módulo de OULAD, no necesariamente a la deserción universitaria completa.


In [ ]:

# ============================================================
# ACTIVIDAD 3.41
# CONSOLIDACIÓN FINAL DE LA EVIDENCIA CIENTÍFICA
#
# NO entrena modelos.
# NO genera predicciones.
# NO modifica umbrales ni hiperparámetros.
# NO selecciona nuevamente el modelo definitivo.
#
# Asistencia IA: ChatGPT, octubre de 2026.
# Prompt:
# "Consolidar la evidencia de un estudio con OULAD,
# distinguiendo evaluación independiente, análisis
# complementarios y experimentos exploratorios.
# Recuperar resultados existentes, auditar su
# consistencia y exportar tablas, figuras y reporte
# académico, sin reajustar los modelos."
# ============================================================

import json
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from IPython.display import display

CARPETA_341 = Path("/content")
SEMANAS_341 = [2, 4, 6, 8]

# ------------------------------------------------------------
# 1. ARCHIVOS DE ORIGEN
# ------------------------------------------------------------

print("=== 3.41.1 RECUPERACIÓN DE EVIDENCIA ===")

fuentes_341 = {
    "metricas_test": "metricas_test_oulad_339.csv",
    "bootstrap_test": "bootstrap_test_oulad_339.csv",
    "diferencias_test": "diferencias_test_oulad_339.csv",
    "retiros_test": "retiros_test_oulad_339.csv",
    "subtipos_test": "subtipos_test_oulad_339.csv",
    "generalizacion": "generalizacion_oulad_340.csv",
    "cohorte_comun": "cohorte_comun_oulad_340.csv",
    "sensibilidad": "sensibilidad_temporal_oulad_340.csv",
}

tablas_341 = {}
inventario_archivos_341 = []

for nombre, archivo in fuentes_341.items():

    ruta = CARPETA_341 / archivo

    if not ruta.exists():
        raise FileNotFoundError(
            f"Falta un resultado previo: {ruta}"
        )

    tabla = pd.read_csv(ruta)
    tablas_341[nombre] = tabla

    sha256 = hashlib.sha256(
        ruta.read_bytes()
    ).hexdigest()

    inventario_archivos_341.append({
        "Fuente": nombre,
        "Archivo": archivo,
        "Filas": len(tabla),
        "SHA256": sha256,
    })

tabla_inventario_341 = pd.DataFrame(
    inventario_archivos_341
)

display(tabla_inventario_341)

print("Archivos originales: recuperados")
print("Nuevos modelos: 0")

# ------------------------------------------------------------
# 2. PROTOCOLO CONGELADO
# ------------------------------------------------------------

print("\n=== 3.41.2 INTEGRIDAD DEL PROTOCOLO ===")

ruta_protocolo_341 = (
    CARPETA_341 / "protocolo_oulad_337.json"
)

hash_esperado_341 = (
    "283bbddb0c56a615dde6a8a1201e149dfd7020f4672d7d18267e4c93ad880625"
)

assert ruta_protocolo_341.exists()

hash_observado_341 = hashlib.sha256(
    ruta_protocolo_341.read_bytes()
).hexdigest()

assert hash_observado_341 == hash_esperado_341

print("Protocolo original: verificado")
print("SHA-256:", hash_observado_341)

# ------------------------------------------------------------
# 3. TABLA MAESTRA DEL TEST INDEPENDIENTE
# ------------------------------------------------------------

print("\n=== 3.41.3 EVIDENCIA PRINCIPAL ===")

metricas = tablas_341["metricas_test"]
bootstrap = tablas_341["bootstrap_test"]
diferencias = tablas_341["diferencias_test"]

assert len(metricas) == 8
assert len(bootstrap) == 4
assert len(diferencias) == 4

assert set(metricas["Semana"]) == set(SEMANAS_341)

assert set(metricas["Experimento"]) == {"A", "B"}

assert not metricas.duplicated(
    ["Semana", "Experimento"]
).any()

filas_maestras_341 = []

for semana in SEMANAS_341:

    s = metricas.loc[
        metricas["Semana"].eq(semana)
    ].set_index("Experimento")

    a = s.loc["A"]
    b = s.loc["B"]

    intervalo = bootstrap.loc[
        bootstrap["Semana"].eq(semana)
    ].iloc[0]

    d = diferencias.loc[
        diferencias["Semana"].eq(semana)
    ].iloc[0]

    assert a["Matriculas"] == b["Matriculas"]
    assert a["Adversos"] == b["Adversos"]

    delta = float(
        b["F1_macro"] - a["F1_macro"]
    )

    assert np.isclose(
        delta,
        intervalo["Delta_F1_observado"],
        atol=1e-4
    )

    assert np.isclose(
        delta,
        d["Delta_F1_macro"],
        atol=1e-4
    )

    filas_maestras_341.append({
        "Semana": semana,
        "N_test": int(a["Matriculas"]),
        "N_adversos": int(a["Adversos"]),
        "F1_A": float(a["F1_macro"]),
        "F1_B": float(b["F1_macro"]),
        "Delta_F1": delta,
        "IC95_inferior": float(
            intervalo["IC95_inferior"]
        ),
        "IC95_superior": float(
            intervalo["IC95_superior"]
        ),
        "Recall_A": float(a["Recall_adverso"]),
        "Recall_B": float(b["Recall_adverso"]),
        "Delta_recall": float(
            b["Recall_adverso"]
            - a["Recall_adverso"]
        ),
        "ROC_AUC_A": float(a["ROC_AUC"]),
        "ROC_AUC_B": float(b["ROC_AUC"]),
        "Brier_A": float(a["Brier"]),
        "Brier_B": float(b["Brier"]),
        "Accuracy_A": float(a["Accuracy"]),
        "Accuracy_B": float(b["Accuracy"]),
        "FN_evitados_B": int(
            a["FN"] - b["FN"]
        ),
        "FP_evitados_B": int(
            a["FP"] - b["FP"]
        ),
    })

tabla_maestra_341 = pd.DataFrame(
    filas_maestras_341
)

assert (tabla_maestra_341["Delta_F1"] > 0).all()
assert (
    tabla_maestra_341["IC95_inferior"] > 0
).all()

display(
    tabla_maestra_341.round(5)
)

print(
    "Las cuatro comparaciones principales "
    "favorecen al Experimento B."
)

# ------------------------------------------------------------
# 4. DESARROLLO VS TEST Y COHORTE COMÚN
# ------------------------------------------------------------

print("\n=== 3.41.4 ROBUSTEZ Y GENERALIZACIÓN ===")

generalizacion = tablas_341["generalizacion"]

comun = tablas_341["cohorte_comun"]

sensibilidad = tablas_341["sensibilidad"]

assert len(generalizacion) == 4
assert len(comun) == 8
assert len(sensibilidad) == 4

f1_comun = comun.pivot(
    index="Semana",
    columns="Experimento",
    values="F1_macro"
)

filas_robustez_341 = []

for semana in SEMANAS_341:

    original = tabla_maestra_341.loc[
        tabla_maestra_341["Semana"].eq(semana)
    ].iloc[0]

    desarrollo = generalizacion.loc[
        generalizacion["Semana"].eq(semana)
    ].iloc[0]

    delta_comun = float(
        f1_comun.loc[semana, "B"]
        - f1_comun.loc[semana, "A"]
    )

    filas_robustez_341.append({
        "Semana": semana,
        "Delta_desarrollo": float(
            desarrollo["Delta_desarrollo"]
        ),
        "Delta_test_original": float(
            original["Delta_F1"]
        ),
        "Delta_test_comun": delta_comun,
        "Cambio_comun_original": (
            delta_comun - original["Delta_F1"]
        ),
        "F1_B_desarrollo": float(
            desarrollo["F1_desarrollo_B"]
        ),
        "F1_B_test": float(
            desarrollo["F1_test_B"]
        ),
    })

tabla_robustez_341 = pd.DataFrame(
    filas_robustez_341
)

for _, fila in tabla_robustez_341.iterrows():

    s = int(fila["Semana"])

    original_340 = sensibilidad.loc[
        sensibilidad["Semana"].eq(s)
    ].iloc[0]

    assert np.isclose(
        fila["Delta_test_comun"],
        original_340["Delta_F1_comun"],
        atol=1e-4
    )

assert (
    tabla_robustez_341["Delta_test_comun"] > 0
).all()

display(
    tabla_robustez_341.round(5)
)

n_comunes_341 = int(
    comun["Matriculas"].iloc[0]
)

n_estudiantes_comunes_341 = int(
    comun["Estudiantes"].iloc[0]
)

assert comun["Matriculas"].nunique() == 1
assert comun["Estudiantes"].nunique() == 1

print(
    "Matrículas de la cohorte común:",
    n_comunes_341
)

print(
    "Estudiantes de la cohorte común:",
    n_estudiantes_comunes_341
)

# ------------------------------------------------------------
# 5. ANTICIPACIÓN DE RETIROS
# ------------------------------------------------------------

print("\n=== 3.41.5 ANTICIPACIÓN DE RETIROS ===")

retiros = tablas_341["retiros_test"]

assert len(retiros) == 8

filas_retiros_341 = []

for semana in SEMANAS_341:

    sub = retiros.loc[
        retiros["Semana"].eq(semana)
    ].set_index("Experimento")

    a = sub.loc["A"]
    b = sub.loc["B"]

    assert (
        a["Retiros_futuros"]
        == b["Retiros_futuros"]
    )

    filas_retiros_341.append({
        "Semana": semana,
        "Retiros_futuros": int(
            b["Retiros_futuros"]
        ),
        "Detectados_A": int(
            a["Retiros_detectados"]
        ),
        "Detectados_B": int(
            b["Retiros_detectados"]
        ),
        "Adicionales_B": int(
            b["Retiros_detectados"]
            - a["Retiros_detectados"]
        ),
        "Recall_retiro_A_pct": float(
            a["Recall_retiro_pct"]
        ),
        "Recall_retiro_B_pct": float(
            b["Recall_retiro_pct"]
        ),
        "Cobertura_14d_B_pct": float(
            b["Cobertura_total_14d_pct"]
        ),
        "Cobertura_28d_B_pct": float(
            b["Cobertura_total_28d_pct"]
        ),
        "Mediana_dias_detectados_B": float(
            b["Mediana_anticipacion_detectados"]
        ),
    })

tabla_retiros_341 = pd.DataFrame(
    filas_retiros_341
)

display(
    tabla_retiros_341.round(2)
)

print(
    "Nota: detección retrospectiva de "
    "retiros de módulo, sin evaluación "
    "de una intervención real."
)

# ------------------------------------------------------------
# 6. EVIDENCIA EXPLORATORIA DE DESARROLLO
# ------------------------------------------------------------

print("\n=== 3.41.6 MODELOS EXPLORATORIOS ===")

# Valores previamente reportados en Actividad 3.24.
# Representan medias de CV externa anidada.
# NO son resultados del test independiente.
# Se transcriben para centralizar su procedencia.

tabla_desarrollo_341 = pd.DataFrame({
    "Semana": [2, 4, 6, 8],
    "LR_A": [
        0.60762, 0.60398, 0.59755, 0.59074
    ],
    "LR_B": [
        0.66560, 0.68547, 0.69968, 0.72280
    ],
    "RF_A": [
        0.59859, 0.59452, 0.58795, 0.58165
    ],
    "RF_B": [
        0.65664, 0.67058, 0.68695, 0.70056
    ],
    "Transformer_C": [
        0.66544, 0.68251, 0.69528, 0.70948
    ],
})

display(
    tabla_desarrollo_341.round(5)
)

# Resumen de semana 8 de la Actividad 3.35.
# Corresponde a resultados OOF del desarrollo;
# utiliza un estimador agregado distinto a la
# media de F1 por fold presentada arriba.

tabla_optuna_341 = pd.DataFrame([
    {
        "Configuracion": "LR B | Grid",
        "F1_OOF": 0.72289,
        "Recall_pct": 59.08526,
        "FPR_pct": 15.35056,
        "Brier": 0.17533,
    },
    {
        "Configuracion": "LR B | Optuna",
        "F1_OOF": 0.72332,
        "Recall_pct": 63.77753,
        "FPR_pct": 19.55886,
        "Brier": 0.17758,
    },
    {
        "Configuracion": "RF B | Grid",
        "F1_OOF": 0.70057,
        "Recall_pct": 52.57482,
        "FPR_pct": 13.18547,
        "Brier": 0.18220,
    },
    {
        "Configuracion": "RF B | Optuna",
        "F1_OOF": 0.71462,
        "Recall_pct": 60.46866,
        "FPR_pct": 18.19400,
        "Brier": 0.18112,
    },
    {
        "Configuracion": "Transformer C",
        "F1_OOF": 0.70993,
        "Recall_pct": 55.16657,
        "FPR_pct": 13.99789,
        "Brier": 0.17880,
    },
])

display(
    tabla_optuna_341.round(5)
)

print(
    "IMPORTANTE: Optuna, RF y Transformer "
    "se reportan como evidencia de desarrollo."
)

# ------------------------------------------------------------
# 7. INVENTARIO DEL NIVEL DE EVIDENCIA
# ------------------------------------------------------------

print("\n=== 3.41.7 CLASIFICACIÓN DE EVIDENCIA ===")

tabla_evidencia_341 = pd.DataFrame([
    {
        "Hallazgo": "Mejora de F1 macro B frente a A",
        "Nivel": "Principal",
        "Procedencia": "3.39",
        "Estado": "Evaluado en test independiente",
    },
    {
        "Hallazgo": "Recall, ROC-AUC y Brier A/B",
        "Nivel": "Complementario",
        "Procedencia": "3.39",
        "Estado": "Evaluado en test independiente",
    },
    {
        "Hallazgo": "Detección futura de Withdrawn",
        "Nivel": "Complementario",
        "Procedencia": "3.39",
        "Estado": "Retrospectivo, test independiente",
    },
    {
        "Hallazgo": "Robustez sobre cohorte común",
        "Nivel": "Complementario",
        "Procedencia": "3.40",
        "Estado": "Sensibilidad retrospectiva",
    },
    {
        "Hallazgo": "Random Forest frente a LR",
        "Nivel": "Exploratorio",
        "Procedencia": "3.24-3.25",
        "Estado": "Validación cruzada de desarrollo",
    },
    {
        "Hallazgo": "Transformer C",
        "Nivel": "Exploratorio",
        "Procedencia": "3.18-3.25",
        "Estado": "Validación cruzada de desarrollo",
    },
    {
        "Hallazgo": "Optimización mediante Optuna",
        "Nivel": "Exploratorio",
        "Procedencia": "3.32-3.36",
        "Estado": "Validación cruzada de desarrollo",
    },
    {
        "Hallazgo": "Calibración y subgrupos",
        "Nivel": "Exploratorio",
        "Procedencia": "3.29-3.30, 3.35",
        "Estado": "Auditoría de desarrollo",
    },
    {
        "Hallazgo": "Capacidad y carga de alertas",
        "Nivel": "Exploratorio",
        "Procedencia": "3.27-3.28, 3.36",
        "Estado": "Simulación retrospectiva",
    },
])

display(tabla_evidencia_341)

# ------------------------------------------------------------
# 8. FIGURAS PARA EL INFORME
# ------------------------------------------------------------

print("\n=== 3.41.8 FIGURAS FINALES ===")

# FIGURA 1: F1 macro sobre test independiente.

fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(
    tabla_maestra_341["Semana"],
    tabla_maestra_341["F1_A"],
    marker="o",
    linewidth=2,
    label="Experimento A - Estático",
)

ax.plot(
    tabla_maestra_341["Semana"],
    tabla_maestra_341["F1_B"],
    marker="s",
    linewidth=2,
    label="Experimento B - Longitudinal",
)

ax.set_title(
    "Comparación A vs B - Prueba independiente"
)

ax.set_xlabel("Semana de predicción")
ax.set_ylabel("F1 macro")

ax.set_xticks(SEMANAS_341)
ax.grid(alpha=0.3)
ax.legend()

fig.tight_layout()

ruta_figura_1_341 = (
    CARPETA_341 / "figura_test_f1_341.png"
)

fig.savefig(
    ruta_figura_1_341,
    dpi=250,
    bbox_inches="tight"
)

plt.show()

# FIGURA 2: consistencia de diferencias.

fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(
    tabla_robustez_341["Semana"],
    tabla_robustez_341["Delta_desarrollo"],
    marker="o",
    label="Desarrollo - CV anidada",
)

ax.plot(
    tabla_robustez_341["Semana"],
    tabla_robustez_341["Delta_test_original"],
    marker="s",
    label="Test - Cohortes originales",
)

ax.plot(
    tabla_robustez_341["Semana"],
    tabla_robustez_341["Delta_test_comun"],
    marker="^",
    label="Test - Cohorte común",
)

ax.axhline(
    0,
    linestyle="--",
    linewidth=1
)

ax.set_title(
    "Robustez del aporte longitudinal"
)

ax.set_xlabel("Semana de predicción")
ax.set_ylabel("Diferencia F1 macro (B - A)")
ax.set_xticks(SEMANAS_341)

ax.legend()
ax.grid(alpha=0.3)

fig.tight_layout()

ruta_figura_2_341 = (
    CARPETA_341 / "figura_robustez_341.png"
)

fig.savefig(
    ruta_figura_2_341,
    dpi=250,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 9. GENERACIÓN DEL RESUMEN ACADÉMICO
# ------------------------------------------------------------

print("\n=== 3.41.9 SÍNTESIS ACADÉMICA ===")

def tabla_markdown_341(df, columnas, decimales=5):

    cabecera = "| " + " | ".join(columnas) + " |"
    separador = "| " + " | ".join(
        ["---"] * len(columnas)
    ) + " |"

    filas = []

    for _, fila in df[columnas].iterrows():

        valores = []

        for valor in fila:

            if isinstance(
                valor,
                (float, np.floating)
            ):
                valores.append(
                    f"{valor:.{decimales}f}"
                )
            else:
                valores.append(str(valor))

        filas.append(
            "| " + " | ".join(valores) + " |"
        )

    return "\n".join(
        [cabecera, separador] + filas
    )


tabla_md_341 = tabla_markdown_341(
    tabla_maestra_341,
    [
        "Semana",
        "N_test",
        "F1_A",
        "F1_B",
        "Delta_F1",
        "IC95_inferior",
        "IC95_superior",
        "Recall_B",
        "ROC_AUC_B",
    ]
)

tabla_robustez_md_341 = tabla_markdown_341(
    tabla_robustez_341,
    [
        "Semana",
        "Delta_desarrollo",
        "Delta_test_original",
        "Delta_test_comun",
    ]
)

tabla_retiros_md_341 = tabla_markdown_341(
    tabla_retiros_341,
    [
        "Semana",
        "Retiros_futuros",
        "Detectados_A",
        "Detectados_B",
        "Adicionales_B",
        "Cobertura_28d_B_pct",
    ],
    decimales=2
)

resumen_341 = f"""# Síntesis de evidencia científica - OULAD

## Pregunta de investigación

¿En qué medida la incorporación de características longitudinales
del comportamiento estudiantil mejora el rendimiento de los modelos
de Machine Learning para predecir tempranamente resultados académicos
adversos, frente al uso exclusivo de variables estáticas, y con
cuánta anticipación pueden identificarse estudiantes en riesgo?

## Hipótesis principal

La incorporación de características longitudinales mejora el
rendimiento predictivo frente a un modelo basado exclusivamente
en variables estáticas.

## Diseño

Dataset: OULAD.

Unidad predictiva: matrícula estudiante-módulo-presentación.

Separación de entrenamiento y prueba: por identificador de estudiante.

Modelo principal: Regresión Logística.

Experimento A: variables estáticas.

Experimento B: variables estáticas y comportamiento longitudinal.

Ventanas de predicción: semanas 2, 4, 6 y 8.

Métrica principal: F1 macro.

Hiperparámetros: seleccionados y congelados antes del test.

## Resultados del conjunto independiente

{tabla_md_341}

Las cuatro diferencias de F1 macro favorecen al Experimento B.
Los intervalos bootstrap descriptivos del 95 % excluyen cero
en las cuatro ventanas.

Estos intervalos no están ajustados por multiplicidad y están
condicionados a los modelos entrenados.

## Robustez temporal

{tabla_robustez_md_341}

Se evaluaron {n_comunes_341} matrículas correspondientes a
{n_estudiantes_comunes_341} estudiantes en la cohorte común.

El aporte longitudinal permaneció positivo en las cuatro semanas.

La cohorte común fue definida retrospectivamente y puede estar
sujeta a sesgo de supervivencia. Los modelos también se entrenaron
sobre distintas poblaciones según la semana.

## Anticipación de retiros

{tabla_retiros_md_341}

Withdrawn se interpreta como retiro del módulo de OULAD.
La anticipación es retrospectiva y no demuestra la eficacia
de una intervención preventiva.

## Evidencia exploratoria

Random Forest, Transformer C y Optuna fueron analizados durante
el desarrollo mediante validación cruzada. Sus resultados no
constituyen evaluaciones independientes adicionales.

La calibración, las diferencias entre subgrupos y las estrategias
de alertas se analizaron exploratoriamente.

No se utilizaron los resultados del test para seleccionar
nuevamente modelos, umbrales o hiperparámetros.

## Limitaciones

1. Los datos proceden de una única fuente institucional.
2. La validación independiente es por estudiante, no entre universidades.
3. Las cohortes elegibles cambian entre ventanas temporales.
4. La cohorte común presenta selección retrospectiva.
5. Los resultados adversos son Fail o Withdrawn a nivel de módulo.
6. Las alertas no fueron implementadas como intervención real.
7. La disponibilidad temporal de ciertas variables estáticas
   depende de su interpretación como información de inscripción.
8. Los intervalos bootstrap no incorporan toda la incertidumbre
   del entrenamiento y de selección de hiperparámetros.

## Conclusión

La evaluación independiente respalda la hipótesis de que las
características longitudinales aportan información predictiva
adicional frente a las características estáticas.

La ventaja se observa desde la segunda semana y se mantiene
hasta la octava semana.

El estudio demuestra capacidad predictiva retrospectiva en OULAD,
pero no establece causalidad, eficacia de intervención ni
generalización automática a otras instituciones.
"""

ruta_resumen_341 = (
    CARPETA_341 / "sintesis_cientifica_oulad_341.md"
)

ruta_resumen_341.write_text(
    resumen_341,
    encoding="utf-8"
)

print(
    "Resumen académico creado:",
    ruta_resumen_341
)

# ------------------------------------------------------------
# 10. EXPORTACIÓN DE TABLAS
# ------------------------------------------------------------

print("\n=== 3.41.10 EXPORTACIÓN FINAL ===")

exportaciones_341 = {
    "tabla_maestra_oulad_341.csv": tabla_maestra_341,
    "robustez_oulad_341.csv": tabla_robustez_341,
    "retiros_resumen_oulad_341.csv": tabla_retiros_341,
    "modelos_desarrollo_oulad_341.csv": tabla_desarrollo_341,
    "optuna_desarrollo_oulad_341.csv": tabla_optuna_341,
    "inventario_evidencia_oulad_341.csv": tabla_evidencia_341,
    "fuentes_auditadas_oulad_341.csv": tabla_inventario_341,
}

for nombre, tabla in exportaciones_341.items():

    ruta = CARPETA_341 / nombre

    tabla.to_csv(
        ruta,
        index=False
    )

    assert ruta.exists()

    print(nombre, ":", len(tabla), "filas")

# Manifest de trazabilidad.
manifest_341 = {
    "actividad": "3.41",
    "protocolo_sha256": hash_observado_341,
    "modelos_reentrenados": 0,
    "predicciones_nuevas": 0,
    "umbrales_modificados": 0,
    "fuentes": inventario_archivos_341,
    "archivos_exportados": (
        list(exportaciones_341.keys())
        + [
            ruta_resumen_341.name,
            ruta_figura_1_341.name,
            ruta_figura_2_341.name,
        ]
    ),
}

ruta_manifest_341 = (
    CARPETA_341 / "manifest_evidencia_oulad_341.json"
)

ruta_manifest_341.write_text(
    json.dumps(
        manifest_341,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

# ------------------------------------------------------------
# 11. CONTROLES FINALES
# ------------------------------------------------------------

print("\n=== 3.41.11 CONTROLES FINALES ===")

assert len(tabla_maestra_341) == 4

assert set(
    tabla_maestra_341["Semana"]
) == set(SEMANAS_341)

assert len(tabla_robustez_341) == 4
assert len(tabla_retiros_341) == 4

assert len(tabla_desarrollo_341) == 4
assert len(tabla_optuna_341) == 5

assert hashlib.sha256(
    ruta_protocolo_341.read_bytes()
).hexdigest() == hash_esperado_341

assert ruta_resumen_341.exists()
assert ruta_manifest_341.exists()

print("Evidencia principal: consolidada")
print("Evidencia complementaria: consolidada")
print("Experimentos exploratorios: documentados")
print("Protocolo: sin modificaciones")
print("Modelos reentrenados: 0")
print("Predicciones nuevas: 0")
print("Umbrales modificados: 0")
print("Tablas exportadas:", len(exportaciones_341))
print("Figuras exportadas: 2")
print("Resumen académico: generado")
print("Manifest reproducible: generado")

print("\n=== ACTIVIDAD 3.41 FINALIZADA ===")



# Actividad 3.42. Auditoría de cumplimiento y preparación de la entrega

## 3.42.1. Objetivo

Verificar el grado de cobertura de los criterios establecidos en la rúbrica del proyecto de Machine Learning, identificar requisitos pendientes y preparar la estructura definitiva del notebook y su repositorio GitHub.

## 3.42.2. Alcance

La evaluación científica principal ha concluido. El modelo definitivo es una Regresión Logística con características longitudinales, optimizada mediante GridSearchCV y evaluada en un conjunto independiente separado por estudiante.

Los resultados del Experimento A (estático) y del Experimento B (longitudinal) constituyen la evidencia principal del estudio.

Los modelos Random Forest, Transformer y las configuraciones optimizadas mediante Optuna forman parte de los experimentos complementarios de desarrollo.

Esta actividad no modifica los resultados previamente obtenidos.

## 3.42.3. Requisitos por verificar

Se comprobará que el notebook presente de forma clara:

- La motivación del problema, pregunta de investigación e hipótesis.
- El análisis exploratorio, incluidos balance de clases, distribuciones, asimetría y patrones temporales.
- La construcción de características longitudinales y la selección de variables.
- Los pipelines y su tratamiento de datos faltantes.
- La optimización de hiperparámetros, curvas de aprendizaje y validación.
- La comparación estadística entre técnicas.
- Las métricas de clasificación exigidas por la rúbrica.
- La visualización t-SNE bidimensional.
- Las conclusiones sustentadas por resultados.
- Las referencias bibliográficas y de código en formato IEEE.
- Los prompts empleados con herramientas de inteligencia artificial.
- La documentación del repositorio GitHub.

## 3.42.4. Consideraciones metodológicas

La comparación principal fue establecida antes de la evaluación independiente y no será modificada.

Las métricas complementarias faltantes podrán calcularse a partir de predicciones existentes, sin necesidad de reentrenar.

La visualización t-SNE, en caso de estar pendiente, se realizará exclusivamente con datos de desarrollo. Será una herramienta exploratoria y no participará en la selección final del modelo.

La entrega deberá organizarse como un notebook ejecutable y comprensible, evitando duplicaciones innecesarias.

## 3.42.5. Resultados esperados

1. Matriz de auditoría de requisitos.
2. Tabla complementaria de métricas de clasificación sobre el test ya evaluado.
3. Índice recomendado para el notebook.
4. Borrador de README para GitHub.
5. Identificación explícita de requisitos todavía pendientes.


In [ ]:

# ============================================================
# ACTIVIDAD 3.42
# AUDITORÍA DE RÚBRICA Y PREPARACIÓN DE ENTREGA
#
# No se reentrenan modelos.
# No se modifican umbrales.
# Se reutilizan predicciones independientes existentes.
#
# Asistencia IA: ChatGPT, octubre de 2026.
# Prompt:
# "Auditar el cumplimiento de una rúbrica de
# Machine Learning, calcular métricas complementarias
# desde predicciones existentes y generar un índice
# de notebook y un borrador de README de GitHub."
# ============================================================

from pathlib import Path
import hashlib
import numpy as np
import pandas as pd

from IPython.display import display
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    accuracy_score
)

CARPETA_342 = Path("/content")
SEMANAS_342 = [2, 4, 6, 8]

print("=== 3.42.1 RECUPERACIÓN ===")

rutas_342 = {
    "predicciones":
        CARPETA_342 / "predicciones_test_oulad_339.csv",
    "metricas":
        CARPETA_342 / "metricas_test_oulad_339.csv",
    "tabla_maestra":
        CARPETA_342 / "tabla_maestra_oulad_341.csv",
    "resumen":
        CARPETA_342 / "sintesis_cientifica_oulad_341.md",
    "manifest":
        CARPETA_342 / "manifest_evidencia_oulad_341.json",
    "protocolo":
        CARPETA_342 / "protocolo_oulad_337.json",
}

for nombre, ruta in rutas_342.items():
    assert ruta.exists(), f"Archivo no encontrado: {ruta}"
    print(f"{nombre}: recuperado")

pred_342 = pd.read_csv(rutas_342["predicciones"])
metricas_342 = pd.read_csv(rutas_342["metricas"])
maestra_342 = pd.read_csv(rutas_342["tabla_maestra"])

assert len(metricas_342) == 8
assert len(maestra_342) == 4
assert len(pred_342) == 43162

assert not pred_342.duplicated([
    "Semana",
    "Experimento",
    "code_module",
    "code_presentation",
    "id_student",
]).any()

# ------------------------------------------------------------
# 2. VERIFICACIÓN DEL PROTOCOLO
# ------------------------------------------------------------

print("\n=== 3.42.2 PROTOCOLO ===")

sha_esperado_342 = (
    "283bbddb0c56a615dde6a8a1201e149dfd7020f4672d7d18267e4c93ad880625"
)

sha_actual_342 = hashlib.sha256(
    rutas_342["protocolo"].read_bytes()
).hexdigest()

assert sha_actual_342 == sha_esperado_342

print("SHA-256:", sha_actual_342)
print("Integridad: verificada")

# ------------------------------------------------------------
# 3. MATRIZ DE AUDITORÍA
# ------------------------------------------------------------

print("\n=== 3.42.3 AUDITORÍA DE RÚBRICA ===")

criterios_342 = [
    (
        "Introducción", 0.10,
        "Contexto, objetivos e hipótesis",
        "Revisar presentación final"
    ),
    (
        "EDA", 0.60,
        "Actividades 2.1-2.7",
        "Revisar balance, skewness y estacionalidad"
    ),
    (
        "Feature Extraction", 0.25,
        "Variables temporales y secuencias",
        "Documentar lags y disponibilidad temporal"
    ),
    (
        "Feature Selection", 0.25,
        "Pipelines con filtro",
        "Verificar explicación del método"
    ),
    (
        "Optimización", 3.50,
        "GridSearchCV, Optuna y curvas",
        "Seleccionar figuras para presentación"
    ),
    (
        "Comparación estadística", 3.50,
        "CV anidada y contrastes",
        "Explicar dependencia de folds y Holm"
    ),
    (
        "Pipelines", 0.50,
        "sklearn Pipeline y Transformer modular",
        "Verificar reproducibilidad"
    ),
    (
        "t-SNE", 1.00,
        "Sin evidencia confirmada",
        "PENDIENTE: verificar o implementar"
    ),
    (
        "Idioma", 0.10,
        "Notebook en español",
        "Revisión editorial"
    ),
    (
        "Referencias IEEE", 0.10,
        "Fuentes y prompts",
        "PENDIENTE: auditar formato y cobertura"
    ),
    (
        "Conclusiones", 0.10,
        "Síntesis de actividades 3.39-3.41",
        "Revisar coherencia con resultados"
    ),
]

auditoria_342 = pd.DataFrame(
    criterios_342,
    columns=[
        "Criterio",
        "Puntaje_maximo",
        "Evidencia_identificada",
        "Accion_pendiente",
    ]
)

assert np.isclose(
    auditoria_342["Puntaje_maximo"].sum(),
    10.0
)

display(auditoria_342)

print(
    "Puntaje total de la rúbrica:",
    auditoria_342["Puntaje_maximo"].sum()
)

print(
    "Nota: esta matriz NO constituye una "
    "calificación ni una verificación automática "
    "del contenido completo del notebook."
)

# ------------------------------------------------------------
# 4. MÉTRICAS COMPLEMENTARIAS
# ------------------------------------------------------------

print("\n=== 3.42.4 MÉTRICAS COMPLEMENTARIAS ===")

filas_metricas_342 = []

for semana in SEMANAS_342:

    for exp in ["A", "B"]:

        sub = pred_342.loc[
            pred_342["Semana"].eq(semana)
            & pred_342["Experimento"].eq(exp)
        ].copy()

        y = sub["y_real"].to_numpy(dtype=int)
        y_pred = sub["y_pred"].to_numpy(dtype=int)

        prob = sub[
            "prob_adverso"
        ].to_numpy(dtype=float)

        assert set(np.unique(y)) == {0, 1}
        assert set(np.unique(y_pred)).issubset({0, 1})
        assert np.isfinite(prob).all()
        assert ((prob >= 0) & (prob <= 1)).all()

        # Probabilidades de ambas clases:
        # columna 0 = favorable
        # columna 1 = adversa
        probs_2clases = np.column_stack([
            1 - prob,
            prob
        ])

        etiquetas_2clases = np.column_stack([
            1 - y,
            y
        ])

        # AUC macro OvR binaria:
        # promedio de la AUC por clase.
        auc_macro = np.mean([
            roc_auc_score(y, prob),
            roc_auc_score(1 - y, 1 - prob)
        ])

        # AUC micro OvR:
        # se agregan decisiones binarizadas
        # de ambas clases para la evaluación.
        auc_micro = roc_auc_score(
            etiquetas_2clases.ravel(),
            probs_2clases.ravel()
        )

        filas_metricas_342.append({
            "Semana": semana,
            "Experimento": exp,
            "Matriculas": len(sub),
            "Accuracy": accuracy_score(
                y, y_pred
            ),
            "Precision_macro": precision_score(
                y, y_pred,
                average="macro",
                zero_division=0
            ),
            "Precision_weighted": precision_score(
                y, y_pred,
                average="weighted",
                zero_division=0
            ),
            "Recall_macro": recall_score(
                y, y_pred,
                average="macro",
                zero_division=0
            ),
            "Recall_adverso": recall_score(
                y, y_pred,
                pos_label=1
            ),
            "F1_macro": f1_score(
                y, y_pred,
                average="macro"
            ),
            "F1_weighted": f1_score(
                y, y_pred,
                average="weighted"
            ),
            "ROC_AUC_binaria": roc_auc_score(
                y, prob
            ),
            "ROC_AUC_macro_OvR": auc_macro,
            "ROC_AUC_micro_OvR": auc_micro
        })

metricas_extra_342 = pd.DataFrame(
    filas_metricas_342
)

display(
    metricas_extra_342.round(5)
)

# Reproducir las métricas originales.
for _, fila in metricas_extra_342.iterrows():

    original = metricas_342.loc[
        metricas_342["Semana"].eq(fila["Semana"])
        & metricas_342["Experimento"].eq(
            fila["Experimento"]
        )
    ].iloc[0]

    assert np.isclose(
        fila["F1_macro"],
        original["F1_macro"],
        atol=1e-8
    )

    assert np.isclose(
        fila["ROC_AUC_binaria"],
        original["ROC_AUC"],
        atol=1e-8
    )

print("Métricas originales: reproducidas")
print("F1 weighted: calculado")
print("AUC macro/micro OvR: calculadas")
print("Predicciones nuevas: 0")

# ------------------------------------------------------------
# 5. ÍNDICE PROPUESTO PARA EL NOTEBOOK
# ------------------------------------------------------------

print("\n=== 3.42.5 ÍNDICE ACADÉMICO ===")

secciones_342 = [
    (
        "1", "Introducción",
        "Contexto, problema, hipótesis y objetivos"
    ),
    (
        "2", "Datos y metodología",
        "OULAD, unidad de análisis y etiquetas"
    ),
    (
        "3", "Análisis exploratorio",
        "Calidad, clases, distribuciones y tiempo"
    ),
    (
        "4", "Ingeniería de características",
        "Variables estáticas, VLE y entregas"
    ),
    (
        "5", "Diseño experimental",
        "Cohortes, ventanas, grupos y test reservado"
    ),
    (
        "6", "Preprocesamiento y selección",
        "Pipelines, imputación y filtros"
    ),
    (
        "7", "Modelos y optimización",
        "LR, RF, GridSearchCV y Optuna"
    ),
    (
        "8", "Validación y comparación estadística",
        "CV anidada, métricas y contrastes"
    ),
    (
        "9", "Experimento Transformer",
        "Arquitectura, secuencias y resultados"
    ),
    (
        "10", "Visualización t-SNE",
        "Representación bidimensional del espacio"
    ),
    (
        "11", "Evaluaciones complementarias",
        "Calibración, subgrupos y alertas"
    ),
    (
        "12", "Evaluación independiente",
        "Resultados definitivos A frente a B"
    ),
    (
        "13", "Discusión",
        "Interpretación, robustez y limitaciones"
    ),
    (
        "14", "Conclusiones",
        "Hallazgos y futuras investigaciones"
    ),
    (
        "15", "Referencias",
        "Formato IEEE y atribución de código"
    ),
    (
        "16", "Anexos",
        "Auditorías y experimentos extensos"
    ),
]

indice_342 = pd.DataFrame(
    secciones_342,
    columns=["Numero", "Seccion", "Contenido"]
)

display(indice_342)

texto_indice_342 = (
    "# Índice académico recomendado\n\n"
    + "\n".join(
        f"## {fila.Numero}. {fila.Seccion}\n"
        f"{fila.Contenido}\n"
        for fila in indice_342.itertuples()
    )
    + "\n"
    + "Nota: este índice es una propuesta narrativa. "
    + "No es necesario mover las celdas de código "
    + "si hacerlo rompe sus dependencias.\n"
)

ruta_indice_342 = (
    CARPETA_342 / "indice_notebook_oulad_342.md"
)

ruta_indice_342.write_text(
    texto_indice_342,
    encoding="utf-8"
)

print("Índice guardado:", ruta_indice_342)

# ------------------------------------------------------------
# 6. BORRADOR DE README PARA GITHUB
# ------------------------------------------------------------

print("\n=== 3.42.6 README ===")

filas_resultado = []

for _, fila in maestra_342.iterrows():

    filas_resultado.append(
        f"| {int(fila['Semana'])} "
        f"| {fila['F1_A']:.5f} "
        f"| {fila['F1_B']:.5f} "
        f"| {fila['Delta_F1']:+.5f} |"
    )

tabla_resultado_md = "\n".join([
    "| Semana | F1 estático | F1 longitudinal | Δ F1 |",
    "|---|---:|---:|---:|"
] + filas_resultado)

readme_342 = f"""# Predicción temprana de riesgo académico con OULAD

## Descripción

Proyecto de Machine Learning para estudiar el aporte de
características longitudinales del comportamiento estudiantil
a la predicción temprana de resultados académicos adversos.

## Pregunta de investigación

¿En qué medida la incorporación de características
longitudinales mejora la predicción temprana frente al uso
exclusivo de variables estáticas?

## Datos

Fuente: Open University Learning Analytics Dataset (OULAD).

La unidad de análisis es una matrícula de estudiante
en un módulo y una presentación.

La etiqueta adversa combina Fail y Withdrawn.
Withdrawn no equivale necesariamente a abandono universitario.

## Diseño experimental

- Semanas de predicción: 2, 4, 6 y 8.
- Experimento A: características estáticas.
- Experimento B: características estáticas y longitudinales.
- Modelo principal: Regresión Logística.
- Optimización principal: GridSearchCV.
- Validación: agrupada por estudiante.
- Métrica principal: F1 macro.
- Prueba independiente: estudiantes no incluidos en desarrollo.

Se investigaron adicionalmente Random Forest,
Transformer y Optuna mediante análisis de desarrollo.

## Resultados independientes

{tabla_resultado_md}

La incorporación de información longitudinal mejoró
el F1 macro en las cuatro ventanas temporales.

## Reproducibilidad

El proyecto se desarrolló en Google Colab.

Consultar el notebook para las dependencias, las semillas
aleatorias, las rutas de datos y los procedimientos de
preprocesamiento, entrenamiento y evaluación.

El entrenamiento de redes neuronales puede requerir
tiempo adicional según el hardware disponible.

Antes de publicar este repositorio, comprobar que se
incluyan el notebook, instrucciones de ejecución,
fuentes de datos y archivos necesarios.

No publicar credenciales ni tokens de acceso.

## Limitaciones

Los resultados proceden de OULAD y no representan
una validación externa entre universidades.

La detección de retiros es retrospectiva y no equivale
a demostrar la efectividad de una intervención preventiva.

## Referencias

Incorporar aquí las referencias IEEE verificadas
utilizadas en el notebook.

## Uso de herramientas de IA

Se utilizó asistencia de ChatGPT durante la elaboración
y revisión de código y documentación. Los prompts
pertinentes deberán permanecer registrados en las
celdas de código correspondientes.
"""

ruta_readme_342 = (
    CARPETA_342 / "README_oulad_borrador_342.md"
)

ruta_readme_342.write_text(
    readme_342,
    encoding="utf-8"
)

print("README borrador:", ruta_readme_342)

# ------------------------------------------------------------
# 7. EXPORTACIÓN
# ------------------------------------------------------------

print("\n=== 3.42.7 EXPORTACIÓN ===")

ruta_auditoria_342 = (
    CARPETA_342 / "auditoria_rubrica_342.csv"
)

ruta_metricas_342 = (
    CARPETA_342 / "metricas_complementarias_342.csv"
)

auditoria_342.to_csv(
    ruta_auditoria_342,
    index=False
)

metricas_extra_342.to_csv(
    ruta_metricas_342,
    index=False
)

for ruta in [
    ruta_auditoria_342,
    ruta_metricas_342,
    ruta_indice_342,
    ruta_readme_342,
]:
    assert ruta.exists()
    print("Archivo creado:", ruta.name)

# ------------------------------------------------------------
# 8. CONTROLES FINALES
# ------------------------------------------------------------

print("\n=== 3.42.8 CONTROLES FINALES ===")

assert len(auditoria_342) == 11
assert len(metricas_extra_342) == 8
assert len(indice_342) == 16

assert (
    metricas_extra_342["Semana"].nunique() == 4
)

assert np.isclose(
    auditoria_342["Puntaje_maximo"].sum(),
    10.0
)

assert hashlib.sha256(
    rutas_342["protocolo"].read_bytes()
).hexdigest() == sha_esperado_342

print("Rúbrica: 11 criterios auditados")
print("Métricas adicionales: 8 configuraciones")
print("Índice académico: preparado")
print("README: borrador preparado")
print("Protocolo: intacto")
print("Modelos entrenados: 0")
print("Predicciones generadas: 0")
print("Test: no reutilizado para selección de modelos")

print("\n=== ACTIVIDAD 3.42 FINALIZADA ===")



### Conclusión de la Actividad 3.42

Se completó la auditoría preliminar de los once criterios de la rúbrica y se verificó la integridad del protocolo experimental mediante SHA-256. La recuperación de los resultados de prueba permitió reproducir las métricas previamente publicadas sin modificar los modelos ni sus predicciones.

Se incorporaron las métricas complementarias de Precision macro y weighted, Recall macro, F1 weighted y ROC-AUC macro/micro One-vs-Rest. Estos resultados confirman que la mejora del Experimento B no se limita al F1 macro, sino que se observa también en diferentes indicadores de clasificación.

En la semana 8, el modelo longitudinal alcanzó un F1 weighted de 0,72822, frente a 0,62264 del modelo estático. Asimismo, el ROC-AUC binario y macro fue de 0,79225 para B y de 0,66289 para A.

Se preparó la estructura académica del notebook y un borrador del README para GitHub. Quedan pendientes la comprobación de la visualización t-SNE y la revisión editorial y bibliográfica final.

La actividad no realizó reentrenamiento, optimización adicional ni nuevas predicciones. Por tanto, no modificó las conclusiones obtenidas mediante la evaluación independiente.



# Actividad 3.43. Visualización del espacio de características mediante t-SNE

## Objetivo

Explorar visualmente la estructura local del espacio de características empleado en los experimentos A y B mediante t-distributed Stochastic Neighbor Embedding (t-SNE), una técnica no lineal de reducción de dimensionalidad.

## Metodología

Se utilizarán exclusivamente las matrículas del conjunto de desarrollo correspondiente a la semana 8.

Para evitar la sobrerrepresentación de estudiantes con múltiples matrículas, se seleccionará una matrícula por estudiante. Posteriormente, se construirá una muestra reproducible y equilibrada entre resultados favorables y adversos, empleando las mismas observaciones para ambos experimentos.

Cada representación será sometida a:

1. Tratamiento exploratorio de datos faltantes.
2. Codificación de variables categóricas.
3. Escalamiento de variables numéricas.
4. Reducción preliminar mediante PCA.
5. Proyección bidimensional mediante t-SNE.

Los puntos se colorearán según el resultado académico binario: favorable (Pass o Distinction) y adverso (Fail o Withdrawn).

## Precauciones de interpretación

t-SNE intenta preservar relaciones locales entre observaciones y no reproduce fielmente todas las distancias globales.

Las coordenadas de las proyecciones A y B no son directamente comparables entre sí. Una aparente separación visual tampoco demuestra, por sí sola, superioridad predictiva.

Debido al muestreo equilibrado, la densidad de puntos no representa la prevalencia real de resultados adversos.

La validez predictiva seguirá fundamentándose en la validación cruzada y en la evaluación independiente ya completadas.

## Integridad experimental

No se utilizarán datos del conjunto de prueba independiente. No se entrenarán nuevos clasificadores, ni se modificarán modelos, hiperparámetros, umbrales o resultados.

**Referencia del método:** t-SNE fue propuesto por van der Maaten y Hinton [4] y se ejecuta con la implementación de scikit-learn [2]. La proyección no es supervisada y sus distancias globales no tienen interpretación métrica directa.


In [ ]:
# ============================================================
# ACTIVIDAD 3.43.A - RECUPERACIÓN SIN RECÁLCULO DE COHORTES
# Se usan los snapshots de la Actividad 3.4.
# ============================================================
from pathlib import Path
if 'experimentos_oulad' not in globals():
    recuperar_estado_oulad('experimentos_oulad')
if 'experimentos_oulad' not in globals():
    raise RuntimeError(
        'Falta snapshot de experimentos_oulad. Ejecuta las celdas originales '
        'de preparación hasta Actividad 3.4, que ahora guardan este snapshot en Drive.'
    )
assert experimentos_oulad[8]['X_A_train'].shape==(21164,12)
assert experimentos_oulad[8]['X_B_train'].shape==(21164,48)
print('=== 3.43.A: MATRICES ORIGINALES RECUPERADAS ===')
print('A semana 8:', experimentos_oulad[8]['X_A_train'].shape)
print('B semana 8:', experimentos_oulad[8]['X_B_train'].shape)
print('Origen: desarrollo; no se reconstruyen features manualmente.')


### 3.43.B. t-SNE sobre las matrices originales de desarrollo

La visualización usa **exactamente las matrices originales** `experimentos_oulad[8]["X_A_train"]` y `experimentos_oulad[8]["X_B_train"]`, almacenadas en los snapshots de la Actividad 3.4. No se reconstruyen a mano las variables VLE ni las entregas.

Se extraen 2.000 matrículas de desarrollo, **1.000 por clase y una sola matrícula por estudiante**. Ambas representaciones utilizan los mismos registros, con preprocesamiento exploratorio sin etiquetas, PCA y proyección t-SNE bidimensional. Las etiquetas solo colorean los puntos; las proyecciones no son estimaciones predictivas ni conservan las distancias globales.

La figura se guarda localmente en `/content` y se respalda inmediatamente en Google Drive. No se ejecuta ningún reentrenamiento ni se evalúa el conjunto de prueba.


In [ ]:
# ============================================================
# ACTIVIDAD 3.43.B - t-SNE SOBRE MATRICES ORIGINALES
# Solo desarrollo; misma muestra para A y B.
# Ningún clasificador se vuelve a entrenar.
# ============================================================
# Asistencia IA: ChatGPT, octubre de 2026.
# Prompt: "Visualizar en t-SNE el espacio estático A y longitudinal B
# de OULAD semana 8 con matrices de desarrollo originales persistidas."
import inspect, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.manifold import TSNE
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

if 'experimentos_oulad' not in globals():
    recuperar_estado_oulad('experimentos_oulad')
assert 'experimentos_oulad' in globals()
d=experimentos_oulad[8]
XA=d['X_A_train'].reset_index(drop=True)
XB=d['X_B_train'].reset_index(drop=True)
y=np.asarray(d['y_train']).astype(int).ravel()
grupos=np.asarray(d['grupos_train']).ravel()
assert XA.shape==(21164,12) and XB.shape==(21164,48)
assert len(y)==len(grupos)==len(XA)
assert set(np.unique(y))=={0,1}

# Una matrícula por estudiante; equilibrar resultados SOLO para la figura.
rng=np.random.default_rng(42)
permutacion=rng.permutation(len(y))
no_repetidas=pd.DataFrame({'idx':permutacion,'id':grupos[permutacion], 'clase':y[permutacion]}).drop_duplicates('id')
seleccion=[]
for clase in (0,1):
    disponibles=no_repetidas.loc[no_repetidas['clase']==clase,'idx'].to_numpy()
    assert len(disponibles)>=1000
    seleccion.extend(rng.choice(disponibles,1000,replace=False))
seleccion=np.asarray(seleccion,dtype=int)
rng.shuffle(seleccion)
assert len(np.unique(grupos[seleccion]))==2000
X_A_343=XA.iloc[seleccion].reset_index(drop=True)
X_B_343=XB.iloc[seleccion].reset_index(drop=True)
y_343=y[seleccion]
assert X_A_343.shape==(2000,12) and X_B_343.shape==(2000,48)

# Preprocesamiento no supervisado diseñado para EXPLORACIÓN,
# ajustado exclusivamente a estas observaciones de desarrollo.
def proyeccion_343(X,etiqueta):
    X=X.copy()
    num=X.select_dtypes(include=np.number).columns.tolist()
    cat=[c for c in X.columns if c not in num]
    for c in num:
        X[c]=pd.to_numeric(X[c],errors='coerce').replace([np.inf,-np.inf],np.nan)
    for c in cat:
        X[c]=X[c].astype('string').fillna('No informado').astype(str)
    # Ignorar columnas completamente vacías en la muestra.
    num=[c for c in num if not X[c].isna().all()]
    cat=[c for c in cat if not X[c].isna().all()]
    try: encoder=OneHotEncoder(handle_unknown='ignore',sparse_output=False)
    except TypeError: encoder=OneHotEncoder(handle_unknown='ignore',sparse=False)
    transformaciones=[]
    if num: transformaciones.append(('num',Pipeline([('imputar',SimpleImputer(strategy='median')),('escalar',StandardScaler())]),num))
    if cat: transformaciones.append(('cat',encoder,cat))
    pre=ColumnTransformer(transformaciones,sparse_threshold=0)
    M=np.asarray(pre.fit_transform(X),dtype=float)
    assert np.isfinite(M).all()
    pca=PCA(n_components=min(25,M.shape[1],M.shape[0]-1),random_state=42)
    Z=pca.fit_transform(M)
    kwargs=dict(n_components=2,random_state=42,perplexity=35,learning_rate='auto',init='pca')
    kwargs['max_iter' if 'max_iter' in inspect.signature(TSNE).parameters else 'n_iter']=1000
    inicio=time.perf_counter()
    P=TSNE(**kwargs).fit_transform(Z)
    print(f'{etiqueta}: original={X.shape}; codificada={M.shape}; varianza PCA={pca.explained_variance_ratio_.sum():.3f}; t-SNE={time.perf_counter()-inicio:.1f}s')
    return P

print('=== 3.43.B: t-SNE ORIGINAL A VS B ===')
print('Muestra de desarrollo: 2.000 matrículas de 2.000 estudiantes')
pa=proyeccion_343(X_A_343,'A')
pb=proyeccion_343(X_B_343,'B')
fig,axs=plt.subplots(1,2,figsize=(15,6),constrained_layout=True)
for ax,p,titulo in zip(axs,(pa,pb),('A: características estáticas','B: características longitudinales')):
    for cls,nombre,color in [(0,'Favorable','#277DA1'),(1,'Adverso','#F3722C')]:
        m=y_343==cls
        ax.scatter(p[m,0],p[m,1],color=color,s=12,alpha=.55,linewidths=0,label=nombre)
    ax.set(title=titulo,xlabel='t-SNE 1',ylabel='t-SNE 2')
    ax.legend();ax.grid(alpha=.15)
fig.suptitle('OULAD - Semana 8: t-SNE exploratorio con matrices de desarrollo originales')
archivo_fig=Path('/content/tsne_comparativo_oulad_343.png')
fig.savefig(archivo_fig,dpi=250,bbox_inches='tight')
plt.show()
coordenadas_343=pd.DataFrame(dict(resultado_binario=y_343,tsne_A_1=pa[:,0],tsne_A_2=pa[:,1],tsne_B_1=pb[:,0],tsne_B_2=pb[:,1]))
archivo_csv=Path('/content/coordenadas_tsne_oulad_343.csv')
coordenadas_343.to_csv(archivo_csv,index=False)
respaldar_checkpoint_oulad(archivo_fig)
respaldar_checkpoint_oulad(archivo_csv)
assert set(np.unique(y_343))=={0,1}
print('=== 3.43 FINALIZADA: desarrollo original, ningún acceso al test ===')
print('El color usa etiquetas, pero la proyección NO está supervisada.')
print('Una separación gráfica no demuestra superioridad predictiva.')



### 3.43.C. Interpretación y conclusión de la visualización t-SNE

Se aplicó t-distributed Stochastic Neighbor Embedding (t-SNE) a las matrices originales de desarrollo correspondientes a la semana 8 de OULAD, con el propósito de explorar la estructura del espacio de características de los experimentos estático (A) y longitudinal (B).

Se seleccionó una muestra reproducible de 2.000 matrículas pertenecientes a 2.000 estudiantes diferentes, equilibrada entre resultados académicos favorables y adversos. Las mismas observaciones fueron utilizadas en ambas representaciones.

El Experimento A incluyó 12 variables originales, expandidas a 51 características codificadas. El Experimento B incorporó 48 variables originales, expandidas a 87 características codificadas.

Previamente a t-SNE se aplicó PCA, conservando el 90,5 % de la varianza en A y el 87,5 % en B.

**Observaciones visuales**

La proyección del Experimento A presenta agrupaciones locales reconocibles, pero con una mezcla considerable de estudiantes con resultados favorables y adversos.

La proyección del Experimento B muestra una distribución más continua y una concentración visual relativamente mayor de resultados adversos hacia uno de sus extremos y de resultados favorables hacia el otro. No obstante, existe un solapamiento sustancial entre ambas clases.

Estas observaciones son compatibles con la presencia de información adicional en las características longitudinales, pero no permiten cuantificar por sí solas el rendimiento predictivo ni establecer una separación estadísticamente significativa entre clases.

**Limitaciones**

t-SNE es una técnica no supervisada de visualización que busca preservar relaciones locales. Las distancias globales, los tamaños de agrupaciones y la geometría relativa entre proyecciones deben interpretarse con precaución.

La muestra fue equilibrada artificialmente, por lo que la densidad de puntos no representa la prevalencia real de resultados adversos.

Las coordenadas de los experimentos A y B no son directamente comparables. Además, la reducción preliminar mediante PCA conservó proporciones de varianza diferentes en ambos espacios.

**Integridad experimental**

La visualización se realizó exclusivamente con matrices originales de desarrollo, sin utilizar registros del conjunto independiente.

Las etiquetas académicas se emplearon únicamente para el muestreo y la representación por colores, sin intervenir en el ajuste de t-SNE.

No se reentrenaron clasificadores ni se modificaron los modelos definitivos.

**Conclusión**

La visualización t-SNE permite explorar diferencias descriptivas entre las representaciones estática y longitudinal de los estudiantes, mostrando patrones de distribución compatibles con la incorporación de información conductual.

La evidencia cuantitativa de la superioridad predictiva del Experimento B procede de la validación cruzada y de la evaluación independiente realizadas anteriormente.

Con esta actividad se completa el componente de visualización bidimensional solicitado por la rúbrica del proyecto.



## Actividad 3.43.D — Análisis de sensibilidad y robustez interpretativa de t-SNE

### Justificación

La Actividad 3.43.B utilizó una muestra equilibrada de 2.000 matrículas para ilustrar la estructura local de los espacios estático y longitudinal de OULAD en la semana 8. Aunque esta muestra cumple el propósito de visualización, no permite apreciar toda la diversidad de los perfiles estudiantiles y altera deliberadamente la prevalencia natural de las etiquetas.

Como **complemento exploratorio de robustez interpretativa**, se amplía la visualización a **10.000 estudiantes** y a la **totalidad de 19.233 estudiantes únicos del conjunto de desarrollo** de la semana 8 (una matrícula por estudiante).

No se incluyen los registros del test independiente reservado ni se modifica la partición original de entrenamiento.

### Objetivo

Evaluar si los patrones visuales observados inicialmente en una muestra reducida se mantienen, cambian o desaparecen al incrementar la cantidad de estudiantes representados, conservando la distribución natural de las clases académicas.

Este análisis busca fortalecer la interpretación exploratoria de los datos, no producir nueva evidencia sobre la superioridad predictiva de los modelos.

### Metodología

Se utilizan las matrices originales:

- `experimentos_oulad[8]["X_A_train"]`: 12 variables estáticas.
- `experimentos_oulad[8]["X_B_train"]`: 48 variables estáticas y longitudinales.

No se reconstruyen las características ni se modifican los modelos originales.

Se selecciona una matrícula por estudiante mediante un procedimiento reproducible con semilla 42.

La muestra de 10.000 estudiantes constituye un subconjunto de la población completa de 19.233 estudiantes únicos de desarrollo.

A diferencia de la Actividad 3.43.B, **no se realiza balanceo artificial entre las clases**. Se conserva la proporción observada de resultados favorables y adversos después de seleccionar una matrícula por estudiante.

Para cada tamaño de muestra y representación se aplica:

1. Imputación exploratoria de valores faltantes.
2. Estandarización de variables numéricas.
3. Codificación One-Hot de variables categóricas.
4. Reducción dimensional preliminar mediante PCA, con un máximo de 25 componentes.
5. Proyección bidimensional mediante t-SNE, utilizando `perplexity=35`, 1.000 iteraciones y semilla aleatoria 42.

Las cuatro proyecciones son ajustadas independientemente. Se documenta la proporción de varianza conservada mediante PCA y el tiempo de ejecución.

Cada resultado se almacena primero en `/content` y posteriormente se respalda en Google Drive, permitiendo recuperar los cálculos después de una desconexión.

### Análisis complementario de cuatro resultados académicos

Cuando se dispone de las etiquetas originales, se representa la proyección completa utilizando cuatro categorías:

- `Pass`
- `Distinction`
- `Fail`
- `Withdrawn`

Estas categorías se utilizan exclusivamente para colorear los puntos, sin intervenir en el ajuste de PCA o t-SNE.

### Consideraciones metodológicas

Esta actividad constituye un **análisis de sensibilidad de la interpretación visual frente al tamaño muestral**.

t-SNE preserva principalmente relaciones locales, por lo que las coordenadas, distancias globales y dimensiones aparentes de los grupos no deben interpretarse como equivalentes entre proyecciones independientes.

Una mayor separación visual no demuestra, por sí sola, una mejora en la capacidad predictiva.

La incorporación de una matrícula por estudiante modifica la unidad de visualización respecto a la totalidad de matrículas y puede alterar sus proporciones originales.

La ampliación de la muestra mejora la cobertura de los perfiles de desarrollo, pero no equivale a una validación externa.

### Integridad experimental

La actividad no utiliza el conjunto de prueba independiente.

No se entrenan nuevos clasificadores, no se modifican hiperparámetros ni umbrales y no se altera el protocolo congelado.

Las conclusiones predictivas principales siguen fundamentándose en los resultados independientes de las Actividades 3.39 a 3.41.

### Resultados esperados

Se generarán:

1. Tabla de composición y cobertura de las muestras.
2. Cuatro proyecciones t-SNE: A y B con 10.000 y 19.233 estudiantes.
3. Visualización opcional con las cuatro categorías académicas originales.
4. Auditoría técnica de dimensiones codificadas, varianza de PCA y tiempos.
5. Archivos y checkpoints respaldados en Google Drive.

**Alcance bibliográfico:** esta sensibilidad visual continúa siendo exploratoria según la naturaleza de t-SNE [4]. Aumentar el número de observaciones no transforma la proyección en una prueba estadística de discriminación.


In [ ]:

# ============================================================
# ACTIVIDAD 3.43.D.1
# ROBUSTEZ INTERPRETATIVA DE t-SNE - 10.000 ESTUDIANTES
#
# Prepara funciones compartidas para la Actividad 3.43.D.2.
# Utiliza exclusivamente matrices de desarrollo originales.
# No entrena clasificadores ni utiliza el test independiente.
#
# Asistencia IA: ChatGPT, octubre de 2026.
# Prompt: analizar la sensibilidad de t-SNE a una muestra
# de 10.000 estudiantes, manteniendo una matrícula por
# estudiante, prevalencia natural de clases y respaldos
# persistentes en Google Drive.
# ============================================================

import os
import json
import time
import hashlib
import inspect
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn

from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.manifold import TSNE
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)
from IPython.display import display

SEMILLA_343D = 42

RUTA_LOCAL_343D = Path("/content")
RUTA_DRIVE_343D = (
    Path(RUTA_BASE) / "artefactos"
)

RUTA_DRIVE_343D.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# 1. RECUPERAR MATRICES ORIGINALES
# ------------------------------------------------------------

print("=== 3.43.D.1.1 MATRICES ORIGINALES ===")

if "experimentos_oulad" not in globals():
    recuperar_estado_oulad("experimentos_oulad")

assert "experimentos_oulad" in globals()

datos_343d = experimentos_oulad[8]

X_A_343d = (
    datos_343d["X_A_train"]
    .reset_index(drop=True)
)

X_B_343d = (
    datos_343d["X_B_train"]
    .reset_index(drop=True)
)

y_343d = np.asarray(
    datos_343d["y_train"],
    dtype=int
).ravel()

grupos_343d = np.asarray(
    datos_343d["grupos_train"]
).ravel()

assert X_A_343d.shape == (21164, 12)
assert X_B_343d.shape == (21164, 48)

assert len(y_343d) == len(grupos_343d) == 21164
assert set(np.unique(y_343d)) == {0, 1}

print("A:", X_A_343d.shape)
print("B:", X_B_343d.shape)
print("Estudiantes únicos:", len(np.unique(grupos_343d)))

# ------------------------------------------------------------
# 2. SELECCIONAR UNA MATRÍCULA POR ESTUDIANTE
# ------------------------------------------------------------

print("\n=== 3.43.D.1.2 MUESTREO REPRODUCIBLE ===")

rng_343d = np.random.default_rng(SEMILLA_343D)

orden_343d = rng_343d.permutation(
    len(y_343d)
)

indices_unicos_343d = (
    pd.DataFrame({
        "indice": orden_343d,
        "estudiante": grupos_343d[orden_343d]
    })
    .drop_duplicates(
        "estudiante",
        keep="first"
    )["indice"]
    .to_numpy(dtype=int)
)

N_TOTAL_343D = len(indices_unicos_343d)

assert N_TOTAL_343D == 19233

# La selección completa se conserva para la celda 2.
indices_10000_343d = indices_unicos_343d[:10000]

assert len(indices_10000_343d) == 10000

assert len(np.unique(
    grupos_343d[indices_10000_343d]
)) == 10000

y_10000_343d = y_343d[
    indices_10000_343d
]

print("Estudiantes seleccionados:", 10000)
print("Favorables:", int((y_10000_343d == 0).sum()))
print("Adversos:", int((y_10000_343d == 1).sum()))
print(
    "Porcentaje adverso:",
    round(100 * y_10000_343d.mean(), 2),
    "%"
)

print(
    "Cobertura de estudiantes:",
    round(100 * 10000 / N_TOTAL_343D, 2),
    "%"
)

# ------------------------------------------------------------
# 3. FUNCIÓN DE PREPROCESAMIENTO EXPLORATORIO
# ------------------------------------------------------------

def preparar_tsne_343d(X):

    X = X.copy()

    numericas = X.select_dtypes(
        include=np.number
    ).columns.tolist()

    categoricas = [
        columna for columna in X.columns
        if columna not in numericas
    ]

    for columna in numericas:

        X[columna] = (
            pd.to_numeric(
                X[columna],
                errors="coerce"
            )
            .replace(
                [np.inf, -np.inf],
                np.nan
            )
        )

    # Eliminar columnas completamente vacías
    # únicamente en esta representación exploratoria.
    numericas = [
        c for c in numericas
        if not X[c].isna().all()
    ]

    for columna in categoricas:

        X[columna] = (
            X[columna]
            .astype("string")
            .fillna("No informado")
            .astype(str)
        )

    try:
        encoder = OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        )
    except TypeError:
        encoder = OneHotEncoder(
            handle_unknown="ignore",
            sparse=False
        )

    transformadores = []

    if numericas:

        transformadores.append((
            "numericas",
            Pipeline([
                (
                    "imputacion",
                    SimpleImputer(strategy="median")
                ),
                (
                    "escalamiento",
                    StandardScaler()
                )
            ]),
            numericas
        ))

    if categoricas:

        transformadores.append((
            "categoricas",
            encoder,
            categoricas
        ))

    preprocesador = ColumnTransformer(
        transformadores,
        sparse_threshold=0
    )

    matriz = np.asarray(
        preprocesador.fit_transform(X),
        dtype=np.float64
    )

    assert np.isfinite(matriz).all()

    componentes = min(
        25,
        matriz.shape[1],
        len(X) - 1
    )

    pca = PCA(
        n_components=componentes,
        random_state=SEMILLA_343D
    )

    reducida = pca.fit_transform(matriz)

    varianza = float(
        pca.explained_variance_ratio_.sum()
    )

    return reducida, matriz.shape[1], varianza

# ------------------------------------------------------------
# 4. FUNCIÓN DE RESPALDO A GOOGLE DRIVE
# ------------------------------------------------------------

def respaldar_archivo_343d(ruta):

    ruta = Path(ruta)

    destino = RUTA_DRIVE_343D / ruta.name

    # Utilizar la función de persistencia
    # configurada previamente en el notebook.
    respaldar_checkpoint_oulad(ruta)

    if not destino.exists():
        raise RuntimeError(
            "No se confirmó el respaldo: "
            f"{destino}"
        )

    return destino

# ------------------------------------------------------------
# 5. FUNCIÓN REUTILIZABLE t-SNE + CHECKPOINT
# ------------------------------------------------------------

def ejecutar_tsne_343d(
    X_original,
    indices,
    experimento
):

    n = len(indices)

    X = (
        X_original
        .iloc[indices]
        .reset_index(drop=True)
    )

    nombre = (
        f"tsne_oulad_343d_"
        f"{n}_{experimento}.npz"
    )

    local = RUTA_LOCAL_343D / nombre
    drive = RUTA_DRIVE_343D / nombre

    parametros = {
        "semilla": SEMILLA_343D,
        "componentes_pca": 25,
        "perplexity": 35,
        "iteraciones": 1000,
        "sklearn": sklearn.__version__,
        "n": n,
        "experimento": experimento,
        "version": "343d_separado_v1"
    }

    # Firma que verifica que los checkpoints
    # correspondan exactamente a la muestra.
    h = hashlib.sha256()

    h.update(
        np.asarray(
            indices,
            dtype=np.int64
        ).tobytes()
    )

    h.update(
        pd.util.hash_pandas_object(
            X,
            index=False
        ).to_numpy().tobytes()
    )

    h.update(
        json.dumps(
            parametros,
            sort_keys=True
        ).encode("utf-8")
    )

    firma = h.hexdigest()

    # Recuperación del checkpoint.
    if not local.exists() and drive.exists():

        shutil.copy2(drive, local)

    if local.exists():

        with np.load(
            local,
            allow_pickle=False
        ) as cache:

            firma_guardada = str(
                cache["firma"].item()
            )

            if firma_guardada != firma:
                raise RuntimeError(
                    f"Checkpoint incompatible: {nombre}. "
                    "No se sobrescribirá automáticamente."
                )

            coords = cache[
                "coordenadas"
            ].copy()

            dimensiones = int(
                cache["dimensiones"].item()
            )

            varianza = float(
                cache["varianza"].item()
            )

            segundos = float(
                cache["segundos"].item()
            )

        assert coords.shape == (n, 2)
        assert np.isfinite(coords).all()

        if not drive.exists():
            respaldar_archivo_343d(local)

        print(
            f"{experimento}: "
            f"{n:,} estudiantes RECUPERADOS "
            "desde checkpoint."
        )

        return coords, dimensiones, varianza, segundos, True

    # --------------------------------------------------------
    # Calcular nueva proyección.
    # --------------------------------------------------------

    inicio = time.perf_counter()

    reducida, dimensiones, varianza = (
        preparar_tsne_343d(X)
    )

    args = {
        "n_components": 2,
        "perplexity": 35,
        "learning_rate": "auto",
        "init": "pca",
        "random_state": SEMILLA_343D,
        "method": "barnes_hut"
    }

    if "max_iter" in inspect.signature(
        TSNE
    ).parameters:
        args["max_iter"] = 1000
    else:
        args["n_iter"] = 1000

    print(
        f"Calculando t-SNE {experimento} "
        f"con {n:,} estudiantes...",
        flush=True
    )

    coords = TSNE(
        **args
    ).fit_transform(
        reducida
    )

    segundos = time.perf_counter() - inicio

    assert coords.shape == (n, 2)
    assert np.isfinite(coords).all()

    # Guardar localmente antes de copiar a Drive.
    temporal = local.with_name(
        local.stem + ".tmp.npz"
    )

    np.savez_compressed(
        temporal,
        coordenadas=coords,
        dimensiones=dimensiones,
        varianza=varianza,
        segundos=segundos,
        firma=firma
    )

    os.replace(
        temporal,
        local
    )

    respaldar_archivo_343d(local)

    print(
        f"{experimento}: completado "
        f"en {segundos:.1f} segundos. "
        "Checkpoint respaldado."
    )

    return coords, dimensiones, varianza, segundos, False

# ------------------------------------------------------------
# 6. FUNCIÓN DE GRÁFICOS A/B
# ------------------------------------------------------------

def graficar_tsne_343d(
    coordenadas_A,
    coordenadas_B,
    etiquetas,
    n
):

    fig, ejes = plt.subplots(
        1, 2,
        figsize=(15, 6),
        constrained_layout=True
    )

    for ax, exp, coords in zip(
        ejes,
        ["A", "B"],
        [coordenadas_A, coordenadas_B]
    ):

        for clase, nombre, color in [
            (0, "Favorable", "#277DA1"),
            (1, "Adverso", "#F3722C")
        ]:

            mascara = etiquetas == clase

            ax.scatter(
                coords[mascara, 0],
                coords[mascara, 1],
                s=3 if n == 10000 else 2,
                alpha=0.30 if n == 10000 else 0.22,
                color=color,
                label=nombre,
                linewidths=0,
                rasterized=True
            )

        ax.set_title(
            f"Experimento {exp} - {n:,} estudiantes"
        )

        ax.set_xlabel("t-SNE 1")
        ax.set_ylabel("t-SNE 2")

        ax.grid(alpha=0.15)
        ax.legend(markerscale=3)

    fig.suptitle(
        f"OULAD - Semana 8: t-SNE con {n:,} estudiantes\n"
        "Distribución natural | Desarrollo original",
        fontsize=14
    )

    nombre = (
        f"figura_tsne_oulad_343d_{n}.png"
    )

    ruta = RUTA_LOCAL_343D / nombre

    fig.savefig(
        ruta,
        dpi=220,
        bbox_inches="tight"
    )

    respaldar_archivo_343d(ruta)

    plt.show()

# ------------------------------------------------------------
# 7. EJECUTAR SOLO EL ANÁLISIS DE 10.000
# ------------------------------------------------------------

print("\n=== 3.43.D.1.3 t-SNE CON 10.000 ===")

coords_10000_A_343d, dimA_10000, varA_10000, tA_10000, cacheA_10000 = (
    ejecutar_tsne_343d(
        X_A_343d,
        indices_10000_343d,
        "A"
    )
)

coords_10000_B_343d, dimB_10000, varB_10000, tB_10000, cacheB_10000 = (
    ejecutar_tsne_343d(
        X_B_343d,
        indices_10000_343d,
        "B"
    )
)

# ------------------------------------------------------------
# 8. AUDITORÍA TÉCNICA
# ------------------------------------------------------------

print("\n=== 3.43.D.1.4 AUDITORÍA ===")

auditoria_10000_343d = pd.DataFrame([
    {
        "Experimento": "A",
        "Estudiantes": 10000,
        "Variables_originales": 12,
        "Variables_codificadas": dimA_10000,
        "Varianza_PCA": varA_10000,
        "Tiempo_segundos": tA_10000,
        "Recuperado_cache": cacheA_10000
    },
    {
        "Experimento": "B",
        "Estudiantes": 10000,
        "Variables_originales": 48,
        "Variables_codificadas": dimB_10000,
        "Varianza_PCA": varB_10000,
        "Tiempo_segundos": tB_10000,
        "Recuperado_cache": cacheB_10000
    }
])

display(auditoria_10000_343d.round(3))

graficar_tsne_343d(
    coords_10000_A_343d,
    coords_10000_B_343d,
    y_10000_343d,
    10000
)

# ------------------------------------------------------------
# 9. EXPORTAR RESULTADOS
# ------------------------------------------------------------

coordenadas_10000_343d = pd.DataFrame({
    "resultado_binario": y_10000_343d,
    "tsne_A_1": coords_10000_A_343d[:, 0],
    "tsne_A_2": coords_10000_A_343d[:, 1],
    "tsne_B_1": coords_10000_B_343d[:, 0],
    "tsne_B_2": coords_10000_B_343d[:, 1]
})

ruta_csv_10000 = (
    RUTA_LOCAL_343D /
    "coordenadas_tsne_oulad_343d_10000.csv"
)

coordenadas_10000_343d.to_csv(
    ruta_csv_10000,
    index=False
)

respaldar_archivo_343d(ruta_csv_10000)

ruta_aud_10000 = (
    RUTA_LOCAL_343D /
    "auditoria_tsne_oulad_343d_10000.csv"
)

auditoria_10000_343d.to_csv(
    ruta_aud_10000,
    index=False
)

respaldar_archivo_343d(ruta_aud_10000)

print("\n=== ACTIVIDAD 3.43.D.1 FINALIZADA ===")
print("Estudiantes:", 10000)
print("Gráficos: A y B")
print("Resultados: guardados en Drive")
print("Test independiente utilizado: NO")
print("Clasificadores reentrenados: 0")


In [ ]:

# ============================================================
# ACTIVIDAD 3.43.D.2
# ROBUSTEZ INTERPRETATIVA - 19.233 ESTUDIANTES
#
# Requiere ejecutar primero la Actividad 3.43.D.1.
#
# Utiliza todos los estudiantes únicos de desarrollo.
# Una matrícula por estudiante.
# Distribución natural de clases.
#
# NO entrena clasificadores ni utiliza el test.
#
# Asistencia IA: ChatGPT, octubre de 2026.
# Prompt: extender el análisis t-SNE a los 19.233
# estudiantes únicos de OULAD, reutilizando funciones,
# evitando recalcular 10.000 observaciones y generando
# visualizaciones binarias y por resultado académico.
# ============================================================

print("=== 3.43.D.2.1 POBLACIÓN COMPLETA ===")

# ------------------------------------------------------------
# 1. RECUPERAR LA MUESTRA COMPLETA
# ------------------------------------------------------------

assert "indices_unicos_343d" in globals()
assert "ejecutar_tsne_343d" in globals()
assert "graficar_tsne_343d" in globals()

indices_19233_343d = indices_unicos_343d.copy()

assert len(indices_19233_343d) == 19233

assert len(np.unique(
    grupos_343d[indices_19233_343d]
)) == 19233

# Comprobar que la muestra anterior es un subconjunto.
assert set(indices_10000_343d).issubset(
    set(indices_19233_343d)
)

y_19233_343d = y_343d[
    indices_19233_343d
]

favorables_19233 = int(
    (y_19233_343d == 0).sum()
)

adversos_19233 = int(
    (y_19233_343d == 1).sum()
)

print("Estudiantes:", 19233)
print("Favorables:", favorables_19233)
print("Adversos:", adversos_19233)

print(
    "Porcentaje adverso:",
    round(100 * y_19233_343d.mean(), 2),
    "%"
)

print("Cobertura del desarrollo:", "100 %")

# ------------------------------------------------------------
# 2. EJECUTAR t-SNE SOBRE LOS 19.233 ESTUDIANTES
# ------------------------------------------------------------

print("\n=== 3.43.D.2.2 t-SNE COMPLETO ===")

coords_19233_A_343d, dimA_19233, varA_19233, tA_19233, cacheA_19233 = (
    ejecutar_tsne_343d(
        X_A_343d,
        indices_19233_343d,
        "A"
    )
)

coords_19233_B_343d, dimB_19233, varB_19233, tB_19233, cacheB_19233 = (
    ejecutar_tsne_343d(
        X_B_343d,
        indices_19233_343d,
        "B"
    )
)

# ------------------------------------------------------------
# 3. AUDITORÍA TÉCNICA
# ------------------------------------------------------------

print("\n=== 3.43.D.2.3 AUDITORÍA ===")

auditoria_19233_343d = pd.DataFrame([
    {
        "Experimento": "A",
        "Estudiantes": 19233,
        "Variables_originales": 12,
        "Variables_codificadas": dimA_19233,
        "Varianza_PCA": varA_19233,
        "Tiempo_segundos": tA_19233,
        "Recuperado_cache": cacheA_19233
    },
    {
        "Experimento": "B",
        "Estudiantes": 19233,
        "Variables_originales": 48,
        "Variables_codificadas": dimB_19233,
        "Varianza_PCA": varB_19233,
        "Tiempo_segundos": tB_19233,
        "Recuperado_cache": cacheB_19233
    }
])

display(
    auditoria_19233_343d.round(3)
)

# ------------------------------------------------------------
# 4. GRÁFICO A VS B CON LA POBLACIÓN COMPLETA
# ------------------------------------------------------------

print("\n=== 3.43.D.2.4 GRÁFICO COMPLETO ===")

graficar_tsne_343d(
    coords_19233_A_343d,
    coords_19233_B_343d,
    y_19233_343d,
    19233
)

# ------------------------------------------------------------
# 5. COMPARACIÓN VISUAL 10.000 VS 19.233
# ------------------------------------------------------------

print("\n=== 3.43.D.2.5 SENSIBILIDAD VISUAL ===")

fig, ejes = plt.subplots(
    2, 2,
    figsize=(16, 12),
    constrained_layout=True
)

configuraciones = [
    (
        10000,
        indices_10000_343d,
        coords_10000_A_343d,
        coords_10000_B_343d
    ),
    (
        19233,
        indices_19233_343d,
        coords_19233_A_343d,
        coords_19233_B_343d
    )
]

for fila, (
    n,
    indices,
    puntos_A,
    puntos_B
) in enumerate(configuraciones):

    etiquetas = y_343d[indices]

    for columna, (
        exp,
        puntos
    ) in enumerate([
        ("A", puntos_A),
        ("B", puntos_B)
    ]):

        ax = ejes[fila, columna]

        for clase, nombre, color in [
            (0, "Favorable", "#277DA1"),
            (1, "Adverso", "#F3722C")
        ]:

            mascara = etiquetas == clase

            ax.scatter(
                puntos[mascara, 0],
                puntos[mascara, 1],
                s=3 if n == 10000 else 2,
                alpha=0.30 if n == 10000 else 0.22,
                color=color,
                label=nombre,
                linewidths=0,
                rasterized=True
            )

        ax.set_title(
            f"Experimento {exp} - {n:,} estudiantes"
        )

        ax.set_xlabel("t-SNE 1")
        ax.set_ylabel("t-SNE 2")

        ax.legend(markerscale=3)
        ax.grid(alpha=0.12)

fig.suptitle(
    "OULAD - Robustez interpretativa de t-SNE\n"
    "Semana 8 | Datos de desarrollo | "
    "Distribución natural de clases",
    fontsize=14
)

ruta_comparativa_343d = (
    RUTA_LOCAL_343D /
    "figura_tsne_oulad_343d_robustez.png"
)

fig.savefig(
    ruta_comparativa_343d,
    dpi=220,
    bbox_inches="tight"
)

respaldar_archivo_343d(
    ruta_comparativa_343d
)

plt.show()

# ------------------------------------------------------------
# 6. VISUALIZACIÓN CON CUATRO RESULTADOS ACADÉMICOS
# ------------------------------------------------------------

print("\n=== 3.43.D.2.6 CUATRO RESULTADOS ===")

info_343d = None

if (
    "oulad_tablas" in globals()
    and "studentInfo.csv" in oulad_tablas
):

    info_343d = oulad_tablas[
        "studentInfo.csv"
    ]

else:

    ruta_info = (
        Path(RUTA_BASE) /
        "datos" /
        "studentInfo.csv"
    )

    if ruta_info.exists():
        info_343d = pd.read_csv(
            ruta_info
        )

if (
    info_343d is not None
    and "claves_train" in datos_343d
):

    claves = [
        "code_module",
        "code_presentation",
        "id_student"
    ]

    claves_train = (
        datos_343d["claves_train"]
        .reset_index(drop=True)
    )

    assert len(claves_train) == 21164

    info_resultados = (
        claves_train[claves]
        .merge(
            info_343d[
                claves + ["final_result"]
            ],
            on=claves,
            how="left",
            validate="one_to_one",
            sort=False
        )
    )

    assert len(info_resultados) == 21164

    # Confirmar que los resultados académicos
    # coinciden con el target binario original.
    assert np.array_equal(
        info_resultados[
            "final_result"
        ].isin(
            ["Fail", "Withdrawn"]
        ).astype(int).to_numpy(),
        y_343d
    )

    etiquetas_4 = info_resultados[
        "final_result"
    ].to_numpy()[indices_19233_343d]

    print(
        "Distribución de los cuatro resultados:"
    )

    display(
        pd.Series(
            etiquetas_4,
            name="Resultado"
        ).value_counts()
    )

    colores_4 = {
        "Distinction": "#457B9D",
        "Pass": "#77B7D5",
        "Fail": "#F4A261",
        "Withdrawn": "#D1495B"
    }

    fig4, ejes4 = plt.subplots(
        1, 2,
        figsize=(16, 6),
        constrained_layout=True
    )

    for ax, exp, puntos in zip(
        ejes4,
        ["A", "B"],
        [
            coords_19233_A_343d,
            coords_19233_B_343d
        ]
    ):

        for resultado, color in colores_4.items():

            mascara = (
                etiquetas_4 == resultado
            )

            ax.scatter(
                puntos[mascara, 0],
                puntos[mascara, 1],
                s=2,
                alpha=0.23,
                color=color,
                label=resultado,
                linewidths=0,
                rasterized=True
            )

        ax.set_title(
            f"Experimento {exp} - 19.233 estudiantes"
        )

        ax.set_xlabel("t-SNE 1")
        ax.set_ylabel("t-SNE 2")

        ax.grid(alpha=0.12)
        ax.legend(markerscale=4)

    fig4.suptitle(
        "OULAD - Resultados académicos originales\n"
        "Pass, Distinction, Fail y Withdrawn",
        fontsize=14
    )

    ruta_cuatro_343d = (
        RUTA_LOCAL_343D /
        "figura_tsne_oulad_343d_cuatro_resultados.png"
    )

    fig4.savefig(
        ruta_cuatro_343d,
        dpi=220,
        bbox_inches="tight"
    )

    respaldar_archivo_343d(
        ruta_cuatro_343d
    )

    plt.show()

    print(
        "Gráfico de cuatro categorías: COMPLETADO"
    )

else:

    print(
        "No se encontraron las etiquetas "
        "originales en esta sesión."
    )

    print(
        "Se omite únicamente el gráfico "
        "opcional de cuatro categorías."
    )

# ------------------------------------------------------------
# 7. EXPORTAR COORDENADAS
# ------------------------------------------------------------

print("\n=== 3.43.D.2.7 EXPORTACIÓN ===")

coordenadas_19233_343d = pd.DataFrame({
    "resultado_binario": y_19233_343d,
    "tsne_A_1": coords_19233_A_343d[:, 0],
    "tsne_A_2": coords_19233_A_343d[:, 1],
    "tsne_B_1": coords_19233_B_343d[:, 0],
    "tsne_B_2": coords_19233_B_343d[:, 1]
})

ruta_csv_19233 = (
    RUTA_LOCAL_343D /
    "coordenadas_tsne_oulad_343d_19233.csv"
)

coordenadas_19233_343d.to_csv(
    ruta_csv_19233,
    index=False
)

respaldar_archivo_343d(
    ruta_csv_19233
)

ruta_aud_19233 = (
    RUTA_LOCAL_343D /
    "auditoria_tsne_oulad_343d_19233.csv"
)

auditoria_19233_343d.to_csv(
    ruta_aud_19233,
    index=False
)

respaldar_archivo_343d(
    ruta_aud_19233
)

# ------------------------------------------------------------
# 8. COMPARACIÓN DE COMPOSICIÓN
# ------------------------------------------------------------

print("\n=== 3.43.D.2.8 COMPOSICIÓN COMPARADA ===")

tabla_muestras_343d = pd.DataFrame([
    {
        "Muestra": 10000,
        "Favorables": int(
            (y_10000_343d == 0).sum()
        ),
        "Adversos": int(
            (y_10000_343d == 1).sum()
        ),
        "Adversos_pct": (
            100 * y_10000_343d.mean()
        ),
        "Cobertura_pct": (
            100 * 10000 / 19233
        )
    },
    {
        "Muestra": 19233,
        "Favorables": favorables_19233,
        "Adversos": adversos_19233,
        "Adversos_pct": (
            100 * y_19233_343d.mean()
        ),
        "Cobertura_pct": 100.0
    }
])

display(
    tabla_muestras_343d.round(2)
)

ruta_muestras = (
    RUTA_LOCAL_343D /
    "resumen_muestras_tsne_oulad_343d.csv"
)

tabla_muestras_343d.to_csv(
    ruta_muestras,
    index=False
)

respaldar_archivo_343d(
    ruta_muestras
)

# ------------------------------------------------------------
# 9. VERIFICACIÓN FINAL
# ------------------------------------------------------------

print("\n=== 3.43.D.2.9 CONTROLES FINALES ===")

assert coords_19233_A_343d.shape == (19233, 2)
assert coords_19233_B_343d.shape == (19233, 2)

assert np.isfinite(
    coords_19233_A_343d
).all()

assert np.isfinite(
    coords_19233_B_343d
).all()

assert ruta_comparativa_343d.exists()
assert ruta_csv_19233.exists()

print("Estudiantes únicos:", 19233)
print("Cobertura del desarrollo: 100 %")
print("Balanceo artificial: NO")
print("Proyecciones acumuladas: 4")
print("Gráficos comparativos: generados")
print("Resultados guardados en Google Drive: SÍ")
print("Modelos predictivos reentrenados: 0")
print("Test independiente utilizado: NO")

print("\n=== ACTIVIDAD 3.43.D.2 FINALIZADA ===")



### 3.43.D.3. Discusión y conclusión del análisis de robustez interpretativa

#### Objetivo

La presente actividad se desarrolló como un complemento metodológico de la visualización t-SNE inicial, con el objetivo de examinar la estabilidad de los patrones visuales al incrementar el tamaño de la población representada.

La visualización original, construida con 2.000 estudiantes y clases artificialmente equilibradas, se amplió a una muestra de 10.000 estudiantes y posteriormente a la totalidad de 19.233 estudiantes únicos disponibles en el conjunto de desarrollo de la semana 8.

Se mantuvo una sola matrícula por estudiante, se conservaron las proporciones naturales de las clases y se utilizaron las matrices originales de características estáticas y longitudinales.

#### Composición de las muestras

La muestra de 10.000 estudiantes representó el 51,99 % de los estudiantes únicos de desarrollo y presentó una proporción de resultados adversos del 41,33 %.

La población completa de 19.233 estudiantes presentó una proporción adversa del 40,75 %, equivalente a 7.837 resultados adversos y 11.396 favorables.

La diferencia de prevalencia adversa entre ambas muestras ampliadas fue de 0,58 puntos porcentuales.

Estas proporciones corresponden a una matrícula seleccionada por estudiante y no necesariamente a la distribución de todas las matrículas originales.

#### Estabilidad del Experimento A

La representación de características estáticas conservó agrupaciones locales reconocibles al incrementar el tamaño muestral.

Sin embargo, los resultados favorables y adversos continuaron presentando una mezcla considerable dentro de las agrupaciones.

No se observó una separación visual clara y consistente entre ambas categorías académicas.

#### Estabilidad del Experimento B

La representación longitudinal conservó un patrón de distribución diferencial entre resultados favorables y adversos en las muestras de 10.000 y 19.233 estudiantes.

En ambas proyecciones se observaron regiones con mayor concentración relativa de resultados adversos y otras con mayor presencia de resultados favorables.

El patrón visual identificado inicialmente con 2.000 estudiantes permaneció reconocible al ampliar la cobertura hasta la totalidad de estudiantes únicos de desarrollo.

No obstante, se mantuvo un solapamiento importante entre clases, por lo que la estructura observada no constituye una separación perfecta de los resultados académicos.

#### Análisis de cuatro categorías académicas

La visualización complementaria de los resultados originales incluyó 9.160 casos Pass, 2.236 Distinction, 5.150 Fail y 2.687 Withdrawn.

La representación permitió observar la distribución espacial de estas categorías, pero no mostró una separación inequívoca entre reprobación y retiro del módulo, ni entre aprobación y distinción.

Esta observación plantea una posible investigación futura mediante modelos longitudinales multiclase, cuya utilidad deberá demostrarse mediante validación predictiva específica.

#### Consideraciones metodológicas

El incremento del tamaño muestral permite examinar la estabilidad descriptiva de los patrones y ampliar la cobertura de perfiles estudiantiles representados.

Sin embargo, las proyecciones t-SNE se ajustaron independientemente y no permiten comparar directamente coordenadas, distancias globales, densidades o geometrías entre tamaños muestrales.

Además, la muestra inicial de 2.000 estudiantes fue artificialmente equilibrada, mientras que las muestras ampliadas conservaron la distribución natural de las clases. Por tanto, las diferencias entre sus gráficos no pueden atribuirse exclusivamente al tamaño de muestra.

La reducción preliminar mediante PCA conservó aproximadamente el 90 % de la varianza en el Experimento A y entre el 86,9 % y el 87 % en el Experimento B para las muestras ampliadas.

Las visualizaciones constituyen evidencia exploratoria y no una prueba estadística de superioridad predictiva.

#### Integridad del experimento

Se utilizaron exclusivamente matrices originales de desarrollo.

No se accedió al conjunto de prueba independiente, no se reentrenaron clasificadores y no se modificaron hiperparámetros ni umbrales.

Las proyecciones, figuras y auditorías técnicas se almacenaron mediante checkpoints persistentes en Google Drive.

#### Conclusión

**El análisis de sensibilidad muestra que las diferencias cualitativas entre las representaciones estática y longitudinal permanecen reconocibles al incrementar la muestra de 2.000 a 10.000 y 19.233 estudiantes.**

La representación estática mantiene una mezcla importante de resultados favorables y adversos dentro de agrupaciones locales.

La representación longitudinal conserva regiones con distintas concentraciones relativas de ambas clases, aunque también presenta un solapamiento considerable.

La persistencia de estos patrones en una población más amplia fortalece la robustez descriptiva de la interpretación visual inicial, sin establecer una relación causal ni cuantificar por sí misma el rendimiento predictivo.

**La evidencia concluyente sobre la mejora del Experimento B procede de la evaluación independiente**, donde el F1 macro fue superior al del Experimento A en las cuatro ventanas temporales.

En consecuencia, la presente actividad aporta evidencia visual complementaria compatible con la hipótesis principal, sin reemplazar los resultados obtenidos mediante validación cruzada y evaluación independiente.


## Anexo metodológico: interpretación estadística y extensión Optuna


### Justificación de la corrección de Holm para comparaciones múltiples

Para comparar el desempeño de los modelos de Machine Learning, se emplearon pruebas t pareadas con corrección por dependencia entre particiones de validación cruzada repetida. Este procedimiento permitió estimar las diferencias de rendimiento considerando que las particiones de entrenamiento y validación no constituyen observaciones completamente independientes.

Sin embargo, debido a que se realizaron múltiples contrastes de hipótesis entre algoritmos y ventanas temporales, se incorporó adicionalmente el **procedimiento de Holm–Bonferroni** para controlar la tasa de error familiar (*Family-Wise Error Rate*, FWER).

La corrección de Holm permite controlar el riesgo de rechazar incorrectamente al menos una hipótesis nula dentro de una familia de comparaciones, evitando interpretar como significativas diferencias que podrían surgir como consecuencia de realizar numerosos contrastes.

En la comparación complementaria entre Transformer C, Regresión Logística B y Random Forest B, se definió una familia de ocho contrastes: dos comparaciones por cada una de las cuatro semanas de predicción (2, 4, 6 y 8). Los valores p obtenidos mediante la prueba t corregida fueron ajustados conjuntamente utilizando el procedimiento secuencial de Holm, manteniendo un nivel de significancia de α = 0,05.

**Resultados:** de los ocho contrastes realizados, únicamente la comparación entre Transformer C y Random Forest B en la semana 6 mantuvo significancia estadística después del ajuste (p ajustado = 0,046376).

Aunque el Transformer presentó ventajas descriptivas frente a Random Forest B en todas las ventanas evaluadas, las restantes diferencias no alcanzaron significancia estadística tras el ajuste de Holm.

Estos resultados indican que la superioridad descriptiva de un algoritmo no necesariamente implica evidencia estadística suficiente para afirmar que presenta un mejor rendimiento general.

El procedimiento se utilizó como una medida de rigor metodológico para evitar conclusiones excesivamente optimistas derivadas de múltiples comparaciones.

**Limitación:** los contrastes se consideran exploratorios, debido a que las arquitecturas y sus resultados fueron examinados durante el desarrollo experimental. Además, la prueba t corregida proporciona una aproximación estadística sujeta a los supuestos del procedimiento, por lo que los valores p ajustados deben interpretarse como evidencia complementaria y no como una demostración definitiva de superioridad.

### Extensión metodológica: optimización automatizada mediante Optuna

Como etapa complementaria al proceso inicial de selección de modelos, se incorporó Optuna para explorar estrategias adaptativas de optimización de hiperparámetros.

Mientras que la evaluación inicial utilizó GridSearchCV sobre espacios discretos de búsqueda, esta extensión propone explorar rangos más amplios mediante el algoritmo Tree-structured Parzen Estimator (TPE).

El objetivo fue determinar si la optimización automatizada permite mejorar el rendimiento de los modelos longitudinales, particularmente Regresión Logística y Random Forest, y analizar su efecto sobre la capacidad de generalización.

La métrica principal siguió siendo el F1 macro, con especial atención al recall de resultados adversos, ROC-AUC y brechas entre entrenamiento y validación.

Para preservar la validez experimental, la optimización se realizó exclusivamente con datos de desarrollo y respetando las agrupaciones por estudiante. Las transformaciones y la selección de características permanecieron integradas en los pipelines de entrenamiento.

Los resultados de Optuna se compararon con los obtenidos mediante GridSearchCV, distinguiendo entre mejoras observadas durante la optimización y estimaciones de generalización obtenidas mediante validación externa.

Esta extensión permitió evaluar si una búsqueda de hiperparámetros más flexible añade valor al modelado longitudinal del riesgo académico, sin modificar retrospectivamente los resultados de los experimentos originales.
**Ubicación y nivel de evidencia:** los contrastes estadísticos y la optimización Optuna descritos en este anexo corresponden a la etapa de desarrollo; no se trata de nuevas evaluaciones sobre el test reservado. Véanse Nadeau y Bengio [5], Holm [6] y Optuna [7].


## Revisión de entrega y decisiones congeladas (14 de octubre de 2026)

**Propósito:** distinguir resultados históricos documentados de celdas que pueden volver a ejecutarse. Esta sección no constituye un nuevo experimento.

**Evaluación principal:** Regresión Logística A frente a B en semanas 2, 4, 6 y 8. Las métricas del test fueron calculadas previamente y quedaron congeladas: mejoras absolutas de F1 macro de +0,06887, +0,07645, +0,09741 y +0,11510. El código actual de la Actividad 3.39 **recupera los CSV históricos** y no recalcula el test. Los análisis complementarios sobre ese test son descriptivos y no se emplean para elegir un nuevo modelo.

**Experimentos de desarrollo:** Random Forest, Transformer, optimización mediante Optuna, calibración, contraste estadístico y simulación de alertas. Sus resultados no constituyen nuevas evaluaciones independientes. Los intervalos bootstrap de la comparación principal fueron obtenidos sobre estudiantes agrupados, sin ajustes simultáneos por multiplicidad y condicionados a los modelos ya ajustados.

**t-SNE:** estudio exploratorio sobre matrices de desarrollo, primero con 2.000 estudiantes y luego con 10.000 y 19.233 estudiantes únicos. Los dos tamaños mayores conservan la prevalencia natural después de seleccionar una matrícula por estudiante; las geometrías de proyecciones independientes no son directamente comparables ni demuestran superioridad predictiva.

**GitHub:** conservar una versión legible del notebook y un README que explique la configuración de `/content/drive/MyDrive/Proyecto_ML`, la descarga de OULAD, la ubicación de los snapshots, la dependencia de los resultados finales históricos y la documentación de paquetes. No subir datos con identificadores individuales. La publicación del repositorio no equivale a validar una ejecución limpia de todo el notebook en un runtime nuevo.


# Actividad 3.44. Referencias bibliográficas y trazabilidad

## 3.44.1. Referencias en formato IEEE

[1] J. Kuzilek, M. Hlosta y Z. Zdrahal, “Open University Learning Analytics dataset,” *Scientific Data*, vol. 4, art. 170171, 2017, doi: 10.1038/sdata.2017.171.

[2] F. Pedregosa *et al*., “Scikit-learn: Machine Learning in Python,” *Journal of Machine Learning Research*, vol. 12, pp. 2825–2830, 2011. [En línea]. Disponible: https://www.jmlr.org/papers/v12/pedregosa11a.html

[3] L. Breiman, “Random Forests,” *Machine Learning*, vol. 45, pp. 5–32, 2001, doi: 10.1023/A:1010933404324.

[4] L. van der Maaten y G. Hinton, “Visualizing Data using t-SNE,” *Journal of Machine Learning Research*, vol. 9, pp. 2579–2605, 2008. [En línea]. Disponible: https://www.jmlr.org/papers/v9/vandermaaten08a.html

[5] C. Nadeau y Y. Bengio, “Inference for the Generalization Error,” *Machine Learning*, vol. 52, pp. 239–281, 2003, doi: 10.1023/A:1024068626366.

[6] S. Holm, “A Simple Sequentially Rejective Multiple Test Procedure,” *Scandinavian Journal of Statistics*, vol. 6, no. 2, pp. 65–70, 1979. [En línea]. Disponible: https://www.jstor.org/stable/4615733

[7] T. Akiba, S. Sano, T. Yanase, T. Ohta y M. Koyama, “Optuna: A Next-generation Hyperparameter Optimization Framework,” en *Proc. 25th ACM SIGKDD Int. Conf. Knowledge Discovery & Data Mining*, 2019, pp. 2623–2631, doi: 10.1145/3292500.3330701.

[8] A. Vaswani *et al*., “Attention Is All You Need,” en *Advances in Neural Information Processing Systems*, vol. 30, 2017. [En línea]. Disponible: https://papers.nips.cc/paper/7181-attention-is-all-you-need

[9] I. Guyon y A. Elisseeff, “An Introduction to Variable and Feature Selection,” *Journal of Machine Learning Research*, vol. 3, pp. 1157–1182, 2003. [En línea]. Disponible: https://www.jmlr.org/papers/v3/guyon03a.html

[10] R. Kohavi, “A Study of Cross-Validation and Bootstrap for Accuracy Estimation and Model Selection,” en *Proc. 14th Int. Joint Conf. Artificial Intelligence (IJCAI)*, 1995, pp. 1137–1145. [En línea]. Disponible: https://www.ijcai.org/Proceedings/95-2/Papers/016.pdf

[11] J. Bergstra y Y. Bengio, “Random Search for Hyper-Parameter Optimization,” *Journal of Machine Learning Research*, vol. 13, pp. 281–305, 2012. [En línea]. Disponible: https://www.jmlr.org/papers/v13/bergstra12a.html

[12] T. Saito y M. Rehmsmeier, “The Precision-Recall Plot Is More Informative than the ROC Plot When Evaluating Binary Classifiers on Imbalanced Datasets,” *PLOS ONE*, vol. 10, no. 3, art. e0118432, 2015, doi: 10.1371/journal.pone.0118432.

[13] A. Paszke *et al*., “PyTorch: An Imperative Style, High-Performance Deep Learning Library,” en *Advances in Neural Information Processing Systems*, vol. 32, 2019. [En línea]. Disponible: https://papers.nips.cc/paper/9015-pytorch-an-imperative-style-high-performance-deep-learning-library

## 3.44.2. Correspondencia entre bibliografía y método

- **Datos OULAD:** descripción original del conjunto y significado de `Fail`, `Withdrawn`, `Pass`, `Distinction` [1]. La descarga emplea un espejo alojado en Kaggle, pero la referencia académica corresponde a la publicación original [1].
- **Implementación clásica y pipelines:** scikit-learn [2]; fundamento de Random Forest [3]; selección tipo filtro y sus limitaciones [9].
- **Validación e inferencia:** estimación por validación cruzada [10], dependencia estadística entre folds [5], control de comparaciones múltiples [6].
- **Modelos secuenciales:** arquitectura conceptual de atención [8], implementación con PyTorch [13]. El Transformer del estudio es una adaptación, no una reproducción literal del modelo del artículo.
- **Optimización:** implementación de Optuna [7] y antecedentes de estrategias de búsqueda [11]. La amplitud de los espacios de búsqueda también cambia entre GridSearchCV y Optuna; la mejora no se atribuye exclusivamente al optimizador.
- **Medición:** ROC-AUC, precisión, recall y F1 mediante scikit-learn [2]; consideraciones sobre precisión-recall y ROC en datasets desbalanceados [12].
- **Visualización:** t-SNE [4] con implementación de scikit-learn [2]; el color representa etiquetas conocidas, no supervisión de t-SNE.

## 3.44.3. Transparencia en el uso de IA y código

ChatGPT se utilizó como apoyo en redacción, construcción, depuración y revisión de código. Las celdas incluyen comentarios de procedencia cuando el encargo original fue documentado. Para las celdas auxiliares que no conservan un prompt histórico, **no debe inventarse una transcripción literal**: se indica su propósito y se reconoce esa limitación de trazabilidad.

Los artículos científicos son **fundamentos metodológicos**, no una afirmación de que se haya copiado código de sus autores. El código implementado se apoya en las API de scikit-learn, PyTorch, pandas, NumPy, Matplotlib y Optuna. Si se incorporó algún fragmento de un tutorial externo, debe atribuirse en la celda concreta además de esta bibliografía general.

## 3.44.4. Límite de reproducibilidad y publicación

Los resultados independientes de 3.39 se obtuvieron una sola vez con el protocolo congelado. El notebook actual los recupera desde CSV existentes; **no vuelve a ejecutar el test**. Para reproducir completamente los resultados históricos desde cero se necesitan los artefactos originales de esa ejecución, las versiones de dependencias y el entorno o procedimiento histórico correspondiente. En un clon público sin dichos archivos, es esperado que las celdas protegidas soliciten restaurarlos.

Los archivos de datos y checkpoints permanecen en Google Drive privado. El repositorio GitHub debe publicar únicamente lo autorizado, nunca credenciales, tokens o ficheros derivados que incluyan identificadores individuales de estudiantes. El nombre `Withdrawn` se refiere al retiro del módulo y no demuestra deserción universitaria definitiva.
